# TTS duration and GPU memory benchmark — Kaggle T4 ×2

This single notebook contains all six adapters: **Kokoro, F5-TTS, EmotiVoice,
StyleTTS2, OmniVoice, and Supertonic 3**. It installs each model in a separate
environment, tests configurable profiles and 10–500-word passages, and creates a
downloadable ZIP with named WAVs, timing/error reports, memory traces, source
provenance, and full generation logs.

This revision fixes the F5 eager-import dependency failure observed in the
uploaded Kaggle run and replaces arbitrary chunk breaks with punctuation-aware
splitting. It removes added per-chunk gaps, protects PCM exports from clipping,
and uses male defaults. **Exact duration alone does not establish word completeness.**
Independent CPU ASR and silence/clipping diagnostics are included after synthesis.

**Before running:** import this notebook into Kaggle, enable **Internet**, select
**GPU T4 x2**, and edit the experiment cell below. Use a fresh kernel. First setup
downloads several GB; runtime environments/weights stay under `/tmp`, and reports
and audio stay under `/kaggle/working`. Nothing is published externally.

Up to two workers run concurrently, each restricted to one physical GPU UUID.
Different models are preferred whenever possible. Generation progress and current
GPU memory appear live. The default target is **5,000,000,000 bytes per GPU**, with
allocator limits and a 10 ms external NVML monitor that terminates an observed
overrun. **Sampling cannot prove an instantaneous peak ceiling.** Both sampled
process/device usage and native allocator peaks are retained separately.

Duration means the **complete exported WAV**, including silence, configured
pauses, and resampling. Frozen duration/frame/token plans are reused in synthesis;
WAV headers are independently checked afterward. F5 and OmniVoice preview a
chosen allocation, rather than predicting an unconstrained natural reading.
StyleTTS2's exact preview needs style diffusion and may be slower. Quality and
intelligibility must be judged by listening, even when duration matches exactly.

This notebook's GPU execution must be validated by your Kaggle run: it has not
been executed on two T4s locally. Failures are reported rather than silently
removing a model from the comparison.


## Voice and configuration reference

This guide is exhaustive for the **six embedded adapters and their pinned
checkpoints**. Numeric settings admit infinitely many combinations; the tables
give every exposed key, its domain, and its effect. Options from unrelated
checkpoints, training pipelines, or other frontends are outside this notebook.
The option inventory is also extracted directly from each adapter's argument
parser, so it includes file/cache/runtime controls as well as synthesis knobs.

| Model | Voice choices in this notebook | Reference WAV? | Explicit emotion / emphasis |
|---|---|---|---|
| Kokoro | 54 shipped presets, compatible `.pt` packs, weighted mixtures | No WAV-to-speaker encoder | No named emotion or word-emphasis control; speed and punctuation affect delivery |
| F5-TTS | Any suitable speaker recording; no finite preset bank | WAV + exact transcript | Reference delivery conditions style; no independent named emotion, pitch, or word-emphasis key |
| EmotiVoice | 2,014 ordered speaker labels in the pinned vocabulary | No | Text style prompt; pitch/energy predictor scales; phoneme-index emphasis |
| StyleTTS2 | Any suitable speaker recording; optional official demo WAVs | WAV only | Reference + sampled style/prosody; continuous style weights and pitch; no named emotion key |
| OmniVoice | Any suitable speaker recording, saved prompt, or supported attribute combinations | WAV + exact transcript, or cached prompt | Voice-design attributes including whisper; no validated named emotion or word-emphasis vocabulary in this pinned release |
| Supertonic 3 | M1–M5 / F1–F5; compatible custom voice-style JSON | No local WAV encoder | Supported expression tags in text; no separate emotion or per-word emphasis flag |

**What “all reference voices” means:** F5, StyleTTS2, and OmniVoice do not restrict
you to a named list of speakers. A valid recording can represent any speaker;
recording suitability and cloning fidelity must be checked by listening. Kokoro
voice packs and Supertonic style JSON are model embeddings, not audio files.
EmotiVoice speaker labels are learned identities, not speaker WAV filenames.

The shared notebook reference field accepts a readable, finite, nonsilent **mono
1–8 second** WAV. Prefer one speaker, steady recording level, little background
noise, and 3–8 seconds of continuous speech. For F5/Omni, transcribe the actual
spoken recording exactly. A transcript is unnecessary for StyleTTS2. Individual
adapter bounds are documented below; the shared notebook uses the narrower
1–8 second contract. Reference preprocessing and model loading are separately
timed and are not part of a warmed duration-only preview.

### How to change settings

Edit `CONFIG`, `TEXT_CASES`, and `MODEL_PROFILES` in the experiment cell. Adapter
keys inside `parameters` use underscores: CLI `--cfg-strength` becomes
`cfg_strength`. JSON/Python booleans must be actual booleans, not strings such as
`"False"`. Omit optional keys to retain their adapter defaults; profile entries
whose value is `None` are omitted before invoking the script.

```python
CONFIG["speaker_mode"] = "auto"
CONFIG["reference_audio"] = "/kaggle/input/my-voice/reference.wav"
CONFIG["reference_text"] = "The exact words spoken in the reference recording."

# Add another take configuration without changing the original profile:
MODEL_PROFILES["kokoro"].append({
    "name": "george_comparison", "enabled": True, "n": 2, "seed": 900,
    "parameters": {"voice": "bm_george", "language": "b", "speed": 0.95},
})
```

The example is a configuration recipe; replace its WAV path and transcript
before using it. It does not upload or download a voice automatically.

Settings merge in this order: global sample rate/pause/take settings → installed
model asset paths → profile `parameters` → reference routing → Omni
`clone_parameters`. Thus a profile can override export rate, pause, voice,
compatible asset paths, and native synthesis controls. The saved per-job
`config.json` is the authoritative resolved configuration. Standalone CLI flags
override a standalone JSON configuration file.

### Every experiment-level setting

These are supervisor settings, distinct from the model parameters below.

| `CONFIG` key | Notebook default / permitted values | Exact purpose |
|---|---|---|
| `install_models` | All six; list of `kokoro`, `f5`, `emotivoice`, `styletts2`, `omnivoice`, `supertonic` | Install and schedule only these models; enabled profiles for omitted models do not run |
| `force_setup` | `False` / boolean | Request reinstall for Torch and the pip-installed Kokoro/Omni source packages; it does not delete all cached assets or force every dependency to reinstall |
| `torch_cuda_variant` | `"auto"`, wheel name, or per-model dictionary | Auto uses cu121 for Torch 2.5.1 models and cu128 for Torch 2.8.0 Omni/Super; 2.5.1 supports cu118/cu121/cu124, 2.8 supports cu126/cu128/cu129; Super must retain cu128 in this setup |
| `strict_cuda_driver` | `False` / boolean | If true, Super setup requires NVIDIA Linux driver >=570.26; runtime CUDA/provider preflight remains necessary either way |
| `n` | `2`, positive integer | Takes per model/profile/text; profile-level `n` overrides it; the notebook rejects invalid signed-64-bit seed ranges |
| `seed` | `1234`, nonnegative integer | Starting seed; takes use consecutive seeds. EmotiVoice is deterministic, so changing seed alone does not create different delivery |
| `sample_rate` | `48000`, integer | Export Hz for all models unless overridden in a profile; synthesis remains at the model's native rate. Adapter-specific bounds apply |
| `pause_ms` | `0.0`, nonnegative number | Extra digital silence between planned chunks; native punctuation/silence remains. Profile override allowed; F5/Style cannot combine a positive pause with a positive crossfade |
| `max_vram_gb` | `5.0`, `(0,5]` | Decimal GB target per GPU; passed to adapters and external monitor. Allocator limits plus NVML polling are not an instantaneous process-wide certificate |
| `nvml_interval_ms` | `10.0`, `[1,1000]` | Requested memory polling interval; actual polling can be slower under load |
| `enforce_observed_budget` | `True` / boolean | Stop workers after a sampled overrun and retain partial files/logs; false records overruns without budget termination |
| `budget_scope` | `"device_total"` or `"process_tree"` | Device total includes baseline/other processes. Process tree counts tracked worker/descendants; if per-process accounting is unavailable it falls back to device total |
| `job_timeout_seconds` | `3600`, positive seconds | Maximum worker lifetime including interpreter startup, model/reference loading, planning, and all takes |
| `gpu_release_timeout_seconds` | `20`, positive seconds | Wait after worker exit for tracked GPU processes to disappear before reusing a slot; release failure disables that slot |
| `cpu_threads_per_job` | `4`, positive integer | Sets OMP/MKL environment limits and CPU quality-helper threads; an adapter's explicit `threads` key can separately configure its own pools |
| `require_two_t4` | `True` / boolean | Require two visible T4s. False allows other two-GPU hardware; it does not change the fixed two-slot scheduler into a single-GPU runner |
| `speaker_mode` | `"auto"`, `"reference"`, `"default"` | Auto uses supplied global WAV for cloning-capable models, else male defaults. Reference requires a global user WAV and blocks preset-only models. Default ignores the global WAV and uses male fallback; explicit per-profile references still work |
| `reference_audio` | `None` or file path | Global reference for F5/Style and requested Omni cloning; if absent in auto/default mode, CPU Kokoro creates a disclosed male reference |
| `reference_text` | `None` or string | Exact global WAV transcript for F5/Omni; Style needs only the WAV |
| `include_reference_in_zip` | `True` / boolean | Include the shared reference WAV and transcript in the final ZIP; per-profile external WAVs are not automatically collected by this switch |
| `preview_players` | `6`, nonnegative integer | Maximum distinct models shown as listening players in the notebook; all created WAVs still enter the ZIP |
| `quality_checks` | `True` / boolean | Run CPU acoustic diagnostics before packaging; false skips them and ASR |
| `asr_quality_check` | `True` / boolean | Add independent CPU/int8 ASR comparison; adds elapsed time without TTS GPU allocation |
| `asr_model` | `"base.en"`, supported faster-whisper name or CTranslate2 path | Base English model is revision-pinned. `small.en` is a larger English alternative; use multilingual models for other languages. Other remote names are not revision-pinned by this notebook |
| `asr_language` | `"en"` or recognizer-supported language code | Force the independent recognizer's language; this does not set TTS language or translate the text |
| `quality_timeout_seconds` | `7200`, positive seconds | Timeout for the quality-helper subprocess; dependency/model downloads occur before that subprocess and are not included in this timeout |

### Texts and profile fields

| Field | Meaning |
|---|---|
| `TEXT_CASES` | List of `{name, text}` objects; each stripped text must have 10–500 whitespace-separated words. Names identify cases; complete paragraphs and punctuation are preserved |
| `WORD_COUNTS` / `SAMPLE_PASSAGES` | Convenience defaults for the three supplied 10/100/500-word texts. Other counts require your own full text; the notebook does not repeat or cut the supplied passages |
| Profile `name` | Human-readable experiment name, sanitized into filenames; ordinal/text/config hashes prevent collisions even if names repeat |
| Profile `enabled` | Boolean, default true when omitted; disabled profiles create no jobs |
| Profile `parameters` | Adapter destination keys documented below; omitted keys use script defaults. A different profile is a separate model-loading job |
| Profile `n`, `seed` | Override global take count/seed; put them at profile level rather than inside `parameters` |
| Omni profile `reference` | `"auto"` / `True` / `False`; auto follows the global reference choice, true requests cloning (fallback reference is possible), false selects design unless an explicit reference/cache or global reference-only mode requires cloning |
| Omni profile `clone_parameters` | Optional adapter overrides applied only when cloning; use this for clone-specific controls. Default design `instruct` is removed unless `clone_parameters` supplies it |

The notebook owns `config`, `text`, `text_file`, `device`, `n`, `seed`,
`output_dir`, `max_vram_gb`, `estimate_only`, and `self_test`. Do not place them
inside `parameters` or `clone_parameters`; use their supervisor/profile controls
or standalone CLI. Every GPU worker sees only its assigned physical GPU and uses
logical `cuda:0`. Up to two jobs run together; different model names are preferred
when the queue permits it.

### Duration and sound quality: effects shared by the models

- `speed > 1` generally shortens duration; `< 1` lengthens it. This changes native
  duration conditioning/allocation, not the WAV playback rate. Frame rounding,
  minimum durations, and text/reference clamps can prevent exact inverse scaling.
- F5/Omni explicit duration selects a synthesis allocation before waveform
  generation. It is quantized and excludes configured joins; it is not an
  independent prediction of natural unconstrained reading time. Allocations
  that are too short can omit words even when the total sample count is exact.
- Voices, reference preprocessing, emotion/style, seed, text normalization,
  chunking, speed, and joins can affect duration where they enter that model's
  planner. Changing any relevant configuration requires a fresh matching plan.
- Sampling steps usually change quality/runtime rather than the fixed allocated
  output length. StyleTTS2's style diffusion occurs before duration prediction,
  so its sampled style can also change the plan.
- `sample_rate` changes export sampling density, not native acoustic bandwidth;
  peak attenuation/gain and short edge fades change amplitude without changing
  length. Crossfades overlap audio and reduce length; pauses add length. Both
  effects are accounted for by the applicable adapter's plan.
- Native rates: Kokoro/F5/Style/Omni 24 kHz, EmotiVoice 16 kHz, Supertonic 44.1 kHz.
  PCM_16/PCM_24 set integer storage precision; FLOAT preserves floating samples
  and may be less convenient for some players. These are not neural quality knobs.
- ASR omissions/substitutions are fallible screening results, not a perceptual
  quality guarantee. Silence diagnostics distinguish interior quiet spans from
  leading/trailing silence; intended punctuation pauses can still be flagged.

The new catalog section can be inspected and exported **without loading model
weights**. It lists all shipped speaker rows and every parser option. Catalog
viewing does not enable all speakers as experiments: default runs remain two
takes for each enabled profile and each of the three passages.


## Where the duration estimate happens, and why it matches exactly

**The requested passage's waveform is not generated first.** Each adapter
computes a length plan, prints/saves its preview, and then enters waveform
synthesis using that same plan. The notebook later reads the actual WAV header
independently for its comparison. A final report is produced after generation,
but the reported planned sample count was already computed before it.

The experiment is therefore testing a **reused synthesis-length contract**.
It is not testing an independent words-per-minute predictor against an arbitrary
unconstrained TTS call. Exact sample-count equality is a consequence of controlling
the lengths accepted by these decoders. “100%” here means zero measured duration
error in successful exports, not 100% accuracy of pronunciation, word completeness,
natural pacing, or arbitrary future versions. A failed length contract is reported
as an error; the scripts do not quietly rescale speech to make the score look good.

### The six algorithms

| Model | Computation before requested waveform synthesis | Frozen data reused by synthesis | Native chunk length |
|---|---|---|---|
| Kokoro | G2P → text/BERT encodings → voice-conditioned duration network. Sum sigmoid duration logits per phoneme; divide by speed, round, clamp to at least one duration unit | Integer phoneme durations, duration encodings, tokens, selected voice embedding | `600 * sum(durations)` samples at 24 kHz |
| F5-TTS | Reference-duration/text-byte ratio or your explicit seconds → allocate mel frames; include reference/STFT/text-token minimum clamps. This is an allocation heuristic, not a learned natural-duration forecast | Exact total flow frame count and reference-slice count for every chunk | `256 * (allocated_total_frames - floor(reference_samples/256) - 1)` samples at 24 kHz |
| EmotiVoice | G2P + text/speaker/style encodings → learned duration predictor → speed and phoneme-emphasis scaling → integer rounding/minimum lexical-phone guard | Integer durations, conditioned phoneme embeddings, pitch/energy factors | Analytic HiFiGAN convolution/transpose-convolution propagation from `sum(durations)` acoustic frames, at 16 kHz |
| StyleTTS2 | Phonemize/BERT encode → sample latent style separately for every take/chunk → reference/style blend → learned phoneme duration network → speed/rounding/minimum | Sampled acoustic/prosodic styles, integer durations, duration embeddings, seeds | `600 * sum(durations) - tail_trim_samples` samples at 24 kHz; default fixed trim 50 |
| OmniVoice | Pinned Unicode/text rule and reference token/text ratio (or default rate), short-text adjustment, speed and chunk limits; optional explicit seconds select a total token budget | Exact audio-token count for every chunk | `960 * target_tokens` samples at 24 kHz; token rate 25 Hz, i.e. 40 ms per token |
| Supertonic 3 | ONNX duration predictor conditioned on text and voice DP style → divide predicted seconds by speed → use SDK float32 conversion to integer samples and latent-frame allocation | Predicted duration, exact integer sample count, latent-frame count, IDs/masks | `int64(float32_duration * 44100)` samples after removing only precomputed latent-allocation padding |

**Kokoro:** durations are generated by the actual loaded model/voice combination,
not fitted from word count. The decoder's verified temporal geometry emits 600
samples per duration unit. Pitch/noise prediction and waveform decoding follow
the preview. Decoder noise may change audio between seeds without changing the
already frozen duration vector.

**F5:** let `R=floor(reference_samples/256)`, `B` be the reference transcript's
UTF-8 byte length, and `b` be the chunk's byte length. Without explicit seconds,
the requested total frames are `R + int(R*b/(B*speed))`. With explicit seconds,
the requested generated allocation is distributed by chunk byte lengths. The
adapter then applies the pinned flow's text/reference minimum and maximum
clamps. The preview uses this final allocation, not the raw requested seconds.
The flow is called with that exact `duration`; the reference region is excluded
and centered Vocos STFT geometry gives the sample formula in the table. F5 can
preview before loading the generator/vocoder weights, but still needs source
utilities, a reference waveform, vocabulary, and tokenization.

**EmotiVoice:** its duration predictor is conditioned on the learned speaker
identity, prompt/style embedding, and content. Durations are rounded after
speed/emphasis scaling. Spoken phonemes obey `min_phoneme_frames`, while pause
tokens may retain zero. The acoustic length regulator receives those exact
durations rather than recomputing them. Output lengths are propagated using
`Conv1d: floor((L+2p-d*(k-1)-1)/s)+1` and
`ConvTranspose1d: (L-1)*s-2p+d*(k-1)+output_padding+1`; residual blocks are checked
to preserve length. This computes sample count without running the vocoder.

**StyleTTS2:** stochastic style generation occurs **during planning**, before
duration prediction. Consequently each seed/take has its own preview. The
cached style and durations are reused by the later F0/energy/waveform path;
there is no second independent duration draw. This preflight does meaningful
neural computation and is not universally near-instant. A matching saved
`reuse_plan` can skip style/duration inference, subject to the documented cache
constraints.

**Omni:** the rule estimator chooses token lengths using Unicode-weighted text
and a reference-rate heuristic. The iterative acoustic-token generator is given
those exact target lengths; its codec produces 960 samples per token. Explicit
`duration` becomes `max(1, floor(seconds*25))` total tokens, apportioned across
chunks with exact integer allocation. The text heuristic can choose an unsuitable
speaking pace, but the resulting WAV still has the chosen length. This makes
allocation previews fast; it does not make the heuristic a <1% predictor of
unconstrained natural speech duration. Automatic silence removal, padding, and
upstream automatic chunking are disabled in this adapter.

**Supertonic:** only the duration graph handles the passage during planning.
The SDK allocates `ceil(samples/3072)` latent frames, each decoding to 3072
samples. It exports the precomputed sample count by removing the known final
allocation padding. This is an explicit native mechanism, not audio-content
silence detection. During **cold engine initialization**, the adapter does run
two tiny zero-latent vocoder probes to check this geometry, plus a duration-only
warm-up. Those probes are not synthesized versions of your passage and are
included in startup cost rather than warmed preview latency.

### Joins and exported sample rate

After planning all chunks, let `N_native` be their total samples, plus configured
pauses and minus actual configured F5/Style overlaps. The adapter calculates
the rational polyphase resampler length by integer ceiling:

```text
N_export = ceil(N_native * export_rate / native_rate)
planned_seconds = N_export / export_rate
actual_seconds = WAV_header_frames / WAV_header_sample_rate
benchmark_error_percent = 100 * abs(actual_seconds - planned_seconds) / actual_seconds
```

Native punctuation/BOS/EOS silence is already part of the generated chunk's
allocated length. Tiny edge fades, gain, and peak protection do not add or
remove samples. Style's fixed end correction and Super's predetermined latent
padding removal are disclosed and included before export-length calculation.
No successful recording is padded, time-stretched, or content-trimmed afterward
to conceal a mismatch. Fixed decoding corrections and requested crossfades are
different from estimating duration by measuring the completed recording.

### Where to inspect the evidence

The ZIP includes the complete Python adapters under `source/`, and each job's
`config.json`, text, preview/manifest, and `generation.log` under `job_details/`.
The catalog explorer also shows exact file/function line locations for planning
and synthesis. In the notebook, look for the worker's **Duration preflight**,
**Style/duration preflight**, or printed **preflight/Duration preview** message
before its generation progress starts.

| Adapter | Preview saved before requested waveform synthesis |
|---|---|
| Kokoro | Per-run `preview.json`, including integer durations and predicted export samples |
| F5 | Initial `manifest.json` with `preview` and `plan`, written before `load_models` and flow/vocoder synthesis |
| EmotiVoice | Initial `emotivoice_manifest.json` with phoneme durations and predicted samples, before `engine.synthesize` |
| StyleTTS2 | `duration_plan.pt` and initial `manifest.json` with per-take previews, before `generate_chunk` |
| OmniVoice | `estimate.json` with `plan`, before model loading/token generation in the normal design path |
| Supertonic | Initial `supertonic_manifest.json`, before passage text encoding/flow/vocoder generation |

Manifests that also contain output rows are updated after takes complete. Their
final modification times alone do not prove the original preview was computed
late; inspect the call order and generation logs. Kokoro's separate preview and
Omni's estimate file preserve standalone plan artifacts. The notebook's report
cell reads these already calculated quantities and independently measures WAV
headers; it does not derive planned duration from those headers.

Model loading, downloads, reference preparation, duration planning, waveform
synthesis, and independent ASR are distinct stages. A no-reference default may
first generate a **separate synthetic speaker-reference recording** on CPU; that
is not pre-generating the requested benchmark passages. The reported planning
latency is model-specific and excludes several cold-start costs. The 500-word
CPU Kokoro preview observed locally was several seconds, so exactness should
not be confused with a universal near-instant performance guarantee.


## Kokoro: every installed preset and every exposed control

The pinned v1.0 snapshot contains **54 speaker packs**. `f`/`m` in the second prefix position indicates the documented female/male preset family. Speaker identity and G2P language are separate controls. Use matching language prefixes for the intended accent. [Pinned voice assets](https://huggingface.co/hexgrad/Kokoro-82M/tree/f3ff3571791e39611d31c381e3a41a3af07b4987/voices), [voice guide](https://huggingface.co/hexgrad/Kokoro-82M/blob/f3ff3571791e39611d31c381e3a41a3af07b4987/VOICES.md).

| `language` and frontend | All installed voice IDs (female first; male after semicolon) |
|---|---|
| a — US English | af_alloy, af_aoede, af_bella, af_heart, af_jessica, af_kore, af_nicole, af_nova, af_river, af_sarah, af_sky; am_adam, am_echo, am_eric, am_fenrir, am_liam, am_michael, am_onyx, am_puck, am_santa |
| b — British English | bf_alice, bf_emma, bf_isabella, bf_lily; bm_daniel, bm_fable, bm_george, bm_lewis |
| e — Spanish | ef_dora; em_alex, em_santa |
| f — French | ff_siwis |
| h — Hindi | hf_alpha, hf_beta; hm_omega, hm_psi |
| i — Italian | if_sara; im_nicola |
| j — Japanese | jf_alpha, jf_gongitsune, jf_nezumi, jf_tebukuro; jm_kumo |
| p — Brazilian Portuguese | pf_dora; pm_alex, pm_santa |
| z — Mandarin Chinese | zf_xiaobei, zf_xiaoni, zf_xiaoxiao, zf_xiaoyi; zm_yunjian, zm_yunxi, zm_yunxia, zm_yunyang |

`voice` also accepts local `.pt` tensors with shape `(L,1,256)` and finite values. Comma-separated voice IDs or local packs can be mixed with `voice_weights` if shapes match. Example: `{"voice":"am_fenrir,am_michael","voice_weights":"0.7,0.3","language":"a"}`. The voice is interpolated in embedding space; its perceived timbre and quality must be auditioned. There is **no reference-WAV cloning encoder** and no native emotion prompt, SSML, calibrated pitch knob, or per-word emphasis switch in this adapter.

Notebook enabled profile: `am_fenrir`, US English, speed1, initial chunk45 words. Standalone chunk default is35. The disabled `bm_george` profile uses British English. The notebook maps each installed ID in a comma-separated blend to its local pack automatically; explicit comma-separated local paths also work.

English uses Misaki with eSpeak fallback; Spanish/French/Hindi/Italian/Portuguese use eSpeak. Japanese requires `misaki[ja]`, Chinese requires `misaki[zh]`; the current pinned setup installs `misaki[en]`, so add the corresponding optional frontend dependencies before using `j` or `z`. [Pinned frontend](https://github.com/hexgrad/kokoro/blob/dfb907a02bba8152ca444717ca5d78747ccb4bec/kokoro/pipeline.py). The standalone second model family `hexgrad/Kokoro-82M-v1.1-zh` is selectable, but its assets are **not installed by this Kaggle setup** and its separate catalog is not represented by the54 v1.0 packs above. Keep matching checkpoint/config/voice assets together.

| JSON key / CLI flag | Standalone default | Allowed values | Exact effect |
|---|---|---|---|
| `config` / `--config` | None | Path to a JSON object; underscore keys | Reads CLI defaults; explicit CLI flags win. Notebook profiles create these JSON files automatically. |
| `text` / `--text` | None | UTF-8 string; exclusive with text_file | Literal text. All three adapters enforce 10..500 whitespace-delimited words. |
| `text_file` / `--text-file` | None | UTF-8/UTF-8-BOM file; exclusive with text | Loads the complete source text. No repetition or clipping to reach a target word count. |
| `n` / `--n` | 1 | Integer >=1 | Number of output WAVs. The notebook normally makes n=2 per profile/text case. |
| `threads` / `--threads` | 4 | Integer >=1 | CPU inference thread count; does not select a voice or alter desired duration. |
| `device` / `--device` | cpu | cpu, cuda, cuda:INDEX | Selects inference device. Notebook sets cuda:0 inside each GPU-isolated worker; CUDA index is local to CUDA_VISIBLE_DEVICES. |
| `max_vram_gb` / `--max-vram-gb` | 5.0 | Finite 0 < value <=5; decimal GB | Allocator/arena target, not a certified whole-device ceiling. Notebook additionally polls total device usage through NVML. |
| `pause_ms` / `--pause-ms` | 0.0 | Finite >=0 milliseconds | Inserts exactly round(pause_ms*native_rate/1000) silence samples between final chunks. Adds to predicted and actual duration; does not remove model-generated silence. |
| `edge_fade_ms` / `--edge-fade-ms` | 3.0 | Finite 0..20 milliseconds | Cosine attenuation at both ends of each chunk, capped at half that chunk. Zero disables it; sample count and timing are unchanged. |
| `gain_db` / `--gain-db` | 0.0 | Finite -120..120 dB | Multiplies the final resampled waveform by 10**(gain_db/20). Changes loudness only; excessive positive gain can trigger peak attenuation. |
| `normalize_peak` / `--normalize-peak` | True | Boolean | Attenuates the whole recording only if its absolute peak exceeds 0.98. Does not amplify quiet audio, compress dynamics, or alter duration. |
| `estimate_only` / `--estimate-only` | False | Boolean | Loads models and writes the reusable duration preflight/report without synthesizing a WAV. Cold loading is separate from preview latency. |
| `seed` / `--seed` | 42 | Integer >=0; seed+n-1 <2**63 | Consecutive seeds are recorded for n samples. The supported eval decoder has no stochastic sampler, so identical settings need not produce distinct voices/audio. |
| `voice` / `--voice` | am_fenrir | Preset ID, local .pt path, or comma-separated compatible IDs/paths | Selects length-indexed speaker embeddings. Comma-separated packs are averaged with voice_weights; .wav is unsupported. |
| `voice_weights` / `--voice-weights` | None (equal weights) | Comma-separated finite nonnegative numbers, one per voice; sum >0 | Normalized weighted average of the voice packs. Packs must have identical tensor shapes; a mix is a new interpolation, not an exact named speaker. |
| `language` / `--language` | a | a, b, e, f, h, i, j, p, z | Selects grapheme-to-phoneme frontend/accent, independently of the voice ID. Match voice prefix to frontend for the intended language/accent. |
| `speed` / `--speed` | 1.0 | Finite >0 | Divides predicted phoneme durations by speed, then rounds and clamps each to >=1 frame before acoustic decoding. Higher is faster; rounding prevents exact inverse scaling, especially at extremes. |
| `repo_id` / `--repo-id` | hexgrad/Kokoro-82M | hexgrad/Kokoro-82M or hexgrad/Kokoro-82M-v1.1-zh | Selects a pinned config/checkpoint/voice download family. This notebook installs v1.0 only; selecting v1.1-zh needs separate matching assets and setup changes. |
| `checkpoint` / `--checkpoint` | None (pinned download) | Path to compatible PyTorch checkpoint | Replaces the downloaded model weights. Architecture and model_config must match; a fine-tune can change sound and durations. |
| `model_config` / `--model-config` | None (pinned download) | Path to compatible JSON config | Defines model architecture. Adapter rejects unsupported decoder upsampling/ISTFT geometry because the exact 600-sample duration mapping would no longer be established. |
| `cache_dir` / `--cache-dir` | <script directory>/.model_cache | Directory path | Stores downloaded config/checkpoint/preset files. No acoustic control. |
| `cuda_reserve_mb` / `--cuda-reserve-mb` | 768 | Integer >=0; must leave allocator budget | Reserves decimal MB outside the PyTorch allocation target for context/other allocations; does not itself cap those allocations. |
| `chunk_words` / `--chunk-words` | 35 | Integer >=1 | Initial word bound; sentence/clause boundaries preferred. G2P/context or max_chunk_seconds can split further. Changes prosody and duration because each chunk is independently planned. |
| `max_chunk_seconds` / `--max-chunk-seconds` | 20.0 | Finite >0 seconds | Splits any predicted overlong chunk and replans it before synthesis. An indivisible overlong token is rejected instead of truncated. |
| `sample_rate` / `--sample-rate` | 24000 | Integer 8000..192000 Hz | Export rate only; native synthesis remains 24 kHz. Rational resampling predicts the exact ceil(native_samples*export_rate/24000) count for the concatenated recording. |
| `subtype` / `--subtype` | PCM_16 | PCM_16, PCM_24, FLOAT | WAV sample encoding/precision, not sample rate. Does not change samples or duration. |
| `output_dir` / `--output-dir` | samples | Directory path | Writes a unique timestamp/UUID run directory with WAVs, preview and manifest. |
| `self_test` / `--self-test` | False | Boolean | Runs pure integer-length/chunk checks without loading models or validating ordinary generation inputs; diagnostic mode only. |

## EmotiVoice: the complete trained speaker table and controls

The pinned pretrained table has **2,014 unique speaker labels** and ordered indices0..2013. The searchable catalog cell contains **every row**, including upstream name/gender where provided, and exports the whole table. [Exact ordered labels](https://github.com/netease-youdao/EmotiVoice/blob/59f0f36de4db12825f4705dd4e0780d79dd6bb01/data/youdao/text/speaker2), [name/gender metadata](https://github.com/netease-youdao/EmotiVoice/blob/59f0f36de4db12825f4705dd4e0780d79dd6bb01/data/youdao/text/README.md).

Select `speaker` as a **string label**, for example`"9017"`, not an embedding index. Selected known voices:9017 John Van Stan (male; default),6097 Phil Benson (male),6671 Tony Oliva (male),6670 Mike Pelton (male),8051 Maria Kasper (female),11614 Sylviamb (female),9136 Helen Taylor (female),11697 Celine Major (female),92 Cori Samuel (female),12787 LikeManyWaters (female). These are examples; all2,014 are listed in the catalog. Gender/name entries are upstream metadata, not measured pitch/style guarantees.

English and Chinese text, including their mixed frontend, are supported. There is **no arbitrary reference-WAV identity input** in this pretrained adapter. The upstream personal-data cloning workflow is training/fine-tuning rather than instant WAV conditioning; new voices require matching retrained weights and speaker tables. `prompt` accepts arbitrary short conditioning text; upstream demonstrates emotions such as Happy/Excited/Sad/Angry. There is no finite exhaustive prompt dictionary, no guaranteed emotion intensity, and changing emotion can also change duration. [Pinned model documentation](https://github.com/netease-youdao/EmotiVoice/blob/59f0f36de4db12825f4705dd4e0780d79dd6bb01/README.md).

Notebook enabled profile: speaker`"9017"`, prompt`"Neutral"`, speed/pitch-predictor/energy-predictor scales1; the disabled comparison changes the prompt to`"Sad"`. This eval model is deterministic: n repeats with different seeds have identical audio for unchanged inputs.

`emphasis_json` uses **zero-based chunk and phoneme indices**, including boundary tokens. First run `estimate_only=True` to inspect `chunks[].phonemes` in the manifest, then write edits such as`[{"chunk":0,"phoneme":2,"duration_scale":1.2,"pitch_scale":1.1,"energy_scale":1.2}]` to an accessible JSON file and set its path. If chunking/text/speaker/prompt/speed changes, replan and inspect indices again. Each optional scale defaults1; all must be0.25..4. This is direct phoneme control, not SSML or automatic English-word alignment.

| JSON key / CLI flag | Standalone default | Allowed values | Exact effect |
|---|---|---|---|
| `config` / `--config` | None | Path to a JSON object; underscore keys | Reads CLI defaults; explicit CLI flags win. Notebook profiles create these JSON files automatically. |
| `text` / `--text` | None | UTF-8 string; exclusive with text_file | Literal text. All three adapters enforce 10..500 whitespace-delimited words. |
| `text_file` / `--text-file` | None | UTF-8/UTF-8-BOM file; exclusive with text | Loads the complete source text. No repetition or clipping to reach a target word count. |
| `n` / `--n` | 1 | Integer >=1 | Number of output WAVs. The notebook normally makes n=2 per profile/text case. |
| `threads` / `--threads` | 4 | Integer >=1 | CPU inference thread count; does not select a voice or alter desired duration. |
| `device` / `--device` | cpu | cpu, cuda, cuda:INDEX | Selects inference device. Notebook sets cuda:0 inside each GPU-isolated worker; CUDA index is local to CUDA_VISIBLE_DEVICES. |
| `max_vram_gb` / `--max-vram-gb` | 5.0 | Finite 0 < value <=5; decimal GB | Allocator/arena target, not a certified whole-device ceiling. Notebook additionally polls total device usage through NVML. |
| `pause_ms` / `--pause-ms` | 0.0 | Finite >=0 milliseconds | Inserts exactly round(pause_ms*native_rate/1000) silence samples between final chunks. Adds to predicted and actual duration; does not remove model-generated silence. |
| `edge_fade_ms` / `--edge-fade-ms` | 3.0 | Finite 0..20 milliseconds | Cosine attenuation at both ends of each chunk, capped at half that chunk. Zero disables it; sample count and timing are unchanged. |
| `gain_db` / `--gain-db` | 0.0 | Finite -120..120 dB | Multiplies the final resampled waveform by 10**(gain_db/20). Changes loudness only; excessive positive gain can trigger peak attenuation. |
| `normalize_peak` / `--normalize-peak` | True | Boolean | Attenuates the whole recording only if its absolute peak exceeds 0.98. Does not amplify quiet audio, compress dynamics, or alter duration. |
| `estimate_only` / `--estimate-only` | False | Boolean | Loads models and writes the reusable duration preflight/report without synthesizing a WAV. Cold loading is separate from preview latency. |
| `repo` / `--repo` | None (required) | Pinned official source checkout path | Supplies the architecture, bilingual phoneme frontend, vocabulary, ordered speaker table and default config. |
| `generator_checkpoint` / `--generator-checkpoint` | None (required) | Compatible checkpoint path containing generator state | Loads acoustic model/vocoder and speaker embeddings. Notebook uses the authenticated pinned g_00140000 asset. |
| `style_checkpoint` / `--style-checkpoint` | None (required) | Compatible checkpoint path containing model state | Loads SimBERT style/content conditioning. Notebook uses pinned checkpoint_163431. |
| `model_yaml` / `--model-yaml` | None -> repo/config/joint/config.yaml | Compatible YAML path; native sr must equal16000 | Architecture/configuration for the generator, not an independent style preset; matching weights required. |
| `bert_path` / `--bert-path` | WangZeJun/simbert-base-chinese | Hugging Face ID or local SimBERT directory | Tokenizer/text embedding model for content and emotion/style prompts; style encoder executes on CPU even in CUDA mode. |
| `local_files_only` / `--local-files-only` | False | Boolean | Passes offline-only loading to the tokenizer. Notebook supplies the full local SimBERT directory, avoiding inference-time downloads. |
| `speaker` / `--speaker` | None -> label9017 | One exact string label from the 2014-row table; exclusive with speaker_index | Chooses the trained speaker identity by label, not integer embedding position. Default resolves9017, John Van Stan. |
| `speaker_index` / `--speaker-index` | None | Integer 0..2013 for the pinned table; exclusive with speaker | Chooses an embedding row by zero-based table position. Index2 is label9017; index9017 is invalid. Custom checkpoints may have a different table. |
| `prompt` / `--prompt` | Neutral | Free text within SimBERT context limit | Conditions style/emotion, e.g. Happy, Excited, Sad, Angry or Neutral. This is learned conditioning, not a guaranteed exhaustive emotion vocabulary or a calibrated intensity slider. |
| `speed` / `--speed` | 1.0 | Finite0.25..4 | Applies inverse speed to predicted phoneme durations before integer rounding and Gaussian upsampling. Higher is faster; not waveform stretching. |
| `pitch_predictor_scale` / `--pitch-predictor-scale` | 1.0 | Finite0.25..4 | Multiplies neural pitch-predictor values before their embedding. Neural units, not Hz, semitones or guaranteed perceived pitch ratios; duration is unchanged with the same cached plan. |
| `energy_predictor_scale` / `--energy-predictor-scale` | 1.0 | Finite0.25..4 | Multiplies neural energy-predictor values before their embedding. Neural units, not dB; acoustic output can change without changing the plan's frame count. |
| `min_phoneme_frames` / `--min-phoneme-frames` | 1 | Integer0..10 | After speed/emphasis rounding, clamps lexical phones to this many acoustic frames; pauses/EOS are not forced. One protects against dropped lexical phones; zero restores upstream zero-frame behavior. Can lengthen the plan. |
| `emphasis_json` / `--emphasis-json` | None | JSON file with phoneme-indexed edit list | Optional per-chunk/per-phone duration_scale, pitch_scale, energy_scale edits; each finite0.25..4. Edits multiply global factors; repeated edits multiply again. Duration edits are incorporated into the exact plan. |
| `max_chunk_chars` / `--max-chunk-chars` | 240 | Integer>=20 | Hard initial character bound with sentence/clause preferences. An indivisible token longer than the bound is rejected; frontend/token context can reject oversized inputs. |
| `max_chunk_frames` / `--max-chunk-frames` | 2000 | Integer>=1 | Rejects a chunk whose planned acoustic allocation exceeds this limit. Does not silently shorten or automatically split that chunk; lower max_chunk_chars or increase speed. |
| `seed` / `--seed` | 1234 | Integer>=0 | Logs consecutive n-sample seeds; eval EmotiVoice is deterministic, so changing seed or n does not add diversity. |
| `allow_unverified_cuda_budget` / `--allow-unverified-cuda-budget` | False | Boolean; must beTrue for CUDA | Acknowledges that the conservative PyTorch allocator budget excludes driver/context/external memory. Notebook sets thisTrue and monitors device usage. |
| `sample_rate` / `--sample-rate` | 0 ->16000 | 0 or integer8000..192000 Hz | Resamples the concatenated native16 kHz recording at export. Higher rates do not create new native bandwidth; count uses the exact rational-resampler ceiling. |
| `wav_subtype` / `--wav-subtype` | PCM_16 | PCM_16, PCM_24, FLOAT | Export precision/encoding; does not affect the frame plan or WAV duration. |
| `output_dir` / `--output-dir` | emotivoice_samples | Fresh directory path | Writes WAVs and emotivoice_manifest.json; refuses an existing manifest to avoid overwriting a prior test. |

## Supertonic3: every preset, custom styles and controls

The pinned open snapshot contains **10 preset style JSONs**. [Exact preset assets](https://huggingface.co/supertone-oss-archive/supertonic-3/tree/aafc6e32416a594460b32413efc49d7fe4ce6d46/voice_styles). Descriptions below summarize the developer's audition guide and are subjective, not physical pitch specifications. [Developer voice descriptions](https://github.com/supertone-oss-archive/supertonic-py/blob/main/docs/voices.md).

| `voice` | Documented gender | Audition guide |
|---|---|---|
| F1 | F | Composed, slightly low female |
| F2 | F | Bright, youthful female |
| F3 | F | Clear female announcer |
| F4 | F | Confident, expressive female |
| F5 | F | Gentle, soothing female |
| M1 | M | Energetic male narration |
| M2 | M | Deep, composed, serious male |
| M3 | M | Authoritative male presenter |
| M4 | M | Gentle, youthful male |
| M5 | M | Warm male storyteller |

The same ten identities support all31 released language codes:`en` English,`ko` Korean,`ja` Japanese,`ar` Arabic,`bg` Bulgarian,`cs` Czech,`da` Danish,`de` German,`el` Greek,`es` Spanish,`et` Estonian,`fi` Finnish,`fr` French,`hi` Hindi,`hr` Croatian,`hu` Hungarian,`id` Indonesian,`it` Italian,`lt` Lithuanian,`lv` Latvian,`nl` Dutch,`pl` Polish,`pt` Portuguese,`ro` Romanian,`ru` Russian,`sk` Slovak,`sl` Slovenian,`sv` Swedish,`tr` Turkish,`uk` Ukrainian,`vi` Vietnamese. The pinned frontend additionally accepts`na` as a sentinel; it is not a32nd documented language. [Model card](https://huggingface.co/supertone-oss-archive/supertonic-3/blob/aafc6e32416a594460b32413efc49d7fe4ce6d46/README.md), [pinned frontend](https://github.com/supertone-oss-archive/supertonic/blob/1e9799e964ea4c0dad7cde993b65c3c813a7b373/py/helper.py). No Farsi or Mandarin preset language is listed.

`voice_style` accepts an already compatible **Supertonic3** JSON, overriding`voice`. Presets contain float32 `style_ttl` tensors of shape`(1,50,256)` and `style_dp` tensors of shape`(1,8,16)`; ttl conditions acoustics and dp conditions duration. The open package contains **no local reference-WAV encoder**. The model card directs custom voice creation to [Voice Builder](https://supertonic.supertone.ai/voice-builder), where purchased styles can be downloaded for Supertonic2/3. Supplying a JSON is local inference; building that JSON through this hosted service is a separate workflow. A schema-valid custom/mixed JSON is not a guarantee of a natural or recognizable voice; this adapter provides no automatic mixing knob. [Pinned model card](https://huggingface.co/supertone-oss-archive/supertonic-3/blob/aafc6e32416a594460b32413efc49d7fe4ce6d46/README.md).

Notebook enabled profile: M2, English, speed1,16 steps, initial360 characters. Standalone is300 characters, or120 automatically for Korean/Japanese when the configured value is300. M2 is the developer's deep/serious preset; there is no native emotion prompt, calibrated pitch slider, per-word emphasis or SSML control. Latent seeds create alternative realizations with the same preplanned length.

| JSON key / CLI flag | Standalone default | Allowed values | Exact effect |
|---|---|---|---|
| `config` / `--config` | None | Path to a JSON object; underscore keys | Reads CLI defaults; explicit CLI flags win. Notebook profiles create these JSON files automatically. |
| `text` / `--text` | None | UTF-8 string; exclusive with text_file | Literal text. All three adapters enforce 10..500 whitespace-delimited words. |
| `text_file` / `--text-file` | None | UTF-8/UTF-8-BOM file; exclusive with text | Loads the complete source text. No repetition or clipping to reach a target word count. |
| `n` / `--n` | 1 | Integer >=1 | Number of output WAVs. The notebook normally makes n=2 per profile/text case. |
| `threads` / `--threads` | 4 | Integer >=1 | CPU inference thread count; does not select a voice or alter desired duration. |
| `device` / `--device` | cpu | cpu, cuda, cuda:INDEX | Selects inference device. Notebook sets cuda:0 inside each GPU-isolated worker; CUDA index is local to CUDA_VISIBLE_DEVICES. |
| `max_vram_gb` / `--max-vram-gb` | 5.0 | Finite 0 < value <=5; decimal GB | Allocator/arena target, not a certified whole-device ceiling. Notebook additionally polls total device usage through NVML. |
| `pause_ms` / `--pause-ms` | 0.0 | Finite >=0 milliseconds | Inserts exactly round(pause_ms*native_rate/1000) silence samples between final chunks. Adds to predicted and actual duration; does not remove model-generated silence. |
| `edge_fade_ms` / `--edge-fade-ms` | 3.0 | Finite 0..20 milliseconds | Cosine attenuation at both ends of each chunk, capped at half that chunk. Zero disables it; sample count and timing are unchanged. |
| `gain_db` / `--gain-db` | 0.0 | Finite -120..120 dB | Multiplies the final resampled waveform by 10**(gain_db/20). Changes loudness only; excessive positive gain can trigger peak attenuation. |
| `normalize_peak` / `--normalize-peak` | True | Boolean | Attenuates the whole recording only if its absolute peak exceeds 0.98. Does not amplify quiet audio, compress dynamics, or alter duration. |
| `estimate_only` / `--estimate-only` | False | Boolean | Loads models and writes the reusable duration preflight/report without synthesizing a WAV. Cold loading is separate from preview latency. |
| `repo` / `--repo` | None (required) | Pinned official source checkout path | Imports the official Unicode language frontend and compatible voice-style loader. |
| `assets_dir` / `--assets-dir` | None (required) | Directory containing onnx/ and voice_styles/ | Contains the four pinned ONNX graphs, configuration/Unicode index and voice style JSONs. Model-file hashes are verified before use. |
| `voice` / `--voice` | M2 | M1..M5 orF1..F5 in the pinned snapshot | Loads assets_dir/voice_styles/<voice>.json; this name is a preset, not a reference WAV or an emotion instruction. |
| `voice_style` / `--voice-style` | None | Compatible Supertonic3 JSON path | Overrides voice. Uses style_dp for duration and style_ttl for acoustic conditioning. No local WAV-to-style encoder is provided; incompatible Supertonic2-only styles are unsupported. |
| `lang` / `--lang` | en | 31 language codes plus frontend sentinelna | Adds the chosen language tags around normalized text. All ten presets can be used for all released languages; pronunciation/accent quality can vary. |
| `speed` / `--speed` | 1.0 | Finite0.25..4 | Divides the voice-conditioned duration predictor's seconds before allocating the latent and integer sample count. Higher is faster, with no post-generation time stretching. |
| `steps` / `--steps` | 16 | Integer>=1 | Number of iterative flow/vector-estimator steps. More steps increase work and may improve quality, without changing the predetermined duration. Not an emotion or timing control. |
| `max_chunk_chars` / `--max-chunk-chars` | 300; automatically120 forko/ja if value is300 | Integer>=20 | Hard sequential chunk bound with sentence/clause preferences. Explicit notebook profile360 stays360 even forko/ja, so use120 for those languages if desired. |
| `seed` / `--seed` | 1234 | Integer>=0 | Seeds NumPy latent-noise sampling; each n output uses seed+index. Changes realization/prosody while the cached duration plan stays fixed. |
| `allow_unverified_cuda_budget` / `--allow-unverified-cuda-budget` | False | Boolean; must beTrue for CUDA | Explicitly accepts that four ONNX CUDA arena caps do not certify whole-process/device VRAM. Notebook setsTrue plus external NVML monitoring. |
| `cuda_reserve_mb` / `--cuda-reserve-mb` | 768 | Integer>=0; reserve must be less than CUDA budget | Decimal MB removed before dividing the remaining target among four CUDA arenas. CUDA context/driver/workspaces can use memory outside those arenas. |
| `sample_rate` / `--sample-rate` | 0 ->44100 | 0 or integer8000..192000 Hz | Export resampling of native44.1 kHz synthesis; exact ceiling of rational resampling is included in the plan. No improvement in native bandwidth from upsampling. |
| `wav_subtype` / `--wav-subtype` | PCM_16 | PCM_16, PCM_24, FLOAT | WAV export precision/encoding only; duration remains the same. |
| `output_dir` / `--output-dir` | supertonic_samples | Fresh directory path | Writes WAVs and supertonic_manifest.json; refuses an existing manifest to protect prior tests. |


## F5-TTS v1 Base: voices and complete adapter configuration

**Voice choices are open-ended:** supply speech from the speaker you want using `reference_audio` and its exact `reference_text`. There is no finite list of F5 speaker IDs and no F5-native serious-male preset. The notebook's male fallback is a disclosed Kokoro `am_fenrir` reference. It is a convenience prompt, and copying a synthetic prompt does not establish the same quality as a clean human reference.

The adapter accepts a finite, nonsilent **1–8 second** reference. The notebook's shared reference validator additionally requires **mono** audio; the standalone script averages channels and resamples to 24 kHz. Preserve complete words and a natural ending; the adapter does not trim silence or automatically transcribe the prompt. The checkpoint covers **English and Mandarin Chinese**. Community language checkpoints are separate models, not additional built-in voices, and compatibility with this fixed v1 Base architecture is not automatic. [Official language/checkpoint list](https://github.com/SWivid/F5-TTS/blob/283252563dbf91be625e0c27926acfaac449186c/src/f5_tts/infer/SHARED.md)

Optional upstream example recordings are `basic_ref_en.wav`, `basic_ref_zh.wav`, and the multi-voice examples `main.flac`, `town.flac`, `country.flac` under `src/f5_tts/infer/examples/`. They are WAV/FLAC prompts, not a speaker-embedding bank. Inspect their duration and obtain the exact transcript before using them in this adapter. The English basic example's supplied transcript is “Some call me nature, others call me mother nature.” [Official examples and reference guidance](https://github.com/SWivid/F5-TTS/blob/283252563dbf91be625e0c27926acfaac449186c/src/f5_tts/infer/README.md)

The following table covers **every CLI/config destination**, including operational/resource fields; `config` itself is a CLI-only file selector and cannot occur inside that JSON. Use underscore keys inside `MODEL_PROFILES['f5'][...]['parameters']`; standalone CLI flags use hyphens. The notebook owns `config`, `text`, `text_file`, `device`, `n`, `seed`, `output_dir`, `max_vram_gb`, and `estimate_only`; profile-level `n` and `seed` are separate supported fields. `repo`, `checkpoint`, `vocab`, and `vocos_dir` are installed automatically but can be replaced by compatible local resource paths. All floating-point settings must be finite. The displayed numeric bounds are validation bounds, not a claim that extreme settings produce intelligible speech.

| Config key | Standalone default | Accepted values | Exact effect |
|---|---|---|---|
| `config` | `unset` | Optional JSON file; standalone CLI only | Loads argparse destination keys; explicit CLI arguments take precedence. Notebook writes this file for you. Unknown keys and incorrect JSON types fail validation. |
| `text` | `unset` | Exactly one of text/text_file; 10–500 whitespace-separated words | Literal text to speak; use TEXT_CASES in the notebook. Text affects pronunciation, native punctuation pauses, chunking, and the plan. |
| `text_file` | `unset` | UTF-8/UTF-8-BOM file; mutually exclusive with text | Reads the complete passage from a file. The supervisor creates this from TEXT_CASES. |
| `repo` | `unset` | Required path to the exact pinned source checkout | Inference implementation. The notebook installs and supplies it; a different revision or modified checked model source is rejected. |
| `n` | `1` | Integer ≥1 | Number of takes; set global CONFIG['n'] or profile-level n. These models generate takes/chunks sequentially within their worker. |
| `seed` | `42` | Integer; notebook accepts 0 through 2**63−n | Base random seed; take i uses seed+i and each chunk has a reproducible derived seed. Reproducibility is conditional on the same software/hardware/settings. |
| `device` | `'cpu'` | cpu or available cuda:N | Standalone compute device. The notebook selects a physical GPU, exposes it to the worker, and supplies cuda:0; do not put device in a profile. |
| `output_dir` | `'f5_samples'` | Fresh output directory | Writes WAVs, manifest, and previews/plans. Existing run artifacts are rejected. The notebook owns naming and directories. |
| `sample_rate` | `24000` | Integer 8000–192000 Hz | Export rate after deterministic polyphase resampling from 24 kHz. Increasing it does not create new acoustic detail. Final duration uses the exact exported sample count, including resampling rounding. |
| `pause_ms` | `0` | Finite ≥0; cannot be positive with crossfade_ms | Inserts this many milliseconds of zero-valued samples between generated chunks. It adds to exported duration; it does not change pauses already generated inside a chunk. |
| `crossfade_ms` | `0` | Finite ≥0; cannot be positive with pause_ms | Linearly overlaps neighboring chunks, subtracting the actual overlap from exported length. Overlap is capped by available samples; large overlaps can superimpose words. |
| `fade_ms` | `5` | Finite 0–20 ms | Linear amplitude fade at both ends of every native chunk, capped at half its length. It changes no sample counts. A large fade can weaken an initial/final phoneme; zero disables it. |
| `max_vram_gb` | `5` | Finite 0<value≤5 decimal GB; CUDA also requires >0.75 GB | GPU allocator budgeting with a 0.75 GB reserve, plus observed peak checks. CUDA library allocations exist outside the allocator; sampled monitoring is not an instantaneous hard-peak certificate. Notebook also supervises NVML externally. |
| `estimate_only` | `False` | Boolean; --estimate-only / --no-estimate-only | Writes the exact preview (and StyleTTS2 tensor plan) without waveform generation. The regular notebook benchmark generates audio, so the supervisor controls this key. |
| `wav_subtype` | `'PCM_24'` | PCM_16, PCM_24, FLOAT | WAV sample encoding: signed 16-/24-bit PCM or 32-bit float. This changes precision/file size, not rate, sample count, or generation style. |
| `peak_limit` | `0.98` | Finite 0<value≤1 | If the final resampled waveform exceeds this absolute peak, applies one uniform gain ≤1. It neither boosts quiet audio nor changes duration; the gain and original peak are logged. |
| `reference_audio` | `unset` | Required finite nonsilent 1–8 s audio; shared notebook reference must be mono | Speaker/style prompt. Standalone reading averages multiple channels and resamples to 24 kHz; no automatic silence trimming. Clean speech from one speaker gives a more useful prompt than music, overlapping speakers, clipping, or a cut word. |
| `reference_text` | `unset` | Required exact nonempty transcript | Conditions the spoken reference together with the waveform. Adapter appends a terminal period if needed and a separating space. Its UTF-8 byte length, with the reference duration, also defines the allocation speaking-rate estimate. |
| `checkpoint` | `unset` | Optional local official-layout F5TTS_v1_Base EMA .safetensors | Replaces the pinned generator file. Architecture is fixed in this adapter and load is strict; Small, E2TTS, non-EMA layouts, and arbitrary unrelated checkpoints are not interchangeable. |
| `vocab` | `unset` | Optional local matching vocabulary file | Text token-to-ID mapping; must match the generator. Otherwise uses the pinned F5TTS_v1_Base vocab. A different vocabulary is not a language-quality guarantee. |
| `vocos_dir` | `unset` | Optional directory containing compatible config.yaml and pytorch_model.bin | Offline Vocos resources. The validated decoder uses vocos==0.1.0 and centered 24 kHz/256-hop output; incompatible temporal topology is rejected. |
| `speed` | `1` | Finite >0 | Divides the reference-derived generated-frame allocation: >1 requests shorter/faster speech, <1 longer/slower. It changes planning, not post-generation playback. Token/reference clamps and chunk boundaries prevent a universal 1/speed duration ratio. |
| `duration_seconds` | `unset` | Optional finite >0 seconds | Requests a TOTAL generated-content allocation divided among chunks by UTF-8 byte length, excluding reference audio and joins. Frame quantization and token/reference minimums may increase/change it: the reported preview, not this raw target, is the final duration. With this set, speed still affects chunk splitting but no longer determines each chunk's duration allocation. |
| `steps` | `32` | Integer ≥2; notebook quality profile 64 | Flow-integration intervals. More intervals cost generation time and may improve quality; they do not alter the frozen allocated frame count. Improvement is not guaranteed to be monotonic. |
| `cfg_strength` | `2` | Finite ≥0 | Classifier-free guidance strength. The pinned flow combines conditional/unconditional predictions as conditional + strength*(conditional−unconditional); near zero uses only the conditional prediction. Stronger guidance can change adherence and sound quality, but leaves the allocation fixed. |
| `sway` | `-1` | Finite −1 through 1 | Warps integration times after EPSS/uniform schedule: t'=t+sway*(cos(pi*t/2)−1+t). Negative values concentrate evaluation toward early flow time; zero removes this warp. Changes generation behavior, not output length. |
| `ode_method` | `'euler'` | euler, midpoint, rk4 | ODE solver: approximately 1, 2, or 4 transformer evaluations per interval respectively. Higher-order solvers are more expensive; their quality is an empirical choice. The progress bar counts evaluations; length stays fixed. |
| `no_epss` | `False` | Boolean; CLI flag --no-epss sets true | True selects uniformly spaced times BEFORE the optional sway warp. False enables pinned Empirically Pruned Step Sampling at 5/6/7/10/12/16 steps; other step counts already fall back to uniform times. At the notebook's 64 steps, toggling this does not change the pre-sway schedule. |
| `target_rms` | `0.1` | Finite >0 | If the reference RMS is below this value, boosts conditioning to this RMS, then reverses that gain on generated audio. It is reference-level handling, not final loudness mastering or an emotion/intensity control. Final peak protection still applies. |
| `max_chunk_seconds` | `6` | Finite 0.2–12 s | Bounds generated allocation per chunk; also determines the reference-derived byte budget. A clamped allocation beyond this limit (with a one-hop allowance) fails rather than truncating speech. Smaller chunks can reduce VRAM but create more joins and reset context. |
| `max_chunk_bytes` | `160` | Integer ≥8 UTF-8 bytes | Hard text byte budget per chunk, combined with the time-derived budget. Sentence/clause boundaries are preferred; tiny final tails are merged/rebalanced when possible. A single over-budget word fails. UTF-8 bytes are not words, especially for Chinese. |

**Notebook defaults versus script defaults:** the enabled quality profile raises `steps` from **32 to 64**; global defaults use **2 takes**, seed **1234**, **48 kHz export**, **0 added pause**, and assigned CUDA workers. Other F5 settings inherit the script defaults shown above unless a profile overrides them.

**What the exact duration means:** the planner freezes a frame allocation derived from reference speaking rate or `duration_seconds`, then applies the model's required text/reference minimum. Native chunk length is `(allocated_frames − floor(reference_samples/256) − 1) * 256`. The preview accounts for joins and export-rate rounding. Every take shares that allocation; noise, solver, steps, guidance, and sway can change the sound without changing its length. This is an exact allocation contract, not an independent forecast of the voice's unconstrained natural speaking time. A target that is too short may make pronunciation poor even when the sample count is exact. [Pinned flow implementation](https://github.com/SWivid/F5-TTS/blob/283252563dbf91be625e0c27926acfaac449186c/src/f5_tts/model/cfm.py)

**Upstream features outside this adapter:** automatic reference ASR, the upstream multi-voice `[speaker]` text markup, speech editing, no-reference experiments, alternate E2/Small architectures, BigVGAN, and content-based silence removal are not parameters here. Multiple reference voices can be tested through separate notebook profiles, one prompt per profile. There is no independent pitch, emotion-label, temperature/top-p, or exact word-emphasis control. Delivery can be influenced by the reference and punctuation, with empirical quality limits. Source is MIT; the official pretrained F5 weights are CC-BY-NC-4.0. [Pinned model card](https://huggingface.co/SWivid/F5-TTS/blob/84e5a410d9cead4de2f847e7c9369a6440bdfaca/README.md)

## StyleTTS2 LibriTTS: voices and complete adapter configuration

**Voice choices are open-ended:** give `reference_audio` from the desired speaker; **no transcript is required**. There is no speaker-ID selector in this adapter. The notebook's fallback is the same disclosed Kokoro `am_fenrir` recording, not a StyleTTS2 male preset. Standalone input accepts finite nonsilent **0.5–15 second** references and averages channels; the shared notebook path is deliberately stricter at **mono, 1–8 seconds**. References are resampled to 24 kHz and not silence-trimmed.

The installed model is the **English multi-speaker LibriTTS** checkpoint. `language='en-us'/'en-gb'` switches English pronunciation rules, not model weights. Other StyleTTS2 checkpoints, including single-speaker LJSpeech, are not interchangeable with this adapter's fixed inference topology. [Official model/inference documentation](https://github.com/yl4579/StyleTTS2/blob/5cedc71c333f8d8b8551ca59378bdcc7af4c9529/README.md)

The official demonstration references are listed below. These are optional **recording filenames**, not the complete universe of clonable voices or guaranteed serious/deep male choices. The emotional filenames refer to recordings with that delivery; they do not introduce an emotion-label API. Download the [pinned reference bundle](https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/3aa7ba7f8f275ec13dce21682a61494c35089e2a/reference_audio.zip) manually if wanted, inspect each clip against the input constraints, and point a profile to its extracted audio file. The benchmark does not automatically test all examples. [Pinned official demo](https://github.com/yl4579/StyleTTS2/blob/5cedc71c333f8d8b8551ca59378bdcc7af4c9529/Demo/Inference_LibriTTS.ipynb)

| Demonstration group | Reference recording filenames |
|---|---|
| Seen speakers | `696_92939_000016_000006.wav`, `1789_142896_000022_000005.wav` |
| Unseen speakers | `1221-135767-0014.wav`, `5639-40744-0020.wav`, `908-157963-0027.wav`, `4077-13754-0000.wav` |
| Zero-shot examples | `3.wav`, `4.wav`, `5.wav` |
| Delivery examples | `anger.wav`, `sleepy.wav`, `amused.wav`, `disgusted.wav` |
| Named examples | `Yinghao.wav`, `Gavin.wav`, `Vinay.wav`, `Nima.wav` |

The table below covers **every CLI/config destination**; `config` is CLI-only. Config keys use underscores; standalone flags use hyphens. Supervisor-controlled keys and per-profile take/seed fields work as described for F5 above. All floating-point settings must be finite; acceptance is not an audio-quality guarantee.

| Config key | Standalone default | Accepted values | Exact effect |
|---|---|---|---|
| `config` | `unset` | Optional JSON file; standalone CLI only | Loads argparse destination keys; explicit CLI arguments take precedence. Notebook writes this file for you. Unknown keys and incorrect JSON types fail validation. |
| `text` | `unset` | Exactly one of text/text_file; 10–500 whitespace-separated words | Literal text to speak; use TEXT_CASES in the notebook. Text affects pronunciation, native punctuation pauses, chunking, and the plan. |
| `text_file` | `unset` | UTF-8/UTF-8-BOM file; mutually exclusive with text | Reads the complete passage from a file. The supervisor creates this from TEXT_CASES. |
| `repo` | `unset` | Required path to the exact pinned source checkout | Inference implementation. The notebook installs and supplies it; a different revision or modified checked model source is rejected. |
| `n` | `1` | Integer ≥1 | Number of takes; set global CONFIG['n'] or profile-level n. These models generate takes/chunks sequentially within their worker. |
| `seed` | `42` | Integer; notebook accepts 0 through 2**63−n | Base random seed; take i uses seed+i and each chunk has a reproducible derived seed. Reproducibility is conditional on the same software/hardware/settings. |
| `device` | `'cpu'` | cpu or available cuda:N | Standalone compute device. The notebook selects a physical GPU, exposes it to the worker, and supplies cuda:0; do not put device in a profile. |
| `output_dir` | `'styletts2_samples'` | Fresh output directory | Writes WAVs, manifest, and previews/plans. Existing run artifacts are rejected. The notebook owns naming and directories. |
| `sample_rate` | `24000` | Integer 8000–192000 Hz | Export rate after deterministic polyphase resampling from 24 kHz. Increasing it does not create new acoustic detail. Final duration uses the exact exported sample count, including resampling rounding. |
| `pause_ms` | `0` | Finite ≥0; cannot be positive with crossfade_ms | Inserts this many milliseconds of zero-valued samples between generated chunks. It adds to exported duration; it does not change pauses already generated inside a chunk. |
| `crossfade_ms` | `0` | Finite ≥0; cannot be positive with pause_ms | Linearly overlaps neighboring chunks, subtracting the actual overlap from exported length. Overlap is capped by available samples; large overlaps can superimpose words. |
| `fade_ms` | `5` | Finite 0–20 ms | Linear amplitude fade at both ends of every native chunk, capped at half its length. It changes no sample counts. A large fade can weaken an initial/final phoneme; zero disables it. |
| `max_vram_gb` | `5` | Finite 0<value≤5 decimal GB; CUDA also requires >0.75 GB | GPU allocator budgeting with a 0.75 GB reserve, plus observed peak checks. CUDA library allocations exist outside the allocator; sampled monitoring is not an instantaneous hard-peak certificate. Notebook also supervises NVML externally. |
| `estimate_only` | `False` | Boolean; --estimate-only / --no-estimate-only | Writes the exact preview (and StyleTTS2 tensor plan) without waveform generation. The regular notebook benchmark generates audio, so the supervisor controls this key. |
| `wav_subtype` | `'PCM_24'` | PCM_16, PCM_24, FLOAT | WAV sample encoding: signed 16-/24-bit PCM or 32-bit float. This changes precision/file size, not rate, sample count, or generation style. |
| `peak_limit` | `0.98` | Finite 0<value≤1 | If the final resampled waveform exceeds this absolute peak, applies one uniform gain ≤1. It neither boosts quiet audio nor changes duration; the gain and original peak are logged. |
| `reference_audio` | `unset` | Required finite nonsilent 0.5–15 s audio; shared notebook reference: mono 1–8 s | Speaker/style prompt; no transcript needed. Standalone reading averages channels and resamples to 24 kHz without silence trimming. Acoustic and prosodic encoders extract a 256-dimensional reference style. |
| `checkpoint` | `unset` | Optional compatible official-layout LibriTTS .pth checkpoint | Replaces the generator weights. Inference modules load strictly; single-speaker LJSpeech and unrelated architectures are not drop-in alternatives for this adapter. |
| `model_config` | `unset` | Optional matching official LibriTTS config.yml | Architecture configuration paired with the checkpoint. It is not a per-generation style configuration. The adapter requires the validated HifiGAN temporal topology. |
| `reuse_plan` | `unset` | Optional duration_plan.pt from an identical run setup | Reuses cached sampled styles, integer durations, and duration-conditioned embeddings. Fingerprint includes text, reference/checkpoint/config contents, seed/takes, style, pitch, chunk, join, and export settings. Use a fresh output_dir; changed settings fail validation. |
| `speed` | `1` | Finite >0 | Divides each predicted phoneme duration BEFORE rounding, then clamps every duration to at least one unit. >1 usually shortens/faster; <1 lengthens/slower. Rounding/minimums mean the total is not exactly proportional to 1/speed. |
| `alpha` | `0.1` | Finite 0–1 | Acoustic/timbre blend: alpha*sampled_acoustic + (1−alpha)*reference_acoustic. Zero maximizes direct reference timbre; one uses sampled acoustic style. Higher alpha can reduce speaker similarity; it is not an emotion label. |
| `beta` | `0.3` | Finite 0–1 | Prosody blend: beta*sampled_prosody + (1−beta)*reference_prosody. Zero favors the reference delivery; one favors text-conditioned sampled prosody. It can change phoneme durations, expressivity, and similarity, so exact preview must be recomputed. |
| `diffusion_steps` | `10` | Integer ≥2 | ADPM2 latent-style sampling steps. Affects preflight cost, sampled style, and therefore potentially the duration plan. This is not waveform diffusion; the cached style is reused during decoding. |
| `embedding_scale` | `1` | Finite >0 | Text-embedding classifier-free guidance in style sampling: fixed_embedding_prediction + scale*(text_prediction−fixed_embedding_prediction). At one, uses the normal text-conditioned path; larger values emphasize its difference and add conditional/unconditional computation. Can affect style and planned duration. |
| `style_continuity` | `0.7` | Finite 0–1 | On chunks after the first, blends previous FINAL blended style with the new sampled style: continuity*previous + (1−continuity)*new. Zero removes that carryover; one carries the previous style into this intermediate blend. Alpha/beta reference blending is still applied afterward, so one does not freeze every final chunk style. |
| `pitch_scale` | `1` | Finite >0 | Multiplies predicted F0 immediately before HifiGAN decoding, without shifting durations. >1 usually higher pitch; <1 lower. This is a model-F0 multiplier, not pitch-preserving playback speed; extreme values can hurt naturalness and speaker identity. |
| `max_chunk_words` | `35` | Integer 1–80; notebook quality profile 45 | Initial word budget, with preferred sentence/clause endings. Smaller chunks lower context/VRAM requirements but add joins and may weaken sentence prosody. |
| `max_chunk_tokens` | `400` | Integer 16–500 phoneme tokens | Further recursively splits chunks that exceed the phonemized token budget, preferring sentence/clause boundaries near the split. Includes the initial special token. One over-budget word fails. |
| `max_chunk_seconds` | `20` | Finite 1–30 s | When building a NEW plan, rejects a chunk whose predicted native audio exceeds this limit before decoding; it does not automatically resplit. Use smaller word/token chunks or an appropriate faster speed. Existing reuse_plan data bypasses this check, and this setting is currently absent from the cache fingerprint: retain the original limit when reusing a plan. |
| `tail_trim_samples` | `50` | Integer ≥0 native 24 kHz samples | Fixed trim at every chunk's decoder tail, matching the official demo's end-pulse workaround. Default is about 2.083 ms per chunk. Zero retains full decoder output. It is included in the preview and is not content-dependent silence removal; a large trim can remove real speech or make a plan invalid. |
| `language` | `'en-us'` | en-us, en-gb | Selects the eSpeak English phonemizer's pronunciation variety. The same English LibriTTS checkpoint is used; this does not turn it into a multilingual model or guarantee a British voice identity. |

**Notebook defaults versus script defaults:** the enabled quality profile uses `max_chunk_words=45` rather than the script's **35**. Both use `alpha=0.1`, `beta=0.3`, `diffusion_steps=10`, `embedding_scale=1`, and `pitch_scale=1`. Global notebook settings supply 2 takes, seed 1234, 48 kHz export, zero added gap, and CUDA workers. An optional expressive profile is disabled until you enable it.

**Duration and style interaction:** exact preflight actually runs BERT, latent style diffusion, and the duration predictor for **every requested take**. It caches sampled styles and integer phoneme durations for reuse; it is heavier than F5's arithmetic allocation and is not universally near-instant. Each native chunk has `sum(integer_durations)*600 − tail_trim_samples` samples; joins/resampling are counted afterward. Changing seed, style weights, guidance, continuity, steps, reference, speed, or pronunciation can change the plan. `pitch_scale` changes F0 during decoding without changing that duration contract, although the saved plan fingerprint conservatively includes it.

For a serious/deeper rendition, start with a clean reference that already has that timbre and delivery. `alpha=0` maximizes direct reference acoustic conditioning; low `beta` favors its prosody. Moderate `pitch_scale<1` can lower predicted F0 but cannot guarantee a convincing deeper speaker. Higher `beta` gives more sampled text-conditioned delivery; it does not select a named emotion. The text-embedding guidance formula and its compute cost are in the [pinned diffusion module](https://github.com/yl4579/StyleTTS2/blob/5cedc71c333f8d8b8551ca59378bdcc7af4c9529/Modules/diffusion/modules.py).

**Upstream features outside this adapter:** direct reference-text style-transfer methods, arbitrary emotion labels, per-word SSML emphasis, other languages/checkpoints, sample-wise speaker changes, and custom diffusion samplers/sigma schedules are not exposed. This adapter fixes ADPM2 with Karras sigma min 0.0001, max 3, rho 9, and uses the official LibriTTS HifiGAN conditioning shift. It caches duration decisions rather than modifying generated speech to fit a target. There is no `duration_seconds` override for StyleTTS2.


## OmniVoice: arbitrary reference voices, voice design, and every adapter control

This section describes the **pinned model and the duration-preserving Python adapter in this notebook**, not every option in other releases. There is **no finite speaker-ID catalog**: a clean recording of a single speaker provides a new reference voice. There is also a voice-design mode with the complete attribute vocabulary below. Similarity, pitch, and style adherence remain empirical; the model cannot guarantee an exact identity or a serious mood.

### Choose a voice

| Voice path | What you provide | What happens |
|---|---|---|
| Clone a recorded speaker | `ref_audio` and its exact `ref_text` | The codec turns the recording into a voice prompt. Use clean single-speaker speech; background music, overlapping voices, and an incorrect transcript can compromise pronunciation and similarity. |
| Reuse a clone | Existing `prompt_cache` `.pt` | Loads saved reference codec tokens, transcript, and RMS; original WAV and transcript arguments must be omitted. This avoids repeatedly preparing and encoding the reference. |
| Save a new clone | New `prompt_cache` path plus `ref_audio`/`ref_text` | The adapter prepares the reference during preflight; synthesis loads the model, encodes it, verifies the planned token count, and saves the cache. An estimate-only run does not encode/save a new prompt. |
| Design a voice | Comma-separated `instruct` attributes | Creates a voice matching supported attributes. Default: `male, middle-aged, low pitch, british accent`. Design has no fixed identity/preset ID. |
| Let the model choose | Explicit `instruct=""`, no reference | Resolves to upstream auto-voice mode. Omitting `instruct` or using `None` instead selects this adapter's male default. |

Upstream recommends **3–10 seconds** of reference audio; the shared notebook reference validator accepts **mono, finite, non-silent 1–8-second WAVs**. Choose **3–8 seconds** for both paths. The standalone adapter downmixes stereo and resamples to 24 kHz, then caps the prepared/cached reference token duration using `max_reference_seconds` (default 10). It rejects an oversized reference rather than truncating speech to fit. At least one 40 ms codec token is required. A same-language reference tends to give more standard pronunciation; a cross-language reference can transfer an accent. Clone mode is the upstream model's most stable mode; voice design was trained on English/Chinese and may be less stable in other languages. [Pinned cloning and language guidance](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/README.md)

To use a single reference for compatible models, set the notebook's global `reference_audio` and `reference_text`, with `speaker_mode="auto"`. In an OmniVoice profile, `reference` may be `"auto"`, `True`, or `False`; `True` requests a reference, `False` selects design unless a reference/cache is explicitly supplied or global `speaker_mode="reference"` forces cloning. A per-profile WAV uses `parameters={"ref_audio": "/kaggle/input/.../speaker.wav", "ref_text": "Exact transcript."}`. The automatic clone route removes default male design instructions. To deliberately combine a clone with matching attributes, use `clone_parameters={"instruct": "male, low pitch"}`. Conflicting attributes usually lose to the reference; matching dialect attributes can help. [Pinned combination tips](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/tips.md)

### Complete voice-design attribute vocabulary

Choose **at most one value per category**. English values are case-insensitive. Chinese equivalents may be used; commas `,` or `，` are normalized. A dialect and an English accent cannot be combined. Unsupported emotion/style words such as `serious`, `angry`, `happy`, `sad`, `deep`, or `narrator` are rejected as instructions: use the actual pitch values or a reference recording with the desired delivery.

| Category | All accepted values, with Chinese equivalents where available |
|---|---|
| Gender | `male` / `男`; `female` / `女` |
| Age | `child` / `儿童`; `teenager` / `少年`; `young adult` / `青年`; `middle-aged` / `中年`; `elderly` / `老年` |
| Pitch | `very low pitch` / `极低音调`; `low pitch` / `低音调`; `moderate pitch` / `中音调`; `high pitch` / `高音调`; `very high pitch` / `极高音调` |
| Style | `whisper` / `耳语` |
| English accent | `american accent`, `british accent`, `australian accent`, `canadian accent`, `indian accent`, `chinese accent`, `korean accent`, `japanese accent`, `portuguese accent`, `russian accent` |
| Chinese dialect | `河南话`, `陕西话`, `四川话`, `贵州话`, `云南话`, `桂林话`, `济南话`, `石家庄话`, `甘肃话`, `宁夏话`, `青岛话`, `东北话` |

Accent controls apply to English speech; dialect controls apply to Chinese speech. Leaving a category out lets the model choose it. Some combinations may be ignored. [Pinned complete voice-design reference](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/voice-design.md)

There are **no named emotion sliders and no per-word emphasis parameter** in this adapter. You can influence delivery with punctuation, a matching expressive reference, and these complete inline non-verbal tags:

`[laughter]`, `[sigh]`, `[confirmation-en]`, `[question-en]`, `[question-ah]`, `[question-oh]`, `[question-ei]`, `[question-yi]`, `[surprise-ah]`, `[surprise-oh]`, `[surprise-wa]`, `[surprise-yo]`, `[dissatisfaction-hnn]`.

English pronunciation may be overridden with bracketed uppercase CMU phonemes, e.g. `[B EY1 S]`; Chinese pronunciations can use pinyin tone numbers. These are pronunciation/non-verbal controls, not general emotion labels or SSML. The chunker preserves complete bracket groups. [Pinned inline-control examples](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/README.md)

### Generation and planning controls

Keys use underscores in a profile `parameters` dictionary; the standalone CLI uses hyphens. “Default” below means **standalone script default**. The enabled notebook profile changes `num_step` to **64**, `max_chunk_seconds` to **12.0**, and `dtype` to **float16**; export is **48,000 Hz** via the global configuration. Unspecified values retain script defaults.

| Key | Default; accepted values | Exactly what it changes |
|---|---|---|
| `language` | `"en"`; one supported ID/name, or `"none"`/`None` | Language conditioning. Full names are case-insensitive; use the lower-case IDs in the full catalog below. `none` omits language conditioning. Unknown non-null values are rejected by the adapter. It does not translate text. |
| `instruct` | Male design above; supported attributes or `""` | Voice-design conditioning. Empty string requests auto voice. With a clone, matching instructions may reinforce the reference; the notebook's clone route removes design defaults unless explicitly provided in `clone_parameters`. |
| `ref_audio` | `None`; readable audio path | Recorded speaker prompt. Requires `ref_text`. File loading/downmixing/resampling/preprocessing occurs before planning; first use is not an instant text-only preview. |
| `ref_text` | `None`; exact nonempty transcript | The words spoken in the reference, not the text to synthesize. Required with new audio, forbidden without audio. The adapter does not load automatic reference ASR. A cache already contains this transcript. |
| `prompt_cache` | `None`; existing/new `.pt` path | Existing path loads a clone and forbids supplying new audio simultaneously. Missing path requires new audio/transcript and is written after reference encoding for synthesis. Cached shape must be 8 codebooks by positive token count. |
| `preprocess_prompt` | `True`; boolean | Removes long reference silences (upstream 200 ms interior/200 ms trailing/100 ms leading handling) and adds terminal punctuation if missing. Changing it changes reference token count and therefore rule-based duration allocation. Existing cache is already prepared and is unaffected. Does not remove generated silence. |
| `normalize_text` | `False`; boolean | Expands numbers/symbols to spoken forms before both planning and synthesis, while preserving inline tags/phonemes/pinyin. English/Chinese require optional OmniVoice `[tn]` dependencies; this notebook does not install that extra by default. Other languages use a best-effort integer conversion when `num2words` supports them. Inspect the logged normalized text; no universal normalization is promised. |
| `speed` | `1.0`; finite `>0` | Divides rule-estimated token allocation: above 1 shortens planned speech, below 1 lengthens it. This is token allocation, not playback-speed resampling. Ignored when `duration` is set. Extreme values can cause rushed/missing speech or surplus silence. |
| `duration` | `None`; finite seconds `>0` | Overrides speed and allocates `max(1, floor(duration*25))` speech tokens across chunks: **40 ms resolution**, before explicitly added inter-chunk pauses. The final file may therefore differ from the literal requested float; the reported sample-count preview is the exact planned export. Too few tokens for the required chunks are rejected. |
| `max_chunk_seconds` | `8.0`; finite `>=0.04` | Maximum planned speech-token duration per manual chunk, rounded down to whole tokens. Smaller values lower context/VRAM costs but produce more boundaries; larger values provide longer context at higher memory cost. Splits prefer sentence/clause boundaries; short terminal chunks are rebalanced when feasible. Changing it can change total allocation through per-chunk rounding and the short-text rule. |
| `max_reference_seconds` | `10.0`; finite `>0` | Rejects prepared/cached prompts whose token duration exceeds the cap. Does not shorten the WAV, set output duration, or guarantee memory usage. Raising it admits longer references and can raise VRAM/inference cost. |
| `num_step` | `32`; integer `>=1` | Number of neural iterative unmasking passes per chunk; 64 in the notebook quality profile. More passes cost more generation time and may improve quality; they do not change frozen token lengths or guarantee monotonic improvement. |
| `guidance_scale` | `2.0`; finite `>=0` | Conditioning extrapolation uses `conditional_logp + scale*(conditional_logp-unconditional_logp)` before normalization. Zero uses the conditional prediction alone; larger values strengthen the difference. Token length stays fixed. Upstream still builds conditional/unconditional inputs, so zero is not a guaranteed half-memory mode. |
| `t_shift` | `0.1`; finite `>0` | Warps a linear time grid by `a*t/(1+(a-1)*t)`. This changes how many masked positions are committed at each iteration. `1.0` is linear; below 1 produces smaller early increments and larger late increments in this pinned implementation. It does not change target duration. |
| `layer_penalty_factor` | `5.0`; finite `>=0` | Subtracts `codebook_index*factor` from selection scores. Larger values favor filling earlier codec codebooks first. Zero removes that ordering preference. Duration remains fixed. |
| `position_temperature` | `5.0`; finite `>=0` | Controls Gumbel randomness in deciding which masked positions to commit. Zero chooses positions greedily; larger values add more randomness. Different seeds can vary waveform content despite identical length. |
| `class_temperature` | `0.0`; finite `>=0` | Zero chooses the most likely token value. Positive values sample using Gumbel noise after a fixed top-10%-of-vocabulary filter. Controls token diversity, separately from which positions get filled; length remains fixed. |
| `denoise` | `True`; boolean | Adds a model conditioning token when reference tokens are present, requesting cleaner speech. It is not a waveform denoising filter. It has no such token effect for a reference-free first chunk; later design/auto chunks use the first generated chunk as a prompt. |
| `fade_ms` | `10.0`; finite `[0,20]` | Applies length-preserving amplitude fades at each manual chunk edge. Zero disables fades. Helps boundary clicks; long fades can attenuate first/last phonemes. No samples are removed or crossfaded away. |
| `peak_limit` | `0.98`; finite `(0,1]` | After export resampling, attenuates the entire WAV only if its absolute peak exceeds this ceiling. Logs gain/original peak. Prevents overdrive; does not restore distorted source speech, compress dynamics, or change length. |
| `sample_rate` | `None` = native **24,000 Hz**; integer `[8000,192000]` | Rational resampling for the exported WAV; 48,000 in the notebook. Codec bandwidth remains native. The planner includes the exact rounded output sample count. This is unrelated to generation temperature. |
| `pause_ms` | `0.0`; finite `>=0` | Adds this rounded number of milliseconds of zero samples **between manual chunks only**. Included in preview. It does not control pauses inside a chunk or strip model-produced gaps. Zero avoids deliberately adding boundary gaps. |
| `dtype` | `"auto"`; `auto`, `float32`, `float16`, `bfloat16` | Auto selects float32 on CPU and float16 on CUDA. Float16 CPU is rejected. Float16 is the tested/configured T4 path; other dtypes require compatible hardware/runtime. Dtype affects compute/memory, not planned sample length. |

Decoder and scheduling behavior above was checked against the [pinned model implementation](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/omnivoice/models/omnivoice.py), not inferred from the parameter names alone. [Upstream parameter reference](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/generation-parameters.md)

**Why the duration preview can be exact:** without explicit duration, the adapter applies the pinned Unicode-weight/reference-rate rule, freezes per-chunk target tokens, then generates exactly those tokens. With explicit duration it freezes an allocation from your requested seconds. Native output is 960 samples per 25 Hz token at 24 kHz. Planned pauses and rational export resampling are included. Changing seed, steps, guidance, or temperatures changes content while keeping this allocation fixed. A changed reference, preprocessing, normalization, text, speed, chunk bound, or duration requires a new plan. Short-text allocation includes the upstream nonlinear boost below 50 tokens. A cached prompt avoids re-encoding; imports/reference work still make cold preflight slower than warm arithmetic. **Exact length does not certify complete words, natural pacing, or an independently accurate prediction of unconstrained speech.** [Pinned rule allocator](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/omnivoice/utils/duration.py)

The adapter deliberately disables upstream `postprocess_output` (silence removal), `pad_duration` (extra edge silence), and automatic `audio_chunk_duration`/`audio_chunk_threshold` chunking. It fixes them to `False`, `0.0`, `0.0`, and infinity, respectively, so output remains sample-exact. They are **not configurable adapter keys**; passing them raises an unknown-key error. `fade_duration` is exposed as `fade_ms`; `preprocess_prompt` is applied while preparing the reference, then disabled at generation to avoid applying it twice. Other upstream-only choices—automatic reference ASR, FlashInfer/CUDA graphs, user batch size, training/fine-tuning, and multi-speaker dialogue—are not exposed by this benchmark adapter.

### Common execution controls present in the OmniVoice script

The notebook supervisor supplies text, take count, seeds, output directories, local assets, and the visible GPU device. Override generation controls in each profile rather than overriding supervisor bookkeeping.

| Key | Script default / bounds | Purpose |
|---|---|---|
| `text`, `text_file` | Exactly one required; nonempty, at most 500 whitespace-delimited words | Direct input string or UTF-8 text-file path. The notebook tests complete 10/100/500-word passages. |
| `config` | None; JSON object path | Loads underscore/hyphen CLI destination keys; explicit CLI values override JSON. Unknown keys and wrong types fail. `help`, `config`, and `self_test` cannot be supplied inside this JSON. |
| `n` | `1`; integer `>=1` | Number of output takes, each reusing the same plan. |
| `seed` | `1234`; integers with `0 <= seed` and `seed+n-1 < 2**32` | Successive take seeds are `seed`, `seed+1`, etc.; seeds are set for Torch/NumPy/CUDA. Greedy settings can reduce variation. Reproducibility across hardware is not guaranteed. |
| `device` | `"cpu"`; `cpu`, `cuda`, `cuda:N` | Execution target. The notebook worker hides the other GPU, so `cuda:0` refers to its assigned physical T4. |
| `max_vram_gb` | `5.0`; finite `(0,5]`, decimal GB | Caps PyTorch's allocator with a context reserve; the notebook additionally samples whole-device NVML usage and kills over-budget jobs. Neither mechanism certifies an unobserved instantaneous driver/library peak. |
| `output_dir` | `outputs/omnivoice`; path | Writes estimate/manifest and numbered seed-labelled float WAVs. Existing intended sample paths are rejected to prevent silent overwrite. |
| `estimate_only` | `False`; boolean | Writes the plan without loading weights or generating speech. New-reference audio work and dependencies can still be needed; an existing prompt cache requires the pinned package but not reference re-encoding. |
| `local_model` | `None`; directory | Uses a local pinned-model snapshot including the official `audio_tokenizer/config.json`. Otherwise downloads the pinned Hugging Face revision. A different checkpoint is not an audited guarantee. |
| `local_files_only` | `False`; boolean | Restricts Hugging Face snapshot retrieval to cache; notebook predownloads assets and sets this true. |
| `self_test` | `False`; CLI flag only | Runs dependency-free arithmetic and chunking sanity checks, not model-quality or GPU validation. |
| `help` | CLI `--help` | Prints standalone CLI usage and exits; not a JSON/profile generation setting. |

### All supported OmniVoice language IDs and names

The pinned mapping contains **646 IDs and 646 names**. Examples relevant to this setup: `en` (English), `fa` (Persian), `zh` (Chinese), `ja` (Japanese), `de` (German), `fr` (French), `es` (Spanish), `hi` (Hindi), `ru` (Russian). Language coverage does not imply uniform accent or quality. Min Nan Chinese (`nan`) requires Tai-lo romanized input in this release. The complete frozen mapping is shown below and is also present in the notebook's machine-readable voice catalog. [Pinned language mapping](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/omnivoice/utils/lang_map.py)

<details>
<summary>Expand all 646 accepted language IDs and names</summary>

| ID | Accepted full name |
|---|---|
| `aae` | Arbëreshë Albanian |
| `aal` | Afade |
| `aao` | Algerian Saharan Arabic |
| `ab` | Abkhazian |
| `abb` | Bankon |
| `abn` | Abua |
| `abr` | Abron |
| `abs` | Ambonese Malay |
| `abv` | Baharna Arabic |
| `acm` | Mesopotamian Arabic |
| `acw` | Hijazi Arabic |
| `acx` | Omani Arabic |
| `adf` | Dhofari Arabic |
| `adx` | Amdo Tibetan |
| `ady` | Adyghe |
| `aeb` | Tunisian Arabic |
| `aec` | Saidi Arabic |
| `af` | Afrikaans |
| `afb` | Gulf Arabic |
| `afo` | Eloyi |
| `ahl` | Igo |
| `ahs` | Ashe |
| `ajg` | Aja (Benin) |
| `aju` | Judeo-Moroccan Arabic |
| `ala` | Alago |
| `aln` | Gheg Albanian |
| `alo` | Larike-Wakasihu |
| `am` | Amharic |
| `amu` | Guerrero Amuzgo |
| `an` | Aragonese |
| `anc` | Ngas |
| `ank` | Goemai |
| `anp` | Angika |
| `anw` | Anaang |
| `aom` | Ömie |
| `apc` | Levantine Arabic |
| `apd` | Sudanese Arabic |
| `arb` | Standard Arabic |
| `arq` | Algerian Arabic |
| `ars` | Najdi Arabic |
| `ary` | Moroccan Arabic |
| `arz` | Egyptian Arabic |
| `as` | Assamese |
| `ast` | Asturian |
| `avl` | Eastern Egyptian Bedawi Arabic |
| `awo` | Awak |
| `ayl` | Libyan Arabic |
| `ayp` | North Mesopotamian Arabic |
| `az` | Azerbaijani |
| `ba` | Bashkir |
| `bag` | Tuki |
| `bas` | Basa (Cameroon) |
| `bax` | Bamun |
| `bba` | Baatonum |
| `bbj` | Ghomálá' |
| `bbl` | Bats |
| `bbu` | Kulung (Nigeria) |
| `bce` | Bamenyam |
| `bci` | Baoulé |
| `bcs` | Kohumono |
| `bcy` | Bacama |
| `bda` | Bayot |
| `bde` | Bade |
| `bdm` | Buduma |
| `be` | Belarusian |
| `beb` | Bebele |
| `bew` | Betawi |
| `bfd` | Bafut |
| `bft` | Balti |
| `bg` | Bulgarian |
| `bgp` | Eastern Balochi |
| `bhb` | Bhili |
| `bhh` | Bukharic |
| `bho` | Bhojpuri |
| `bhp` | Bima |
| `bhr` | Bara Malagasy |
| `bjj` | Kanauji |
| `bjk` | Barok |
| `bjn` | Banjar |
| `bjt` | Balanta-Ganja |
| `bkh` | Bakoko |
| `bkm` | Kom (Cameroon) |
| `bky` | Bokyi |
| `bmm` | Northern Betsimisaraka Malagasy |
| `bmq` | Bomu |
| `bn` | Bengali |
| `bnm` | Batanga |
| `bnn` | Bunun |
| `bns` | Bundeli |
| `bo` | Tibetan |
| `bou` | Bondei |
| `bqg` | Bago-Kusuntu |
| `br` | Breton |
| `bra` | Braj |
| `brh` | Brahui |
| `bri` | Mokpwe |
| `brx` | Bodo |
| `bs` | Bosnian |
| `bsh` | Kati |
| `bsj` | Bangwinji |
| `bsk` | Burushaski |
| `btm` | Batak Mandailing |
| `btv` | Bateri |
| `bug` | Buginese |
| `bum` | Bulu (Cameroon) |
| `buo` | Terei |
| `bux` | Boghom |
| `bwr` | Bura-Pabir |
| `bxf` | Bilur |
| `byc` | Ubaghara |
| `bys` | Burak |
| `byv` | Medumba |
| `byx` | Qaqet |
| `bzc` | Southern Betsimisaraka Malagasy |
| `bzw` | Basa (Nigeria) |
| `ca` | Catalan |
| `ccg` | Samba Daka |
| `ceb` | Cebuano |
| `cen` | Cen |
| `cfa` | Dijim-Bwilim |
| `cgg` | Chiga |
| `chq` | Quiotepec Chinantec |
| `cjk` | Chokwe |
| `ckb` | Central Kurdish |
| `ckl` | Cibak |
| `ckr` | Kairak |
| `cky` | Cakfem-Mushere |
| `cnh` | Hakha Chin |
| `cpy` | South Ucayali Ashéninka |
| `cs` | Czech |
| `cte` | Tepinapa Chinantec |
| `ctl` | Tlacoatzintepec Chinantec |
| `cut` | Teutila Cuicatec |
| `cux` | Tepeuxila Cuicatec |
| `cv` | Chuvash |
| `cy` | Welsh |
| `da` | Danish |
| `dag` | Dagbani |
| `dar` | Dargwa |
| `dav` | Taita |
| `dbd` | Dadiya |
| `dcc` | Deccan |
| `de` | German |
| `deg` | Degema |
| `dgh` | Dghwede |
| `dgo` | Dogri |
| `dje` | Zarma |
| `dmk` | Domaaki |
| `dml` | Dameli |
| `dru` | Rukai |
| `dty` | Dotyali |
| `dua` | Duala |
| `dv` | Dhivehi |
| `dyu` | Dyula |
| `dzg` | Dazaga |
| `ebr` | Ebrié |
| `ebu` | Embu |
| `ego` | Eggon |
| `eiv` | Askopan |
| `eko` | Koti |
| `ekr` | Yace |
| `el` | Greek |
| `elm` | Eleme |
| `en` | English |
| `eo` | Esperanto |
| `es` | Spanish |
| `esu` | Central Yupik |
| `et` | Estonian |
| `eto` | Eton (Cameroon) |
| `ets` | Yekhee |
| `etu` | Ejagham |
| `eu` | Basque |
| `ewo` | Ewondo |
| `ext` | Extremaduran |
| `eyo` | Keiyo |
| `fa` | Persian |
| `fan` | Fang (Equatorial Guinea) |
| `fat` | Fanti |
| `ff` | Fulah |
| `ffm` | Maasina Fulfulde |
| `fi` | Finnish |
| `fia` | Nobiin |
| `fil` | Filipino |
| `fip` | Fipa |
| `fkk` | Kirya-Konzəl |
| `fmp` | Fe'Fe' |
| `fr` | French |
| `fub` | Adamawa Fulfulde |
| `fuc` | Pulaar |
| `fue` | Borgu Fulfulde |
| `fuf` | Pular |
| `fuh` | Western Niger Fulfulde |
| `fui` | Bagirmi Fulfulde |
| `fuq` | Central-Eastern Niger Fulfulde |
| `fuv` | Nigerian Fulfulde |
| `fy` | Western Frisian |
| `ga` | Irish |
| `gbm` | Garhwali |
| `gbr` | Gbagyi |
| `gby` | Gbari |
| `gcc` | Mali |
| `gdf` | Guduf-Gava |
| `gej` | Gen |
| `ges` | Geser-Gorom |
| `ggg` | Gurgula |
| `gid` | Gidar |
| `gig` | Goaria |
| `giz` | South Giziga |
| `gjk` | Kachi Koli |
| `gju` | Gujari |
| `gl` | Galician |
| `glw` | Glavda |
| `gn` | Guarani |
| `gol` | Gola |
| `gom` | Goan Konkani |
| `gsl` | Gusilay |
| `gu` | Gujarati |
| `gui` | Eastern Bolivian Guaraní |
| `gur` | Farefare |
| `guz` | Gusii |
| `gv` | Manx |
| `gwc` | Gawri |
| `gwe` | Gweno |
| `gwt` | Gawar-Bati |
| `gya` | Northwest Gbaya |
| `gyz` | Geji |
| `ha` | Hausa |
| `hah` | Hahon |
| `hao` | Hakö |
| `haw` | Hawaiian |
| `haz` | Hazaragi |
| `hbb` | Huba |
| `he` | Hebrew |
| `hem` | Hemba |
| `hi` | Hindi |
| `hia` | Lamang |
| `hkk` | Hunjara-Kaina Ke |
| `hla` | Halia |
| `hno` | Northern Hindko |
| `hoj` | Hadothi |
| `hr` | Croatian |
| `hsb` | Upper Sorbian |
| `ht` | Haitian |
| `hu` | Hungarian |
| `hue` | San Francisco Del Mar Huave |
| `hul` | Hula |
| `hux` | Nüpode Huitoto |
| `hwo` | Hwana |
| `hy` | Armenian |
| `hz` | Herero |
| `ia` | Interlingua (International Auxiliary Language Association) |
| `ibb` | Ibibio |
| `id` | Indonesian |
| `ida` | Idakho-Isukha-Tiriki |
| `idu` | Idoma |
| `ig` | Igbo |
| `ijc` | Izon |
| `ijn` | Kalabari |
| `ik` | Inupiaq |
| `ikw` | Ikwere |
| `is` | Icelandic |
| `ish` | Esan |
| `iso` | Isoko |
| `it` | Italian |
| `its` | Isekiri |
| `itw` | Ito |
| `itz` | Itzá |
| `ja` | Japanese |
| `jal` | Yalahatan |
| `jax` | Jambi Malay |
| `jgo` | Ngomba |
| `jmx` | Western Juxtlahuaca Mixtec |
| `jns` | Jaunsari |
| `jqr` | Jaqaru |
| `juk` | Wapan |
| `juo` | Jiba |
| `jv` | Javanese |
| `ka` | Georgian |
| `kab` | Kabyle |
| `kai` | Karekare |
| `kaj` | Jju |
| `kam` | Kamba |
| `kbd` | Kabardian |
| `kbl` | Kanembu |
| `kbt` | Abadi |
| `kcq` | Kamo |
| `kdh` | Tem |
| `kea` | Kabuverdianu |
| `keu` | Akebu |
| `kfe` | Kota (India) |
| `kfk` | Kinnauri |
| `kfp` | Korwa |
| `khg` | Khams Tibetan |
| `khw` | Khowar |
| `kj` | Kuanyama |
| `kjc` | Coastal Konjo |
| `kjk` | Highland Konjo |
| `kk` | Kazakh |
| `kln` | Kalenjin |
| `kls` | Kalasha |
| `km` | Khmer |
| `kmr` | Northern Kurdish |
| `kmy` | Koma |
| `kn` | Kannada |
| `kna` | Dera (Nigeria) |
| `knn` | Konkani |
| `ko` | Korean |
| `kol` | Kol (Papua New Guinea) |
| `koo` | Konzo |
| `kpo` | Ikposo |
| `kqo` | Eastern Krahn |
| `ks` | Kashmiri |
| `ksd` | Kuanua |
| `ksf` | Bafia |
| `kto` | Kuot |
| `kuh` | Kushi |
| `kvx` | Parkari Koli |
| `kw` | Cornish |
| `kwm` | Kwambi |
| `kxp` | Wadiyara Koli |
| `ky` | Kirghiz |
| `kyx` | Rapoisi |
| `lag` | Rangi |
| `lb` | Luxembourgish |
| `lcm` | Tungag |
| `ldb` | DũYa |
| `lg` | Ganda |
| `lij` | Ligurian |
| `lir` | Liberian English |
| `lkb` | Kabras |
| `lla` | Lala-Roba |
| `ln` | Lingala |
| `lnu` | Longuda |
| `lo` | Lao |
| `loa` | Loloda |
| `lrk` | Loarki |
| `lss` | Lasi |
| `lt` | Lithuanian |
| `ltg` | Latgalian |
| `lto` | Tsotso |
| `lua` | Luba-Lulua |
| `luo` | Luo |
| `lus` | Lushai |
| `lv` | Latvian |
| `lwg` | Wanga |
| `mab` | Yutanduchi Mixtec |
| `maf` | Mafa |
| `mai` | Maithili |
| `mau` | Huautla Mazatec |
| `max` | North Moluccan Malay |
| `mbo` | Mbo (Cameroon) |
| `mcf` | Matsés |
| `mcn` | Masana |
| `mcx` | Mpiemo |
| `mdd` | Mbum |
| `mde` | Maba (Chad) |
| `mdf` | Moksha |
| `mek` | Mekeo |
| `mer` | Meru |
| `meu` | Motu |
| `mfm` | Marghi South |
| `mfn` | Cross River Mbembe |
| `mfo` | Mbe |
| `mfv` | Mandjak |
| `mgg` | Mpumpong |
| `mgi` | Lijili |
| `mhk` | Mungaka |
| `mhr` | Eastern Mari |
| `mi` | Maori |
| `mig` | San Miguel El Grande Mixtec |
| `miu` | Cacaloxtepec Mixtec |
| `mk` | Macedonian |
| `mkf` | Miya |
| `mki` | Dhatki |
| `ml` | Malayalam |
| `mlq` | Western Maninkakan |
| `mn` | Mongolian |
| `mne` | Naba |
| `mni` | Manipuri |
| `mqy` | Manggarai |
| `mr` | Marathi |
| `mrj` | Western Mari |
| `mrr` | Maria (India) |
| `mrt` | Marghi Central |
| `ms` | Malay |
| `mse` | Musey |
| `msh` | Masikoro Malagasy |
| `msw` | Mansoanka |
| `mt` | Maltese |
| `mtr` | Mewari |
| `mtu` | Tututepec Mixtec |
| `mtx` | Tidaá Mixtec |
| `mua` | Mundang |
| `mug` | Musgu |
| `mui` | Musi |
| `mve` | Marwari (Pakistan) |
| `mvy` | Indus Kohistani |
| `mxs` | Huitepec Mixtec |
| `mxu` | Mada (Cameroon) |
| `mxy` | Southeastern Nochixtlán Mixtec |
| `my` | Burmese |
| `myv` | Erzya |
| `mzl` | Mazatlán Mixe |
| `nal` | Nalik |
| `nan` | Min Nan Chinese |
| `nap` | Neapolitan |
| `nb` | Norwegian Bokmål |
| `nbh` | Ngamo |
| `ncf` | Notsi |
| `nco` | Sibe |
| `ncx` | Central Puebla Nahuatl |
| `ndi` | Samba Leko |
| `ng` | Ndonga |
| `ngi` | Ngizim |
| `nhg` | Tetelcingo Nahuatl |
| `nhi` | Zacatlán-Ahuacatlán-Tepetzintla Nahuatl |
| `nhn` | Central Nahuatl |
| `nhq` | Huaxcaleca Nahuatl |
| `nja` | Nzanyi |
| `nl` | Dutch |
| `nla` | Ngombale |
| `nlv` | Orizaba Nahuatl |
| `nmg` | Kwasio |
| `nmz` | Nawdm |
| `nn` | Norwegian Nynorsk |
| `nnh` | Ngiemboon |
| `no` | Norwegian |
| `noe` | Nimadi |
| `npi` | Nepali |
| `nso` | Pedi |
| `ny` | Chichewa |
| `nyu` | Nyungwe |
| `oc` | Occitan |
| `odk` | Od |
| `odu` | Odual |
| `ogo` | Khana |
| `om` | Oromo |
| `orc` | Orma |
| `oru` | Ormuri |
| `ory` | Odia |
| `os` | Iron Ossetic |
| `pa` | Panjabi |
| `pbs` | Central Pame |
| `pbt` | Southern Pashto |
| `pbu` | Northern Pashto |
| `pcm` | Nigerian Pidgin |
| `pex` | Petats |
| `phl` | Phalura |
| `phr` | Pahari-Potwari |
| `pip` | Pero |
| `piy` | Piya-Kwonci |
| `pko` | Pökoot |
| `pl` | Polish |
| `plk` | Kohistani Shina |
| `plt` | Plateau Malagasy |
| `pmq` | Northern Pame |
| `pms` | Piemontese |
| `pmy` | Papuan Malay |
| `pnb` | Western Panjabi |
| `poc` | Poqomam |
| `poe` | San Juan Atzingo Popoloca |
| `pow` | San Felipe Otlaltepec Popoloca |
| `prq` | Ashéninka Perené |
| `ps` | Pushto |
| `pst` | Central Pashto |
| `pt` | Portuguese |
| `pua` | Western Highland Purepecha |
| `pwn` | Paiwan |
| `qug` | Chimborazo Highland Quichua |
| `qum` | Sipacapense |
| `qup` | Southern Pastaza Quechua |
| `qur` | Yanahuanca Pasco Quechua |
| `qus` | Santiago Del Estero Quichua |
| `quv` | Sacapulteco |
| `qux` | Yauyos Quechua |
| `quy` | Ayacucho Quechua |
| `qva` | Ambo-Pasco Quechua |
| `qvi` | Imbabura Highland Quichua |
| `qvj` | Loja Highland Quichua |
| `qvl` | Cajatambo North Lima Quechua |
| `qwa` | Corongo Ancash Quechua |
| `qws` | Sihuas Ancash Quechua |
| `qxa` | Chiquián Ancash Quechua |
| `qxp` | Puno Quechua |
| `qxt` | Santa Ana De Tusi Pasco Quechua |
| `qxu` | Arequipa-La Unión Quechua |
| `qxw` | Jauja Wanca Quechua |
| `rag` | Logooli |
| `rm` | Romansh |
| `ro` | Romanian |
| `rob` | Tae' |
| `rof` | Rombo |
| `roo` | Rotokas |
| `rth` | Ratahan |
| `ru` | Russian |
| `rup` | Macedo-Romanian |
| `rw` | Kinyarwanda |
| `sa` | Sanskrit |
| `sah` | Yakut |
| `sat` | Santali |
| `sau` | Saleman |
| `say` | Saya |
| `sbn` | Sindhi Bhil |
| `sc` | Sardinian |
| `scl` | Shina |
| `scn` | Sicilian |
| `sd` | Sindhi |
| `sei` | Seri |
| `shu` | Chadian Arabic |
| `si` | Sinhala |
| `sip` | Sikkimese |
| `siw` | Siwai |
| `sjr` | Siar-Lak |
| `sk` | Slovak |
| `skg` | Sakalava Malagasy |
| `skr` | Saraiki |
| `sl` | Slovenian |
| `sn` | Shona |
| `snc` | Sinaugoro |
| `snk` | Soninke |
| `so` | Somali |
| `sol` | Solos |
| `sps` | Saposa |
| `sq` | Albanian |
| `sr` | Serbian |
| `src` | Logudorese Sardinian |
| `sro` | Campidanese Sardinian |
| `ssi` | Sansi |
| `ste` | Liana-Seti |
| `sua` | Sulka |
| `sv` | Swedish |
| `sva` | Svan |
| `sw` | Swahili |
| `szy` | Sakizaya |
| `ta` | Tamil |
| `tan` | Tangale |
| `tar` | Central Tarahumara |
| `tay` | Atayal |
| `tbf` | Mandara |
| `tcf` | Malinaltepec Me'Phaa |
| `tcy` | Tulu |
| `tdn` | Tondano |
| `tdx` | Tandroy-Mahafaly Malagasy |
| `te` | Telugu |
| `tg` | Tajik |
| `tgc` | Tigak |
| `th` | Thai |
| `the` | Chitwania Tharu |
| `thq` | Kochila Tharu |
| `thr` | Rana Tharu |
| `thv` | Tahaggart Tamahaq |
| `ti` | Tigrinya |
| `tig` | Tigre |
| `tio` | Teop |
| `tk` | Turkmen |
| `tkg` | Tesaka Malagasy |
| `tkt` | Kathoriya Tharu |
| `tli` | Tlingit |
| `tlp` | Filomena Mata-Coahuitlán Totonac |
| `tn` | Tswana |
| `tok` | Toki Pona |
| `tpl` | Tlacoapa Me'Phaa |
| `tpz` | Tinputz |
| `tqp` | Tomoip |
| `tr` | Turkish |
| `trp` | Kok Borok |
| `trq` | San Martín Itunyoso Triqui |
| `trv` | Sediq |
| `trw` | Torwali |
| `tt` | Tatar |
| `ttj` | Tooro |
| `ttr` | Tera |
| `ttu` | Torau |
| `tui` | Tupuri |
| `tul` | Tula |
| `tuq` | Tedaga |
| `tuv` | Turkana |
| `tuy` | Tugen |
| `tvo` | Tidore |
| `tvu` | Tunen |
| `tw` | Twi |
| `twu` | Termanu |
| `txs` | Tonsea |
| `txy` | Tanosy Malagasy |
| `udl` | Wuzlam |
| `ug` | Uighur |
| `uk` | Ukrainian |
| `uki` | Kui (India) |
| `umb` | Umbundu |
| `ur` | Urdu |
| `ush` | Ushojo |
| `uz` | Uzbek |
| `uzn` | Northern Uzbek |
| `vai` | Vai |
| `var` | Huarijio |
| `ver` | Mom Jango |
| `vi` | Vietnamese |
| `vmc` | Juxtlahuaca Mixtec |
| `vmj` | Ixtayutla Mixtec |
| `vmm` | Mitlatongo Mixtec |
| `vmp` | Soyaltepec Mazatec |
| `vmz` | Mazatlán Mazatec |
| `vot` | Votic |
| `vro` | Võro |
| `wbl` | Wakhi |
| `wci` | Waci Gbe |
| `weo` | Wemale |
| `wes` | Cameroon Pidgin |
| `wja` | Waja |
| `wji` | Warji |
| `wo` | Wolof |
| `wof` | Gambian Wolof |
| `xh` | Xhosa |
| `xhe` | Khetrani |
| `xka` | Kalkoti |
| `xmf` | Mingrelian |
| `xmv` | Antankarana Malagasy |
| `xmw` | Tsimihety Malagasy |
| `xpe` | Liberia Kpelle |
| `xti` | Sinicahua Mixtec |
| `xtu` | Cuyamecalco Mixtec |
| `yaq` | Yaqui |
| `yav` | Yangben |
| `yay` | Agwagwune |
| `ydd` | Eastern Yiddish |
| `ydg` | Yidgha |
| `yer` | Tarok |
| `yes` | Nyankpa |
| `yi` | Yiddish |
| `yo` | Yoruba |
| `yue` | Cantonese |
| `zga` | Kinga |
| `zgh` | Standard Moroccan Tamazight |
| `zh` | Chinese |
| `zoc` | Copainalá Zoque |
| `zoh` | Chimalapa Zoque |
| `zor` | Rayón Zoque |
| `zpv` | Chichicapan Zapotec |
| `zpy` | Mazaltepec Zapotec |
| `ztg` | Xanaguía Zapotec |
| `ztn` | Santa Catarina Albarradas Zapotec |
| `ztp` | Loxicha Zapotec |
| `zts` | Tilquiapan Zapotec |
| `ztu` | Güilá Zapotec |
| `zu` | Zulu |
| `zza` | Zaza |

</details>


## 1. Experiment configuration

Defaults: all six models, **10/100/500 words**, **two takes per job**, one enabled
profile per model. Enable alternate profiles or add any native option shown in
the saved help logs. Profile-level `n` and `seed` override the global values;
put other controls in `parameters`. GPU selection, text, output paths and the
VRAM target are controlled by the supervisor.

The built-in cases are three distinct, complete English passages: a ten-word
sentence, a one-hundred-word historical vignette, and a five-hundred-word story
with five paragraphs. No sentence is looped to reach a target word count. Edit
`TEXT_CASES` for your own passages or other lengths within 10–500 words.

For reference-based models, supply a clean **mono 1–8 second** reference and
its exact transcript (3–8 seconds is recommended), or keep both fields `None`
to create a disclosed male Kokoro reference on CPU. Synthetic reference quality
can affect cloning quality.

`speaker_mode="auto"` uses your supplied WAV for F5, StyleTTS2 and OmniVoice.
Otherwise defaults are Kokoro **am_fenrir**, EmotiVoice **9017/Neutral**,
Supertonic **M2** (documented deep/serious), and OmniVoice male/low-pitch design;
F5/Style use the male reference. `speaker_mode="reference"` tests only the three
cloning-capable models and reports the others as unsupported. `speaker_mode="default"`
uses male defaults. Style needs only the WAV; F5/Omni also require its transcript.
For Omni clone-specific instructions, use a profile's `clone_parameters` dictionary;
the default male design instruction is removed when your reference is selected.


In [ ]:
import datetime as dt
import uuid
from pathlib import Path

ROOT = Path("/kaggle/working/tts_benchmark")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / "runs" / RUN_ID
PACKAGE_DIR = RUN_DIR / "download"
RUN_DIR.mkdir(parents=True, exist_ok=False)
PACKAGE_DIR.mkdir(parents=True, exist_ok=False)

# Change these before running the installation/benchmark cells.
CONFIG = {
    "install_models": ["kokoro", "f5", "emotivoice", "styletts2", "omnivoice", "supertonic"],
    "force_setup": False,
    "torch_cuda_variant": "auto",
    "strict_cuda_driver": False,
    "n": 2,
    "seed": 1234,
    "sample_rate": 48000,
    "pause_ms": 0.0,           # Native punctuation already contributes pauses; no extra gap per chunk.
    "max_vram_gb": 5.0,          # Decimal GB: 5,000,000,000 bytes.
    "nvml_interval_ms": 10.0,
    "enforce_observed_budget": True,
    "budget_scope": "device_total", # Conservative; includes pre-existing allocations.
    "job_timeout_seconds": 3600,
    "gpu_release_timeout_seconds": 20,
    "cpu_threads_per_job": 4,
    "require_two_t4": True,
    "speaker_mode": "auto",  # auto: user WAV if supplied, otherwise male defaults. reference: cloning models only. default: male defaults.
    "reference_audio": None,    # e.g. /kaggle/input/my-voice/reference.wav
    "reference_text": None,     # Exact transcript; use a clean 1-8 second reference.
    "include_reference_in_zip": True,
    "preview_players": 6,
    "quality_checks": True,
    "asr_quality_check": True, # CPU-only word comparison after synthesis; it adds runtime, not GPU memory.
    "asr_model": "base.en",    # Use small.en for a stronger English check, or a local CTranslate2 model path.
    "asr_language": "en",
    "quality_timeout_seconds": 7200,
}

# Three distinct passages, with no repeated sentences. For other lengths,
# supply your own TEXT_CASES rather than cutting or cycling a passage.
WORD_COUNTS = [10, 100, 500]
SAMPLE_PASSAGES = {
    "10": "The final decision required patience, clear evidence, and careful judgment.",
    "100": "Before the library opened, an archivist discovered a folded letter behind a damaged map. Its author described a bridge that had disappeared from the city decades earlier. The location seemed familiar, but the surviving records offered conflicting dates. She compared photographs, examined a survey, and followed the river on foot. Beneath an ordinary railway crossing, she found the weathered old stone bridge foundations. By afternoon, the letter had become part of a much larger story about the neighborhood. The discovery changed no grand theory, yet it restored a small piece of history that residents could finally recognize as their own.",
    "500": "The field station stood above a narrow valley, where pine trees sheltered the road and clouds often concealed the distant peaks. Each morning, a small research team checked the instruments before carrying fresh batteries and supplies to the ridge. Their project concerned a faint signal that appeared in recordings during cold weather. Previous measurements suggested a pattern, but the evidence remained uneven. Some nights produced clear observations; others yielded only noise. The team had arrived with competing explanations and a shared commitment to patient work. They expected the investigation to continue for several weeks, perhaps longer if conditions became difficult.\n\nOn the third evening, an unexpected result interrupted their routine. A sensor beside the eastern window registered a sudden, sharp increase while the equipment outside remained completely stable. Mira, the lead engineer, examined the cables and repeated the calibration. Daniel reviewed the software, searching for changes in timing or numerical precision. A younger colleague suggested that the window itself might be influencing the reading. Everyone agreed to test the possibility. They moved the sensor, exchanged two amplifiers, and recorded another sequence. The signal returned at almost the same moment, leaving the explanation unresolved and making the next experiment more important.\n\nThe following afternoon brought heavy rain and a brief regional power failure. Inside the station, conversation slowed as the team waited for the backup supply to stabilize. Then Mira noticed a quiet mechanical vibration beneath the floor. A ventilation fan had restarted, and its bearings produced a faint rhythm. She asked Daniel to mark the event in the log. When they compared the timestamps, the vibration aligned with the unexplained signal. They switched the fan off, waited, and repeated the measurement. The disturbance disappeared. A simple maintenance problem had shaped an observation that initially seemed to demand a sophisticated explanation.\n\nFinding the cause changed the project without ending it. The earlier records still contained useful information, although several sessions now required particularly careful review. The researchers revised their analysis and documented every exclusion. They also invited a technician from the nearby town to inspect the remaining equipment. His practical suggestions improved the mounting brackets and reduced electrical interference. During dinner, the discussion turned toward the habits that make collaborative work reliable. Clear notes, independent checks, and a willingness to revise assumptions had mattered as much as technical expertise. The team began planning another series with quieter equipment and stronger safeguards.\n\nBy the end of the visit, the valley had emerged from its clouds. The road remained wet, but sunlight reached the station for the first time in several days. Mira prepared a report that described the original question, the misleading observation, and the correction. She included the unsuccessful tests because they explained how the conclusion had developed. Before leaving, the team stored copies of the recordings and checked the inventory together. Their findings were modest, precise, and useful. As the vehicle descended toward town, the station grew smaller behind them, carrying the promise of further questions and more careful measurements."
}
TEXT_CASES = [
    {"name": f"w{count:03d}", "text": SAMPLE_PASSAGES[str(count)]}
    for count in WORD_COUNTS
]
# Example replacement:
# TEXT_CASES = [{"name": "my_passage", "text": Path("/kaggle/input/my-text/passage.txt").read_text()}]

# Each profile is a separate experiment, repeated n times. Toggle enabled,
# change parameters, or add profiles. Names appear in filenames and reports.
# Parameters use each script's own CLI destination names (saved --help files explain them).
MODEL_PROFILES = {
    "kokoro": [
        {"name": "male_fenrir", "enabled": True, "parameters": {"voice": "am_fenrir", "speed": 1.0, "chunk_words": 45, "normalize_peak": True}},
        {"name": "male_george", "enabled": False, "parameters": {"voice": "bm_george", "language": "b", "speed": 1.0, "chunk_words": 45, "normalize_peak": True}},
    ],
    "f5": [
        {"name": "reference_quality", "enabled": True, "parameters": {"speed": 1.0, "steps": 64, "cfg_strength": 2.0, "sway": -1.0, "ode_method": "euler", "max_chunk_seconds": 6.0}},
        {"name": "reference_fast", "enabled": False, "parameters": {"speed": 1.2, "steps": 32, "cfg_strength": 2.0, "sway": -1.0, "max_chunk_seconds": 6.0}},
    ],
    "emotivoice": [
        {"name": "male9017_neutral", "enabled": True, "parameters": {"speaker": "9017", "prompt": "Neutral", "speed": 1.0, "pitch_predictor_scale": 1.0, "energy_predictor_scale": 1.0}},
        {"name": "male9017_sad", "enabled": False, "parameters": {"speaker": "9017", "prompt": "Sad", "speed": 1.0, "pitch_predictor_scale": 1.0, "energy_predictor_scale": 1.0}},
    ],
    "styletts2": [
        {"name": "reference_quality", "enabled": True, "parameters": {"speed": 1.0, "alpha": 0.1, "beta": 0.3, "diffusion_steps": 10, "embedding_scale": 1.0, "pitch_scale": 1.0, "max_chunk_words": 45}},
        {"name": "reference_expressive", "enabled": False, "parameters": {"speed": 1.1, "alpha": 0.5, "beta": 0.9, "diffusion_steps": 7, "embedding_scale": 1.2, "pitch_scale": 1.05, "max_chunk_words": 35}},
    ],
    "omnivoice": [
        {"name": "male_or_user_reference", "enabled": True, "reference": "auto", "parameters": {"instruct": "male, middle-aged, low pitch, british accent", "speed": 1.0, "num_step": 64, "guidance_scale": 2.0, "dtype": "float16", "max_chunk_seconds": 12.0}},
        {"name": "clone_reference", "enabled": False, "reference": True, "parameters": {"speed": 1.0, "num_step": 64, "guidance_scale": 2.0, "dtype": "float16", "max_chunk_seconds": 12.0}},
    ],
    "supertonic": [
        {"name": "m2_deep_serious", "enabled": True, "parameters": {"voice": "M2", "lang": "en", "speed": 1.0, "steps": 16, "max_chunk_chars": 360}},
        {"name": "f1_fast", "enabled": False, "parameters": {"voice": "F1", "lang": "en", "speed": 1.2, "steps": 12, "max_chunk_chars": 300}},
    ],
}
print("Run:", RUN_ID)
print("Cases:", [(c["name"], len(c["text"].split())) for c in TEXT_CASES])
print("Enabled profiles:", {m: [p["name"] for p in profiles if p.get("enabled", True)] for m, profiles in MODEL_PROFILES.items()})


## Explore and export the complete voice/option catalogs

Run the configuration cell, then this cell to inspect catalogs without installing
TTS models. Set `CATALOG_MODEL`, `CATALOG_QUERY`, `CATALOG_GENDER`, and
`CATALOG_LIMIT` inside the cell; set `OPTION_MODEL` to see another adapter's
actual CLI defaults. Use `CATALOG_LIMIT=None` for every matching speaker.
All rows, model metadata, option CSVs, and this reference guide are saved under
`catalog/` and included in the final ZIP. To generate a selected voice, copy its
selection key into a model profile; catalog browsing does not add jobs.


In [ ]:
# No weight downloads or GPU allocations. Run the experiment configuration cell first.
import csv as catalog_csv
import html as catalog_markup
import json

VOICE_CATALOG = json.loads('{"kokoro": [{"label": "af_alloy", "voice": "af_alloy", "name": "af_alloy", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_alloy.pt"}, {"label": "af_aoede", "voice": "af_aoede", "name": "af_aoede", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_aoede.pt"}, {"label": "af_bella", "voice": "af_bella", "name": "af_bella", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_bella.pt"}, {"label": "af_heart", "voice": "af_heart", "name": "af_heart", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_heart.pt"}, {"label": "af_jessica", "voice": "af_jessica", "name": "af_jessica", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_jessica.pt"}, {"label": "af_kore", "voice": "af_kore", "name": "af_kore", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_kore.pt"}, {"label": "af_nicole", "voice": "af_nicole", "name": "af_nicole", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_nicole.pt"}, {"label": "af_nova", "voice": "af_nova", "name": "af_nova", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_nova.pt"}, {"label": "af_river", "voice": "af_river", "name": "af_river", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_river.pt"}, {"label": "af_sarah", "voice": "af_sarah", "name": "af_sarah", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_sarah.pt"}, {"label": "af_sky", "voice": "af_sky", "name": "af_sky", "language_code": "a", "language": "US English", "gender": "F", "file": "voices/af_sky.pt"}, {"label": "am_adam", "voice": "am_adam", "name": "am_adam", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_adam.pt"}, {"label": "am_echo", "voice": "am_echo", "name": "am_echo", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_echo.pt"}, {"label": "am_eric", "voice": "am_eric", "name": "am_eric", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_eric.pt"}, {"label": "am_fenrir", "voice": "am_fenrir", "name": "am_fenrir", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_fenrir.pt"}, {"label": "am_liam", "voice": "am_liam", "name": "am_liam", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_liam.pt"}, {"label": "am_michael", "voice": "am_michael", "name": "am_michael", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_michael.pt"}, {"label": "am_onyx", "voice": "am_onyx", "name": "am_onyx", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_onyx.pt"}, {"label": "am_puck", "voice": "am_puck", "name": "am_puck", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_puck.pt"}, {"label": "am_santa", "voice": "am_santa", "name": "am_santa", "language_code": "a", "language": "US English", "gender": "M", "file": "voices/am_santa.pt"}, {"label": "bf_alice", "voice": "bf_alice", "name": "bf_alice", "language_code": "b", "language": "British English", "gender": "F", "file": "voices/bf_alice.pt"}, {"label": "bf_emma", "voice": "bf_emma", "name": "bf_emma", "language_code": "b", "language": "British English", "gender": "F", "file": "voices/bf_emma.pt"}, {"label": "bf_isabella", "voice": "bf_isabella", "name": "bf_isabella", "language_code": "b", "language": "British English", "gender": "F", "file": "voices/bf_isabella.pt"}, {"label": "bf_lily", "voice": "bf_lily", "name": "bf_lily", "language_code": "b", "language": "British English", "gender": "F", "file": "voices/bf_lily.pt"}, {"label": "bm_daniel", "voice": "bm_daniel", "name": "bm_daniel", "language_code": "b", "language": "British English", "gender": "M", "file": "voices/bm_daniel.pt"}, {"label": "bm_fable", "voice": "bm_fable", "name": "bm_fable", "language_code": "b", "language": "British English", "gender": "M", "file": "voices/bm_fable.pt"}, {"label": "bm_george", "voice": "bm_george", "name": "bm_george", "language_code": "b", "language": "British English", "gender": "M", "file": "voices/bm_george.pt"}, {"label": "bm_lewis", "voice": "bm_lewis", "name": "bm_lewis", "language_code": "b", "language": "British English", "gender": "M", "file": "voices/bm_lewis.pt"}, {"label": "ef_dora", "voice": "ef_dora", "name": "ef_dora", "language_code": "e", "language": "Spanish", "gender": "F", "file": "voices/ef_dora.pt"}, {"label": "em_alex", "voice": "em_alex", "name": "em_alex", "language_code": "e", "language": "Spanish", "gender": "M", "file": "voices/em_alex.pt"}, {"label": "em_santa", "voice": "em_santa", "name": "em_santa", "language_code": "e", "language": "Spanish", "gender": "M", "file": "voices/em_santa.pt"}, {"label": "ff_siwis", "voice": "ff_siwis", "name": "ff_siwis", "language_code": "f", "language": "French", "gender": "F", "file": "voices/ff_siwis.pt"}, {"label": "hf_alpha", "voice": "hf_alpha", "name": "hf_alpha", "language_code": "h", "language": "Hindi", "gender": "F", "file": "voices/hf_alpha.pt"}, {"label": "hf_beta", "voice": "hf_beta", "name": "hf_beta", "language_code": "h", "language": "Hindi", "gender": "F", "file": "voices/hf_beta.pt"}, {"label": "hm_omega", "voice": "hm_omega", "name": "hm_omega", "language_code": "h", "language": "Hindi", "gender": "M", "file": "voices/hm_omega.pt"}, {"label": "hm_psi", "voice": "hm_psi", "name": "hm_psi", "language_code": "h", "language": "Hindi", "gender": "M", "file": "voices/hm_psi.pt"}, {"label": "if_sara", "voice": "if_sara", "name": "if_sara", "language_code": "i", "language": "Italian", "gender": "F", "file": "voices/if_sara.pt"}, {"label": "im_nicola", "voice": "im_nicola", "name": "im_nicola", "language_code": "i", "language": "Italian", "gender": "M", "file": "voices/im_nicola.pt"}, {"label": "jf_alpha", "voice": "jf_alpha", "name": "jf_alpha", "language_code": "j", "language": "Japanese", "gender": "F", "file": "voices/jf_alpha.pt"}, {"label": "jf_gongitsune", "voice": "jf_gongitsune", "name": "jf_gongitsune", "language_code": "j", "language": "Japanese", "gender": "F", "file": "voices/jf_gongitsune.pt"}, {"label": "jf_nezumi", "voice": "jf_nezumi", "name": "jf_nezumi", "language_code": "j", "language": "Japanese", "gender": "F", "file": "voices/jf_nezumi.pt"}, {"label": "jf_tebukuro", "voice": "jf_tebukuro", "name": "jf_tebukuro", "language_code": "j", "language": "Japanese", "gender": "F", "file": "voices/jf_tebukuro.pt"}, {"label": "jm_kumo", "voice": "jm_kumo", "name": "jm_kumo", "language_code": "j", "language": "Japanese", "gender": "M", "file": "voices/jm_kumo.pt"}, {"label": "pf_dora", "voice": "pf_dora", "name": "pf_dora", "language_code": "p", "language": "Brazilian Portuguese", "gender": "F", "file": "voices/pf_dora.pt"}, {"label": "pm_alex", "voice": "pm_alex", "name": "pm_alex", "language_code": "p", "language": "Brazilian Portuguese", "gender": "M", "file": "voices/pm_alex.pt"}, {"label": "pm_santa", "voice": "pm_santa", "name": "pm_santa", "language_code": "p", "language": "Brazilian Portuguese", "gender": "M", "file": "voices/pm_santa.pt"}, {"label": "zf_xiaobei", "voice": "zf_xiaobei", "name": "zf_xiaobei", "language_code": "z", "language": "Mandarin Chinese", "gender": "F", "file": "voices/zf_xiaobei.pt"}, {"label": "zf_xiaoni", "voice": "zf_xiaoni", "name": "zf_xiaoni", "language_code": "z", "language": "Mandarin Chinese", "gender": "F", "file": "voices/zf_xiaoni.pt"}, {"label": "zf_xiaoxiao", "voice": "zf_xiaoxiao", "name": "zf_xiaoxiao", "language_code": "z", "language": "Mandarin Chinese", "gender": "F", "file": "voices/zf_xiaoxiao.pt"}, {"label": "zf_xiaoyi", "voice": "zf_xiaoyi", "name": "zf_xiaoyi", "language_code": "z", "language": "Mandarin Chinese", "gender": "F", "file": "voices/zf_xiaoyi.pt"}, {"label": "zm_yunjian", "voice": "zm_yunjian", "name": "zm_yunjian", "language_code": "z", "language": "Mandarin Chinese", "gender": "M", "file": "voices/zm_yunjian.pt"}, {"label": "zm_yunxi", "voice": "zm_yunxi", "name": "zm_yunxi", "language_code": "z", "language": "Mandarin Chinese", "gender": "M", "file": "voices/zm_yunxi.pt"}, {"label": "zm_yunxia", "voice": "zm_yunxia", "name": "zm_yunxia", "language_code": "z", "language": "Mandarin Chinese", "gender": "M", "file": "voices/zm_yunxia.pt"}, {"label": "zm_yunyang", "voice": "zm_yunyang", "name": "zm_yunyang", "language_code": "z", "language": "Mandarin Chinese", "gender": "M", "file": "voices/zm_yunyang.pt"}], "emotivoice": [{"index": 0, "label": "8051", "name": "Maria Kasper", "gender": "F", "description": "Clear, soothing, expressive"}, {"index": 1, "label": "11614", "name": "Sylviamb", "gender": "F", "description": "Crisp, melodic, captivating"}, {"index": 2, "label": "9017", "name": "John Van Stan", "gender": "M", "description": "Rich, resonant, engaging"}, {"index": 3, "label": "6097", "name": "Phil Benson", "gender": "M", "description": "Smooth, mellow, resonant."}, {"index": 4, "label": "6671", "name": "Tony Oliva", "gender": "M", "description": "Smooth, charismatic, dynamic"}, {"index": 5, "label": "6670", "name": "Mike Pelton", "gender": "M", "description": ""}, {"index": 6, "label": "9136", "name": "Helen Taylor", "gender": "F", "description": ""}, {"index": 7, "label": "11697", "name": "Celine Major", "gender": "F", "description": ""}, {"index": 8, "label": "92", "name": "Cori Samuel", "gender": "F", "description": "Lively, expressive, energetic"}, {"index": 9, "label": "12787", "name": "LikeManyWaters", "gender": "F", "description": ""}, {"index": 10, "label": "1006", "name": "Marta Kornowska", "gender": "F", "description": ""}, {"index": 11, "label": "1012", "name": "Lizzie Oldfather", "gender": "F", "description": ""}, {"index": 12, "label": "1018", "name": "JimmyLogan", "gender": "M", "description": ""}, {"index": 13, "label": "101", "name": "paulino", "gender": "M", "description": ""}, {"index": 14, "label": "1025", "name": "rdmagpie", "gender": "M", "description": ""}, {"index": 15, "label": "1027", "name": "Brooks Jensen", "gender": "M", "description": ""}, {"index": 16, "label": "1028", "name": "Tim Lundeen", "gender": "M", "description": ""}, {"index": 17, "label": "102", "name": "Linda Leu", "gender": "F", "description": ""}, {"index": 18, "label": "1034", "name": "Kevin O\'Coin", "gender": "M", "description": ""}, {"index": 19, "label": "103", "name": "Karen Savage", "gender": "F", "description": ""}, {"index": 20, "label": "1040", "name": "John Garvin", "gender": "M", "description": ""}, {"index": 21, "label": "1046", "name": "durnburr", "gender": "M", "description": ""}, {"index": 22, "label": "1049", "name": "Sam Fold", "gender": "M", "description": ""}, {"index": 23, "label": "104", "name": "Laura M.D.", "gender": "F", "description": ""}, {"index": 24, "label": "1050", "name": "entada", "gender": "F", "description": ""}, {"index": 25, "label": "1051", "name": "E. Moulton", "gender": "F", "description": ""}, {"index": 26, "label": "1052", "name": "Kathy Jacobs", "gender": "F", "description": ""}, {"index": 27, "label": "1058", "name": "James Tiley", "gender": "M", "description": ""}, {"index": 28, "label": "1061", "name": "Missie", "gender": "F", "description": ""}, {"index": 29, "label": "1065", "name": "Justin Brett", "gender": "M", "description": ""}, {"index": 30, "label": "1066", "name": "Laurie Anne Walden", "gender": "F", "description": ""}, {"index": 31, "label": "1069", "name": "Dawn", "gender": "F", "description": ""}, {"index": 32, "label": "1079", "name": "Mary aka Breadchick", "gender": "F", "description": ""}, {"index": 33, "label": "107", "name": "John Greenman", "gender": "M", "description": ""}, {"index": 34, "label": "1081", "name": "Fracture", "gender": "M", "description": ""}, {"index": 35, "label": "1084", "name": "Nichole Karl", "gender": "F", "description": ""}, {"index": 36, "label": "1085", "name": "hefyd", "gender": "M", "description": ""}, {"index": 37, "label": "1088", "name": "Christabel", "gender": "F", "description": ""}, {"index": 38, "label": "1092", "name": "Maria", "gender": "F", "description": ""}, {"index": 39, "label": "1093", "name": "Kiki Baessell", "gender": "F", "description": ""}, {"index": 40, "label": "1094", "name": "tubeyes", "gender": "M", "description": ""}, {"index": 41, "label": "1096", "name": "Geoff Dugwyler", "gender": "M", "description": ""}, {"index": 42, "label": "1097", "name": "Euthymius", "gender": "M", "description": ""}, {"index": 43, "label": "1098", "name": "Merryb", "gender": "F", "description": ""}, {"index": 44, "label": "1107", "name": "Jason Oakley", "gender": "M", "description": ""}, {"index": 45, "label": "110", "name": "Cynthia Lyons (1946-2011)", "gender": "F", "description": ""}, {"index": 46, "label": "1110", "name": "Graeme Jolliffe", "gender": "M", "description": ""}, {"index": 47, "label": "1112", "name": "RedToby", "gender": "M", "description": ""}, {"index": 48, "label": "1116", "name": "Megan Stemm-Wade", "gender": "F", "description": ""}, {"index": 49, "label": "111", "name": "Linda Wilcox", "gender": "F", "description": ""}, {"index": 50, "label": "1121", "name": "John Lieder", "gender": "M", "description": ""}, {"index": 51, "label": "1124", "name": "Ancilla", "gender": "F", "description": ""}, {"index": 52, "label": "112", "name": "Christiane Levesque", "gender": "F", "description": ""}, {"index": 53, "label": "1132", "name": "Giles Baker", "gender": "M", "description": ""}, {"index": 54, "label": "1152", "name": "Millbeach", "gender": "F", "description": ""}, {"index": 55, "label": "1154", "name": "Larysa Jaworski", "gender": "F", "description": ""}, {"index": 56, "label": "1160", "name": "Gary Gilberd", "gender": "M", "description": ""}, {"index": 57, "label": "1161", "name": "Dominic Moore", "gender": "M", "description": ""}, {"index": 58, "label": "1165", "name": "Bob Graff", "gender": "M", "description": ""}, {"index": 59, "label": "1166", "name": "Debra Lynn", "gender": "F", "description": ""}, {"index": 60, "label": "1168", "name": "Ree", "gender": "F", "description": ""}, {"index": 61, "label": "1171", "name": "Julia Claussen", "gender": "F", "description": ""}, {"index": 62, "label": "1175", "name": "Brother Patrick", "gender": "M", "description": ""}, {"index": 63, "label": "1179", "name": "Alan Chant", "gender": "M", "description": ""}, {"index": 64, "label": "1182", "name": "Brett Condron", "gender": "M", "description": ""}, {"index": 65, "label": "1183", "name": "roolynninms", "gender": "F", "description": ""}, {"index": 66, "label": "1184", "name": "Jeremy Pavier", "gender": "M", "description": ""}, {"index": 67, "label": "1187", "name": "Paul Hansen", "gender": "M", "description": ""}, {"index": 68, "label": "118", "name": "Alex Buie", "gender": "M", "description": ""}, {"index": 69, "label": "1195", "name": "Jennette Selig", "gender": "F", "description": ""}, {"index": 70, "label": "119", "name": "Alex Patterson", "gender": "M", "description": ""}, {"index": 71, "label": "1200", "name": "hosmer_angel", "gender": "M", "description": ""}, {"index": 72, "label": "1222", "name": "Joseph Ugoretz", "gender": "M", "description": ""}, {"index": 73, "label": "1224", "name": "Heather Duncan", "gender": "F", "description": ""}, {"index": 74, "label": "1225", "name": "Chris Langston", "gender": "M", "description": ""}, {"index": 75, "label": "1226", "name": "Russ Lemker", "gender": "M", "description": ""}, {"index": 76, "label": "122", "name": "J.C.", "gender": "M", "description": ""}, {"index": 77, "label": "1230", "name": "Ian Skillen", "gender": "M", "description": ""}, {"index": 78, "label": "1235", "name": "Tim Gregory", "gender": "M", "description": ""}, {"index": 79, "label": "1239", "name": "Scott Robbins", "gender": "M", "description": ""}, {"index": 80, "label": "123", "name": "Ezwa", "gender": "F", "description": ""}, {"index": 81, "label": "1246", "name": "Sandra", "gender": "F", "description": ""}, {"index": 82, "label": "1250", "name": "Christina Boyles", "gender": "F", "description": ""}, {"index": 83, "label": "1252", "name": "Avery", "gender": "F", "description": ""}, {"index": 84, "label": "1258", "name": "Mellors", "gender": "M", "description": ""}, {"index": 85, "label": "1259", "name": "Elizabeth Klett", "gender": "F", "description": ""}, {"index": 86, "label": "125", "name": "Claire Goget", "gender": "F", "description": ""}, {"index": 87, "label": "1260", "name": "Chris Hughes", "gender": "M", "description": ""}, {"index": 88, "label": "1261", "name": "Mans Broo", "gender": "M", "description": ""}, {"index": 89, "label": "1263", "name": "Leonie Rose", "gender": "F", "description": ""}, {"index": 90, "label": "1265", "name": "Edward Elmer", "gender": "M", "description": ""}, {"index": 91, "label": "1266", "name": "Jenilee", "gender": "F", "description": ""}, {"index": 92, "label": "126", "name": "Susan Denney", "gender": "F", "description": ""}, {"index": 93, "label": "1271", "name": "Christian Pecaut", "gender": "M", "description": ""}, {"index": 94, "label": "1274", "name": "Larry Gilman", "gender": "M", "description": ""}, {"index": 95, "label": "127", "name": "John Hicken", "gender": "M", "description": ""}, {"index": 96, "label": "1280", "name": "Tim Makarios", "gender": "M", "description": ""}, {"index": 97, "label": "1283", "name": "Paul Siegel", "gender": "M", "description": ""}, {"index": 98, "label": "128", "name": "ML Cohen", "gender": "M", "description": ""}, {"index": 99, "label": "1291", "name": "Patti Brugman", "gender": "F", "description": ""}, {"index": 100, "label": "1296", "name": "Gigi Minden", "gender": "F", "description": ""}, {"index": 101, "label": "1298", "name": "Wina Hathaway", "gender": "F", "description": ""}, {"index": 102, "label": "1311", "name": "Scott D. Farquhar", "gender": "M", "description": ""}, {"index": 103, "label": "1313", "name": "Scott Sherris", "gender": "M", "description": ""}, {"index": 104, "label": "1323", "name": "Leon Mire", "gender": "M", "description": ""}, {"index": 105, "label": "1331", "name": "Adrian Praetzellis", "gender": "M", "description": ""}, {"index": 106, "label": "1334", "name": "John Schell", "gender": "M", "description": ""}, {"index": 107, "label": "1335", "name": "Clarica", "gender": "F", "description": ""}, {"index": 108, "label": "1336", "name": "Charlie Blakemore", "gender": "M", "description": ""}, {"index": 109, "label": "133", "name": "Mick", "gender": "M", "description": ""}, {"index": 110, "label": "1341", "name": "Coastalbloke", "gender": "M", "description": ""}, {"index": 111, "label": "1342", "name": "SueAnn Dozier", "gender": "F", "description": ""}, {"index": 112, "label": "1343", "name": "Laura Koskinen", "gender": "F", "description": ""}, {"index": 113, "label": "1347", "name": "Ted Nugent", "gender": "M", "description": ""}, {"index": 114, "label": "1349", "name": "John Pruden", "gender": "M", "description": ""}, {"index": 115, "label": "1353", "name": "Clarke Bell", "gender": "M", "description": ""}, {"index": 116, "label": "1355", "name": "Chris Gladis", "gender": "M", "description": ""}, {"index": 117, "label": "1363", "name": "Tammy Sanders", "gender": "F", "description": ""}, {"index": 118, "label": "1365", "name": "Joel Poortenga", "gender": "M", "description": ""}, {"index": 119, "label": "1367", "name": "Joe Brenneman", "gender": "M", "description": ""}, {"index": 120, "label": "1370", "name": "Lee Elliott", "gender": "F", "description": ""}, {"index": 121, "label": "1373", "name": "Kira Belkin", "gender": "F", "description": ""}, {"index": 122, "label": "1374", "name": "Graham Thomsen", "gender": "M", "description": ""}, {"index": 123, "label": "1379", "name": "Ken Crooker", "gender": "M", "description": ""}, {"index": 124, "label": "1384", "name": "Stephen Lamb", "gender": "M", "description": ""}, {"index": 125, "label": "1387", "name": "Scott Mather", "gender": "M", "description": ""}, {"index": 126, "label": "1392", "name": "Chris Masterson", "gender": "M", "description": ""}, {"index": 127, "label": "1403", "name": "tipaew", "gender": "M", "description": ""}, {"index": 128, "label": "1414", "name": "guava", "gender": "F", "description": ""}, {"index": 129, "label": "1417", "name": "Psuke Bariah", "gender": "F", "description": ""}, {"index": 130, "label": "1421", "name": "Madame Tusk", "gender": "F", "description": ""}, {"index": 131, "label": "1422", "name": "Yazpistachio", "gender": "F", "description": ""}, {"index": 132, "label": "1425", "name": "Jeanette Ferguson", "gender": "F", "description": ""}, {"index": 133, "label": "1430", "name": "Alok Karulkar", "gender": "M", "description": ""}, {"index": 134, "label": "1444", "name": "Ryan Mease", "gender": "M", "description": ""}, {"index": 135, "label": "1445", "name": "Michael Yard", "gender": "M", "description": ""}, {"index": 136, "label": "1447", "name": "Luigina", "gender": "F", "description": ""}, {"index": 137, "label": "1448", "name": "marevalo", "gender": "F", "description": ""}, {"index": 138, "label": "1455", "name": "webslog", "gender": "M", "description": ""}, {"index": 139, "label": "1460", "name": "E. Plein", "gender": "F", "description": ""}, {"index": 140, "label": "1463", "name": "Vivian Bush", "gender": "F", "description": ""}, {"index": 141, "label": "1469", "name": "Fr. Richard Zeile of Detroit", "gender": "M", "description": ""}, {"index": 142, "label": "1474", "name": "Jc Guan", "gender": "F", "description": ""}, {"index": 143, "label": "147", "name": "Thomas Hoover", "gender": "M", "description": ""}, {"index": 144, "label": "1485", "name": "Robert Flach", "gender": "M", "description": ""}, {"index": 145, "label": "1487", "name": "radioreader", "gender": "M", "description": ""}, {"index": 146, "label": "1492", "name": "mb", "gender": "M", "description": ""}, {"index": 147, "label": "1494", "name": "George Deprez, PhD", "gender": "M", "description": ""}, {"index": 148, "label": "1495", "name": "Glendower Jones", "gender": "M", "description": ""}, {"index": 149, "label": "1498", "name": "Lori Hebel", "gender": "F", "description": ""}, {"index": 150, "label": "1502", "name": "Ann Boyer", "gender": "F", "description": ""}, {"index": 151, "label": "1505", "name": "Mark Norman", "gender": "M", "description": ""}, {"index": 152, "label": "1509", "name": "Miranda Stinson", "gender": "F", "description": ""}, {"index": 153, "label": "150", "name": "Fox in the Stars", "gender": "F", "description": ""}, {"index": 154, "label": "1513", "name": "Simon-Peter Zak", "gender": "M", "description": ""}, {"index": 155, "label": "151", "name": "Gwen", "gender": "F", "description": ""}, {"index": 156, "label": "152", "name": "Andy Minter", "gender": "M", "description": ""}, {"index": 157, "label": "1535", "name": "Robert Scott", "gender": "M", "description": ""}, {"index": 158, "label": "1536", "name": "Marco", "gender": "M", "description": ""}, {"index": 159, "label": "153", "name": "Graham Williams", "gender": "M", "description": ""}, {"index": 160, "label": "1544", "name": "LilianaVale", "gender": "F", "description": ""}, {"index": 161, "label": "1545", "name": "AmyAG", "gender": "F", "description": ""}, {"index": 162, "label": "1547", "name": "Riddleman", "gender": "F", "description": ""}, {"index": 163, "label": "154", "name": "Robert Foster", "gender": "M", "description": ""}, {"index": 164, "label": "1552", "name": "Roger Turnau", "gender": "M", "description": ""}, {"index": 165, "label": "1553", "name": "Mim Ritty", "gender": "F", "description": ""}, {"index": 166, "label": "1556", "name": "geofred", "gender": "M", "description": ""}, {"index": 167, "label": "1559", "name": "Luke Harrison", "gender": "M", "description": ""}, {"index": 168, "label": "1563", "name": "Chandra Gioiello", "gender": "F", "description": ""}, {"index": 169, "label": "1564", "name": "Hedvig", "gender": "F", "description": ""}, {"index": 170, "label": "1566", "name": "Anna Christensen", "gender": "F", "description": ""}, {"index": 171, "label": "1569", "name": "Kristine Mackin", "gender": "F", "description": ""}, {"index": 172, "label": "1571", "name": "Bob Tassinari", "gender": "M", "description": ""}, {"index": 173, "label": "1572", "name": "Alan Clare", "gender": "M", "description": ""}, {"index": 174, "label": "1578", "name": "Lorelle Anderson", "gender": "F", "description": ""}, {"index": 175, "label": "1579", "name": "Philippa Willitts", "gender": "F", "description": ""}, {"index": 176, "label": "1593", "name": "Kristine Bekere", "gender": "F", "description": ""}, {"index": 177, "label": "1594", "name": "Jon Scott Jones", "gender": "M", "description": ""}, {"index": 178, "label": "1595", "name": "Riccardo Fasol", "gender": "M", "description": ""}, {"index": 179, "label": "159", "name": "hugh mac", "gender": "M", "description": ""}, {"index": 180, "label": "1601", "name": "Michael Yourshaw", "gender": "M", "description": ""}, {"index": 181, "label": "1603", "name": "Arouet", "gender": "M", "description": ""}, {"index": 182, "label": "1607", "name": "Claude Banta", "gender": "M", "description": ""}, {"index": 183, "label": "1614", "name": "FNH", "gender": "M", "description": ""}, {"index": 184, "label": "1618", "name": "Nicholas James Bridgewater", "gender": "M", "description": ""}, {"index": 185, "label": "161", "name": "Cyril Law, Jr.", "gender": "M", "description": ""}, {"index": 186, "label": "1621", "name": "Caliban", "gender": "M", "description": ""}, {"index": 187, "label": "1624", "name": "Daniel Shorten", "gender": "M", "description": ""}, {"index": 188, "label": "1633", "name": "Beecher", "gender": "F", "description": ""}, {"index": 189, "label": "1636", "name": "Sandra Zera", "gender": "F", "description": ""}, {"index": 190, "label": "1639", "name": "Joe Konno", "gender": "M", "description": ""}, {"index": 191, "label": "163", "name": "Andrew Miller", "gender": "M", "description": ""}, {"index": 192, "label": "1641", "name": "Rohanna", "gender": "F", "description": ""}, {"index": 193, "label": "1643", "name": "Chris Leslie-Hynan", "gender": "M", "description": ""}, {"index": 194, "label": "1645", "name": "David Shamp", "gender": "M", "description": ""}, {"index": 195, "label": "1646", "name": "Ben Cobbett", "gender": "M", "description": ""}, {"index": 196, "label": "1647", "name": "Rich Meyers", "gender": "M", "description": ""}, {"index": 197, "label": "1648", "name": "Accent", "gender": "F", "description": ""}, {"index": 198, "label": "1653", "name": "Carmina Sansone", "gender": "F", "description": ""}, {"index": 199, "label": "1664", "name": "Shauna M", "gender": "F", "description": ""}, {"index": 200, "label": "1665", "name": "Jessica AC Snyder", "gender": "F", "description": ""}, {"index": 201, "label": "1668", "name": "stepheather", "gender": "F", "description": ""}, {"index": 202, "label": "1674", "name": "Jo", "gender": "F", "description": ""}, {"index": 203, "label": "1679", "name": "Polly", "gender": "F", "description": ""}, {"index": 204, "label": "167", "name": "Peter Yearsley", "gender": "M", "description": ""}, {"index": 205, "label": "1680", "name": "intothelight", "gender": "F", "description": ""}, {"index": 206, "label": "1681", "name": "islajane", "gender": "F", "description": ""}, {"index": 207, "label": "1685", "name": "Jonathan Horniblow", "gender": "M", "description": ""}, {"index": 208, "label": "168", "name": "Chris Goringe", "gender": "M", "description": ""}, {"index": 209, "label": "1690", "name": "Anne-Marie", "gender": "F", "description": ""}, {"index": 210, "label": "1691", "name": "Classicsfan", "gender": "F", "description": ""}, {"index": 211, "label": "1693", "name": "4Cullen", "gender": "F", "description": ""}, {"index": 212, "label": "1695", "name": "Steph", "gender": "F", "description": ""}, {"index": 213, "label": "1696", "name": "Darcywil", "gender": "F", "description": ""}, {"index": 214, "label": "1699", "name": "Gavin Smith", "gender": "M", "description": ""}, {"index": 215, "label": "16", "name": "Alys AtteWater", "gender": "F", "description": ""}, {"index": 216, "label": "1704", "name": "JB", "gender": "F", "description": ""}, {"index": 217, "label": "1705", "name": "tittletattle", "gender": "F", "description": ""}, {"index": 218, "label": "1708", "name": "Alaaious", "gender": "M", "description": ""}, {"index": 219, "label": "1710", "name": "Gilly", "gender": "F", "description": ""}, {"index": 220, "label": "1714", "name": "lauralee", "gender": "F", "description": ""}, {"index": 221, "label": "1717", "name": "PJ", "gender": "F", "description": ""}, {"index": 222, "label": "1721", "name": "Linnea", "gender": "F", "description": ""}, {"index": 223, "label": "1726", "name": "janeite", "gender": "F", "description": ""}, {"index": 224, "label": "1731", "name": "Dani", "gender": "F", "description": ""}, {"index": 225, "label": "1733", "name": "Mira Cheskis", "gender": "F", "description": ""}, {"index": 226, "label": "1736", "name": "LC", "gender": "F", "description": ""}, {"index": 227, "label": "1737", "name": "Erin Hastings", "gender": "F", "description": ""}, {"index": 228, "label": "173", "name": "vlooi", "gender": "F", "description": ""}, {"index": 229, "label": "1743", "name": "Bryan Ness", "gender": "M", "description": ""}, {"index": 230, "label": "1746", "name": "Theo Bacher", "gender": "M", "description": ""}, {"index": 231, "label": "1748", "name": "Brad Powers", "gender": "M", "description": ""}, {"index": 232, "label": "1750", "name": "Lorie Heinrichs", "gender": "F", "description": ""}, {"index": 233, "label": "1754", "name": "Joan Freeman", "gender": "F", "description": ""}, {"index": 234, "label": "1756", "name": "Tamara Hamilton", "gender": "F", "description": ""}, {"index": 235, "label": "1757", "name": "cricket", "gender": "F", "description": ""}, {"index": 236, "label": "175", "name": "Meredith Hughes", "gender": "F", "description": ""}, {"index": 237, "label": "1760", "name": "Matthew Howell", "gender": "F", "description": ""}, {"index": 238, "label": "1765", "name": "Kelly Elizabeth", "gender": "F", "description": ""}, {"index": 239, "label": "1767", "name": "Cori Dean", "gender": "F", "description": ""}, {"index": 240, "label": "1769", "name": "ej", "gender": "M", "description": ""}, {"index": 241, "label": "176", "name": "Vinny Bove", "gender": "M", "description": ""}, {"index": 242, "label": "1772", "name": "David A. Stokely", "gender": "M", "description": ""}, {"index": 243, "label": "1773", "name": "Eliza Horne", "gender": "F", "description": ""}, {"index": 244, "label": "1777", "name": "Professor Chronotis", "gender": "M", "description": ""}, {"index": 245, "label": "1779", "name": "Cynthia Zocca", "gender": "F", "description": ""}, {"index": 246, "label": "177", "name": "Kymm Zuckert", "gender": "F", "description": ""}, {"index": 247, "label": "1780", "name": "Micah", "gender": "M", "description": ""}, {"index": 248, "label": "1784", "name": "grovejade", "gender": "F", "description": ""}, {"index": 249, "label": "1789", "name": "Vin Reilly", "gender": "M", "description": ""}, {"index": 250, "label": "1795", "name": "Muhammad Mussnoon", "gender": "M", "description": ""}, {"index": 251, "label": "1801", "name": "Antonio", "gender": "M", "description": ""}, {"index": 252, "label": "1804", "name": "Marie Manis", "gender": "F", "description": ""}, {"index": 253, "label": "1809", "name": "cucciasv", "gender": "F", "description": ""}, {"index": 254, "label": "1811", "name": "Eric Ray", "gender": "M", "description": ""}, {"index": 255, "label": "1813", "name": "tesoro007", "gender": "F", "description": ""}, {"index": 256, "label": "1815", "name": "Aringguth", "gender": "M", "description": ""}, {"index": 257, "label": "1819", "name": "Shannon", "gender": "F", "description": ""}, {"index": 258, "label": "1825", "name": "srshel", "gender": "F", "description": ""}, {"index": 259, "label": "1826", "name": "Jacob Miller", "gender": "M", "description": ""}, {"index": 260, "label": "1828", "name": "James Gladwin", "gender": "M", "description": ""}, {"index": 261, "label": "1841", "name": "Laura Caldwell", "gender": "F", "description": ""}, {"index": 262, "label": "1844", "name": "noonday", "gender": "M", "description": ""}, {"index": 263, "label": "1846", "name": "valikojohn", "gender": "M", "description": ""}, {"index": 264, "label": "1849", "name": "Kelly Dougherty", "gender": "M", "description": ""}, {"index": 265, "label": "1851", "name": "Kehinde", "gender": "F", "description": ""}, {"index": 266, "label": "1859", "name": "Jan Baxter", "gender": "F", "description": ""}, {"index": 267, "label": "1863", "name": "Ania", "gender": "F", "description": ""}, {"index": 268, "label": "1867", "name": "Rowdy Delaney", "gender": "M", "description": ""}, {"index": 269, "label": "1868", "name": "Graham Redman", "gender": "M", "description": ""}, {"index": 270, "label": "1870", "name": "Stuart Bell", "gender": "M", "description": ""}, {"index": 271, "label": "1874", "name": "Ernst Schnell", "gender": "M", "description": ""}, {"index": 272, "label": "1878", "name": "BLRossow", "gender": "M", "description": ""}, {"index": 273, "label": "1885", "name": "inkwelldragon", "gender": "F", "description": ""}, {"index": 274, "label": "188", "name": "Mary Anderson", "gender": "F", "description": ""}, {"index": 275, "label": "1898", "name": "Jennifer", "gender": "F", "description": ""}, {"index": 276, "label": "1901", "name": "Allyson Hester", "gender": "F", "description": ""}, {"index": 277, "label": "1903", "name": "Michael Thomas Robinson", "gender": "M", "description": ""}, {"index": 278, "label": "1913", "name": "Geoff Cowgill", "gender": "M", "description": ""}, {"index": 279, "label": "1920", "name": "Annika Feilbach", "gender": "F", "description": ""}, {"index": 280, "label": "1923", "name": "Maire Rhode", "gender": "F", "description": ""}, {"index": 281, "label": "1924", "name": "Andrew Drinkwater", "gender": "M", "description": ""}, {"index": 282, "label": "1926", "name": "Nikki Sullivan", "gender": "F", "description": ""}, {"index": 283, "label": "1931", "name": "poormedea", "gender": "F", "description": ""}, {"index": 284, "label": "1938", "name": "icyjumbo (1964-2010)", "gender": "M", "description": ""}, {"index": 285, "label": "1943", "name": "Corun", "gender": "M", "description": ""}, {"index": 286, "label": "1944", "name": "Carolyn Frances", "gender": "F", "description": ""}, {"index": 287, "label": "1963", "name": "Belinda Brown", "gender": "F", "description": ""}, {"index": 288, "label": "1968", "name": "lizzyblack", "gender": "F", "description": ""}, {"index": 289, "label": "196", "name": "Stewart Wills", "gender": "M", "description": ""}, {"index": 290, "label": "1970", "name": "Dawn Larsen", "gender": "F", "description": ""}, {"index": 291, "label": "1974", "name": "Katie Baynes", "gender": "F", "description": ""}, {"index": 292, "label": "1977", "name": "Jennie Hughes", "gender": "F", "description": ""}, {"index": 293, "label": "1985", "name": "Jonny Lee", "gender": "M", "description": ""}, {"index": 294, "label": "1987", "name": "Michael Macedonia", "gender": "M", "description": ""}, {"index": 295, "label": "1989", "name": "Sergio Baldelli", "gender": "M", "description": ""}, {"index": 296, "label": "198", "name": "Heather Barnett", "gender": "F", "description": ""}, {"index": 297, "label": "1992", "name": "Michelle White", "gender": "F", "description": ""}, {"index": 298, "label": "199", "name": "Maria Morabe", "gender": "F", "description": ""}, {"index": 299, "label": "19", "name": "Kara Shallenberg", "gender": "F", "description": ""}, {"index": 300, "label": "2001", "name": "Phillip David", "gender": "M", "description": ""}, {"index": 301, "label": "2002", "name": "Larry Maddocks", "gender": "M", "description": ""}, {"index": 302, "label": "2003", "name": "The Penang Lawyer", "gender": "M", "description": ""}, {"index": 303, "label": "2004", "name": "Kim S", "gender": "F", "description": ""}, {"index": 304, "label": "2007", "name": "Sheila Morton", "gender": "F", "description": ""}, {"index": 305, "label": "200", "name": "Maureen S. O\'Brien", "gender": "F", "description": ""}, {"index": 306, "label": "2010", "name": "Julie Bynum", "gender": "F", "description": ""}, {"index": 307, "label": "2012", "name": "jburby", "gender": "M", "description": ""}, {"index": 308, "label": "2013", "name": "Mark", "gender": "M", "description": ""}, {"index": 309, "label": "201", "name": "Joplin James", "gender": "M", "description": ""}, {"index": 310, "label": "2021", "name": "Keri Ford", "gender": "M", "description": ""}, {"index": 311, "label": "2026", "name": "Mil Nicholson", "gender": "F", "description": ""}, {"index": 312, "label": "202", "name": "Geetu Melwani", "gender": "F", "description": ""}, {"index": 313, "label": "2039", "name": "Anton", "gender": "M", "description": ""}, {"index": 314, "label": "203", "name": "Marian Brown", "gender": "F", "description": ""}, {"index": 315, "label": "2042", "name": "Charlene V. Smith", "gender": "F", "description": ""}, {"index": 316, "label": "2046", "name": "kyleti", "gender": "M", "description": ""}, {"index": 317, "label": "204", "name": "Mark F. Smith", "gender": "M", "description": ""}, {"index": 318, "label": "2050", "name": "Xe Sands", "gender": "F", "description": ""}, {"index": 319, "label": "2051", "name": "Grant Petersen", "gender": "M", "description": ""}, {"index": 320, "label": "205", "name": "Esther", "gender": "F", "description": ""}, {"index": 321, "label": "2060", "name": "Julie Pandya", "gender": "F", "description": ""}, {"index": 322, "label": "2061", "name": "Jodi Krangle", "gender": "F", "description": ""}, {"index": 323, "label": "2062", "name": "Mindy H", "gender": "F", "description": ""}, {"index": 324, "label": "2063", "name": "hearhis", "gender": "M", "description": ""}, {"index": 325, "label": "2067", "name": "Nick Gisburne", "gender": "M", "description": ""}, {"index": 326, "label": "2068", "name": "Priya, India", "gender": "F", "description": ""}, {"index": 327, "label": "207", "name": "Kevin McAsh", "gender": "M", "description": ""}, {"index": 328, "label": "2089", "name": "Martina", "gender": "F", "description": ""}, {"index": 329, "label": "208", "name": "Andrea L", "gender": "F", "description": ""}, {"index": 330, "label": "2090", "name": "Melissa", "gender": "F", "description": ""}, {"index": 331, "label": "2092", "name": "Elaine Hamby", "gender": "F", "description": ""}, {"index": 332, "label": "2096", "name": "Nick Marsh", "gender": "M", "description": ""}, {"index": 333, "label": "209", "name": "Moira Fogarty", "gender": "F", "description": ""}, {"index": 334, "label": "20", "name": "Gesine", "gender": "F", "description": ""}, {"index": 335, "label": "2100", "name": "Katherine Holt", "gender": "F", "description": ""}, {"index": 336, "label": "2104", "name": "R. S. Steinberg", "gender": "M", "description": ""}, {"index": 337, "label": "210", "name": "Aldark", "gender": "M", "description": ""}, {"index": 338, "label": "2110", "name": "Aaron Elliott", "gender": "M", "description": ""}, {"index": 339, "label": "2113", "name": "Andrew Vidal", "gender": "M", "description": ""}, {"index": 340, "label": "211", "name": "shanda_w", "gender": "F", "description": ""}, {"index": 341, "label": "2122", "name": "Kenneth R. Morefield", "gender": "M", "description": ""}, {"index": 342, "label": "2127", "name": "wrongshore", "gender": "M", "description": ""}, {"index": 343, "label": "2133", "name": "Mat Messerschmidt", "gender": "M", "description": ""}, {"index": 344, "label": "2136", "name": "Great Plains", "gender": "M", "description": ""}, {"index": 345, "label": "2140", "name": "Ralph Snelson", "gender": "M", "description": ""}, {"index": 346, "label": "2143", "name": "Cat Schirf", "gender": "F", "description": ""}, {"index": 347, "label": "2148", "name": "BethAnne", "gender": "F", "description": ""}, {"index": 348, "label": "2149", "name": "Mark Penfold", "gender": "M", "description": ""}, {"index": 349, "label": "2152", "name": "redabrus", "gender": "F", "description": ""}, {"index": 350, "label": "2156", "name": "Roger Melin", "gender": "M", "description": ""}, {"index": 351, "label": "2159", "name": "Matthew Westra", "gender": "M", "description": ""}, {"index": 352, "label": "215", "name": "Alice Elizabeth Still", "gender": "F", "description": ""}, {"index": 353, "label": "2167", "name": "spiderman0521", "gender": "M", "description": ""}, {"index": 354, "label": "2182", "name": "Susan Umpleby", "gender": "F", "description": ""}, {"index": 355, "label": "2185", "name": "Jonathan Feldman", "gender": "M", "description": ""}, {"index": 356, "label": "218", "name": "Kelly Clear", "gender": "M", "description": ""}, {"index": 357, "label": "2194", "name": "RobbieRogers", "gender": "F", "description": ""}, {"index": 358, "label": "2195", "name": "Joe Earley", "gender": "M", "description": ""}, {"index": 359, "label": "2196", "name": "Andrea Fiore", "gender": "F", "description": ""}, {"index": 360, "label": "2198", "name": "Clive Catterall", "gender": "M", "description": ""}, {"index": 361, "label": "2208", "name": "Alan Brown", "gender": "M", "description": ""}, {"index": 362, "label": "2229", "name": "Pete Williams, Pittsburgh, PA", "gender": "M", "description": ""}, {"index": 363, "label": "2230", "name": "Isosceles", "gender": "F", "description": ""}, {"index": 364, "label": "2234", "name": "Lars Rolander", "gender": "M", "description": ""}, {"index": 365, "label": "2237", "name": "Chloey Winters", "gender": "F", "description": ""}, {"index": 366, "label": "2238", "name": "Will Larson", "gender": "M", "description": ""}, {"index": 367, "label": "2240", "name": "Ralph Volpi", "gender": "M", "description": ""}, {"index": 368, "label": "2246", "name": "RaySee", "gender": "M", "description": ""}, {"index": 369, "label": "224", "name": "Caitlin Kelly", "gender": "F", "description": ""}, {"index": 370, "label": "2254", "name": "Heidi Preuss", "gender": "F", "description": ""}, {"index": 371, "label": "2256", "name": "tamurile", "gender": "F", "description": ""}, {"index": 372, "label": "225", "name": "Brenda Dayne", "gender": "F", "description": ""}, {"index": 373, "label": "2262", "name": "Andy", "gender": "M", "description": ""}, {"index": 374, "label": "2269", "name": "Rhonda Federman", "gender": "F", "description": ""}, {"index": 375, "label": "226", "name": "Deb Bacon-Ziegler", "gender": "F", "description": ""}, {"index": 376, "label": "2270", "name": "Megan Kunkel", "gender": "F", "description": ""}, {"index": 377, "label": "2272", "name": "Alec Daitsman", "gender": "M", "description": ""}, {"index": 378, "label": "2273", "name": "Peter Kelleher", "gender": "M", "description": ""}, {"index": 379, "label": "2275", "name": "Lori H", "gender": "F", "description": ""}, {"index": 380, "label": "2276", "name": "Andrea", "gender": "F", "description": ""}, {"index": 381, "label": "2279", "name": "Gilles Lehoux", "gender": "M", "description": ""}, {"index": 382, "label": "227", "name": "Sarah Key-DeLyria", "gender": "F", "description": ""}, {"index": 383, "label": "2284", "name": "Zapo", "gender": "M", "description": ""}, {"index": 384, "label": "2288", "name": "Ellis Christoff", "gender": "F", "description": ""}, {"index": 385, "label": "2289", "name": "David Kleparek", "gender": "M", "description": ""}, {"index": 386, "label": "228", "name": "Arctura", "gender": "F", "description": ""}, {"index": 387, "label": "2292", "name": "Dick Durette", "gender": "M", "description": ""}, {"index": 388, "label": "2294", "name": "James Christopher", "gender": "M", "description": ""}, {"index": 389, "label": "2297", "name": "Philippa", "gender": "F", "description": ""}, {"index": 390, "label": "2299", "name": "cpalmer17", "gender": "M", "description": ""}, {"index": 391, "label": "229", "name": "carnright", "gender": "M", "description": ""}, {"index": 392, "label": "2301", "name": "Chris Jones", "gender": "F", "description": ""}, {"index": 393, "label": "2309", "name": "Wyatt", "gender": "M", "description": ""}, {"index": 394, "label": "2312", "name": "Lucy Lo Faro", "gender": "F", "description": ""}, {"index": 395, "label": "2319", "name": "Jack Farrell", "gender": "M", "description": ""}, {"index": 396, "label": "231", "name": "Sean McGaughey", "gender": "M", "description": ""}, {"index": 397, "label": "2339", "name": "skellie", "gender": "F", "description": ""}, {"index": 398, "label": "233", "name": "Steve Karafit", "gender": "M", "description": ""}, {"index": 399, "label": "2341", "name": "webround", "gender": "M", "description": ""}, {"index": 400, "label": "2346", "name": "FirstKnight", "gender": "F", "description": ""}, {"index": 401, "label": "2351", "name": "hugoceline", "gender": "F", "description": ""}, {"index": 402, "label": "2356", "name": "Paul Curran", "gender": "M", "description": ""}, {"index": 403, "label": "2361", "name": "M. J. Boyle", "gender": "F", "description": ""}, {"index": 404, "label": "2364", "name": "Anna-Maria Viola", "gender": "F", "description": ""}, {"index": 405, "label": "2368", "name": "Alex C. Telander", "gender": "M", "description": ""}, {"index": 406, "label": "2374", "name": "M.C.Y.", "gender": "F", "description": ""}, {"index": 407, "label": "2380", "name": "Jacob Cherry", "gender": "M", "description": ""}, {"index": 408, "label": "2384", "name": "Ger", "gender": "M", "description": ""}, {"index": 409, "label": "238", "name": "Maria Elmvang", "gender": "F", "description": ""}, {"index": 410, "label": "2391", "name": "treefingers", "gender": "F", "description": ""}, {"index": 411, "label": "2393", "name": "Michael Bradford", "gender": "M", "description": ""}, {"index": 412, "label": "2397", "name": "texttalker", "gender": "M", "description": ""}, {"index": 413, "label": "2401", "name": "Matt Warzel", "gender": "F", "description": ""}, {"index": 414, "label": "2404", "name": "n8evv", "gender": "M", "description": ""}, {"index": 415, "label": "2405", "name": "musil", "gender": "M", "description": ""}, {"index": 416, "label": "2407", "name": "ajmacbeth", "gender": "M", "description": ""}, {"index": 417, "label": "240", "name": "Shurtagal", "gender": "M", "description": ""}, {"index": 418, "label": "2416", "name": "Julia Albath", "gender": "F", "description": ""}, {"index": 419, "label": "242", "name": "Cagirlwithasoutherndrawl", "gender": "F", "description": ""}, {"index": 420, "label": "2436", "name": "Seth Adam Sher", "gender": "M", "description": ""}, {"index": 421, "label": "2437", "name": "Wetcoast", "gender": "M", "description": ""}, {"index": 422, "label": "243", "name": "Jennifer Stearns", "gender": "F", "description": ""}, {"index": 423, "label": "2445", "name": "musici123", "gender": "F", "description": ""}, {"index": 424, "label": "2448", "name": "David Federman", "gender": "M", "description": ""}, {"index": 425, "label": "245", "name": "Peter of Buckinghamshire England", "gender": "M", "description": ""}, {"index": 426, "label": "2485", "name": "Serin", "gender": "F", "description": ""}, {"index": 427, "label": "2487", "name": "Rachel Lintern", "gender": "F", "description": ""}, {"index": 428, "label": "2488", "name": "Lisa Wilson", "gender": "F", "description": ""}, {"index": 429, "label": "248", "name": "Becky Miller", "gender": "F", "description": ""}, {"index": 430, "label": "2491", "name": "johnb", "gender": "M", "description": ""}, {"index": 431, "label": "2494", "name": "Mark Cawley", "gender": "M", "description": ""}, {"index": 432, "label": "2496", "name": "Ben Dutton", "gender": "M", "description": ""}, {"index": 433, "label": "2498", "name": "B. Grebe", "gender": "F", "description": ""}, {"index": 434, "label": "2499", "name": "Paul Henry Tremblay", "gender": "M", "description": ""}, {"index": 435, "label": "249", "name": "pww214", "gender": "M", "description": ""}, {"index": 436, "label": "2504", "name": "Helen Elsbeth", "gender": "F", "description": ""}, {"index": 437, "label": "250", "name": "Mary Reagan", "gender": "F", "description": ""}, {"index": 438, "label": "2512", "name": "Marion", "gender": "F", "description": ""}, {"index": 439, "label": "2514", "name": "S. Young", "gender": "M", "description": ""}, {"index": 440, "label": "2517", "name": "Gayland Darnell", "gender": "M", "description": ""}, {"index": 441, "label": "2518", "name": "Rob Powell", "gender": "M", "description": ""}, {"index": 442, "label": "2522", "name": "senshisteph", "gender": "F", "description": ""}, {"index": 443, "label": "2526", "name": "Bob Gilham", "gender": "M", "description": ""}, {"index": 444, "label": "252", "name": "Rainer", "gender": "M", "description": ""}, {"index": 445, "label": "2531", "name": "Greg Weeks", "gender": "M", "description": ""}, {"index": 446, "label": "253", "name": "Stefan Schmelz", "gender": "M", "description": ""}, {"index": 447, "label": "2541", "name": "Eddie Winter", "gender": "M", "description": ""}, {"index": 448, "label": "2544", "name": "Annise", "gender": "F", "description": ""}, {"index": 449, "label": "2545", "name": "the quiet fox", "gender": "M", "description": ""}, {"index": 450, "label": "254", "name": "Alan Davis Drake (1945-2010)", "gender": "M", "description": ""}, {"index": 451, "label": "2552", "name": "Daniel Cranston", "gender": "M", "description": ""}, {"index": 452, "label": "2553", "name": "daisy55", "gender": "F", "description": ""}, {"index": 453, "label": "255", "name": "Chris Hawk", "gender": "M", "description": ""}, {"index": 454, "label": "2568", "name": "Elena the Quiet", "gender": "F", "description": ""}, {"index": 455, "label": "2570", "name": "kindlibrarian", "gender": "F", "description": ""}, {"index": 456, "label": "2573", "name": "Becca B", "gender": "F", "description": ""}, {"index": 457, "label": "2574", "name": "TimSC", "gender": "M", "description": ""}, {"index": 458, "label": "2577", "name": "K Hindall", "gender": "F", "description": ""}, {"index": 459, "label": "2587", "name": "Anne Cheng", "gender": "F", "description": ""}, {"index": 460, "label": "2588", "name": "Padraig O\'hIceadha", "gender": "M", "description": ""}, {"index": 461, "label": "2589", "name": "Jordan", "gender": "M", "description": ""}, {"index": 462, "label": "258", "name": "Kurt Wong", "gender": "M", "description": ""}, {"index": 463, "label": "2598", "name": "Barbara Bulkeley", "gender": "F", "description": ""}, {"index": 464, "label": "25", "name": "John Gonzalez", "gender": "M", "description": ""}, {"index": 465, "label": "2606", "name": "Marc Tanti", "gender": "M", "description": ""}, {"index": 466, "label": "2607", "name": "Ruth Golding", "gender": "F", "description": ""}, {"index": 467, "label": "2624", "name": "David Nicol", "gender": "M", "description": ""}, {"index": 468, "label": "2628", "name": "Zloot", "gender": "M", "description": ""}, {"index": 469, "label": "2638", "name": "Dawn", "gender": "F", "description": ""}, {"index": 470, "label": "263", "name": "Michael Sirois", "gender": "M", "description": ""}, {"index": 471, "label": "264", "name": "Chris Hughes", "gender": "M", "description": ""}, {"index": 472, "label": "2652", "name": "MixieArmadillo", "gender": "F", "description": ""}, {"index": 473, "label": "2654", "name": "Notelrac", "gender": "M", "description": ""}, {"index": 474, "label": "265", "name": "Janice", "gender": "M", "description": ""}, {"index": 475, "label": "2660", "name": "mpetranech", "gender": "M", "description": ""}, {"index": 476, "label": "2671", "name": "Foreign Girl", "gender": "F", "description": ""}, {"index": 477, "label": "2673", "name": "jude kaider", "gender": "M", "description": ""}, {"index": 478, "label": "2674", "name": "Jason Procopio", "gender": "M", "description": ""}, {"index": 479, "label": "2676", "name": "missizii", "gender": "F", "description": ""}, {"index": 480, "label": "2691", "name": "Donna Stewart", "gender": "F", "description": ""}, {"index": 481, "label": "2694", "name": "DianaJMB", "gender": "F", "description": ""}, {"index": 482, "label": "26", "name": "Denny Sayers", "gender": "M", "description": ""}, {"index": 483, "label": "2712", "name": "anoldfashiongirl", "gender": "F", "description": ""}, {"index": 484, "label": "2724", "name": "Michael Dalling", "gender": "M", "description": ""}, {"index": 485, "label": "272", "name": "Mr. Baby Man", "gender": "M", "description": ""}, {"index": 486, "label": "2730", "name": "Shirley Anderson", "gender": "F", "description": ""}, {"index": 487, "label": "2733", "name": "CalmDragon", "gender": "M", "description": ""}, {"index": 488, "label": "2735", "name": "ASchindler", "gender": "F", "description": ""}, {"index": 489, "label": "273", "name": "Jim Cadwell", "gender": "M", "description": ""}, {"index": 490, "label": "2740", "name": "Tony Ashworth", "gender": "M", "description": ""}, {"index": 491, "label": "2741", "name": "Jan Dawn Doronila", "gender": "F", "description": ""}, {"index": 492, "label": "2748", "name": "Annoying Twit", "gender": "F", "description": ""}, {"index": 493, "label": "274", "name": "Eileen George", "gender": "F", "description": ""}, {"index": 494, "label": "2754", "name": "peaceuntoyou", "gender": "F", "description": ""}, {"index": 495, "label": "2762", "name": "Phillipa Chantry", "gender": "F", "description": ""}, {"index": 496, "label": "2764", "name": "Piper Hale", "gender": "F", "description": ""}, {"index": 497, "label": "2769", "name": "Anthony Wilson", "gender": "M", "description": ""}, {"index": 498, "label": "2775", "name": "ink tree", "gender": "F", "description": ""}, {"index": 499, "label": "277", "name": "Alessia", "gender": "F", "description": ""}, {"index": 500, "label": "2785", "name": "humeangel", "gender": "F", "description": ""}, {"index": 501, "label": "2787", "name": "Quentin Manuel", "gender": "M", "description": ""}, {"index": 502, "label": "278", "name": "AliceG", "gender": "F", "description": ""}, {"index": 503, "label": "27", "name": "Sean McKinley", "gender": "M", "description": ""}, {"index": 504, "label": "2812", "name": "Greg Hartley", "gender": "M", "description": ""}, {"index": 505, "label": "2817", "name": "Catherine Millward", "gender": "F", "description": ""}, {"index": 506, "label": "2825", "name": "Ernst Pattynama", "gender": "M", "description": ""}, {"index": 507, "label": "2827", "name": "David Leeson", "gender": "M", "description": ""}, {"index": 508, "label": "2834", "name": "Ksushi", "gender": "F", "description": ""}, {"index": 509, "label": "2836", "name": "Linda McDaniel", "gender": "F", "description": ""}, {"index": 510, "label": "283", "name": "Asaf Bartov", "gender": "M", "description": ""}, {"index": 511, "label": "2843", "name": "ricell", "gender": "M", "description": ""}, {"index": 512, "label": "2853", "name": "Jane Greensmith", "gender": "F", "description": ""}, {"index": 513, "label": "2854", "name": "Andrew Coleman", "gender": "M", "description": ""}, {"index": 514, "label": "2882", "name": "ayngelwing", "gender": "F", "description": ""}, {"index": 515, "label": "2893", "name": "Ryan Sutter", "gender": "M", "description": ""}, {"index": 516, "label": "2895", "name": "Jenny Lundak", "gender": "F", "description": ""}, {"index": 517, "label": "289", "name": "Barbara Wedge", "gender": "F", "description": ""}, {"index": 518, "label": "28", "name": "Kristin Hughes", "gender": "F", "description": ""}, {"index": 519, "label": "2909", "name": "Petra", "gender": "F", "description": ""}, {"index": 520, "label": "2910", "name": "Janna", "gender": "F", "description": ""}, {"index": 521, "label": "2911", "name": "David Lawrence", "gender": "M", "description": ""}, {"index": 522, "label": "2919", "name": "Raerity", "gender": "F", "description": ""}, {"index": 523, "label": "2920", "name": "Rechelle", "gender": "F", "description": ""}, {"index": 524, "label": "2925", "name": "Darla", "gender": "F", "description": ""}, {"index": 525, "label": "2930", "name": "BUAES", "gender": "M", "description": ""}, {"index": 526, "label": "2943", "name": "Sarah Gutierrez", "gender": "F", "description": ""}, {"index": 527, "label": "2946", "name": "Markus Wachenheim", "gender": "M", "description": ""}, {"index": 528, "label": "294", "name": "Calliope", "gender": "F", "description": ""}, {"index": 529, "label": "2952", "name": "Scott Carpenter", "gender": "M", "description": ""}, {"index": 530, "label": "2960", "name": "Scotty", "gender": "M", "description": ""}, {"index": 531, "label": "2967", "name": "Landii", "gender": "F", "description": ""}, {"index": 532, "label": "2971", "name": "Matthew C. Heckel", "gender": "M", "description": ""}, {"index": 533, "label": "2975", "name": "Sarafina Suransky", "gender": "F", "description": ""}, {"index": 534, "label": "2979", "name": "Jilliane Brandt", "gender": "F", "description": ""}, {"index": 535, "label": "2985", "name": "Cantor", "gender": "M", "description": ""}, {"index": 536, "label": "2988", "name": "Larissa Little", "gender": "F", "description": ""}, {"index": 537, "label": "2989", "name": "Jamie Strassenburg, Cypress, California", "gender": "F", "description": ""}, {"index": 538, "label": "298", "name": "Caroline Morse", "gender": "F", "description": ""}, {"index": 539, "label": "2990", "name": "Tom Crawford", "gender": "M", "description": ""}, {"index": 540, "label": "2992", "name": "davechase", "gender": "M", "description": ""}, {"index": 541, "label": "2997", "name": "Violet", "gender": "F", "description": ""}, {"index": 542, "label": "2998", "name": "Kim Jansen", "gender": "M", "description": ""}, {"index": 543, "label": "2999", "name": "Joseph Finkberg", "gender": "M", "description": ""}, {"index": 544, "label": "29", "name": "Linton", "gender": "M", "description": ""}, {"index": 545, "label": "3003", "name": "Sue Anderson", "gender": "F", "description": ""}, {"index": 546, "label": "3006", "name": "MorganScorpion", "gender": "F", "description": ""}, {"index": 547, "label": "3008", "name": "Gloria Zbilicki", "gender": "F", "description": ""}, {"index": 548, "label": "3020", "name": "Mike Vendetti", "gender": "M", "description": ""}, {"index": 549, "label": "3021", "name": "leetcr", "gender": "M", "description": ""}, {"index": 550, "label": "3025", "name": "MichelleHarris", "gender": "F", "description": ""}, {"index": 551, "label": "302", "name": "Chris Peterson", "gender": "F", "description": ""}, {"index": 552, "label": "3033", "name": "valli", "gender": "F", "description": ""}, {"index": 553, "label": "303", "name": "Tony Hightower", "gender": "M", "description": ""}, {"index": 554, "label": "3045", "name": "Cameron Conaway", "gender": "M", "description": ""}, {"index": 555, "label": "3046", "name": "Stephanie Land", "gender": "F", "description": ""}, {"index": 556, "label": "3053", "name": "Tracy Yonemoto", "gender": "F", "description": ""}, {"index": 557, "label": "3054", "name": "GerryR", "gender": "M", "description": ""}, {"index": 558, "label": "3060", "name": "Didier", "gender": "M", "description": ""}, {"index": 559, "label": "3063", "name": "markman", "gender": "M", "description": ""}, {"index": 560, "label": "3079", "name": "nivedita", "gender": "F", "description": ""}, {"index": 561, "label": "307", "name": "Randy Phillips", "gender": "M", "description": ""}, {"index": 562, "label": "3082", "name": "Logan McCamon", "gender": "M", "description": ""}, {"index": 563, "label": "3088", "name": "Sean O\'Hara", "gender": "M", "description": ""}, {"index": 564, "label": "3090", "name": "kelcymx", "gender": "F", "description": ""}, {"index": 565, "label": "3094", "name": "moe", "gender": "F", "description": ""}, {"index": 566, "label": "3097", "name": "nitram", "gender": "M", "description": ""}, {"index": 567, "label": "3098", "name": "woggy298", "gender": "M", "description": ""}, {"index": 568, "label": "30", "name": "Annie Coleman Rothenberg", "gender": "F", "description": ""}, {"index": 569, "label": "3100", "name": "David Higham", "gender": "M", "description": ""}, {"index": 570, "label": "3105", "name": "gfairch511", "gender": "M", "description": ""}, {"index": 571, "label": "3109", "name": "Diogenes Dog", "gender": "M", "description": ""}, {"index": 572, "label": "310", "name": "Dexnell Peters", "gender": "F", "description": ""}, {"index": 573, "label": "3112", "name": "Jessica Louise", "gender": "F", "description": ""}, {"index": 574, "label": "3114", "name": "Geoffrey Edwards", "gender": "M", "description": ""}, {"index": 575, "label": "3118", "name": "George Yeager", "gender": "M", "description": ""}, {"index": 576, "label": "3119", "name": "Sharon Riskedahl", "gender": "F", "description": ""}, {"index": 577, "label": "311", "name": "deadwhitemales", "gender": "M", "description": ""}, {"index": 578, "label": "3125", "name": "Fran", "gender": "F", "description": ""}, {"index": 579, "label": "3132", "name": "Andy Yu", "gender": "M", "description": ""}, {"index": 580, "label": "3135", "name": "Steve Foreman", "gender": "M", "description": ""}, {"index": 581, "label": "3137", "name": "Parrot", "gender": "M", "description": ""}, {"index": 582, "label": "3138", "name": "Labyrinth Composer", "gender": "F", "description": ""}, {"index": 583, "label": "313", "name": "Dilini Jayasinghe", "gender": "F", "description": ""}, {"index": 584, "label": "3142", "name": "RogerA", "gender": "M", "description": ""}, {"index": 585, "label": "3143", "name": "suzanne", "gender": "F", "description": ""}, {"index": 586, "label": "3144", "name": "jfmarchini", "gender": "M", "description": ""}, {"index": 587, "label": "3148", "name": "artos", "gender": "M", "description": ""}, {"index": 588, "label": "3157", "name": "TriciaG", "gender": "F", "description": ""}, {"index": 589, "label": "3168", "name": "David Anton", "gender": "M", "description": ""}, {"index": 590, "label": "3172", "name": "Beatrice", "gender": "F", "description": ""}, {"index": 591, "label": "3179", "name": "Robin", "gender": "F", "description": ""}, {"index": 592, "label": "317", "name": "Mike Gardom", "gender": "M", "description": ""}, {"index": 593, "label": "3180", "name": "Mike Conrad", "gender": "M", "description": ""}, {"index": 594, "label": "3192", "name": "Arfuhrm", "gender": "M", "description": ""}, {"index": 595, "label": "3196", "name": "Elizabeth Harker", "gender": "F", "description": ""}, {"index": 596, "label": "319", "name": "Ed Good", "gender": "M", "description": ""}, {"index": 597, "label": "31", "name": "Martin Clifton", "gender": "M", "description": ""}, {"index": 598, "label": "3214", "name": "fourteatoo", "gender": "M", "description": ""}, {"index": 599, "label": "3215", "name": "Shirley Ellen", "gender": "F", "description": ""}, {"index": 600, "label": "3224", "name": "Acacia Wood", "gender": "F", "description": ""}, {"index": 601, "label": "3227", "name": "Betina", "gender": "F", "description": ""}, {"index": 602, "label": "322", "name": "Elisabeth Shields", "gender": "F", "description": ""}, {"index": 603, "label": "3230", "name": "rasputin", "gender": "M", "description": ""}, {"index": 604, "label": "3235", "name": "Karen Commins", "gender": "F", "description": ""}, {"index": 605, "label": "3238", "name": "nihilist00", "gender": "M", "description": ""}, {"index": 606, "label": "3240", "name": "flakker", "gender": "M", "description": ""}, {"index": 607, "label": "3242", "name": "peac", "gender": "M", "description": ""}, {"index": 608, "label": "3244", "name": "wilwarin", "gender": "F", "description": ""}, {"index": 609, "label": "3245", "name": "wendy", "gender": "F", "description": ""}, {"index": 610, "label": "3257", "name": "Jay Vance", "gender": "M", "description": ""}, {"index": 611, "label": "3258", "name": "mwalimu", "gender": "F", "description": ""}, {"index": 612, "label": "3259", "name": "Kate West", "gender": "F", "description": ""}, {"index": 613, "label": "3261", "name": "fourgrays", "gender": "M", "description": ""}, {"index": 614, "label": "3268", "name": "Patrick McHaffie", "gender": "M", "description": ""}, {"index": 615, "label": "3271", "name": "Ancient mariner", "gender": "M", "description": ""}, {"index": 616, "label": "3272", "name": "Matthew_J_Almeida", "gender": "M", "description": ""}, {"index": 617, "label": "3274", "name": "Morgan Saletta", "gender": "M", "description": ""}, {"index": 618, "label": "3285", "name": "David Collins", "gender": "M", "description": ""}, {"index": 619, "label": "3288", "name": "Euan Bayliss", "gender": "M", "description": ""}, {"index": 620, "label": "3289", "name": "lukeprog", "gender": "M", "description": ""}, {"index": 621, "label": "3290", "name": "Marian Martin", "gender": "F", "description": ""}, {"index": 622, "label": "32", "name": "Betsie Bush", "gender": "F", "description": ""}, {"index": 623, "label": "3307", "name": "Doug Allison", "gender": "M", "description": ""}, {"index": 624, "label": "3314", "name": "Carol Stripling", "gender": "F", "description": ""}, {"index": 625, "label": "3318", "name": "Magdalena", "gender": "F", "description": ""}, {"index": 626, "label": "3319", "name": "mjbrichant", "gender": "F", "description": ""}, {"index": 627, "label": "331", "name": "Richard Grove", "gender": "M", "description": ""}, {"index": 628, "label": "332", "name": "Aaron Teiser", "gender": "M", "description": ""}, {"index": 629, "label": "3334", "name": "joi", "gender": "F", "description": ""}, {"index": 630, "label": "3340", "name": "Preston McConkie", "gender": "M", "description": ""}, {"index": 631, "label": "3346", "name": "Hannah Dowell", "gender": "F", "description": ""}, {"index": 632, "label": "3356", "name": "Diana Solomon", "gender": "F", "description": ""}, {"index": 633, "label": "3357", "name": "swade", "gender": "F", "description": ""}, {"index": 634, "label": "335", "name": "Paul Harvey", "gender": "M", "description": ""}, {"index": 635, "label": "3361", "name": "Linda Lee Paquet", "gender": "F", "description": ""}, {"index": 636, "label": "336", "name": "Harvey Chinn", "gender": "M", "description": ""}, {"index": 637, "label": "3370", "name": "Glenn Simonsen", "gender": "M", "description": ""}, {"index": 638, "label": "3373", "name": "cvd", "gender": "M", "description": ""}, {"index": 639, "label": "3374", "name": "Craig Campbell", "gender": "M", "description": ""}, {"index": 640, "label": "3379", "name": "Kele", "gender": "F", "description": ""}, {"index": 641, "label": "337", "name": "Barbara Harvey", "gender": "F", "description": ""}, {"index": 642, "label": "3380", "name": "DrBeccaAnne", "gender": "F", "description": ""}, {"index": 643, "label": "3381", "name": "Cagliostro", "gender": "M", "description": ""}, {"index": 644, "label": "3389", "name": "von", "gender": "M", "description": ""}, {"index": 645, "label": "3394", "name": "Jackie Provau", "gender": "F", "description": ""}, {"index": 646, "label": "3400", "name": "Laura Davis", "gender": "F", "description": ""}, {"index": 647, "label": "3409", "name": "Philippa Brodie", "gender": "F", "description": ""}, {"index": 648, "label": "340", "name": "Scott Henkel", "gender": "M", "description": ""}, {"index": 649, "label": "3411", "name": "SuD", "gender": "F", "description": ""}, {"index": 650, "label": "3417", "name": "Albatross", "gender": "M", "description": ""}, {"index": 651, "label": "3433", "name": "Bob Sherman", "gender": "M", "description": ""}, {"index": 652, "label": "3436", "name": "Anders Lankford", "gender": "M", "description": ""}, {"index": 653, "label": "3440", "name": "Heidi Will", "gender": "F", "description": ""}, {"index": 654, "label": "3446", "name": "kayo", "gender": "F", "description": ""}, {"index": 655, "label": "3448", "name": "Todd Lennon", "gender": "M", "description": ""}, {"index": 656, "label": "345", "name": "Micah Sheppard", "gender": "M", "description": ""}, {"index": 657, "label": "3465", "name": "ravenotation", "gender": "F", "description": ""}, {"index": 658, "label": "3467", "name": "js392", "gender": "F", "description": ""}, {"index": 659, "label": "3470", "name": "Jason Mills", "gender": "M", "description": ""}, {"index": 660, "label": "3479", "name": "Karan Yamada", "gender": "F", "description": ""}, {"index": 661, "label": "3482", "name": "Hayden", "gender": "F", "description": ""}, {"index": 662, "label": "3486", "name": "Robin Balmer", "gender": "M", "description": ""}, {"index": 663, "label": "3488", "name": "Tom Weiss", "gender": "M", "description": ""}, {"index": 664, "label": "348", "name": "ianish", "gender": "M", "description": ""}, {"index": 665, "label": "3490", "name": "Gregg Margarite (1957-2012)", "gender": "M", "description": ""}, {"index": 666, "label": "3500", "name": "B. Treadgold", "gender": "F", "description": ""}, {"index": 667, "label": "3503", "name": "Christian Al-Kadi", "gender": "M", "description": ""}, {"index": 668, "label": "3513", "name": "Symmie", "gender": "F", "description": ""}, {"index": 669, "label": "3521", "name": "NickNumber", "gender": "M", "description": ""}, {"index": 670, "label": "3526", "name": "Bereni", "gender": "F", "description": ""}, {"index": 671, "label": "3537", "name": "Tracy Datlen", "gender": "F", "description": ""}, {"index": 672, "label": "353", "name": "Jamey Osborne", "gender": "M", "description": ""}, {"index": 673, "label": "3541", "name": "Termin Dyan", "gender": "M", "description": ""}, {"index": 674, "label": "3546", "name": "Jeannie", "gender": "F", "description": ""}, {"index": 675, "label": "3547", "name": "BenW", "gender": "M", "description": ""}, {"index": 676, "label": "3551", "name": "Annie Kirkpatrick", "gender": "F", "description": ""}, {"index": 677, "label": "3553", "name": "Luc Kordas", "gender": "M", "description": ""}, {"index": 678, "label": "3554", "name": "LaraC, Louisville, KY", "gender": "F", "description": ""}, {"index": 679, "label": "3557", "name": "Rachel Triska", "gender": "F", "description": ""}, {"index": 680, "label": "3559", "name": "Kerry Hiles", "gender": "F", "description": "Soothing, clear, inviting"}, {"index": 681, "label": "3564", "name": "Vanessa", "gender": "F", "description": ""}, {"index": 682, "label": "3567", "name": "David Lazarus", "gender": "M", "description": ""}, {"index": 683, "label": "3571", "name": "HarryInk", "gender": "M", "description": ""}, {"index": 684, "label": "3584", "name": "Bridget Gaige", "gender": "F", "description": ""}, {"index": 685, "label": "3587", "name": "John Nixon", "gender": "M", "description": ""}, {"index": 686, "label": "3588", "name": "Neeru Iyer", "gender": "F", "description": ""}, {"index": 687, "label": "3592", "name": "Paul McCartan", "gender": "M", "description": ""}, {"index": 688, "label": "3595", "name": "Martin Geeson", "gender": "M", "description": ""}, {"index": 689, "label": "3598", "name": "Dawn", "gender": "F", "description": ""}, {"index": 690, "label": "3606", "name": "Ashwath Ganesan", "gender": "M", "description": ""}, {"index": 691, "label": "3607", "name": "Richard Wallis", "gender": "M", "description": ""}, {"index": 692, "label": "3618", "name": "Timothy Ferguson", "gender": "M", "description": ""}, {"index": 693, "label": "362", "name": "Judith Brown", "gender": "F", "description": ""}, {"index": 694, "label": "3630", "name": "Rachel Gatwood", "gender": "F", "description": ""}, {"index": 695, "label": "3638", "name": "AmyG", "gender": "F", "description": ""}, {"index": 696, "label": "3641", "name": "Joelle Peebles", "gender": "F", "description": ""}, {"index": 697, "label": "3645", "name": "MaryAnn", "gender": "F", "description": ""}, {"index": 698, "label": "3647", "name": "Channe", "gender": "F", "description": ""}, {"index": 699, "label": "3650", "name": "Jonathan Ross", "gender": "M", "description": ""}, {"index": 700, "label": "3654", "name": "Mark Wilson", "gender": "M", "description": ""}, {"index": 701, "label": "3656", "name": "Kai Lu", "gender": "M", "description": ""}, {"index": 702, "label": "3657", "name": "Bellona Times", "gender": "M", "description": ""}, {"index": 703, "label": "365", "name": "Jon Ingram", "gender": "M", "description": ""}, {"index": 704, "label": "3664", "name": "Barry Eads", "gender": "M", "description": ""}, {"index": 705, "label": "3665", "name": "C.J. Casey", "gender": "M", "description": ""}, {"index": 706, "label": "366", "name": "Katy Preston", "gender": "F", "description": ""}, {"index": 707, "label": "3675", "name": "Linda Ferguson", "gender": "F", "description": ""}, {"index": 708, "label": "3679", "name": "veronasser", "gender": "F", "description": ""}, {"index": 709, "label": "3681", "name": "Ann Boulais", "gender": "F", "description": ""}, {"index": 710, "label": "3691", "name": "Hollis Hanover", "gender": "M", "description": ""}, {"index": 711, "label": "3698", "name": "Nadine Eckert-Boulet", "gender": "F", "description": ""}, {"index": 712, "label": "3699", "name": "Bruce Pirie", "gender": "M", "description": ""}, {"index": 713, "label": "369", "name": "Kevin Readdean", "gender": "M", "description": ""}, {"index": 714, "label": "36", "name": "Chip", "gender": "M", "description": ""}, {"index": 715, "label": "3703", "name": "Linda Andrus", "gender": "F", "description": ""}, {"index": 716, "label": "3723", "name": "Kevin Lavin", "gender": "M", "description": ""}, {"index": 717, "label": "3728", "name": "amycs", "gender": "F", "description": ""}, {"index": 718, "label": "3733", "name": "Melanie Schleeter McCalmont", "gender": "F", "description": ""}, {"index": 719, "label": "3738", "name": "Collee McKinnon", "gender": "F", "description": ""}, {"index": 720, "label": "373", "name": "Kim Braun", "gender": "F", "description": ""}, {"index": 721, "label": "3744", "name": "Jonathan Burchard", "gender": "M", "description": ""}, {"index": 722, "label": "3747", "name": "Keith Henige", "gender": "M", "description": ""}, {"index": 723, "label": "374", "name": "kumarei", "gender": "M", "description": ""}, {"index": 724, "label": "3757", "name": "EmAllise", "gender": "F", "description": ""}, {"index": 725, "label": "3779", "name": "Rachel Steely", "gender": "F", "description": ""}, {"index": 726, "label": "377", "name": "Lenny Glionna Jr.", "gender": "M", "description": ""}, {"index": 727, "label": "3780", "name": "Pete", "gender": "M", "description": ""}, {"index": 728, "label": "3781", "name": "Celena Arter", "gender": "F", "description": ""}, {"index": 729, "label": "3783", "name": "TexasSteve", "gender": "M", "description": ""}, {"index": 730, "label": "3793", "name": "Nathan", "gender": "M", "description": ""}, {"index": 731, "label": "3796", "name": "Mario Pineda", "gender": "M", "description": ""}, {"index": 732, "label": "3798", "name": "Bianca Kramer", "gender": "F", "description": ""}, {"index": 733, "label": "37", "name": "wolvrin", "gender": "M", "description": ""}, {"index": 734, "label": "3807", "name": "Jesse Noar", "gender": "M", "description": ""}, {"index": 735, "label": "380", "name": "Laurie Campbell", "gender": "F", "description": ""}, {"index": 736, "label": "3819", "name": "StarrDog", "gender": "M", "description": ""}, {"index": 737, "label": "3830", "name": "rymd80", "gender": "M", "description": ""}, {"index": 738, "label": "3835", "name": "M.White", "gender": "M", "description": ""}, {"index": 739, "label": "3843", "name": "storm", "gender": "F", "description": ""}, {"index": 740, "label": "3845", "name": "Ray Smith", "gender": "M", "description": ""}, {"index": 741, "label": "3848", "name": "skoval", "gender": "F", "description": ""}, {"index": 742, "label": "3851", "name": "mbousquet", "gender": "F", "description": ""}, {"index": 743, "label": "3852", "name": "selniff", "gender": "F", "description": ""}, {"index": 744, "label": "3857", "name": "Epistomolus", "gender": "M", "description": ""}, {"index": 745, "label": "3864", "name": "Tom Watts", "gender": "M", "description": ""}, {"index": 746, "label": "3866", "name": "SilverG", "gender": "M", "description": ""}, {"index": 747, "label": "3867", "name": "Roberta Carlisle", "gender": "F", "description": ""}, {"index": 748, "label": "3869", "name": "Timothy Pinkham", "gender": "M", "description": ""}, {"index": 749, "label": "3871", "name": "Figura", "gender": "M", "description": ""}, {"index": 750, "label": "3876", "name": "Frances Marcinkiewicz", "gender": "F", "description": ""}, {"index": 751, "label": "3879", "name": "Keneva", "gender": "F", "description": ""}, {"index": 752, "label": "3885", "name": "Elli", "gender": "F", "description": ""}, {"index": 753, "label": "3894", "name": "Indy Gosal", "gender": "M", "description": ""}, {"index": 754, "label": "3895", "name": "porlob", "gender": "M", "description": ""}, {"index": 755, "label": "3896", "name": "Will Zufall", "gender": "M", "description": ""}, {"index": 756, "label": "38", "name": "R. Francis Smith", "gender": "M", "description": ""}, {"index": 757, "label": "3905", "name": "J. Rebecca Franklin", "gender": "F", "description": ""}, {"index": 758, "label": "3906", "name": "tabithat", "gender": "F", "description": ""}, {"index": 759, "label": "3909", "name": "Evelyn Clarke", "gender": "F", "description": ""}, {"index": 760, "label": "3911", "name": "Joseph Couves", "gender": "F", "description": ""}, {"index": 761, "label": "3912", "name": "Bob Neufeld", "gender": "M", "description": ""}, {"index": 762, "label": "3922", "name": "Ashley Candland", "gender": "F", "description": ""}, {"index": 763, "label": "3923", "name": "Sean Michael Hogan", "gender": "M", "description": ""}, {"index": 764, "label": "3925", "name": "Viglione", "gender": "F", "description": ""}, {"index": 765, "label": "3926", "name": "Denise Lacey", "gender": "F", "description": ""}, {"index": 766, "label": "3928", "name": "Marianne Coleman-Hipkins", "gender": "F", "description": ""}, {"index": 767, "label": "392", "name": "Maria Celano", "gender": "F", "description": ""}, {"index": 768, "label": "3934", "name": "Lynne Carroll", "gender": "F", "description": ""}, {"index": 769, "label": "3945", "name": "Floyd Wilde", "gender": "M", "description": ""}, {"index": 770, "label": "3947", "name": "johnell", "gender": "F", "description": ""}, {"index": 771, "label": "3955", "name": "Fredrik Karlsson", "gender": "M", "description": ""}, {"index": 772, "label": "3959", "name": "rjhargrav", "gender": "M", "description": ""}, {"index": 773, "label": "3962", "name": "writerboyontour", "gender": "M", "description": ""}, {"index": 774, "label": "3969", "name": "CM Slosson", "gender": "F", "description": ""}, {"index": 775, "label": "3972", "name": "Joy Easton", "gender": "F", "description": ""}, {"index": 776, "label": "3977", "name": "LivelyHive", "gender": "M", "description": ""}, {"index": 777, "label": "3979", "name": "Dale A. Bade", "gender": "F", "description": ""}, {"index": 778, "label": "3982", "name": "Kate Adams", "gender": "F", "description": ""}, {"index": 779, "label": "3983", "name": "lavocedorata", "gender": "F", "description": ""}, {"index": 780, "label": "398", "name": "James Smith", "gender": "M", "description": ""}, {"index": 781, "label": "3990", "name": "Jessi", "gender": "F", "description": ""}, {"index": 782, "label": "3992", "name": "perpetualdreamworld", "gender": "F", "description": ""}, {"index": 783, "label": "3994", "name": "Miriam Esther Goldman", "gender": "F", "description": ""}, {"index": 784, "label": "39", "name": "Sherry Crowther", "gender": "F", "description": ""}, {"index": 785, "label": "4005", "name": "Jhiu", "gender": "F", "description": ""}, {"index": 786, "label": "4009", "name": "Diana Majlinger", "gender": "F", "description": ""}, {"index": 787, "label": "4010", "name": "David Baldwin", "gender": "M", "description": ""}, {"index": 788, "label": "4013", "name": "Kevin Maxson", "gender": "M", "description": ""}, {"index": 789, "label": "4014", "name": "Tom Clifton", "gender": "M", "description": ""}, {"index": 790, "label": "4015", "name": "JimOCR", "gender": "M", "description": ""}, {"index": 791, "label": "4017", "name": "gsolgaard", "gender": "M", "description": ""}, {"index": 792, "label": "4018", "name": "Nicholas Clifford", "gender": "M", "description": ""}, {"index": 793, "label": "4019", "name": "brrrrrr6", "gender": "M", "description": ""}, {"index": 794, "label": "4020", "name": "Linda", "gender": "M", "description": ""}, {"index": 795, "label": "4021", "name": "Linda Woods", "gender": "F", "description": ""}, {"index": 796, "label": "402", "name": "Sharmini Kumar", "gender": "F", "description": ""}, {"index": 797, "label": "4034", "name": "Sienna", "gender": "F", "description": ""}, {"index": 798, "label": "4039", "name": "Shawn Craig Smith", "gender": "M", "description": ""}, {"index": 799, "label": "403", "name": "Nocturna", "gender": "F", "description": ""}, {"index": 800, "label": "4042", "name": "Ryan DeRamos", "gender": "M", "description": ""}, {"index": 801, "label": "4044", "name": "serenitylee", "gender": "F", "description": ""}, {"index": 802, "label": "404", "name": "Nomenphile", "gender": "F", "description": ""}, {"index": 803, "label": "4051", "name": "Liz Devens", "gender": "F", "description": ""}, {"index": 804, "label": "4054", "name": "Ryan Gubele", "gender": "M", "description": ""}, {"index": 805, "label": "4057", "name": "RoseA", "gender": "F", "description": ""}, {"index": 806, "label": "4059", "name": "Troy Bond", "gender": "M", "description": ""}, {"index": 807, "label": "405", "name": "Eric Dennison", "gender": "M", "description": ""}, {"index": 808, "label": "4063", "name": "Abigail Bartels", "gender": "F", "description": ""}, {"index": 809, "label": "4064", "name": "Margaret Espaillat", "gender": "F", "description": ""}, {"index": 810, "label": "4071", "name": "Nichelle von Lauder", "gender": "F", "description": ""}, {"index": 811, "label": "4078", "name": "Richard Kilmer", "gender": "M", "description": ""}, {"index": 812, "label": "4085", "name": "Paul P Miller", "gender": "M", "description": ""}, {"index": 813, "label": "4088", "name": "Blazin48", "gender": "F", "description": ""}, {"index": 814, "label": "408", "name": "Claudine Chen", "gender": "F", "description": ""}, {"index": 815, "label": "4090", "name": "madmouth", "gender": "F", "description": ""}, {"index": 816, "label": "4098", "name": "Rachell Lovett", "gender": "F", "description": ""}, {"index": 817, "label": "409", "name": "Mike Kauffmann", "gender": "M", "description": ""}, {"index": 818, "label": "40", "name": "Vicki Barbour", "gender": "F", "description": ""}, {"index": 819, "label": "4104", "name": "Linda Dodge", "gender": "F", "description": ""}, {"index": 820, "label": "4108", "name": "garymacf", "gender": "M", "description": ""}, {"index": 821, "label": "4110", "name": "Richard Ellwood", "gender": "M", "description": ""}, {"index": 822, "label": "4111", "name": "Rachel P.", "gender": "F", "description": ""}, {"index": 823, "label": "4122", "name": "Wendel Topper", "gender": "M", "description": ""}, {"index": 824, "label": "412", "name": "Brian Roberg", "gender": "M", "description": ""}, {"index": 825, "label": "4137", "name": "Sarah LuAnn", "gender": "F", "description": ""}, {"index": 826, "label": "4138", "name": "daltongirl", "gender": "F", "description": ""}, {"index": 827, "label": "413", "name": "Daniel Watkins", "gender": "M", "description": ""}, {"index": 828, "label": "4145", "name": "Patti Cunningham", "gender": "F", "description": ""}, {"index": 829, "label": "4152", "name": "Ata Khudayberdiev", "gender": "M", "description": ""}, {"index": 830, "label": "4156", "name": "Caroline Shapiro", "gender": "F", "description": ""}, {"index": 831, "label": "4160", "name": "Rosie", "gender": "F", "description": ""}, {"index": 832, "label": "4161", "name": "Daniel Paashaus", "gender": "M", "description": ""}, {"index": 833, "label": "4172", "name": "Graeme Dunlop", "gender": "M", "description": ""}, {"index": 834, "label": "4174", "name": "Availle", "gender": "F", "description": ""}, {"index": 835, "label": "4179", "name": "Robert Keiper", "gender": "M", "description": ""}, {"index": 836, "label": "4189", "name": "Megan Argo", "gender": "F", "description": ""}, {"index": 837, "label": "4191", "name": "Grant Hurlock", "gender": "M", "description": ""}, {"index": 838, "label": "4192", "name": "MaryModern", "gender": "F", "description": ""}, {"index": 839, "label": "4193", "name": "Ethan Rampton", "gender": "M", "description": ""}, {"index": 840, "label": "4195", "name": "bj", "gender": "F", "description": ""}, {"index": 841, "label": "4196", "name": "Jim Clevenger", "gender": "M", "description": ""}, {"index": 842, "label": "4205", "name": "psi_mon", "gender": "M", "description": ""}, {"index": 843, "label": "4211", "name": "A. Knight", "gender": "F", "description": ""}, {"index": 844, "label": "4214", "name": "A. Janelle Risa", "gender": "F", "description": ""}, {"index": 845, "label": "4216", "name": "Hannah Skoonberg", "gender": "F", "description": ""}, {"index": 846, "label": "4217", "name": "Brendan Tannam", "gender": "M", "description": ""}, {"index": 847, "label": "4218", "name": "David Cole", "gender": "M", "description": ""}, {"index": 848, "label": "421", "name": "Patrick", "gender": "M", "description": ""}, {"index": 849, "label": "4225", "name": "gmiteva", "gender": "F", "description": ""}, {"index": 850, "label": "4234", "name": "laineyben", "gender": "F", "description": ""}, {"index": 851, "label": "4235", "name": "Haylayer Flaga", "gender": "F", "description": ""}, {"index": 852, "label": "4238", "name": "Chela", "gender": "F", "description": ""}, {"index": 853, "label": "4243", "name": "Ian Grae", "gender": "M", "description": ""}, {"index": 854, "label": "4257", "name": "garbageman99", "gender": "M", "description": ""}, {"index": 855, "label": "4262", "name": "LoraBeth Davis", "gender": "F", "description": ""}, {"index": 856, "label": "4263", "name": "Chelsea Baker", "gender": "F", "description": ""}, {"index": 857, "label": "4267", "name": "Ric F", "gender": "M", "description": ""}, {"index": 858, "label": "426", "name": "Norah Piehl", "gender": "F", "description": ""}, {"index": 859, "label": "4273", "name": "as101", "gender": "F", "description": ""}, {"index": 860, "label": "4277", "name": "JohanG", "gender": "M", "description": ""}, {"index": 861, "label": "4280", "name": "BB", "gender": "F", "description": ""}, {"index": 862, "label": "428", "name": "Rayburn Beale", "gender": "M", "description": ""}, {"index": 863, "label": "4295", "name": "Slawek", "gender": "M", "description": ""}, {"index": 864, "label": "4297", "name": "Tina Horning", "gender": "F", "description": ""}, {"index": 865, "label": "4305", "name": "Aspergine", "gender": "F", "description": ""}, {"index": 866, "label": "4310", "name": "grace4him", "gender": "F", "description": ""}, {"index": 867, "label": "4313", "name": "May Low", "gender": "F", "description": ""}, {"index": 868, "label": "4321", "name": "Xenutia", "gender": "F", "description": ""}, {"index": 869, "label": "4327", "name": "brokenaltar", "gender": "F", "description": ""}, {"index": 870, "label": "432", "name": "Steve Andersen", "gender": "M", "description": ""}, {"index": 871, "label": "4331", "name": "Barbara Clements", "gender": "F", "description": ""}, {"index": 872, "label": "4335", "name": "davidb", "gender": "M", "description": ""}, {"index": 873, "label": "4340", "name": "kiwafruit", "gender": "F", "description": ""}, {"index": 874, "label": "4344", "name": "Tadhg", "gender": "M", "description": ""}, {"index": 875, "label": "4345", "name": "Richard Schipper", "gender": "M", "description": ""}, {"index": 876, "label": "4352", "name": "Martina Hutchins", "gender": "F", "description": ""}, {"index": 877, "label": "4356", "name": "Patricia Rutledge", "gender": "F", "description": ""}, {"index": 878, "label": "4362", "name": "Michelle Montano", "gender": "F", "description": ""}, {"index": 879, "label": "4363", "name": "Emily Livingston", "gender": "F", "description": ""}, {"index": 880, "label": "4379", "name": "Scott Dahlem", "gender": "M", "description": ""}, {"index": 881, "label": "4381", "name": "Denise Resnik", "gender": "F", "description": ""}, {"index": 882, "label": "4396", "name": "Michael Wolf", "gender": "F", "description": ""}, {"index": 883, "label": "4397", "name": "John Dennison", "gender": "M", "description": ""}, {"index": 884, "label": "439", "name": "Robert Garrison", "gender": "M", "description": ""}, {"index": 885, "label": "4402", "name": "Christopher Sanner", "gender": "M", "description": ""}, {"index": 886, "label": "4406", "name": "Matthew Scott Surprenant", "gender": "M", "description": ""}, {"index": 887, "label": "4407", "name": "Ted Drury", "gender": "M", "description": ""}, {"index": 888, "label": "4411", "name": "Zarnaz", "gender": "F", "description": ""}, {"index": 889, "label": "4415", "name": "Anita Fleming", "gender": "F", "description": ""}, {"index": 890, "label": "4420", "name": "Steve Lomas", "gender": "M", "description": ""}, {"index": 891, "label": "4422", "name": "Kamna", "gender": "F", "description": ""}, {"index": 892, "label": "4423", "name": "David Dwight", "gender": "M", "description": ""}, {"index": 893, "label": "4427", "name": "Lynne Handler", "gender": "F", "description": ""}, {"index": 894, "label": "4428", "name": "Blueoyster101", "gender": "M", "description": ""}, {"index": 895, "label": "4433", "name": "Ken Sterry", "gender": "M", "description": ""}, {"index": 896, "label": "4434", "name": "Sarah Nuxoll", "gender": "F", "description": ""}, {"index": 897, "label": "4441", "name": "William Peck", "gender": "M", "description": ""}, {"index": 898, "label": "4442", "name": "Pep", "gender": "M", "description": ""}, {"index": 899, "label": "4443", "name": "THOVO", "gender": "F", "description": ""}, {"index": 900, "label": "4447", "name": "Cascades", "gender": "M", "description": ""}, {"index": 901, "label": "444", "name": "Sage Tyrtle", "gender": "F", "description": ""}, {"index": 902, "label": "4455", "name": "Patrick Wells", "gender": "F", "description": ""}, {"index": 903, "label": "4463", "name": "Cate Mackenzie", "gender": "F", "description": ""}, {"index": 904, "label": "446", "name": "Steve Hartzog", "gender": "M", "description": ""}, {"index": 905, "label": "4474", "name": "John L. Clark", "gender": "M", "description": ""}, {"index": 906, "label": "4480", "name": "Iskander Shafikov", "gender": "M", "description": ""}, {"index": 907, "label": "4481", "name": "margo zinberg", "gender": "F", "description": ""}, {"index": 908, "label": "4484", "name": "Phil Griffiths", "gender": "M", "description": ""}, {"index": 909, "label": "4487", "name": "Nienke", "gender": "F", "description": ""}, {"index": 910, "label": "448", "name": "scrappylibrarian", "gender": "F", "description": ""}, {"index": 911, "label": "4490", "name": "Rachel Weaver", "gender": "F", "description": ""}, {"index": 912, "label": "4492", "name": "P.Hynes", "gender": "F", "description": ""}, {"index": 913, "label": "4495", "name": "Dillon Stiles", "gender": "M", "description": ""}, {"index": 914, "label": "44", "name": "travelbratd", "gender": "F", "description": ""}, {"index": 915, "label": "4511", "name": "Jeanie", "gender": "F", "description": ""}, {"index": 916, "label": "4513", "name": "Gabi", "gender": "F", "description": ""}, {"index": 917, "label": "4519", "name": "Mimi Wang", "gender": "F", "description": ""}, {"index": 918, "label": "451", "name": "Sonserae Leese-Calver", "gender": "F", "description": ""}, {"index": 919, "label": "4520", "name": "Dorlene Kaplan", "gender": "F", "description": ""}, {"index": 920, "label": "4535", "name": "Brett W. Downey", "gender": "M", "description": ""}, {"index": 921, "label": "453", "name": "Glen Hallstrom", "gender": "M", "description": ""}, {"index": 922, "label": "4545", "name": "SunshinePaul", "gender": "M", "description": ""}, {"index": 923, "label": "4546", "name": "Estelle Jobson", "gender": "F", "description": ""}, {"index": 924, "label": "4549", "name": "Kristen Zaza", "gender": "F", "description": ""}, {"index": 925, "label": "454", "name": "Tom Yates", "gender": "M", "description": ""}, {"index": 926, "label": "4563", "name": "Wiebke", "gender": "F", "description": ""}, {"index": 927, "label": "4576", "name": "Snaefaxi", "gender": "F", "description": ""}, {"index": 928, "label": "4583", "name": "Jersey City Frankie", "gender": "M", "description": ""}, {"index": 929, "label": "4586", "name": "Chris Caron", "gender": "M", "description": ""}, {"index": 930, "label": "458", "name": "Scott Splavec", "gender": "M", "description": ""}, {"index": 931, "label": "4591", "name": "Robert White", "gender": "M", "description": ""}, {"index": 932, "label": "4592", "name": "WestWestest", "gender": "F", "description": ""}, {"index": 933, "label": "4594", "name": "mailman61953", "gender": "M", "description": ""}, {"index": 934, "label": "4598", "name": "cher0520", "gender": "F", "description": ""}, {"index": 935, "label": "4599", "name": "xibu", "gender": "M", "description": ""}, {"index": 936, "label": "459", "name": "Mark Bradford", "gender": "M", "description": ""}, {"index": 937, "label": "45", "name": "Kelly Bescherer", "gender": "F", "description": ""}, {"index": 938, "label": "460", "name": "Dave Ranson", "gender": "M", "description": ""}, {"index": 939, "label": "4640", "name": "Karen Mason", "gender": "F", "description": ""}, {"index": 940, "label": "4652", "name": "Savanna Herrold", "gender": "F", "description": ""}, {"index": 941, "label": "4659", "name": "Stephen Marsh", "gender": "M", "description": ""}, {"index": 942, "label": "4660", "name": "SusieSA", "gender": "F", "description": ""}, {"index": 943, "label": "4667", "name": "Maria Therese", "gender": "F", "description": ""}, {"index": 944, "label": "466", "name": "Joy Chan", "gender": "F", "description": ""}, {"index": 945, "label": "4680", "name": "pachayes", "gender": "F", "description": ""}, {"index": 946, "label": "4681", "name": "Frank Adams", "gender": "M", "description": ""}, {"index": 947, "label": "4687", "name": "mevans", "gender": "M", "description": ""}, {"index": 948, "label": "4693", "name": "Robert Parker", "gender": "M", "description": ""}, {"index": 949, "label": "4697", "name": "Jannie Meisberger", "gender": "F", "description": ""}, {"index": 950, "label": "4699", "name": "Jason Justice", "gender": "M", "description": ""}, {"index": 951, "label": "46", "name": "Aaron Hochwimmer", "gender": "M", "description": ""}, {"index": 952, "label": "4701", "name": "CC", "gender": "M", "description": ""}, {"index": 953, "label": "4703", "name": "HurstPP", "gender": "F", "description": ""}, {"index": 954, "label": "4705", "name": "Algy Pug", "gender": "M", "description": ""}, {"index": 955, "label": "4706", "name": "SallyMc", "gender": "F", "description": ""}, {"index": 956, "label": "470", "name": "Chris Chapman", "gender": "M", "description": ""}, {"index": 957, "label": "4710", "name": "David Huston", "gender": "M", "description": ""}, {"index": 958, "label": "4712", "name": "Alisson Veldhuis", "gender": "F", "description": ""}, {"index": 959, "label": "4719", "name": "M. Craun", "gender": "M", "description": ""}, {"index": 960, "label": "472", "name": "Tina Tilney", "gender": "F", "description": ""}, {"index": 961, "label": "4731", "name": "Becky Cook", "gender": "F", "description": ""}, {"index": 962, "label": "4738", "name": "Leonard Wilson", "gender": "M", "description": ""}, {"index": 963, "label": "4741", "name": "Nullifidian", "gender": "M", "description": ""}, {"index": 964, "label": "4742", "name": "Little Tee", "gender": "F", "description": ""}, {"index": 965, "label": "4744", "name": "Amy Gramour", "gender": "F", "description": ""}, {"index": 966, "label": "4748", "name": "Dirk Eichhorn", "gender": "M", "description": ""}, {"index": 967, "label": "474", "name": "Zachary Brewster-Geisz", "gender": "M", "description": ""}, {"index": 968, "label": "4750", "name": "Paul Huckerby", "gender": "M", "description": ""}, {"index": 969, "label": "4757", "name": "Grace Dobson", "gender": "F", "description": ""}, {"index": 970, "label": "475", "name": "Jason X.", "gender": "M", "description": ""}, {"index": 971, "label": "4766", "name": "Melanie", "gender": "F", "description": ""}, {"index": 972, "label": "4767", "name": "Delysid", "gender": "M", "description": ""}, {"index": 973, "label": "476", "name": "Chuck Spann", "gender": "M", "description": ""}, {"index": 974, "label": "4771", "name": "scrawl", "gender": "M", "description": ""}, {"index": 975, "label": "4773", "name": "Kathryn Lois", "gender": "F", "description": ""}, {"index": 976, "label": "4779", "name": "Angel5", "gender": "F", "description": ""}, {"index": 977, "label": "4788", "name": "Bill Boerst", "gender": "M", "description": ""}, {"index": 978, "label": "4791", "name": "MelanieMae", "gender": "F", "description": ""}, {"index": 979, "label": "4799", "name": "Matt Judd", "gender": "M", "description": ""}, {"index": 980, "label": "479", "name": "wedschild", "gender": "F", "description": ""}, {"index": 981, "label": "47", "name": "Amanda", "gender": "F", "description": ""}, {"index": 982, "label": "4800", "name": "Mary Herndon Bell", "gender": "F", "description": ""}, {"index": 983, "label": "4806", "name": "Jason Ingolfsland", "gender": "M", "description": ""}, {"index": 984, "label": "480", "name": "Chris Vee", "gender": "M", "description": ""}, {"index": 985, "label": "4813", "name": "Steve Mattern", "gender": "M", "description": ""}, {"index": 986, "label": "481", "name": "Neal Foley", "gender": "M", "description": ""}, {"index": 987, "label": "4821", "name": "Zachary Johnson", "gender": "M", "description": ""}, {"index": 988, "label": "4824", "name": "meyerli", "gender": "F", "description": ""}, {"index": 989, "label": "4830", "name": "George Aalto", "gender": "M", "description": ""}, {"index": 990, "label": "4836", "name": "Preston Scrape", "gender": "M", "description": ""}, {"index": 991, "label": "4837", "name": "Sharon Kilmer", "gender": "F", "description": ""}, {"index": 992, "label": "483", "name": "junk science", "gender": "F", "description": ""}, {"index": 993, "label": "4841", "name": "Haley Pereira", "gender": "F", "description": ""}, {"index": 994, "label": "4846", "name": "Kathrin Salazar", "gender": "F", "description": ""}, {"index": 995, "label": "4848", "name": "Dee Wykoff", "gender": "M", "description": ""}, {"index": 996, "label": "4853", "name": "Barbara Derksen", "gender": "F", "description": ""}, {"index": 997, "label": "4854", "name": "Doctor_wu", "gender": "M", "description": ""}, {"index": 998, "label": "4859", "name": "nathank", "gender": "M", "description": ""}, {"index": 999, "label": "4860", "name": "Jonah Cummings", "gender": "M", "description": ""}, {"index": 1000, "label": "4863", "name": "I M CLIFFORD", "gender": "M", "description": ""}, {"index": 1001, "label": "4872", "name": "R E Faust", "gender": "M", "description": ""}, {"index": 1002, "label": "4894", "name": "Gary Dzierlenga", "gender": "M", "description": ""}, {"index": 1003, "label": "4898", "name": "greatbasinrain", "gender": "M", "description": ""}, {"index": 1004, "label": "4899", "name": "Theresa L. Downey", "gender": "F", "description": ""}, {"index": 1005, "label": "489", "name": "Tora", "gender": "F", "description": ""}, {"index": 1006, "label": "4910", "name": "Amanda Martin Sandino", "gender": "F", "description": ""}, {"index": 1007, "label": "4915", "name": "Andrew Bowles", "gender": "M", "description": ""}, {"index": 1008, "label": "4926", "name": "browneyedgirl32382", "gender": "F", "description": ""}, {"index": 1009, "label": "4930", "name": "Adrian Levitsky", "gender": "M", "description": ""}, {"index": 1010, "label": "4931", "name": "KevS", "gender": "M", "description": ""}, {"index": 1011, "label": "4936", "name": "jedopi", "gender": "F", "description": ""}, {"index": 1012, "label": "4948", "name": "Tom Barron", "gender": "M", "description": ""}, {"index": 1013, "label": "4955", "name": "Elwood Mott", "gender": "M", "description": ""}, {"index": 1014, "label": "4958", "name": "David P. Sroka", "gender": "M", "description": ""}, {"index": 1015, "label": "4959", "name": "Piper Hayes", "gender": "F", "description": ""}, {"index": 1016, "label": "4964", "name": "rashada", "gender": "F", "description": ""}, {"index": 1017, "label": "4965", "name": "Rachel Craig", "gender": "F", "description": ""}, {"index": 1018, "label": "4969", "name": "Garth Burton", "gender": "M", "description": ""}, {"index": 1019, "label": "4973", "name": "Michael Lipschultz", "gender": "M", "description": ""}, {"index": 1020, "label": "4979", "name": "Eric", "gender": "M", "description": ""}, {"index": 1021, "label": "497", "name": "audiotoshokan", "gender": "M", "description": ""}, {"index": 1022, "label": "4991", "name": "Jacqueline (Jacqui) Grady", "gender": "F", "description": ""}, {"index": 1023, "label": "4993", "name": "BillMac", "gender": "M", "description": ""}, {"index": 1024, "label": "49", "name": "vrk74", "gender": "M", "description": ""}, {"index": 1025, "label": "5000", "name": "Anna-Lisa Ott", "gender": "F", "description": ""}, {"index": 1026, "label": "5002", "name": "Brendan Stallard", "gender": "M", "description": ""}, {"index": 1027, "label": "5005", "name": "pinhsien", "gender": "F", "description": ""}, {"index": 1028, "label": "5007", "name": "Kathleen Nelson", "gender": "F", "description": ""}, {"index": 1029, "label": "5009", "name": "Michael Reuss", "gender": "M", "description": ""}, {"index": 1030, "label": "5013", "name": "Joseph Dsouza", "gender": "M", "description": ""}, {"index": 1031, "label": "5019", "name": "Sandra Estenson", "gender": "F", "description": ""}, {"index": 1032, "label": "5022", "name": "Kathleen Costa", "gender": "F", "description": ""}, {"index": 1033, "label": "5023", "name": "Elizabeth Zaranka", "gender": "F", "description": ""}, {"index": 1034, "label": "5029", "name": "Courtney Sandhu", "gender": "F", "description": ""}, {"index": 1035, "label": "5036", "name": "NicolaRuth", "gender": "F", "description": ""}, {"index": 1036, "label": "5038", "name": "John Kooz", "gender": "M", "description": ""}, {"index": 1037, "label": "5039", "name": "Inga Parsons", "gender": "F", "description": ""}, {"index": 1038, "label": "5043", "name": "edbucks", "gender": "M", "description": ""}, {"index": 1039, "label": "5044", "name": "Andrea Keene", "gender": "F", "description": ""}, {"index": 1040, "label": "5045", "name": "Mounica", "gender": "F", "description": ""}, {"index": 1041, "label": "5049", "name": "Bradley Smith", "gender": "M", "description": ""}, {"index": 1042, "label": "5054", "name": "Denice Stradling", "gender": "F", "description": ""}, {"index": 1043, "label": "505", "name": "Menno", "gender": "M", "description": ""}, {"index": 1044, "label": "5060", "name": "Linda Moreau", "gender": "F", "description": ""}, {"index": 1045, "label": "5062", "name": "E Ogston", "gender": "F", "description": ""}, {"index": 1046, "label": "5076", "name": "Jules Hawryluk", "gender": "M", "description": ""}, {"index": 1047, "label": "5077", "name": "Sandra G", "gender": "F", "description": ""}, {"index": 1048, "label": "5082", "name": "SummerWind", "gender": "F", "description": ""}, {"index": 1049, "label": "5092", "name": "Josh Smith", "gender": "M", "description": ""}, {"index": 1050, "label": "5093", "name": "Nicole Kay", "gender": "F", "description": ""}, {"index": 1051, "label": "5101", "name": "E.Lee", "gender": "F", "description": ""}, {"index": 1052, "label": "5104", "name": "Chuck Burke", "gender": "M", "description": ""}, {"index": 1053, "label": "5115", "name": "Terry Goodyer", "gender": "F", "description": ""}, {"index": 1054, "label": "5118", "name": "klbonds", "gender": "M", "description": ""}, {"index": 1055, "label": "5123", "name": "wvthcomp", "gender": "M", "description": ""}, {"index": 1056, "label": "5132", "name": "David Stryker", "gender": "M", "description": ""}, {"index": 1057, "label": "5133", "name": "Jo Karabasz", "gender": "F", "description": ""}, {"index": 1058, "label": "5136", "name": "Felicity C", "gender": "F", "description": ""}, {"index": 1059, "label": "5141", "name": "David Goldfarb", "gender": "M", "description": ""}, {"index": 1060, "label": "5147", "name": "Cody2", "gender": "M", "description": ""}, {"index": 1061, "label": "5152", "name": "Liam Neely", "gender": "M", "description": ""}, {"index": 1062, "label": "5154", "name": "Danarra", "gender": "F", "description": ""}, {"index": 1063, "label": "5157", "name": "drewmac", "gender": "M", "description": ""}, {"index": 1064, "label": "5163", "name": "LilyAnne", "gender": "F", "description": ""}, {"index": 1065, "label": "5164", "name": "Vsilverlining", "gender": "F", "description": ""}, {"index": 1066, "label": "5172", "name": "Equilibrium33", "gender": "M", "description": ""}, {"index": 1067, "label": "517", "name": "Matthew Walton", "gender": "M", "description": ""}, {"index": 1068, "label": "5181", "name": "Joel Nisbet", "gender": "M", "description": ""}, {"index": 1069, "label": "5183", "name": "Kenneth Sergeant Gaghan", "gender": "M", "description": ""}, {"index": 1070, "label": "5185", "name": "okei", "gender": "M", "description": ""}, {"index": 1071, "label": "5186", "name": "Brendan Brown", "gender": "M", "description": ""}, {"index": 1072, "label": "5189", "name": "Samanem", "gender": "M", "description": ""}, {"index": 1073, "label": "5190", "name": "Rom Maczka", "gender": "M", "description": ""}, {"index": 1074, "label": "5192", "name": "Jason Esteves", "gender": "M", "description": ""}, {"index": 1075, "label": "5198", "name": "Brian", "gender": "M", "description": ""}, {"index": 1076, "label": "5199", "name": "Diana Fast", "gender": "F", "description": ""}, {"index": 1077, "label": "51", "name": "rakkar", "gender": "M", "description": ""}, {"index": 1078, "label": "5217", "name": "Anqi Wang", "gender": "F", "description": ""}, {"index": 1079, "label": "5220", "name": "Mary Schneider", "gender": "F", "description": ""}, {"index": 1080, "label": "5224", "name": "Matt Soar", "gender": "M", "description": ""}, {"index": 1081, "label": "5230", "name": "Aubrey Anne", "gender": "F", "description": ""}, {"index": 1082, "label": "5233", "name": "summerdaze", "gender": "F", "description": ""}, {"index": 1083, "label": "5239", "name": "compozr", "gender": "M", "description": ""}, {"index": 1084, "label": "5242", "name": "Michael Monhollon", "gender": "M", "description": ""}, {"index": 1085, "label": "5244", "name": "Carolin Ksr", "gender": "F", "description": ""}, {"index": 1086, "label": "5245", "name": "Dymmesdale", "gender": "M", "description": ""}, {"index": 1087, "label": "5246", "name": "Mike Bloomfield", "gender": "M", "description": ""}, {"index": 1088, "label": "5248", "name": "Don Stirno", "gender": "M", "description": ""}, {"index": 1089, "label": "5252", "name": "Michael Deng", "gender": "M", "description": ""}, {"index": 1090, "label": "5261", "name": "Ali Kazerani", "gender": "M", "description": ""}, {"index": 1091, "label": "5269", "name": "Elizabeth Barr", "gender": "F", "description": ""}, {"index": 1092, "label": "5271", "name": "Amanda", "gender": "F", "description": ""}, {"index": 1093, "label": "5278", "name": "Mike Harris", "gender": "M", "description": ""}, {"index": 1094, "label": "5280", "name": "penboy7000", "gender": "M", "description": ""}, {"index": 1095, "label": "5285", "name": "Sweetlilbirdy", "gender": "F", "description": ""}, {"index": 1096, "label": "5287", "name": "adsum iam", "gender": "F", "description": ""}, {"index": 1097, "label": "5290", "name": "Natalie", "gender": "F", "description": ""}, {"index": 1098, "label": "5293", "name": "Ned Troxel", "gender": "M", "description": ""}, {"index": 1099, "label": "5296", "name": "Joselyn Hasty", "gender": "F", "description": ""}, {"index": 1100, "label": "5299", "name": "Becky Doughty", "gender": "F", "description": ""}, {"index": 1101, "label": "52", "name": "Luisa Hall", "gender": "F", "description": ""}, {"index": 1102, "label": "5304", "name": "Don W. Jenkins", "gender": "M", "description": ""}, {"index": 1103, "label": "5321", "name": "Adam Whybray", "gender": "M", "description": ""}, {"index": 1104, "label": "5322", "name": "Jay Bidal", "gender": "M", "description": ""}, {"index": 1105, "label": "5325", "name": "Rory Lawton", "gender": "M", "description": ""}, {"index": 1106, "label": "5328", "name": "Nathan Jordan", "gender": "M", "description": ""}, {"index": 1107, "label": "5333", "name": "Robert Fletcher", "gender": "M", "description": ""}, {"index": 1108, "label": "5337", "name": "Danielle", "gender": "F", "description": ""}, {"index": 1109, "label": "5339", "name": "Lauren McCullough", "gender": "F", "description": ""}, {"index": 1110, "label": "5340", "name": "Mary-Beth Blackburn", "gender": "F", "description": ""}, {"index": 1111, "label": "534", "name": "Jean O\'Sullivan", "gender": "F", "description": ""}, {"index": 1112, "label": "5350", "name": "Cath Garde", "gender": "F", "description": ""}, {"index": 1113, "label": "5355", "name": "Carrie Heyes", "gender": "F", "description": ""}, {"index": 1114, "label": "5361", "name": "scarlettraces", "gender": "F", "description": ""}, {"index": 1115, "label": "5375", "name": "BumbleVee", "gender": "F", "description": ""}, {"index": 1116, "label": "5379", "name": "lennich", "gender": "M", "description": ""}, {"index": 1117, "label": "5386", "name": "Tim Ferreira", "gender": "M", "description": ""}, {"index": 1118, "label": "5389", "name": "Joseph Lawler", "gender": "M", "description": ""}, {"index": 1119, "label": "5390", "name": "Charles Bice", "gender": "M", "description": ""}, {"index": 1120, "label": "5393", "name": "Amy Hengst", "gender": "F", "description": ""}, {"index": 1121, "label": "5400", "name": "Natalie Sullivan", "gender": "F", "description": ""}, {"index": 1122, "label": "5401", "name": "Andrew Nelson", "gender": "M", "description": ""}, {"index": 1123, "label": "5405", "name": "Anka", "gender": "F", "description": ""}, {"index": 1124, "label": "5412", "name": "tommybascue", "gender": "F", "description": ""}, {"index": 1125, "label": "5424", "name": "Julia Niedermaier", "gender": "F", "description": ""}, {"index": 1126, "label": "5429", "name": "Dennis Lane Pretoria", "gender": "M", "description": ""}, {"index": 1127, "label": "542", "name": "J. Hall", "gender": "M", "description": ""}, {"index": 1128, "label": "5439", "name": "Laine S.", "gender": "F", "description": ""}, {"index": 1129, "label": "5445", "name": "Liz DeLassus", "gender": "F", "description": ""}, {"index": 1130, "label": "544", "name": "bozgeez", "gender": "M", "description": ""}, {"index": 1131, "label": "5456", "name": "e_scarab", "gender": "M", "description": ""}, {"index": 1132, "label": "5459", "name": "bryan.peterson", "gender": "M", "description": ""}, {"index": 1133, "label": "5460", "name": "Max Lindberg", "gender": "M", "description": ""}, {"index": 1134, "label": "5463", "name": "GLM", "gender": "M", "description": ""}, {"index": 1135, "label": "5468", "name": "ashleighjane", "gender": "F", "description": ""}, {"index": 1136, "label": "5471", "name": "ElleyKat", "gender": "F", "description": ""}, {"index": 1137, "label": "5480", "name": "ESFJ Girl", "gender": "F", "description": ""}, {"index": 1138, "label": "5487", "name": "Michaela O\'Connor", "gender": "F", "description": ""}, {"index": 1139, "label": "5506", "name": "Frank Booker", "gender": "M", "description": ""}, {"index": 1140, "label": "5513", "name": "David Callahan", "gender": "M", "description": ""}, {"index": 1141, "label": "5514", "name": "Ella Jane Quentin", "gender": "F", "description": ""}, {"index": 1142, "label": "551", "name": "Guntar", "gender": "M", "description": ""}, {"index": 1143, "label": "5538", "name": "Linda Velwest", "gender": "F", "description": ""}, {"index": 1144, "label": "5545", "name": "Gary Olman", "gender": "M", "description": ""}, {"index": 1145, "label": "5561", "name": "Ellen Jones", "gender": "F", "description": ""}, {"index": 1146, "label": "5565", "name": "Ken Padgitt", "gender": "M", "description": ""}, {"index": 1147, "label": "5567", "name": "BStapley", "gender": "M", "description": ""}, {"index": 1148, "label": "5569", "name": "Beth Hitesman", "gender": "F", "description": ""}, {"index": 1149, "label": "557", "name": "fieldsofgold", "gender": "M", "description": ""}, {"index": 1150, "label": "5583", "name": "Claudia Wilson", "gender": "F", "description": ""}, {"index": 1151, "label": "55", "name": "David Jaquay", "gender": "M", "description": ""}, {"index": 1152, "label": "5604", "name": "kittyandcheese", "gender": "F", "description": ""}, {"index": 1153, "label": "5606", "name": "PrincessG", "gender": "F", "description": ""}, {"index": 1154, "label": "5620", "name": "jerryB", "gender": "M", "description": ""}, {"index": 1155, "label": "5628", "name": "Adrian Wheal", "gender": "M", "description": ""}, {"index": 1156, "label": "5635", "name": "Matthew Reece", "gender": "M", "description": ""}, {"index": 1157, "label": "5636", "name": "Joyce Couch", "gender": "F", "description": ""}, {"index": 1158, "label": "5641", "name": "Gerald Peter Morgan", "gender": "M", "description": ""}, {"index": 1159, "label": "5649", "name": "hazelra", "gender": "F", "description": ""}, {"index": 1160, "label": "5652", "name": "amicrazy2u", "gender": "F", "description": ""}, {"index": 1161, "label": "5653", "name": "Kimberly Anderson", "gender": "F", "description": ""}, {"index": 1162, "label": "5660", "name": "hollz", "gender": "F", "description": ""}, {"index": 1163, "label": "5661", "name": "Dianne Lanning", "gender": "F", "description": ""}, {"index": 1164, "label": "5665", "name": "Lynda Sizemore", "gender": "F", "description": ""}, {"index": 1165, "label": "5671", "name": "Trihypoo", "gender": "F", "description": ""}, {"index": 1166, "label": "5678", "name": "jgoffena", "gender": "M", "description": ""}, {"index": 1167, "label": "567", "name": "Aaron Benedict", "gender": "M", "description": ""}, {"index": 1168, "label": "5682", "name": "Lmwong", "gender": "F", "description": ""}, {"index": 1169, "label": "5684", "name": "Elsa Youngsteadt", "gender": "F", "description": ""}, {"index": 1170, "label": "5688", "name": "Jennifer Dionne", "gender": "F", "description": ""}, {"index": 1171, "label": "568", "name": "JD Weber", "gender": "M", "description": ""}, {"index": 1172, "label": "569", "name": "Frank", "gender": "M", "description": ""}, {"index": 1173, "label": "56", "name": "Kirsten Ferreri", "gender": "F", "description": ""}, {"index": 1174, "label": "5700", "name": "Paul Adams", "gender": "M", "description": ""}, {"index": 1175, "label": "5703", "name": "Garth Comira", "gender": "M", "description": ""}, {"index": 1176, "label": "5717", "name": "Phil Chenevert", "gender": "M", "description": ""}, {"index": 1177, "label": "5719", "name": "John Fricker", "gender": "M", "description": ""}, {"index": 1178, "label": "5720", "name": "Thelma Meyer", "gender": "F", "description": ""}, {"index": 1179, "label": "5723", "name": "Linda Hogan", "gender": "F", "description": ""}, {"index": 1180, "label": "5724", "name": "paintgirl", "gender": "F", "description": ""}, {"index": 1181, "label": "5725", "name": "doublemirrors", "gender": "F", "description": ""}, {"index": 1182, "label": "5727", "name": "JDavidMoore", "gender": "M", "description": ""}, {"index": 1183, "label": "572", "name": "Rebecca Dittman", "gender": "M", "description": ""}, {"index": 1184, "label": "5733", "name": "Spinhop", "gender": "F", "description": ""}, {"index": 1185, "label": "5735", "name": "Filipa", "gender": "F", "description": ""}, {"index": 1186, "label": "5740", "name": "Alisa", "gender": "F", "description": ""}, {"index": 1187, "label": "5746", "name": "DL Pead", "gender": "M", "description": ""}, {"index": 1188, "label": "5750", "name": "laurencetrask", "gender": "M", "description": ""}, {"index": 1189, "label": "5756", "name": "Mich_elle", "gender": "F", "description": ""}, {"index": 1190, "label": "5765", "name": "A. J. Carroll", "gender": "M", "description": ""}, {"index": 1191, "label": "5767", "name": "taijohn", "gender": "M", "description": ""}, {"index": 1192, "label": "576", "name": "Caroline Mercier", "gender": "F", "description": ""}, {"index": 1193, "label": "5772", "name": "jessecoy", "gender": "M", "description": ""}, {"index": 1194, "label": "5776", "name": "Chris Pauley", "gender": "M", "description": ""}, {"index": 1195, "label": "5778", "name": "Laura Victoria", "gender": "F", "description": ""}, {"index": 1196, "label": "5781", "name": "Parrot17", "gender": "M", "description": ""}, {"index": 1197, "label": "5784", "name": "Chris Donnelly", "gender": "M", "description": ""}, {"index": 1198, "label": "5789", "name": "Kirsten Wever", "gender": "F", "description": ""}, {"index": 1199, "label": "5791", "name": "croudy", "gender": "M", "description": ""}, {"index": 1200, "label": "5796", "name": "Linette Geisel", "gender": "F", "description": ""}, {"index": 1201, "label": "57", "name": "Ophelia Darcy", "gender": "F", "description": ""}, {"index": 1202, "label": "5802", "name": "Christopher Maust", "gender": "M", "description": ""}, {"index": 1203, "label": "5808", "name": "jeandelfrio", "gender": "M", "description": ""}, {"index": 1204, "label": "5809", "name": "kattekliek", "gender": "F", "description": ""}, {"index": 1205, "label": "580", "name": "Ryan", "gender": "M", "description": ""}, {"index": 1206, "label": "5810", "name": "Patrick Reinhart", "gender": "M", "description": ""}, {"index": 1207, "label": "581", "name": "C. Berrius", "gender": "M", "description": ""}, {"index": 1208, "label": "5825", "name": "Vira Denton", "gender": "F", "description": ""}, {"index": 1209, "label": "5826", "name": "Phineas Redux", "gender": "M", "description": ""}, {"index": 1210, "label": "5831", "name": "Kirsty Leishman", "gender": "F", "description": ""}, {"index": 1211, "label": "5837", "name": "Debbie Pieterse", "gender": "F", "description": ""}, {"index": 1212, "label": "583", "name": "Russ Maxwell", "gender": "M", "description": ""}, {"index": 1213, "label": "5840", "name": "Dick Summerfield", "gender": "M", "description": ""}, {"index": 1214, "label": "584", "name": "miette", "gender": "F", "description": ""}, {"index": 1215, "label": "5854", "name": "Gabriel Glenn", "gender": "M", "description": ""}, {"index": 1216, "label": "585", "name": "pheo", "gender": "F", "description": ""}, {"index": 1217, "label": "5860", "name": "J L Raimundo", "gender": "F", "description": ""}, {"index": 1218, "label": "5867", "name": "Sharon Omi", "gender": "F", "description": ""}, {"index": 1219, "label": "5874", "name": "Paul Andrews", "gender": "M", "description": ""}, {"index": 1220, "label": "5876", "name": "kelleywyskiel", "gender": "F", "description": ""}, {"index": 1221, "label": "587", "name": "Joy Scaglione", "gender": "F", "description": ""}, {"index": 1222, "label": "5886", "name": "Simon Brown", "gender": "M", "description": ""}, {"index": 1223, "label": "5890", "name": "GabrielleC", "gender": "F", "description": ""}, {"index": 1224, "label": "5893", "name": "Elizabeth Fiedler", "gender": "F", "description": ""}, {"index": 1225, "label": "5894", "name": "Paul Stephens", "gender": "M", "description": ""}, {"index": 1226, "label": "589", "name": "Stephanie Konig", "gender": "F", "description": ""}, {"index": 1227, "label": "58", "name": "George Coutts", "gender": "M", "description": ""}, {"index": 1228, "label": "5906", "name": "Lily-LLM", "gender": "F", "description": ""}, {"index": 1229, "label": "5909", "name": "Mark Mickelson", "gender": "M", "description": ""}, {"index": 1230, "label": "5910", "name": "Anthony Lee", "gender": "M", "description": ""}, {"index": 1231, "label": "5911", "name": "Paul C. Newman", "gender": "M", "description": ""}, {"index": 1232, "label": "5913", "name": "dmbrought", "gender": "M", "description": ""}, {"index": 1233, "label": "5914", "name": "bobbybrill", "gender": "M", "description": ""}, {"index": 1234, "label": "5929", "name": "Gryphon Perkins", "gender": "F", "description": ""}, {"index": 1235, "label": "5933", "name": "Barry O\'Neill", "gender": "M", "description": ""}, {"index": 1236, "label": "5935", "name": "Mike Okonek", "gender": "M", "description": ""}, {"index": 1237, "label": "593", "name": "Eric S. Piotrowski", "gender": "M", "description": ""}, {"index": 1238, "label": "5940", "name": "Gargoyle", "gender": "M", "description": ""}, {"index": 1239, "label": "5949", "name": "debolee", "gender": "F", "description": ""}, {"index": 1240, "label": "594", "name": "KentF", "gender": "M", "description": ""}, {"index": 1241, "label": "5951", "name": "Stevan Simmons", "gender": "M", "description": ""}, {"index": 1242, "label": "5952", "name": "sherlock85", "gender": "F", "description": ""}, {"index": 1243, "label": "5968", "name": "Cate Barratt", "gender": "F", "description": ""}, {"index": 1244, "label": "5970", "name": "Larry Degala", "gender": "M", "description": ""}, {"index": 1245, "label": "5975", "name": "Tony Maxey", "gender": "M", "description": ""}, {"index": 1246, "label": "5977", "name": "Ken Felt", "gender": "M", "description": ""}, {"index": 1247, "label": "5979", "name": "Pentti Hirvonen", "gender": "M", "description": ""}, {"index": 1248, "label": "597", "name": "Lisa Chau", "gender": "F", "description": ""}, {"index": 1249, "label": "5980", "name": "Nichole Thompson", "gender": "F", "description": ""}, {"index": 1250, "label": "5983", "name": "Katew", "gender": "F", "description": ""}, {"index": 1251, "label": "598", "name": "Kim", "gender": "F", "description": ""}, {"index": 1252, "label": "5993", "name": "garyday", "gender": "M", "description": ""}, {"index": 1253, "label": "6000", "name": "MissRose", "gender": "F", "description": ""}, {"index": 1254, "label": "6003", "name": "sparks0314", "gender": "M", "description": ""}, {"index": 1255, "label": "6006", "name": "Stephanie Lee", "gender": "F", "description": ""}, {"index": 1256, "label": "6009", "name": "AVG", "gender": "M", "description": ""}, {"index": 1257, "label": "6010", "name": "Ralph Kerwin", "gender": "M", "description": ""}, {"index": 1258, "label": "6014", "name": "Tina Nuzzi", "gender": "F", "description": ""}, {"index": 1259, "label": "6019", "name": "DerekP", "gender": "M", "description": ""}, {"index": 1260, "label": "6025", "name": "Lonelle Yoder", "gender": "F", "description": ""}, {"index": 1261, "label": "6030", "name": "Shiromi", "gender": "F", "description": ""}, {"index": 1262, "label": "6035", "name": "Marty Kris", "gender": "M", "description": ""}, {"index": 1263, "label": "6051", "name": "Ric Cornwall", "gender": "M", "description": ""}, {"index": 1264, "label": "6060", "name": "Jonathan Lange", "gender": "M", "description": ""}, {"index": 1265, "label": "6064", "name": "Deborah Knight", "gender": "F", "description": ""}, {"index": 1266, "label": "6065", "name": "David Olson", "gender": "M", "description": ""}, {"index": 1267, "label": "6072", "name": "CrowGirl", "gender": "F", "description": ""}, {"index": 1268, "label": "6076", "name": "Saethram", "gender": "M", "description": ""}, {"index": 1269, "label": "6077", "name": "Steve Belleguelle", "gender": "M", "description": ""}, {"index": 1270, "label": "6078", "name": "dobsonfly", "gender": "F", "description": ""}, {"index": 1271, "label": "6080", "name": "progressingamerica", "gender": "M", "description": ""}, {"index": 1272, "label": "6081", "name": "Lazuli", "gender": "M", "description": ""}, {"index": 1273, "label": "6084", "name": "Amallen", "gender": "M", "description": ""}, {"index": 1274, "label": "6087", "name": "Caroline Driggs", "gender": "F", "description": ""}, {"index": 1275, "label": "6088", "name": "burk", "gender": "F", "description": ""}, {"index": 1276, "label": "608", "name": "Baranduin", "gender": "F", "description": ""}, {"index": 1277, "label": "6098", "name": "Kancamagus", "gender": "M", "description": ""}, {"index": 1278, "label": "6099", "name": "Cheri Gardner", "gender": "F", "description": ""}, {"index": 1279, "label": "60", "name": "CBW Simon", "gender": "M", "description": ""}, {"index": 1280, "label": "6102", "name": "Rachel Bossier", "gender": "F", "description": ""}, {"index": 1281, "label": "6106", "name": "Ruth Kidson", "gender": "F", "description": ""}, {"index": 1282, "label": "6111", "name": "Jill Preston", "gender": "F", "description": ""}, {"index": 1283, "label": "6119", "name": "Vinnie Tesla", "gender": "M", "description": ""}, {"index": 1284, "label": "6120", "name": "Terra Mendoza", "gender": "F", "description": ""}, {"index": 1285, "label": "6121", "name": "Imagine", "gender": "F", "description": ""}, {"index": 1286, "label": "6126", "name": "David Abbott", "gender": "M", "description": ""}, {"index": 1287, "label": "6127", "name": "Andee", "gender": "F", "description": ""}, {"index": 1288, "label": "6131", "name": "Malane", "gender": "F", "description": ""}, {"index": 1289, "label": "6135", "name": "Spike Holcomb", "gender": "F", "description": ""}, {"index": 1290, "label": "6138", "name": "Deborah Brabyn", "gender": "F", "description": ""}, {"index": 1291, "label": "6139", "name": "Lois C. Johnson", "gender": "F", "description": ""}, {"index": 1292, "label": "613", "name": "D.E. Wittkower", "gender": "M", "description": ""}, {"index": 1293, "label": "6145", "name": "Madeleine Brook", "gender": "F", "description": ""}, {"index": 1294, "label": "6147", "name": "Liberty Stump", "gender": "F", "description": ""}, {"index": 1295, "label": "614", "name": "Christine Blachford", "gender": "F", "description": ""}, {"index": 1296, "label": "6153", "name": "anjieliu", "gender": "F", "description": ""}, {"index": 1297, "label": "6159", "name": "Ted Garvin", "gender": "M", "description": ""}, {"index": 1298, "label": "6173", "name": "Carol Box", "gender": "F", "description": ""}, {"index": 1299, "label": "6177", "name": "wminbru", "gender": "M", "description": ""}, {"index": 1300, "label": "6178", "name": "alegriavida", "gender": "F", "description": ""}, {"index": 1301, "label": "6181", "name": "Mike", "gender": "M", "description": ""}, {"index": 1302, "label": "6184", "name": "Adam Tomkins", "gender": "M", "description": ""}, {"index": 1303, "label": "6188", "name": "Roxanna Nazari", "gender": "F", "description": ""}, {"index": 1304, "label": "6196", "name": "Martin Gradwell", "gender": "M", "description": ""}, {"index": 1305, "label": "6199", "name": "bobolink", "gender": "M", "description": ""}, {"index": 1306, "label": "6209", "name": "deckerteach", "gender": "M", "description": ""}, {"index": 1307, "label": "6211", "name": "Caeliveres", "gender": "F", "description": ""}, {"index": 1308, "label": "6221", "name": "pwu909", "gender": "M", "description": ""}, {"index": 1309, "label": "6224", "name": "dave k", "gender": "M", "description": ""}, {"index": 1310, "label": "6227", "name": "EliMarieHK", "gender": "F", "description": ""}, {"index": 1311, "label": "622", "name": "Andrew Lebrun", "gender": "M", "description": ""}, {"index": 1312, "label": "6232", "name": "Paul Denton", "gender": "M", "description": ""}, {"index": 1313, "label": "6233", "name": "Gen Jones", "gender": "F", "description": ""}, {"index": 1314, "label": "6235", "name": "ReadWriteLib", "gender": "F", "description": ""}, {"index": 1315, "label": "6242", "name": "Lucinda Gainey", "gender": "F", "description": ""}, {"index": 1316, "label": "6248", "name": "Kevin Green", "gender": "M", "description": ""}, {"index": 1317, "label": "6249", "name": "Thomas A. Copeland", "gender": "M", "description": ""}, {"index": 1318, "label": "6251", "name": "Theresa Sheridan", "gender": "F", "description": ""}, {"index": 1319, "label": "6254", "name": "Chelsea S.", "gender": "F", "description": ""}, {"index": 1320, "label": "625", "name": "toriasuncle", "gender": "M", "description": ""}, {"index": 1321, "label": "6272", "name": "jlenardon", "gender": "F", "description": ""}, {"index": 1322, "label": "6276", "name": "dan_h", "gender": "M", "description": ""}, {"index": 1323, "label": "6281", "name": "Beth Thomas", "gender": "F", "description": ""}, {"index": 1324, "label": "6284", "name": "cklee", "gender": "F", "description": ""}, {"index": 1325, "label": "6286", "name": "Wendy Almeida", "gender": "F", "description": ""}, {"index": 1326, "label": "6288", "name": "Sunni West", "gender": "F", "description": ""}, {"index": 1327, "label": "6294", "name": "humanode", "gender": "M", "description": ""}, {"index": 1328, "label": "62", "name": "Faris", "gender": "M", "description": ""}, {"index": 1329, "label": "6300", "name": "Sarah Crampton", "gender": "F", "description": ""}, {"index": 1330, "label": "6308", "name": "Easton", "gender": "F", "description": ""}, {"index": 1331, "label": "6311", "name": "TRUEBRIT", "gender": "M", "description": ""}, {"index": 1332, "label": "6317", "name": "Sarah K", "gender": "F", "description": ""}, {"index": 1333, "label": "6323", "name": "Caluminium", "gender": "M", "description": ""}, {"index": 1334, "label": "6324", "name": "Barbara E. McCarthy", "gender": "F", "description": ""}, {"index": 1335, "label": "6330", "name": "David Cummings", "gender": "M", "description": ""}, {"index": 1336, "label": "6332", "name": "Kelli England", "gender": "F", "description": ""}, {"index": 1337, "label": "6333", "name": "Max Korlinge", "gender": "M", "description": ""}, {"index": 1338, "label": "6339", "name": "Michelle Remington", "gender": "F", "description": ""}, {"index": 1339, "label": "6346", "name": "Matthew Pagan", "gender": "M", "description": ""}, {"index": 1340, "label": "6351", "name": "C. L. W. Rollins", "gender": "F", "description": ""}, {"index": 1341, "label": "6352", "name": "stmacduff", "gender": "F", "description": ""}, {"index": 1342, "label": "6353", "name": "Jennifer Randall", "gender": "F", "description": ""}, {"index": 1343, "label": "6356", "name": "John O", "gender": "M", "description": ""}, {"index": 1344, "label": "6358", "name": "selway", "gender": "F", "description": ""}, {"index": 1345, "label": "6364", "name": "Todd Ulbrich", "gender": "M", "description": ""}, {"index": 1346, "label": "6367", "name": "Vince Dee", "gender": "M", "description": ""}, {"index": 1347, "label": "6368", "name": "Marc Pizzuti", "gender": "M", "description": ""}, {"index": 1348, "label": "636", "name": "Zale Schafer (Rose May Chamberlin Memorial Foundat", "gender": "F", "description": ""}, {"index": 1349, "label": "6370", "name": "Dan Craig", "gender": "M", "description": ""}, {"index": 1350, "label": "6377", "name": "Maxim Babich", "gender": "M", "description": ""}, {"index": 1351, "label": "6378", "name": "Michelle Day", "gender": "F", "description": ""}, {"index": 1352, "label": "637", "name": "Michael Scherer", "gender": "M", "description": ""}, {"index": 1353, "label": "6385", "name": "Novella Serena", "gender": "F", "description": ""}, {"index": 1354, "label": "6391", "name": "polkadotish", "gender": "F", "description": ""}, {"index": 1355, "label": "6395", "name": "Richard Carpenter", "gender": "M", "description": ""}, {"index": 1356, "label": "6399", "name": "Edmund Bloxam", "gender": "M", "description": ""}, {"index": 1357, "label": "639", "name": "Robert Beach", "gender": "M", "description": ""}, {"index": 1358, "label": "6402", "name": "Chiquito Crasto", "gender": "M", "description": ""}, {"index": 1359, "label": "6406", "name": "Abigail Rasmussen", "gender": "F", "description": ""}, {"index": 1360, "label": "6407", "name": "AdamH", "gender": "M", "description": ""}, {"index": 1361, "label": "6411", "name": "Feyaza", "gender": "F", "description": ""}, {"index": 1362, "label": "6415", "name": "Daryl Wor", "gender": "F", "description": ""}, {"index": 1363, "label": "6418", "name": "Iolo Jones", "gender": "M", "description": ""}, {"index": 1364, "label": "6436", "name": "Steve Mattingly", "gender": "M", "description": ""}, {"index": 1365, "label": "6437", "name": "John Hoerr", "gender": "M", "description": ""}, {"index": 1366, "label": "6446", "name": "Bob Gonzalez", "gender": "M", "description": ""}, {"index": 1367, "label": "644", "name": "Daniele", "gender": "M", "description": ""}, {"index": 1368, "label": "6454", "name": "David Wales", "gender": "M", "description": ""}, {"index": 1369, "label": "6458", "name": "Dennis Blake", "gender": "M", "description": ""}, {"index": 1370, "label": "6459", "name": "Nigel Boydell", "gender": "M", "description": ""}, {"index": 1371, "label": "6476", "name": "Viridian", "gender": "F", "description": ""}, {"index": 1372, "label": "6482", "name": "Delmar H Dolbier", "gender": "M", "description": ""}, {"index": 1373, "label": "6484", "name": "ilianthe", "gender": "F", "description": ""}, {"index": 1374, "label": "6488", "name": "Kendall Ashyby", "gender": "F", "description": ""}, {"index": 1375, "label": "6492", "name": "Hugh Gillis", "gender": "M", "description": ""}, {"index": 1376, "label": "6494", "name": "Morey Kunin", "gender": "M", "description": ""}, {"index": 1377, "label": "6499", "name": "dave7", "gender": "M", "description": ""}, {"index": 1378, "label": "64", "name": "Robin Cotter", "gender": "F", "description": ""}, {"index": 1379, "label": "6505", "name": "Elena", "gender": "F", "description": ""}, {"index": 1380, "label": "6506", "name": "Tim Quinn", "gender": "M", "description": ""}, {"index": 1381, "label": "6512", "name": "mcgovern1934", "gender": "M", "description": ""}, {"index": 1382, "label": "6513", "name": "Susan de Raadt", "gender": "F", "description": ""}, {"index": 1383, "label": "6529", "name": "Fred DeBerardinis", "gender": "M", "description": ""}, {"index": 1384, "label": "6531", "name": "janesandberg", "gender": "F", "description": ""}, {"index": 1385, "label": "6533", "name": "drewmore", "gender": "M", "description": ""}, {"index": 1386, "label": "6534", "name": "Helen Falconer", "gender": "F", "description": ""}, {"index": 1387, "label": "6535", "name": "Ron Altman", "gender": "M", "description": ""}, {"index": 1388, "label": "6538", "name": "Juan Federico", "gender": "M", "description": ""}, {"index": 1389, "label": "6539", "name": "David W. Wolfe", "gender": "M", "description": ""}, {"index": 1390, "label": "6540", "name": "Craig Gulliver", "gender": "M", "description": ""}, {"index": 1391, "label": "6548", "name": "Kristin Gjerlw", "gender": "F", "description": ""}, {"index": 1392, "label": "6549", "name": "Sandra Luna", "gender": "F", "description": ""}, {"index": 1393, "label": "6550", "name": "elfpen", "gender": "M", "description": ""}, {"index": 1394, "label": "6553", "name": "DublinGothic", "gender": "M", "description": ""}, {"index": 1395, "label": "6557", "name": "Tika Sabu", "gender": "F", "description": ""}, {"index": 1396, "label": "6563", "name": "William Tomcho", "gender": "M", "description": ""}, {"index": 1397, "label": "6567", "name": "vikvenom", "gender": "F", "description": ""}, {"index": 1398, "label": "6568", "name": "Heather Hamtil", "gender": "F", "description": ""}, {"index": 1399, "label": "6574", "name": "Caden Vaughn Clegg", "gender": "M", "description": ""}, {"index": 1400, "label": "6575", "name": "Patrick Painter", "gender": "M", "description": ""}, {"index": 1401, "label": "6583", "name": "NastassiaS", "gender": "F", "description": ""}, {"index": 1402, "label": "6590", "name": "Marea Brook", "gender": "F", "description": ""}, {"index": 1403, "label": "6594", "name": "Amelia Chantarotwong", "gender": "F", "description": ""}, {"index": 1404, "label": "65", "name": "chriss the girl", "gender": "F", "description": ""}, {"index": 1405, "label": "6609", "name": "Patrick Wallace", "gender": "M", "description": ""}, {"index": 1406, "label": "6610", "name": "sylly", "gender": "F", "description": ""}, {"index": 1407, "label": "6614", "name": "Veronica Schlette", "gender": "F", "description": ""}, {"index": 1408, "label": "6620", "name": "Amy Koenig", "gender": "F", "description": ""}, {"index": 1409, "label": "6625", "name": "hearmeout7", "gender": "M", "description": ""}, {"index": 1410, "label": "6627", "name": "Amber Hamilton", "gender": "F", "description": ""}, {"index": 1411, "label": "6636", "name": "Philip Martin", "gender": "M", "description": ""}, {"index": 1412, "label": "6637", "name": "Christine Nendza", "gender": "F", "description": ""}, {"index": 1413, "label": "663", "name": "Bruce Stafford", "gender": "M", "description": ""}, {"index": 1414, "label": "6643", "name": "cbooren", "gender": "F", "description": ""}, {"index": 1415, "label": "664", "name": "Wendy G.", "gender": "F", "description": ""}, {"index": 1416, "label": "6652", "name": "Brendon Wright", "gender": "M", "description": ""}, {"index": 1417, "label": "6660", "name": "Nicole Lee", "gender": "F", "description": ""}, {"index": 1418, "label": "6668", "name": "Ecological Humanist", "gender": "M", "description": ""}, {"index": 1419, "label": "6673", "name": "Jenna Lanman", "gender": "F", "description": ""}, {"index": 1420, "label": "6674", "name": "jimmylee", "gender": "M", "description": ""}, {"index": 1421, "label": "6676", "name": "Chill28", "gender": "F", "description": ""}, {"index": 1422, "label": "667", "name": "Bethany Simpson", "gender": "F", "description": ""}, {"index": 1423, "label": "6685", "name": "Jonathan Drury", "gender": "M", "description": ""}, {"index": 1424, "label": "6686", "name": "Elin", "gender": "M", "description": ""}, {"index": 1425, "label": "6687", "name": "KirksVoice", "gender": "M", "description": ""}, {"index": 1426, "label": "6689", "name": "Tiffany Halla Colonna", "gender": "F", "description": ""}, {"index": 1427, "label": "6690", "name": "T Michael Burke", "gender": "M", "description": ""}, {"index": 1428, "label": "6694", "name": "Ross Williamson", "gender": "M", "description": ""}, {"index": 1429, "label": "6695", "name": "Linda Fredericks", "gender": "F", "description": ""}, {"index": 1430, "label": "669", "name": "Anne", "gender": "F", "description": ""}, {"index": 1431, "label": "66", "name": "Alex Foster", "gender": "M", "description": ""}, {"index": 1432, "label": "6701", "name": "Dayle", "gender": "M", "description": ""}, {"index": 1433, "label": "6705", "name": "D. A. Frank", "gender": "F", "description": ""}, {"index": 1434, "label": "6707", "name": "Lewis", "gender": "M", "description": ""}, {"index": 1435, "label": "6709", "name": "vinphizz", "gender": "M", "description": ""}, {"index": 1436, "label": "6713", "name": "stephenreader", "gender": "M", "description": ""}, {"index": 1437, "label": "6724", "name": "Kristin Young", "gender": "F", "description": ""}, {"index": 1438, "label": "6726", "name": "metgmz", "gender": "F", "description": ""}, {"index": 1439, "label": "6727", "name": "Tony Russell", "gender": "M", "description": ""}, {"index": 1440, "label": "6733", "name": "Snapdragon", "gender": "F", "description": ""}, {"index": 1441, "label": "6735", "name": "fshort", "gender": "F", "description": ""}, {"index": 1442, "label": "6741", "name": "Sharon C.", "gender": "F", "description": ""}, {"index": 1443, "label": "6743", "name": "Rick Rodney", "gender": "M", "description": ""}, {"index": 1444, "label": "6746", "name": "Robin Skelcey", "gender": "M", "description": ""}, {"index": 1445, "label": "6747", "name": "Ryan Lothian", "gender": "M", "description": ""}, {"index": 1446, "label": "6749", "name": "MaryA", "gender": "F", "description": ""}, {"index": 1447, "label": "6752", "name": "maxvon_d", "gender": "M", "description": ""}, {"index": 1448, "label": "6753", "name": "T.E. McHenry", "gender": "M", "description": ""}, {"index": 1449, "label": "6754", "name": "ToddHW", "gender": "M", "description": ""}, {"index": 1450, "label": "6758", "name": "The Gypsy", "gender": "F", "description": ""}, {"index": 1451, "label": "6773", "name": "MostafaRazavi", "gender": "M", "description": ""}, {"index": 1452, "label": "6777", "name": "Rick Saffery", "gender": "M", "description": ""}, {"index": 1453, "label": "6784", "name": "SteveBuys", "gender": "M", "description": ""}, {"index": 1454, "label": "6788", "name": "Pamela Krantz", "gender": "F", "description": ""}, {"index": 1455, "label": "6792", "name": "montmorency", "gender": "M", "description": ""}, {"index": 1456, "label": "6794", "name": "Rachel Moyar", "gender": "F", "description": ""}, {"index": 1457, "label": "6798", "name": "Aesthete\'s Readings", "gender": "M", "description": ""}, {"index": 1458, "label": "679", "name": "rhodian", "gender": "M", "description": ""}, {"index": 1459, "label": "6804", "name": "Nick Duncan", "gender": "M", "description": ""}, {"index": 1460, "label": "6807", "name": "Lisa Caputo", "gender": "F", "description": ""}, {"index": 1461, "label": "6818", "name": "beckyboyd", "gender": "F", "description": ""}, {"index": 1462, "label": "681", "name": "Lucy Burgoyne", "gender": "F", "description": ""}, {"index": 1463, "label": "6821", "name": "Rholdah", "gender": "F", "description": ""}, {"index": 1464, "label": "6836", "name": "John", "gender": "M", "description": ""}, {"index": 1465, "label": "6846", "name": "John Leonard", "gender": "M", "description": ""}, {"index": 1466, "label": "6848", "name": "KarlHenning", "gender": "M", "description": ""}, {"index": 1467, "label": "6849", "name": "Dan Raynham", "gender": "M", "description": ""}, {"index": 1468, "label": "684", "name": "Lizzie Driver", "gender": "F", "description": ""}, {"index": 1469, "label": "6853", "name": "J. McKnight", "gender": "F", "description": ""}, {"index": 1470, "label": "6865", "name": "Jing Li", "gender": "F", "description": ""}, {"index": 1471, "label": "6875", "name": "Bill Miller", "gender": "M", "description": ""}, {"index": 1472, "label": "6877", "name": "Bear Schacht", "gender": "M", "description": ""}, {"index": 1473, "label": "6880", "name": "Capybara", "gender": "M", "description": ""}, {"index": 1474, "label": "6882", "name": "David Isenhower", "gender": "M", "description": ""}, {"index": 1475, "label": "6883", "name": "Adam Doughty", "gender": "M", "description": ""}, {"index": 1476, "label": "6892", "name": "Piotr Nater", "gender": "M", "description": ""}, {"index": 1477, "label": "6895", "name": "Reeses118", "gender": "F", "description": ""}, {"index": 1478, "label": "6902", "name": "Barbara Edelman", "gender": "F", "description": ""}, {"index": 1479, "label": "6906", "name": "Joanna1", "gender": "F", "description": ""}, {"index": 1480, "label": "690", "name": "Silver", "gender": "F", "description": ""}, {"index": 1481, "label": "6912", "name": "Richard Beck", "gender": "M", "description": ""}, {"index": 1482, "label": "6913", "name": "daisyb", "gender": "F", "description": ""}, {"index": 1483, "label": "6914", "name": "Katalina Watt", "gender": "F", "description": ""}, {"index": 1484, "label": "6918", "name": "Marilyn Mack", "gender": "F", "description": ""}, {"index": 1485, "label": "6923", "name": "Szindbad", "gender": "M", "description": ""}, {"index": 1486, "label": "6924", "name": "Rapunzelina", "gender": "F", "description": ""}, {"index": 1487, "label": "6925", "name": "Thomas Meaney", "gender": "M", "description": ""}, {"index": 1488, "label": "6927", "name": "Sarika Pawar", "gender": "F", "description": ""}, {"index": 1489, "label": "6943", "name": "Chieko Steely", "gender": "F", "description": ""}, {"index": 1490, "label": "6945", "name": "Daniel George", "gender": "M", "description": ""}, {"index": 1491, "label": "6947", "name": "Grace", "gender": "F", "description": ""}, {"index": 1492, "label": "6950", "name": "elmay", "gender": "F", "description": ""}, {"index": 1493, "label": "6951", "name": "redhed3095", "gender": "F", "description": ""}, {"index": 1494, "label": "6954", "name": "Paul Richards", "gender": "M", "description": ""}, {"index": 1495, "label": "6962", "name": "mattoscarlomas", "gender": "M", "description": ""}, {"index": 1496, "label": "6963", "name": "Kasper", "gender": "M", "description": ""}, {"index": 1497, "label": "6965", "name": "NoelBadrian", "gender": "M", "description": ""}, {"index": 1498, "label": "6967", "name": "sganatra81", "gender": "F", "description": ""}, {"index": 1499, "label": "696", "name": "Tamara R. Schwartz", "gender": "F", "description": ""}, {"index": 1500, "label": "6974", "name": "Michael Armenta", "gender": "M", "description": ""}, {"index": 1501, "label": "6978", "name": "Debra", "gender": "F", "description": ""}, {"index": 1502, "label": "7000", "name": "Kevin Alix", "gender": "M", "description": ""}, {"index": 1503, "label": "7001", "name": "Arienne", "gender": "F", "description": ""}, {"index": 1504, "label": "7008", "name": "John Trevithick", "gender": "M", "description": ""}, {"index": 1505, "label": "7009", "name": "roeg11", "gender": "M", "description": ""}, {"index": 1506, "label": "7011", "name": "Icprice", "gender": "M", "description": ""}, {"index": 1507, "label": "7012", "name": "Erin McKelle", "gender": "F", "description": ""}, {"index": 1508, "label": "7026", "name": "Michele Eaton", "gender": "F", "description": ""}, {"index": 1509, "label": "7030", "name": "Conrad T", "gender": "M", "description": ""}, {"index": 1510, "label": "7046", "name": "Pascal Ramseier", "gender": "M", "description": ""}, {"index": 1511, "label": "7051", "name": "Andrew White", "gender": "M", "description": ""}, {"index": 1512, "label": "7055", "name": "gemtwist", "gender": "M", "description": ""}, {"index": 1513, "label": "7059", "name": "Joannemmp", "gender": "F", "description": ""}, {"index": 1514, "label": "705", "name": "eva", "gender": "F", "description": ""}, {"index": 1515, "label": "7061", "name": "AllenJohns", "gender": "M", "description": ""}, {"index": 1516, "label": "7062", "name": "Rebecca Thomas", "gender": "F", "description": ""}, {"index": 1517, "label": "7065", "name": "smitaj", "gender": "M", "description": ""}, {"index": 1518, "label": "7067", "name": "Matthew Wall", "gender": "M", "description": ""}, {"index": 1519, "label": "7069", "name": "John Schuurman", "gender": "M", "description": ""}, {"index": 1520, "label": "7073", "name": "Jill Janovetz", "gender": "F", "description": ""}, {"index": 1521, "label": "7078", "name": "Mary in Arkansas", "gender": "F", "description": ""}, {"index": 1522, "label": "7079", "name": "Chuck Williamson", "gender": "F", "description": ""}, {"index": 1523, "label": "708", "name": "Kevin Devine", "gender": "M", "description": ""}, {"index": 1524, "label": "7090", "name": "Jon Sindell", "gender": "M", "description": ""}, {"index": 1525, "label": "7092", "name": "Lorraine B", "gender": "F", "description": ""}, {"index": 1526, "label": "7095", "name": "Wesseling", "gender": "M", "description": ""}, {"index": 1527, "label": "7096", "name": "Gary Iredale", "gender": "M", "description": ""}, {"index": 1528, "label": "7097", "name": "novelreader", "gender": "F", "description": ""}, {"index": 1529, "label": "70", "name": "Kurt Copeland", "gender": "M", "description": ""}, {"index": 1530, "label": "7107", "name": "titankin77", "gender": "M", "description": ""}, {"index": 1531, "label": "7113", "name": "Sukaina Jaffer", "gender": "F", "description": ""}, {"index": 1532, "label": "7117", "name": "Art Leung", "gender": "F", "description": ""}, {"index": 1533, "label": "711", "name": "Roy Schreiber", "gender": "M", "description": ""}, {"index": 1534, "label": "7121", "name": "384403", "gender": "M", "description": ""}, {"index": 1535, "label": "7125", "name": "Peggy", "gender": "F", "description": ""}, {"index": 1536, "label": "7128", "name": "morganreads", "gender": "M", "description": ""}, {"index": 1537, "label": "712", "name": "Michael Shook", "gender": "M", "description": ""}, {"index": 1538, "label": "7131", "name": "Eden Rea-Hedrick", "gender": "F", "description": ""}, {"index": 1539, "label": "7134", "name": "Steve Jackson", "gender": "M", "description": ""}, {"index": 1540, "label": "7135", "name": "Maggie Smallwood", "gender": "F", "description": ""}, {"index": 1541, "label": "7138", "name": "CaprishaPage", "gender": "F", "description": ""}, {"index": 1542, "label": "7139", "name": "gabrielv", "gender": "M", "description": ""}, {"index": 1543, "label": "713", "name": "Onjana Yawnghwe", "gender": "F", "description": ""}, {"index": 1544, "label": "7143", "name": "Minni Ang", "gender": "F", "description": ""}, {"index": 1545, "label": "7147", "name": "S.Nevets", "gender": "M", "description": ""}, {"index": 1546, "label": "7148", "name": "Vickie Ranz", "gender": "F", "description": ""}, {"index": 1547, "label": "7150", "name": "asterix", "gender": "M", "description": ""}, {"index": 1548, "label": "7155", "name": "pklipp", "gender": "M", "description": ""}, {"index": 1549, "label": "7169", "name": "Ryan Ransom", "gender": "M", "description": ""}, {"index": 1550, "label": "716", "name": "martyd", "gender": "M", "description": ""}, {"index": 1551, "label": "7170", "name": "alanmapstone", "gender": "M", "description": ""}, {"index": 1552, "label": "7177", "name": "shana", "gender": "F", "description": ""}, {"index": 1553, "label": "7178", "name": "J.K. Neely", "gender": "F", "description": ""}, {"index": 1554, "label": "7188", "name": "Marty", "gender": "M", "description": ""}, {"index": 1555, "label": "7189", "name": "doonaboon", "gender": "M", "description": ""}, {"index": 1556, "label": "718", "name": "clarknova", "gender": "M", "description": ""}, {"index": 1557, "label": "7190", "name": "Tony Posante", "gender": "M", "description": ""}, {"index": 1558, "label": "7197", "name": "Christine Richardson", "gender": "F", "description": ""}, {"index": 1559, "label": "7198", "name": "Daniela Austin", "gender": "F", "description": ""}, {"index": 1560, "label": "7199", "name": "T.K. Kirven", "gender": "F", "description": ""}, {"index": 1561, "label": "7205", "name": "genierose", "gender": "F", "description": ""}, {"index": 1562, "label": "7208", "name": "KK", "gender": "M", "description": ""}, {"index": 1563, "label": "720", "name": "Peter Gallagher", "gender": "M", "description": ""}, {"index": 1564, "label": "7215", "name": "BensonBrunswin", "gender": "M", "description": ""}, {"index": 1565, "label": "7218", "name": "MJ Franck", "gender": "F", "description": ""}, {"index": 1566, "label": "7220", "name": "LynnAlison", "gender": "F", "description": ""}, {"index": 1567, "label": "7223", "name": "eye hear voices", "gender": "F", "description": ""}, {"index": 1568, "label": "7226", "name": "Jonathan Moore", "gender": "M", "description": ""}, {"index": 1569, "label": "7228", "name": "Peter John Keeble", "gender": "M", "description": ""}, {"index": 1570, "label": "7229", "name": "Linda Ciano", "gender": "F", "description": ""}, {"index": 1571, "label": "7238", "name": "Uday Sagar", "gender": "M", "description": ""}, {"index": 1572, "label": "7239", "name": "manofwealth", "gender": "M", "description": ""}, {"index": 1573, "label": "7240", "name": "Lucretia B.", "gender": "F", "description": ""}, {"index": 1574, "label": "7242", "name": "Renate", "gender": "F", "description": ""}, {"index": 1575, "label": "7245", "name": "Laura Atkinson", "gender": "F", "description": ""}, {"index": 1576, "label": "7246", "name": "Lyn Silva", "gender": "F", "description": ""}, {"index": 1577, "label": "7247", "name": "Robert Hoffman", "gender": "M", "description": ""}, {"index": 1578, "label": "7250", "name": "A. J. Elliot", "gender": "F", "description": ""}, {"index": 1579, "label": "7255", "name": "Jendia", "gender": "F", "description": ""}, {"index": 1580, "label": "7263", "name": "rachaelg", "gender": "F", "description": ""}, {"index": 1581, "label": "7264", "name": "Sean McClain", "gender": "M", "description": ""}, {"index": 1582, "label": "7265", "name": "Annie", "gender": "F", "description": ""}, {"index": 1583, "label": "726", "name": "Paul", "gender": "M", "description": ""}, {"index": 1584, "label": "7277", "name": "Jenny Bradshaw", "gender": "F", "description": ""}, {"index": 1585, "label": "7278", "name": "Jon Smith", "gender": "M", "description": ""}, {"index": 1586, "label": "727", "name": "Andrew Richards", "gender": "M", "description": ""}, {"index": 1587, "label": "7285", "name": "Christina M. Glavas", "gender": "F", "description": ""}, {"index": 1588, "label": "7286", "name": "kindfish", "gender": "F", "description": ""}, {"index": 1589, "label": "728", "name": "Eric Connover", "gender": "M", "description": ""}, {"index": 1590, "label": "7299", "name": "Barbara Miller", "gender": "F", "description": ""}, {"index": 1591, "label": "7301", "name": "jonseverity", "gender": "M", "description": ""}, {"index": 1592, "label": "7302", "name": "Asta1234", "gender": "F", "description": ""}, {"index": 1593, "label": "7307", "name": "Anthony Ogus", "gender": "M", "description": ""}, {"index": 1594, "label": "730", "name": "Karen Labenz", "gender": "F", "description": ""}, {"index": 1595, "label": "7312", "name": "nkneer", "gender": "M", "description": ""}, {"index": 1596, "label": "7313", "name": "Mark DeVol", "gender": "M", "description": ""}, {"index": 1597, "label": "7315", "name": "Charlotte Duckett", "gender": "F", "description": ""}, {"index": 1598, "label": "731", "name": "Megan-Jane Daniels Suyasu", "gender": "F", "description": ""}, {"index": 1599, "label": "7320", "name": "Monika Rolley", "gender": "F", "description": ""}, {"index": 1600, "label": "7326", "name": "Adam", "gender": "M", "description": ""}, {"index": 1601, "label": "7327", "name": "Tommy Howell", "gender": "M", "description": ""}, {"index": 1602, "label": "7331", "name": "AnabelleC", "gender": "F", "description": ""}, {"index": 1603, "label": "7333", "name": "Lydia Paterson", "gender": "F", "description": ""}, {"index": 1604, "label": "7337", "name": "Adina Owen", "gender": "F", "description": ""}, {"index": 1605, "label": "7338", "name": "Eberle Thomas", "gender": "M", "description": ""}, {"index": 1606, "label": "7342", "name": "Carol", "gender": "F", "description": ""}, {"index": 1607, "label": "7346", "name": "James Bendall", "gender": "M", "description": ""}, {"index": 1608, "label": "7348", "name": "acousticwave", "gender": "F", "description": ""}, {"index": 1609, "label": "7354", "name": "TrevorD777", "gender": "M", "description": ""}, {"index": 1610, "label": "7357", "name": "rebcult", "gender": "M", "description": ""}, {"index": 1611, "label": "7360", "name": "Caroline Hemmerly Kunkle", "gender": "F", "description": ""}, {"index": 1612, "label": "7367", "name": "NIneFive83", "gender": "M", "description": ""}, {"index": 1613, "label": "7376", "name": "Nyssa E. Schmidt", "gender": "F", "description": ""}, {"index": 1614, "label": "737", "name": "Roger W. Barnett", "gender": "M", "description": ""}, {"index": 1615, "label": "7383", "name": "Meg Cowan", "gender": "F", "description": ""}, {"index": 1616, "label": "7387", "name": "Tara Dow", "gender": "F", "description": ""}, {"index": 1617, "label": "7389", "name": "Steve C", "gender": "M", "description": ""}, {"index": 1618, "label": "7391", "name": "Cary Simz", "gender": "M", "description": ""}, {"index": 1619, "label": "7392", "name": "MicheleW", "gender": "F", "description": ""}, {"index": 1620, "label": "7395", "name": "Katherine", "gender": "F", "description": ""}, {"index": 1621, "label": "7398", "name": "Marie Daum", "gender": "F", "description": ""}, {"index": 1622, "label": "7402", "name": "Canby Ibarra", "gender": "M", "description": ""}, {"index": 1623, "label": "7408", "name": "David Clarke", "gender": "M", "description": ""}, {"index": 1624, "label": "7416", "name": "Jeanni Hall", "gender": "F", "description": ""}, {"index": 1625, "label": "7423", "name": "Gilles G. Le Blanc", "gender": "M", "description": ""}, {"index": 1626, "label": "7424", "name": "Christopher Smith", "gender": "M", "description": ""}, {"index": 1627, "label": "742", "name": "Peter Groom", "gender": "M", "description": ""}, {"index": 1628, "label": "7433", "name": "Gabriela Cowan", "gender": "F", "description": ""}, {"index": 1629, "label": "7436", "name": "Gregory Eccles", "gender": "M", "description": ""}, {"index": 1630, "label": "7445", "name": "Emily Anderson", "gender": "F", "description": ""}, {"index": 1631, "label": "7447", "name": "dasbury", "gender": "M", "description": ""}, {"index": 1632, "label": "7448", "name": "Kevin Johnson", "gender": "M", "description": ""}, {"index": 1633, "label": "7460", "name": "Larry Beasley", "gender": "M", "description": ""}, {"index": 1634, "label": "7463", "name": "Pete Milan", "gender": "M", "description": ""}, {"index": 1635, "label": "7467", "name": "freshface", "gender": "F", "description": ""}, {"index": 1636, "label": "7475", "name": "Jared Hess", "gender": "M", "description": ""}, {"index": 1637, "label": "7478", "name": "Dan Froese", "gender": "M", "description": ""}, {"index": 1638, "label": "7480", "name": "Tricia F.", "gender": "F", "description": ""}, {"index": 1639, "label": "7491", "name": "Lilaread", "gender": "F", "description": ""}, {"index": 1640, "label": "7492", "name": "Scheherazade", "gender": "F", "description": ""}, {"index": 1641, "label": "7495", "name": "Arlene Joyce", "gender": "F", "description": ""}, {"index": 1642, "label": "7498", "name": "MarianneVictoria", "gender": "F", "description": ""}, {"index": 1643, "label": "7502", "name": "mikefernlea", "gender": "M", "description": ""}, {"index": 1644, "label": "7505", "name": "Ron Lockhart", "gender": "M", "description": ""}, {"index": 1645, "label": "7507", "name": "UnaVersal", "gender": "F", "description": ""}, {"index": 1646, "label": "7510", "name": "Kathrine Engan", "gender": "F", "description": ""}, {"index": 1647, "label": "7511", "name": "Sherri Vance", "gender": "F", "description": ""}, {"index": 1648, "label": "7512", "name": "dudeman", "gender": "M", "description": ""}, {"index": 1649, "label": "7514", "name": "Barbara Baker", "gender": "F", "description": ""}, {"index": 1650, "label": "7515", "name": "Cynthia Moyer", "gender": "F", "description": ""}, {"index": 1651, "label": "7517", "name": "Raz Mason", "gender": "F", "description": ""}, {"index": 1652, "label": "7518", "name": "TJDasch", "gender": "M", "description": ""}, {"index": 1653, "label": "7520", "name": "Richard Jackson", "gender": "M", "description": ""}, {"index": 1654, "label": "7522", "name": "Kelsey P", "gender": "F", "description": ""}, {"index": 1655, "label": "7525", "name": "Jeannie Tirado", "gender": "F", "description": ""}, {"index": 1656, "label": "7538", "name": "Logan West", "gender": "M", "description": ""}, {"index": 1657, "label": "753", "name": "Tim Bulkeley", "gender": "M", "description": ""}, {"index": 1658, "label": "7540", "name": "Christopher Webber", "gender": "M", "description": ""}, {"index": 1659, "label": "7552", "name": "KateC", "gender": "F", "description": ""}, {"index": 1660, "label": "7553", "name": "BDSEmpire", "gender": "M", "description": ""}, {"index": 1661, "label": "7555", "name": "Suebee", "gender": "F", "description": ""}, {"index": 1662, "label": "7556", "name": "Alex Lau", "gender": "M", "description": ""}, {"index": 1663, "label": "7558", "name": "J. A. Cook", "gender": "M", "description": ""}, {"index": 1664, "label": "7559", "name": "gassumpcao", "gender": "M", "description": ""}, {"index": 1665, "label": "7561", "name": "Shawn Bayern", "gender": "M", "description": ""}, {"index": 1666, "label": "7565", "name": "Derek", "gender": "M", "description": ""}, {"index": 1667, "label": "7584", "name": "kandice stehlik", "gender": "F", "description": ""}, {"index": 1668, "label": "7585", "name": "MaxSitting", "gender": "M", "description": ""}, {"index": 1669, "label": "7594", "name": "Kristel Tretter", "gender": "F", "description": ""}, {"index": 1670, "label": "7597", "name": "Inah Derby", "gender": "F", "description": ""}, {"index": 1671, "label": "75", "name": "Jim Mowatt", "gender": "M", "description": ""}, {"index": 1672, "label": "7603", "name": "Wupperhippo", "gender": "M", "description": ""}, {"index": 1673, "label": "7607", "name": "C F de Rosset", "gender": "F", "description": ""}, {"index": 1674, "label": "7608", "name": "rookieblue", "gender": "M", "description": ""}, {"index": 1675, "label": "7609", "name": "detroitreads", "gender": "F", "description": ""}, {"index": 1676, "label": "7618", "name": "Stipsky", "gender": "M", "description": ""}, {"index": 1677, "label": "7635", "name": "Judy Guinan", "gender": "F", "description": ""}, {"index": 1678, "label": "7640", "name": "Dan Darbandi", "gender": "M", "description": ""}, {"index": 1679, "label": "7644", "name": "Raphael Platt", "gender": "F", "description": ""}, {"index": 1680, "label": "7647", "name": "Daniel T. Miller", "gender": "M", "description": ""}, {"index": 1681, "label": "7649", "name": "Lanerd", "gender": "F", "description": ""}, {"index": 1682, "label": "7654", "name": "Word And Mouth", "gender": "M", "description": ""}, {"index": 1683, "label": "7657", "name": "shalclark", "gender": "F", "description": ""}, {"index": 1684, "label": "766", "name": "Jean Crevier", "gender": "M", "description": ""}, {"index": 1685, "label": "7672", "name": "Cooper Leith", "gender": "M", "description": ""}, {"index": 1686, "label": "7679", "name": "Libby Gohn", "gender": "F", "description": ""}, {"index": 1687, "label": "7683", "name": "Carolyne", "gender": "F", "description": ""}, {"index": 1688, "label": "7687", "name": "Demosthenes", "gender": "F", "description": ""}, {"index": 1689, "label": "7691", "name": "engineerdst", "gender": "M", "description": ""}, {"index": 1690, "label": "7699", "name": "DylanTG", "gender": "F", "description": ""}, {"index": 1691, "label": "7700", "name": "Dave Wills", "gender": "M", "description": ""}, {"index": 1692, "label": "7702", "name": "Linda Jenner", "gender": "F", "description": ""}, {"index": 1693, "label": "7704", "name": "Robert Harder", "gender": "M", "description": ""}, {"index": 1694, "label": "7708", "name": "Savannah", "gender": "F", "description": ""}, {"index": 1695, "label": "770", "name": "Justin S Barrett", "gender": "M", "description": ""}, {"index": 1696, "label": "7713", "name": "Sacha Chander", "gender": "M", "description": ""}, {"index": 1697, "label": "7717", "name": "Astrid Fingerhut", "gender": "F", "description": ""}, {"index": 1698, "label": "7720", "name": "jeffwiltzius", "gender": "M", "description": ""}, {"index": 1699, "label": "7733", "name": "Jewel Raquel", "gender": "F", "description": ""}, {"index": 1700, "label": "7737", "name": "Elisabeth Sollog", "gender": "F", "description": ""}, {"index": 1701, "label": "7739", "name": "Melissa Burns-Price", "gender": "F", "description": ""}, {"index": 1702, "label": "7746", "name": "Tom Causby", "gender": "M", "description": ""}, {"index": 1703, "label": "7749", "name": "Sammy Bean", "gender": "M", "description": ""}, {"index": 1704, "label": "774", "name": "Sara Walsh", "gender": "F", "description": ""}, {"index": 1705, "label": "7752", "name": "Samantha Miles", "gender": "F", "description": ""}, {"index": 1706, "label": "7756", "name": "Lynne Thompson", "gender": "F", "description": ""}, {"index": 1707, "label": "7762", "name": "Fannie", "gender": "F", "description": ""}, {"index": 1708, "label": "7764", "name": "Mike Nelson", "gender": "M", "description": ""}, {"index": 1709, "label": "7766", "name": "Maryanka", "gender": "F", "description": ""}, {"index": 1710, "label": "7769", "name": "David Biro", "gender": "M", "description": ""}, {"index": 1711, "label": "7777", "name": "Jacob Paine", "gender": "M", "description": ""}, {"index": 1712, "label": "7780", "name": "tazzle", "gender": "F", "description": ""}, {"index": 1713, "label": "7786", "name": "Kevin W. Davidson", "gender": "M", "description": ""}, {"index": 1714, "label": "7789", "name": "Lauren Jane", "gender": "F", "description": ""}, {"index": 1715, "label": "778", "name": "Branko Collin", "gender": "M", "description": ""}, {"index": 1716, "label": "7794", "name": "mlcui", "gender": "F", "description": ""}, {"index": 1717, "label": "7795", "name": "Cubrick", "gender": "M", "description": ""}, {"index": 1718, "label": "7796", "name": "Raybrite", "gender": "M", "description": ""}, {"index": 1719, "label": "779", "name": "Greg", "gender": "M", "description": ""}, {"index": 1720, "label": "77", "name": "JemmaBlythe", "gender": "F", "description": ""}, {"index": 1721, "label": "7800", "name": "Arie", "gender": "F", "description": ""}, {"index": 1722, "label": "7802", "name": "Samantha J Gubitz", "gender": "F", "description": ""}, {"index": 1723, "label": "780", "name": "Lloyd Davis", "gender": "M", "description": ""}, {"index": 1724, "label": "7816", "name": "Stefanie Heinrichs", "gender": "F", "description": ""}, {"index": 1725, "label": "781", "name": "Mitchell Dwyer", "gender": "M", "description": ""}, {"index": 1726, "label": "7823", "name": "Bart in \'t Veld", "gender": "M", "description": ""}, {"index": 1727, "label": "7825", "name": "PDyer", "gender": "M", "description": ""}, {"index": 1728, "label": "7826", "name": "Dave Shaw", "gender": "M", "description": ""}, {"index": 1729, "label": "7828", "name": "Dan Mason", "gender": "M", "description": ""}, {"index": 1730, "label": "782", "name": "Michele Pacey", "gender": "F", "description": ""}, {"index": 1731, "label": "7833", "name": "Jesse Crisp-Sears", "gender": "M", "description": ""}, {"index": 1732, "label": "7835", "name": "kageokami", "gender": "M", "description": ""}, {"index": 1733, "label": "7839", "name": "Loveday", "gender": "F", "description": ""}, {"index": 1734, "label": "783", "name": "Gina", "gender": "F", "description": ""}, {"index": 1735, "label": "7843", "name": "Annette McGuinness", "gender": "F", "description": ""}, {"index": 1736, "label": "7848", "name": "bala", "gender": "M", "description": ""}, {"index": 1737, "label": "7859", "name": "xinamarieuhl", "gender": "F", "description": ""}, {"index": 1738, "label": "7871", "name": "Rocit", "gender": "M", "description": ""}, {"index": 1739, "label": "7879", "name": "deongines", "gender": "F", "description": ""}, {"index": 1740, "label": "7883", "name": "Annapurna", "gender": "F", "description": ""}, {"index": 1741, "label": "7886", "name": "Mariah Lyons", "gender": "F", "description": ""}, {"index": 1742, "label": "7892", "name": "Alexis Castro", "gender": "F", "description": ""}, {"index": 1743, "label": "7898", "name": "Coreena", "gender": "F", "description": ""}, {"index": 1744, "label": "789", "name": "Eva", "gender": "F", "description": ""}, {"index": 1745, "label": "78", "name": "Hugh McGuire", "gender": "M", "description": ""}, {"index": 1746, "label": "7909", "name": "MGreenberg", "gender": "M", "description": ""}, {"index": 1747, "label": "7912", "name": "Nathan Dickey", "gender": "M", "description": ""}, {"index": 1748, "label": "791", "name": "Ivan", "gender": "M", "description": ""}, {"index": 1749, "label": "7923", "name": "amaskill", "gender": "M", "description": ""}, {"index": 1750, "label": "7925", "name": "Etel Buss", "gender": "F", "description": ""}, {"index": 1751, "label": "792", "name": "mjd-s", "gender": "F", "description": ""}, {"index": 1752, "label": "7932", "name": "Tammy Stalcup", "gender": "F", "description": ""}, {"index": 1753, "label": "7938", "name": "Goss45", "gender": "M", "description": ""}, {"index": 1754, "label": "7939", "name": "DPranitis", "gender": "M", "description": ""}, {"index": 1755, "label": "7942", "name": "Luke Sartor", "gender": "M", "description": ""}, {"index": 1756, "label": "7945", "name": "Lois Browne", "gender": "F", "description": ""}, {"index": 1757, "label": "7946", "name": "Flash", "gender": "F", "description": ""}, {"index": 1758, "label": "7949", "name": "Jon Miller", "gender": "M", "description": ""}, {"index": 1759, "label": "7957", "name": "Wiley Combs", "gender": "M", "description": ""}, {"index": 1760, "label": "7959", "name": "Lily Took", "gender": "F", "description": ""}, {"index": 1761, "label": "7962", "name": "Jolie O\'Dell", "gender": "F", "description": ""}, {"index": 1762, "label": "7967", "name": "Lois Hill", "gender": "F", "description": ""}, {"index": 1763, "label": "797", "name": "Mike Shapiro", "gender": "M", "description": ""}, {"index": 1764, "label": "7981", "name": "timothyFR", "gender": "M", "description": ""}, {"index": 1765, "label": "7988", "name": "JaySands", "gender": "M", "description": ""}, {"index": 1766, "label": "7997", "name": "Tom Merritt", "gender": "M", "description": ""}, {"index": 1767, "label": "8005", "name": "Julia Wells", "gender": "F", "description": ""}, {"index": 1768, "label": "8006", "name": "TyroneS", "gender": "M", "description": ""}, {"index": 1769, "label": "8008", "name": "Paul Adamson", "gender": "M", "description": ""}, {"index": 1770, "label": "8009", "name": "Frances Brown", "gender": "F", "description": ""}, {"index": 1771, "label": "8012", "name": "WoollyBee", "gender": "F", "description": ""}, {"index": 1772, "label": "8014", "name": "constatine", "gender": "F", "description": ""}, {"index": 1773, "label": "8015", "name": "Underhill", "gender": "F", "description": ""}, {"index": 1774, "label": "8023", "name": "JamesMcAndrew", "gender": "M", "description": ""}, {"index": 1775, "label": "8033", "name": "geoffbl", "gender": "M", "description": ""}, {"index": 1776, "label": "803", "name": "Greg Elmensdorp", "gender": "M", "description": ""}, {"index": 1777, "label": "8040", "name": "NatalieOram", "gender": "F", "description": ""}, {"index": 1778, "label": "8042", "name": "Jack Watson Warr", "gender": "M", "description": ""}, {"index": 1779, "label": "8044", "name": "Sarah Hannah", "gender": "F", "description": ""}, {"index": 1780, "label": "8057", "name": "Linda Dougherty", "gender": "F", "description": ""}, {"index": 1781, "label": "8058", "name": "RLC", "gender": "M", "description": ""}, {"index": 1782, "label": "8063", "name": "Robert Snoza", "gender": "M", "description": ""}, {"index": 1783, "label": "8066", "name": "Tim Cote", "gender": "M", "description": ""}, {"index": 1784, "label": "8071", "name": "Vanessa Garcia", "gender": "F", "description": ""}, {"index": 1785, "label": "8072", "name": "Kimberly Krause", "gender": "F", "description": ""}, {"index": 1786, "label": "8075", "name": "Melora", "gender": "F", "description": ""}, {"index": 1787, "label": "807", "name": "Luke Venediger", "gender": "M", "description": ""}, {"index": 1788, "label": "8080", "name": "e.a.zokaites", "gender": "F", "description": ""}, {"index": 1789, "label": "8087", "name": "Arnie Horton", "gender": "M", "description": ""}, {"index": 1790, "label": "8088", "name": "Jason Bolestridge", "gender": "M", "description": ""}, {"index": 1791, "label": "8095", "name": "Theodulf", "gender": "M", "description": ""}, {"index": 1792, "label": "8097", "name": "lewildesen", "gender": "F", "description": ""}, {"index": 1793, "label": "8098", "name": "Arnold", "gender": "M", "description": ""}, {"index": 1794, "label": "8108", "name": "drakaunus", "gender": "M", "description": ""}, {"index": 1795, "label": "810", "name": "Joseph Loverti", "gender": "M", "description": ""}, {"index": 1796, "label": "8112", "name": "Christine Lamberton", "gender": "F", "description": ""}, {"index": 1797, "label": "8118", "name": "Katie McClain", "gender": "F", "description": ""}, {"index": 1798, "label": "8119", "name": "Malcolm Cameron", "gender": "M", "description": ""}, {"index": 1799, "label": "811", "name": "Elizabeth", "gender": "F", "description": ""}, {"index": 1800, "label": "8123", "name": "Sheila Wood", "gender": "F", "description": ""}, {"index": 1801, "label": "8138", "name": "Lee Smalley", "gender": "M", "description": ""}, {"index": 1802, "label": "8143", "name": "Nick Whitley", "gender": "M", "description": ""}, {"index": 1803, "label": "8148", "name": "Xiph", "gender": "F", "description": ""}, {"index": 1804, "label": "8152", "name": "Bigcowfeet", "gender": "M", "description": ""}, {"index": 1805, "label": "8156", "name": "Brooke Cunningham", "gender": "F", "description": ""}, {"index": 1806, "label": "815", "name": "mawrtea", "gender": "M", "description": ""}, {"index": 1807, "label": "8163", "name": "Greengecko", "gender": "F", "description": ""}, {"index": 1808, "label": "8164", "name": "Rob Board", "gender": "M", "description": ""}, {"index": 1809, "label": "8168", "name": "Kiera Davidson", "gender": "F", "description": ""}, {"index": 1810, "label": "8169", "name": "tovarisch", "gender": "M", "description": ""}, {"index": 1811, "label": "816", "name": "Jeff Robinson", "gender": "M", "description": ""}, {"index": 1812, "label": "8172", "name": "Alan Weyman", "gender": "M", "description": ""}, {"index": 1813, "label": "8176", "name": "Jon Kerfoot", "gender": "M", "description": ""}, {"index": 1814, "label": "8180", "name": "Elanor Sakamoto", "gender": "F", "description": ""}, {"index": 1815, "label": "8183", "name": "Cheri Jordan", "gender": "F", "description": ""}, {"index": 1816, "label": "8190", "name": "Lisa Phelps Gonzalez", "gender": "F", "description": ""}, {"index": 1817, "label": "8193", "name": "helengraves", "gender": "F", "description": ""}, {"index": 1818, "label": "8194", "name": "MsMO", "gender": "F", "description": ""}, {"index": 1819, "label": "8195", "name": "Mr Krause", "gender": "M", "description": ""}, {"index": 1820, "label": "8197", "name": "Victoria P", "gender": "F", "description": ""}, {"index": 1821, "label": "8199", "name": "maryagneskatherine", "gender": "F", "description": ""}, {"index": 1822, "label": "81", "name": "Aaron Decker", "gender": "M", "description": ""}, {"index": 1823, "label": "8200", "name": "Dan S", "gender": "M", "description": ""}, {"index": 1824, "label": "8208", "name": "zaanta", "gender": "M", "description": ""}, {"index": 1825, "label": "820", "name": "Scoot", "gender": "M", "description": ""}, {"index": 1826, "label": "8215", "name": "readread", "gender": "M", "description": ""}, {"index": 1827, "label": "8222", "name": "Greg Golding", "gender": "M", "description": ""}, {"index": 1828, "label": "8225", "name": "Matt Lusher", "gender": "M", "description": ""}, {"index": 1829, "label": "8226", "name": "Adam Picot", "gender": "M", "description": ""}, {"index": 1830, "label": "8228", "name": "hgal2010", "gender": "F", "description": ""}, {"index": 1831, "label": "8238", "name": "Madam Fickle", "gender": "F", "description": ""}, {"index": 1832, "label": "8240", "name": "Gertrude Durette", "gender": "F", "description": ""}, {"index": 1833, "label": "8242", "name": "Anita Slusser", "gender": "F", "description": ""}, {"index": 1834, "label": "8245", "name": "gscheids", "gender": "M", "description": ""}, {"index": 1835, "label": "8246", "name": "beeveeo", "gender": "M", "description": ""}, {"index": 1836, "label": "8250", "name": "Stephen Gibbons", "gender": "M", "description": ""}, {"index": 1837, "label": "8259", "name": "Lawrence", "gender": "F", "description": ""}, {"index": 1838, "label": "8262", "name": "Jack Powell", "gender": "M", "description": ""}, {"index": 1839, "label": "8266", "name": "Jeff K.", "gender": "M", "description": ""}, {"index": 1840, "label": "826", "name": "Thomas Wells", "gender": "M", "description": ""}, {"index": 1841, "label": "8272", "name": "Haili", "gender": "F", "description": ""}, {"index": 1842, "label": "8273", "name": "Lucas Boulding", "gender": "M", "description": ""}, {"index": 1843, "label": "8291", "name": "Elliot Gage", "gender": "M", "description": ""}, {"index": 1844, "label": "8295", "name": "Susan Morin", "gender": "F", "description": ""}, {"index": 1845, "label": "8296", "name": "Bria Snow", "gender": "F", "description": ""}, {"index": 1846, "label": "829", "name": "frenchfry", "gender": "F", "description": ""}, {"index": 1847, "label": "82", "name": "Marlo Dianne", "gender": "F", "description": ""}, {"index": 1848, "label": "8300", "name": "Judith Parker", "gender": "F", "description": ""}, {"index": 1849, "label": "8302", "name": "Charlotte Day", "gender": "F", "description": ""}, {"index": 1850, "label": "8307", "name": "Nick Bulka", "gender": "M", "description": ""}, {"index": 1851, "label": "830", "name": "sebrazer", "gender": "F", "description": ""}, {"index": 1852, "label": "8312", "name": "Jaimie Noy", "gender": "F", "description": ""}, {"index": 1853, "label": "8316", "name": "Rocket Rodger", "gender": "M", "description": ""}, {"index": 1854, "label": "831", "name": "Nick Gallant", "gender": "M", "description": ""}, {"index": 1855, "label": "8321", "name": "SamR", "gender": "F", "description": ""}, {"index": 1856, "label": "8322", "name": "yeknod", "gender": "M", "description": ""}, {"index": 1857, "label": "8324", "name": "Kathy Wright", "gender": "F", "description": ""}, {"index": 1858, "label": "8328", "name": "William Gavula", "gender": "M", "description": ""}, {"index": 1859, "label": "8329", "name": "Betty Perry", "gender": "F", "description": ""}, {"index": 1860, "label": "8334", "name": "Doug Reed", "gender": "M", "description": ""}, {"index": 1861, "label": "8337", "name": "Claudia Salto", "gender": "F", "description": ""}, {"index": 1862, "label": "8346", "name": "Al Rocca", "gender": "M", "description": ""}, {"index": 1863, "label": "8347", "name": "Terry Torres", "gender": "M", "description": ""}, {"index": 1864, "label": "8356", "name": "Paul Mazumdar", "gender": "M", "description": ""}, {"index": 1865, "label": "8367", "name": "ophiuroidea", "gender": "M", "description": ""}, {"index": 1866, "label": "836", "name": "Kevin LaVergne", "gender": "M", "description": ""}, {"index": 1867, "label": "8382", "name": "Claire Schreuder", "gender": "F", "description": ""}, {"index": 1868, "label": "8388", "name": "Jacki Horn", "gender": "F", "description": ""}, {"index": 1869, "label": "8389", "name": "Steven Bateman", "gender": "M", "description": ""}, {"index": 1870, "label": "8392", "name": "Katharina Huang", "gender": "F", "description": ""}, {"index": 1871, "label": "8394", "name": "Matthew Walker", "gender": "M", "description": ""}, {"index": 1872, "label": "8396", "name": "gloriousjob", "gender": "M", "description": ""}, {"index": 1873, "label": "839", "name": "rovert405", "gender": "M", "description": ""}, {"index": 1874, "label": "83", "name": "Catharine Eastman", "gender": "F", "description": ""}, {"index": 1875, "label": "8401", "name": "Alexander Hatton", "gender": "M", "description": ""}, {"index": 1876, "label": "8410", "name": "Shauna Kennett", "gender": "F", "description": ""}, {"index": 1877, "label": "8413", "name": "PaulMichael1084", "gender": "M", "description": ""}, {"index": 1878, "label": "8414", "name": "jtueller", "gender": "F", "description": ""}, {"index": 1879, "label": "8415", "name": "Rusty Dancer", "gender": "F", "description": ""}, {"index": 1880, "label": "8419", "name": "Jon Kissack", "gender": "M", "description": ""}, {"index": 1881, "label": "8421", "name": "Jackie Drown", "gender": "F", "description": ""}, {"index": 1882, "label": "8422", "name": "pjhoury", "gender": "M", "description": ""}, {"index": 1883, "label": "8424", "name": "Schums", "gender": "F", "description": ""}, {"index": 1884, "label": "8425", "name": "Larry Wilson", "gender": "M", "description": ""}, {"index": 1885, "label": "8430", "name": "shihping", "gender": "F", "description": ""}, {"index": 1886, "label": "8432", "name": "Emma Joyce", "gender": "F", "description": ""}, {"index": 1887, "label": "8441", "name": "ryoko", "gender": "F", "description": ""}, {"index": 1888, "label": "8443", "name": "Marsha Payne", "gender": "F", "description": ""}, {"index": 1889, "label": "8444", "name": "Anthony Webster", "gender": "M", "description": ""}, {"index": 1890, "label": "8445", "name": "Brett G. Hirsch", "gender": "M", "description": ""}, {"index": 1891, "label": "8447", "name": "Anastasiia Solokha", "gender": "F", "description": ""}, {"index": 1892, "label": "844", "name": "Martina", "gender": "F", "description": ""}, {"index": 1893, "label": "8459", "name": "sdaeley17", "gender": "M", "description": ""}, {"index": 1894, "label": "845", "name": "DaveF", "gender": "M", "description": ""}, {"index": 1895, "label": "8465", "name": "TinaNygard2", "gender": "F", "description": ""}, {"index": 1896, "label": "8466", "name": "Chris Cartwright", "gender": "M", "description": ""}, {"index": 1897, "label": "8468", "name": "Jennifer Dorr", "gender": "F", "description": ""}, {"index": 1898, "label": "846", "name": "Anadaxis_Canejia", "gender": "M", "description": ""}, {"index": 1899, "label": "8470", "name": "Rebecca Braunert-Plunkett", "gender": "F", "description": ""}, {"index": 1900, "label": "8474", "name": "Scott Snowman", "gender": "M", "description": ""}, {"index": 1901, "label": "8476", "name": "Julie Mansius", "gender": "F", "description": ""}, {"index": 1902, "label": "8490", "name": "Matt Parker", "gender": "M", "description": ""}, {"index": 1903, "label": "8494", "name": "Bill Yallalee", "gender": "M", "description": ""}, {"index": 1904, "label": "8498", "name": "Ian Quinlan", "gender": "M", "description": ""}, {"index": 1905, "label": "8499", "name": "ReadAllDay", "gender": "M", "description": ""}, {"index": 1906, "label": "8500", "name": "Ravi Shankar", "gender": "M", "description": ""}, {"index": 1907, "label": "8506", "name": "Denise Nordell", "gender": "F", "description": ""}, {"index": 1908, "label": "850", "name": "tonypettit", "gender": "M", "description": ""}, {"index": 1909, "label": "851", "name": "brenthumphries", "gender": "M", "description": ""}, {"index": 1910, "label": "8527", "name": "Gary Bohannon", "gender": "M", "description": ""}, {"index": 1911, "label": "8531", "name": "imenadel", "gender": "M", "description": ""}, {"index": 1912, "label": "8534", "name": "Chris Clark", "gender": "M", "description": ""}, {"index": 1913, "label": "8536", "name": "Jonathan Brubaker", "gender": "M", "description": ""}, {"index": 1914, "label": "8543", "name": "luckyemma", "gender": "F", "description": ""}, {"index": 1915, "label": "8544", "name": "Pooja DSr", "gender": "F", "description": ""}, {"index": 1916, "label": "8545", "name": "Joanne Rochon", "gender": "F", "description": ""}, {"index": 1917, "label": "8565", "name": "Patrick Eaton", "gender": "M", "description": ""}, {"index": 1918, "label": "8573", "name": "Paige G", "gender": "F", "description": ""}, {"index": 1919, "label": "8575", "name": "Jeremy Robertson", "gender": "M", "description": ""}, {"index": 1920, "label": "8576", "name": "Phil Schempf", "gender": "M", "description": ""}, {"index": 1921, "label": "8580", "name": "Gary Dana", "gender": "M", "description": ""}, {"index": 1922, "label": "8587", "name": "Drew Johnson", "gender": "M", "description": ""}, {"index": 1923, "label": "8590", "name": "bobrose", "gender": "M", "description": ""}, {"index": 1924, "label": "8591", "name": "Jude Somers", "gender": "F", "description": ""}, {"index": 1925, "label": "8592", "name": "Dylan Posa", "gender": "M", "description": ""}, {"index": 1926, "label": "85", "name": "David Leaman", "gender": "M", "description": ""}, {"index": 1927, "label": "8609", "name": "noblesavage", "gender": "M", "description": ""}, {"index": 1928, "label": "8625", "name": "Ellen Preckel", "gender": "F", "description": ""}, {"index": 1929, "label": "8629", "name": "Shivansh Dhar", "gender": "M", "description": ""}, {"index": 1930, "label": "8630", "name": "Eduardo", "gender": "M", "description": ""}, {"index": 1931, "label": "8631", "name": "Edward Kirkby", "gender": "M", "description": ""}, {"index": 1932, "label": "8632", "name": "Geremia", "gender": "M", "description": ""}, {"index": 1933, "label": "8635", "name": "ACBowgus", "gender": "M", "description": ""}, {"index": 1934, "label": "8644", "name": "Eenae", "gender": "F", "description": ""}, {"index": 1935, "label": "8664", "name": "Pam Castille", "gender": "F", "description": ""}, {"index": 1936, "label": "8666", "name": "josembi", "gender": "M", "description": ""}, {"index": 1937, "label": "8671", "name": "Simon Smoke", "gender": "M", "description": ""}, {"index": 1938, "label": "8675", "name": "Larry Greene", "gender": "M", "description": ""}, {"index": 1939, "label": "8677", "name": "KHand", "gender": "F", "description": ""}, {"index": 1940, "label": "8678", "name": "Amy", "gender": "F", "description": ""}, {"index": 1941, "label": "8684", "name": "Alison Stewart", "gender": "F", "description": ""}, {"index": 1942, "label": "8687", "name": "EccentricOwl", "gender": "M", "description": ""}, {"index": 1943, "label": "868", "name": "Mike Rosenlof", "gender": "M", "description": ""}, {"index": 1944, "label": "8699", "name": "Jessica Atha", "gender": "F", "description": ""}, {"index": 1945, "label": "8705", "name": "dsilber01", "gender": "M", "description": ""}, {"index": 1946, "label": "8710", "name": "Ralph Crown", "gender": "M", "description": ""}, {"index": 1947, "label": "8713", "name": "Expatriate", "gender": "M", "description": ""}, {"index": 1948, "label": "8722", "name": "Emily Maynard", "gender": "F", "description": ""}, {"index": 1949, "label": "8725", "name": "Gary Ericson", "gender": "M", "description": ""}, {"index": 1950, "label": "8742", "name": "Adam Taylor", "gender": "M", "description": ""}, {"index": 1951, "label": "8747", "name": "DeanOBuchanan", "gender": "M", "description": ""}, {"index": 1952, "label": "8753", "name": "Walt Allan", "gender": "M", "description": ""}, {"index": 1953, "label": "8758", "name": "Krzysztof Rowinski", "gender": "M", "description": ""}, {"index": 1954, "label": "8765", "name": "jciesielski", "gender": "M", "description": ""}, {"index": 1955, "label": "876", "name": "Alisha", "gender": "F", "description": ""}, {"index": 1956, "label": "8770", "name": "Paul Simonin", "gender": "M", "description": ""}, {"index": 1957, "label": "8772", "name": "Martin Reyto", "gender": "M", "description": ""}, {"index": 1958, "label": "8776", "name": "Taliesin", "gender": "F", "description": ""}, {"index": 1959, "label": "8778", "name": "Francoise", "gender": "F", "description": ""}, {"index": 1960, "label": "8786", "name": "Bruce Kachuk", "gender": "M", "description": ""}, {"index": 1961, "label": "8791", "name": "ScottReyonoldsVoice", "gender": "M", "description": ""}, {"index": 1962, "label": "8797", "name": "Sean Grabosky", "gender": "M", "description": ""}, {"index": 1963, "label": "8799", "name": "Peter Tucker", "gender": "M", "description": ""}, {"index": 1964, "label": "87", "name": "Rosalind Wills", "gender": "F", "description": ""}, {"index": 1965, "label": "8803", "name": "Ben Lindsey-Clark", "gender": "M", "description": ""}, {"index": 1966, "label": "8808", "name": "davidpr", "gender": "M", "description": ""}, {"index": 1967, "label": "8820", "name": "Ignare", "gender": "M", "description": ""}, {"index": 1968, "label": "8825", "name": "Erin Schellhase", "gender": "F", "description": ""}, {"index": 1969, "label": "882", "name": "Mur Lafferty", "gender": "F", "description": ""}, {"index": 1970, "label": "8838", "name": "Kevin Owens", "gender": "M", "description": ""}, {"index": 1971, "label": "8846", "name": "MariaS", "gender": "F", "description": ""}, {"index": 1972, "label": "8848", "name": "Craig Kenneth Bryant", "gender": "M", "description": ""}, {"index": 1973, "label": "884", "name": "sayeth", "gender": "M", "description": ""}, {"index": 1974, "label": "8855", "name": "Eric Metzler", "gender": "M", "description": ""}, {"index": 1975, "label": "8867", "name": "sorbet87", "gender": "F", "description": ""}, {"index": 1976, "label": "886", "name": "Paul S. Jenkins", "gender": "M", "description": ""}, {"index": 1977, "label": "8875", "name": "David D\'Huet", "gender": "M", "description": ""}, {"index": 1978, "label": "887", "name": "Lana Taylor", "gender": "F", "description": ""}, {"index": 1979, "label": "8887", "name": "Andrew Hernandez", "gender": "M", "description": ""}, {"index": 1980, "label": "8897", "name": "beyondutopia", "gender": "F", "description": ""}, {"index": 1981, "label": "895", "name": "Max Porter Zasada", "gender": "M", "description": ""}, {"index": 1982, "label": "8975", "name": "Daisy Flaim", "gender": "F", "description": ""}, {"index": 1983, "label": "89", "name": "Kristen McQuillin", "gender": "F", "description": ""}, {"index": 1984, "label": "9000", "name": "Ramon Escamilla", "gender": "M", "description": ""}, {"index": 1985, "label": "9022", "name": "Claire M", "gender": "F", "description": ""}, {"index": 1986, "label": "909", "name": "Greg Bryant", "gender": "M", "description": ""}, {"index": 1987, "label": "90", "name": "Dan Threetrees", "gender": "M", "description": ""}, {"index": 1988, "label": "911", "name": "frankjf", "gender": "M", "description": ""}, {"index": 1989, "label": "915", "name": "Ted Kaouk", "gender": "M", "description": ""}, {"index": 1990, "label": "91", "name": "Stephan Mobius", "gender": "M", "description": ""}, {"index": 1991, "label": "921", "name": "Brian J. Callaghan", "gender": "M", "description": ""}, {"index": 1992, "label": "922", "name": "Steven H. Wilson", "gender": "M", "description": ""}, {"index": 1993, "label": "923", "name": "Layna", "gender": "F", "description": ""}, {"index": 1994, "label": "925", "name": "pattymarie", "gender": "F", "description": ""}, {"index": 1995, "label": "927", "name": "Nerijus", "gender": "M", "description": ""}, {"index": 1996, "label": "937", "name": "Susie G.", "gender": "F", "description": ""}, {"index": 1997, "label": "93", "name": "Kathy", "gender": "F", "description": ""}, {"index": 1998, "label": "949", "name": "ontheroad", "gender": "M", "description": ""}, {"index": 1999, "label": "94", "name": "David Barnes", "gender": "M", "description": ""}, {"index": 2000, "label": "951", "name": "thomasina", "gender": "F", "description": ""}, {"index": 2001, "label": "956", "name": "krithiga", "gender": "F", "description": ""}, {"index": 2002, "label": "957", "name": "iscatel", "gender": "M", "description": ""}, {"index": 2003, "label": "960", "name": "Marloes Schoonheim", "gender": "F", "description": ""}, {"index": 2004, "label": "964", "name": "Paul Sze", "gender": "M", "description": ""}, {"index": 2005, "label": "969", "name": "Czechchris", "gender": "M", "description": ""}, {"index": 2006, "label": "976", "name": "Alison Raouf", "gender": "F", "description": ""}, {"index": 2007, "label": "978", "name": "Rick Box", "gender": "M", "description": ""}, {"index": 2008, "label": "979", "name": "Kelli Robinson", "gender": "F", "description": ""}, {"index": 2009, "label": "982", "name": "Mike Roop", "gender": "M", "description": ""}, {"index": 2010, "label": "984", "name": "J A Carter", "gender": "M", "description": ""}, {"index": 2011, "label": "985", "name": "George Pilling", "gender": "M", "description": ""}, {"index": 2012, "label": "986", "name": "Michael Kirkpatrick", "gender": "M", "description": ""}, {"index": 2013, "label": "98", "name": "Patricia Oakley", "gender": "F", "description": ""}], "supertonic": [{"label": "F1", "voice": "F1", "name": "F1", "gender": "F", "description": "Composed, slightly low female", "file": "voice_styles/F1.json"}, {"label": "F2", "voice": "F2", "name": "F2", "gender": "F", "description": "Bright, youthful female", "file": "voice_styles/F2.json"}, {"label": "F3", "voice": "F3", "name": "F3", "gender": "F", "description": "Clear female announcer", "file": "voice_styles/F3.json"}, {"label": "F4", "voice": "F4", "name": "F4", "gender": "F", "description": "Confident, expressive female", "file": "voice_styles/F4.json"}, {"label": "F5", "voice": "F5", "name": "F5", "gender": "F", "description": "Gentle, soothing female", "file": "voice_styles/F5.json"}, {"label": "M1", "voice": "M1", "name": "M1", "gender": "M", "description": "Energetic male narration", "file": "voice_styles/M1.json"}, {"label": "M2", "voice": "M2", "name": "M2", "gender": "M", "description": "Deep, composed, serious male", "file": "voice_styles/M2.json"}, {"label": "M3", "voice": "M3", "name": "M3", "gender": "M", "description": "Authoritative male presenter", "file": "voice_styles/M3.json"}, {"label": "M4", "voice": "M4", "name": "M4", "gender": "M", "description": "Gentle, youthful male", "file": "voice_styles/M4.json"}, {"label": "M5", "voice": "M5", "name": "M5", "gender": "M", "description": "Warm male storyteller", "file": "voice_styles/M5.json"}], "f5": {"checkpoint": "F5TTS_v1_Base/model_1250000.safetensors", "source_revision": "283252563dbf91be625e0c27926acfaac449186c", "weight_revision": "84e5a410d9cead4de2f847e7c9369a6440bdfaca", "voice_mode": "open-ended reference audio; no finite speaker-ID bank", "arbitrary_reference_audio": true, "reference_transcript_required": true, "reference_seconds": [1, 8], "native_sample_rate": 24000, "checkpoint_languages": ["English", "Mandarin Chinese"], "notebook_fallback": "Kokoro am_fenrir synthetic reference; not an F5 built-in voice", "source_example_references": ["src/f5_tts/infer/examples/basic/basic_ref_en.wav", "src/f5_tts/infer/examples/basic/basic_ref_zh.wav", "src/f5_tts/infer/examples/multi/main.flac", "src/f5_tts/infer/examples/multi/town.flac", "src/f5_tts/infer/examples/multi/country.flac"], "controls": [{"key": "config", "default": null, "accepted": "Optional JSON file; standalone CLI only", "effect": "Loads argparse destination keys; explicit CLI arguments take precedence. Notebook writes this file for you. Unknown keys and incorrect JSON types fail validation."}, {"key": "text", "default": null, "accepted": "Exactly one of text/text_file; 10–500 whitespace-separated words", "effect": "Literal text to speak; use TEXT_CASES in the notebook. Text affects pronunciation, native punctuation pauses, chunking, and the plan."}, {"key": "text_file", "default": null, "accepted": "UTF-8/UTF-8-BOM file; mutually exclusive with text", "effect": "Reads the complete passage from a file. The supervisor creates this from TEXT_CASES."}, {"key": "repo", "default": null, "accepted": "Required path to the exact pinned source checkout", "effect": "Inference implementation. The notebook installs and supplies it; a different revision or modified checked model source is rejected."}, {"key": "n", "default": 1, "accepted": "Integer ≥1", "effect": "Number of takes; set global CONFIG[\'n\'] or profile-level n. These models generate takes/chunks sequentially within their worker."}, {"key": "seed", "default": 42, "accepted": "Integer; notebook accepts 0 through 2**63−n", "effect": "Base random seed; take i uses seed+i and each chunk has a reproducible derived seed. Reproducibility is conditional on the same software/hardware/settings."}, {"key": "device", "default": "cpu", "accepted": "cpu or available cuda:N", "effect": "Standalone compute device. The notebook selects a physical GPU, exposes it to the worker, and supplies cuda:0; do not put device in a profile."}, {"key": "output_dir", "default": "f5_samples", "accepted": "Fresh output directory", "effect": "Writes WAVs, manifest, and previews/plans. Existing run artifacts are rejected. The notebook owns naming and directories."}, {"key": "sample_rate", "default": 24000, "accepted": "Integer 8000–192000 Hz", "effect": "Export rate after deterministic polyphase resampling from 24 kHz. Increasing it does not create new acoustic detail. Final duration uses the exact exported sample count, including resampling rounding."}, {"key": "pause_ms", "default": 0, "accepted": "Finite ≥0; cannot be positive with crossfade_ms", "effect": "Inserts this many milliseconds of zero-valued samples between generated chunks. It adds to exported duration; it does not change pauses already generated inside a chunk."}, {"key": "crossfade_ms", "default": 0, "accepted": "Finite ≥0; cannot be positive with pause_ms", "effect": "Linearly overlaps neighboring chunks, subtracting the actual overlap from exported length. Overlap is capped by available samples; large overlaps can superimpose words."}, {"key": "fade_ms", "default": 5, "accepted": "Finite 0–20 ms", "effect": "Linear amplitude fade at both ends of every native chunk, capped at half its length. It changes no sample counts. A large fade can weaken an initial/final phoneme; zero disables it."}, {"key": "max_vram_gb", "default": 5, "accepted": "Finite 0<value≤5 decimal GB; CUDA also requires >0.75 GB", "effect": "GPU allocator budgeting with a 0.75 GB reserve, plus observed peak checks. CUDA library allocations exist outside the allocator; sampled monitoring is not an instantaneous hard-peak certificate. Notebook also supervises NVML externally."}, {"key": "estimate_only", "default": false, "accepted": "Boolean; --estimate-only / --no-estimate-only", "effect": "Writes the exact preview (and StyleTTS2 tensor plan) without waveform generation. The regular notebook benchmark generates audio, so the supervisor controls this key."}, {"key": "wav_subtype", "default": "PCM_24", "accepted": "PCM_16, PCM_24, FLOAT", "effect": "WAV sample encoding: signed 16-/24-bit PCM or 32-bit float. This changes precision/file size, not rate, sample count, or generation style."}, {"key": "peak_limit", "default": 0.98, "accepted": "Finite 0<value≤1", "effect": "If the final resampled waveform exceeds this absolute peak, applies one uniform gain ≤1. It neither boosts quiet audio nor changes duration; the gain and original peak are logged."}, {"key": "reference_audio", "default": null, "accepted": "Required finite nonsilent 1–8 s audio; shared notebook reference must be mono", "effect": "Speaker/style prompt. Standalone reading averages multiple channels and resamples to 24 kHz; no automatic silence trimming. Clean speech from one speaker gives a more useful prompt than music, overlapping speakers, clipping, or a cut word."}, {"key": "reference_text", "default": null, "accepted": "Required exact nonempty transcript", "effect": "Conditions the spoken reference together with the waveform. Adapter appends a terminal period if needed and a separating space. Its UTF-8 byte length, with the reference duration, also defines the allocation speaking-rate estimate."}, {"key": "checkpoint", "default": null, "accepted": "Optional local official-layout F5TTS_v1_Base EMA .safetensors", "effect": "Replaces the pinned generator file. Architecture is fixed in this adapter and load is strict; Small, E2TTS, non-EMA layouts, and arbitrary unrelated checkpoints are not interchangeable."}, {"key": "vocab", "default": null, "accepted": "Optional local matching vocabulary file", "effect": "Text token-to-ID mapping; must match the generator. Otherwise uses the pinned F5TTS_v1_Base vocab. A different vocabulary is not a language-quality guarantee."}, {"key": "vocos_dir", "default": null, "accepted": "Optional directory containing compatible config.yaml and pytorch_model.bin", "effect": "Offline Vocos resources. The validated decoder uses vocos==0.1.0 and centered 24 kHz/256-hop output; incompatible temporal topology is rejected."}, {"key": "speed", "default": 1, "accepted": "Finite >0", "effect": "Divides the reference-derived generated-frame allocation: >1 requests shorter/faster speech, <1 longer/slower. It changes planning, not post-generation playback. Token/reference clamps and chunk boundaries prevent a universal 1/speed duration ratio."}, {"key": "duration_seconds", "default": null, "accepted": "Optional finite >0 seconds", "effect": "Requests a TOTAL generated-content allocation divided among chunks by UTF-8 byte length, excluding reference audio and joins. Frame quantization and token/reference minimums may increase/change it: the reported preview, not this raw target, is the final duration. With this set, speed still affects chunk splitting but no longer determines each chunk\'s duration allocation."}, {"key": "steps", "default": 32, "accepted": "Integer ≥2; notebook quality profile 64", "effect": "Flow-integration intervals. More intervals cost generation time and may improve quality; they do not alter the frozen allocated frame count. Improvement is not guaranteed to be monotonic."}, {"key": "cfg_strength", "default": 2, "accepted": "Finite ≥0", "effect": "Classifier-free guidance strength. The pinned flow combines conditional/unconditional predictions as conditional + strength*(conditional−unconditional); near zero uses only the conditional prediction. Stronger guidance can change adherence and sound quality, but leaves the allocation fixed."}, {"key": "sway", "default": -1, "accepted": "Finite −1 through 1", "effect": "Warps integration times after EPSS/uniform schedule: t\'=t+sway*(cos(pi*t/2)−1+t). Negative values concentrate evaluation toward early flow time; zero removes this warp. Changes generation behavior, not output length."}, {"key": "ode_method", "default": "euler", "accepted": "euler, midpoint, rk4", "effect": "ODE solver: approximately 1, 2, or 4 transformer evaluations per interval respectively. Higher-order solvers are more expensive; their quality is an empirical choice. The progress bar counts evaluations; length stays fixed."}, {"key": "no_epss", "default": false, "accepted": "Boolean; CLI flag --no-epss sets true", "effect": "True selects uniformly spaced times BEFORE the optional sway warp. False enables pinned Empirically Pruned Step Sampling at 5/6/7/10/12/16 steps; other step counts already fall back to uniform times. At the notebook\'s 64 steps, toggling this does not change the pre-sway schedule."}, {"key": "target_rms", "default": 0.1, "accepted": "Finite >0", "effect": "If the reference RMS is below this value, boosts conditioning to this RMS, then reverses that gain on generated audio. It is reference-level handling, not final loudness mastering or an emotion/intensity control. Final peak protection still applies."}, {"key": "max_chunk_seconds", "default": 6, "accepted": "Finite 0.2–12 s", "effect": "Bounds generated allocation per chunk; also determines the reference-derived byte budget. A clamped allocation beyond this limit (with a one-hop allowance) fails rather than truncating speech. Smaller chunks can reduce VRAM but create more joins and reset context."}, {"key": "max_chunk_bytes", "default": 160, "accepted": "Integer ≥8 UTF-8 bytes", "effect": "Hard text byte budget per chunk, combined with the time-derived budget. Sentence/clause boundaries are preferred; tiny final tails are merged/rebalanced when possible. A single over-budget word fails. UTF-8 bytes are not words, especially for Chinese."}]}, "styletts2": {"checkpoint": "Models/LibriTTS/epochs_2nd_00020.pth", "source_revision": "5cedc71c333f8d8b8551ca59378bdcc7af4c9529", "weight_revision": "3aa7ba7f8f275ec13dce21682a61494c35089e2a", "voice_mode": "open-ended reference audio; no finite speaker-ID bank", "arbitrary_reference_audio": true, "reference_transcript_required": false, "standalone_reference_seconds": [0.5, 15], "notebook_global_reference_seconds": [1, 8], "native_sample_rate": 24000, "checkpoint_languages": ["English"], "phonemizer_variants": ["en-us", "en-gb"], "notebook_fallback": "Kokoro am_fenrir synthetic reference; not a StyleTTS2 built-in voice", "official_demo_reference_files": [{"group": "seen-speaker examples", "file": "696_92939_000016_000006.wav"}, {"group": "seen-speaker examples", "file": "1789_142896_000022_000005.wav"}, {"group": "unseen-speaker examples", "file": "1221-135767-0014.wav"}, {"group": "unseen-speaker examples", "file": "5639-40744-0020.wav"}, {"group": "unseen-speaker examples", "file": "908-157963-0027.wav"}, {"group": "unseen-speaker examples", "file": "4077-13754-0000.wav"}, {"group": "zero-shot examples", "file": "3.wav"}, {"group": "zero-shot examples", "file": "4.wav"}, {"group": "zero-shot examples", "file": "5.wav"}, {"group": "delivery examples", "file": "anger.wav"}, {"group": "delivery examples", "file": "sleepy.wav"}, {"group": "delivery examples", "file": "amused.wav"}, {"group": "delivery examples", "file": "disgusted.wav"}, {"group": "named examples", "file": "Yinghao.wav"}, {"group": "named examples", "file": "Gavin.wav"}, {"group": "named examples", "file": "Vinay.wav"}, {"group": "named examples", "file": "Nima.wav"}], "reference_bundle_url": "https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/3aa7ba7f8f275ec13dce21682a61494c35089e2a/reference_audio.zip", "controls": [{"key": "config", "default": null, "accepted": "Optional JSON file; standalone CLI only", "effect": "Loads argparse destination keys; explicit CLI arguments take precedence. Notebook writes this file for you. Unknown keys and incorrect JSON types fail validation."}, {"key": "text", "default": null, "accepted": "Exactly one of text/text_file; 10–500 whitespace-separated words", "effect": "Literal text to speak; use TEXT_CASES in the notebook. Text affects pronunciation, native punctuation pauses, chunking, and the plan."}, {"key": "text_file", "default": null, "accepted": "UTF-8/UTF-8-BOM file; mutually exclusive with text", "effect": "Reads the complete passage from a file. The supervisor creates this from TEXT_CASES."}, {"key": "repo", "default": null, "accepted": "Required path to the exact pinned source checkout", "effect": "Inference implementation. The notebook installs and supplies it; a different revision or modified checked model source is rejected."}, {"key": "n", "default": 1, "accepted": "Integer ≥1", "effect": "Number of takes; set global CONFIG[\'n\'] or profile-level n. These models generate takes/chunks sequentially within their worker."}, {"key": "seed", "default": 42, "accepted": "Integer; notebook accepts 0 through 2**63−n", "effect": "Base random seed; take i uses seed+i and each chunk has a reproducible derived seed. Reproducibility is conditional on the same software/hardware/settings."}, {"key": "device", "default": "cpu", "accepted": "cpu or available cuda:N", "effect": "Standalone compute device. The notebook selects a physical GPU, exposes it to the worker, and supplies cuda:0; do not put device in a profile."}, {"key": "output_dir", "default": "styletts2_samples", "accepted": "Fresh output directory", "effect": "Writes WAVs, manifest, and previews/plans. Existing run artifacts are rejected. The notebook owns naming and directories."}, {"key": "sample_rate", "default": 24000, "accepted": "Integer 8000–192000 Hz", "effect": "Export rate after deterministic polyphase resampling from 24 kHz. Increasing it does not create new acoustic detail. Final duration uses the exact exported sample count, including resampling rounding."}, {"key": "pause_ms", "default": 0, "accepted": "Finite ≥0; cannot be positive with crossfade_ms", "effect": "Inserts this many milliseconds of zero-valued samples between generated chunks. It adds to exported duration; it does not change pauses already generated inside a chunk."}, {"key": "crossfade_ms", "default": 0, "accepted": "Finite ≥0; cannot be positive with pause_ms", "effect": "Linearly overlaps neighboring chunks, subtracting the actual overlap from exported length. Overlap is capped by available samples; large overlaps can superimpose words."}, {"key": "fade_ms", "default": 5, "accepted": "Finite 0–20 ms", "effect": "Linear amplitude fade at both ends of every native chunk, capped at half its length. It changes no sample counts. A large fade can weaken an initial/final phoneme; zero disables it."}, {"key": "max_vram_gb", "default": 5, "accepted": "Finite 0<value≤5 decimal GB; CUDA also requires >0.75 GB", "effect": "GPU allocator budgeting with a 0.75 GB reserve, plus observed peak checks. CUDA library allocations exist outside the allocator; sampled monitoring is not an instantaneous hard-peak certificate. Notebook also supervises NVML externally."}, {"key": "estimate_only", "default": false, "accepted": "Boolean; --estimate-only / --no-estimate-only", "effect": "Writes the exact preview (and StyleTTS2 tensor plan) without waveform generation. The regular notebook benchmark generates audio, so the supervisor controls this key."}, {"key": "wav_subtype", "default": "PCM_24", "accepted": "PCM_16, PCM_24, FLOAT", "effect": "WAV sample encoding: signed 16-/24-bit PCM or 32-bit float. This changes precision/file size, not rate, sample count, or generation style."}, {"key": "peak_limit", "default": 0.98, "accepted": "Finite 0<value≤1", "effect": "If the final resampled waveform exceeds this absolute peak, applies one uniform gain ≤1. It neither boosts quiet audio nor changes duration; the gain and original peak are logged."}, {"key": "reference_audio", "default": null, "accepted": "Required finite nonsilent 0.5–15 s audio; shared notebook reference: mono 1–8 s", "effect": "Speaker/style prompt; no transcript needed. Standalone reading averages channels and resamples to 24 kHz without silence trimming. Acoustic and prosodic encoders extract a 256-dimensional reference style."}, {"key": "checkpoint", "default": null, "accepted": "Optional compatible official-layout LibriTTS .pth checkpoint", "effect": "Replaces the generator weights. Inference modules load strictly; single-speaker LJSpeech and unrelated architectures are not drop-in alternatives for this adapter."}, {"key": "model_config", "default": null, "accepted": "Optional matching official LibriTTS config.yml", "effect": "Architecture configuration paired with the checkpoint. It is not a per-generation style configuration. The adapter requires the validated HifiGAN temporal topology."}, {"key": "reuse_plan", "default": null, "accepted": "Optional duration_plan.pt from an identical run setup", "effect": "Reuses cached sampled styles, integer durations, and duration-conditioned embeddings. Fingerprint includes text, reference/checkpoint/config contents, seed/takes, style, pitch, chunk, join, and export settings. Use a fresh output_dir; changed settings fail validation."}, {"key": "speed", "default": 1, "accepted": "Finite >0", "effect": "Divides each predicted phoneme duration BEFORE rounding, then clamps every duration to at least one unit. >1 usually shortens/faster; <1 lengthens/slower. Rounding/minimums mean the total is not exactly proportional to 1/speed."}, {"key": "alpha", "default": 0.1, "accepted": "Finite 0–1", "effect": "Acoustic/timbre blend: alpha*sampled_acoustic + (1−alpha)*reference_acoustic. Zero maximizes direct reference timbre; one uses sampled acoustic style. Higher alpha can reduce speaker similarity; it is not an emotion label."}, {"key": "beta", "default": 0.3, "accepted": "Finite 0–1", "effect": "Prosody blend: beta*sampled_prosody + (1−beta)*reference_prosody. Zero favors the reference delivery; one favors text-conditioned sampled prosody. It can change phoneme durations, expressivity, and similarity, so exact preview must be recomputed."}, {"key": "diffusion_steps", "default": 10, "accepted": "Integer ≥2", "effect": "ADPM2 latent-style sampling steps. Affects preflight cost, sampled style, and therefore potentially the duration plan. This is not waveform diffusion; the cached style is reused during decoding."}, {"key": "embedding_scale", "default": 1, "accepted": "Finite >0", "effect": "Text-embedding classifier-free guidance in style sampling: fixed_embedding_prediction + scale*(text_prediction−fixed_embedding_prediction). At one, uses the normal text-conditioned path; larger values emphasize its difference and add conditional/unconditional computation. Can affect style and planned duration."}, {"key": "style_continuity", "default": 0.7, "accepted": "Finite 0–1", "effect": "On chunks after the first, blends previous FINAL blended style with the new sampled style: continuity*previous + (1−continuity)*new. Zero removes that carryover; one carries the previous style into this intermediate blend. Alpha/beta reference blending is still applied afterward, so one does not freeze every final chunk style."}, {"key": "pitch_scale", "default": 1, "accepted": "Finite >0", "effect": "Multiplies predicted F0 immediately before HifiGAN decoding, without shifting durations. >1 usually higher pitch; <1 lower. This is a model-F0 multiplier, not pitch-preserving playback speed; extreme values can hurt naturalness and speaker identity."}, {"key": "max_chunk_words", "default": 35, "accepted": "Integer 1–80; notebook quality profile 45", "effect": "Initial word budget, with preferred sentence/clause endings. Smaller chunks lower context/VRAM requirements but add joins and may weaken sentence prosody."}, {"key": "max_chunk_tokens", "default": 400, "accepted": "Integer 16–500 phoneme tokens", "effect": "Further recursively splits chunks that exceed the phonemized token budget, preferring sentence/clause boundaries near the split. Includes the initial special token. One over-budget word fails."}, {"key": "max_chunk_seconds", "default": 20, "accepted": "Finite 1–30 s", "effect": "When building a NEW plan, rejects a chunk whose predicted native audio exceeds this limit before decoding; it does not automatically resplit. Use smaller word/token chunks or an appropriate faster speed. Existing reuse_plan data bypasses this check, and this setting is currently absent from the cache fingerprint: retain the original limit when reusing a plan."}, {"key": "tail_trim_samples", "default": 50, "accepted": "Integer ≥0 native 24 kHz samples", "effect": "Fixed trim at every chunk\'s decoder tail, matching the official demo\'s end-pulse workaround. Default is about 2.083 ms per chunk. Zero retains full decoder output. It is included in the preview and is not content-dependent silence removal; a large trim can remove real speech or make a plan invalid."}, {"key": "language", "default": "en-us", "accepted": "en-us, en-gb", "effect": "Selects the eSpeak English phonemizer\'s pronunciation variety. The same English LibriTTS checkpoint is used; this does not turn it into a multilingual model or guarantee a British voice identity."}]}, "omnivoice": {"model": "omnivoice", "source_commit": "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", "model_id": "k2-fsa/OmniVoice", "model_revision": "c5fdb5ccb189668d56333f77ba2629f4cd7535f4", "arbitrary_reference_audio": true, "reference_transcript_required_by_adapter": true, "reference_recommendation_seconds": [3, 10], "notebook_shared_reference_seconds": [1, 8], "preset_speaker_ids": [], "voice_design": {"gender": {"male": "男", "female": "女"}, "age": {"child": "儿童", "teenager": "少年", "young adult": "青年", "middle-aged": "中年", "elderly": "老年"}, "pitch": {"very low pitch": "极低音调", "low pitch": "低音调", "moderate pitch": "中音调", "high pitch": "高音调", "very high pitch": "极高音调"}, "style": {"whisper": "耳语"}, "english_accent": ["american accent", "british accent", "australian accent", "chinese accent", "canadian accent", "indian accent", "korean accent", "portuguese accent", "russian accent", "japanese accent"], "chinese_dialect": ["河南话", "陕西话", "四川话", "贵州话", "云南话", "桂林话", "济南话", "石家庄话", "甘肃话", "宁夏话", "青岛话", "东北话"]}, "emotion_labels": [], "nonverbal_text_tags": ["[laughter]", "[sigh]", "[confirmation-en]", "[question-en]", "[question-ah]", "[question-oh]", "[question-ei]", "[question-yi]", "[surprise-ah]", "[surprise-oh]", "[surprise-wa]", "[surprise-yo]", "[dissatisfaction-hnn]"], "language_name_to_id": {"abadi": "kbt", "abkhazian": "ab", "abron": "abr", "abua": "abn", "adamawa fulfulde": "fub", "adyghe": "ady", "afade": "aal", "afrikaans": "af", "agwagwune": "yay", "aja (benin)": "ajg", "akebu": "keu", "alago": "ala", "albanian": "sq", "algerian arabic": "arq", "algerian saharan arabic": "aao", "ambo-pasco quechua": "qva", "ambonese malay": "abs", "amdo tibetan": "adx", "amharic": "am", "anaang": "anw", "angika": "anp", "antankarana malagasy": "xmv", "aragonese": "an", "arbëreshë albanian": "aae", "arequipa-la unión quechua": "qxu", "armenian": "hy", "ashe": "ahs", "ashéninka perené": "prq", "askopan": "eiv", "assamese": "as", "asturian": "ast", "atayal": "tay", "awak": "awo", "ayacucho quechua": "quy", "azerbaijani": "az", "baatonum": "bba", "bacama": "bcy", "bade": "bde", "bafia": "ksf", "bafut": "bfd", "bagirmi fulfulde": "fui", "bago-kusuntu": "bqg", "baharna arabic": "abv", "bakoko": "bkh", "balanta-ganja": "bjt", "balti": "bft", "bamenyam": "bce", "bamun": "bax", "bangwinji": "bsj", "banjar": "bjn", "bankon": "abb", "baoulé": "bci", "bara malagasy": "bhr", "barok": "bjk", "basa (cameroon)": "bas", "basa (nigeria)": "bzw", "bashkir": "ba", "basque": "eu", "batak mandailing": "btm", "batanga": "bnm", "bateri": "btv", "bats": "bbl", "bayot": "bda", "bebele": "beb", "belarusian": "be", "bengali": "bn", "betawi": "bew", "bhili": "bhb", "bhojpuri": "bho", "bilur": "bxf", "bima": "bhp", "bodo": "brx", "boghom": "bux", "bokyi": "bky", "bomu": "bmq", "bondei": "bou", "borgu fulfulde": "fue", "bosnian": "bs", "brahui": "brh", "braj": "bra", "breton": "br", "buduma": "bdm", "buginese": "bug", "bukharic": "bhh", "bulgarian": "bg", "bulu (cameroon)": "bum", "bundeli": "bns", "bunun": "bnn", "bura-pabir": "bwr", "burak": "bys", "burmese": "my", "burushaski": "bsk", "cacaloxtepec mixtec": "miu", "cajatambo north lima quechua": "qvl", "cakfem-mushere": "cky", "cameroon pidgin": "wes", "campidanese sardinian": "sro", "cantonese": "yue", "catalan": "ca", "cebuano": "ceb", "cen": "cen", "central kurdish": "ckb", "central nahuatl": "nhn", "central pame": "pbs", "central pashto": "pst", "central puebla nahuatl": "ncx", "central tarahumara": "tar", "central yupik": "esu", "central-eastern niger fulfulde": "fuq", "chadian arabic": "shu", "chichewa": "ny", "chichicapan zapotec": "zpv", "chiga": "cgg", "chimalapa zoque": "zoh", "chimborazo highland quichua": "qug", "chinese": "zh", "chiquián ancash quechua": "qxa", "chitwania tharu": "the", "chokwe": "cjk", "chuvash": "cv", "cibak": "ckl", "coastal konjo": "kjc", "copainalá zoque": "zoc", "cornish": "kw", "corongo ancash quechua": "qwa", "croatian": "hr", "cross river mbembe": "mfn", "cuyamecalco mixtec": "xtu", "czech": "cs", "dadiya": "dbd", "dagbani": "dag", "dameli": "dml", "danish": "da", "dargwa": "dar", "dazaga": "dzg", "deccan": "dcc", "degema": "deg", "dera (nigeria)": "kna", "dghwede": "dgh", "dhatki": "mki", "dhivehi": "dv", "dhofari arabic": "adf", "dijim-bwilim": "cfa", "dogri": "dgo", "domaaki": "dmk", "dotyali": "dty", "duala": "dua", "dutch": "nl", "dũya": "ldb", "dyula": "dyu", "eastern balochi": "bgp", "eastern bolivian guaraní": "gui", "eastern egyptian bedawi arabic": "avl", "eastern krahn": "kqo", "eastern mari": "mhr", "eastern yiddish": "ydd", "ebrié": "ebr", "eggon": "ego", "egyptian arabic": "arz", "ejagham": "etu", "eleme": "elm", "eloyi": "afo", "embu": "ebu", "english": "en", "erzya": "myv", "esan": "ish", "esperanto": "eo", "estonian": "et", "eton (cameroon)": "eto", "ewondo": "ewo", "extremaduran": "ext", "fang (equatorial guinea)": "fan", "fanti": "fat", "farefare": "gur", "fe\'fe\'": "fmp", "filipino": "fil", "filomena mata-coahuitlán totonac": "tlp", "finnish": "fi", "fipa": "fip", "french": "fr", "fulah": "ff", "galician": "gl", "gambian wolof": "wof", "ganda": "lg", "garhwali": "gbm", "gawar-bati": "gwt", "gawri": "gwc", "gbagyi": "gbr", "gbari": "gby", "geji": "gyz", "gen": "gej", "georgian": "ka", "german": "de", "geser-gorom": "ges", "gheg albanian": "aln", "ghomálá\'": "bbj", "gidar": "gid", "glavda": "glw", "goan konkani": "gom", "goaria": "gig", "goemai": "ank", "gola": "gol", "greek": "el", "guarani": "gn", "guduf-gava": "gdf", "guerrero amuzgo": "amu", "gujarati": "gu", "gujari": "gju", "gulf arabic": "afb", "gurgula": "ggg", "gusii": "guz", "gusilay": "gsl", "gweno": "gwe", "güilá zapotec": "ztu", "hadothi": "hoj", "hahon": "hah", "haitian": "ht", "hakha chin": "cnh", "hakö": "hao", "halia": "hla", "hausa": "ha", "hawaiian": "haw", "hazaragi": "haz", "hebrew": "he", "hemba": "hem", "herero": "hz", "highland konjo": "kjk", "hijazi arabic": "acw", "hindi": "hi", "huarijio": "var", "huautla mazatec": "mau", "huaxcaleca nahuatl": "nhq", "huba": "hbb", "huitepec mixtec": "mxs", "hula": "hul", "hungarian": "hu", "hunjara-kaina ke": "hkk", "hwana": "hwo", "ibibio": "ibb", "icelandic": "is", "idakho-isukha-tiriki": "ida", "idoma": "idu", "igbo": "ig", "igo": "ahl", "ikposo": "kpo", "ikwere": "ikw", "imbabura highland quichua": "qvi", "indonesian": "id", "indus kohistani": "mvy", "interlingua (international auxiliary language association)": "ia", "inupiaq": "ik", "irish": "ga", "iron ossetic": "os", "isekiri": "its", "isoko": "iso", "italian": "it", "ito": "itw", "itzá": "itz", "ixtayutla mixtec": "vmj", "izon": "ijc", "jambi malay": "jax", "japanese": "ja", "jaqaru": "jqr", "jauja wanca quechua": "qxw", "jaunsari": "jns", "javanese": "jv", "jiba": "juo", "jju": "kaj", "judeo-moroccan arabic": "aju", "juxtlahuaca mixtec": "vmc", "kabardian": "kbd", "kabras": "lkb", "kabuverdianu": "kea", "kabyle": "kab", "kachi koli": "gjk", "kairak": "ckr", "kalabari": "ijn", "kalasha": "kls", "kalenjin": "kln", "kalkoti": "xka", "kamba": "kam", "kamo": "kcq", "kanauji": "bjj", "kanembu": "kbl", "kannada": "kn", "karekare": "kai", "kashmiri": "ks", "kathoriya tharu": "tkt", "kati": "bsh", "kazakh": "kk", "keiyo": "eyo", "khams tibetan": "khg", "khana": "ogo", "khetrani": "xhe", "khmer": "km", "khowar": "khw", "kinga": "zga", "kinnauri": "kfk", "kinyarwanda": "rw", "kirghiz": "ky", "kirya-konzəl": "fkk", "kochila tharu": "thq", "kohistani shina": "plk", "kohumono": "bcs", "kok borok": "trp", "kol (papua new guinea)": "kol", "kom (cameroon)": "bkm", "koma": "kmy", "konkani": "knn", "konzo": "koo", "korean": "ko", "korwa": "kfp", "kota (india)": "kfe", "koti": "eko", "kuanua": "ksd", "kuanyama": "kj", "kui (india)": "uki", "kulung (nigeria)": "bbu", "kuot": "kto", "kushi": "kuh", "kwambi": "kwm", "kwasio": "nmg", "lala-roba": "lla", "lamang": "hia", "lao": "lo", "larike-wakasihu": "alo", "lasi": "lss", "latgalian": "ltg", "latvian": "lv", "levantine arabic": "apc", "liana-seti": "ste", "liberia kpelle": "xpe", "liberian english": "lir", "libyan arabic": "ayl", "ligurian": "lij", "lijili": "mgi", "lingala": "ln", "lithuanian": "lt", "loarki": "lrk", "logooli": "rag", "logudorese sardinian": "src", "loja highland quichua": "qvj", "loloda": "loa", "longuda": "lnu", "loxicha zapotec": "ztp", "luba-lulua": "lua", "luo": "luo", "lushai": "lus", "luxembourgish": "lb", "maasina fulfulde": "ffm", "maba (chad)": "mde", "macedo-romanian": "rup", "macedonian": "mk", "mada (cameroon)": "mxu", "mafa": "maf", "maithili": "mai", "malay": "ms", "malayalam": "ml", "mali": "gcc", "malinaltepec me\'phaa": "tcf", "maltese": "mt", "mandara": "tbf", "mandjak": "mfv", "manggarai": "mqy", "manipuri": "mni", "mansoanka": "msw", "manx": "gv", "maori": "mi", "marathi": "mr", "marghi central": "mrt", "marghi south": "mfm", "maria (india)": "mrr", "marwari (pakistan)": "mve", "masana": "mcn", "masikoro malagasy": "msh", "matsés": "mcf", "mazaltepec zapotec": "zpy", "mazatlán mazatec": "vmz", "mazatlán mixe": "mzl", "mbe": "mfo", "mbo (cameroon)": "mbo", "mbum": "mdd", "medumba": "byv", "mekeo": "mek", "meru": "mer", "mesopotamian arabic": "acm", "mewari": "mtr", "min nan chinese": "nan", "mingrelian": "xmf", "mitlatongo mixtec": "vmm", "miya": "mkf", "mokpwe": "bri", "moksha": "mdf", "mom jango": "ver", "mongolian": "mn", "moroccan arabic": "ary", "motu": "meu", "mpiemo": "mcx", "mpumpong": "mgg", "mundang": "mua", "mungaka": "mhk", "musey": "mse", "musgu": "mug", "musi": "mui", "naba": "mne", "najdi arabic": "ars", "nalik": "nal", "nawdm": "nmz", "ndonga": "ng", "neapolitan": "nap", "nepali": "npi", "ngamo": "nbh", "ngas": "anc", "ngiemboon": "nnh", "ngizim": "ngi", "ngomba": "jgo", "ngombale": "nla", "nigerian fulfulde": "fuv", "nigerian pidgin": "pcm", "nimadi": "noe", "nobiin": "fia", "north mesopotamian arabic": "ayp", "north moluccan malay": "max", "northern betsimisaraka malagasy": "bmm", "northern hindko": "hno", "northern kurdish": "kmr", "northern pame": "pmq", "northern pashto": "pbu", "northern uzbek": "uzn", "northwest gbaya": "gya", "norwegian": "no", "norwegian bokmål": "nb", "norwegian nynorsk": "nn", "notsi": "ncf", "nyankpa": "yes", "nyungwe": "nyu", "nzanyi": "nja", "nüpode huitoto": "hux", "occitan": "oc", "od": "odk", "odia": "ory", "odual": "odu", "omani arabic": "acx", "orizaba nahuatl": "nlv", "orma": "orc", "ormuri": "oru", "oromo": "om", "pahari-potwari": "phr", "paiwan": "pwn", "panjabi": "pa", "papuan malay": "pmy", "parkari koli": "kvx", "pedi": "nso", "pero": "pip", "persian": "fa", "petats": "pex", "phalura": "phl", "piemontese": "pms", "piya-kwonci": "piy", "plateau malagasy": "plt", "polish": "pl", "poqomam": "poc", "portuguese": "pt", "pulaar": "fuc", "pular": "fuf", "puno quechua": "qxp", "pushto": "ps", "pökoot": "pko", "qaqet": "byx", "quiotepec chinantec": "chq", "rana tharu": "thr", "rangi": "lag", "rapoisi": "kyx", "ratahan": "rth", "rayón zoque": "zor", "romanian": "ro", "romansh": "rm", "rombo": "rof", "rotokas": "roo", "rukai": "dru", "russian": "ru", "sacapulteco": "quv", "saidi arabic": "aec", "sakalava malagasy": "skg", "sakizaya": "szy", "saleman": "sau", "samba daka": "ccg", "samba leko": "ndi", "san felipe otlaltepec popoloca": "pow", "san francisco del mar huave": "hue", "san juan atzingo popoloca": "poe", "san martín itunyoso triqui": "trq", "san miguel el grande mixtec": "mig", "sansi": "ssi", "sanskrit": "sa", "santa ana de tusi pasco quechua": "qxt", "santa catarina albarradas zapotec": "ztn", "santali": "sat", "santiago del estero quichua": "qus", "saposa": "sps", "saraiki": "skr", "sardinian": "sc", "saya": "say", "sediq": "trv", "serbian": "sr", "seri": "sei", "shina": "scl", "shona": "sn", "siar-lak": "sjr", "sibe": "nco", "sicilian": "scn", "sihuas ancash quechua": "qws", "sikkimese": "sip", "sinaugoro": "snc", "sindhi": "sd", "sindhi bhil": "sbn", "sinhala": "si", "sinicahua mixtec": "xti", "sipacapense": "qum", "siwai": "siw", "slovak": "sk", "slovenian": "sl", "solos": "sol", "somali": "so", "soninke": "snk", "south giziga": "giz", "south ucayali ashéninka": "cpy", "southeastern nochixtlán mixtec": "mxy", "southern betsimisaraka malagasy": "bzc", "southern pashto": "pbt", "southern pastaza quechua": "qup", "soyaltepec mazatec": "vmp", "spanish": "es", "standard arabic": "arb", "standard moroccan tamazight": "zgh", "sudanese arabic": "apd", "sulka": "sua", "svan": "sva", "swahili": "sw", "swedish": "sv", "tae\'": "rob", "tahaggart tamahaq": "thv", "taita": "dav", "tajik": "tg", "tamil": "ta", "tandroy-mahafaly malagasy": "tdx", "tangale": "tan", "tanosy malagasy": "txy", "tarok": "yer", "tatar": "tt", "tedaga": "tuq", "telugu": "te", "tem": "kdh", "teop": "tio", "tepeuxila cuicatec": "cux", "tepinapa chinantec": "cte", "tera": "ttr", "terei": "buo", "termanu": "twu", "tesaka malagasy": "tkg", "tetelcingo nahuatl": "nhg", "teutila cuicatec": "cut", "thai": "th", "tibetan": "bo", "tidaá mixtec": "mtx", "tidore": "tvo", "tigak": "tgc", "tigre": "tig", "tigrinya": "ti", "tilquiapan zapotec": "zts", "tinputz": "tpz", "tlacoapa me\'phaa": "tpl", "tlacoatzintepec chinantec": "ctl", "tlingit": "tli", "toki pona": "tok", "tomoip": "tqp", "tondano": "tdn", "tonsea": "txs", "tooro": "ttj", "torau": "ttu", "torwali": "trw", "tsimihety malagasy": "xmw", "tsotso": "lto", "tswana": "tn", "tugen": "tuy", "tuki": "bag", "tula": "tul", "tulu": "tcy", "tunen": "tvu", "tungag": "lcm", "tunisian arabic": "aeb", "tupuri": "tui", "turkana": "tuv", "turkish": "tr", "turkmen": "tk", "tututepec mixtec": "mtu", "twi": "tw", "ubaghara": "byc", "uighur": "ug", "ukrainian": "uk", "umbundu": "umb", "upper sorbian": "hsb", "urdu": "ur", "ushojo": "ush", "uzbek": "uz", "vai": "vai", "vietnamese": "vi", "votic": "vot", "võro": "vro", "waci gbe": "wci", "wadiyara koli": "kxp", "waja": "wja", "wakhi": "wbl", "wanga": "lwg", "wapan": "juk", "warji": "wji", "welsh": "cy", "wemale": "weo", "western frisian": "fy", "western highland purepecha": "pua", "western juxtlahuaca mixtec": "jmx", "western maninkakan": "mlq", "western mari": "mrj", "western niger fulfulde": "fuh", "western panjabi": "pnb", "wolof": "wo", "wuzlam": "udl", "xanaguía zapotec": "ztg", "xhosa": "xh", "yace": "ekr", "yakut": "sah", "yalahatan": "jal", "yanahuanca pasco quechua": "qur", "yangben": "yav", "yaqui": "yaq", "yauyos quechua": "qux", "yekhee": "ets", "yiddish": "yi", "yidgha": "ydg", "yoruba": "yo", "yutanduchi mixtec": "mab", "zacatlán-ahuacatlán-tepetzintla nahuatl": "nhi", "zarma": "dje", "zaza": "zza", "zulu": "zu", "ömie": "aom"}}}')
ADAPTER_OPTIONS = json.loads('{"kokoro": [{"key": "text", "cli_flags": ["--text"], "type": "str", "script_default": null, "choices": null, "help": "Literal UTF-8 text.", "supervisor_controlled": true}, {"key": "text_file", "cli_flags": ["--text-file"], "type": "Path", "script_default": null, "choices": null, "help": "UTF-8 or UTF-8-BOM text file.", "supervisor_controlled": true}, {"key": "config", "cli_flags": ["--config"], "type": "Path", "script_default": null, "choices": null, "help": "JSON object of CLI defaults; explicit flags take precedence.", "supervisor_controlled": true}, {"key": "n", "cli_flags": ["--n"], "type": "int", "script_default": 1, "choices": null, "help": "Number of speech samples, with consecutive seeds.", "supervisor_controlled": true}, {"key": "seed", "cli_flags": ["--seed"], "type": "int", "script_default": 42, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "voice", "cli_flags": ["--voice"], "type": "str", "script_default": "am_fenrir", "choices": null, "help": "Preset, local .pt voice pack, or comma-separated presets.", "supervisor_controlled": false}, {"key": "voice_weights", "cli_flags": ["--voice-weights"], "type": "str", "script_default": null, "choices": null, "help": "Comma-separated nonnegative mixing weights; default equal weights.", "supervisor_controlled": false}, {"key": "language", "cli_flags": ["--language"], "type": "str", "script_default": "a", "choices": ["a", "b", "e", "f", "h", "i", "j", "p", "z"], "help": "Kokoro language code; a=US English.", "supervisor_controlled": false}, {"key": "speed", "cli_flags": ["--speed"], "type": "float", "script_default": 1.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "repo_id", "cli_flags": ["--repo-id"], "type": "str", "script_default": "hexgrad/Kokoro-82M", "choices": ["hexgrad/Kokoro-82M", "hexgrad/Kokoro-82M-v1.1-zh"], "help": null, "supervisor_controlled": false}, {"key": "checkpoint", "cli_flags": ["--checkpoint"], "type": "Path", "script_default": null, "choices": null, "help": "Local PyTorch checkpoint instead of the pinned download.", "supervisor_controlled": false}, {"key": "model_config", "cli_flags": ["--model-config"], "type": "Path", "script_default": null, "choices": null, "help": "Local model config; architecture must match the supported decoder.", "supervisor_controlled": false}, {"key": "cache_dir", "cli_flags": ["--cache-dir"], "type": "Path", "script_default": ".model_cache", "choices": null, "help": null, "supervisor_controlled": false}, {"key": "device", "cli_flags": ["--device"], "type": "str", "script_default": "cpu", "choices": null, "help": "cpu or cuda[:index].", "supervisor_controlled": true}, {"key": "max_vram_gb", "cli_flags": ["--max-vram-gb"], "type": "float", "script_default": 5.0, "choices": null, "help": "Decimal GB, at most 5; CUDA allocator target.", "supervisor_controlled": true}, {"key": "cuda_reserve_mb", "cli_flags": ["--cuda-reserve-mb"], "type": "int", "script_default": 768, "choices": null, "help": "Reserve outside the PyTorch allocator target.", "supervisor_controlled": false}, {"key": "threads", "cli_flags": ["--threads"], "type": "int", "script_default": 4, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "chunk_words", "cli_flags": ["--chunk-words"], "type": "int", "script_default": 35, "choices": null, "help": "Initial text chunk limit; long chunks are split again.", "supervisor_controlled": false}, {"key": "max_chunk_seconds", "cli_flags": ["--max-chunk-seconds"], "type": "float", "script_default": 20.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "sample_rate", "cli_flags": ["--sample-rate"], "type": "int", "script_default": 24000, "choices": null, "help": "Export sample rate; native synthesis stays 24 kHz.", "supervisor_controlled": false}, {"key": "pause_ms", "cli_flags": ["--pause-ms"], "type": "float", "script_default": 0.0, "choices": null, "help": "Explicit silence between final planned chunks.", "supervisor_controlled": false}, {"key": "gain_db", "cli_flags": ["--gain-db"], "type": "float", "script_default": 0.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "edge_fade_ms", "cli_flags": ["--edge-fade-ms"], "type": "float", "script_default": 3.0, "choices": null, "help": "Cosine fade at each chunk edge; preserves sample count; 0 disables it.", "supervisor_controlled": false}, {"key": "normalize_peak", "cli_flags": ["--normalize-peak", "--no-normalize-peak"], "type": "bool", "script_default": true, "choices": null, "help": "Reduce peaks above 0.98 without changing length.", "supervisor_controlled": false}, {"key": "subtype", "cli_flags": ["--subtype"], "type": "str", "script_default": "PCM_16", "choices": ["PCM_16", "PCM_24", "FLOAT"], "help": null, "supervisor_controlled": false}, {"key": "output_dir", "cli_flags": ["--output-dir"], "type": "Path", "script_default": "samples", "choices": null, "help": null, "supervisor_controlled": true}, {"key": "estimate_only", "cli_flags": ["--estimate-only", "--no-estimate-only"], "type": "bool", "script_default": false, "choices": null, "help": "Write a preview without generating any waveform.", "supervisor_controlled": true}, {"key": "self_test", "cli_flags": ["--self-test", "--no-self-test"], "type": "bool", "script_default": false, "choices": null, "help": "Run integer-length checks without models or third-party imports.", "supervisor_controlled": true}], "f5": [{"key": "config", "cli_flags": ["--config"], "type": "Path", "script_default": null, "choices": null, "help": "JSON object of argparse destination names; CLI overrides it", "supervisor_controlled": true}, {"key": "text", "cli_flags": ["--text"], "type": "str", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "text_file", "cli_flags": ["--text-file"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "repo", "cli_flags": ["--repo"], "type": "Path", "script_default": null, "choices": null, "help": "Official F5-TTS checkout at the pinned revision", "supervisor_controlled": false}, {"key": "reference_audio", "cli_flags": ["--reference-audio"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "reference_text", "cli_flags": ["--reference-text"], "type": "str", "script_default": null, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "checkpoint", "cli_flags": ["--checkpoint"], "type": "Path", "script_default": null, "choices": null, "help": "Optional local standard F5TTS_v1_Base EMA safetensors", "supervisor_controlled": false}, {"key": "vocab", "cli_flags": ["--vocab"], "type": "Path", "script_default": null, "choices": null, "help": "Optional matching local vocabulary", "supervisor_controlled": false}, {"key": "vocos_dir", "cli_flags": ["--vocos-dir"], "type": "Path", "script_default": null, "choices": null, "help": "Local pinned Vocos config.yaml and pytorch_model.bin", "supervisor_controlled": false}, {"key": "n", "cli_flags": ["--n"], "type": "int", "script_default": 1, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "seed", "cli_flags": ["--seed"], "type": "int", "script_default": 42, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "device", "cli_flags": ["--device"], "type": "str", "script_default": "cpu", "choices": null, "help": "cpu or cuda:N", "supervisor_controlled": true}, {"key": "output_dir", "cli_flags": ["--output-dir"], "type": "Path", "script_default": "f5_samples", "choices": null, "help": null, "supervisor_controlled": true}, {"key": "sample_rate", "cli_flags": ["--sample-rate"], "type": "int", "script_default": 24000, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "pause_ms", "cli_flags": ["--pause-ms"], "type": "float", "script_default": 0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "crossfade_ms", "cli_flags": ["--crossfade-ms"], "type": "float", "script_default": 0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "fade_ms", "cli_flags": ["--fade-ms"], "type": "float", "script_default": 5, "choices": null, "help": "Length-preserving per-chunk edge fades, 0..20 ms", "supervisor_controlled": false}, {"key": "max_vram_gb", "cli_flags": ["--max-vram-gb"], "type": "float", "script_default": 5, "choices": null, "help": "Decimal GB, maximum 5", "supervisor_controlled": true}, {"key": "estimate_only", "cli_flags": ["--estimate-only", "--no-estimate-only"], "type": "bool", "script_default": false, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "speed", "cli_flags": ["--speed"], "type": "float", "script_default": 1, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "duration_seconds", "cli_flags": ["--duration-seconds"], "type": "float", "script_default": null, "choices": null, "help": "Target generated seconds, excluding pauses/overlaps; quantized to frames", "supervisor_controlled": false}, {"key": "steps", "cli_flags": ["--steps"], "type": "int", "script_default": 32, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "cfg_strength", "cli_flags": ["--cfg-strength"], "type": "float", "script_default": 2, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "sway", "cli_flags": ["--sway"], "type": "float", "script_default": -1, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "ode_method", "cli_flags": ["--ode-method"], "type": "str", "script_default": "euler", "choices": ["euler", "midpoint", "rk4"], "help": null, "supervisor_controlled": false}, {"key": "no_epss", "cli_flags": ["--no-epss"], "type": "bool", "script_default": false, "choices": null, "help": "Use uniformly spaced flow sampling times", "supervisor_controlled": false}, {"key": "target_rms", "cli_flags": ["--target-rms"], "type": "float", "script_default": 0.1, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "max_chunk_seconds", "cli_flags": ["--max-chunk-seconds"], "type": "float", "script_default": 6, "choices": null, "help": "Bound requested generated duration per chunk", "supervisor_controlled": false}, {"key": "max_chunk_bytes", "cli_flags": ["--max-chunk-bytes"], "type": "int", "script_default": 160, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "wav_subtype", "cli_flags": ["--wav-subtype"], "type": "str", "script_default": "PCM_24", "choices": ["PCM_16", "PCM_24", "FLOAT"], "help": null, "supervisor_controlled": false}, {"key": "peak_limit", "cli_flags": ["--peak-limit"], "type": "float", "script_default": 0.98, "choices": null, "help": "Peak ceiling via amplitude-only attenuation, (0,1]", "supervisor_controlled": false}], "emotivoice": [{"key": "config", "cli_flags": ["--config"], "type": "Path", "script_default": null, "choices": null, "help": "JSON defaults; CLI overrides; keys use underscores", "supervisor_controlled": true}, {"key": "text", "cli_flags": ["--text"], "type": "str", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "text_file", "cli_flags": ["--text-file"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "repo", "cli_flags": ["--repo"], "type": "Path", "script_default": null, "choices": null, "help": "Pinned official source checkout", "supervisor_controlled": false}, {"key": "generator_checkpoint", "cli_flags": ["--generator-checkpoint"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "style_checkpoint", "cli_flags": ["--style-checkpoint"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "model_yaml", "cli_flags": ["--model-yaml"], "type": "Path", "script_default": null, "choices": null, "help": "Defaults to repo/config/joint/config.yaml", "supervisor_controlled": false}, {"key": "bert_path", "cli_flags": ["--bert-path"], "type": "str", "script_default": "WangZeJun/simbert-base-chinese", "choices": null, "help": "Local SimBERT directory or Hugging Face model ID", "supervisor_controlled": false}, {"key": "local_files_only", "cli_flags": ["--local-files-only", "--no-local-files-only"], "type": "bool", "script_default": false, "choices": null, "help": "Do not download SimBERT/tokenizer files", "supervisor_controlled": false}, {"key": "speaker", "cli_flags": ["--speaker"], "type": "str", "script_default": null, "choices": null, "help": "Exact speaker label; default male speaker 9017", "supervisor_controlled": false}, {"key": "speaker_index", "cli_flags": ["--speaker-index"], "type": "int", "script_default": null, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "prompt", "cli_flags": ["--prompt"], "type": "str", "script_default": "Neutral", "choices": null, "help": "Style/emotion conditioning text", "supervisor_controlled": false}, {"key": "speed", "cli_flags": ["--speed"], "type": "float", "script_default": 1.0, "choices": null, "help": "Higher is faster; integer duration scaling before decoding", "supervisor_controlled": false}, {"key": "pitch_predictor_scale", "cli_flags": ["--pitch-predictor-scale"], "type": "float", "script_default": 1.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "energy_predictor_scale", "cli_flags": ["--energy-predictor-scale"], "type": "float", "script_default": 1.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "min_phoneme_frames", "cli_flags": ["--min-phoneme-frames"], "type": "int", "script_default": 1, "choices": null, "help": "Minimum acoustic frames for lexical phones after speed scaling; 0 restores upstream behavior", "supervisor_controlled": false}, {"key": "emphasis_json", "cli_flags": ["--emphasis-json"], "type": "Path", "script_default": null, "choices": null, "help": "Phoneme-indexed duration/pitch/energy scale edits", "supervisor_controlled": false}, {"key": "max_chunk_chars", "cli_flags": ["--max-chunk-chars"], "type": "int", "script_default": 240, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "max_chunk_frames", "cli_flags": ["--max-chunk-frames"], "type": "int", "script_default": 2000, "choices": null, "help": "Reject oversized acoustic allocations before generation", "supervisor_controlled": false}, {"key": "threads", "cli_flags": ["--threads"], "type": "int", "script_default": 4, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "n", "cli_flags": ["--n"], "type": "int", "script_default": 1, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "seed", "cli_flags": ["--seed"], "type": "int", "script_default": 1234, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "device", "cli_flags": ["--device"], "type": "str", "script_default": "cpu", "choices": null, "help": "cpu or cuda[:index]", "supervisor_controlled": true}, {"key": "allow_unverified_cuda_budget", "cli_flags": ["--allow-unverified-cuda-budget", "--no-allow-unverified-cuda-budget"], "type": "bool", "script_default": false, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "output_dir", "cli_flags": ["--output-dir"], "type": "Path", "script_default": "emotivoice_samples", "choices": null, "help": null, "supervisor_controlled": true}, {"key": "sample_rate", "cli_flags": ["--sample-rate"], "type": "int", "script_default": 0, "choices": null, "help": "0 uses native 16000 Hz", "supervisor_controlled": false}, {"key": "pause_ms", "cli_flags": ["--pause-ms"], "type": "float", "script_default": 0.0, "choices": null, "help": "Explicit silence between chunks only", "supervisor_controlled": false}, {"key": "edge_fade_ms", "cli_flags": ["--edge-fade-ms"], "type": "float", "script_default": 3.0, "choices": null, "help": "Cosine fade at each chunk edge; preserves sample count; 0 disables it", "supervisor_controlled": false}, {"key": "gain_db", "cli_flags": ["--gain-db"], "type": "float", "script_default": 0.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "normalize_peak", "cli_flags": ["--normalize-peak", "--no-normalize-peak"], "type": "bool", "script_default": true, "choices": null, "help": "Reduce peaks above 0.98 to avoid PCM clipping; preserves sample count", "supervisor_controlled": false}, {"key": "max_vram_gb", "cli_flags": ["--max-vram-gb"], "type": "float", "script_default": 5.0, "choices": null, "help": "Decimal GB, at most 5", "supervisor_controlled": true}, {"key": "estimate_only", "cli_flags": ["--estimate-only", "--no-estimate-only"], "type": "bool", "script_default": false, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "wav_subtype", "cli_flags": ["--wav-subtype"], "type": "str", "script_default": "PCM_16", "choices": ["PCM_16", "PCM_24", "FLOAT"], "help": null, "supervisor_controlled": false}], "styletts2": [{"key": "config", "cli_flags": ["--config"], "type": "Path", "script_default": null, "choices": null, "help": "JSON object of argparse destination names; CLI takes precedence", "supervisor_controlled": true}, {"key": "text", "cli_flags": ["--text"], "type": "str", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "text_file", "cli_flags": ["--text-file"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "repo", "cli_flags": ["--repo"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "reference_audio", "cli_flags": ["--reference-audio"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "checkpoint", "cli_flags": ["--checkpoint"], "type": "Path", "script_default": null, "choices": null, "help": "Optional local official-layout LibriTTS .pth checkpoint", "supervisor_controlled": false}, {"key": "model_config", "cli_flags": ["--model-config"], "type": "Path", "script_default": null, "choices": null, "help": "Optional local official LibriTTS config.yml", "supervisor_controlled": false}, {"key": "n", "cli_flags": ["--n"], "type": "int", "script_default": 1, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "seed", "cli_flags": ["--seed"], "type": "int", "script_default": 42, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "device", "cli_flags": ["--device"], "type": "str", "script_default": "cpu", "choices": null, "help": "cpu or cuda:N", "supervisor_controlled": true}, {"key": "output_dir", "cli_flags": ["--output-dir"], "type": "Path", "script_default": "styletts2_samples", "choices": null, "help": null, "supervisor_controlled": true}, {"key": "sample_rate", "cli_flags": ["--sample-rate"], "type": "int", "script_default": 24000, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "pause_ms", "cli_flags": ["--pause-ms"], "type": "float", "script_default": 0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "crossfade_ms", "cli_flags": ["--crossfade-ms"], "type": "float", "script_default": 0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "fade_ms", "cli_flags": ["--fade-ms"], "type": "float", "script_default": 5, "choices": null, "help": "Length-preserving per-chunk edge fades, 0..20 ms", "supervisor_controlled": false}, {"key": "max_vram_gb", "cli_flags": ["--max-vram-gb"], "type": "float", "script_default": 5, "choices": null, "help": "Decimal GB, maximum 5", "supervisor_controlled": true}, {"key": "estimate_only", "cli_flags": ["--estimate-only", "--no-estimate-only"], "type": "bool", "script_default": false, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "reuse_plan", "cli_flags": ["--reuse-plan"], "type": "Path", "script_default": null, "choices": null, "help": "A duration_plan.pt produced with identical synthesis settings", "supervisor_controlled": false}, {"key": "speed", "cli_flags": ["--speed"], "type": "float", "script_default": 1, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "alpha", "cli_flags": ["--alpha"], "type": "float", "script_default": 0.1, "choices": null, "help": "Weight of sampled acoustic style versus reference [0,1]", "supervisor_controlled": false}, {"key": "beta", "cli_flags": ["--beta"], "type": "float", "script_default": 0.3, "choices": null, "help": "Weight of sampled prosodic style versus reference [0,1]", "supervisor_controlled": false}, {"key": "diffusion_steps", "cli_flags": ["--diffusion-steps"], "type": "int", "script_default": 10, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "embedding_scale", "cli_flags": ["--embedding-scale"], "type": "float", "script_default": 1, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "style_continuity", "cli_flags": ["--style-continuity"], "type": "float", "script_default": 0.7, "choices": null, "help": "Previous sampled style weight between chunks [0,1]", "supervisor_controlled": false}, {"key": "pitch_scale", "cli_flags": ["--pitch-scale"], "type": "float", "script_default": 1, "choices": null, "help": "Multiply predicted F0 before decoding", "supervisor_controlled": false}, {"key": "max_chunk_words", "cli_flags": ["--max-chunk-words"], "type": "int", "script_default": 35, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "max_chunk_tokens", "cli_flags": ["--max-chunk-tokens"], "type": "int", "script_default": 400, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "max_chunk_seconds", "cli_flags": ["--max-chunk-seconds"], "type": "float", "script_default": 20, "choices": null, "help": "Reject overly long planned chunks before decoding", "supervisor_controlled": false}, {"key": "tail_trim_samples", "cli_flags": ["--tail-trim-samples"], "type": "int", "script_default": 50, "choices": null, "help": "Fixed per-chunk trim for notebook\'s end pulse", "supervisor_controlled": false}, {"key": "language", "cli_flags": ["--language"], "type": "str", "script_default": "en-us", "choices": ["en-us", "en-gb"], "help": null, "supervisor_controlled": false}, {"key": "wav_subtype", "cli_flags": ["--wav-subtype"], "type": "str", "script_default": "PCM_24", "choices": ["PCM_16", "PCM_24", "FLOAT"], "help": null, "supervisor_controlled": false}, {"key": "peak_limit", "cli_flags": ["--peak-limit"], "type": "float", "script_default": 0.98, "choices": null, "help": "Peak ceiling via amplitude-only attenuation, (0,1]", "supervisor_controlled": false}], "omnivoice": [{"key": "text", "cli_flags": ["--text"], "type": "str", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "text_file", "cli_flags": ["--text-file"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "config", "cli_flags": ["--config"], "type": "Path", "script_default": null, "choices": null, "help": "JSON object of CLI destinations; CLI values override it.", "supervisor_controlled": true}, {"key": "n", "cli_flags": ["--n"], "type": "int", "script_default": 1, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "seed", "cli_flags": ["--seed"], "type": "int", "script_default": 1234, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "device", "cli_flags": ["--device"], "type": "str", "script_default": "cpu", "choices": null, "help": "cpu (default) or cuda[:index]", "supervisor_controlled": true}, {"key": "output_dir", "cli_flags": ["--output-dir"], "type": "Path", "script_default": "outputs\\\\omnivoice", "choices": null, "help": null, "supervisor_controlled": true}, {"key": "sample_rate", "cli_flags": ["--sample-rate"], "type": "int", "script_default": null, "choices": null, "help": "Export Hz; native default 24000.", "supervisor_controlled": false}, {"key": "pause_ms", "cli_flags": ["--pause-ms"], "type": "float", "script_default": 0.0, "choices": null, "help": "Configured gap between manual chunks.", "supervisor_controlled": false}, {"key": "max_vram_gb", "cli_flags": ["--max-vram-gb"], "type": "float", "script_default": 5.0, "choices": null, "help": "Decimal GB, <=5; CUDA allocator budget only.", "supervisor_controlled": true}, {"key": "estimate_only", "cli_flags": ["--estimate-only"], "type": "bool", "script_default": false, "choices": null, "help": "No model download/loading or speech generation.", "supervisor_controlled": true}, {"key": "language", "cli_flags": ["--language"], "type": "str", "script_default": "en", "choices": null, "help": null, "supervisor_controlled": false}, {"key": "instruct", "cli_flags": ["--instruct"], "type": "str", "script_default": null, "choices": null, "help": "Comma-separated supported attributes, e.g. \'male, whisper\'.", "supervisor_controlled": false}, {"key": "ref_audio", "cli_flags": ["--ref-audio"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "ref_text", "cli_flags": ["--ref-text"], "type": "str", "script_default": null, "choices": null, "help": "Required exact reference transcript; ASR is never loaded.", "supervisor_controlled": false}, {"key": "prompt_cache", "cli_flags": ["--prompt-cache"], "type": "Path", "script_default": null, "choices": null, "help": "Load an existing VoiceClonePrompt or save a new one.", "supervisor_controlled": false}, {"key": "preprocess_prompt", "cli_flags": ["--preprocess-prompt", "--no-preprocess-prompt"], "type": "bool", "script_default": true, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "normalize_text", "cli_flags": ["--normalize-text", "--no-normalize-text"], "type": "bool", "script_default": false, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "speed", "cli_flags": ["--speed"], "type": "float", "script_default": 1.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "duration", "cli_flags": ["--duration"], "type": "float", "script_default": null, "choices": null, "help": "Total speech-token seconds, before inter-chunk pauses; overrides speed.", "supervisor_controlled": false}, {"key": "max_chunk_seconds", "cli_flags": ["--max-chunk-seconds"], "type": "float", "script_default": 8.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "max_reference_seconds", "cli_flags": ["--max-reference-seconds"], "type": "float", "script_default": 10.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "num_step", "cli_flags": ["--num-step"], "type": "int", "script_default": 32, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "guidance_scale", "cli_flags": ["--guidance-scale"], "type": "float", "script_default": 2.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "t_shift", "cli_flags": ["--t-shift"], "type": "float", "script_default": 0.1, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "layer_penalty_factor", "cli_flags": ["--layer-penalty-factor"], "type": "float", "script_default": 5.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "position_temperature", "cli_flags": ["--position-temperature"], "type": "float", "script_default": 5.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "class_temperature", "cli_flags": ["--class-temperature"], "type": "float", "script_default": 0.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "denoise", "cli_flags": ["--denoise", "--no-denoise"], "type": "bool", "script_default": true, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "fade_ms", "cli_flags": ["--fade-ms"], "type": "float", "script_default": 10.0, "choices": null, "help": "Length-preserving fades per manual chunk.", "supervisor_controlled": false}, {"key": "dtype", "cli_flags": ["--dtype"], "type": "str", "script_default": "auto", "choices": ["auto", "float32", "float16", "bfloat16"], "help": null, "supervisor_controlled": false}, {"key": "peak_limit", "cli_flags": ["--peak-limit"], "type": "float", "script_default": 0.98, "choices": null, "help": "Peak ceiling via amplitude-only attenuation, (0,1]", "supervisor_controlled": false}, {"key": "local_model", "cli_flags": ["--local-model"], "type": "Path", "script_default": null, "choices": null, "help": "Local snapshot of the pinned official model.", "supervisor_controlled": false}, {"key": "local_files_only", "cli_flags": ["--local-files-only"], "type": "bool", "script_default": false, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "self_test", "cli_flags": ["--self-test"], "type": "bool", "script_default": false, "choices": null, "help": "Dependency-free arithmetic/chunking checks.", "supervisor_controlled": true}], "supertonic": [{"key": "config", "cli_flags": ["--config"], "type": "Path", "script_default": null, "choices": null, "help": "JSON defaults; CLI values override them; keys use underscores", "supervisor_controlled": true}, {"key": "text", "cli_flags": ["--text"], "type": "str", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "text_file", "cli_flags": ["--text-file"], "type": "Path", "script_default": null, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "repo", "cli_flags": ["--repo"], "type": "Path", "script_default": null, "choices": null, "help": "Pinned official Supertonic source checkout", "supervisor_controlled": false}, {"key": "assets_dir", "cli_flags": ["--assets-dir"], "type": "Path", "script_default": null, "choices": null, "help": "Downloaded snapshot containing onnx/ and voice_styles/", "supervisor_controlled": false}, {"key": "voice", "cli_flags": ["--voice"], "type": "str", "script_default": "M2", "choices": null, "help": "Preset basename; M2 is deep/serious male", "supervisor_controlled": false}, {"key": "voice_style", "cli_flags": ["--voice-style"], "type": "Path", "script_default": null, "choices": null, "help": "Explicit compatible voice-style JSON instead of --voice", "supervisor_controlled": false}, {"key": "lang", "cli_flags": ["--lang"], "type": "str", "script_default": "en", "choices": null, "help": null, "supervisor_controlled": false}, {"key": "speed", "cli_flags": ["--speed"], "type": "float", "script_default": 1.0, "choices": null, "help": "Higher is faster; modifies the cached duration plan", "supervisor_controlled": false}, {"key": "steps", "cli_flags": ["--steps"], "type": "int", "script_default": 16, "choices": null, "help": "Flow generation steps; duration is unaffected", "supervisor_controlled": false}, {"key": "max_chunk_chars", "cli_flags": ["--max-chunk-chars"], "type": "int", "script_default": 300, "choices": null, "help": "Bounded sequential chunks; Korean/Japanese default is 120", "supervisor_controlled": false}, {"key": "threads", "cli_flags": ["--threads"], "type": "int", "script_default": 4, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "n", "cli_flags": ["--n"], "type": "int", "script_default": 1, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "seed", "cli_flags": ["--seed"], "type": "int", "script_default": 1234, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "device", "cli_flags": ["--device"], "type": "str", "script_default": "cpu", "choices": null, "help": "cpu or cuda[:index]; respects CUDA_VISIBLE_DEVICES", "supervisor_controlled": true}, {"key": "allow_unverified_cuda_budget", "cli_flags": ["--allow-unverified-cuda-budget", "--no-allow-unverified-cuda-budget"], "type": "bool", "script_default": false, "choices": null, "help": "Acknowledge that the ORT arena cap cannot guarantee a whole-process VRAM ceiling", "supervisor_controlled": false}, {"key": "cuda_reserve_mb", "cli_flags": ["--cuda-reserve-mb"], "type": "int", "script_default": 768, "choices": null, "help": "Decimal MB reserved outside the four CUDA arenas", "supervisor_controlled": false}, {"key": "output_dir", "cli_flags": ["--output-dir"], "type": "Path", "script_default": "supertonic_samples", "choices": null, "help": null, "supervisor_controlled": true}, {"key": "sample_rate", "cli_flags": ["--sample-rate"], "type": "int", "script_default": 0, "choices": null, "help": "Export sample rate; 0 uses native 44100 Hz", "supervisor_controlled": false}, {"key": "pause_ms", "cli_flags": ["--pause-ms"], "type": "float", "script_default": 0.0, "choices": null, "help": "Explicit silence between chunks only", "supervisor_controlled": false}, {"key": "edge_fade_ms", "cli_flags": ["--edge-fade-ms"], "type": "float", "script_default": 3.0, "choices": null, "help": "Cosine fade at each chunk edge; preserves sample count; 0 disables it", "supervisor_controlled": false}, {"key": "gain_db", "cli_flags": ["--gain-db"], "type": "float", "script_default": 0.0, "choices": null, "help": null, "supervisor_controlled": false}, {"key": "normalize_peak", "cli_flags": ["--normalize-peak", "--no-normalize-peak"], "type": "bool", "script_default": true, "choices": null, "help": "Reduce peaks above 0.98 to avoid PCM clipping; preserves sample count", "supervisor_controlled": false}, {"key": "max_vram_gb", "cli_flags": ["--max-vram-gb"], "type": "float", "script_default": 5.0, "choices": null, "help": "Decimal GB, at most 5; CUDA arena budget target", "supervisor_controlled": true}, {"key": "estimate_only", "cli_flags": ["--estimate-only", "--no-estimate-only"], "type": "bool", "script_default": false, "choices": null, "help": null, "supervisor_controlled": true}, {"key": "wav_subtype", "cli_flags": ["--wav-subtype"], "type": "str", "script_default": "PCM_16", "choices": ["PCM_16", "PCM_24", "FLOAT"], "help": null, "supervisor_controlled": false}]}')
ALGORITHM_LOCATIONS = json.loads('[{"model": "kokoro", "file": "kokoro_samples.py", "function": "plan_one", "line": 425}, {"model": "kokoro", "file": "kokoro_samples.py", "function": "plan_text", "line": 457}, {"model": "kokoro", "file": "kokoro_samples.py", "function": "synthesize", "line": 551}, {"model": "f5", "file": "f5_samples.py", "function": "build_plan", "line": 352}, {"model": "f5", "file": "f5_samples.py", "function": "main", "line": 485}, {"model": "emotivoice", "file": "emotivoice_samples.py", "function": "apply_duration_controls", "line": 348}, {"model": "emotivoice", "file": "emotivoice_samples.py", "function": "vocoder_samples", "line": 429}, {"model": "emotivoice", "file": "emotivoice_samples.py", "function": "plan", "line": 634}, {"model": "emotivoice", "file": "emotivoice_samples.py", "function": "synthesize", "line": 711}, {"model": "styletts2", "file": "styletts2_samples.py", "function": "build_plans", "line": 497}, {"model": "styletts2", "file": "styletts2_samples.py", "function": "generate_chunk", "line": 607}, {"model": "styletts2", "file": "styletts2_samples.py", "function": "main", "line": 640}, {"model": "omnivoice", "file": "omnivoice_samples.py", "function": "make_plan", "line": 461}, {"model": "omnivoice", "file": "omnivoice_samples.py", "function": "main", "line": 605}, {"model": "supertonic", "file": "supertonic_samples.py", "function": "validate_decoder", "line": 474}, {"model": "supertonic", "file": "supertonic_samples.py", "function": "plan", "line": 484}, {"model": "supertonic", "file": "supertonic_samples.py", "function": "synthesize", "line": 511}]')
REFERENCE_GUIDE = '\n## Voice and configuration reference\n\nThis guide is exhaustive for the **six embedded adapters and their pinned\ncheckpoints**. Numeric settings admit infinitely many combinations; the tables\ngive every exposed key, its domain, and its effect. Options from unrelated\ncheckpoints, training pipelines, or other frontends are outside this notebook.\nThe option inventory is also extracted directly from each adapter\'s argument\nparser, so it includes file/cache/runtime controls as well as synthesis knobs.\n\n| Model | Voice choices in this notebook | Reference WAV? | Explicit emotion / emphasis |\n|---|---|---|---|\n| Kokoro | 54 shipped presets, compatible `.pt` packs, weighted mixtures | No WAV-to-speaker encoder | No named emotion or word-emphasis control; speed and punctuation affect delivery |\n| F5-TTS | Any suitable speaker recording; no finite preset bank | WAV + exact transcript | Reference delivery conditions style; no independent named emotion, pitch, or word-emphasis key |\n| EmotiVoice | 2,014 ordered speaker labels in the pinned vocabulary | No | Text style prompt; pitch/energy predictor scales; phoneme-index emphasis |\n| StyleTTS2 | Any suitable speaker recording; optional official demo WAVs | WAV only | Reference + sampled style/prosody; continuous style weights and pitch; no named emotion key |\n| OmniVoice | Any suitable speaker recording, saved prompt, or supported attribute combinations | WAV + exact transcript, or cached prompt | Voice-design attributes including whisper; no validated named emotion or word-emphasis vocabulary in this pinned release |\n| Supertonic 3 | M1–M5 / F1–F5; compatible custom voice-style JSON | No local WAV encoder | Supported expression tags in text; no separate emotion or per-word emphasis flag |\n\n**What “all reference voices” means:** F5, StyleTTS2, and OmniVoice do not restrict\nyou to a named list of speakers. A valid recording can represent any speaker;\nrecording suitability and cloning fidelity must be checked by listening. Kokoro\nvoice packs and Supertonic style JSON are model embeddings, not audio files.\nEmotiVoice speaker labels are learned identities, not speaker WAV filenames.\n\nThe shared notebook reference field accepts a readable, finite, nonsilent **mono\n1–8 second** WAV. Prefer one speaker, steady recording level, little background\nnoise, and 3–8 seconds of continuous speech. For F5/Omni, transcribe the actual\nspoken recording exactly. A transcript is unnecessary for StyleTTS2. Individual\nadapter bounds are documented below; the shared notebook uses the narrower\n1–8 second contract. Reference preprocessing and model loading are separately\ntimed and are not part of a warmed duration-only preview.\n\n### How to change settings\n\nEdit `CONFIG`, `TEXT_CASES`, and `MODEL_PROFILES` in the experiment cell. Adapter\nkeys inside `parameters` use underscores: CLI `--cfg-strength` becomes\n`cfg_strength`. JSON/Python booleans must be actual booleans, not strings such as\n`"False"`. Omit optional keys to retain their adapter defaults; profile entries\nwhose value is `None` are omitted before invoking the script.\n\n```python\nCONFIG["speaker_mode"] = "auto"\nCONFIG["reference_audio"] = "/kaggle/input/my-voice/reference.wav"\nCONFIG["reference_text"] = "The exact words spoken in the reference recording."\n\n# Add another take configuration without changing the original profile:\nMODEL_PROFILES["kokoro"].append({\n    "name": "george_comparison", "enabled": True, "n": 2, "seed": 900,\n    "parameters": {"voice": "bm_george", "language": "b", "speed": 0.95},\n})\n```\n\nThe example is a configuration recipe; replace its WAV path and transcript\nbefore using it. It does not upload or download a voice automatically.\n\nSettings merge in this order: global sample rate/pause/take settings → installed\nmodel asset paths → profile `parameters` → reference routing → Omni\n`clone_parameters`. Thus a profile can override export rate, pause, voice,\ncompatible asset paths, and native synthesis controls. The saved per-job\n`config.json` is the authoritative resolved configuration. Standalone CLI flags\noverride a standalone JSON configuration file.\n\n### Every experiment-level setting\n\nThese are supervisor settings, distinct from the model parameters below.\n\n| `CONFIG` key | Notebook default / permitted values | Exact purpose |\n|---|---|---|\n| `install_models` | All six; list of `kokoro`, `f5`, `emotivoice`, `styletts2`, `omnivoice`, `supertonic` | Install and schedule only these models; enabled profiles for omitted models do not run |\n| `force_setup` | `False` / boolean | Request reinstall for Torch and the pip-installed Kokoro/Omni source packages; it does not delete all cached assets or force every dependency to reinstall |\n| `torch_cuda_variant` | `"auto"`, wheel name, or per-model dictionary | Auto uses cu121 for Torch 2.5.1 models and cu128 for Torch 2.8.0 Omni/Super; 2.5.1 supports cu118/cu121/cu124, 2.8 supports cu126/cu128/cu129; Super must retain cu128 in this setup |\n| `strict_cuda_driver` | `False` / boolean | If true, Super setup requires NVIDIA Linux driver >=570.26; runtime CUDA/provider preflight remains necessary either way |\n| `n` | `2`, positive integer | Takes per model/profile/text; profile-level `n` overrides it; the notebook rejects invalid signed-64-bit seed ranges |\n| `seed` | `1234`, nonnegative integer | Starting seed; takes use consecutive seeds. EmotiVoice is deterministic, so changing seed alone does not create different delivery |\n| `sample_rate` | `48000`, integer | Export Hz for all models unless overridden in a profile; synthesis remains at the model\'s native rate. Adapter-specific bounds apply |\n| `pause_ms` | `0.0`, nonnegative number | Extra digital silence between planned chunks; native punctuation/silence remains. Profile override allowed; F5/Style cannot combine a positive pause with a positive crossfade |\n| `max_vram_gb` | `5.0`, `(0,5]` | Decimal GB target per GPU; passed to adapters and external monitor. Allocator limits plus NVML polling are not an instantaneous process-wide certificate |\n| `nvml_interval_ms` | `10.0`, `[1,1000]` | Requested memory polling interval; actual polling can be slower under load |\n| `enforce_observed_budget` | `True` / boolean | Stop workers after a sampled overrun and retain partial files/logs; false records overruns without budget termination |\n| `budget_scope` | `"device_total"` or `"process_tree"` | Device total includes baseline/other processes. Process tree counts tracked worker/descendants; if per-process accounting is unavailable it falls back to device total |\n| `job_timeout_seconds` | `3600`, positive seconds | Maximum worker lifetime including interpreter startup, model/reference loading, planning, and all takes |\n| `gpu_release_timeout_seconds` | `20`, positive seconds | Wait after worker exit for tracked GPU processes to disappear before reusing a slot; release failure disables that slot |\n| `cpu_threads_per_job` | `4`, positive integer | Sets OMP/MKL environment limits and CPU quality-helper threads; an adapter\'s explicit `threads` key can separately configure its own pools |\n| `require_two_t4` | `True` / boolean | Require two visible T4s. False allows other two-GPU hardware; it does not change the fixed two-slot scheduler into a single-GPU runner |\n| `speaker_mode` | `"auto"`, `"reference"`, `"default"` | Auto uses supplied global WAV for cloning-capable models, else male defaults. Reference requires a global user WAV and blocks preset-only models. Default ignores the global WAV and uses male fallback; explicit per-profile references still work |\n| `reference_audio` | `None` or file path | Global reference for F5/Style and requested Omni cloning; if absent in auto/default mode, CPU Kokoro creates a disclosed male reference |\n| `reference_text` | `None` or string | Exact global WAV transcript for F5/Omni; Style needs only the WAV |\n| `include_reference_in_zip` | `True` / boolean | Include the shared reference WAV and transcript in the final ZIP; per-profile external WAVs are not automatically collected by this switch |\n| `preview_players` | `6`, nonnegative integer | Maximum distinct models shown as listening players in the notebook; all created WAVs still enter the ZIP |\n| `quality_checks` | `True` / boolean | Run CPU acoustic diagnostics before packaging; false skips them and ASR |\n| `asr_quality_check` | `True` / boolean | Add independent CPU/int8 ASR comparison; adds elapsed time without TTS GPU allocation |\n| `asr_model` | `"base.en"`, supported faster-whisper name or CTranslate2 path | Base English model is revision-pinned. `small.en` is a larger English alternative; use multilingual models for other languages. Other remote names are not revision-pinned by this notebook |\n| `asr_language` | `"en"` or recognizer-supported language code | Force the independent recognizer\'s language; this does not set TTS language or translate the text |\n| `quality_timeout_seconds` | `7200`, positive seconds | Timeout for the quality-helper subprocess; dependency/model downloads occur before that subprocess and are not included in this timeout |\n\n### Texts and profile fields\n\n| Field | Meaning |\n|---|---|\n| `TEXT_CASES` | List of `{name, text}` objects; each stripped text must have 10–500 whitespace-separated words. Names identify cases; complete paragraphs and punctuation are preserved |\n| `WORD_COUNTS` / `SAMPLE_PASSAGES` | Convenience defaults for the three supplied 10/100/500-word texts. Other counts require your own full text; the notebook does not repeat or cut the supplied passages |\n| Profile `name` | Human-readable experiment name, sanitized into filenames; ordinal/text/config hashes prevent collisions even if names repeat |\n| Profile `enabled` | Boolean, default true when omitted; disabled profiles create no jobs |\n| Profile `parameters` | Adapter destination keys documented below; omitted keys use script defaults. A different profile is a separate model-loading job |\n| Profile `n`, `seed` | Override global take count/seed; put them at profile level rather than inside `parameters` |\n| Omni profile `reference` | `"auto"` / `True` / `False`; auto follows the global reference choice, true requests cloning (fallback reference is possible), false selects design unless an explicit reference/cache or global reference-only mode requires cloning |\n| Omni profile `clone_parameters` | Optional adapter overrides applied only when cloning; use this for clone-specific controls. Default design `instruct` is removed unless `clone_parameters` supplies it |\n\nThe notebook owns `config`, `text`, `text_file`, `device`, `n`, `seed`,\n`output_dir`, `max_vram_gb`, `estimate_only`, and `self_test`. Do not place them\ninside `parameters` or `clone_parameters`; use their supervisor/profile controls\nor standalone CLI. Every GPU worker sees only its assigned physical GPU and uses\nlogical `cuda:0`. Up to two jobs run together; different model names are preferred\nwhen the queue permits it.\n\n### Duration and sound quality: effects shared by the models\n\n- `speed > 1` generally shortens duration; `< 1` lengthens it. This changes native\n  duration conditioning/allocation, not the WAV playback rate. Frame rounding,\n  minimum durations, and text/reference clamps can prevent exact inverse scaling.\n- F5/Omni explicit duration selects a synthesis allocation before waveform\n  generation. It is quantized and excludes configured joins; it is not an\n  independent prediction of natural unconstrained reading time. Allocations\n  that are too short can omit words even when the total sample count is exact.\n- Voices, reference preprocessing, emotion/style, seed, text normalization,\n  chunking, speed, and joins can affect duration where they enter that model\'s\n  planner. Changing any relevant configuration requires a fresh matching plan.\n- Sampling steps usually change quality/runtime rather than the fixed allocated\n  output length. StyleTTS2\'s style diffusion occurs before duration prediction,\n  so its sampled style can also change the plan.\n- `sample_rate` changes export sampling density, not native acoustic bandwidth;\n  peak attenuation/gain and short edge fades change amplitude without changing\n  length. Crossfades overlap audio and reduce length; pauses add length. Both\n  effects are accounted for by the applicable adapter\'s plan.\n- Native rates: Kokoro/F5/Style/Omni 24 kHz, EmotiVoice 16 kHz, Supertonic 44.1 kHz.\n  PCM_16/PCM_24 set integer storage precision; FLOAT preserves floating samples\n  and may be less convenient for some players. These are not neural quality knobs.\n- ASR omissions/substitutions are fallible screening results, not a perceptual\n  quality guarantee. Silence diagnostics distinguish interior quiet spans from\n  leading/trailing silence; intended punctuation pauses can still be flagged.\n\nThe new catalog section can be inspected and exported **without loading model\nweights**. It lists all shipped speaker rows and every parser option. Catalog\nviewing does not enable all speakers as experiments: default runs remain two\ntakes for each enabled profile and each of the three passages.\n\n\n\n## Where the duration estimate happens, and why it matches exactly\n\n**The requested passage\'s waveform is not generated first.** Each adapter\ncomputes a length plan, prints/saves its preview, and then enters waveform\nsynthesis using that same plan. The notebook later reads the actual WAV header\nindependently for its comparison. A final report is produced after generation,\nbut the reported planned sample count was already computed before it.\n\nThe experiment is therefore testing a **reused synthesis-length contract**.\nIt is not testing an independent words-per-minute predictor against an arbitrary\nunconstrained TTS call. Exact sample-count equality is a consequence of controlling\nthe lengths accepted by these decoders. “100%” here means zero measured duration\nerror in successful exports, not 100% accuracy of pronunciation, word completeness,\nnatural pacing, or arbitrary future versions. A failed length contract is reported\nas an error; the scripts do not quietly rescale speech to make the score look good.\n\n### The six algorithms\n\n| Model | Computation before requested waveform synthesis | Frozen data reused by synthesis | Native chunk length |\n|---|---|---|---|\n| Kokoro | G2P → text/BERT encodings → voice-conditioned duration network. Sum sigmoid duration logits per phoneme; divide by speed, round, clamp to at least one duration unit | Integer phoneme durations, duration encodings, tokens, selected voice embedding | `600 * sum(durations)` samples at 24 kHz |\n| F5-TTS | Reference-duration/text-byte ratio or your explicit seconds → allocate mel frames; include reference/STFT/text-token minimum clamps. This is an allocation heuristic, not a learned natural-duration forecast | Exact total flow frame count and reference-slice count for every chunk | `256 * (allocated_total_frames - floor(reference_samples/256) - 1)` samples at 24 kHz |\n| EmotiVoice | G2P + text/speaker/style encodings → learned duration predictor → speed and phoneme-emphasis scaling → integer rounding/minimum lexical-phone guard | Integer durations, conditioned phoneme embeddings, pitch/energy factors | Analytic HiFiGAN convolution/transpose-convolution propagation from `sum(durations)` acoustic frames, at 16 kHz |\n| StyleTTS2 | Phonemize/BERT encode → sample latent style separately for every take/chunk → reference/style blend → learned phoneme duration network → speed/rounding/minimum | Sampled acoustic/prosodic styles, integer durations, duration embeddings, seeds | `600 * sum(durations) - tail_trim_samples` samples at 24 kHz; default fixed trim 50 |\n| OmniVoice | Pinned Unicode/text rule and reference token/text ratio (or default rate), short-text adjustment, speed and chunk limits; optional explicit seconds select a total token budget | Exact audio-token count for every chunk | `960 * target_tokens` samples at 24 kHz; token rate 25 Hz, i.e. 40 ms per token |\n| Supertonic 3 | ONNX duration predictor conditioned on text and voice DP style → divide predicted seconds by speed → use SDK float32 conversion to integer samples and latent-frame allocation | Predicted duration, exact integer sample count, latent-frame count, IDs/masks | `int64(float32_duration * 44100)` samples after removing only precomputed latent-allocation padding |\n\n**Kokoro:** durations are generated by the actual loaded model/voice combination,\nnot fitted from word count. The decoder\'s verified temporal geometry emits 600\nsamples per duration unit. Pitch/noise prediction and waveform decoding follow\nthe preview. Decoder noise may change audio between seeds without changing the\nalready frozen duration vector.\n\n**F5:** let `R=floor(reference_samples/256)`, `B` be the reference transcript\'s\nUTF-8 byte length, and `b` be the chunk\'s byte length. Without explicit seconds,\nthe requested total frames are `R + int(R*b/(B*speed))`. With explicit seconds,\nthe requested generated allocation is distributed by chunk byte lengths. The\nadapter then applies the pinned flow\'s text/reference minimum and maximum\nclamps. The preview uses this final allocation, not the raw requested seconds.\nThe flow is called with that exact `duration`; the reference region is excluded\nand centered Vocos STFT geometry gives the sample formula in the table. F5 can\npreview before loading the generator/vocoder weights, but still needs source\nutilities, a reference waveform, vocabulary, and tokenization.\n\n**EmotiVoice:** its duration predictor is conditioned on the learned speaker\nidentity, prompt/style embedding, and content. Durations are rounded after\nspeed/emphasis scaling. Spoken phonemes obey `min_phoneme_frames`, while pause\ntokens may retain zero. The acoustic length regulator receives those exact\ndurations rather than recomputing them. Output lengths are propagated using\n`Conv1d: floor((L+2p-d*(k-1)-1)/s)+1` and\n`ConvTranspose1d: (L-1)*s-2p+d*(k-1)+output_padding+1`; residual blocks are checked\nto preserve length. This computes sample count without running the vocoder.\n\n**StyleTTS2:** stochastic style generation occurs **during planning**, before\nduration prediction. Consequently each seed/take has its own preview. The\ncached style and durations are reused by the later F0/energy/waveform path;\nthere is no second independent duration draw. This preflight does meaningful\nneural computation and is not universally near-instant. A matching saved\n`reuse_plan` can skip style/duration inference, subject to the documented cache\nconstraints.\n\n**Omni:** the rule estimator chooses token lengths using Unicode-weighted text\nand a reference-rate heuristic. The iterative acoustic-token generator is given\nthose exact target lengths; its codec produces 960 samples per token. Explicit\n`duration` becomes `max(1, floor(seconds*25))` total tokens, apportioned across\nchunks with exact integer allocation. The text heuristic can choose an unsuitable\nspeaking pace, but the resulting WAV still has the chosen length. This makes\nallocation previews fast; it does not make the heuristic a <1% predictor of\nunconstrained natural speech duration. Automatic silence removal, padding, and\nupstream automatic chunking are disabled in this adapter.\n\n**Supertonic:** only the duration graph handles the passage during planning.\nThe SDK allocates `ceil(samples/3072)` latent frames, each decoding to 3072\nsamples. It exports the precomputed sample count by removing the known final\nallocation padding. This is an explicit native mechanism, not audio-content\nsilence detection. During **cold engine initialization**, the adapter does run\ntwo tiny zero-latent vocoder probes to check this geometry, plus a duration-only\nwarm-up. Those probes are not synthesized versions of your passage and are\nincluded in startup cost rather than warmed preview latency.\n\n### Joins and exported sample rate\n\nAfter planning all chunks, let `N_native` be their total samples, plus configured\npauses and minus actual configured F5/Style overlaps. The adapter calculates\nthe rational polyphase resampler length by integer ceiling:\n\n```text\nN_export = ceil(N_native * export_rate / native_rate)\nplanned_seconds = N_export / export_rate\nactual_seconds = WAV_header_frames / WAV_header_sample_rate\nbenchmark_error_percent = 100 * abs(actual_seconds - planned_seconds) / actual_seconds\n```\n\nNative punctuation/BOS/EOS silence is already part of the generated chunk\'s\nallocated length. Tiny edge fades, gain, and peak protection do not add or\nremove samples. Style\'s fixed end correction and Super\'s predetermined latent\npadding removal are disclosed and included before export-length calculation.\nNo successful recording is padded, time-stretched, or content-trimmed afterward\nto conceal a mismatch. Fixed decoding corrections and requested crossfades are\ndifferent from estimating duration by measuring the completed recording.\n\n### Where to inspect the evidence\n\nThe ZIP includes the complete Python adapters under `source/`, and each job\'s\n`config.json`, text, preview/manifest, and `generation.log` under `job_details/`.\nThe catalog explorer also shows exact file/function line locations for planning\nand synthesis. In the notebook, look for the worker\'s **Duration preflight**,\n**Style/duration preflight**, or printed **preflight/Duration preview** message\nbefore its generation progress starts.\n\n| Adapter | Preview saved before requested waveform synthesis |\n|---|---|\n| Kokoro | Per-run `preview.json`, including integer durations and predicted export samples |\n| F5 | Initial `manifest.json` with `preview` and `plan`, written before `load_models` and flow/vocoder synthesis |\n| EmotiVoice | Initial `emotivoice_manifest.json` with phoneme durations and predicted samples, before `engine.synthesize` |\n| StyleTTS2 | `duration_plan.pt` and initial `manifest.json` with per-take previews, before `generate_chunk` |\n| OmniVoice | `estimate.json` with `plan`, before model loading/token generation in the normal design path |\n| Supertonic | Initial `supertonic_manifest.json`, before passage text encoding/flow/vocoder generation |\n\nManifests that also contain output rows are updated after takes complete. Their\nfinal modification times alone do not prove the original preview was computed\nlate; inspect the call order and generation logs. Kokoro\'s separate preview and\nOmni\'s estimate file preserve standalone plan artifacts. The notebook\'s report\ncell reads these already calculated quantities and independently measures WAV\nheaders; it does not derive planned duration from those headers.\n\nModel loading, downloads, reference preparation, duration planning, waveform\nsynthesis, and independent ASR are distinct stages. A no-reference default may\nfirst generate a **separate synthetic speaker-reference recording** on CPU; that\nis not pre-generating the requested benchmark passages. The reported planning\nlatency is model-specific and excludes several cold-start costs. The 500-word\nCPU Kokoro preview observed locally was several seconds, so exactness should\nnot be confused with a universal near-instant performance guarantee.\n\n\n## Kokoro: every installed preset and every exposed control\n\nThe pinned v1.0 snapshot contains **54 speaker packs**. `f`/`m` in the second prefix position indicates the documented female/male preset family. Speaker identity and G2P language are separate controls. Use matching language prefixes for the intended accent. [Pinned voice assets](https://huggingface.co/hexgrad/Kokoro-82M/tree/f3ff3571791e39611d31c381e3a41a3af07b4987/voices), [voice guide](https://huggingface.co/hexgrad/Kokoro-82M/blob/f3ff3571791e39611d31c381e3a41a3af07b4987/VOICES.md).\n\n| `language` and frontend | All installed voice IDs (female first; male after semicolon) |\n|---|---|\n| a — US English | af_alloy, af_aoede, af_bella, af_heart, af_jessica, af_kore, af_nicole, af_nova, af_river, af_sarah, af_sky; am_adam, am_echo, am_eric, am_fenrir, am_liam, am_michael, am_onyx, am_puck, am_santa |\n| b — British English | bf_alice, bf_emma, bf_isabella, bf_lily; bm_daniel, bm_fable, bm_george, bm_lewis |\n| e — Spanish | ef_dora; em_alex, em_santa |\n| f — French | ff_siwis |\n| h — Hindi | hf_alpha, hf_beta; hm_omega, hm_psi |\n| i — Italian | if_sara; im_nicola |\n| j — Japanese | jf_alpha, jf_gongitsune, jf_nezumi, jf_tebukuro; jm_kumo |\n| p — Brazilian Portuguese | pf_dora; pm_alex, pm_santa |\n| z — Mandarin Chinese | zf_xiaobei, zf_xiaoni, zf_xiaoxiao, zf_xiaoyi; zm_yunjian, zm_yunxi, zm_yunxia, zm_yunyang |\n\n`voice` also accepts local `.pt` tensors with shape `(L,1,256)` and finite values. Comma-separated voice IDs or local packs can be mixed with `voice_weights` if shapes match. Example: `{"voice":"am_fenrir,am_michael","voice_weights":"0.7,0.3","language":"a"}`. The voice is interpolated in embedding space; its perceived timbre and quality must be auditioned. There is **no reference-WAV cloning encoder** and no native emotion prompt, SSML, calibrated pitch knob, or per-word emphasis switch in this adapter.\n\nNotebook enabled profile: `am_fenrir`, US English, speed1, initial chunk45 words. Standalone chunk default is35. The disabled `bm_george` profile uses British English. The notebook maps each installed ID in a comma-separated blend to its local pack automatically; explicit comma-separated local paths also work.\n\nEnglish uses Misaki with eSpeak fallback; Spanish/French/Hindi/Italian/Portuguese use eSpeak. Japanese requires `misaki[ja]`, Chinese requires `misaki[zh]`; the current pinned setup installs `misaki[en]`, so add the corresponding optional frontend dependencies before using `j` or `z`. [Pinned frontend](https://github.com/hexgrad/kokoro/blob/dfb907a02bba8152ca444717ca5d78747ccb4bec/kokoro/pipeline.py). The standalone second model family `hexgrad/Kokoro-82M-v1.1-zh` is selectable, but its assets are **not installed by this Kaggle setup** and its separate catalog is not represented by the54 v1.0 packs above. Keep matching checkpoint/config/voice assets together.\n\n| JSON key / CLI flag | Standalone default | Allowed values | Exact effect |\n|---|---|---|---|\n| `config` / `--config` | None | Path to a JSON object; underscore keys | Reads CLI defaults; explicit CLI flags win. Notebook profiles create these JSON files automatically. |\n| `text` / `--text` | None | UTF-8 string; exclusive with text_file | Literal text. All three adapters enforce 10..500 whitespace-delimited words. |\n| `text_file` / `--text-file` | None | UTF-8/UTF-8-BOM file; exclusive with text | Loads the complete source text. No repetition or clipping to reach a target word count. |\n| `n` / `--n` | 1 | Integer >=1 | Number of output WAVs. The notebook normally makes n=2 per profile/text case. |\n| `threads` / `--threads` | 4 | Integer >=1 | CPU inference thread count; does not select a voice or alter desired duration. |\n| `device` / `--device` | cpu | cpu, cuda, cuda:INDEX | Selects inference device. Notebook sets cuda:0 inside each GPU-isolated worker; CUDA index is local to CUDA_VISIBLE_DEVICES. |\n| `max_vram_gb` / `--max-vram-gb` | 5.0 | Finite 0 < value <=5; decimal GB | Allocator/arena target, not a certified whole-device ceiling. Notebook additionally polls total device usage through NVML. |\n| `pause_ms` / `--pause-ms` | 0.0 | Finite >=0 milliseconds | Inserts exactly round(pause_ms*native_rate/1000) silence samples between final chunks. Adds to predicted and actual duration; does not remove model-generated silence. |\n| `edge_fade_ms` / `--edge-fade-ms` | 3.0 | Finite 0..20 milliseconds | Cosine attenuation at both ends of each chunk, capped at half that chunk. Zero disables it; sample count and timing are unchanged. |\n| `gain_db` / `--gain-db` | 0.0 | Finite -120..120 dB | Multiplies the final resampled waveform by 10**(gain_db/20). Changes loudness only; excessive positive gain can trigger peak attenuation. |\n| `normalize_peak` / `--normalize-peak` | True | Boolean | Attenuates the whole recording only if its absolute peak exceeds 0.98. Does not amplify quiet audio, compress dynamics, or alter duration. |\n| `estimate_only` / `--estimate-only` | False | Boolean | Loads models and writes the reusable duration preflight/report without synthesizing a WAV. Cold loading is separate from preview latency. |\n| `seed` / `--seed` | 42 | Integer >=0; seed+n-1 <2**63 | Consecutive seeds are recorded for n samples. The supported eval decoder has no stochastic sampler, so identical settings need not produce distinct voices/audio. |\n| `voice` / `--voice` | am_fenrir | Preset ID, local .pt path, or comma-separated compatible IDs/paths | Selects length-indexed speaker embeddings. Comma-separated packs are averaged with voice_weights; .wav is unsupported. |\n| `voice_weights` / `--voice-weights` | None (equal weights) | Comma-separated finite nonnegative numbers, one per voice; sum >0 | Normalized weighted average of the voice packs. Packs must have identical tensor shapes; a mix is a new interpolation, not an exact named speaker. |\n| `language` / `--language` | a | a, b, e, f, h, i, j, p, z | Selects grapheme-to-phoneme frontend/accent, independently of the voice ID. Match voice prefix to frontend for the intended language/accent. |\n| `speed` / `--speed` | 1.0 | Finite >0 | Divides predicted phoneme durations by speed, then rounds and clamps each to >=1 frame before acoustic decoding. Higher is faster; rounding prevents exact inverse scaling, especially at extremes. |\n| `repo_id` / `--repo-id` | hexgrad/Kokoro-82M | hexgrad/Kokoro-82M or hexgrad/Kokoro-82M-v1.1-zh | Selects a pinned config/checkpoint/voice download family. This notebook installs v1.0 only; selecting v1.1-zh needs separate matching assets and setup changes. |\n| `checkpoint` / `--checkpoint` | None (pinned download) | Path to compatible PyTorch checkpoint | Replaces the downloaded model weights. Architecture and model_config must match; a fine-tune can change sound and durations. |\n| `model_config` / `--model-config` | None (pinned download) | Path to compatible JSON config | Defines model architecture. Adapter rejects unsupported decoder upsampling/ISTFT geometry because the exact 600-sample duration mapping would no longer be established. |\n| `cache_dir` / `--cache-dir` | <script directory>/.model_cache | Directory path | Stores downloaded config/checkpoint/preset files. No acoustic control. |\n| `cuda_reserve_mb` / `--cuda-reserve-mb` | 768 | Integer >=0; must leave allocator budget | Reserves decimal MB outside the PyTorch allocation target for context/other allocations; does not itself cap those allocations. |\n| `chunk_words` / `--chunk-words` | 35 | Integer >=1 | Initial word bound; sentence/clause boundaries preferred. G2P/context or max_chunk_seconds can split further. Changes prosody and duration because each chunk is independently planned. |\n| `max_chunk_seconds` / `--max-chunk-seconds` | 20.0 | Finite >0 seconds | Splits any predicted overlong chunk and replans it before synthesis. An indivisible overlong token is rejected instead of truncated. |\n| `sample_rate` / `--sample-rate` | 24000 | Integer 8000..192000 Hz | Export rate only; native synthesis remains 24 kHz. Rational resampling predicts the exact ceil(native_samples*export_rate/24000) count for the concatenated recording. |\n| `subtype` / `--subtype` | PCM_16 | PCM_16, PCM_24, FLOAT | WAV sample encoding/precision, not sample rate. Does not change samples or duration. |\n| `output_dir` / `--output-dir` | samples | Directory path | Writes a unique timestamp/UUID run directory with WAVs, preview and manifest. |\n| `self_test` / `--self-test` | False | Boolean | Runs pure integer-length/chunk checks without loading models or validating ordinary generation inputs; diagnostic mode only. |\n\n## EmotiVoice: the complete trained speaker table and controls\n\nThe pinned pretrained table has **2,014 unique speaker labels** and ordered indices0..2013. The searchable catalog cell contains **every row**, including upstream name/gender where provided, and exports the whole table. [Exact ordered labels](https://github.com/netease-youdao/EmotiVoice/blob/59f0f36de4db12825f4705dd4e0780d79dd6bb01/data/youdao/text/speaker2), [name/gender metadata](https://github.com/netease-youdao/EmotiVoice/blob/59f0f36de4db12825f4705dd4e0780d79dd6bb01/data/youdao/text/README.md).\n\nSelect `speaker` as a **string label**, for example`"9017"`, not an embedding index. Selected known voices:9017 John Van Stan (male; default),6097 Phil Benson (male),6671 Tony Oliva (male),6670 Mike Pelton (male),8051 Maria Kasper (female),11614 Sylviamb (female),9136 Helen Taylor (female),11697 Celine Major (female),92 Cori Samuel (female),12787 LikeManyWaters (female). These are examples; all2,014 are listed in the catalog. Gender/name entries are upstream metadata, not measured pitch/style guarantees.\n\nEnglish and Chinese text, including their mixed frontend, are supported. There is **no arbitrary reference-WAV identity input** in this pretrained adapter. The upstream personal-data cloning workflow is training/fine-tuning rather than instant WAV conditioning; new voices require matching retrained weights and speaker tables. `prompt` accepts arbitrary short conditioning text; upstream demonstrates emotions such as Happy/Excited/Sad/Angry. There is no finite exhaustive prompt dictionary, no guaranteed emotion intensity, and changing emotion can also change duration. [Pinned model documentation](https://github.com/netease-youdao/EmotiVoice/blob/59f0f36de4db12825f4705dd4e0780d79dd6bb01/README.md).\n\nNotebook enabled profile: speaker`"9017"`, prompt`"Neutral"`, speed/pitch-predictor/energy-predictor scales1; the disabled comparison changes the prompt to`"Sad"`. This eval model is deterministic: n repeats with different seeds have identical audio for unchanged inputs.\n\n`emphasis_json` uses **zero-based chunk and phoneme indices**, including boundary tokens. First run `estimate_only=True` to inspect `chunks[].phonemes` in the manifest, then write edits such as`[{"chunk":0,"phoneme":2,"duration_scale":1.2,"pitch_scale":1.1,"energy_scale":1.2}]` to an accessible JSON file and set its path. If chunking/text/speaker/prompt/speed changes, replan and inspect indices again. Each optional scale defaults1; all must be0.25..4. This is direct phoneme control, not SSML or automatic English-word alignment.\n\n| JSON key / CLI flag | Standalone default | Allowed values | Exact effect |\n|---|---|---|---|\n| `config` / `--config` | None | Path to a JSON object; underscore keys | Reads CLI defaults; explicit CLI flags win. Notebook profiles create these JSON files automatically. |\n| `text` / `--text` | None | UTF-8 string; exclusive with text_file | Literal text. All three adapters enforce 10..500 whitespace-delimited words. |\n| `text_file` / `--text-file` | None | UTF-8/UTF-8-BOM file; exclusive with text | Loads the complete source text. No repetition or clipping to reach a target word count. |\n| `n` / `--n` | 1 | Integer >=1 | Number of output WAVs. The notebook normally makes n=2 per profile/text case. |\n| `threads` / `--threads` | 4 | Integer >=1 | CPU inference thread count; does not select a voice or alter desired duration. |\n| `device` / `--device` | cpu | cpu, cuda, cuda:INDEX | Selects inference device. Notebook sets cuda:0 inside each GPU-isolated worker; CUDA index is local to CUDA_VISIBLE_DEVICES. |\n| `max_vram_gb` / `--max-vram-gb` | 5.0 | Finite 0 < value <=5; decimal GB | Allocator/arena target, not a certified whole-device ceiling. Notebook additionally polls total device usage through NVML. |\n| `pause_ms` / `--pause-ms` | 0.0 | Finite >=0 milliseconds | Inserts exactly round(pause_ms*native_rate/1000) silence samples between final chunks. Adds to predicted and actual duration; does not remove model-generated silence. |\n| `edge_fade_ms` / `--edge-fade-ms` | 3.0 | Finite 0..20 milliseconds | Cosine attenuation at both ends of each chunk, capped at half that chunk. Zero disables it; sample count and timing are unchanged. |\n| `gain_db` / `--gain-db` | 0.0 | Finite -120..120 dB | Multiplies the final resampled waveform by 10**(gain_db/20). Changes loudness only; excessive positive gain can trigger peak attenuation. |\n| `normalize_peak` / `--normalize-peak` | True | Boolean | Attenuates the whole recording only if its absolute peak exceeds 0.98. Does not amplify quiet audio, compress dynamics, or alter duration. |\n| `estimate_only` / `--estimate-only` | False | Boolean | Loads models and writes the reusable duration preflight/report without synthesizing a WAV. Cold loading is separate from preview latency. |\n| `repo` / `--repo` | None (required) | Pinned official source checkout path | Supplies the architecture, bilingual phoneme frontend, vocabulary, ordered speaker table and default config. |\n| `generator_checkpoint` / `--generator-checkpoint` | None (required) | Compatible checkpoint path containing generator state | Loads acoustic model/vocoder and speaker embeddings. Notebook uses the authenticated pinned g_00140000 asset. |\n| `style_checkpoint` / `--style-checkpoint` | None (required) | Compatible checkpoint path containing model state | Loads SimBERT style/content conditioning. Notebook uses pinned checkpoint_163431. |\n| `model_yaml` / `--model-yaml` | None -> repo/config/joint/config.yaml | Compatible YAML path; native sr must equal16000 | Architecture/configuration for the generator, not an independent style preset; matching weights required. |\n| `bert_path` / `--bert-path` | WangZeJun/simbert-base-chinese | Hugging Face ID or local SimBERT directory | Tokenizer/text embedding model for content and emotion/style prompts; style encoder executes on CPU even in CUDA mode. |\n| `local_files_only` / `--local-files-only` | False | Boolean | Passes offline-only loading to the tokenizer. Notebook supplies the full local SimBERT directory, avoiding inference-time downloads. |\n| `speaker` / `--speaker` | None -> label9017 | One exact string label from the 2014-row table; exclusive with speaker_index | Chooses the trained speaker identity by label, not integer embedding position. Default resolves9017, John Van Stan. |\n| `speaker_index` / `--speaker-index` | None | Integer 0..2013 for the pinned table; exclusive with speaker | Chooses an embedding row by zero-based table position. Index2 is label9017; index9017 is invalid. Custom checkpoints may have a different table. |\n| `prompt` / `--prompt` | Neutral | Free text within SimBERT context limit | Conditions style/emotion, e.g. Happy, Excited, Sad, Angry or Neutral. This is learned conditioning, not a guaranteed exhaustive emotion vocabulary or a calibrated intensity slider. |\n| `speed` / `--speed` | 1.0 | Finite0.25..4 | Applies inverse speed to predicted phoneme durations before integer rounding and Gaussian upsampling. Higher is faster; not waveform stretching. |\n| `pitch_predictor_scale` / `--pitch-predictor-scale` | 1.0 | Finite0.25..4 | Multiplies neural pitch-predictor values before their embedding. Neural units, not Hz, semitones or guaranteed perceived pitch ratios; duration is unchanged with the same cached plan. |\n| `energy_predictor_scale` / `--energy-predictor-scale` | 1.0 | Finite0.25..4 | Multiplies neural energy-predictor values before their embedding. Neural units, not dB; acoustic output can change without changing the plan\'s frame count. |\n| `min_phoneme_frames` / `--min-phoneme-frames` | 1 | Integer0..10 | After speed/emphasis rounding, clamps lexical phones to this many acoustic frames; pauses/EOS are not forced. One protects against dropped lexical phones; zero restores upstream zero-frame behavior. Can lengthen the plan. |\n| `emphasis_json` / `--emphasis-json` | None | JSON file with phoneme-indexed edit list | Optional per-chunk/per-phone duration_scale, pitch_scale, energy_scale edits; each finite0.25..4. Edits multiply global factors; repeated edits multiply again. Duration edits are incorporated into the exact plan. |\n| `max_chunk_chars` / `--max-chunk-chars` | 240 | Integer>=20 | Hard initial character bound with sentence/clause preferences. An indivisible token longer than the bound is rejected; frontend/token context can reject oversized inputs. |\n| `max_chunk_frames` / `--max-chunk-frames` | 2000 | Integer>=1 | Rejects a chunk whose planned acoustic allocation exceeds this limit. Does not silently shorten or automatically split that chunk; lower max_chunk_chars or increase speed. |\n| `seed` / `--seed` | 1234 | Integer>=0 | Logs consecutive n-sample seeds; eval EmotiVoice is deterministic, so changing seed or n does not add diversity. |\n| `allow_unverified_cuda_budget` / `--allow-unverified-cuda-budget` | False | Boolean; must beTrue for CUDA | Acknowledges that the conservative PyTorch allocator budget excludes driver/context/external memory. Notebook sets thisTrue and monitors device usage. |\n| `sample_rate` / `--sample-rate` | 0 ->16000 | 0 or integer8000..192000 Hz | Resamples the concatenated native16 kHz recording at export. Higher rates do not create new native bandwidth; count uses the exact rational-resampler ceiling. |\n| `wav_subtype` / `--wav-subtype` | PCM_16 | PCM_16, PCM_24, FLOAT | Export precision/encoding; does not affect the frame plan or WAV duration. |\n| `output_dir` / `--output-dir` | emotivoice_samples | Fresh directory path | Writes WAVs and emotivoice_manifest.json; refuses an existing manifest to avoid overwriting a prior test. |\n\n## Supertonic3: every preset, custom styles and controls\n\nThe pinned open snapshot contains **10 preset style JSONs**. [Exact preset assets](https://huggingface.co/supertone-oss-archive/supertonic-3/tree/aafc6e32416a594460b32413efc49d7fe4ce6d46/voice_styles). Descriptions below summarize the developer\'s audition guide and are subjective, not physical pitch specifications. [Developer voice descriptions](https://github.com/supertone-oss-archive/supertonic-py/blob/main/docs/voices.md).\n\n| `voice` | Documented gender | Audition guide |\n|---|---|---|\n| F1 | F | Composed, slightly low female |\n| F2 | F | Bright, youthful female |\n| F3 | F | Clear female announcer |\n| F4 | F | Confident, expressive female |\n| F5 | F | Gentle, soothing female |\n| M1 | M | Energetic male narration |\n| M2 | M | Deep, composed, serious male |\n| M3 | M | Authoritative male presenter |\n| M4 | M | Gentle, youthful male |\n| M5 | M | Warm male storyteller |\n\nThe same ten identities support all31 released language codes:`en` English,`ko` Korean,`ja` Japanese,`ar` Arabic,`bg` Bulgarian,`cs` Czech,`da` Danish,`de` German,`el` Greek,`es` Spanish,`et` Estonian,`fi` Finnish,`fr` French,`hi` Hindi,`hr` Croatian,`hu` Hungarian,`id` Indonesian,`it` Italian,`lt` Lithuanian,`lv` Latvian,`nl` Dutch,`pl` Polish,`pt` Portuguese,`ro` Romanian,`ru` Russian,`sk` Slovak,`sl` Slovenian,`sv` Swedish,`tr` Turkish,`uk` Ukrainian,`vi` Vietnamese. The pinned frontend additionally accepts`na` as a sentinel; it is not a32nd documented language. [Model card](https://huggingface.co/supertone-oss-archive/supertonic-3/blob/aafc6e32416a594460b32413efc49d7fe4ce6d46/README.md), [pinned frontend](https://github.com/supertone-oss-archive/supertonic/blob/1e9799e964ea4c0dad7cde993b65c3c813a7b373/py/helper.py). No Farsi or Mandarin preset language is listed.\n\n`voice_style` accepts an already compatible **Supertonic3** JSON, overriding`voice`. Presets contain float32 `style_ttl` tensors of shape`(1,50,256)` and `style_dp` tensors of shape`(1,8,16)`; ttl conditions acoustics and dp conditions duration. The open package contains **no local reference-WAV encoder**. The model card directs custom voice creation to [Voice Builder](https://supertonic.supertone.ai/voice-builder), where purchased styles can be downloaded for Supertonic2/3. Supplying a JSON is local inference; building that JSON through this hosted service is a separate workflow. A schema-valid custom/mixed JSON is not a guarantee of a natural or recognizable voice; this adapter provides no automatic mixing knob. [Pinned model card](https://huggingface.co/supertone-oss-archive/supertonic-3/blob/aafc6e32416a594460b32413efc49d7fe4ce6d46/README.md).\n\nNotebook enabled profile: M2, English, speed1,16 steps, initial360 characters. Standalone is300 characters, or120 automatically for Korean/Japanese when the configured value is300. M2 is the developer\'s deep/serious preset; there is no native emotion prompt, calibrated pitch slider, per-word emphasis or SSML control. Latent seeds create alternative realizations with the same preplanned length.\n\n| JSON key / CLI flag | Standalone default | Allowed values | Exact effect |\n|---|---|---|---|\n| `config` / `--config` | None | Path to a JSON object; underscore keys | Reads CLI defaults; explicit CLI flags win. Notebook profiles create these JSON files automatically. |\n| `text` / `--text` | None | UTF-8 string; exclusive with text_file | Literal text. All three adapters enforce 10..500 whitespace-delimited words. |\n| `text_file` / `--text-file` | None | UTF-8/UTF-8-BOM file; exclusive with text | Loads the complete source text. No repetition or clipping to reach a target word count. |\n| `n` / `--n` | 1 | Integer >=1 | Number of output WAVs. The notebook normally makes n=2 per profile/text case. |\n| `threads` / `--threads` | 4 | Integer >=1 | CPU inference thread count; does not select a voice or alter desired duration. |\n| `device` / `--device` | cpu | cpu, cuda, cuda:INDEX | Selects inference device. Notebook sets cuda:0 inside each GPU-isolated worker; CUDA index is local to CUDA_VISIBLE_DEVICES. |\n| `max_vram_gb` / `--max-vram-gb` | 5.0 | Finite 0 < value <=5; decimal GB | Allocator/arena target, not a certified whole-device ceiling. Notebook additionally polls total device usage through NVML. |\n| `pause_ms` / `--pause-ms` | 0.0 | Finite >=0 milliseconds | Inserts exactly round(pause_ms*native_rate/1000) silence samples between final chunks. Adds to predicted and actual duration; does not remove model-generated silence. |\n| `edge_fade_ms` / `--edge-fade-ms` | 3.0 | Finite 0..20 milliseconds | Cosine attenuation at both ends of each chunk, capped at half that chunk. Zero disables it; sample count and timing are unchanged. |\n| `gain_db` / `--gain-db` | 0.0 | Finite -120..120 dB | Multiplies the final resampled waveform by 10**(gain_db/20). Changes loudness only; excessive positive gain can trigger peak attenuation. |\n| `normalize_peak` / `--normalize-peak` | True | Boolean | Attenuates the whole recording only if its absolute peak exceeds 0.98. Does not amplify quiet audio, compress dynamics, or alter duration. |\n| `estimate_only` / `--estimate-only` | False | Boolean | Loads models and writes the reusable duration preflight/report without synthesizing a WAV. Cold loading is separate from preview latency. |\n| `repo` / `--repo` | None (required) | Pinned official source checkout path | Imports the official Unicode language frontend and compatible voice-style loader. |\n| `assets_dir` / `--assets-dir` | None (required) | Directory containing onnx/ and voice_styles/ | Contains the four pinned ONNX graphs, configuration/Unicode index and voice style JSONs. Model-file hashes are verified before use. |\n| `voice` / `--voice` | M2 | M1..M5 orF1..F5 in the pinned snapshot | Loads assets_dir/voice_styles/<voice>.json; this name is a preset, not a reference WAV or an emotion instruction. |\n| `voice_style` / `--voice-style` | None | Compatible Supertonic3 JSON path | Overrides voice. Uses style_dp for duration and style_ttl for acoustic conditioning. No local WAV-to-style encoder is provided; incompatible Supertonic2-only styles are unsupported. |\n| `lang` / `--lang` | en | 31 language codes plus frontend sentinelna | Adds the chosen language tags around normalized text. All ten presets can be used for all released languages; pronunciation/accent quality can vary. |\n| `speed` / `--speed` | 1.0 | Finite0.25..4 | Divides the voice-conditioned duration predictor\'s seconds before allocating the latent and integer sample count. Higher is faster, with no post-generation time stretching. |\n| `steps` / `--steps` | 16 | Integer>=1 | Number of iterative flow/vector-estimator steps. More steps increase work and may improve quality, without changing the predetermined duration. Not an emotion or timing control. |\n| `max_chunk_chars` / `--max-chunk-chars` | 300; automatically120 forko/ja if value is300 | Integer>=20 | Hard sequential chunk bound with sentence/clause preferences. Explicit notebook profile360 stays360 even forko/ja, so use120 for those languages if desired. |\n| `seed` / `--seed` | 1234 | Integer>=0 | Seeds NumPy latent-noise sampling; each n output uses seed+index. Changes realization/prosody while the cached duration plan stays fixed. |\n| `allow_unverified_cuda_budget` / `--allow-unverified-cuda-budget` | False | Boolean; must beTrue for CUDA | Explicitly accepts that four ONNX CUDA arena caps do not certify whole-process/device VRAM. Notebook setsTrue plus external NVML monitoring. |\n| `cuda_reserve_mb` / `--cuda-reserve-mb` | 768 | Integer>=0; reserve must be less than CUDA budget | Decimal MB removed before dividing the remaining target among four CUDA arenas. CUDA context/driver/workspaces can use memory outside those arenas. |\n| `sample_rate` / `--sample-rate` | 0 ->44100 | 0 or integer8000..192000 Hz | Export resampling of native44.1 kHz synthesis; exact ceiling of rational resampling is included in the plan. No improvement in native bandwidth from upsampling. |\n| `wav_subtype` / `--wav-subtype` | PCM_16 | PCM_16, PCM_24, FLOAT | WAV export precision/encoding only; duration remains the same. |\n| `output_dir` / `--output-dir` | supertonic_samples | Fresh directory path | Writes WAVs and supertonic_manifest.json; refuses an existing manifest to protect prior tests. |\n\n## F5-TTS v1 Base: voices and complete adapter configuration\n\n**Voice choices are open-ended:** supply speech from the speaker you want using `reference_audio` and its exact `reference_text`. There is no finite list of F5 speaker IDs and no F5-native serious-male preset. The notebook\'s male fallback is a disclosed Kokoro `am_fenrir` reference. It is a convenience prompt, and copying a synthetic prompt does not establish the same quality as a clean human reference.\n\nThe adapter accepts a finite, nonsilent **1–8 second** reference. The notebook\'s shared reference validator additionally requires **mono** audio; the standalone script averages channels and resamples to 24 kHz. Preserve complete words and a natural ending; the adapter does not trim silence or automatically transcribe the prompt. The checkpoint covers **English and Mandarin Chinese**. Community language checkpoints are separate models, not additional built-in voices, and compatibility with this fixed v1 Base architecture is not automatic. [Official language/checkpoint list](https://github.com/SWivid/F5-TTS/blob/283252563dbf91be625e0c27926acfaac449186c/src/f5_tts/infer/SHARED.md)\n\nOptional upstream example recordings are `basic_ref_en.wav`, `basic_ref_zh.wav`, and the multi-voice examples `main.flac`, `town.flac`, `country.flac` under `src/f5_tts/infer/examples/`. They are WAV/FLAC prompts, not a speaker-embedding bank. Inspect their duration and obtain the exact transcript before using them in this adapter. The English basic example\'s supplied transcript is “Some call me nature, others call me mother nature.” [Official examples and reference guidance](https://github.com/SWivid/F5-TTS/blob/283252563dbf91be625e0c27926acfaac449186c/src/f5_tts/infer/README.md)\n\nThe following table covers **every CLI/config destination**, including operational/resource fields; `config` itself is a CLI-only file selector and cannot occur inside that JSON. Use underscore keys inside `MODEL_PROFILES[\'f5\'][...][\'parameters\']`; standalone CLI flags use hyphens. The notebook owns `config`, `text`, `text_file`, `device`, `n`, `seed`, `output_dir`, `max_vram_gb`, and `estimate_only`; profile-level `n` and `seed` are separate supported fields. `repo`, `checkpoint`, `vocab`, and `vocos_dir` are installed automatically but can be replaced by compatible local resource paths. All floating-point settings must be finite. The displayed numeric bounds are validation bounds, not a claim that extreme settings produce intelligible speech.\n\n| Config key | Standalone default | Accepted values | Exact effect |\n|---|---|---|---|\n| `config` | `unset` | Optional JSON file; standalone CLI only | Loads argparse destination keys; explicit CLI arguments take precedence. Notebook writes this file for you. Unknown keys and incorrect JSON types fail validation. |\n| `text` | `unset` | Exactly one of text/text_file; 10–500 whitespace-separated words | Literal text to speak; use TEXT_CASES in the notebook. Text affects pronunciation, native punctuation pauses, chunking, and the plan. |\n| `text_file` | `unset` | UTF-8/UTF-8-BOM file; mutually exclusive with text | Reads the complete passage from a file. The supervisor creates this from TEXT_CASES. |\n| `repo` | `unset` | Required path to the exact pinned source checkout | Inference implementation. The notebook installs and supplies it; a different revision or modified checked model source is rejected. |\n| `n` | `1` | Integer ≥1 | Number of takes; set global CONFIG[\'n\'] or profile-level n. These models generate takes/chunks sequentially within their worker. |\n| `seed` | `42` | Integer; notebook accepts 0 through 2**63−n | Base random seed; take i uses seed+i and each chunk has a reproducible derived seed. Reproducibility is conditional on the same software/hardware/settings. |\n| `device` | `\'cpu\'` | cpu or available cuda:N | Standalone compute device. The notebook selects a physical GPU, exposes it to the worker, and supplies cuda:0; do not put device in a profile. |\n| `output_dir` | `\'f5_samples\'` | Fresh output directory | Writes WAVs, manifest, and previews/plans. Existing run artifacts are rejected. The notebook owns naming and directories. |\n| `sample_rate` | `24000` | Integer 8000–192000 Hz | Export rate after deterministic polyphase resampling from 24 kHz. Increasing it does not create new acoustic detail. Final duration uses the exact exported sample count, including resampling rounding. |\n| `pause_ms` | `0` | Finite ≥0; cannot be positive with crossfade_ms | Inserts this many milliseconds of zero-valued samples between generated chunks. It adds to exported duration; it does not change pauses already generated inside a chunk. |\n| `crossfade_ms` | `0` | Finite ≥0; cannot be positive with pause_ms | Linearly overlaps neighboring chunks, subtracting the actual overlap from exported length. Overlap is capped by available samples; large overlaps can superimpose words. |\n| `fade_ms` | `5` | Finite 0–20 ms | Linear amplitude fade at both ends of every native chunk, capped at half its length. It changes no sample counts. A large fade can weaken an initial/final phoneme; zero disables it. |\n| `max_vram_gb` | `5` | Finite 0<value≤5 decimal GB; CUDA also requires >0.75 GB | GPU allocator budgeting with a 0.75 GB reserve, plus observed peak checks. CUDA library allocations exist outside the allocator; sampled monitoring is not an instantaneous hard-peak certificate. Notebook also supervises NVML externally. |\n| `estimate_only` | `False` | Boolean; --estimate-only / --no-estimate-only | Writes the exact preview (and StyleTTS2 tensor plan) without waveform generation. The regular notebook benchmark generates audio, so the supervisor controls this key. |\n| `wav_subtype` | `\'PCM_24\'` | PCM_16, PCM_24, FLOAT | WAV sample encoding: signed 16-/24-bit PCM or 32-bit float. This changes precision/file size, not rate, sample count, or generation style. |\n| `peak_limit` | `0.98` | Finite 0<value≤1 | If the final resampled waveform exceeds this absolute peak, applies one uniform gain ≤1. It neither boosts quiet audio nor changes duration; the gain and original peak are logged. |\n| `reference_audio` | `unset` | Required finite nonsilent 1–8 s audio; shared notebook reference must be mono | Speaker/style prompt. Standalone reading averages multiple channels and resamples to 24 kHz; no automatic silence trimming. Clean speech from one speaker gives a more useful prompt than music, overlapping speakers, clipping, or a cut word. |\n| `reference_text` | `unset` | Required exact nonempty transcript | Conditions the spoken reference together with the waveform. Adapter appends a terminal period if needed and a separating space. Its UTF-8 byte length, with the reference duration, also defines the allocation speaking-rate estimate. |\n| `checkpoint` | `unset` | Optional local official-layout F5TTS_v1_Base EMA .safetensors | Replaces the pinned generator file. Architecture is fixed in this adapter and load is strict; Small, E2TTS, non-EMA layouts, and arbitrary unrelated checkpoints are not interchangeable. |\n| `vocab` | `unset` | Optional local matching vocabulary file | Text token-to-ID mapping; must match the generator. Otherwise uses the pinned F5TTS_v1_Base vocab. A different vocabulary is not a language-quality guarantee. |\n| `vocos_dir` | `unset` | Optional directory containing compatible config.yaml and pytorch_model.bin | Offline Vocos resources. The validated decoder uses vocos==0.1.0 and centered 24 kHz/256-hop output; incompatible temporal topology is rejected. |\n| `speed` | `1` | Finite >0 | Divides the reference-derived generated-frame allocation: >1 requests shorter/faster speech, <1 longer/slower. It changes planning, not post-generation playback. Token/reference clamps and chunk boundaries prevent a universal 1/speed duration ratio. |\n| `duration_seconds` | `unset` | Optional finite >0 seconds | Requests a TOTAL generated-content allocation divided among chunks by UTF-8 byte length, excluding reference audio and joins. Frame quantization and token/reference minimums may increase/change it: the reported preview, not this raw target, is the final duration. With this set, speed still affects chunk splitting but no longer determines each chunk\'s duration allocation. |\n| `steps` | `32` | Integer ≥2; notebook quality profile 64 | Flow-integration intervals. More intervals cost generation time and may improve quality; they do not alter the frozen allocated frame count. Improvement is not guaranteed to be monotonic. |\n| `cfg_strength` | `2` | Finite ≥0 | Classifier-free guidance strength. The pinned flow combines conditional/unconditional predictions as conditional + strength*(conditional−unconditional); near zero uses only the conditional prediction. Stronger guidance can change adherence and sound quality, but leaves the allocation fixed. |\n| `sway` | `-1` | Finite −1 through 1 | Warps integration times after EPSS/uniform schedule: t\'=t+sway*(cos(pi*t/2)−1+t). Negative values concentrate evaluation toward early flow time; zero removes this warp. Changes generation behavior, not output length. |\n| `ode_method` | `\'euler\'` | euler, midpoint, rk4 | ODE solver: approximately 1, 2, or 4 transformer evaluations per interval respectively. Higher-order solvers are more expensive; their quality is an empirical choice. The progress bar counts evaluations; length stays fixed. |\n| `no_epss` | `False` | Boolean; CLI flag --no-epss sets true | True selects uniformly spaced times BEFORE the optional sway warp. False enables pinned Empirically Pruned Step Sampling at 5/6/7/10/12/16 steps; other step counts already fall back to uniform times. At the notebook\'s 64 steps, toggling this does not change the pre-sway schedule. |\n| `target_rms` | `0.1` | Finite >0 | If the reference RMS is below this value, boosts conditioning to this RMS, then reverses that gain on generated audio. It is reference-level handling, not final loudness mastering or an emotion/intensity control. Final peak protection still applies. |\n| `max_chunk_seconds` | `6` | Finite 0.2–12 s | Bounds generated allocation per chunk; also determines the reference-derived byte budget. A clamped allocation beyond this limit (with a one-hop allowance) fails rather than truncating speech. Smaller chunks can reduce VRAM but create more joins and reset context. |\n| `max_chunk_bytes` | `160` | Integer ≥8 UTF-8 bytes | Hard text byte budget per chunk, combined with the time-derived budget. Sentence/clause boundaries are preferred; tiny final tails are merged/rebalanced when possible. A single over-budget word fails. UTF-8 bytes are not words, especially for Chinese. |\n\n**Notebook defaults versus script defaults:** the enabled quality profile raises `steps` from **32 to 64**; global defaults use **2 takes**, seed **1234**, **48 kHz export**, **0 added pause**, and assigned CUDA workers. Other F5 settings inherit the script defaults shown above unless a profile overrides them.\n\n**What the exact duration means:** the planner freezes a frame allocation derived from reference speaking rate or `duration_seconds`, then applies the model\'s required text/reference minimum. Native chunk length is `(allocated_frames − floor(reference_samples/256) − 1) * 256`. The preview accounts for joins and export-rate rounding. Every take shares that allocation; noise, solver, steps, guidance, and sway can change the sound without changing its length. This is an exact allocation contract, not an independent forecast of the voice\'s unconstrained natural speaking time. A target that is too short may make pronunciation poor even when the sample count is exact. [Pinned flow implementation](https://github.com/SWivid/F5-TTS/blob/283252563dbf91be625e0c27926acfaac449186c/src/f5_tts/model/cfm.py)\n\n**Upstream features outside this adapter:** automatic reference ASR, the upstream multi-voice `[speaker]` text markup, speech editing, no-reference experiments, alternate E2/Small architectures, BigVGAN, and content-based silence removal are not parameters here. Multiple reference voices can be tested through separate notebook profiles, one prompt per profile. There is no independent pitch, emotion-label, temperature/top-p, or exact word-emphasis control. Delivery can be influenced by the reference and punctuation, with empirical quality limits. Source is MIT; the official pretrained F5 weights are CC-BY-NC-4.0. [Pinned model card](https://huggingface.co/SWivid/F5-TTS/blob/84e5a410d9cead4de2f847e7c9369a6440bdfaca/README.md)\n\n## StyleTTS2 LibriTTS: voices and complete adapter configuration\n\n**Voice choices are open-ended:** give `reference_audio` from the desired speaker; **no transcript is required**. There is no speaker-ID selector in this adapter. The notebook\'s fallback is the same disclosed Kokoro `am_fenrir` recording, not a StyleTTS2 male preset. Standalone input accepts finite nonsilent **0.5–15 second** references and averages channels; the shared notebook path is deliberately stricter at **mono, 1–8 seconds**. References are resampled to 24 kHz and not silence-trimmed.\n\nThe installed model is the **English multi-speaker LibriTTS** checkpoint. `language=\'en-us\'/\'en-gb\'` switches English pronunciation rules, not model weights. Other StyleTTS2 checkpoints, including single-speaker LJSpeech, are not interchangeable with this adapter\'s fixed inference topology. [Official model/inference documentation](https://github.com/yl4579/StyleTTS2/blob/5cedc71c333f8d8b8551ca59378bdcc7af4c9529/README.md)\n\nThe official demonstration references are listed below. These are optional **recording filenames**, not the complete universe of clonable voices or guaranteed serious/deep male choices. The emotional filenames refer to recordings with that delivery; they do not introduce an emotion-label API. Download the [pinned reference bundle](https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/3aa7ba7f8f275ec13dce21682a61494c35089e2a/reference_audio.zip) manually if wanted, inspect each clip against the input constraints, and point a profile to its extracted audio file. The benchmark does not automatically test all examples. [Pinned official demo](https://github.com/yl4579/StyleTTS2/blob/5cedc71c333f8d8b8551ca59378bdcc7af4c9529/Demo/Inference_LibriTTS.ipynb)\n\n| Demonstration group | Reference recording filenames |\n|---|---|\n| Seen speakers | `696_92939_000016_000006.wav`, `1789_142896_000022_000005.wav` |\n| Unseen speakers | `1221-135767-0014.wav`, `5639-40744-0020.wav`, `908-157963-0027.wav`, `4077-13754-0000.wav` |\n| Zero-shot examples | `3.wav`, `4.wav`, `5.wav` |\n| Delivery examples | `anger.wav`, `sleepy.wav`, `amused.wav`, `disgusted.wav` |\n| Named examples | `Yinghao.wav`, `Gavin.wav`, `Vinay.wav`, `Nima.wav` |\n\nThe table below covers **every CLI/config destination**; `config` is CLI-only. Config keys use underscores; standalone flags use hyphens. Supervisor-controlled keys and per-profile take/seed fields work as described for F5 above. All floating-point settings must be finite; acceptance is not an audio-quality guarantee.\n\n| Config key | Standalone default | Accepted values | Exact effect |\n|---|---|---|---|\n| `config` | `unset` | Optional JSON file; standalone CLI only | Loads argparse destination keys; explicit CLI arguments take precedence. Notebook writes this file for you. Unknown keys and incorrect JSON types fail validation. |\n| `text` | `unset` | Exactly one of text/text_file; 10–500 whitespace-separated words | Literal text to speak; use TEXT_CASES in the notebook. Text affects pronunciation, native punctuation pauses, chunking, and the plan. |\n| `text_file` | `unset` | UTF-8/UTF-8-BOM file; mutually exclusive with text | Reads the complete passage from a file. The supervisor creates this from TEXT_CASES. |\n| `repo` | `unset` | Required path to the exact pinned source checkout | Inference implementation. The notebook installs and supplies it; a different revision or modified checked model source is rejected. |\n| `n` | `1` | Integer ≥1 | Number of takes; set global CONFIG[\'n\'] or profile-level n. These models generate takes/chunks sequentially within their worker. |\n| `seed` | `42` | Integer; notebook accepts 0 through 2**63−n | Base random seed; take i uses seed+i and each chunk has a reproducible derived seed. Reproducibility is conditional on the same software/hardware/settings. |\n| `device` | `\'cpu\'` | cpu or available cuda:N | Standalone compute device. The notebook selects a physical GPU, exposes it to the worker, and supplies cuda:0; do not put device in a profile. |\n| `output_dir` | `\'styletts2_samples\'` | Fresh output directory | Writes WAVs, manifest, and previews/plans. Existing run artifacts are rejected. The notebook owns naming and directories. |\n| `sample_rate` | `24000` | Integer 8000–192000 Hz | Export rate after deterministic polyphase resampling from 24 kHz. Increasing it does not create new acoustic detail. Final duration uses the exact exported sample count, including resampling rounding. |\n| `pause_ms` | `0` | Finite ≥0; cannot be positive with crossfade_ms | Inserts this many milliseconds of zero-valued samples between generated chunks. It adds to exported duration; it does not change pauses already generated inside a chunk. |\n| `crossfade_ms` | `0` | Finite ≥0; cannot be positive with pause_ms | Linearly overlaps neighboring chunks, subtracting the actual overlap from exported length. Overlap is capped by available samples; large overlaps can superimpose words. |\n| `fade_ms` | `5` | Finite 0–20 ms | Linear amplitude fade at both ends of every native chunk, capped at half its length. It changes no sample counts. A large fade can weaken an initial/final phoneme; zero disables it. |\n| `max_vram_gb` | `5` | Finite 0<value≤5 decimal GB; CUDA also requires >0.75 GB | GPU allocator budgeting with a 0.75 GB reserve, plus observed peak checks. CUDA library allocations exist outside the allocator; sampled monitoring is not an instantaneous hard-peak certificate. Notebook also supervises NVML externally. |\n| `estimate_only` | `False` | Boolean; --estimate-only / --no-estimate-only | Writes the exact preview (and StyleTTS2 tensor plan) without waveform generation. The regular notebook benchmark generates audio, so the supervisor controls this key. |\n| `wav_subtype` | `\'PCM_24\'` | PCM_16, PCM_24, FLOAT | WAV sample encoding: signed 16-/24-bit PCM or 32-bit float. This changes precision/file size, not rate, sample count, or generation style. |\n| `peak_limit` | `0.98` | Finite 0<value≤1 | If the final resampled waveform exceeds this absolute peak, applies one uniform gain ≤1. It neither boosts quiet audio nor changes duration; the gain and original peak are logged. |\n| `reference_audio` | `unset` | Required finite nonsilent 0.5–15 s audio; shared notebook reference: mono 1–8 s | Speaker/style prompt; no transcript needed. Standalone reading averages channels and resamples to 24 kHz without silence trimming. Acoustic and prosodic encoders extract a 256-dimensional reference style. |\n| `checkpoint` | `unset` | Optional compatible official-layout LibriTTS .pth checkpoint | Replaces the generator weights. Inference modules load strictly; single-speaker LJSpeech and unrelated architectures are not drop-in alternatives for this adapter. |\n| `model_config` | `unset` | Optional matching official LibriTTS config.yml | Architecture configuration paired with the checkpoint. It is not a per-generation style configuration. The adapter requires the validated HifiGAN temporal topology. |\n| `reuse_plan` | `unset` | Optional duration_plan.pt from an identical run setup | Reuses cached sampled styles, integer durations, and duration-conditioned embeddings. Fingerprint includes text, reference/checkpoint/config contents, seed/takes, style, pitch, chunk, join, and export settings. Use a fresh output_dir; changed settings fail validation. |\n| `speed` | `1` | Finite >0 | Divides each predicted phoneme duration BEFORE rounding, then clamps every duration to at least one unit. >1 usually shortens/faster; <1 lengthens/slower. Rounding/minimums mean the total is not exactly proportional to 1/speed. |\n| `alpha` | `0.1` | Finite 0–1 | Acoustic/timbre blend: alpha*sampled_acoustic + (1−alpha)*reference_acoustic. Zero maximizes direct reference timbre; one uses sampled acoustic style. Higher alpha can reduce speaker similarity; it is not an emotion label. |\n| `beta` | `0.3` | Finite 0–1 | Prosody blend: beta*sampled_prosody + (1−beta)*reference_prosody. Zero favors the reference delivery; one favors text-conditioned sampled prosody. It can change phoneme durations, expressivity, and similarity, so exact preview must be recomputed. |\n| `diffusion_steps` | `10` | Integer ≥2 | ADPM2 latent-style sampling steps. Affects preflight cost, sampled style, and therefore potentially the duration plan. This is not waveform diffusion; the cached style is reused during decoding. |\n| `embedding_scale` | `1` | Finite >0 | Text-embedding classifier-free guidance in style sampling: fixed_embedding_prediction + scale*(text_prediction−fixed_embedding_prediction). At one, uses the normal text-conditioned path; larger values emphasize its difference and add conditional/unconditional computation. Can affect style and planned duration. |\n| `style_continuity` | `0.7` | Finite 0–1 | On chunks after the first, blends previous FINAL blended style with the new sampled style: continuity*previous + (1−continuity)*new. Zero removes that carryover; one carries the previous style into this intermediate blend. Alpha/beta reference blending is still applied afterward, so one does not freeze every final chunk style. |\n| `pitch_scale` | `1` | Finite >0 | Multiplies predicted F0 immediately before HifiGAN decoding, without shifting durations. >1 usually higher pitch; <1 lower. This is a model-F0 multiplier, not pitch-preserving playback speed; extreme values can hurt naturalness and speaker identity. |\n| `max_chunk_words` | `35` | Integer 1–80; notebook quality profile 45 | Initial word budget, with preferred sentence/clause endings. Smaller chunks lower context/VRAM requirements but add joins and may weaken sentence prosody. |\n| `max_chunk_tokens` | `400` | Integer 16–500 phoneme tokens | Further recursively splits chunks that exceed the phonemized token budget, preferring sentence/clause boundaries near the split. Includes the initial special token. One over-budget word fails. |\n| `max_chunk_seconds` | `20` | Finite 1–30 s | When building a NEW plan, rejects a chunk whose predicted native audio exceeds this limit before decoding; it does not automatically resplit. Use smaller word/token chunks or an appropriate faster speed. Existing reuse_plan data bypasses this check, and this setting is currently absent from the cache fingerprint: retain the original limit when reusing a plan. |\n| `tail_trim_samples` | `50` | Integer ≥0 native 24 kHz samples | Fixed trim at every chunk\'s decoder tail, matching the official demo\'s end-pulse workaround. Default is about 2.083 ms per chunk. Zero retains full decoder output. It is included in the preview and is not content-dependent silence removal; a large trim can remove real speech or make a plan invalid. |\n| `language` | `\'en-us\'` | en-us, en-gb | Selects the eSpeak English phonemizer\'s pronunciation variety. The same English LibriTTS checkpoint is used; this does not turn it into a multilingual model or guarantee a British voice identity. |\n\n**Notebook defaults versus script defaults:** the enabled quality profile uses `max_chunk_words=45` rather than the script\'s **35**. Both use `alpha=0.1`, `beta=0.3`, `diffusion_steps=10`, `embedding_scale=1`, and `pitch_scale=1`. Global notebook settings supply 2 takes, seed 1234, 48 kHz export, zero added gap, and CUDA workers. An optional expressive profile is disabled until you enable it.\n\n**Duration and style interaction:** exact preflight actually runs BERT, latent style diffusion, and the duration predictor for **every requested take**. It caches sampled styles and integer phoneme durations for reuse; it is heavier than F5\'s arithmetic allocation and is not universally near-instant. Each native chunk has `sum(integer_durations)*600 − tail_trim_samples` samples; joins/resampling are counted afterward. Changing seed, style weights, guidance, continuity, steps, reference, speed, or pronunciation can change the plan. `pitch_scale` changes F0 during decoding without changing that duration contract, although the saved plan fingerprint conservatively includes it.\n\nFor a serious/deeper rendition, start with a clean reference that already has that timbre and delivery. `alpha=0` maximizes direct reference acoustic conditioning; low `beta` favors its prosody. Moderate `pitch_scale<1` can lower predicted F0 but cannot guarantee a convincing deeper speaker. Higher `beta` gives more sampled text-conditioned delivery; it does not select a named emotion. The text-embedding guidance formula and its compute cost are in the [pinned diffusion module](https://github.com/yl4579/StyleTTS2/blob/5cedc71c333f8d8b8551ca59378bdcc7af4c9529/Modules/diffusion/modules.py).\n\n**Upstream features outside this adapter:** direct reference-text style-transfer methods, arbitrary emotion labels, per-word SSML emphasis, other languages/checkpoints, sample-wise speaker changes, and custom diffusion samplers/sigma schedules are not exposed. This adapter fixes ADPM2 with Karras sigma min 0.0001, max 3, rho 9, and uses the official LibriTTS HifiGAN conditioning shift. It caches duration decisions rather than modifying generated speech to fit a target. There is no `duration_seconds` override for StyleTTS2.\n\n## OmniVoice: arbitrary reference voices, voice design, and every adapter control\n\nThis section describes the **pinned model and the duration-preserving Python adapter in this notebook**, not every option in other releases. There is **no finite speaker-ID catalog**: a clean recording of a single speaker provides a new reference voice. There is also a voice-design mode with the complete attribute vocabulary below. Similarity, pitch, and style adherence remain empirical; the model cannot guarantee an exact identity or a serious mood.\n\n### Choose a voice\n\n| Voice path | What you provide | What happens |\n|---|---|---|\n| Clone a recorded speaker | `ref_audio` and its exact `ref_text` | The codec turns the recording into a voice prompt. Use clean single-speaker speech; background music, overlapping voices, and an incorrect transcript can compromise pronunciation and similarity. |\n| Reuse a clone | Existing `prompt_cache` `.pt` | Loads saved reference codec tokens, transcript, and RMS; original WAV and transcript arguments must be omitted. This avoids repeatedly preparing and encoding the reference. |\n| Save a new clone | New `prompt_cache` path plus `ref_audio`/`ref_text` | The adapter prepares the reference during preflight; synthesis loads the model, encodes it, verifies the planned token count, and saves the cache. An estimate-only run does not encode/save a new prompt. |\n| Design a voice | Comma-separated `instruct` attributes | Creates a voice matching supported attributes. Default: `male, middle-aged, low pitch, british accent`. Design has no fixed identity/preset ID. |\n| Let the model choose | Explicit `instruct=""`, no reference | Resolves to upstream auto-voice mode. Omitting `instruct` or using `None` instead selects this adapter\'s male default. |\n\nUpstream recommends **3–10 seconds** of reference audio; the shared notebook reference validator accepts **mono, finite, non-silent 1–8-second WAVs**. Choose **3–8 seconds** for both paths. The standalone adapter downmixes stereo and resamples to 24 kHz, then caps the prepared/cached reference token duration using `max_reference_seconds` (default 10). It rejects an oversized reference rather than truncating speech to fit. At least one 40 ms codec token is required. A same-language reference tends to give more standard pronunciation; a cross-language reference can transfer an accent. Clone mode is the upstream model\'s most stable mode; voice design was trained on English/Chinese and may be less stable in other languages. [Pinned cloning and language guidance](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/README.md)\n\nTo use a single reference for compatible models, set the notebook\'s global `reference_audio` and `reference_text`, with `speaker_mode="auto"`. In an OmniVoice profile, `reference` may be `"auto"`, `True`, or `False`; `True` requests a reference, `False` selects design unless a reference/cache is explicitly supplied or global `speaker_mode="reference"` forces cloning. A per-profile WAV uses `parameters={"ref_audio": "/kaggle/input/.../speaker.wav", "ref_text": "Exact transcript."}`. The automatic clone route removes default male design instructions. To deliberately combine a clone with matching attributes, use `clone_parameters={"instruct": "male, low pitch"}`. Conflicting attributes usually lose to the reference; matching dialect attributes can help. [Pinned combination tips](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/tips.md)\n\n### Complete voice-design attribute vocabulary\n\nChoose **at most one value per category**. English values are case-insensitive. Chinese equivalents may be used; commas `,` or `，` are normalized. A dialect and an English accent cannot be combined. Unsupported emotion/style words such as `serious`, `angry`, `happy`, `sad`, `deep`, or `narrator` are rejected as instructions: use the actual pitch values or a reference recording with the desired delivery.\n\n| Category | All accepted values, with Chinese equivalents where available |\n|---|---|\n| Gender | `male` / `男`; `female` / `女` |\n| Age | `child` / `儿童`; `teenager` / `少年`; `young adult` / `青年`; `middle-aged` / `中年`; `elderly` / `老年` |\n| Pitch | `very low pitch` / `极低音调`; `low pitch` / `低音调`; `moderate pitch` / `中音调`; `high pitch` / `高音调`; `very high pitch` / `极高音调` |\n| Style | `whisper` / `耳语` |\n| English accent | `american accent`, `british accent`, `australian accent`, `canadian accent`, `indian accent`, `chinese accent`, `korean accent`, `japanese accent`, `portuguese accent`, `russian accent` |\n| Chinese dialect | `河南话`, `陕西话`, `四川话`, `贵州话`, `云南话`, `桂林话`, `济南话`, `石家庄话`, `甘肃话`, `宁夏话`, `青岛话`, `东北话` |\n\nAccent controls apply to English speech; dialect controls apply to Chinese speech. Leaving a category out lets the model choose it. Some combinations may be ignored. [Pinned complete voice-design reference](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/voice-design.md)\n\nThere are **no named emotion sliders and no per-word emphasis parameter** in this adapter. You can influence delivery with punctuation, a matching expressive reference, and these complete inline non-verbal tags:\n\n`[laughter]`, `[sigh]`, `[confirmation-en]`, `[question-en]`, `[question-ah]`, `[question-oh]`, `[question-ei]`, `[question-yi]`, `[surprise-ah]`, `[surprise-oh]`, `[surprise-wa]`, `[surprise-yo]`, `[dissatisfaction-hnn]`.\n\nEnglish pronunciation may be overridden with bracketed uppercase CMU phonemes, e.g. `[B EY1 S]`; Chinese pronunciations can use pinyin tone numbers. These are pronunciation/non-verbal controls, not general emotion labels or SSML. The chunker preserves complete bracket groups. [Pinned inline-control examples](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/README.md)\n\n### Generation and planning controls\n\nKeys use underscores in a profile `parameters` dictionary; the standalone CLI uses hyphens. “Default” below means **standalone script default**. The enabled notebook profile changes `num_step` to **64**, `max_chunk_seconds` to **12.0**, and `dtype` to **float16**; export is **48,000 Hz** via the global configuration. Unspecified values retain script defaults.\n\n| Key | Default; accepted values | Exactly what it changes |\n|---|---|---|\n| `language` | `"en"`; one supported ID/name, or `"none"`/`None` | Language conditioning. Full names are case-insensitive; use the lower-case IDs in the full catalog below. `none` omits language conditioning. Unknown non-null values are rejected by the adapter. It does not translate text. |\n| `instruct` | Male design above; supported attributes or `""` | Voice-design conditioning. Empty string requests auto voice. With a clone, matching instructions may reinforce the reference; the notebook\'s clone route removes design defaults unless explicitly provided in `clone_parameters`. |\n| `ref_audio` | `None`; readable audio path | Recorded speaker prompt. Requires `ref_text`. File loading/downmixing/resampling/preprocessing occurs before planning; first use is not an instant text-only preview. |\n| `ref_text` | `None`; exact nonempty transcript | The words spoken in the reference, not the text to synthesize. Required with new audio, forbidden without audio. The adapter does not load automatic reference ASR. A cache already contains this transcript. |\n| `prompt_cache` | `None`; existing/new `.pt` path | Existing path loads a clone and forbids supplying new audio simultaneously. Missing path requires new audio/transcript and is written after reference encoding for synthesis. Cached shape must be 8 codebooks by positive token count. |\n| `preprocess_prompt` | `True`; boolean | Removes long reference silences (upstream 200 ms interior/200 ms trailing/100 ms leading handling) and adds terminal punctuation if missing. Changing it changes reference token count and therefore rule-based duration allocation. Existing cache is already prepared and is unaffected. Does not remove generated silence. |\n| `normalize_text` | `False`; boolean | Expands numbers/symbols to spoken forms before both planning and synthesis, while preserving inline tags/phonemes/pinyin. English/Chinese require optional OmniVoice `[tn]` dependencies; this notebook does not install that extra by default. Other languages use a best-effort integer conversion when `num2words` supports them. Inspect the logged normalized text; no universal normalization is promised. |\n| `speed` | `1.0`; finite `>0` | Divides rule-estimated token allocation: above 1 shortens planned speech, below 1 lengthens it. This is token allocation, not playback-speed resampling. Ignored when `duration` is set. Extreme values can cause rushed/missing speech or surplus silence. |\n| `duration` | `None`; finite seconds `>0` | Overrides speed and allocates `max(1, floor(duration*25))` speech tokens across chunks: **40 ms resolution**, before explicitly added inter-chunk pauses. The final file may therefore differ from the literal requested float; the reported sample-count preview is the exact planned export. Too few tokens for the required chunks are rejected. |\n| `max_chunk_seconds` | `8.0`; finite `>=0.04` | Maximum planned speech-token duration per manual chunk, rounded down to whole tokens. Smaller values lower context/VRAM costs but produce more boundaries; larger values provide longer context at higher memory cost. Splits prefer sentence/clause boundaries; short terminal chunks are rebalanced when feasible. Changing it can change total allocation through per-chunk rounding and the short-text rule. |\n| `max_reference_seconds` | `10.0`; finite `>0` | Rejects prepared/cached prompts whose token duration exceeds the cap. Does not shorten the WAV, set output duration, or guarantee memory usage. Raising it admits longer references and can raise VRAM/inference cost. |\n| `num_step` | `32`; integer `>=1` | Number of neural iterative unmasking passes per chunk; 64 in the notebook quality profile. More passes cost more generation time and may improve quality; they do not change frozen token lengths or guarantee monotonic improvement. |\n| `guidance_scale` | `2.0`; finite `>=0` | Conditioning extrapolation uses `conditional_logp + scale*(conditional_logp-unconditional_logp)` before normalization. Zero uses the conditional prediction alone; larger values strengthen the difference. Token length stays fixed. Upstream still builds conditional/unconditional inputs, so zero is not a guaranteed half-memory mode. |\n| `t_shift` | `0.1`; finite `>0` | Warps a linear time grid by `a*t/(1+(a-1)*t)`. This changes how many masked positions are committed at each iteration. `1.0` is linear; below 1 produces smaller early increments and larger late increments in this pinned implementation. It does not change target duration. |\n| `layer_penalty_factor` | `5.0`; finite `>=0` | Subtracts `codebook_index*factor` from selection scores. Larger values favor filling earlier codec codebooks first. Zero removes that ordering preference. Duration remains fixed. |\n| `position_temperature` | `5.0`; finite `>=0` | Controls Gumbel randomness in deciding which masked positions to commit. Zero chooses positions greedily; larger values add more randomness. Different seeds can vary waveform content despite identical length. |\n| `class_temperature` | `0.0`; finite `>=0` | Zero chooses the most likely token value. Positive values sample using Gumbel noise after a fixed top-10%-of-vocabulary filter. Controls token diversity, separately from which positions get filled; length remains fixed. |\n| `denoise` | `True`; boolean | Adds a model conditioning token when reference tokens are present, requesting cleaner speech. It is not a waveform denoising filter. It has no such token effect for a reference-free first chunk; later design/auto chunks use the first generated chunk as a prompt. |\n| `fade_ms` | `10.0`; finite `[0,20]` | Applies length-preserving amplitude fades at each manual chunk edge. Zero disables fades. Helps boundary clicks; long fades can attenuate first/last phonemes. No samples are removed or crossfaded away. |\n| `peak_limit` | `0.98`; finite `(0,1]` | After export resampling, attenuates the entire WAV only if its absolute peak exceeds this ceiling. Logs gain/original peak. Prevents overdrive; does not restore distorted source speech, compress dynamics, or change length. |\n| `sample_rate` | `None` = native **24,000 Hz**; integer `[8000,192000]` | Rational resampling for the exported WAV; 48,000 in the notebook. Codec bandwidth remains native. The planner includes the exact rounded output sample count. This is unrelated to generation temperature. |\n| `pause_ms` | `0.0`; finite `>=0` | Adds this rounded number of milliseconds of zero samples **between manual chunks only**. Included in preview. It does not control pauses inside a chunk or strip model-produced gaps. Zero avoids deliberately adding boundary gaps. |\n| `dtype` | `"auto"`; `auto`, `float32`, `float16`, `bfloat16` | Auto selects float32 on CPU and float16 on CUDA. Float16 CPU is rejected. Float16 is the tested/configured T4 path; other dtypes require compatible hardware/runtime. Dtype affects compute/memory, not planned sample length. |\n\nDecoder and scheduling behavior above was checked against the [pinned model implementation](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/omnivoice/models/omnivoice.py), not inferred from the parameter names alone. [Upstream parameter reference](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/generation-parameters.md)\n\n**Why the duration preview can be exact:** without explicit duration, the adapter applies the pinned Unicode-weight/reference-rate rule, freezes per-chunk target tokens, then generates exactly those tokens. With explicit duration it freezes an allocation from your requested seconds. Native output is 960 samples per 25 Hz token at 24 kHz. Planned pauses and rational export resampling are included. Changing seed, steps, guidance, or temperatures changes content while keeping this allocation fixed. A changed reference, preprocessing, normalization, text, speed, chunk bound, or duration requires a new plan. Short-text allocation includes the upstream nonlinear boost below 50 tokens. A cached prompt avoids re-encoding; imports/reference work still make cold preflight slower than warm arithmetic. **Exact length does not certify complete words, natural pacing, or an independently accurate prediction of unconstrained speech.** [Pinned rule allocator](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/omnivoice/utils/duration.py)\n\nThe adapter deliberately disables upstream `postprocess_output` (silence removal), `pad_duration` (extra edge silence), and automatic `audio_chunk_duration`/`audio_chunk_threshold` chunking. It fixes them to `False`, `0.0`, `0.0`, and infinity, respectively, so output remains sample-exact. They are **not configurable adapter keys**; passing them raises an unknown-key error. `fade_duration` is exposed as `fade_ms`; `preprocess_prompt` is applied while preparing the reference, then disabled at generation to avoid applying it twice. Other upstream-only choices—automatic reference ASR, FlashInfer/CUDA graphs, user batch size, training/fine-tuning, and multi-speaker dialogue—are not exposed by this benchmark adapter.\n\n### Common execution controls present in the OmniVoice script\n\nThe notebook supervisor supplies text, take count, seeds, output directories, local assets, and the visible GPU device. Override generation controls in each profile rather than overriding supervisor bookkeeping.\n\n| Key | Script default / bounds | Purpose |\n|---|---|---|\n| `text`, `text_file` | Exactly one required; nonempty, at most 500 whitespace-delimited words | Direct input string or UTF-8 text-file path. The notebook tests complete 10/100/500-word passages. |\n| `config` | None; JSON object path | Loads underscore/hyphen CLI destination keys; explicit CLI values override JSON. Unknown keys and wrong types fail. `help`, `config`, and `self_test` cannot be supplied inside this JSON. |\n| `n` | `1`; integer `>=1` | Number of output takes, each reusing the same plan. |\n| `seed` | `1234`; integers with `0 <= seed` and `seed+n-1 < 2**32` | Successive take seeds are `seed`, `seed+1`, etc.; seeds are set for Torch/NumPy/CUDA. Greedy settings can reduce variation. Reproducibility across hardware is not guaranteed. |\n| `device` | `"cpu"`; `cpu`, `cuda`, `cuda:N` | Execution target. The notebook worker hides the other GPU, so `cuda:0` refers to its assigned physical T4. |\n| `max_vram_gb` | `5.0`; finite `(0,5]`, decimal GB | Caps PyTorch\'s allocator with a context reserve; the notebook additionally samples whole-device NVML usage and kills over-budget jobs. Neither mechanism certifies an unobserved instantaneous driver/library peak. |\n| `output_dir` | `outputs/omnivoice`; path | Writes estimate/manifest and numbered seed-labelled float WAVs. Existing intended sample paths are rejected to prevent silent overwrite. |\n| `estimate_only` | `False`; boolean | Writes the plan without loading weights or generating speech. New-reference audio work and dependencies can still be needed; an existing prompt cache requires the pinned package but not reference re-encoding. |\n| `local_model` | `None`; directory | Uses a local pinned-model snapshot including the official `audio_tokenizer/config.json`. Otherwise downloads the pinned Hugging Face revision. A different checkpoint is not an audited guarantee. |\n| `local_files_only` | `False`; boolean | Restricts Hugging Face snapshot retrieval to cache; notebook predownloads assets and sets this true. |\n| `self_test` | `False`; CLI flag only | Runs dependency-free arithmetic and chunking sanity checks, not model-quality or GPU validation. |\n| `help` | CLI `--help` | Prints standalone CLI usage and exits; not a JSON/profile generation setting. |\n\n### All supported OmniVoice language IDs and names\n\nThe pinned mapping contains **646 IDs and 646 names**. Examples relevant to this setup: `en` (English), `fa` (Persian), `zh` (Chinese), `ja` (Japanese), `de` (German), `fr` (French), `es` (Spanish), `hi` (Hindi), `ru` (Russian). Language coverage does not imply uniform accent or quality. Min Nan Chinese (`nan`) requires Tai-lo romanized input in this release. The complete frozen mapping is shown below and is also present in the notebook\'s machine-readable voice catalog. [Pinned language mapping](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/omnivoice/utils/lang_map.py)\n\n<details>\n<summary>Expand all 646 accepted language IDs and names</summary>\n\n| ID | Accepted full name |\n|---|---|\n| `aae` | Arbëreshë Albanian |\n| `aal` | Afade |\n| `aao` | Algerian Saharan Arabic |\n| `ab` | Abkhazian |\n| `abb` | Bankon |\n| `abn` | Abua |\n| `abr` | Abron |\n| `abs` | Ambonese Malay |\n| `abv` | Baharna Arabic |\n| `acm` | Mesopotamian Arabic |\n| `acw` | Hijazi Arabic |\n| `acx` | Omani Arabic |\n| `adf` | Dhofari Arabic |\n| `adx` | Amdo Tibetan |\n| `ady` | Adyghe |\n| `aeb` | Tunisian Arabic |\n| `aec` | Saidi Arabic |\n| `af` | Afrikaans |\n| `afb` | Gulf Arabic |\n| `afo` | Eloyi |\n| `ahl` | Igo |\n| `ahs` | Ashe |\n| `ajg` | Aja (Benin) |\n| `aju` | Judeo-Moroccan Arabic |\n| `ala` | Alago |\n| `aln` | Gheg Albanian |\n| `alo` | Larike-Wakasihu |\n| `am` | Amharic |\n| `amu` | Guerrero Amuzgo |\n| `an` | Aragonese |\n| `anc` | Ngas |\n| `ank` | Goemai |\n| `anp` | Angika |\n| `anw` | Anaang |\n| `aom` | Ömie |\n| `apc` | Levantine Arabic |\n| `apd` | Sudanese Arabic |\n| `arb` | Standard Arabic |\n| `arq` | Algerian Arabic |\n| `ars` | Najdi Arabic |\n| `ary` | Moroccan Arabic |\n| `arz` | Egyptian Arabic |\n| `as` | Assamese |\n| `ast` | Asturian |\n| `avl` | Eastern Egyptian Bedawi Arabic |\n| `awo` | Awak |\n| `ayl` | Libyan Arabic |\n| `ayp` | North Mesopotamian Arabic |\n| `az` | Azerbaijani |\n| `ba` | Bashkir |\n| `bag` | Tuki |\n| `bas` | Basa (Cameroon) |\n| `bax` | Bamun |\n| `bba` | Baatonum |\n| `bbj` | Ghomálá\' |\n| `bbl` | Bats |\n| `bbu` | Kulung (Nigeria) |\n| `bce` | Bamenyam |\n| `bci` | Baoulé |\n| `bcs` | Kohumono |\n| `bcy` | Bacama |\n| `bda` | Bayot |\n| `bde` | Bade |\n| `bdm` | Buduma |\n| `be` | Belarusian |\n| `beb` | Bebele |\n| `bew` | Betawi |\n| `bfd` | Bafut |\n| `bft` | Balti |\n| `bg` | Bulgarian |\n| `bgp` | Eastern Balochi |\n| `bhb` | Bhili |\n| `bhh` | Bukharic |\n| `bho` | Bhojpuri |\n| `bhp` | Bima |\n| `bhr` | Bara Malagasy |\n| `bjj` | Kanauji |\n| `bjk` | Barok |\n| `bjn` | Banjar |\n| `bjt` | Balanta-Ganja |\n| `bkh` | Bakoko |\n| `bkm` | Kom (Cameroon) |\n| `bky` | Bokyi |\n| `bmm` | Northern Betsimisaraka Malagasy |\n| `bmq` | Bomu |\n| `bn` | Bengali |\n| `bnm` | Batanga |\n| `bnn` | Bunun |\n| `bns` | Bundeli |\n| `bo` | Tibetan |\n| `bou` | Bondei |\n| `bqg` | Bago-Kusuntu |\n| `br` | Breton |\n| `bra` | Braj |\n| `brh` | Brahui |\n| `bri` | Mokpwe |\n| `brx` | Bodo |\n| `bs` | Bosnian |\n| `bsh` | Kati |\n| `bsj` | Bangwinji |\n| `bsk` | Burushaski |\n| `btm` | Batak Mandailing |\n| `btv` | Bateri |\n| `bug` | Buginese |\n| `bum` | Bulu (Cameroon) |\n| `buo` | Terei |\n| `bux` | Boghom |\n| `bwr` | Bura-Pabir |\n| `bxf` | Bilur |\n| `byc` | Ubaghara |\n| `bys` | Burak |\n| `byv` | Medumba |\n| `byx` | Qaqet |\n| `bzc` | Southern Betsimisaraka Malagasy |\n| `bzw` | Basa (Nigeria) |\n| `ca` | Catalan |\n| `ccg` | Samba Daka |\n| `ceb` | Cebuano |\n| `cen` | Cen |\n| `cfa` | Dijim-Bwilim |\n| `cgg` | Chiga |\n| `chq` | Quiotepec Chinantec |\n| `cjk` | Chokwe |\n| `ckb` | Central Kurdish |\n| `ckl` | Cibak |\n| `ckr` | Kairak |\n| `cky` | Cakfem-Mushere |\n| `cnh` | Hakha Chin |\n| `cpy` | South Ucayali Ashéninka |\n| `cs` | Czech |\n| `cte` | Tepinapa Chinantec |\n| `ctl` | Tlacoatzintepec Chinantec |\n| `cut` | Teutila Cuicatec |\n| `cux` | Tepeuxila Cuicatec |\n| `cv` | Chuvash |\n| `cy` | Welsh |\n| `da` | Danish |\n| `dag` | Dagbani |\n| `dar` | Dargwa |\n| `dav` | Taita |\n| `dbd` | Dadiya |\n| `dcc` | Deccan |\n| `de` | German |\n| `deg` | Degema |\n| `dgh` | Dghwede |\n| `dgo` | Dogri |\n| `dje` | Zarma |\n| `dmk` | Domaaki |\n| `dml` | Dameli |\n| `dru` | Rukai |\n| `dty` | Dotyali |\n| `dua` | Duala |\n| `dv` | Dhivehi |\n| `dyu` | Dyula |\n| `dzg` | Dazaga |\n| `ebr` | Ebrié |\n| `ebu` | Embu |\n| `ego` | Eggon |\n| `eiv` | Askopan |\n| `eko` | Koti |\n| `ekr` | Yace |\n| `el` | Greek |\n| `elm` | Eleme |\n| `en` | English |\n| `eo` | Esperanto |\n| `es` | Spanish |\n| `esu` | Central Yupik |\n| `et` | Estonian |\n| `eto` | Eton (Cameroon) |\n| `ets` | Yekhee |\n| `etu` | Ejagham |\n| `eu` | Basque |\n| `ewo` | Ewondo |\n| `ext` | Extremaduran |\n| `eyo` | Keiyo |\n| `fa` | Persian |\n| `fan` | Fang (Equatorial Guinea) |\n| `fat` | Fanti |\n| `ff` | Fulah |\n| `ffm` | Maasina Fulfulde |\n| `fi` | Finnish |\n| `fia` | Nobiin |\n| `fil` | Filipino |\n| `fip` | Fipa |\n| `fkk` | Kirya-Konzəl |\n| `fmp` | Fe\'Fe\' |\n| `fr` | French |\n| `fub` | Adamawa Fulfulde |\n| `fuc` | Pulaar |\n| `fue` | Borgu Fulfulde |\n| `fuf` | Pular |\n| `fuh` | Western Niger Fulfulde |\n| `fui` | Bagirmi Fulfulde |\n| `fuq` | Central-Eastern Niger Fulfulde |\n| `fuv` | Nigerian Fulfulde |\n| `fy` | Western Frisian |\n| `ga` | Irish |\n| `gbm` | Garhwali |\n| `gbr` | Gbagyi |\n| `gby` | Gbari |\n| `gcc` | Mali |\n| `gdf` | Guduf-Gava |\n| `gej` | Gen |\n| `ges` | Geser-Gorom |\n| `ggg` | Gurgula |\n| `gid` | Gidar |\n| `gig` | Goaria |\n| `giz` | South Giziga |\n| `gjk` | Kachi Koli |\n| `gju` | Gujari |\n| `gl` | Galician |\n| `glw` | Glavda |\n| `gn` | Guarani |\n| `gol` | Gola |\n| `gom` | Goan Konkani |\n| `gsl` | Gusilay |\n| `gu` | Gujarati |\n| `gui` | Eastern Bolivian Guaraní |\n| `gur` | Farefare |\n| `guz` | Gusii |\n| `gv` | Manx |\n| `gwc` | Gawri |\n| `gwe` | Gweno |\n| `gwt` | Gawar-Bati |\n| `gya` | Northwest Gbaya |\n| `gyz` | Geji |\n| `ha` | Hausa |\n| `hah` | Hahon |\n| `hao` | Hakö |\n| `haw` | Hawaiian |\n| `haz` | Hazaragi |\n| `hbb` | Huba |\n| `he` | Hebrew |\n| `hem` | Hemba |\n| `hi` | Hindi |\n| `hia` | Lamang |\n| `hkk` | Hunjara-Kaina Ke |\n| `hla` | Halia |\n| `hno` | Northern Hindko |\n| `hoj` | Hadothi |\n| `hr` | Croatian |\n| `hsb` | Upper Sorbian |\n| `ht` | Haitian |\n| `hu` | Hungarian |\n| `hue` | San Francisco Del Mar Huave |\n| `hul` | Hula |\n| `hux` | Nüpode Huitoto |\n| `hwo` | Hwana |\n| `hy` | Armenian |\n| `hz` | Herero |\n| `ia` | Interlingua (International Auxiliary Language Association) |\n| `ibb` | Ibibio |\n| `id` | Indonesian |\n| `ida` | Idakho-Isukha-Tiriki |\n| `idu` | Idoma |\n| `ig` | Igbo |\n| `ijc` | Izon |\n| `ijn` | Kalabari |\n| `ik` | Inupiaq |\n| `ikw` | Ikwere |\n| `is` | Icelandic |\n| `ish` | Esan |\n| `iso` | Isoko |\n| `it` | Italian |\n| `its` | Isekiri |\n| `itw` | Ito |\n| `itz` | Itzá |\n| `ja` | Japanese |\n| `jal` | Yalahatan |\n| `jax` | Jambi Malay |\n| `jgo` | Ngomba |\n| `jmx` | Western Juxtlahuaca Mixtec |\n| `jns` | Jaunsari |\n| `jqr` | Jaqaru |\n| `juk` | Wapan |\n| `juo` | Jiba |\n| `jv` | Javanese |\n| `ka` | Georgian |\n| `kab` | Kabyle |\n| `kai` | Karekare |\n| `kaj` | Jju |\n| `kam` | Kamba |\n| `kbd` | Kabardian |\n| `kbl` | Kanembu |\n| `kbt` | Abadi |\n| `kcq` | Kamo |\n| `kdh` | Tem |\n| `kea` | Kabuverdianu |\n| `keu` | Akebu |\n| `kfe` | Kota (India) |\n| `kfk` | Kinnauri |\n| `kfp` | Korwa |\n| `khg` | Khams Tibetan |\n| `khw` | Khowar |\n| `kj` | Kuanyama |\n| `kjc` | Coastal Konjo |\n| `kjk` | Highland Konjo |\n| `kk` | Kazakh |\n| `kln` | Kalenjin |\n| `kls` | Kalasha |\n| `km` | Khmer |\n| `kmr` | Northern Kurdish |\n| `kmy` | Koma |\n| `kn` | Kannada |\n| `kna` | Dera (Nigeria) |\n| `knn` | Konkani |\n| `ko` | Korean |\n| `kol` | Kol (Papua New Guinea) |\n| `koo` | Konzo |\n| `kpo` | Ikposo |\n| `kqo` | Eastern Krahn |\n| `ks` | Kashmiri |\n| `ksd` | Kuanua |\n| `ksf` | Bafia |\n| `kto` | Kuot |\n| `kuh` | Kushi |\n| `kvx` | Parkari Koli |\n| `kw` | Cornish |\n| `kwm` | Kwambi |\n| `kxp` | Wadiyara Koli |\n| `ky` | Kirghiz |\n| `kyx` | Rapoisi |\n| `lag` | Rangi |\n| `lb` | Luxembourgish |\n| `lcm` | Tungag |\n| `ldb` | DũYa |\n| `lg` | Ganda |\n| `lij` | Ligurian |\n| `lir` | Liberian English |\n| `lkb` | Kabras |\n| `lla` | Lala-Roba |\n| `ln` | Lingala |\n| `lnu` | Longuda |\n| `lo` | Lao |\n| `loa` | Loloda |\n| `lrk` | Loarki |\n| `lss` | Lasi |\n| `lt` | Lithuanian |\n| `ltg` | Latgalian |\n| `lto` | Tsotso |\n| `lua` | Luba-Lulua |\n| `luo` | Luo |\n| `lus` | Lushai |\n| `lv` | Latvian |\n| `lwg` | Wanga |\n| `mab` | Yutanduchi Mixtec |\n| `maf` | Mafa |\n| `mai` | Maithili |\n| `mau` | Huautla Mazatec |\n| `max` | North Moluccan Malay |\n| `mbo` | Mbo (Cameroon) |\n| `mcf` | Matsés |\n| `mcn` | Masana |\n| `mcx` | Mpiemo |\n| `mdd` | Mbum |\n| `mde` | Maba (Chad) |\n| `mdf` | Moksha |\n| `mek` | Mekeo |\n| `mer` | Meru |\n| `meu` | Motu |\n| `mfm` | Marghi South |\n| `mfn` | Cross River Mbembe |\n| `mfo` | Mbe |\n| `mfv` | Mandjak |\n| `mgg` | Mpumpong |\n| `mgi` | Lijili |\n| `mhk` | Mungaka |\n| `mhr` | Eastern Mari |\n| `mi` | Maori |\n| `mig` | San Miguel El Grande Mixtec |\n| `miu` | Cacaloxtepec Mixtec |\n| `mk` | Macedonian |\n| `mkf` | Miya |\n| `mki` | Dhatki |\n| `ml` | Malayalam |\n| `mlq` | Western Maninkakan |\n| `mn` | Mongolian |\n| `mne` | Naba |\n| `mni` | Manipuri |\n| `mqy` | Manggarai |\n| `mr` | Marathi |\n| `mrj` | Western Mari |\n| `mrr` | Maria (India) |\n| `mrt` | Marghi Central |\n| `ms` | Malay |\n| `mse` | Musey |\n| `msh` | Masikoro Malagasy |\n| `msw` | Mansoanka |\n| `mt` | Maltese |\n| `mtr` | Mewari |\n| `mtu` | Tututepec Mixtec |\n| `mtx` | Tidaá Mixtec |\n| `mua` | Mundang |\n| `mug` | Musgu |\n| `mui` | Musi |\n| `mve` | Marwari (Pakistan) |\n| `mvy` | Indus Kohistani |\n| `mxs` | Huitepec Mixtec |\n| `mxu` | Mada (Cameroon) |\n| `mxy` | Southeastern Nochixtlán Mixtec |\n| `my` | Burmese |\n| `myv` | Erzya |\n| `mzl` | Mazatlán Mixe |\n| `nal` | Nalik |\n| `nan` | Min Nan Chinese |\n| `nap` | Neapolitan |\n| `nb` | Norwegian Bokmål |\n| `nbh` | Ngamo |\n| `ncf` | Notsi |\n| `nco` | Sibe |\n| `ncx` | Central Puebla Nahuatl |\n| `ndi` | Samba Leko |\n| `ng` | Ndonga |\n| `ngi` | Ngizim |\n| `nhg` | Tetelcingo Nahuatl |\n| `nhi` | Zacatlán-Ahuacatlán-Tepetzintla Nahuatl |\n| `nhn` | Central Nahuatl |\n| `nhq` | Huaxcaleca Nahuatl |\n| `nja` | Nzanyi |\n| `nl` | Dutch |\n| `nla` | Ngombale |\n| `nlv` | Orizaba Nahuatl |\n| `nmg` | Kwasio |\n| `nmz` | Nawdm |\n| `nn` | Norwegian Nynorsk |\n| `nnh` | Ngiemboon |\n| `no` | Norwegian |\n| `noe` | Nimadi |\n| `npi` | Nepali |\n| `nso` | Pedi |\n| `ny` | Chichewa |\n| `nyu` | Nyungwe |\n| `oc` | Occitan |\n| `odk` | Od |\n| `odu` | Odual |\n| `ogo` | Khana |\n| `om` | Oromo |\n| `orc` | Orma |\n| `oru` | Ormuri |\n| `ory` | Odia |\n| `os` | Iron Ossetic |\n| `pa` | Panjabi |\n| `pbs` | Central Pame |\n| `pbt` | Southern Pashto |\n| `pbu` | Northern Pashto |\n| `pcm` | Nigerian Pidgin |\n| `pex` | Petats |\n| `phl` | Phalura |\n| `phr` | Pahari-Potwari |\n| `pip` | Pero |\n| `piy` | Piya-Kwonci |\n| `pko` | Pökoot |\n| `pl` | Polish |\n| `plk` | Kohistani Shina |\n| `plt` | Plateau Malagasy |\n| `pmq` | Northern Pame |\n| `pms` | Piemontese |\n| `pmy` | Papuan Malay |\n| `pnb` | Western Panjabi |\n| `poc` | Poqomam |\n| `poe` | San Juan Atzingo Popoloca |\n| `pow` | San Felipe Otlaltepec Popoloca |\n| `prq` | Ashéninka Perené |\n| `ps` | Pushto |\n| `pst` | Central Pashto |\n| `pt` | Portuguese |\n| `pua` | Western Highland Purepecha |\n| `pwn` | Paiwan |\n| `qug` | Chimborazo Highland Quichua |\n| `qum` | Sipacapense |\n| `qup` | Southern Pastaza Quechua |\n| `qur` | Yanahuanca Pasco Quechua |\n| `qus` | Santiago Del Estero Quichua |\n| `quv` | Sacapulteco |\n| `qux` | Yauyos Quechua |\n| `quy` | Ayacucho Quechua |\n| `qva` | Ambo-Pasco Quechua |\n| `qvi` | Imbabura Highland Quichua |\n| `qvj` | Loja Highland Quichua |\n| `qvl` | Cajatambo North Lima Quechua |\n| `qwa` | Corongo Ancash Quechua |\n| `qws` | Sihuas Ancash Quechua |\n| `qxa` | Chiquián Ancash Quechua |\n| `qxp` | Puno Quechua |\n| `qxt` | Santa Ana De Tusi Pasco Quechua |\n| `qxu` | Arequipa-La Unión Quechua |\n| `qxw` | Jauja Wanca Quechua |\n| `rag` | Logooli |\n| `rm` | Romansh |\n| `ro` | Romanian |\n| `rob` | Tae\' |\n| `rof` | Rombo |\n| `roo` | Rotokas |\n| `rth` | Ratahan |\n| `ru` | Russian |\n| `rup` | Macedo-Romanian |\n| `rw` | Kinyarwanda |\n| `sa` | Sanskrit |\n| `sah` | Yakut |\n| `sat` | Santali |\n| `sau` | Saleman |\n| `say` | Saya |\n| `sbn` | Sindhi Bhil |\n| `sc` | Sardinian |\n| `scl` | Shina |\n| `scn` | Sicilian |\n| `sd` | Sindhi |\n| `sei` | Seri |\n| `shu` | Chadian Arabic |\n| `si` | Sinhala |\n| `sip` | Sikkimese |\n| `siw` | Siwai |\n| `sjr` | Siar-Lak |\n| `sk` | Slovak |\n| `skg` | Sakalava Malagasy |\n| `skr` | Saraiki |\n| `sl` | Slovenian |\n| `sn` | Shona |\n| `snc` | Sinaugoro |\n| `snk` | Soninke |\n| `so` | Somali |\n| `sol` | Solos |\n| `sps` | Saposa |\n| `sq` | Albanian |\n| `sr` | Serbian |\n| `src` | Logudorese Sardinian |\n| `sro` | Campidanese Sardinian |\n| `ssi` | Sansi |\n| `ste` | Liana-Seti |\n| `sua` | Sulka |\n| `sv` | Swedish |\n| `sva` | Svan |\n| `sw` | Swahili |\n| `szy` | Sakizaya |\n| `ta` | Tamil |\n| `tan` | Tangale |\n| `tar` | Central Tarahumara |\n| `tay` | Atayal |\n| `tbf` | Mandara |\n| `tcf` | Malinaltepec Me\'Phaa |\n| `tcy` | Tulu |\n| `tdn` | Tondano |\n| `tdx` | Tandroy-Mahafaly Malagasy |\n| `te` | Telugu |\n| `tg` | Tajik |\n| `tgc` | Tigak |\n| `th` | Thai |\n| `the` | Chitwania Tharu |\n| `thq` | Kochila Tharu |\n| `thr` | Rana Tharu |\n| `thv` | Tahaggart Tamahaq |\n| `ti` | Tigrinya |\n| `tig` | Tigre |\n| `tio` | Teop |\n| `tk` | Turkmen |\n| `tkg` | Tesaka Malagasy |\n| `tkt` | Kathoriya Tharu |\n| `tli` | Tlingit |\n| `tlp` | Filomena Mata-Coahuitlán Totonac |\n| `tn` | Tswana |\n| `tok` | Toki Pona |\n| `tpl` | Tlacoapa Me\'Phaa |\n| `tpz` | Tinputz |\n| `tqp` | Tomoip |\n| `tr` | Turkish |\n| `trp` | Kok Borok |\n| `trq` | San Martín Itunyoso Triqui |\n| `trv` | Sediq |\n| `trw` | Torwali |\n| `tt` | Tatar |\n| `ttj` | Tooro |\n| `ttr` | Tera |\n| `ttu` | Torau |\n| `tui` | Tupuri |\n| `tul` | Tula |\n| `tuq` | Tedaga |\n| `tuv` | Turkana |\n| `tuy` | Tugen |\n| `tvo` | Tidore |\n| `tvu` | Tunen |\n| `tw` | Twi |\n| `twu` | Termanu |\n| `txs` | Tonsea |\n| `txy` | Tanosy Malagasy |\n| `udl` | Wuzlam |\n| `ug` | Uighur |\n| `uk` | Ukrainian |\n| `uki` | Kui (India) |\n| `umb` | Umbundu |\n| `ur` | Urdu |\n| `ush` | Ushojo |\n| `uz` | Uzbek |\n| `uzn` | Northern Uzbek |\n| `vai` | Vai |\n| `var` | Huarijio |\n| `ver` | Mom Jango |\n| `vi` | Vietnamese |\n| `vmc` | Juxtlahuaca Mixtec |\n| `vmj` | Ixtayutla Mixtec |\n| `vmm` | Mitlatongo Mixtec |\n| `vmp` | Soyaltepec Mazatec |\n| `vmz` | Mazatlán Mazatec |\n| `vot` | Votic |\n| `vro` | Võro |\n| `wbl` | Wakhi |\n| `wci` | Waci Gbe |\n| `weo` | Wemale |\n| `wes` | Cameroon Pidgin |\n| `wja` | Waja |\n| `wji` | Warji |\n| `wo` | Wolof |\n| `wof` | Gambian Wolof |\n| `xh` | Xhosa |\n| `xhe` | Khetrani |\n| `xka` | Kalkoti |\n| `xmf` | Mingrelian |\n| `xmv` | Antankarana Malagasy |\n| `xmw` | Tsimihety Malagasy |\n| `xpe` | Liberia Kpelle |\n| `xti` | Sinicahua Mixtec |\n| `xtu` | Cuyamecalco Mixtec |\n| `yaq` | Yaqui |\n| `yav` | Yangben |\n| `yay` | Agwagwune |\n| `ydd` | Eastern Yiddish |\n| `ydg` | Yidgha |\n| `yer` | Tarok |\n| `yes` | Nyankpa |\n| `yi` | Yiddish |\n| `yo` | Yoruba |\n| `yue` | Cantonese |\n| `zga` | Kinga |\n| `zgh` | Standard Moroccan Tamazight |\n| `zh` | Chinese |\n| `zoc` | Copainalá Zoque |\n| `zoh` | Chimalapa Zoque |\n| `zor` | Rayón Zoque |\n| `zpv` | Chichicapan Zapotec |\n| `zpy` | Mazaltepec Zapotec |\n| `ztg` | Xanaguía Zapotec |\n| `ztn` | Santa Catarina Albarradas Zapotec |\n| `ztp` | Loxicha Zapotec |\n| `zts` | Tilquiapan Zapotec |\n| `ztu` | Güilá Zapotec |\n| `zu` | Zulu |\n| `zza` | Zaza |\n\n</details>'
CATALOG_DIR = PACKAGE_DIR / "catalog"
CATALOG_DIR.mkdir(parents=True, exist_ok=True)
for filename, payload in (("voices.json", VOICE_CATALOG), ("adapter_options.json", ADAPTER_OPTIONS),
                          ("notebook_default_profiles.json", MODEL_PROFILES), ("duration_algorithm_locations.json", ALGORITHM_LOCATIONS)):
    (CATALOG_DIR / filename).write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
(CATALOG_DIR / "REFERENCE_GUIDE.md").write_text(REFERENCE_GUIDE, encoding="utf-8")

def write_catalog_csv(path, rows):
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = catalog_csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value
                          for key, value in row.items()} for row in rows)

for model, catalog in VOICE_CATALOG.items():
    if isinstance(catalog, list):
        write_catalog_csv(CATALOG_DIR / f"{model}_all_voices.csv", catalog)
for model, rows in ADAPTER_OPTIONS.items():
    write_catalog_csv(CATALOG_DIR / f"{model}_all_options.csv", rows)

# Edit these filters and rerun this cell to explore every embedded speaker row.
CATALOG_MODEL = "kokoro"       # kokoro / emotivoice / supertonic
CATALOG_QUERY = ""             # Case-insensitive substring across fields; e.g. a speaker label/name.
CATALOG_GENDER = "all"         # all / male / female (exact normalized gender match).
CATALOG_LIMIT = 100            # None displays every matched row, including all 2,014 EmotiVoice speakers.
OPTION_MODEL = "omnivoice"    # Any of the six model keys.

if CATALOG_MODEL not in VOICE_CATALOG or not isinstance(VOICE_CATALOG[CATALOG_MODEL], list):
    raise ValueError("CATALOG_MODEL must select a finite preset catalog: kokoro, emotivoice, or supertonic")
if CATALOG_GENDER not in {"all", "male", "female"}:
    raise ValueError("CATALOG_GENDER must be all, male, or female")
if CATALOG_LIMIT is not None and (type(CATALOG_LIMIT) is not int or CATALOG_LIMIT < 1):
    raise ValueError("CATALOG_LIMIT must be a positive integer or None")
if OPTION_MODEL not in ADAPTER_OPTIONS:
    raise ValueError("OPTION_MODEL must select one of the six models")

def normalized_gender(row):
    gender = str(row.get("gender", "")).casefold().strip()
    return {"m": "male", "f": "female", "man": "male", "woman": "female"}.get(gender, gender)

catalog_rows = [row for row in VOICE_CATALOG[CATALOG_MODEL]
                if (CATALOG_GENDER == "all" or normalized_gender(row) == CATALOG_GENDER)
                and CATALOG_QUERY.casefold() in " ".join(str(value) for value in row.values()).casefold()]
shown_rows = catalog_rows if CATALOG_LIMIT is None else catalog_rows[:CATALOG_LIMIT]

def catalog_html(rows):
    columns = list(dict.fromkeys(key for row in rows for key in row))
    headers = "".join(f"<th>{catalog_markup.escape(key)}</th>" for key in columns)
    body = "".join("<tr>" + "".join(f"<td>{catalog_markup.escape(str(row.get(key, '')))}</td>" for key in columns) + "</tr>" for row in rows)
    return '<div style="max-height:520px;overflow:auto"><table border="1" cellpadding="5"><thead><tr>' + headers + '</tr></thead><tbody>' + body + '</tbody></table></div>'

print("Complete preset catalogs:", {key: len(value) for key, value in VOICE_CATALOG.items() if isinstance(value, list)})
print(f"Showing {len(shown_rows)} / {len(catalog_rows)} matched {CATALOG_MODEL} voices; all rows are saved to CSV.")
print("Option counts:", {key: len(value) for key, value in ADAPTER_OPTIONS.items()})
try:
    from IPython.display import HTML, FileLink, display
    display(HTML(catalog_html(shown_rows)))
    print(f"Actual {OPTION_MODEL} parser defaults (not profile overrides):")
    display(HTML(catalog_html(ADAPTER_OPTIONS[OPTION_MODEL])))
    print("Pre-synthesis planning and generation functions in the embedded Python files:")
    display(HTML(catalog_html(ALGORITHM_LOCATIONS)))
    for filename in (f"{CATALOG_MODEL}_all_voices.csv", f"{OPTION_MODEL}_all_options.csv", "voices.json", "REFERENCE_GUIDE.md"):
        display(FileLink(str(CATALOG_DIR / filename)))
except ImportError:
    print(json.dumps(shown_rows, ensure_ascii=False, indent=2))
    print(json.dumps(ADAPTER_OPTIONS[OPTION_MODEL], ensure_ascii=False, indent=2))

# Catch misspelled/unsupported profile keys before downloading model weights.
# Numeric bounds and reference/checkpoint compatibility remain adapter checks.
for model, profiles in MODEL_PROFILES.items():
    if model not in ADAPTER_OPTIONS:
        raise ValueError(f"Unknown profile model {model}")
    options = {row["key"]: row for row in ADAPTER_OPTIONS[model]}
    for profile in profiles:
        for section in ("parameters", "clone_parameters"):
            for key, value in profile.get(section, {}).items():
                if key not in options:
                    raise ValueError(f"Unknown {model}/{profile['name']} {section} key: {key}")
                if options[key]["supervisor_controlled"]:
                    raise ValueError(f"Set {key} through its supervisor/profile control, not {section}")
                if value is not None and options[key]["choices"] is not None and value not in options[key]["choices"]:
                    raise ValueError(f"Unsupported {model}/{profile['name']} value for {key}: {value!r}")
print("Profile key/choice audit passed. Catalog files and the guide will enter the final WAV/report ZIP.")


## 2. Notebook helpers and hardware verification


In [ ]:
import csv
import hashlib
import html
import importlib
import json
import os
import platform
import signal
import subprocess
import sys
import threading
import zipfile
from collections import deque


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    temp.replace(path)

# Lightweight notebook/report packages only; model packages stay in their own venvs.
kernel_packages = {
    "pynvml": "nvidia-ml-py>=12,<14", "psutil": "psutil>=5.9", "tqdm": "tqdm>=4.66",
    "soundfile": "soundfile>=0.12", "pandas": "pandas>=2", "matplotlib": "matplotlib>=3.7", "uv": "uv>=0.6",
}
KERNEL_READY = True
for module, package in kernel_packages.items():
    try:
        importlib.import_module(module)
    except ImportError:
        try:
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = ""
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], env=env, check=True)
            importlib.invalidate_caches()
            importlib.import_module(module)
        except Exception as exc:
            KERNEL_READY = False
            atomic_json(RUN_DIR / "kernel_failure.json", {"module": module, "error": str(exc)})
            print("Kernel dependency failed:", module, exc)
            break

if KERNEL_READY:
    import pandas as pd
    import psutil
    import pynvml
    import soundfile as sf
    from IPython.display import HTML, Audio, FileLink, display
    from tqdm.auto import tqdm
print("Kernel helpers ready:", KERNEL_READY)


In [ ]:
GPU_INVENTORY = []
GPU_HANDLES = {}
HARDWARE_READY = False
hardware_error = None
try:
    if not KERNEL_READY:
        raise RuntimeError("Notebook dependencies are unavailable; enable Internet and rerun.")
    if not (0 < CONFIG["max_vram_gb"] <= 5):
        raise ValueError("max_vram_gb must be in (0, 5] decimal GB")
    if not 1 <= CONFIG["nvml_interval_ms"] <= 1000:
        raise ValueError("nvml_interval_ms must be between 1 and 1000")
    if CONFIG["budget_scope"] not in {"device_total", "process_tree"}:
        raise ValueError("budget_scope must be device_total or process_tree")
    pynvml.nvmlInit()
    visible = os.environ.get("CUDA_VISIBLE_DEVICES")
    physical = list(range(pynvml.nvmlDeviceGetCount()))
    if visible is not None:
        tokens = [token.strip() for token in visible.split(",") if token.strip()]
        handles = [pynvml.nvmlDeviceGetHandleByUUID(token) if token.startswith("GPU-") else pynvml.nvmlDeviceGetHandleByIndex(int(token)) for token in tokens]
    else:
        handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in physical]
    driver = pynvml.nvmlSystemGetDriverVersion()
    driver = driver.decode() if isinstance(driver, bytes) else driver
    for slot, handle in enumerate(handles):
        name = pynvml.nvmlDeviceGetName(handle)
        name = name.decode() if isinstance(name, bytes) else name
        gpu_uuid = pynvml.nvmlDeviceGetUUID(handle)
        gpu_uuid = gpu_uuid.decode() if isinstance(gpu_uuid, bytes) else gpu_uuid
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        GPU_INVENTORY.append({"slot": slot, "uuid": gpu_uuid, "name": name,
                              "total_bytes": info.total, "baseline_used_bytes": info.used, "driver": driver})
        GPU_HANDLES[slot] = handle
    if len(GPU_INVENTORY) != 2:
        raise RuntimeError(f"Expected two GPUs; found {len(GPU_INVENTORY)}. Select Kaggle GPU T4 x2.")
    if CONFIG["require_two_t4"] and not all("T4" in g["name"] for g in GPU_INVENTORY):
        raise RuntimeError("Select GPU T4 x2, or explicitly disable require_two_t4 to test other hardware.")
    HARDWARE_READY = True
except Exception as exc:
    hardware_error = f"{type(exc).__name__}: {exc}"
atomic_json(RUN_DIR / "hardware.json", {"ready": HARDWARE_READY, "error": hardware_error,
    "gpus": GPU_INVENTORY, "kernel_python": sys.version, "platform": platform.platform(),
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "peak_measurement": "NVML sampled total/device and process tree; allocator peaks separately. No instantaneous hard-peak certificate."})
print(json.dumps({"hardware_ready": HARDWARE_READY, "error": hardware_error, "gpus": GPU_INVENTORY}, indent=2))


## 3. Extract the embedded standalone adapters

The compressed payload contains the six complete Python adapters and the CPU
quality checker. They are
verified by SHA-256, written locally for inspection, and included in the ZIP.
No external source archive upload is required.


In [ ]:
# Six complete standalone adapters and the CPU quality checker. No Python file uploads are needed.
import base64
import gzip

EMBEDDED_SOURCES = {
  "kokoro_samples.py": {
    "sha256": "734f492085135cbc9519d319da4ebddf5a39b124e9a207dd01366819ec6c2e7d",
    "gzip_base64": "H4sIAAAAAAAC/9U9a3PbOJLfU5X/gOHe1FAeipZk56WMZjeTx25uJo+Kk9m6c1QsSoQkjiWSS1K2Fa/vt18/ABB8yHZSu1V3qYotkUCj0Wj0C93wn7473Bb54SxODmVyLrJduUqTo/v3HMd5nUQyk/AjKcWv6Vmap6IIN9k6TpYiTCKR5bJ/EZ7LRZpvRLTNwzJOE5GtwySBJv79e/fvvSdw4sgfjkRciFzO080GQUa+eJ0UZbhei/e7j2k+XwkAI3bpNkcIJcL0RLmSyfj+PaHQEv2NyOJMxKqns4zLH1dlmRXjw0P4vNrOfBjgcCUvl3kYHZ4R0j68+Uu0mD0ZPAoHo9ksfDx8MJqHx8fHj4aP5uGD6NHjR8eP5vPZ8UzOHeFs4iI8i09lMnVEmYdJgbjIvJhMHvhH/kAU8zjbiSLdJtEiXkuc5stLpIu0UeWxAyKYLHzo0e+X8rLsYx+RhDmTyy8vS3hznsZzKcJNsJBJHufwpMikjMQQxuv3gYDfDFkWZbwJS9lPk/UOcf24krhy57G8EPk2KcRfR+9pOYHYIpyX23AtCJ3+PE2iGEEBImZ5E1lepPkZLF8polQW9+8laYmAxKvBYZLGBUGP4jm29hTQdAtYzAViKWQyTyOZewKWG99WHCTpBfDN83C+ssYsCD25mckoAsaCr7kEVtoW0AaZptglAKiIC1+8UkCLbZaleSmj+/cA1PwsS2Ng4hDYLC4lzDGXjJoaE4DHZSHkJcx/vRMPBwNxADA2Lk4lADx6QpH7/r2wFKNjcfa3Lx7w4Xy9RZTEL+9ODl++OxEFLEEyl754m8JEZb7c9Wch4qleANqb9BwoDJsB8fd5PSK5CLfrEp+GYhOuiYaFLH2978L5XGaAIT8+XKdzvUoWXTwBSwEIwiALmdNwf3/2+1OcKBAMYCdp9aofAuawoTMZniEBeFVgStAcmlZUw/VYbZMzMUOOD/NYEhoARsBeA5qUQG4QEQiUV2q+DmFusBof42QHjZJFvISlnMG0ZLSEDbMII2iKawTTKcQfMAwScb6O52eFuICNnG5htWhWHpOMPsHi8ke9GiRjnr//hJMrLToiFoiC+CKBeH+FBr9/ePbGF88/vXiGbdOs7MfJGNoJkCNAzBJAz7aAHNIPOm9kWACTREY4IZk04zFnwfSIUCDtYAPMZV7Gix0s3yrMUTSmQI2ifxFHMF8cXKxj4DFPgIxB6ogoj89lfgjUwV0BOyfp67EUSsz5MCLBh3Z5ul4TywBzxYle4nU8y8N8h8QJoSWsNUioPM5Ks5ZRDHIXZrjDqccbhT6JeKTfIk83IggWW9wWQaBaCJoXY4Gt9NN8mYV5Ic2DCGQLSBjY5TBQaR6vwmIFmJnv/Aue+BtZhtApNK/+KNLEfAFRtTJfQPhG6ab6Wo1a7ArzGUc3X7bbOFJTygAUDKjn854gA/GefXz9+8vgw7OPL8UEtnIwGAzu3zt59ub9by9PgvcvPwQvPsHL1+/ewuuH9O7dpw/PocfL31+f8GPnzurk/r037168/M10PoHeVyjMhXC0muI93n88euOMhbM4WiyOHgCsJ0N59OThcBgdDedHj+FLeDwMj8LF4NHs+MnjR463F0z/fOgP+19WCG4wlI8eDB7Moofho3AUzo+fPHpw/PBoODyeH4WPHj4YhE8Wj0bDxwjuWmP76jXQ4lZMWRHBYMHAz8rVHRGqeuET3fEal+b+PdjAgvgrd3ui/7NhN/9ZvtyC4VC+p5djHikDFPe0cEHC0C4A/p0EQZTOg8BDbQEMVso8ABlVFBPT+UN48aLq8De5zl7ppj0ei3TXRGQ+iKVgs0U1ud4F8hI0QAE7OVjm6TZzrcbUMFRIuQ7raMcTKwA+cX4DPZSDDP/08VX/Mbd3butMCh4glLtMTpCdNTAGAgKAPvR/efeG0cX2BmzWgsmSuQvgf54Am6ezP0BsiHQhnv/2WovW4iloSTAB5zGAX4egjEvQH6gQ5jIi1bd/vEQPBVrF0wAnQz3o2y1oshzHQ9sH5KAS8h5pBJSAhZxvSyA26BsZFTeMhO87Bzsedffhh8S9yhhzqk6OscvM8r0nPewJVsTAxEoZZ+H8jPQU2rkh4AHsBQIyUoobcOaR7opF/0LGy1VZmIGfN+CC3kjkMiSqbOJL1I2qy1OjDeU/0KZTj78CAzDkl9twiaSYrxCbYrKOCyBwOJOLVfxH9sXp2VQySCqjRfcXaFo8FeHk04l4mSwBxupmLBxl/uolXKzT0OYYf7C3H2rnfhw1UW4IYRvtDtm2f8cYq6hr1/xGzKC1uGV3or8iwwhZG62ULE7IpE4vEphYtI+Pa2uxARKub9ixPDa1UhbX05q5KzZggqN2BcxqBrIxu29bkTka5f2INoE1vKYjfnPBigCREwQ9H/gT3cZD4fiEVEDd91M2Aoekvunm2dYwFHymTbWNwtNxDO7j5fRuVAsv++d5uOkvZ3t46YE/0IO8kHPwlNbir794Aoz8TQoEe/CUbcbKSixhJDDM776LEOk+7v78XPY3s06x9OjhY43FB24pwAIuwHikxWoYhnfEArXGKge+K7pF4V1wR8O/D/7eHhhHDzTarxPwFIF4pHeoGxu8T0FEommPT9iaLUB7gB25DOPkVvxx/RgHEP3gjRZ7VnE02CcSatNhhdJHyamNFVK4el7VIw3YMhattzzll5dsihJQgUCfgvfN+kl7pKIow12hHEZfD3oHVDN0EvqbfRMeVGz7Uqtj7WLOwEOXMgHtn8CCUDgGtjmvwK0Ux2XpR7P9w94Bd3Ty+ujjEf4NOjPENqWPYEZNCj9PiziRAkHhjpQhiVXkLRziKStV2CyFXoQ5uKjAcgNweQr0N8HdKb+G6gkafev4i+yjv2fjHlI8ozIYf0nTtQyTd2QvhutnHO5oT+tjvpWteX2Q0RaNBfYpZylwzMB/8th4vvMVaE5U5bCgy3J1ywyQsbczJK6l9U6d98/fBMOH8Ig+jY7x06vf3j376EwtKata7YUM6GTb8hax7yhLzendhTvsgNS/gsSvwnXRpvHfc1B94I7rUJem7RLDMrBLKYa5M9Gnr+GSQq4XYJAX5b8P+w/bBKyGUi5l3mcmYHuiio6QVi04iBbnEQiMvNwpX7doziaXYAMkIms4WTi/woUf52OBZtJpUeZT8U/xNoVdN6FfdS/sbbiRBdi40nLAtLumRgov4BnZXOCj+wj7dDieiniBUM4xAkHQAXdJTzS9c4n6IgnznScCcrQYxbMEzCRGFGCr1gDN6uCzwTNuEbcAOBhd8NHMKtx2Fx/VY4A6y6UAGHDFxNmWC3CiCrCzNDurITEOExcUfAY56+pBYDPE87JnDc8MJPM8zSs/iy0wjOOAiAXGtHwsp9fkI8T8KvTBiS3H0BajnCHGczI/0A2QotSA8UrElYO8g9tcmYnX1xVYEJpISAALbohBHVaXviuY9emqLnvmtXA+KZAmwEfx4TO5K8biqiDj0lUwetf2FHE20MwT5+F6KxF1jY8Pe3ZTuE1aMnro7DOep9B7Wm8SF8GMdxw0sxYpVHFo1/BwcAIGlES5rPakuHnP2kygKGMNhsHCBl/QrDyBLZoTqRPwCqZxzWwxk5ojFGCnPSpPxUchTIG8hOOc7p6hUTR0YwY9e9+AmZJHt2NGKv4bcXPJviQIvW9BMqEgwu04ot4iFBUPFoQPiab9Swri8RtQwjgkdgXhcjteDQxqfPbvR3Uvlt1DW7h3jFXmu46n+I+hTez+jHev3V5e4smHcD9Co5eIuyd+x6b0uYdRZ2ixZ5xqtq8TgB9HgmYN4gn6XO9fDGVEtebLeCtxW297I62boxOc7/IWBloMknwD+lC7qknmg6wOdCP34MCI8ZrcBrPGxdCcmEyEDjvihsNnPnAJ2AdoROio4sTpkUSmLjAv1LItmV8bGHuRoz8hC+G2wTlseQMG1ODb0LAxQBvBMhzaJgM+8dF4C9B4s4Ar8wjfm8YutSZ3Vm3LHs6pekwUMO/G9jIpzX+yzbL1zpwqIhelC3VYS/xk06eBZiJ+EkM606Hx2I+vPSNHKCDfHJ93IQAOjSdMFEANafn0Zvtn4Gih4+q0iAWe40/gT+lB6cGPGsU+oPPzRIwODh4edY3/3A7ZsodGkVsed4FhgFoAE2wudFUfHvdncdnUN8gcCdifyBzgcaj4oLMJLwMM8QQU4qGvTJkqYuCQMx2QM+2ggxuQg+ugExmgc4mv7BVUth6eRflxAX40mCXuUpZhWebEAB5h0uvttY2u8H0lXhlEi7p0xP/TxKJvC33zGhEawFqYZmrS2OBBF/HfqzWlaHp0yH6zOcZHXkBwVoSMIKkjzn9s4xzo20BY01GzBOL0GM/PsCtPiVY5wGAIPhs+GdFrw7PbKAyU0x5sZginC3UtL2k8z46wUGydInIKio3irODdqZa4J34Ww9Gge2OoOIdZIR006UMPIg78tmAr6qtZ2oyDD/cNYgdCWiPxMK1Rcukvtus1RWrdHCOf/0SquX8ejz9HP/b+7HiMAwdMe90D80szZEf41Kl7hiz6tHMoKbIVsOPpckxL55uMBVlouBj0kRxE+K0QcRznDQWZKUvGxx0drsHNUoyRpevdDyrRQsxlvKZz/W2B7rja74AL6IaNLOO5TyfUFpoNXMQB4QECyT7UBanUE4eH9rNqaiqJYReAqcPTQ0k4JnupNZP32wTzYTityWROwAKcpzGlfeApD7xThxrAgTOMNvCh+WHMIdGimgWaV1kGe34icFQ/pweu89n5vB0NhhH9fPJDb3ptMQUwRCExlA/ccPrdn7HN6OHn7dFgMPq8XSwGQ/w5XEx//A/HMyP02tptZCDqRr4EAcOK2K/JP3se6DHq9uv0gg5m0dmsSz5nk/t2NEQ9KzoedjyLOjpnebpoPy06Wv7R8awo28/OO0aW5bzjob9sP4x92X4Y+psOzBsPLS9csQTQFO16t7HXYY+fPuv/d9j/Mv3s9360l7MdiKitkfEDGH5DLdUZQH0btnjLezomBjymn0fEXHNirhn9DBssxjGcQbW31OiswNg+o43l8VFAJS5MrKnaar9KmdFZB6XP4O7grSouVjGnQ8Huy9WeayQdHeqEI7PRYGo0ZMMwykNMT6vcB7BQSCvScNxhn0WkDjAm4nRamSN4CLvMw2xFqTZ4bE+HGyDUZD2IwdbWpOrADV1rWXme1LCxfLBLoesmTlyVQgRymaRW0RGawMY/WS06/JI5cEuMaTs0nW7PiZQEyNUhzZO+eUwmmKkEJxtjqJLHOB3DoNMejt4tXclwHrUHmrYfKeauUNzj2d1lDv+eeQy7B+uey63z4LWtmp32hw1I6twmzDAT13WE42O6XA3jBhdoZuMm0GI8re18hmhpe23NuJQSqO2twKh4D5hvDXuWLdKxis5Ue/dEJ/LR3mVDE5HkYDXY+MUKzAkNt0+HNH11doM7ugwzPH82m5fztWBxsh369wmFrXkCBeYzTOCRHxaY6LrTOEd0NgHPCbmjUc+fp9lObzAaUm2iHNfWtQ3Vg9r8xKEYgs3a433GQ5I9Map0MsP7uSFcNhgOdwf+AzBB8OcB4jlPCxd+gUygwLk78PBpFnsMpNfrwVQoAmIhbytvHP90TI2n4mBC47TeM1HHpsHpeGw4SS07t6yWnUxsmH4g2XoJyvRMJiC26RcfCHjoH8Sb7SbIVuC+bpT5R0Ic3d6KBz5IShYqztjAYR5EtQR7Lj6PixizPxUQkZ7LHKZ6AYIWtqVkltEp4wjwDaVg477Q2MtNVu40AGCsFVjQSSrevXshFuF6PQvnZ5Q9ex7Gazz505mzPyiPPtvO1vEcs8climdxkeOOyqHHukhFmYOlh6yAmNGBtUaVI2KYy4rppZRIyQBlwgSLv0iO8FE68kzitDAvmuSYhuuJGSa0JhV8Jku0Ddcq6QthVQZ7uF2uSk0eSk2m+Wu5pDYZ0mcmKXmZMzrVYlQaisGiduJVtQ5jOZ+NnnOYA5rrrkQxTWvdyDxoNFRRnznsfrDqYSfD4q1h97oc0DHPtZLsGXNFQ/TZDm4qq5a6bsvQhYPZ62Ar4oEqAsXPCRBdGUaIwBUO+l1+/VTXDRzqEwsg3QkewlYsRJlbOa4yGSOcl1cftm6KoYxoTgMdz9a++YapvUvYIPihaMwKY6IYRqFkIsWoKpF4bE0XbCb2rEqLwzm3rmNS5mQwT5cozd0INzjL/zIFuqnPsL3kunKYLDGwyM1BDfYFQQpUcIequ6LbRRzBZsHEW/4+C3NMqP2TA+IS4Lv8/sAAU93wMLEoI7AB/Qs82XUXzuf8inC5HovTKwAz/umKOl9fT8UVInB9eEUj294UITaZqBlVRGjBdz6bMxjr3WK9LVauRS1qHOAho4tSYCz4eDwLd3jmOOajwQadSokaDrfxhESHj2oywIgRwfApyvUjaPpyk1VJoaoPo8dnlhX6dMoZgcYsXDW0BxIKU9iDEDzxWB0zkwWUlJNRdYBP6wiDfcbszMYhaD1ZpEIhl9kadRlia9HChFbd5ozDAnR92YgrPByALuT8a7KsKNf6htbASMOHpvXxYHgb7OPjYKhaD4eDo1rzhq/iIFuUF7BPVrmUn5MF1iEt0P4HFAnCqW6CkUNqhR+qdpZbYIJsFMZOlhL3ANkUto/NeHTmnR9UIGphDMTDenM80GfnuHF01qUOo6iomS7H0vkDGQ7MOYd63TYhmEW3ZwFUoREdaa/nENwh0G5xSCs+MdAxkpDSEieU1++Djl4EZNxUOQb1g6Uuc7HxztRq4fti0XpfYnaddT6N1QOr7RJTcBbA58Fqa6oIVgv8FujczUanM7UE3PbXN5ij4Ylf3yuro9HaDpEJU+NgRcqUpcEnYK+pAWmL9qkX65MPQCYgWpdGcXS1HZfJcFnfHGt5wDi/SEytCFIJ9E2UztnywVVPdWkYRpNlch6DMsKsGFuJ9HhKgJW24aIYIYDdwxqhXQDi2y1cVReg5R2bWsE2x2CJ3dBK2XCqRj4KQCPmeaoyCjBCUvD4Vh5I1a3nLyXAOZ8XQQx7Hnb01bV6hqG9uAziyCGP0MKHwh64Kwy/dwz43UQ0yka+ZrUWteVS+cK6BBNEDhWSqZGuGuOA4l9Q5OSqjRaYBbVF07YnsD+drcEmCtRpkWsfPOlF4bDyRPXgr64djlY6nSqpgtmO/XOUTq1jiwMxRFmu/1samiDxgTgeImLU2mkfzjAK+BLszcDY/XuMyBqtHTpAuAgLOugAaYSG+5bO+baJgfQULXphYunzbOvb57XW+HwqSbSokYGbwRIEG7lJSd1bvZamVwAmV4b1Y7LQAHy7n6U0dAKwoS16tDa5vdqImN/TefBiUb9uyDcGwNOnuxJUgQeTODwn/6dVW3cTAYEAgaqaU8gH2gR0m1gd1ibpiQ6qG9hUdxGgla/BwqYoi2qlNGuDHuT3XODHNgyab3Xes9nzuw72tDTalaOy2sdCZbM7y2zLO4AmAi/ATHEwbBLgG4rvBQXo92IRywheY5rSdTfNdsl8BYIY/M821ymCsRqt+uD+U5M0Ldqd1Tru76sbtLvGCzL5DXDPAEOfyObTOzDVm2YNJnEZ+WvsAEmuWbY3cYvL9FI0Txz0woBi0dNoxvFp8sw5ZkJm3aop7u+l52466QfNPmYGgU0i6FDb2Y1OxDV6x1Tcw4WozD3N3FLqB8KTOPKZ2ZzzMCOXvF7rSqVtUbNQlRpylalVpto879CBJ6N9G/U3pySUcKOBgp3ae1AbVi4aIugIVZ4m/G6fajXsMdcG7AkNxDO4TPQHT1BFSgCKfcIiUn81MoHDBAFFd9TJr6pl4YROIB2lP5vBVeKlMkZ6NpS6BWJB3p+CaiDoIvWJgsUGCthEizKRJRstVd6KnQXLlT3UfJtZIc8CjBoQXqdDED8PrYAzOi+dnc5knsDEC5A2uu8I+g5HN3ReyiQgJINVmlFX7vngLl2SYLEouf1IqajeV9m7n5JWiVM9U5Xq4tHpVGUhlWe1CbOMTF8O4pkd1WE6WUVeE52WY560+MMqrq3vAL3SXL81UZ6Dq15P6kzNs5jwL4/7oBvvVkP30IAwtoQ8D9fadzIB0EnllrhYqBcgiXgkXbfniW4EeEQSLgoq1SpSBIPOY/CDDobV8nYIDjVWB2COx+a13WNay4ewQN94kqeSrcIS7Y+ipGwrHkmrA1UAiTH6CtAphdzdS8bzso5koGsmK2Snxr+ttWCn4HToD6ZgW2FQ0MK7Vwuj1PcondQxDGL27q5qr2CYtZ2YdEnptJeUlGMmoWHW+rf74h0Xumm9rf2GzMB9m3BvENP5nUpiNYWsAkSaYOFVmVGHVhaYRyoGhsewZWgOYjkZsmMLYsktwGKi4bUDzINTFIvtzLFuduKnStDTnqX8LsOZVqJEhlKJVru+txcOz+qQU7+onUVRxNIYVNShGhNPWbJAK1Nd+ajoFmCZDJUR3ViMwIXHDP6jTArMjaUT6vmZn0TxBnnryDwpVmEmqS4DHmN0avTg4R0cJ8z5l5cZOMIgVKuCZ0HgONZOkx+LK50tx3j9QC1+8DiYVMt7rblzhi+xW88HC8O9E1pvURZix+adKJSqQjg5jcUwR6o0VJMVmI9IkGEirrrNAukGlLKWjrU32WcuCN2VvIzipRXfUmcjGVMcqU1jnw6mfkW0jCo78PmNu0vtJ2rIu2kFjpaIMaIbY/UvQSxw22ziS9RuGyP7NMbEhHSxDYgp9wI8qZoEYHSA9xCjLyCM1eYyDSy1UrsWoZ4Ve1tZjZGhVR6trgfiZzUNoA+n9OeO06I2rT6qxF06NzNkoODiNjPJlZ1BRqyJ4SCkMmO3CZqFepZcEUdPDsVCha2Cq6j09VUofpJeuPD908fnvfH3//X95vvo4/d/+/7N9yfXwRVeUOLjj2OX+OV0/Hh67dRG8jdn8NPlEuqCSwjBHAfDKEjPanoX02LWuDQBYbwvaMqN/6L2mL45JkA97vYq2xtDxSBupEspKnzaS+k71eGvtsUbfrFVXcR1P/U9e6UPqYi91GfkeX2iiPVc6rHKricbA3TsPJw1jwtBSvBReu+6pQxuLF2683ki+xYg47aWEWmQvVLQW2dsVICLmd5RnmaNI7tit5ml6+Kmw0Tix1Nr4qfZ1JIPNq0aFJq2NUNEx7RoTcBHPOAZ4aEkdVGenDoq0c00fHTUK2mhNMVgehspnff6wBx9WSxDtg8paeBDls1qeA6vFRsMUOamOtmmR4LbjKnCnFuSXnNPT8HzOIAXnhhMpzr1Q2tWzCdlCTVphSd4xm2AZiitGL8O7CYsKt2ON04V4IufSdeArQOjCijbkV1gqYch+Wl9NfrCzgiaSdrlvJD4xR4Er6tJkNsCxGji/g/+6vkYfrVNEb7qK6qBCdT9Xy5+ASmP196B5SXdPhgHfTshpeqnrnlLcxbjGoICj87D4nSMPuLj8dTTtPeIWha8Sy73bIJcF+XGjexxzWVwmtKgSjZpHLnNrrolBkf+AOu456OOC0F+TvqYGWwl398EnvOEzPOeP1+Dn+hu4mQy7BFHgADPJXEMkMmub8yVI4SUNwAIix5VONaWY0+Eisp2xsKI4laWqeIQaKM/tlJWI3xbMTfYlW4rymXwg6YVrp1NjVWFTbubwJpTnGvR/VrXiY81jQ46D0C7gklGQZFZQfKiUkd0bInLP60Ha2vuLApNJ5w1g7WBp5Jj8ERTecT+cpTxGI3cupuTpjwKxz/AqEqnqO2LkdclXJsZfGpBA5N0SvXUBjkr9UgN3ASghTBpFBfrVbKCjTv+jHuu0ilqHEvMoDXWrG0pVF32bSSqj00taPhpM2nNSqbVXhpjh6gZKONWiiuZAl0JlXfQ76zj36ZWJhnejWgSiMiHWRak2imcUl1ixDee2jrdPkZtKvaGcrezhUjFIqN8HY90TFhnei6Ldj5xY1TOBhY/idGeTNQ7Eo+28TOBJg9Y7ZxedkM6Et8HyVckzrAIbCHBPtLJSFEX1Toop9lDWwtx0szfWJJioWQja7aUtNnbN2FiQh+QJC/QiBY1SK8DB5xUnNglmURhkOi4MYz1zNutXTUO7U6NCJw2Ujt+3lMA9n9/2Tuu6cFUuzk4gYWqPqPgVfQH+qzIJHRS8f9o7TvEoQVFxRFwedtqvcq6rVURdJRKeK2yTu3krelSA7vwQCWycxZoLZOdAYLVVasMAwjtybLwtsMiOvNPgUdqMryeJ5wX5oJn7XQ2KtcQ8s0hehS8XNmnbvhT9GAZzBcV+c7dUwHuchxquci3OP0NZ1pHF7HkUdebTQRbhfVqyIPaRlbp49y9Ua/GsZemIKAVRSES8+3Z6K7Vxz0QLqkOeon1bWZuaPTIyBrgpto9r1WkWQGiS3ImthnqkH7Cs047+m+950wUk1wCLRupKHZjHc60WpujOKsZXagYVCcZAWZM53Gkz2obZyxVKMk6dzEJOjVkZYmX/qDteR7mnN5i26fa4KataD1mVcyBP+xrhQftZjVCE2mhcfftYY5aonrb5srYHap1rq8ndGsscGen2njkMDT5prufrqDu6GC8qG6EG3E2Wrvak8aA9R3KA9af1ejHN0ileXAGkoqyK/jC8LUM89pl5bhh8B5M2OJgcMV0Cbk+DFSnfuqwzz7AdlgsAeBGZdFVWwdotolP+dPU62hj+WrQznzrbFutUmzJiK6WtucGbauvU79MyWNo+l/XjSwzlDta6FSv7MEcTvmAERrJMbXVNiif6r5qJCs7WkdRD2nBUdrwMbmnhU/PTmbtzGy+fTH28C6CP+142b1YX8mO38T09miInZJu+4h0w0Laryg73b5yrpGSq69eozOlcVcG7h0jxfp+Q1C2qLjGrRuwOGzHig7DENPWwUEz6MJt60zc0cNT8TJuXkUk7OaeeovBiG4wWH0zl1asBzy5sAwwdTqn9DWXn4ecvm1HBpthQK+aRS0BK15Sjmw9Lug2QCk8fDTf1m6vpwOFkU+/94ccDfxTBUKf/WmM64CbgPBMa9iNrfnsb5MCfHAJazyoRQ8xIbcdIRR/qbpaHInlbvRHIdohvleDtwAF/GCZ1IOFd4nY/stjtbdEac3fsNDzqMU8qXc9wtlND/6jCxqGUkWuBu5V9DIUGQNJprVAI4fWfEp4d9vVSDQE5RI07Ux1hpZk1TkrN77rQWuH/HqhtOl5nK4p4ZCCAOrOUftWTjp7ruq09HktIfAUk0XZSrnhgKShgVriq1XHaptf7Ys7zLnYJkziBd6jNyEx5jb1kXKKBp6+9oYTPMgYB7/ohF0Zfa9klYZOXivbR1yGbMpBGE6vVhhBAXn7jh27q+1V4Z9EoDYu/rDIAOvafCu+F+7o4OBo1Gslx8Kst2BpdwGqpnLzsWKNqYNWhXzDWdUeTuWrEgXb8T0mFoXbbCeowwG3B9ZuOFCBRW2tc0eBcMPH74LVUHOtMK1mjdoqW9wBq2gK0L+We8h80n8aZ8KVxBQ0S0K9bxW2HQXEmCSW7WpHxbXSHKv4GeSEtU8aRKbS3RSPviltaDlXrq9t9tvdm5XoFf61uhZXv2g7pBi3UaN6dbfaPO8mkPkIvvJwIA4ORO0yIDCpRoNasAPziCdcy46UwwgSlpXPCoNdr9dBO3NFMOX4Mpvih5/pCt/xHgIcTPiG30Nq3JLZZkTKEGk5Wx2C2/S4q+zW11Wbv2ilLK9bBXdNONumGFqTib4jprJcF0rjBFe1bQFbYDw4iq5J4sA7+HntwywcuyS5qou0oFvlkfbTAuUJVUlWpZEAzsKwWKhaTtPCE/s5zxPqFmXOdlRf9N8MmTh/f/Z7/Xh6gYocxsBP1RA1hcUJ7FbkCdv66uzrkL8xCoiBZWRRldeErrSyu/TbzIEHmq2HNQZrjIIs1tp1MBqP+TMy6oBumbMH7mLLBs9Vy7BN1nFy1oxKd/EkXtIMEg3/TJVYhPEaWI+vhjIe/LnM40XMmXH18pFWNarFHDWDj7W7ZQhp8X4HfxKknikSqKB3en+SUu7xV9frihcq/76i7809jD/ZfNTVLZdrDgmZY3BaWOhNv7u62PrejNUdI22aBl3gVILUWp7LNUK66o7BO1QjwVcc4FE9/qEKjgviC++mTsU8XGPMzBKpd5TPHBsc1u6ab8MPFyVXI1U43aInOsBddy/qreGUpn9vabvOaIpmcB1O0d9r8ZSFcxJiSc8V0+ha/20dvtaU9pjZeyzra39/D7Mo1ZDXjZvj1P1PQP6AZHQQUKg+CLCqOAh0uL5es8uy4GSHMvzlZVy6VIKs1a2+8tWWFfatr554d6I+UHQIr65lP8S6FBY/2UcgigwvP3x49wEvgMX31w6XiEyqCv/eDUgOdZEr9r1/738BM6dP4jpzAAA="
  },
  "f5_samples.py": {
    "sha256": "9b6b0018c9fb89a83d7d37f356f0399d091f8d83eafffc6ebd1ea429dc86e09a",
    "gzip_base64": "H4sIAAAAAAAC/7197Y7buLLgfz8FR2cXR+6x5Y/+SLcznotMPiZZnHwgyWSw6DEEWaLaOm1LGknujpNt4Oz9vS+wu1jsi8zT3PsA8wpbVSRFUpLdnbPAbcykbYksFotVxfoi+y/fjbZlMVom6YinNyzfVassPe45jvOhCtIoWGcpZy9Ohx8/fmA3E1YGm3zNS3abVCsWpIx/DsJqwAq+LXnEom0RVEmWsnwdpF6v946gsWNvMh7CP1NoF2abDU8jHnnsVVpWwXrNNkEVrpL0ilVZEa5G9G+wjZJswMowyXeDXplt0yhO1nzAqt+jDTwPYl7xtMyKcsBW26sr6B4HIfdX2+WA3WRhVs7nY2/ijQHJiFUrDkP/vk0KDoNXZS+L6VkWx0mYBGuYILzPszKBwXcee0qzTqqSwchFyAlIuOLhNcu2Ffvw9pf3T5/7759/evXh1ds3sx5jV0nFQuq1qqq8nI1G8ASQ8WC+ow+/JjdJNBJU9LCp+Cj7DZ8qAtMQOML0/Hh6Oj09O46W8cVkyc+mp3wcTh9dTM+CMA6C8OTkYnJ+FgIEsWJsuGF5krNE0nTImTeqR5Ft4lNfrp+X79hwiFOum8H3in+uhkhmlgdlGVxxr/pcsd8AAP5g+5gXPA35kJaH1d+92+DGeo+QmPMcmYNVRZCWYZHkledAI2CHXg/ofcVTDtySFeyWJ1erqpyxp0+HP/3X4ZunwxNYN7denLzgACNJuVyDPEvSqu/13gCv3cDCBhWsVcmmJ+z65ZfHMISY5JBeFFyxLC44/5xnRQWAfn3yyeu9V/iOymoHsw6ztCqydcmCgrNym4u2Hvu4gmY4RJoBgSOeIwOnFeObjLg9K3q3WREN+SZfBSW0e/LuFTSEEeFzEAV5xQuPfcg5DAxPaimBhcpC+jgA0BVwWQ/oDm8BJeBNluAoSYgk4MCFebWFjxWMkrEY6MZveLEDZk9gAkhwELiPxOhyVjhJHI9EMyhpRusEPrubLM1GijLRgKQZ2a6EtYeOvapINhsQqf5jIH4GvAsSAOzNQeAnw3NWghSDQOiBUDxQWnhzwXtPV9v0usQlhLbQMa2I3uE6AKzYEuU6KBJYHQRxzXnOqiTdsSpI1lLJ4KLQ5D6sYDmAAwgg9lZSzUtc0ZLIC5SFxYJXgCr8l8AEgYkqNvaOWbnObqPsNlWgeHTFWRxEXJAG6AFrk5Q4YJJukyqReg4pAxopRSUDQHdSCQJmW9AlHhEdet8k/BbJrbQiO5LrC2gBiBygAHWvqtWRXG34GhQprU0JY6NmQsWUBhWwwHq9YwEgleVFgnwM0wuuSUkmG+6hwiqrBAQdGA70JothjkgPEjwQj6wHJKgMDvPYO9DJKUKIMl4SBsU2pU4xEIZtsoivgZVJe0bIsE+zddQDCgKtQMsi5dZZEIml0msPMwcsQ5gAkQcEBTGE9eB5gBK43gFfPn33C5IGB4t4HGzXFUGBVSzZF15k7Gdo8On9k9cw6i/Pnvy1ZO92H3EfUDMAvKB/CBThUW/JEeHhcBN8Ht4UwWZ4tRzIHQnlnRegFVBA0izV3TUpSpRoIA5AA6A9sRaroIgEPzKQzCqrSPHQvAixGSHG1smykAwrF9ea6GsegPzC7HG1BKWyHMcEaFLa2JtPr/9Wg96imiWqoTJGdfMYkAFq9HBx+OcQmJpWHXerJU0N0NwC51bAuiAkQDBN26stYALiAm0AxBB3SVpXD/fzXi8usg3z/XgLDMZ9n4m1BSyBBIIyvZ56VlzBrEquvgObrWDu6qv4BQ+8Da9Aw1WBevP3MkvVZ9jYV+pzVqpPRQ203C4lHeonu/pjtSp4gHOvHwBbiSnkABfGVvi/w2HoRbXLkVjy+ZN01+s19mo2Z85DN1en9+vzVz+//Gh1Pj/hp8HJZBxdhIDeScSn8fnJI/4ovDg+uwjOTk7GywgskcDpfXr79O0Hq+845svjOIouphfLMDo7O7ngY+h5Gpydn4fxoygaT6Y8nk6c3psnH199AqyffHw+YC/fvoPe05PxeDxggLea08snH14+/wCvvtIO7ZRFOIItvqrKEQnzKIw3sNE7M8D6HLA9jc8fLR+dB8cXUcingMRkHIYw2+UFP72IzqaTR5N4enLKp8v4LJg8CscXJ0CR49Pp+NwZ7BkC/t0KewKHuZgGk+ji0aPTs2X46NGj4+DRaTgdwySncXgxOV4eT6fjeBrwKT+LxyfTyfQsPg6i8+D0URQAPXGYu16vByqCEff5wIalC//czEDyyuqyrIoF+2/sDRpac/rVZ8Mfa2713gQbXuZgCM4I3xwa1e+eFFdb1LHv8FvhgtqnPQrYfu77URb6/gC1BjAtbNc+7FBlOa87vw9un+kOL/k6f6Ga9sVQXhBFiC+N4TrDIewjcXLlDJAr+RyZFExV6Dh3/ssH4Ids+XcOmwQofDUIqEbQ6amwDVKcyWP29G+vWAb7fJHgNpVUjhhNmqVzOexmi6bBeueDwlhvS7CK/Ksi2+au2bqFIO4Vzr0tyCA0Z7FvvqjAumb7VhvZtpEbiC0LRBY3QdxAS5i5sx++ZX0+DCXTIN0LWVuVnfgrBY76HtQ4uUWwWbw4hdn4NxP/pwDW7vnrJ6ZXsncs2F2D5cFhandIjEcdtuug2B2CmZXDKCm64P6NoEgif8KWTHCmtws2a9qiwD3AvdYnkfbAFdw7UqpGAFoN1FY+n+xrXsJO1NnjZLqvSwRsECK/qaZOmG8dNRn4jCZKuI2C2Zu9WAprq00QBRK/uY72hpz+3gloT6JzHoae3gciR0t3uClVfzC2AgPCeC9TFllZonX6rX2luwbaejg83P+05hGyS4e5MJ6Q98DhGJKpbVjJA7CivemYATwaYx/mhll238DPeJhsgD1//mkAfP852Ww37HTvuqJ6BOngwyxd7wA0GNmovWsl/VOWoY+iJOlJKFwrNeqLYF3yvSudG7zaQHdyL52VSzcUrlEp90v8MQDWz8TkPwIoMOOkI0wmM3UeMFLjZPcR95Qj3ALWQQ4bwu9bMPCSL9Aarf4Cdwk52N6ZVTwvO7n3eK8UhvEV9CuILfYQZW/f8jbY7ekz3KspMuRTDr4WrkG4Qp+2nLsOB+MChdjZJJFS0E5xfeL0DQUhGu1VWdkQ5l9qdnFK0Hbcr4otr/XKL6DBt2mC2z/4XWRBRMItIg2g/K79ar2itRwW+2XVu5+LUHBI6DQbdcI6U2j/RP6K9oE1K+lIHHjDBPN+mRVDL3cV72aXydleZXUb3AzBmsc+1vq9e/ran5zhouGn6Ql+evG3t08+Wgso391LH3Sqhutkk1R7iXxxrmjzDhqzkCe0eDdJwGghq20k1AdD2y3dyuiLOx5MFiaFrlNwdwfMJxNL2KL0SFukol0Si6ae2FNnNbolr8CWuyoBALpEHrnOrtnWQ//GR8PEBRMlQ3GfO9sqHp6r/Qh/BNMimK/io4dW4kw+JzdXfkxgsT1ftQfEjPbk7kMD10Hi4CpI67R/p0cCh/YWWGeOuLsSjkYEABKQkkKMYFO5aopA/ySs+rgxY0/1GCxzBVOTRaww2LNZ4TpPCQe22QKCS4wiKaMYQzBBQsEKWiuwzokbMFYiGaNtLDsaV3L+4dmA3QTrLceZK6y8pOKb0u3bKEkSzhW5L7H3wmoC8zfmHsi9xa03H/8DqpWPoFXUvnN4X+o3UMAfwDbBBTAGogkM2BJA9K32fK2XGGUBYwAgrt8CFMOoZAA2FvabxiQR/JZRXVIq1K3/rcOX/FtGgh2s31xDHEv0qCXHk/qKcBEMI8XFft8eWnFy7LxKBVBy7ZA0IwEIOfErMtN3xZ3BoIqLUa7LXemhQrmczBYktvAZCUtOLk6ZnphyGKQ7MUU2nzPlyFHoDh96QISiKjEepny4udMnTGppqMdvMKGSkksHu/nk+i2kp30fBsJRPIQGtfgncWmjkXvQwpeqv3SPjmrFQ00CQ3U3lLZSD0IjSr/VqZ1FX/mXjuUYOsp1c8jf8oV74QhnQ33TZDNmIykF23MFu07hBgMa3tKtkhwiMIvzbBHD7DqgoJdYgn69E7mBJ+K/pQyNwOLIZ4RT/WLWa+niD5gc2InINfxG3stimRPCNTXXuB6QQqcepZEo3Cq+WWTseqFgdj0HtimS3O1CUYQYBh2JKBTcAMO9fJNXu3YmSsRXKf8XSexT4LkiCWFVtIkReOQFDIwHYBf56Mv4V0vzsTD3fDD3mo3JiPKVJW+8JEvetzvUPl7jeccjtH18sn0sAPGVr+x08zna4KZXoNZqvXYxKOslZQzbawWaUgojioIkSRfpn6zX6jWGQJRto3ZuAczmizH7waYf+2HOTknreim8m4iP5J3A16n8SlkcaDkW3zWh6WEHau/FutJ4hudJo4GwsB/nbIJbAQ4Dn6cDmSn6kcnccG254yM9B4stKAzgU0LxB3Y+Ho/rlwJr4/2PbHIxbbQgenhTxKmDS/DxZGpD1I3IHsdRxXIeoICZ9kQcPU9gQq617Vgwb+phPh4J0DD9kUjnmgyacQEHuSgm26qnrtGQxNFq1YX2O3JtRWpdNdUsBcKc8iuR4aUmlKUAo6Ba4WueBst1Lcya5eZaevDLtItjZFCkHgpY/3I8mI4Xj9k6A5NTIBJiHq/A0B4ydxKsR8DlyPzAPeGqi9e1iNKKdoysHRhzcOF8GBRXzlvNIMa2QMRw8ZstyO1efbFYLWAoR13r0Yxh1ChSXYRah6ao2zpIcQPiN5yI5UBdtI8ginXNeAMyoBBN6omrMwQJrklU8GpbgNkgEwW0R/sigO3iBjGjvZHyAnofVdFltLfqrJNXbFMt5pfOFTmWzvCpQ7YjQeuTZXAzJEMC3758/uSZsxhg9pDSaGL/n6PhP6D0q/xIeNFnIbig6SJoq/Y3Rb8ase/mXXUl4qcIEuDD99sUgxDPpcmJKWCRBqgdqIp9bcC4ewz6HYMEX9VAygxFtV/wNUkYRpxycLywPiFlVnqp7S+BgYB1CHOVE/TKVTA9PXOJXGxUA+0LB5eUiotf8nUAS7R0fit+wzgyfEjB0/VW/HOUXIE351oGkRwGyKJwsy2iTpq8Bjc6Tng0SsCj3uSAB2gJHQwR9JohMQSOihiU3he+7zdzB4wY04sh8DKvah6hR/hvM2lG3EPcQRFJe5uWmHiCyzETf5ANnI86gyIy+JInsC5AJBA2RBOZ+lb5HvA5ctIfacmLCjQQTUktIaLs4JKV2fqGu30wMXs9yoix13yTFbvXWYqlUgI1x0H7QKbZMamer7egPVQqXKfg70uKUzLco1Q1wkXp9n3UN77vlnwdD0SF2Awzuxj0wUzBDBEfMFKqYGTMpGNpBmLWsUf9BuKz6Ce/CGU8Z7IB/ZItXPGrT36yqwZgR2zCL/o2+LLKcp/fYFhirhPX3nN84jbapjebtY+UmCFcO5nZQJoAzQyAH+nToT5xIXxWSYc9TYHPDFIIVx4dOUy1NKSM2PBgx++w4zYKHKX4BRHxEWxPfnATJGvcrJsxF8HNn9CFkbz8FmM9mOIhA0L1k9keud6N4izDpzaGFY4hLaKBrtm0Ig1mdLmqu/hYdANCkYDaMnt71MvfkABobajQ9GVVxpxtktTVzDUQo2E0DPlo7D06NfkHNV0ThG3pdtMKa1JUIQjp/zUPYItG8Oznn0i7U8EKZg5WwDROv5tRMObVHH8kUN5HWSCOL3WkpIavoLkWbEvgOhcKE04VSYOCVIKfW+5ZtWJnU0XWeeQ7lKmeHWOkZyRsr0CBuHYcCHRshCWBc3ZpNHxG4/3Mq5f0+qfdqzTin91EuEgJbo0FKme3q89TLAYDTbmw8UAdJut23IZJYv7crtA9R+Fp6BOUIAyqdgQL8UeUDwG7dL6tpQRa6K2m+YOTEwTBGUrSdA/XuQqtoBiwBpK2m0obMJY4bGEY3H0nmAjkTIzaPwgX8UTYFOjGMQ6j0XLj7sPZyzGMOGdZicoAvrj9B/Wl4kcdeyRQWHL5c74Vm6UItD4YVhsAWNUuGNM/MJWAOfTTnz1oJME53887hntQf5O3aiO36wdr2HIlpB7u86TB7uEhMDV6e1a0Hnk/CHujRZUcfHYbD4GbwOohMnQTtSmJt0GC6nt82u/1Wu3EHm3t/mKzdkV8YS6VAJgtARA5Jcegvw+OiJ0a/CeJ+Jx+4fZu9fyLMJ/AS1R21WP2a5JG2W3Jfn327DVMf0fKBcufS14XHxrVob0W9WsTrEyDvFxlFRGQdBimey7J7AJDbDH7p80K9tW5yrdC/Yu04wzXxJE1IDMmaz8cLM70xfYrgkmqzhHaICHvOqw9uV/t0nBVgLX6ZY8poGt0563OGISR21LdrBuKrDq9B4hqtRcG1sWqcr467ylqc4hINRY1ueong72d1Kh1H/Wg0YVEQ9rj9f5urY0tQY3utZlfGwOqR20O2B2EsWEDbwQ4qd2hxaddTbe/M1kRZV6TR8+adSiCPvvRQOABjuZbVYuLhcsERtTqltImIx8T19PM+kiuFy+0gIVrEEotXbaF0NRCZAy0JE66Cnv1yd8zsEkR/2xL5SwidFJiQlxkoPEf6UrhgvhqFpQiBKTq6sva5RO1B798fDE8ZyJwKIwYUepfYO425UGx3NVV/yKOJyr/eRpRNhadPTkRY1wMDzfdXtP4pVMF1MEygY1YlaQ6HssoB5Li6OOCWkUlTYkGnL0L7t0l+OPjOiaCXdDEoK6Wblvz1MWnHqXruVsn64F5TKIdttxj54kYQzFM2JwM8A42+K64ewyIhLCAAKFdoiFnSEQXE/uhRtEMuAKpUS1Ri+/ZpGfbnPjW7EeLhF8d5gi2oeeXov+M2gOURb9NBXBb9pKBus2N0YGezaGbm9oTxUAYyllzKrWXnCRPpYC0cSzBxe9lBtuSOi1CB0Aa4MItZS1Yma05+Jm3K56yQNCdTnIAmDhZr0u2zT32LKMNE4lfcTqT0oCWb1MMUJELJc+kEKuveUijBOpwy45mQZWuYqWJkvaGCx4jlb7pZULlNRkwF2k0FE/7bDRiU9tmIJuYqpCp/sMtnEvvu3/593/8659//Pc///i/i8vfnL/+2z/+z7/943/+tujfLY7+E+ym0GbwePbnH//jzz/+959//K+ONh0mZAhskURBJZympOkRyQkMFHv0RWgTtFUA+5ArURwIoYLuQ+SgRa/DutMDdZt3gpt1q8vhZNHZcAlLd93YXD08P5JGnbw9gxcLo/BG6Qp43DMUgACEIj+pRUU8Q0w6hIJNDF8eOHlJB9nmTHWaLoBggBD8q+E01Y7q921CVw8xw+T5pQKiobcLKugcU7YtLQSVqmyxHoiU5gHEU3XHaAewLxhzw0kHN+HxLjwQoZZBdbsEgLNF/xBBGoRBSN+kj7vZ42FQ59ZaHqB0e2IzmNkC9hqEfB+/NswE2qopp4WluKV7G9xwGQKV9cu0TQ9YmtNj2q/hd71Tvwfr84aLE05Uzds65KY2blX9i8oKh8F6SHlord6pSVfKKBdqqXGNBkyPlDl0NHWVzJFRv1nTFsLGPcv6xSdemOU7yW9g8uExjjQH2yylykwcEpiLAII/RYWA8JoCnqqkVS7IjBot2NGc4JivhgKh+t3lbKb4rGsBRPwXLT1zAWR9oYy22ktQbYEolyJUXTtM1G6hrah3hQgbwwLyK14M8chmuE7yXCxDkW2vVlSMsW6VaF+ts2WwZldBktaLkxXJFWYpld9Lw7lAHFwp+BUsBf/0paLDznItJ964ng8b2ZBInduwqTAJ+vQai8mOCOgAfDurAxjr1nfw6rAdPBbNHTk0PJCf1EkgFCTwSAQBXPGrlOeBgGxgvFHiWQqBrJTWlmtdE6eCvhLC5XhR23ziEaoy9XIyM/xa0RNjJWS7fq/aD4lwckQZ5x3Il1auUoRT9XREaQythJoJetL7Z7JHuEVGWsiMNSFaiSSVL6zJmPJbo45yS8BMGbbcjIb4/vOSWe8fEnd4D8oInbQU/m+H6wRelzMhr0Aj8V2LL8bGhoQMbhyIuRZ7ejwQD2UHG4v+gb1wD4YCIVwRD4+uli4tWsd0xbh9ixOor+SE5TZZg6e9DmSaMZh1HGQTOWXpmoFr5RvaB78ab6k+zd8gy2iFgxLS0y6cHbqRbEFnJ0WG0hNngLYVnr+W8XWYPPBJ5Yewg4Aj7oNu2qHIIkDM/OOzJPrck/OMfXEsgUwIgS+miHOO+zjuCy/fvusJI/op2vOYI3xxyjZcHGYG6mWYXFVFYer4/oitMlrhiTz24MntCCxBtMJBY9gDyy/K5cGyGFlMM5dWm6Bee6MfMbcLc9Y67WO6rEKiWpU7VHzjnoskooHDUWcl0JEqgVIKWroW86avbrnp0hMU1yig/UEmI/Zsz43MNvJDEuW4LLR69AUQyoNvXAlR1uUDnxJwrWWo+0AOjPC+JLkYt1RPrcqfFRboEf27Wcq9VGsCq0ZwFmbW6JoTAg22cxVYQwAavXyl83ANXAnouznaph7Os09FDq6t8LrLcNrpm7+wFwXnX7g49y+vG9CRVXXKEVRE8deyeV+B1wCF2fxtjpUwwYb9MBmTq49N6QBliIEAKpqizvOxdzxA4y1coUPUACWKZPR1CnQrAF7jkIGuTsItWKXgBYPjjDVPPJLaDh2iESk0r+FKKLwbwkVsrR+M7hGtI8Uto5rTD2hgHe9UWmDetS5HpmSaQ5g8/bDpoLBOhbC2ByfVRSrIOGqgVA5K/9np6fGZkPh6BPHV4MNBl84SUC0GlJB/aGozDFjKb0d0NtxSTOD7TKf3hKBae6wMqAF/iwJavFxERqemU2n+mEiP5EEjqqkRi/CYlXipBk91pGNUK3Fnz54r6yaVeLpyWkNzxkOMJhzVm4aKepo9f9yjS02yfI8QvpkuL0godI5AHomUtOk4scWCuOIi2vP0xWuMdm7yx3RbScDKDQCCl2bcEo1PTBAppSxON+0hF6lgFcOw2nxto05F8zOpo9uvldaEJrUCbbeqRU2eNJxJzutoauzYa6yiqHvotezo1SEKiHT7aUfflEq46kO8M4spOtobMgiNTYm02uokgmW5EfGVM5gFkTgtXbp7rDaxF2GNlahHNGuXmu5hw0fEMPs+y0zZZMBcA/Ys+fgQ+w2zLDTbBOxV3aFxbZZqvYrxm6+ue9HtjdPtIr+lehA18GCAbksnJNR7OnTeU7GA9tUhHpgCWJnoioMVTp9Kiuj2Luf+4jc6d0PCKQ63R5yur0FlZt4DVhf6qRMdtJUYX0ES6UyFPqRrE8J1rGu86CSjeQOAqO3zJ9PTMdZem3cBDOoqz3njRpF+s/hPoQN2SRwnn4kSFqxDaZGPxn1TLAgxV1yKA3TqLq3hOtjh1Qt4W4EJ1hhZJRZqY0rxcgnsIw6xaG5ysWpQszqWQ4bbsso2Eojg2Tmyq2vU8YAkYWiJF3NgYVuTRclmPhlPTwaNMpq8Ws2nU/sp1jWV88mZ/TSO/Q0dULYf0ykYhH466XqzCcprmENExzCpgtdq8/u1nwLGc7T/7DdoyvpAVuDheQPrnPtBVaU+4tnRk94tgxBoGc1FxtbpaEGYyXr3eSPhWaOfbjc+3yxhV5rr1bLbgYPlC+qO9Yu+/ojv0dL0r2/xvNS8kYaGPSBGvdleHAccMxmegfewzTZe3yapft3RHaYIQ2PhawoKldqMm+lscQzEOGcBzcxLcuzm9VToQPJMntgyiHunPwKHAtfXk3bkKXTQ6l6G5wjoq9FekJccFxCOuRYTQ5SxiA1lpdaMJCfG5XWqelXcbmF30nS/5juvoOAtJlWTz67DN4G8pcPpz8SZMevQK/Qwz7wiRFXAbVpP0EyfCMYC24ruPnDwlIzE5q5nwBQNFVU9+lDFlfgQBmvuxXQwrt0Cv1VFdgV7uHdLJSlmdkkgmGe5S4jT2TStN+jMNBUEch93R5c+0v4I34wKGip7xnfim6g7kWWGFIyZnJHNiOfn6mpO83gila/2RUzT6HY8tZSYQIoDeV0sBpX1wfOgrigWESD61zgfUZ8WNGNwePDZFzdaivd0Pm9uNsc6bOOmFrzAwH7Zvral13ajjKEAvL3Dde5yyNkBrNlI3CoDKzmcnlyvvuhD4wqdelOzb7rq99qWqzXH/08k2tN+MCryajvAgWwFD20Vf4XXt4GEkIhqavWtLi1eFGxC2OrJoa8HqlC6/urqGhkJ8XEnFlzbwcMaNTWiZrTAqhVKULCMdri7ePSwPrpG3zy5mw1ED09rafWEVDpZW0BxCsU5dNvYgLR0/6DtZSvcX1JdDyjMMBkiB3pgKuOxOn6pr1wNVexP3Ns5mp6eDQFFZb853Va4HpeW31LL2E0/aGxfxlFeCpda2thuKs75ilambdNqBasur1GQIA3u6WwtOUHDlnzTb+1MqkRnE2BQ8d4L0OokB2k1CmvQPZE5L2JfVje4xrnorlPRFHdDDSTOEMdMnyauD5+J4+j6RPH+OyyGJd4uYQI2T/fWxTm17y/CRl6c4J3Da7dwfvvwvSOC330rjzihU35GXzzoOR4fsItt61BYyToAhGIm6zgoj43znYw9D2BibK0SDt1Q3fIYiRqGx+wKcPmq0bhrMqx1Lk0ckjYO1PSM+nWw3fIdXtCa5ubj+tJjurs1Nl+R8vmPdP3wGmYPVhSjIfU1jtIay7P1TjfF+5m9YFtlNa7woGecUNfn5XEP2wRpEsNCeHhJitP3+GfgcrBW6OxiurPae5jxdB057BFeOuz0W1rqBVDsOUHpXP234iZWAAeGCdZWB2vk4Z26c6SkW1FBiBKgDl5K/NVE4e4xXm6TUTgnBgqsmHXbWIMFNuKUFHC3dWrKlYeNtOFgnZOWQmOW2VtU2FzDvy5wI96ZIA8bEtn87LpRVowZDNhrEIMyJll1Wwf0ldniSJMH7Yz1bbAr/WnUAU5l6Dc8AMUEw84ndjligfuJYZo3cmpBKChCp1evwsgti0E7sdIY0eI1V8zKDO6NRhIwTbb+1veCEufmduUglTpB/WGULVhw4dW5OtUEMOrDttTWQz1175mmd/X9yaLvQN6jrC/QXfLqlvNU1hOp25XN62zwbLqRyy9/L0ROH5eAFqMxzbOTPjs6YtO+HVCWh+gnfHh2H9b1zdgUE45jLGq7wao5uiG6smpaRaJMG9xoU5FKc41Sjr63TUtQufwLd8dtrKyT/s0ipnqA+uMRXWuhrwYYIRQbp3o3675eoskGRsYiEk6B63hobf77P/7VaV7Vo7NUGJ1xep3PmdNwGdGIUBZ+sGzHm/6ZmBOBwlva740xtexxFdzxvzWmU5sZ27yOtXeaG6pwkUdWBJuCuvfbKWbO0ciUB8Im0LlwnQbvTACi57xGq+teXJu4aRBU9QHshxuyec+BnWEYYdzC4G1ZXCHTjo1bEw53lZFtZRvZFTCX+WUz9L0QhaB0MgpJpipJ6iIS8ySJMNN10sUa7Khxm8X3Fp6UixlZOrJnVQ7y2/b5hVaY3hywYSbX2OnWNsKN9kpXNpuJ9KKex6DpDGCOGnpRzSJSrGmvNxicLHbrSaN9i80wr9J81pysuM4yKzBC9eLUSDOJlMdjSiEhZ+tMmsoo1d4TeVuPwWuCr1iFCzBDYphSXfCPRVXN2Qnq+iW42eDPyiSPPEzTdZwiL5CH6SK7CFR76X7VMxYzxbW/G9CfR0jxckgsdduWq4YFocy9NpeIk/GzhoJrIU7mtF/fFTxr3pnQDD2Sw2W2byjHTjfNaG7HE1rcJ+7BgXZfr6VP129cFHczkNdRkOt0IyJqA3Hhzg24YW7QVxG6u2bQU2TwSLU1plW7HfBef2nzJK6KXp/me2B8fIn833glzE2c2OWiFVtFUxbeSOvWq8+H9bs4R6243v4OOQDd3bzbAkgk3EyDB1UbzXVYf46XDvgBeCyJvPAVHzauV7TrPNS1shSg6Tynpo/XqjDQAzYvEaCoAxNGfY6VPjRThTJJ2JaXS0eFDrC+WH7uaCVyUAJJqYcWezc8cy69A+ZWOwQlKkw+ycwbZbsDWKLVhqP2CbCisxRlL/IWj5jjH7Xg6CLiCY4IjF4QBM7KXQqtyqT0GteL+Nps0xUW5N+qGHNdotXXNVrt+0PacMA7IY6/HC8uO9PPi3uPf7VTy9pexjK2epYyCIY3hPCilA6ylbNUenz44eOLjypati//T3/iAv1pl0oy5niR1pG5h+GlmOHc0X9dR5UsYA5yC+I6F1uf08eYQo5xeV427G1UUPIPnNSl/TBQRxW/3F8fJg2mR2eUkXXB4wQM7xrgeJhEPO51HQdHuf88oPVEZDldUYY1moIe3bX6gorkqySpYg7kLffA0WkdJRI1SOI+sSP2Vd4+PMO6V31L8QzvGaOLimfs5O7STCUt9o6hl/fg2Wix9gZCg4PNiSVi50V9mTFnX+Xv79nkbiALT74SLemRcxig4CMc/56GdCNFV8bS4m4maj5vfc2O98wHZDu6CUikoZ975Ouiiu7bFVqsYw7nbXMURXfSPzzuKsuuayVkpLDBwbwCq4sXPny/xbOq2NI1UTx8aP/eCxWoUF/J8wDT53+XsSuFjljPh111gCpvXjtPD+pCt1qhSF3qsqHFw7oq21J2b9YTPRAKSdtcSt3Dupi3ks33XZR4cMzbYOcrAxoUGo/n5r2K93bHAk1To30vtBXeH4QFIuPjwQPvaeA+3l8+F5dlphl9u7/vPXdpYF3b+gF8h7wsc9AHrsPA/Khmy96B6zhqPpab9mShNuQO5jiMXldBEDpMwmbDi6dS0FX1xWYEVBxNcPbPZEOZ3hrLy9lAIrenvG3BoMVsIeJubh93v822ooMQsANM9g8kLi7VWTyR9nJhfLp3C4njDg/0f0DkbN+Q4vcRc0XozASwfzxRWyktQ4LghfkWZixCfX3zyNY+FjGU2AZNYwKz75qPuoZdGHmaTZqxj/2T7uIQOzuZlPTXVjA9GW9L8TcoWR7Q5dD4R/Ds2x47sGxGhg2Mu+LD92H3OrPoxN4Eb0av0lhQag8W1soY5wT180EdujoYfsIjLd1DkNGmCk+NKfb23KXT3Fg7Ja3lPvY6tZSMf5lHpprRrW7EybczY1mH8hL78xN2HGl/qqKJtJ2zkGeGbISMtEVnNqPOX+zlIZ3Y6D1sE1C38ll5V3Hyq9c+1CouSfTpsk4U+yU43TaAYSPwhhkUu0nXujSAiDseDSh7Dtk+wBPTiT7xx0frqx/pb3viE/yrefg3ML/aQ96J+C+8wWrUbcm+2kjeIV9H2xD/QJ+gx1ebPjPv7D/f9R/j3wiV5rUawHng2kgeoStXsNgQnMQvgbyjzTh+WrOScSV0pwA8QMLMP6LQis3EKtVq+Auz8XF0RwaNekpf7igZ2/boYhG6cY1xBqxLFgZM/vkQMLIAlC+/dUh1GmcinYmfTMCdGgAbedrzbzAeuZDQQCCilISF2INZ0cGju2LJ1fYi3Xz6Ox3EdR0aXMduVMxt0Vnhv7/SX8fzAqqwxDCkQRaj6GC/1ejQX+qamS74gcZtDsUwYuvhIQiwBkn48EB/IwIpF1H3NZb5QMfmwSHVUT0/0NWWdTrVYD440FMflTGG3ZMtswIqB2CCIs7WYGbKW6HUH7ZxZp0HvoV637f8d70H7Bo6wCiivwtyPe9VL/9BUVwrSttybRSe4polLKxK8F5Z/DMKvk/3k/k+lln5viyutxxyIekfduB6bp5/TiqXKrJkYr15LRvGMuDZrJE3gc3o/fu372cM9pzwzkENuOZzvIO3rMD2N/5ISWu0SV9EDaFj7/8BPYQlcqt9AAA="
  },
  "emotivoice_samples.py": {
    "sha256": "d896d9e96a308b433d30f3e2391ae30479ebffd83a8b1271d22cc133c3bffd21",
    "gzip_base64": "H4sIAAAAAAAC/719a5PbRpLgd/6KMny7BiUSJFuSx6ZMz8qyZGvOYykk2ROx7Q4ESIAkLBLA4NEtuq/vt18+6gmAbGo8e45wiwDqkVWVla/KzPr8s0lTlZNlmk2S7FoUh3qbZ48Gnue9q6MsjnZ5logX+7xOf83TVSKqaF/sklLcpPVWrKLVNolFAVWSfSLipozqNM8qATVF8jFa1fC3yMta1Ok+zTbBYPBLlYg31Il4FMxm3E69TUS+XqerNNqJMinyKq3z8vBFBQ//bNISGs/qShS7phLVKi0OgXiOgA22dV1U88lkA600y2CV7ydZUidRlYwPeRNH+cSAHkAhAgxgXn0QeVMPnny9nq4ffRknj+Pl7OKriyfrx3+ZPonjx8n0L19N4798HcdfLpfTWSC+z2+yXR7F4rvX738UmyRLYKh5Se1V9WGXiCRb5XFSDqj1Ik8R4Gi1yssYBi7qXHwKrJOb9EM6eVMmdRmlWRKP99D2rgoG72Gm6vxDkqV/JOXkXbr/7sXb94K+ihQ63FW5mrNY+HGyjppdLf4RZZv/Tv7WZJMq3S+Tsh4vsdvVFtqukiEsy4uPtK7zgZAoIBIE5hqBCXnNq6A4iPEYl0dMiqjeTmobZPEb1BVQQE/O2EyFqbAJp9PZ4yn8pyvQ/PUWNu/C2ZePHj+aYekiiT4AAn49nf1FN1GU+b6oxc9JAxO241IwAbNgCr/r5GM9XqewRllUMoYG9ccavgAODgamSZxAsYuWOJeZiKM6msiVwSYmstjFSGQ5YhJM0TKJaXnTLE4+BqapMb1A5N8BUte7AyBdnldJ1VMN/laABnHA2yyHRcIJGfCYoEaZiKiAp4/pPqoTudirPKvLHFBCvKOhAuwprhLuFHhcHkQSpzWh3hbbgqcVfoC5TDYw1u6eXSbrHLr6IWqqKgUwm4LWHZoYwbcVvE6oLXhfl0m0h6bWSQlonwhcL5FuMqgPYCDMu2IbAeSbBuEJxBvuHzZMkdar7QRRZHMYVKtol/AIswQA2cH8RdCeGhzPdJXsgRxkyST+zgx7MHhdINy02skeuqvSavx7BZjLy5hWNdAUceuttk32wZuL6Uh4ctTwdDEaeGrsIcEBL2fBBRZCEK13M3jHANsF7wLxKoNBJUikYCIGsMWRjsVpVeyiQxJP9lGWrpMKZ51gUHMOw0qz1a4hFFjmDVDZ8sC7Wu5wtW8txHRI5B6gELCmVVLzPvD/lm8z8SusGlLtIVPVaJDJ/cAUihEqEM8YYFq5fzz7Fah4htO8IlIfCdrz0F+ZNxskzdB5oQnRwCBvHS13yVMR8VMEK3ijAZYkiMHWCENtYOGbvPyw3uU3weBNk63qhlZhHN0gItBUwRRVW+QbSbxJxDqKcdKQ1t5sc6AVZaLoKvYnorpOMm5kEAH4saC/CJ/4PacuJ6tdWhQEaFmna+BMFU0ScAFA1jop5U7R+wGYTZQBlj1/84uafrUqCMemicoIthIMEQhxLn6AYr++ffZ3YEy/fP8Ma+QKPbG4mg+gENEOBj5usmvoc50CaV8BhRkvGxhoPafNAzheJeU1gHGNvPJ9Xq5gMaHailgOF4UuBkSGxDYqY0YjAWDHJdQqJ7hPkGbB/0lJUHB93OmBeJ0BRcKRrZsdEMYdbJgBrBHsUGoN14EoWJHAHyDI3yyeiB++M4OmDX2dwoKXTcaTwxQS8WwyMHOotj1MB3ZJ3BdlBq6DUyyA6Iib6BqJz15Uhww+wE6Gecx3MQAFy9UUvPQRFChkvyxPMOlAjlQybdMg7w5Pia4CuEDRAT1Wh4HE82ViBhJrJFhCke0+Kj8gIgDVB9aOiLnTEwK4gFtTs9oYeHcJQACdSVfIL5Jr3JnweS4yhCmJaiy1S5fIDhMY/haGKVIELwU6MoiAAuQ0fv0O136dbhRNhlnYAutGkADeOIcJApgr3HMVkEagNbDWICsdAhTYBoM17HARhuumbsokDJEj4CaicXOLg4F6V26KqKwS9Qz0cwugqkf+B14ETZ3u1Fskr+o38KKt+p1X6lepG6yaJRAcII/6W3XQP2H5EoZWIir0pKCVb4B6RsCouBTy4tUuqpCDymL6FZdAFmQ18gahow/1gba9fP8sO4zEq5oFlMHg3etf3j5/Eb598eurd69e/ywWwjtXJvQGPz97/+pXqPzs/QuoOPsSZJrBYABUQtDMhjDFlQ9/rufEjS5hh12J/yN+RjK7oH+GYvytXong5wh4A/K/OUk19LKEkrrAM8lP39AXn0rhf0AdV2VK9GYRhnG+CsMR4tUeCWMZ0iwtdCtvo5vvTYUfk13xUhWlFodW7wFgWqjYuO+Nx4yf3ginNVngLI/EFppYeH97B/MnSWT1VDz/6ZXIATtLwG14/JAcKoESBJApQNkVSgoe94SLDaO0Otw3uPN2hzD5CGyyAiQPN8CNCt9U6ACGL717vpMQaMN+cqhIVvoG+ibNgBEaZlwBsUCZBWVVoCTeyUb7hOOzIWoLymdXJKI1PkT7Xd+AvpeLhkoKDnrCizxB1lnLh4AqH+9E4yL0RgoG7kdvZKEo9bHwTqsiVg2G7accqaLSc2LgochODgKI5o/NhijjS5QYmSy/+l42cB6gyBF3hBTVGPmT1TsQf9xNetN8l4PcEWVK5nxGn7vDewkKWNIew/c5ibGxUiHlaCZajxMEggO6EqQ+ZV/IOl204few9ApMpD1q9V+Qmq76I+3nqRZ1SNC0NS7vrK5Y/VG4Blh0EjtZLrXA86Qe5ykY35FckbB6hAwCVZucRElr39+33KQTKqBA/Ixq0yVoiqqzH9PNlmXudQTCe/lU60xarkE1gIR3VpliEEdRGvXup5+kW4y1XDRmheIoTCfbYp3k39GYPUsg0oylojJel8iSrG2h1rOL+bM21v8dRKN9s4edlDckInFjJO/sko8k7VA/IM6sYZoFq+xyap+KKdCiqiaNUisQywREqDQvP2mXO8phL0GXoyWkBRDUMk9sXZUAS0iprs5Y5330cUyaDPyFEtZWMEty8fjcJdGNfcKCXIA40l6Tt8nvQD+JL1dAeGKzOJZ6oLBasqoUJ+3+AYO+mERx/0Afn6yZ9daZneaE1k52q108Ot1bDPrDKrGJzapoNKGB38haUCW7nBM+XJ1JXE6pddD8efzE5SP3TzqIG0VTj4EtunitmsEn3+ta8rzh6dmlUmPUWnonWZPKKYp0oImwqkoSsPjxj9MSUIGGpPG+OkKnpoYQv5DWM1EBe0RjxTKpb5IkkyYCUinPXB00I4zRjEAdu7uHAejsn0dBZ/s8zyuQUsgcIaJaJKDJSssOtv+ULTLlNcwIzyBwqiarkZTFaYXWCDSOnbOZNhEQ4Xh5fIrOG3WGcv0ONvoY2fKfl27el03SJSlxgxZA6AAI+RJIi5gGX3+FsiSbYt48/7tQxpdjU3TOnCARvAbyN94cm5cnBnW+T1YpDF388N0IV2qfV7V4chovgdmQfVVKgv/Sjj3ZwU10PQalGCGH5ldb3JDV4tKDCQpnX8Ir+nXxGH+9/On1s/felUWlZCnuIYnKHeixoREPWeP8kIGMafROLpyuuXzAkvy8vawVtIKsMUDhtPLtsgHS9BBFLJ+ON2AJF15Tr8dfKQoiO0D5Nq3Y4LJK1LlDBfCDZDI0XVrTA6phXhqVUuyBEZHuD7gvUEUmhTJfIsfyhi3URZhv+WcA+mU9l+9JupA/oRnZVSgr3elWYM/iVEErVVJrcEEdp2dZ3BmhrHFiKGvvF9kqQY4a71zcVmSa8mX14Z01FIQVSo3EdbRrEjpykIAEaZ3sK781cQAFFMdytx6iOSKK1Mfv3JI9sCmYxC38+ay8Q/mW7YcwYjwnsAAz84zWB56MS6h21QZHAl6xTQONdHpNWH+QneDn8yDkFnFmEM47YWx2WbPbtWC0ITjeDZSyUDOS+9bXGzt8h1Im0ja1p0/v+eGw2wf+B6AAuVsQafIJriECtoQ2OuWTHUAl5wrL0/FNVn9Sw1D+rHaJSn5Cy4BexO2p3l1PF1VyujVrtqnNkQDxfdi3LrhoVKu/wRaCvMq4B0IUGp3Gky5i2JOAiHkGotTloR8OtS1pD+gtEViz1p2lj6ukqIX/Hsq8QPhH4lcsSb+HIqqwxKePurU9gL5AM/boNbEnq9rR4eoReVguJLvYlTRKDhxkMq1RqXOb7Lam27IOte5rSRYN2ZxwCkCn5Cc03W1UTj9QxVCV9h880CyCyiGHbfPeLtfFp0ovBNt7FwvrtZ5R+jYf9HPIdwUIM+sD+1PsDsQb87U810ZtxTJu2ieLjkHW01BR7z1rwOTb+nhqLvshjFwzEsJmXlFrfKC7zOutBVB28EFJiuq6pKkBypMmu3io+QqiO71C2uR70i7raVNq6FhEPbKS2u+GR6eWjbyjY04LOCM97gl85sQ+FnIYCGIGqjlDiBqtZxRiD0TXkPSEUBkDrFdSpbdg7MpT7vRgR0Nc6XqIU9zzUXwjZicllVssdscy1xIPfcnlBpU4aetq4UtrBND+xfT4pLaMH6abWgA7hd8X0zY+ounnG3GiTSqhGsryLEs2pHX2I3YXd4+h9+lOHW+Ok71jP1PxzULOV5qF0ogmFxg/zU7NWcfspvtTau+UxjCb9uKcMm5K7wVtFAyVUVC5MLS/dBEPT/aCtFqnGYihfbhHaEcDDi6e4MB6UXAhHn8KDsIoLrG9kXh89RSIXY1eX9rfA4A64BlelAJNoXPTfzbAFuuDuwIu6LQUZG8I9xUB7bw5vfbKTqEB5EZpDY5iQE//aBoI0WSgYHAxxf6ML0/tK9uKcQQ77J11BCA0LoTxkudjWbVefitmp0CQlom+SVGAjKEBxtQzgEHCglp9uFnakyPaH3FqnpymONI4YCOUPx2JJ1don5fWgDa1IBtEiIYuua61+AqNWGp17AK4gb9Gq+oJemEMZxqMKQ4LGw0Crm4sZAoMtkmKzxaCjJEalDIJ1qD1wLSttn7poUXR/+t8/lv8cPhXb2TXPcHhZOMKnOMWzvugoY9k5gyNmTPEhkLpvdIPg0MDPHKPkVqddsMQdV7jcQA686AXjXQ4CQR6i0qILV+6e6ytukNDJ4+aBAzMLFq3aZllYxixmDboKhnSltNhN9wk8fKsVfY0ZbR6NQRSyn23bkMt4f/frOnqxbfaHbR0PlTwHCV3+C8MLmJxdclgyDGVSd2UGSGedKxQ3nIhHghlm3rr3+RlPCcFE30ptA7teZ7lWoZmR+nxNmKzJArIq3y/h28vss0urbZAC5foXMSHIBOkUWm0qwJ0rKEjz7pMiyJBBRf7DEp64Xu/eb81F9NZTH+//mJ4dWf2E2zgCvQT2r2Xn/0Vi1x8+VvzaDq9+K1Zr6cz/DtbXz38X95Id2BNn5yAC9WeKhIkWVyh55LvBTYDt4eAti1VHDZLUvqs27vbcV8Gll1Yvqq677qv4m7NoszXnZdVt9zv3VdV3Xl13e0zqVfdd8Gm8y4Nks67KNh3AXbfGWOHXH2YQ8RIv0WIgQhfPhv/dzT+4+q3YPjQXruOVdRZEr2buHl3nziLLR9mHSwaPZ0Tpj2mv48IjVaERkv6G7WQiew1QjkmsQxPBl38QxtnJHbpPkXfQ1QoYBNpZyW9lf53khRCOgiiSA9bGDQ62oyVuNmm7I26Rm8f3lfABYEFVuiLDHuumqx2KE2ZzQSjok5dRaWMUoDV2El87zmdrpguuZIiGkppcaUMcv/hzTk81fZ76bWIZDvZF1qU5NMl22drIS6vNCsBchZtyqjY4m7ivoAx1Tv0X7E7RCIhjQRcnsv5LgMB3XeXZETFUPbi8ZFLJLzBHqiZFkXtjGXtPWP/YbI7oYcgulfy0RQ3eUv/2MyCl42ad1sH2gIoQdQVwAcJaup87gDXtWIBe4/TGIWghWrooTDDfMibCvrpNQqa2nI++s1ksK2iD50v3N0IBwGd66b4xUMxa5sJ8fU3GrZqeGIwuJiXvaCwegitszMnGzrUHCVZsycHUO7hcg5dXg2x735mRjaii04/V8fspwa+/nk6A/7/mTHMevu6OrnkR8bQWs3qcjxz2+E9G0QFuv/6nvACdF5zoHVxTe1OLgEF5lc23eX2JNHUKppPmuecnUgt1YBI50js0x3QjATdk4B8kPmc6CkU15T0XVPgEWjFXtu8QxFU440OhGu/p6ALnNIaHePgNxEe8rQ3NJQ9WmFpigMalLNiIIdQ4dHLAl4EUYURLwcGfCRiOjaF9wTco4thsMqLgyRJdBIL1aBzv8Rl9W3t54EzPDERM1BlhkQofO5xKCYTcaFJMTf3rUvf9wV04E+DJ2Is8O8DBHKVVz78AxSUvF9RaYPHIh1xG8PhEMZBxnYLcktGws4v51T2SjxYUDftz2P6PNffL+dzhUNyybmgcuAt8zpZ1SFqJM6qSz1ZLi+aM+UROxWdE2WjRa8bmLpLqoPmlktitFTpyvDVZ0UhDbqbXb4EeQObf6rjGRL21NnmeV0B1DW6pdXbKKPzdCTx+38rMsBq+LOpeAD/yGHCMl9Mla8IjC+UrkELHgmuB2je+A8aEiQayPLsL8XoRN515BYwsdsZMg2zZ5BEJLsrEl+gvj0aWENqvbt8I0u+9axmQrmg6Mw0t9sftYqrAB/6t/2R/NTcpk7OA9e/kziVVqH0e1OmQV/+a/QXRB9DKxKUHVCOKoi3S0Vl8neMyyxRl2X3OXJzl4Q4TTj4suBwrWjjyFwq6Auosxd6RFT4TUCBFlKl+MYjU4xXFZ5QveyTjlZifJSUngbCjBbVgrSiIDDQOpChGBHOtDiEiSHfdPFDA9QwieUAf7h4o+fgbUImP4oWYb9rGC1KlfEXFYZwjvP1mGY1Z2nExFOR5RC6DHgjgFofAoNSUy9AOVRupOhSC7JkwaFrOzRQ4/k7SEgw6YWlRa5zNDtUATX4CsvKnpclaMZbwewHpMlsUxUz0IqbmpqEAakiHAYjy8FernGNqL2myBGLV0l6zfFUICwmFLyFRDEQyDmgLtXhKDl2m4wUeqxhHMtoxTIRzUWVO+CXSbGL0E4RSfSwSUqcgMhOgRIA8DrNlHcmD/a9mf9liv6ZGBmjVoKmOs2ahDzIobEYNv5zduiWKg9vcQ69VEvL64I7IwxRIQpDv0p2a5g2GARyZKS4lkCGHwP5DRUz/mW3Altrp1vR+k2fTkMEjXfPgr75duukGw07grqL20Bq+3Cbqnb8NlBQk/1JIyc22FIrUae8eng5HV9coXmPynMP9FPvnD4ptaMS9MpRaw9REQgY+mzhurRIi6YoeMCMI/msvHuK0YJNJYOY4zIvAH8QvbxOF86MKd/eTMaHV5fTK2virPfAiMnaKLyW4Yg+W0LdsE2CuISkrxHuj1DHdKqTA54JDDKQLByD//Kykk8qJtPS91CSI99leXRD4t2gzdWpOHy4sn4azv4GQ9tysjdJlzecQI5pbHk/y/BftO2R2CUjEfciuokObQZfYzCgxHkdubvg1wFLbThU5uk+snQ5XBD1diD3hMiOp8Ngl2cbKfcpcFQrsKUrQKDLy2MMqw8nYZquyF0NbccLDZoyT1NHGCiJp25kK1W9ZXmG733fGg5onUPxnwqwEcgzIU28dLK7nI/E7CpYFY0/DOqctq9S3aNsQya6o41/01rcf62j7twDsygTX7dkZsBMe6tjq5Bj7tRvR86MjdToJLpX2+jiyZc+xtnMKdyNEBTwl3EwTjdJhYKfDOwLZHnuiuKDsWqQw+byvXLpkVMK+9nPHRe15S5ffcCFRn7kw2iWcTSXJclBEITGi8eAafgPjGrpeS0CxbAETYHKm0/tuSPm79vkI//yUTL4r1YcII7ZpIYI5VcfX1njV6F9l2iEl9uRYkbzBtcKywWrm1hFzBzwNBAPpTDitkbNA8ZFbUryn8MCbuNUvht03IQO6JfAtqGUBAnzSVdV/Rvypfstkz2I+L7VKS8unWuAUk2hZe0xGlcMbLiStl4dbhmUTWbI/6W3ScktYvzcs0YHL6DymGzz+PXHF8++R39TcnNgR1/QtwuKI2Uvc/kSJ51+sscuCI4xfFf2NmP4loABXW/FWLZNcm9BL0v3fVxrjR7gCWUtsBOQcLjdbatZ4FMbYKu3quc7jJ2VeT5s8VF9D9rHRCUIQNcRacCXHqfbwIlRQk5QHOxHkCXbb1bqTV/4HHy5kvuyrA//0nqBiF43BNN4DLxglexASeNH+PtAwX9ltBZrMc27nkU1/vBqcUcnV5fGcP46qohJnS9ioiVHuZhAowQF40kaV1lCB2ai0Od8IooB/qS0Fw9IhY4F9qHlP5KMxjCUygXZkJEfz3V0KJGvgZYxXPYvjSFSOUjikDMzkLxA3/ZR9cE8aXJtXrELhi1nsNc2O2B03ttSBj5zOIWK1TDviRto+UaxZske5kYKYSAlszHyEPIhGE1PeXk2kGfXEi1CNqP6/I8yb4HUuFNCFKhWGciOVRKzzQMwmnmmcwr3ISmzZMenETGKQmzCQpPITh1S+dahS629Tbgz6UsCkuOw5e7CLYcYxcQ+X9gBub9wF/hTdSLlRo2+FuxtgdJFXDl+MRYzlKS4E3i6gAfZj3hoxvJA+AwWV3hIA+KBBHJeZS0akG2Ndtihr23mdkfjox3Rn8lEwYftSi6SU0oihUm+hWgj9ZG1rM7ZKYj50QZtD4gV+a6hfhmuysoNkgi9M3WQ3o/py/SHZz+zeKHEV32e0INlDJaGKKAiwDTN8b00R8uMNIlrklbV4Ksld5zqUB0SqPZsdJako1cAUj2BQE/vqku2mj/QXwCokJemEnPha6P6sL/MVUdRBEpI+E/tMy+5rmZej8ZHmid8nY3onwsE8A+g0lQz4HojYT1dHPFbT3u3/olpo16HstshsnhJJ44eLvRwCO+XrGoKmbgDJjSN0aTAUy3XjnlBy7W73/1cTYbCVJgLa+SfMm57kLK1/19DVGcPx4Fyd0he1caE9gLThMiZ6bGpoA/FvCfVhOv9Ry4+xIjJNbkK0VzljvlzseW4e2BjSbhtloLcXDlPUFQLK8HHU0Fx2y84MdoXFLFAyWhaDT5r6vzvKqkVEJGGEkXFecLuNJhBDg1Jog2YyS/lNAhyd5JdpyBiXHo/vgx//OW78PXLlz+9+vkFOXt7MyM8OEq1xiXKGoIEAVNoJKXOOIJwvlcx+27xQ7SqrPwlGM8EIg2+Dflx4JqxqEelO7qfpG85Ob04HyjxmvRLw99IhvIdaA9mg7hqg65lCrCuir7tSIWki7J0SOeHodupdAZTWi4/+rbnWcs+Ry5Y4fLA543IATuufKAphtPpVP3fakEnOgrp8Fc3NbUR1YKNQzzQoo0+YF6v6Y2Bx+9BWoXRdQScdLlL/KOGtOPCrOvIdhNxikLO8rUkVQP2gUraRD5TO8m1UcjNMCpS9s6ZomQ88glbmlk3GgAHJtAyWLMwbIU54LuQveoWdu2Nro3nGAUovSkIBnZDAdUK96CZloeWgwqFbkKDf/nyK6nvy3+ccieXEQ0hHUwZORCjRCP7ai/miaa/WdjevJ+8nGjhNitJS6MTbYlqD30qS3lNJQm8WCUrmmBGM7bK7wHfQS/7tAUt+FwJ1UE582glokMm/8SYJ868jcRRhLC6o6xxdNamekLFsvLvqStN4xU2kmWBTpelBH/jV4KmpR5DjaFG7ioR/dQKoaShm4sC9GlQso8XhY+q9A8XxYjYUCgPZLq1ZP5MTjUS1jVMcx6T4Br8ntSadv/txft3P6hQkE9qRWazUQ3ZnG/guioQPwirIlkhgXTybGFswirErojNhSonQxdn7aB+nZPJsImJjgzFn2SJ8MxLtEYMjuMmC0YaSBWGwyKWek3RwhxB1B9N1ycQPWe/X8V27ZSGloXHyYHWEv8kBKxVdeeP3/MMIpS+BfHw2DLIoQTJR5h8bsF3+mnVXG9YrTAFguf05HcKBogTIaXHlJxbv2gJQTbCGDmosoQjJT3tDoEgh7xnP/0kKC2SScQJbOb5m19ceYjjoWR6WgDD7skHGIfSwI3561oj4KpIH4wIgJPFPL0daGUEkhFQwUJjr0qxcZOkm23NsltLzeoPe7K6l5HkdC5Nu9BTftVWofOQUBsVOSumFdglo89VKsgvqKMvBI8f+8fD4vLQRkn0WKZDB4zJlCZddIdMP/qexA9Q4KzYTTfo2xqANDx6VyoA3A2/3adVReaTJkvUIBbu+tLycGMhQuxL6EbSfdPOU9A63kTjiuMHgKgKkDuWFwnCeYLTmhM32VMM5EI2waiLHg57zKSIsfLySzual8gaj5JGx/kecb67ZFE2ETZZA8p8uEWmgJH4Htu7fAX/qFvVTKpTw7xuVbo7BqTZas5zt7hJALZwlYuACJjJutql/y4tGXV0o0WvKneC6GN+N0WojnRHuBliwR7Vo61FmsK9EeTn8ypKPHcCcJI3NPR8ntVJVtF3tmVzAhi0pazxJgLS6HOdLucIshB08WiJjACofBZWh/0y31V9BU0qNyyqog+zkGJkq/ZORI2JKlYl2R2sJJN/SrpF7Zxt50IaJVo5hVX2HocTO7TRYsr3SLecjW/hilM0rBPMxurM4TX9yNgX/fvJDOi0ENTiRqYfmxlpODRDMuX+xIJpJmVSzZ9gVLpQD7O6Z6k+F693MXvpYFpdUlt5nkJ02yHCX7Uy1wp2EcDE3BG+qVyBwyGjMsUKYURg8aVhZ1NdJyUztNu7jsO5yzHNTFxa83/VnzWl7ZB+gkef9kaXy+uMrt8eSAmlM+5O1w+k+5XvBXJ+6ZAikBywlO5Rlfwa5GW6wSPlqTfUVY9aH3Wb12e1OfN6Wxoes5OaAZ03Ae051/X7iC+v+6Uuj0Y6WuouSelKN6q6lm+6ciVxGkWRuJk6P67usrq9J7xVFeG5WwZ0VCjUcV30QRttYzeVV86CC0dVpbN/hxnKD8QDQbEpUwztXm2VmyNeXeD1ta916MUR9dnIHFWv8ESfdOQMH7r0WtrlcQxHnDhnMb3YQFnokPdRhRClKuLVw1PppZyomntoWJ8kxmxW+Xz4Fuu9SON/tft7swzMj0Iiyy56GnDzmbTzc3Sb/ZzcMU3+YVkWrwyRNHqNbsB0MUBUcQ5X536AUas5vi1AwP7ZTpCFZugGQbYFvCxg1VQ1Juo27IdTddGFFugu2moNM2Bx/g8rZBePXd2bBOzLDZpMW0VdVtKbkqd3Wp1VD+it71kZa1uJeYyn5NFcPOe5VKrsyd1UuXQPxRJj4J6SyEWBy6oApW1xs+V2kZyNTgDc4Pjp15+bIRvZ/mfnyWRHc3PE3NogkLtplXBK/8nRC1jOmio7J0fPfHDQlzMn90XZed3rXUywnTHrt27I8FrnGyodrVoRIkLqpUwZIZ9CTMvXqi5l2A4BtwShufINPCooDzvRt5ae2qrftvIM++JleR4P2WoLTAg0WPTgw/f/hbw6XYEsss1jfTbpDtl4MMrk+MZh20TOPMsOV9ZpOaXhdfMWSn+64/kKySkZeyH98/LqRL4dap79es9ACvf2GSePId9D08r5h60T28Re5vdoHVjI0jfwq7rSxrrP5i5Iq6pZoqSNFc5xEPdeYPpSjY0EVEbozFFp2kGbDD/kM9TNsqY7FOM+sNAJx71jp3W9TvtmnbtzAFeExIUdpNEWfFJxYIehDmzD3kyBFGaG7VGut6Hi7cj15MpxErhv+g6bjsyzAlO5PvfkEVJpl6pTQ/i0qewZn0pxgcPAk0Cf1CqdGbxvKmRSQKn73MqoN4D2TmGjm9RFlm/lB+KOuxmBzpg0eUVUK98MCJxLOpm3kgZ1wwNog5kAEZs+IcFqOdN+4tGyOceXx3it1rVyofuXx21svjQQuETuDFiKxuvNHHDLUWqbouGjdjojlBdf4Ut54qqzrsAnDy8BIicPc42RlQrcu2tPaYvpUIc0CaGsBnqh6rc9Q+gDIOdAF3amanS0cXXeek7bquypph2Q3UNV1Xjvx1Yzx2b1F+OwymfwmNTmqXWH0yoqBF3iEANuc0oceZAs72/qu7fJyVShsUqLG9LThxrKrNAnHX7MPpZFU2vZQxug/UtZ7WokVzpkG0+18OhqhrpsMmlMa50gwMxTnv9FjwmcLNQqlzAuEWdsABKmoa6c7DUE3aVH/4ZpXHlXGLdQJBQj9K3q6x6W3NVZ5QUScoyc9UEJb/I2ji8qcSubvxuzZkvrjp7IlOHaZFQgp9PB0cAnj9Vc3K80BEU8eXAtfi8H3KpyZRYIMRy3ZgVo+CHxe2aoQ/a66+B3O8W6i57WRoOOSUDDtTgGrluJwpdpldFTWldyX9uVhpdegYl/Yun2BqvettBodMc7w9oRfiPxYETXgzVFjzcySZLaCdwisMoJy8z0wM0ngOY7N2HKyPhotXMiDjvC6eUVogPDZR16KLGlHV5IwoWUUwDZyeXQZ0CGXbnxXrn0WVtA0imQMAAUZI/kI0wMbgfu0sCPU6z8w828mSwo+tSFsVOfPuNhONneP8H7g89gF2pmbFLGs8y+Fl2piGdeWqLUxnZsUXLqevJOS59/1a0ynPmymZG2740cy92wk8Glm078UonMaxNcbQfJ2VHY6gDbmOV6k2P0Zh4/kQpGicfaHmhiHG5lW3d9tu646oYAWrBdFlc8qCMxf9Z2VbkF1CFOthkeSdTS7RKRXrU+7G292xbSE7WY0Z583PAVu4HIjqQPbdWt/ZET6avaimSq56pchRjuiizLh0kC/P6/2HrQZNU/myQBcW98Mew2a479dEvFh9BwXHfYXQtF/9h7+pGHMN1NJNG5Z8T2cLE4+p79zjkNZv3mGylqRfXxM4jjX3iWT36WQ8eIS4rcC5KPBWig/njGKUY0SozEeDa8py2kKf+eluTc/unGTnyK0/1iPBudeSDTfYPhvtZ6aiVR53k1BNr/OBJt1J31oJSMUNJ7E4PUWfzAzkBjFBOLD1LmWXfLd5K0aNJrR0Ydbd+i/n3JbM/syw23OqOz/gS5Z/Z2j2HHTjnl8PnPFjYrO3GcJnMsHNmdKI8pQy93oOwcV8fgsCtJaUOj831xDvdZe60QzmMyiOKCtxYcdxitWWEQE0vat9bU3HnHd1e/xZkX/nI6sodKGYC0AaRrU+m1hnRQ93SjrlnmaIsugp5u0jHtHGnzeOC4G/qHh1CnciX0URidNmFk3WJtycKdfNdnUC7p7WJC5SvMDEGn+P6QUoidtLHd70WhHbu7FxJP5JAxWeEyzWR6xqpKN3g0RbYQHocVibqM6NAqWW3PPT6XibwXp8bZNy2y3re92oaM4vvTM7OW2RWdTabsDdD7LXd0177BT2vE3UTvdDa8KhNkSZyQ/MyJkhGKePLfG7M4sg/j9YlGD39B1UVlDOntWys0/klSm9hX6h1TII6X+AjyWh2ttr70dzpekvjxPWWsZBKnCzo06r7CLvm5r7Rch6Pf5XodL+CQpONz75KqTxeLek6i2kljEEccqzBdOS7NtiP6Pjd4gglc8g3m7DOhso5J7YQZ4YSW7O5fZNt04Xw37vyks4rUfKgqYdKpwnbSEqph8OpUNUIqS8JsyWRHhEqMXMY+XJQ81Y+8Z9PWxFx57J6eWvh8qquPvU6uXWeXh61Bk7Lk028LCIAh0AHEPmgGPeLhw/aouCl++IS2uuNQ7bJ+G5bJptlFcrYsiYCv/CXNtXOK91EaWTkPE03mMSbT51D/Q9QA48Q8DRxPjbzyOs13aN9nsylQQvuyUWpc3vzXwjW51Vo2lpaKTvftJjRCNyU7bQny81JF6zyEF/7H07N6pNub6Np1A9Ocx4dG+5zUe+nOiR7IPnjN049zj7DNmAQFbhqIPxGquPZUSL4TdRzPQfbGbEe+BmF4N9LufeL2BDB397mV6hM4lToV+kAJd6rSKVGmSjQWWg7OjjJmX5uhs3ISJQDSiDHBZa2FhJ6MGSOVRsNK0Cpr6Tdu2gOVfcFtDKiLVQ32sdWszrxgvZOgwlpz8lRKtomjl4kzzoRKcxj0CLB6BKHYLt/mbdDPQEerVau0OAQo0kY7FXOl4Sry3UEywfQ6RW/eBR/lblaxb/XowDdsZdw0TeEInaI4LbJlZ9DWe53W1ZNaNV6LmRcHdcLEM3lTgqBMnjCWpwiQtqiO5q2z09aBLsjXAA7mglxwMgpkx2HVrNHrl17wb1hUL6j3haRFulrAXZP9Wk80eZzEgLyVjyBgYukK8+lEMNupvP+dHCSzenExEnxfRhapYzPs6zfM2tP2T7FSlOSVdvjVsIxoBGpO9lGa0V1jdpZydp6R15m5qG3fV9a+qOxz8Za95pFUAz3BfvQd0Ei9ZY4e8gCiZEoSpnR3gGKgPyUy3fhNHHQvJjIpVPnerkHXYcm5v2vQckrq/2aiS+y3MmkCXYc8sCIlLW8qddPsffd+UN4Z+4TmnvsDq05t6a9uYhJ0/hlySHUDeIYBncY4zutuCb7Owe+t2+qCjjeloyknuF8L5yo6fUkJHUq519Edd6EaAyXxnB7sTPvtDPz3Zd6nU8He/PsqNTdnlg7WsJlgF/ml99s7vGrBys0pu5uRjx9Xw2uC7Bt6es5HdGhF7YR8z6ZBAFUxL37NuTHGgGN4BAwfqXGZYYx+60s+unmXrdcVJmSkS/4w/mh9JJHjsduIOi+PhQOp3mx6D/s6y/8JJPLl4ynnGWI9ZB9lTbQL8To1X1+9Zm41DMweCvYfKHtdBGpLXcn0b4SmYf7B8rOHFtN1UrlhrqYZdGO3gpVV6YB2oxq/00bPXuBlfAkz+YK+qbV8Td0AlUWsPTB0FElot0ep4HLOIbKGvbIVzs3pMhUgBj42RUj/Im6DTBUUSbkOSU7F61NkEi/MswIFOOEKzaEipP+Iyj2FNU4o4Zw0cMoDpUnXCEWeN08pKYQSjaVoi5YUlezF6jag02+O5uJcovImERnooW/cwD5w4Zgyt3vGn1lSY86CwFMn59aK6rnonwe8JdmeLZNkMbm5d/5I+0YjtTUg2tJuK0d7dvoZyHAjLXT0XNsFM2ltGnlBN14AZ2xOnDfVvRvugV1LpbvvSc9Gqfz2fo+QzKelmAc2lUYH5P5u3w8o2Zcvv6I4OXAiqEwnZ8q+LXm6I7upjdEWnRx/YhmFPBeeiQV0ODNJBKFKpQgFW8kYrbKGf4fsVQyl5dpLV+YTLN+O9zX1OqHAdnccSTpvOap9SA7kKaIvOV733FLMCTaJJx6L1b4GCYq3vAo2My5Z9jice2sN4PbrnuLkEd8pjtdTdKtb7jN84e6cOKP1lg6tWcmeMwOzPloSOXy1ccYSnAzyIFDmyVkxSQj4Xgh0nlNvbFCAwoR65+qi+o2daF/jucRo7trZDv3FdbOd3TM5AjzvRXcDIVj2FrWK6+MSdmcn90m2bEjvXU3yLbtLBMz4gJRd6eA9CfXwpB/oG6ZsJCYgNStKY2i614lP0NWQen8pM77r9N2Y3ZvlIArYVPE3wBboJoG8BBnGzgifJdcINaa+12nkaQxPWSLTOeXRfpFBRXk7GkD4O02xQxUkzCHIZOQF+b0zKIz4NZkq0MkSve/inC+xIYuEPF3BNGKglsnsOngu1SvO0ymr2S3qvUN70AMIirjX3vSkDZA7qAjcPWQwRVr8qcxx878qp0/WuIJ+whToKo90T20lBag7gbHuMbu710XbFv/pqXMyg6i6KkOhVxGcLD7qS75wOucoNXpfoU9p11qW/puNzGJdylNVHb9IXPleaLr3MJ1KAmH8o+iMoafOgwcSXV1ncAsbvLIhtLm8UteHsGCER4gmjMT73pLpkIjSVfCn6N48+HJ9Jyrhd8rdyWIYkyfte9aGW++aamsnBe6Hh2RfA4sR5B6Q6DQPLqD3PdCfVb6LFZOAgppdzINHCN9Tff/arZGM7u6FR+lQmXvjUAtMDq03+Q1M4hwQqB0aPEcnxoQs2ESKxjKrn5ram3S3o0tmVUx8+27BNpgGVMug5agpIy2cuUOCF+keI57dZIzSCsdp8RpQA6owZvuP/UqlgJOT80BY4uYD8cjyq9UG6m5URAY9U3Zm083AuD7qnh/ad36p9GXQt1VkYoM1MA4GaH30PvcAJFyvR1O8t0A2QCazMX7C9+OeAsNTy732fitlCgqE5vIWOru7EreAlFYb8y8RP/8DNocB9m5yawF7hzi7cbcGK2bxwjtr7d1stkglSmR6rHdkzkUnlAbBXIz+sBWL3dXkjRLP+s0RRXxta+K3OjvtfPoovqOGbvHPXXATXXvOXR79WvlxzZyNlcYhL5KpHY5phXzOmLBziuVm3PH0VfqUcfNlZO6E0Vm1zIVI+oSgxzENO1eeAxnzPgwqtEXB7pVZ7ZNSuxEltZvz5cI5TrYq843mi/apAd2NhmEXSUYjpeaH96l4usGR0FlkrzEh8aLnWjN5i7S870s+uVdyWY7tFAe8oFvDkY5QC6i2tjnKEHDNFGgHWsKwzCXgVAKDbXyKGrOa/WzRleTRyY6A+BbIXDCd9eHi6QMxwzknDMKYJx+kU8w+Qdc8RYDN8dyo3Isuy6SboWH7LW4NxHh4hp1CcfxnHnz1H3fH4kSY/Rs7SEsc6GQc6LuUm+tcnoiTwpgZ2YGdbvPeWfNeL2V+S5XJVCe8lIA794zK1EBu3kzrZPfkQQwdvRDFsQjGms9e7PMPia62JoeXKNCOpCmCJkL5wpq/bI0bCxrEX6Y9Z4Z9/KYkbaoiLUjcC71ADQVTdhNe+h2N1IVr1qJH5kipyUBF+qDSkLmG1HuS1rGE9o9nvwrMagZ6mzxjb9v4dpGdNPHUYZLxImG545Jlz6teB6oerQmZhceaXI/UjnZvj60uMjZbnVL0FN5oFm0p8idsj7098pa0VCJ7XY8X1/1ZxY/ZDNpnTUxaoa7z3FNeKxhEG3R2V9Rf8UW3xoMHvNWO6Rvd5VMX7HA966bfMwVOlp5AYMreRbjVOTVDdqeFXiQ6rmndLCqI6Jbc4xwU4+XTKWYkx2OxMOTbB0M8wwxDGcnqZLFg/H93qABhQaiofTruHCrTKOWl8F/RmQftjpF4/U7+MAc9I2cDjez7Vp5jMt/4DT/R52EnuYWaixdv375+S7rVikYIOL3A23uqGnag5YDYgXlmpaT4fzsI4TVkqwAA"
  },
  "styletts2_samples.py": {
    "sha256": "6289d800a2cf4878533bd5adfea2ea321b704ec24bb1ae6223bc5108359ee3f0",
    "gzip_base64": "H4sIAAAAAAAC/71965LbRrPYfz7FfFBSBiQSS3Iv2qVMn5Il2VaOdSlJtiuhWTggMSTxCQRoAFyJ3mzqS37nBZJUKi/ipzl5AL9CunsumMGFu3JVzpZKJAczPT09PT3dPd2DB3872Rf5ySJOT3h6zXaHcpOlpz3Hcd6XYRqFSZZy9r48JPzDh/dj9mO8yGP4xopwu0t4wT7F5YYtw+WGR4x/Dpcli/Z5WMZZynZJmBZ+r/eWYLJTfzQcwH9jlvNltt3yNOKRz94syjBOWbnhLFut4mUcJqzI9vmSs7Bk79/89O7Zi+Ddi59fvn/55vWkx9g6LtmS0NqU5a6YnJxAyWa/8AHoySE5O398daIR9rG2/iVbD54ZIwLMlx+zfcnOlzxaPh4tT09PV5fR5eLy/Hy0DM+vTh9fLqLl8nG4OltenY+vei/TogyThFDO+S4r4jLLD/D1t32ccxhXWfTZDobMt/HvPO8zXux4+HGQrvusWMa7Qx8HmEarOOH93ma/XsfpehUueQCjYEB0tg3L5QYKGQBebk7o/3AfxZnPXhAstt0XJVtwFsXFMrvmebhIOFscelW3TxjQ/Jc4jbJPBSt4yd7+8Ob1i1cv/9OLd8GL929fPP3n4MeX3757+u4/Qi8sLguWxAuNKXv+448+kEvwAyuQXGVZjAM57/7uwAYDHD3zK2pDUck/lwMcGduFRRGuuV9+LtmvAAr/sMmK5zxd8gENiOnf/qfwGp4Dn/R6PxXAWUje1T5JDgyGkCZZGNEgNZPUp3qgeXOZpat4jZTs0ezusjgtkZt+efHy+x8+aG7y2YdNXDCjzh47/iFexd8/haevgY+vocd9uduXPag5PmMff/j9CTD6LstL4HiiBk4UsDy0DHOYEV7yfBuncVHGSxiequKzn7MYmRqwInICmlsYYZ5tKyIwQRTkgTDZbcKTBS/DE1E9ilerfYELC8ZX5lkCa+v9jgMWgFm4g07gK0zlLudRvETsBDNwvSIL4JhVBjjmyH6E04cNdAwAemkGTLILcRyAThITrfk2o5WchAuesCxnCf8cL4H2fAvIFdDv07cvGa1exCEKdzB2WPEvSBBIRBBACmgW7NsX7z70WX00OFic60puiHYZrJwF0D3KgLJpVrJ8n7LvhiCmeL4+mNABMQQADITD28IULLMIEIHR7UFChVAEta9j/qmXwPjS5QFJtuVhsQcghEAsutjBbMSFoCOsrpSH+SDG5Q68gajCqt+EeQQgud8D2kkxGIlBwbJXgxjAHEUxfkO5mAI+yCXUU1ryNc+NSQFgPSlBoY9nb3+iejnfIyJA3OKQwvCA2iBJrnnKPm3gP1wpgG20XwJ1kJjIQKXgrOJEA/ehIgdOBInCB1kKi+lTHpe0umDIKKGf0KKEvgb4Cxl2v4XncSkGGPFVuE9Ktoo/Azbnw4EYMQOhDRIwj7dIOkt4Ax35Iss+fgXw90nBe5+y/GNILPeEBBEICWg8wMYDtY0MkeI511sBrA0oL7maS7kI/d7rTKwRJDBMZTmI+A53EpggBLhFMmcVdUEUlSWW7cKIpgCwRbr6vWebffqxQMYA0lX9FRzBAlFPlkmI4gBw3sBiB9ZP1+VmAPULnl8jKB6tYf2GES96wEYwEbAtxUuACZLm7yBOgPjv9MKGKVjk/ASmrMiiA0mKVXidIf8tDorKsHRw+iVFZeGEgTTP2Pfw4Od3T1/57NlPz5+yNQeJHaIoQHSEoHp7+IA7BbJptgxhAQFb7cSGongdxXtB6woa71NolRcwZ2UGOxqMLYOBF+ycff8tW+9BGAAtYFRhCnPag/WwhcHCulgiYQQasGfkYR6j7BOdkrgsYqhXboAOGhXY6Hc4IWHSU2vm9c+vfoTJgD2CJFhSZLSdomD12VNY1gscGhJoD5Qu2XWcJUJCrIB/QPqBktLrkQANgtW+hAEGAYu3CIIRzmIN9HqqLF+DhAOOlL9BhG1gAOrn34ssVd9hvWzU96xQ33LdtNgvJLV0yUF/LTc5D5HbdEG85QLRHcCFLhWWb7EbelAedsSeovxpeuj1aroPmzLnvjqK06ttddj4NAwfL8LHq8vV+PE5X45OoyUfjy4ux+HF6OzqbHl6Pry84uPQ6b1++uHlz9Dz0w8voOH4bDgcKmx+ePr+hxfvofSGtnRnC6szQXXAmTAnulqdny2uLs9HZ9HobBxF4+HFxenp6PLx5Vl4cXo5PF+MF49Hl6DrPA7PzhaPTy/PTkfLi2i4OruMwpXTF0BfgVxDMbaBnXgdphL6aMmvhuNwFfKzy8vx8ipaLa5WlyH2NuJ8cRbyU34R8cdDqMSvxqMxXy3OsOPLaDgMo8cK+k8lcM/J2x9xOzrZww8Jf3XFz3h0dRmNV8Pz1fDqYgSAw3B8MV5dPo4uLx8vRhcX5xz6Da+uzi/O+OlFdMaXi/EouoIBnQL8216vB8uWEZsFwG+FC/9dT2CdFOWsKPM5+8/sNequU/rw2OAbzZb+6xAk7w6IMyFEd1BJP3uar/eoWr7FX7kLQmeZx7SipkEQZcsg6DPc/MIStuAAZFdRTHXjd+Gn51WDH3iy+05V9URXPohHxJf6cKW2BrQaDIQy5fSRQ/kUGbbPNgBh6vyH98BX2eLvHPb6bKVRBbkFG04qVmqKQ3rCnv34EnaMj5zkLfAwikSHehH9r2GoAoctrGMQGoeAf14moCBc82ANe8fOlRVtRB2hcDrHHpI2auLfOmRHaLP3q2epsPdpUqmYbYRUkpGhrEz0TjpIwgMaJlqv9Xdoa1WgunqjNXlk4jr6YzUF2j9sk84+UgUYEOmrrWo66qpegALY2uJs3NUkAo1tiROnqjrL3d5RY4DvuM8v91E4ed2JpdAaBlGc23RQIPGX6zSsG8frHAdVGKCa3DocQ3J2gdjh7jnYFqr9CkwbA8Kwk4dAdShQ3fjStuZqPt7+XFH3x4ays+M5cDGoTIba02dD3x8PGcAz1nILP4afB9d5uB2sF3d1/JwvQVdNQAHpwx78Od7ut+y8c3otzRZAh2QPVHLv2yxLeJgqfn9Kj6tevwOdg99Jsko5bltLT7WqGWAVWKJKLY+EbyRG3ZSMJq3HoxaMSukdZCt2xqKpEWx0J9pkP3bxiT9S+P/C4/WG5LdSy8JltifbVRhqqCGCFVWZqLNhfzQ/grmJBJqvnTicduMg1OS/jgOKD2VgDoqS74p2cdW52Ph2wclgGBQwdfwvzwLhT7ZKnO7j8iB1EPwzAOqyijiPq0JBpLdowcLM2CYn+yRoB4T+xMEsXAqrRtCnf5RAu7hcbo4PT03QK/gV75KD4Vv4bqicCWShAaWcY8uf8BqAJRi1z8Tp+d2ty+wjT9ubg4Z651RUgAqO1nlxz7kYD+tT8Y4LnQfYEmiSZCgeYe2jqS/pXydN/37LpWEXtw72fKjm5TsyyivRTPY49Gza4GAcCzv8+HIB/MHiWyMrLDfoqCqmrsPTwR5xwC8guz1jLxaPuqB9Cq8HYCIh7hbAt89eBaMLhIjfxmf47bsf3zz9YMGWz+4kF/kqk3gbl50y5upS0eotuk2XPCZ33XUcMvLLlftI+kVQG073odgh3Lp8+Zhmn6A8ID1VaPdUVOn4ol68ElV9oUJNNLpK5AMAtDN99GcWrlnXR6MxQI3VVQ6jqbMvV4NLpY7gn9jiEMyN+Oqjuj2R5TT58mucAqqBqg+IGfXJ2QUVXAeJg7MgtUXvtuoJTPdPwF5TxN2VcCpEACABKYR/bMldNUSgP8gID9UzbKmKwdZRMCuyiBnmeZ7lrvNM+G3Jt40iE51BNDdg39Ds4+4kGaFpZTgVbrQEoKzPrsNkz8mJJrHw45JvC9ezUZAkmyryzrD13KoC4zXGGkpVwtW6RvC+hCX/Id9zpWYcV0O8Ggr4B9jGSHCjIxpAn8FiTjyrPk+qKUXeRx8KiIkvAQrVPXIP1Sbyi/qkJfclvbokzKiZ96XdF/xLegJ726vPIfYlWuiV4kv5RLgIhpHLw37e7Fpx7sp5mQqgZBMjaU4EIOTEG2Smv+W3BoMqLsZ1XBwKHwXIbDSZ0zKF70hYchPgkKnEXHdhehBDZNMpUyYwjocKfSBCXhaofCoTeOp4hIleDbr/GhOqVTJzsFlAlvNc+iruwkDY2cfQoBp/EZcmGjsfagRS1Bfuw4da0FCV0BDVNSGtxANJwGpDEfa/+Vuqm4G086tHpj1flZLdHSi7uyoXVmhAVmhVWlHY6hOMjUAYG2L7mZh0X4M+XZa5G/ZpAJ46t0DCNKhn1uyTq9Gtt/f0ruWGPiKkGM/DiZVlhKR+MOk15Pb7/Q6VQzr+hU/kW9DlxZwjP5j8oTtEtEOfThCFwkK/LIq3dCXPHHGlNs8S0akuT2Ej2VG6B0U+x2VWTXTogyoYoEEarBd9o5gMLrOAbCizAO0Z87c2EwLSoy1gqJ4HleZvPiPVu9mEPAPBtjDLtNVfK28pQnUoIHWoXxsraYeBVHtNXVTNQ5K46PH242IVp7BDisVdW6iSlm0c8BpWcI7mmtJy1Pm0AGfP+pB9bc8B+3rKzkke+yk8G4mvNBv4aCh+G0Sj0hY03onJpx4MpwPBB45n30wZ2DQC8DdQC9nIMIOwzGkQZojNr/G/kaAHiQ3FHIopWmbca6OUbIVtqPu6gagpB50IK04j1GA3gzja2g3I2oXxj9WyGmE1kw/ICsPCy2M0rFnASJs+qxnVSNCxjFiwbDw28v3LGi1HF3VEhEFHszMcdqIr2RaLT48hXDcUoUPfR8A2fgocYHhaw/ByOByK3oX5FdBpOE781Xg4lJRGGy1AQ0s5C4HWx9AyPIbUge9LaHTu3TgIBYoOzSlXMgF7EQiYEkGVukZFCh6warXx4Vs63KTKumq1bNMMrNm1OFWkKnScBipZucHHPMXwgKixrKeVYMIf4za6SA9kjc/Hw/kTYUkLRKBDxvOwwOdxGYfJCUgS9KHB2l1u2uRJJf1oxlp6rsxFs3PXWGQ5L/c5aCTyFId2+UAEI7m470xoF6VDm2rHxRPzpERFTp8J+vk+rfabmbMmC9UZPHNIKSVYYPDCbn89IA0Fn/7w4ulzZw7Wcrijo0yhNUzRogDrFrU48ZWwou+2IBd4gBSKoCV85PHO9djfpm0xVOIvD2Mg8Lt9iueTL6QmWwXzyCgsRauwZDc1UEqlRbGY84T4pY/RMZx8REBd69SwaXtF8Rqt0ak6ivWLTTg+v3CJQuxEA/WEcbw4lBzao9aQhDAnC+fX/Fd0y8KXFKxkf8M/C5CuZajKboAWCjdbWWolxCswwVcxj07iFKMDAA8MidFRKhhkIGMTBKEm7Eahq+iy3ITpWpjQXbxh4SEYxS5CprE1O81AdkWUzY3GAxBCvAGTWK1RtV5SHe3WHsjj2Xox6Yp4ntrWxjx0NZ7Nq6+Cr8knX1NRJBl9sTyR5kd52MG4lV1M/rg4VS5jyc3AaPLYa0vzuxQxAnLC0BbbkSKEYRUlyAZNbui9yJJr7qLS3OvRCSt7xbdZfniVpRgCKJBCwREEKLWCwC14suqLKL4JHuqjawpPtSYIts9IGIECNJHmsOkuSlY+teuL76Kd/CGE2JTJCvQha7jiwyPr3lUdsIdsxK88G3xRZrsAI4pKCVbELSBcFcDgv8DHLkCzTC+qnV5vkwBlqmy9yrUXBSv3G0ajMQ7hPUDbEc/0aouRZvlow79hw30UOkppEBTAIlBhg/AatlXcoepuHsEsP6NCK1nlDbqX8CyRdk3VTh4ryskScX0FGDkUm2KY8Ua3whalGTDQNauWwHNTs8laNwlANuyA32KQbmZrn1oFW+IyyzFHMTWBDIyZsm2culTXZBB0uCETDP3H5+bko15RB2Er1O20wugjFYtDm0LCQ9APADrGC+EeQCFB6BbfAPs4NXYz+KPR/YmgTxdhgTaBFJ+SGIGC5lqwrcXSOk94olkS2ypIBZjERcek5QebKDI8Z3dA5u/ZXkwqo1XxEha/a3ueQIJFqN5N2cyo+Jz6+56XP9Djbw8v04h/dmNheMW4geYo+ty2Ns+yPS5Nb27jgfJHRk65NU3F/Pu0QaMe105NFuACQrdti3sS/yheq1+tBhgRCEmS2q31cSBi8DgaSYZ20K0Ub4OHrEDu7VaqbEFr4rAhpCC/3gqmgWUlevaOA1e+kDsr0YQDbDBKIxRjWYGrGX7Upr3rT4Scan8lgcKwxO93e7GlCOfsvWE1AZAd/fXX7GJ4J5B7EEXPPHs0bensXu1NltHaa9sf/7zkO7XIfIwSJAF0HEuMNu91zKfuuRuEvaOhRA0/u7VC4DxgdSJD+8zUV9KnMEbpOzz3er1GvZbN9gN9c8swB2aaykUMKkMIRE5J4/e64Ahvq8F9kogv6AMkoz30ByL4Mi5YpmKBhJV8QE1/n1Y74T5FDVflEfzy/Pkrv9egu1Z8ijTcFZusJNKR9MGjoBkpO6D+zCd/WR9gN856txeCm8wAZ4Kz4cgooQmT0UEOxmcHYt8U7iUVxwp1kIS3LTqW3GkO6XKTgy73e8cerkJcxZmY3Rj9FXJD0dXaoYiYmruAqFqdMITNeVNTwRGWIJLGQpNLl/Q7G6ledRtVUGtCi0IGIuid2ZqbmopoN6/2f7VxqxZ6I7cbCDXBBl7zdFK9Y5NvWBb4d2uyIq72ijzVqFmLCPDYNwYC9zAk36g4ZgzhFtNLyxNTIsTAyHTEGTXPiCTfiwfVElsmWcGr9WXv7nUJRBt5Y80JkTHplCUYvO7iCLI9xbgIbwieWAfCJnTDSUvUqmnoEG7lHvhjRmaPsH20LMBIWCkMpEp1CLdC/aNo6HpSlArYXuGvQOUDVfWlRVq5QaUHL9cB1c/fvhq/F2V99lxV0yX/HOZ5WLzHJAwAVAHe7kEmKCCv8Ef1rATWKjDmFg8YVD/JAvR4cZBd1STj1xfGr48Wsqr+DNTobCsa9Yxeicqq0mIfJ5Lyoo55zsTIWWoWAI/SWU8VtGJTzXU6M6dQfr6ivk9U0YkRC4oL4zpGuk1r4eWeYbAb6VSIm/HzTszsxfxFaIKFvtwUwTiNguFwOB5irOwRhHVPFuaKosiMwEErHhBaJnnvjtLAJKotJoGhtk8s4+OkPgfmdwWMmTwppJqFM/fk4hbepKkBwZeFvYZmqnKccOOshBqGmbvq0X5nuLEL765aH3meAlIFbH9mZQMZccARxVvDaSMHtMu53gbkqGZOkTvSv0N+SNeR8fwwL7MRbN7nfXbaZ+M5/BzDTyy66LMz+D0aX/aZGUt73OGjfO1WHpKOZpb5fCcAVMTjnYg8PrGO5BCkW0/KX1yPQRc7uPLY8qSWSmBGTd/NJtgFwDYlgGtKD/fhQwOLBssIIA/Y0/fv+uIMC/aUEndPPCQAXQ4zQ3PYoVLcZ4ULg2JpKKsMrSioHKON5FcyBfAxhI1bTX3fcOn0CXXpnFPuNeCeg33K6mAt82ibRkPE4NY5uE73MwsldzbOy4+1b3soWLblgd4prNg8MzKAVgkRwsfBuZ59FE/VZJyITYZJzR5PBHGNmCIRtFlodxCxVSUoMfR5F5A2gqJLarayUYCzaBgDFSz5beakvHTmjfF0IVkbi4RyD73mWSXZMZ0yLgrKYNIe1y1tyiL8xVRs0OGCCTA3gAesk212zTELL/7sOqKJ73gTVsXPQDUjkEuNkqipThQqZe4BKPnGtpxmdKrE8XxiDRPKc5H5aGfpLvaYP4mBlAXWScvk4BuWlDzOmJrz6NMoAuQ81w48A3Q1+1Clmn2NIYIwcJ5GMjhGn7QWPiWT4yluHBWOCI5CcHJq2g8vFJNRXzOoXsWtmfiS+mYgTV9JE4NfNePSbMeB8q7aoRJF/mYcjVzv9XVure/WxdpYpObiNGhnIlVmIIRNowimHYNNrExbkUMpj2jK8KCya5GLk0bKMaYH0+nALscU3UgwgFIjpw2d0bURM1RP/a0SNxLM1FRDXWOTLeL1NgwKqX5ObW3UFU+BW6dDH5QbDGEQJeHnKWyg+SabXhnAlgl00DxIwdlSmz3OmERbqgFrTKumRE2oYxgUwB4RsIlQDvoqjbaPWZ6WuCcPkYTui1az4bzf/txQNborya66K8iwqrZ6ls0jemMDNmYPdR7wIxoBaCUC1zvFXWPJYaoKsRaZAbBwMWk5vc6SvQhJ1YJnK4LnMUdW5BDrQKnFge5dEBg4Vg/WeXSN1RpsVRXU3AFGmBqdOBl7jNd22Kdi10Rls6hx2CgGr2IgyEOvE2H2aYwdOhfDobHl3ipmFOZkFetFy7/LpCQk5NmZeY6W7io7Ez4tW1LfrsFgORcr81F1lUZlbdGdHD4sKXSD6UxfqRXvsuQgrC68XqBPFz2g52ZFGp7biFkDK5ci0R060TsdY1ZS8ik8gGkSmbs2npxM6cPf8jB1w89xofJJ8DAfuwHuNHTgikVXQsubUpayv15Gbk6yvJF8ZvRkDcgVYzHTfE9OJFg5RP3b88MCR+SmO1+OyY6+8M/x9AhYm6B6oBabcDEW41wd0wEIHWFGlX0MsGoo9+ah09s8u8Zs8hDEGtBJBpQZqUDZivnnvg+dqLQOjR6hi3gXv+X0SZQmmtfGdHHmsYcP2djzPHSc88HFMZyqrH7QJDioDks89IcFXQDP6bzMLcc4IamLKFVP3OKiiwv/FU/e466eZ2tQtt3a0ibTbTqChQS2URoAyGJ6SV9XK8xMOQMz6VOcBkKwTEdjrLfJdqrgdDg0doAHeN1H86IGcY5XmF1isEeF71cYS4UhSzLYVgITR0gFBX/SnivvR6kmZ8JkBqG8DYBMDutUng7bFbmQo02iuUI7xkUapPvtTvCtv8zgmwpdFc1cVynSaxdm7xzEOzzw92nx257z37k7hIl9xM6QO888o3zkNXUJStsT8CqdGeWQxanKAyvdt8Bo0liwtBkX0PCkFPMbyg89xcyXeKvWfs3rR6KywEQVYUzifzJ4AAPtAuU/pFB+EIU6z1xg6jiYN4a6rbrXQl5rgWlBFKiJg8UgbKkIEtTBEo8W2QLFqI+XLMj1ZPSIq+TYEqHbNaiBdWK8gBknFfdaxYRSuGBf5k2hOhqSWYzj9GncGHwwm/eZYGTJc1TrayF0EIAxMZjvJA7ERa1HJtp9o4mlI2CrdnjEZCB0MBN1yhRMdBiDHuZiu4Eo9VBmju0TGjqtpDx7ysdxc2fm/+2f/u8//tuff/zXP//4P/PZr85X//qP//2v//gfv8692/nDf4cuK2fWfzL584///ucf/+vPP/5nS52WA7slKLxxRJcQTdksrp8fywH0aZyP2MgTAWN+wUPgXleiKHiqgOYDNpp7817LWVrVUfthmiB/VWs2GM1bKy5g//xoBP/g/PvhDi9zcR3mCCe0wIfIO4EHc8+yIolToNhcOAKQXDgUi4j5ygVJDqk6SJ2FFk2HIoHXn4Fcy/E2FDxTEGAHlPpMV27JcN1Y3oCGuiRFKaEclCo2LSK9fMSSEoyJzDPUeHh9c/PULCS3V2o3qR8KYGU7+NCQjOIBAIdi2OCSOCVdCrsEliWASkup7+kC3GxClebs4ZTgmI+EbJjoZ7PJRM1wm+gSAUh4dmLOgEzDk2FP9hwYhwbVeQHVm1dC7SklHNTz9uiaKX0JFh6AsLfPXsnbuvBynh3esaKnJMvjNQaVqiNfrS7g/MBHuBBso7aaNeajiRkc+UM9CthSLEi0tmzYlMUDbXq1KQR7BIH22Y1jNQDl2foNJjHWg2JR3ZFdQ4H8pi4eodBnMKtE7HT3QU21jdTObGgDsQ9tJdlJV06T8qOvOlHK8oec80WYfvwFVusH+Syv2lR30vmLcPkRJYRsKC6z+1YUGgcrOpBQ9wAlz1D/A7jCya5hwoxYYFyVyToNffW1r/WQYLdPl6VgFxlPi+sXbBBc64aCrsYINFiKjilsoWWguEEZ+LledVYntnmX5oLIXZEYZI9xGi5vakPhbVBLf3VnBGLugZHb1kbLS421niNXVfOqsFnnX/4F9pmvnK+Moq++kkV1UQNdwo4haVBBM878aZcmSW3cc3MHM2mZb2Ivtl4jaovC+KcWGa1dG+WmqOW15xa0RhLMbkRG2URg0McTcMosn0hlDgRKAUqMKFWiUimY6dqz9MrbeR0jMQyMZhjdHX7IhZKkzoPRK6Wu7aP+hQQ1vKebMFmxqdkR7hm9f1ON4w5tY2SpWV+uatyhZkgKoByuqqGL6jBNwu0iClkMUg+ENwLHyl4zXMfWPiRjGJw8m2DDORoNZikWTuaS+fXeO6uc7YKn47Sus/ebGTgNY8KHeijARIdUU6nAXrdxgCuHR9Lgc8WHWogoZNBnBwq/1HbozoJwVxkMOmVYBahKCErS0KBId0Gftnw2mhjLWDR8NJXVHon+gPRGUKpUOmTv1mBE4KcYizh8ontjhSXcvn31Ko1x0ipj+tWRltA4TF+uUWIYqrrMcDKJ379FW/mzkmltnUqSPKBrLpm8nrBKoxJHcHFa3Vdo3HdCGxjPxU2feD0sXUi4x9v08GpUAmxA0ix3L1PV4ktJNmtBaEFLD2dKHLa42fUJsmAEbf5KiTkTLTGLY8dR8583pacKgZ+2Q96GxUcNFi8+LFyUJ3XAng0Yc8XvACzPW4UlTqetSr6TBkmOS+x86v4X/PD8mKJr7QR02ogiC4y25elU1NfeYHcAaA9q9qBx1CNNHWrmL3d7VCMkfPVTUln9RLTEdxVpgAulzgzArmLVTTGb8iFJYjHpdMVGsZyKFJ8T8/BjleCpDmwF6LqdOlQfz58Ktghzm5OqAwQh7KGX2u6wkzfVKAuHFrTOCpjNGxYyLorPYEzL7VjcherRiVeKiaUg393f452rnAQVFdsuUVBZgzRPGD/O0eSVaD8SnWE+xBD/ThvNBUNtQ7ApEoLiVgBb9hGYPT1vesb6io1FHq21juycWjnWZnj2fRa2nRWEtwFNlZDrDmFOM1BC5LKBGYxSWl/wb3x+4TXXUL87TFfNwpSIcHc9kbQ6PZI13Qj25iGmwhVTLai76wKniNzXaSMbtr2R51dOwK7IYcXK3Xn2bXPQTAXGgycF6hFzR3gVSjNfGGpRWa8j3Jxujp3KjGNVeTahcJnJvAKMjxGYppqu0wpZX/g1VSnvJuwJNDRg0/MGcKrUCvwBe4YsTortw0VCl75L6CIKxfRI4/4wIHe5QbrKRdwkiSSp5YVVw+krkkkH66BjmvWxlT48ntb9tb4Ze+Lq1S7h15Z9ey9Jto7xXvigBXpSlFu3gcZxbPWY8Sg4i2WQmgG0upYuz/7uiu7BBiz20NURasjrqSua0sXNGjvMn8E9GHY39KXLLH3PpzNndDNORx5t8y2JECnlRuq06SklJ6n+CDGP4jlgfwP+uhgOieEa+dZtXskaaJUXXyv+pjWv/GH7IdtRw61TBDhvzYvA6HxouysPJ2UmzoOfYPwVK7ZhglEFjbvREOlVSGEq6v6/dtnVKDW2WqVatLa86UbdNIrVpSv9I9WV0VzTGo80afA4tG6USXXnbijYt2afuxrJtSoOm52JWrt3tlPSRDdUBXe2tJkPWtoFR1pWCoeiLv1ob3Hbu5s5QInz9zu0k+t7nXhHh2QYg4csE40qKZd6TCGglGONktlIiQdTSKbAdmR1G8drJNQz6NV18oXQNNEFF25tZXORZEsyW0Au5K6w7ieypjiAvyRVbnwmPzzMBa+7KwQ+igIE0xqffG4mj6vBYgQHBQTdw5OqIudqce42bYx74KqliEFNk8blPo3rzOz7igST1fOsxUW7tbJdS6G8H7SWZ0H3dTbzyg1lqv64psw1+q4pOfXnxq0ujfxz21/S/VhdDFlLa6lvHPUK1ZWHNZSrc/AGtvKSjXq5edtG/VlHcXVfhVOPW7q1OEXxbeVwwbsgyKlUHYZRcnugo91r9zRUFVUQaVdcfN/wSen4HFq909pibkS+WFc0WFcx1HL8Kf5IQTXlCSUH6ChWI3hp7nn1kPMOrGwQVlDT3LsLQ0so1ADTpY3RfrsrjHsOiwzsf4ymFQcHvnRU1y6iELJEBtjJk5kucYKidlJLmvvCIKimAYtAZ+augvr8eHT2+Ozy9OLs8RdEPWhPkQB53FH0/8dJ1HQQwSr6yFu99fYVgoYSX7lwLM1ewejQ5yM98KYi00GDB/hKC7BiA1RUcpEwj0rhb/v4OkzorSUZ2UDGja1gYw42dPlUvE7xZjwD2jYEifZZxxSKCHEZFRFegx0gAnF+24cRYrisgKj0P1TijUWOyBWNcXXOaWWFSuugPjo3arrA+qofYYgZPrGwyI+AMiesG8YD9kbZjzLTQ72JiBWbeAWj0+/ioTc1ibumU3z1Eqw7v3NoZEzKQrJx0cyda6tPFUlnZPe4hFFa5CYM42dre+VJELNi665dM4MbKbqi6O1ITSPzu+FrCr9ytdVK4OxAQRmqZoUGI+4qxeRh7fI62V1fIlpTlm1MW7JXVZemVSn0al/Ee3lmSMMDJu47xnMk0AeX6BoaJPyaJ+KCY1a9aAhEI5R3vwboBDZsEQxQ3bTV0Bdagr1UfAQbtNSf94xDQFGT8p4EbWrmwPxoRpPtInUUX8v44W1c0BvimHtjdHXLrgt2Q519ZXf21fzWe4Ib+r4QL5UDxbaUb1SSMJ2aCiKjT8yATdlNe8imnY9FyXl6x4vY6/D1yct0Jd7c5Hj26ZkRpCPK+uL9ZG51l9hDO1SVfLcebn/dx2JbZPW7372ij78onIj2BRyHv+P5KqBTKzrR1xeKtl0nStdcTqsbNImZ5F2aymkoL3Kt7tPsThAbFHgPs6du8NJRcoEKIRJz7sO0RDgXufPr+0eOsEQ8+8Y9upLNaCtu2jsSt1fjO8yrAx0GX2GGHmux2ZDbH8c2Gor79T7haiT9ZaBeIReJ490nbA1o3FQY3Nb5zLpiTeTVGRctmZHbJBDQTkx3XxLr/SVh3lUAym/R1g/3+F5ECQoKej3j8lvAtbrVFTiSDDaffwZWA92ylhkDuli8QgsTlUjKb6m9MgKmm+72A8DGzZ1VB/46yRauMkwe4qsS5SpqLMTvgBQvCI3WGX0jFj3A5Et6a2WYICMe1H3bBb1lD5ZCjNHfYMvemHjcCvdVCFQCgc3M96sA++MLvOjddHiJrdBDyJVQn/OtuC4Ls2PN67NceZtVqO+6si4JlWvNvB/GItH2I/yPKYt4lbAM7aH5CLKPbelNfZ09YaTQWTnuUtf2miYR7QS19IW+AiuHoYRTdVZNGkEtPEss20ZPSLfA8D6gmDR9EaqZxLwWFbnfaedmqzhTEas8sjpMyYi/Uw7qDbO6q7h2lUAobrcwchvNyl+U3WheXYBgZ45BB7Bj5PZq0qrpwm0JTxavW6SDSWXOnegpPTFy1sWrFaVY2NAd69cSdafFlUYncIQn/XSqQ5Hm5eWqgRn1EFah0HUurc7A9L1vIJVM1Vwe4945+fXJNnK9YZJ0mAZyudyI9eWiHTsxMaV9M2lrzZ51dMM/GafX9vmy0CjtI2CiUM21JxUdtTXawTAz6Zmuq152RASCVVEytRAVIwWU7g/S/bhWvw9rN8Y+soY+wHDrEysrpnlcDqTodNu3u+zlboC5Wupoe9TvqCk8ybWz8I7KXW5rEcLT3kbTp2plk6wTMZGxU68uTpYqijab3/baPd5CPloPDc/NfUlbRURM9PR0DcEWuJQ6Z5X07+jCaNko62gbp8tkH/EiUBcyqGxZccNRC7WaRTHFIE3H9pPaecYq2RebqQ2zsUnJI29LHWpqOLW7+FBIujeWMJ80JDnmC5MUFY+K237VZ4UGzm+gQ2Es/Gf3nPCPE3YNtiqlbHu1N31cq/s1P5D7SiZl011VlbdEqPfXnTfgfeyLa8NFUJ9MWu8KOqi/DqOOAMKr9d+7+1SoIfOsGs2gHCV/xeFPs55SapuXUNHGhfmfbTe31Fzttt8aGnV6rqm+0BLM+p3+aylg5IvWQJEVEdjudcsEiyMsMYfWfF2DveeGnr5moBUdhC6/tdyIi1HFlq9etNQmETbWPxqHJ18kXP6KUHGkfOuUdLQE4Wm1zOpnH6RLBbIaklgvUsOUawEa3LH67RZCvuCIZvP6SRCZEfBEWha+vobOa7vvS3Ful+yyzbX2Zj69RlvY78YGo+r0K/kKCgy+ejkIwf6M5SsH+6xxKYytWqsXG5I63BrEPqxdP2D6sdsdhHa+f2uVegQhxmSAso5RxUL0Fp4dREj3hFd+HqG6toUR7jC/lRe1wNMv1/eMUIf7WSqmd7OmbXZKxvYgEIKglLT6aU5f5RI0jMB2Md/k1K6gLyJbx+k9YU/34gtfamHmh5jDo2SjOJWVrPBt808qv4GZrCbLREg/dWYmrHVuYhao7pC5zjQ1q31XulrrdivpAXVB3OGFfql1etr2d/eFrwR1NhlYaB0JdbHb2c0weU6E0+HoMcUASTmb2LAxxA4e390FNa51cLxVN/Han7S/d+sOggtewSUgw7ilgdWYyT7r4CSShKZd1XUfgoVN424Eo0nfBtg+WjUe+8YEORobIePShNa7FPTdCd1Rq5oO94zzUncAgaThIW4sdCgpd+6ZQG4+a1pkc3MBNz0k8sUFAb0qA/fEReHqi48Gui+M+VPf2+ZL49T5qoUvuWKlclu+wBtTqthHdFbiVWdA5hjTJ9wb1dmtfhOnfGfsjULpVr+X9wkTo7yxRz3xL/49npNIcxq1cEXF+wbhSSah0I4U37CexL+raE0jE1bzkvHGJstf+AW7hPkaxIYqs1Iu45vKRzAZnka3FAYQ3NQ8ArfkWG7utyuh7bhGX33WtiDwXnB66ec0RFCB/NWytNNVJm5RwW8m4FYxgJV8Oq4tkLM0l1HuADySPjIpIyyU7s19zi9Pf1apwvKcjV7pJ0grGM1pmSKp8c20gjr/q/GXGBMnNWiDIJ069F928VThRxaPogreKDwaUSnfBWx7fYR06m6mJs9oZUzvfYJHK9uGGqryI03thU7XGpsFR1pKPc/utsOlbqmlx0JKFwXeEsXlPc3qNbTOpDUDXUjsrsm/T/RnxaLSUJrTEf+douXfyOCxDBoarWnyKDzFtccYTBXj+1XwcC0IyAsSBHjWGwTy2nDrhEis8/eHAuz3F5/j0qVjYalr1K9IRzsFyiY1ZyLsPu/evXk3YbDJLG+dPoXATvFNMUUJlpTxitFGbyNPHBxAw97/A6YmzNlpjwAA"
  },
  "omnivoice_samples.py": {
    "sha256": "e26e798cdf6001ede357d6641fa32998f40676c2962d25a1b82ff4eb7b02eef0",
    "gzip_base64": "H4sIAAAAAAAC/8V925LbSLLYuyL0DzVYbwiUQDQvfaWGs9ZoWjuyZySFLnPC7qERIFAksSIBLgB2q0fuiLWf/eYn2+Hwj+zXrD9gf8GZWRdU4cKmZmePZ2NbBFCXrKysrMyszKzffXW0K/KjeZIe8fSabW/LVZaOHz5wHOf1Jk1+ypKIsyLcbNe8YDdJuWJhyvinMCo9ts15v7hNyxUvEvgYXvNFlm/6a54uodx2Hab+wwcPH7xMizJcr1mSspAVfBvmYcnZG+qJjf3h4AmDrpM8Szc8LZkLzS54bhQYHsGfUW/y8AGTALL+hm2TLTQpmi6zPFpNpyP/3B+Ih3AXJ5l8013PWSblk1VZbovJ0RH8Xu3mfpRtjj6O+osiPNIY8OHbvx2cz/lgfhxF8zAa8+GYj8+O+fnpYr7gYXwMH+D18ZiPHOaUeZgWiAyeF9PpiT/2Bw4romR7y8o/x5uHD15kOXv+4btnbFdwBghkm7CMVkm6hGG/R/jF15sV54i4mH/y2YsXmy1fQslbNucs5REvijC/ffgAOmIANeCtKHjMCH08hXkjJPjs9bZMsjRcsxQgCtfJLyE+Qws8Lpge46OCXZXpjKbs8pOYcRPlGRS8xoKBJAcfRtPvl/xT2V8ka4AozHNq2S8/lfAF5g7+8qJMYHC8n6Xr23+8OZy5fBeVzIGRcI9tkjhe83645LHH1tkNTC8g0mPzPCmTAqg1ioConH+kY8BnnzCJmPWBzOU7rMGcS1wLBs5p6qM82Za+AwW3OUxM2QfagEmmbv1t+Y9A09rgl+O54GscQVHidL+HpnZbQCwPNyzfQa+yNSAsWCZZBO0WgppgdX3kaeGz9ytY82KgLArX0W5NhcpVCE3KSkhl316+eP32ki15ynEcQOHFlvMI+UjMit12u06gGnGUNayOVYYLIsyXvAyAkUCD2cMHGrakpEauOawJiXCfvSwZgJJmJbImXCtbDn9SbO4aqD0GuGLkVXESIUQPH2QLtkujDCkpTFL4mIblLof1AZCFHxFEGDw0/CrTTA17QBiAtpDQyjzZbPBHlj98sA3jGBopM1iEABC0E7J4JyYN6LOgpe2zdzCbSCA532TXWAjGrwe2ztJlHzt6+CBa7VICIsw5i5MinK95/BQwBJiKEhhrCkjoUym2DYF/FFQyynbwIfbZj2G6g+apQAFTAd/m8A1BJJTzP+8AOUm4BswBR96V2VEMDHwJsGYxF/VYliI/kJNGg8NZB+CyNE5oYMVuLpuSXTHBhlLYC0oY6C3tFtkO14YiThzVIs9+4SkTm0QB3Oa5qC25fgENIpqKo2gtB4dAr7K8ZDDuTZLqsdGwcz4PYaOJAEZgljC5yEwBikWy3MGUi6GzLdYsgWr/RcIUGisWwDaQK9gLjhBaeviAlg6TCBLwNHhOn3gOdIYsiX0rOQ8RE89xpXCgaQD24QOkdxgJTkUJNDTfwZLxBOmy5Q42Rhg8ALSAR9gmywRXVc4SpOakvEVIC54n2a6AucpiYtWvxHqAQW13tBBGx+zj97947OJ0oDbuI5o+n11+wt5h6PQeZwM2ynQJgxIFHz7A2ZYAAUHBrEVsDhNwk8TlCusjrxNl+0Rw0ACP5FQA5mCFwRwCD0jZ8PcA3huYmlQtKMBytN4BKlmyQThg5HF2k66zMD7CP1DOo9mupgaIgsQFZH8PH/xLCAvRYC0w2EVYlE9hvtdxH3Z0GBtyISQ+XK9S1AA+AM0Q2n32DLa9Gxin7kLsgkZHT4WEA1SMgNcYbgj0AMWTEgn3zQfsJuaLcLcuxXpGAvmF5xn7I3z86e2zH43dl3Z0oseCtny502tuicyWRk/IgT5glnlYEBkjLTUIBUk+g5mIc6CAnMoAWM9SVjUYhVv4f4oVNcNjovzROpnnID8YGIUF8gFIFQeGi4mFDx+swjxmX09P2B+/pTFFPCHCgYEDq02AyFPiUD77o+TwMDF/BgYE9OoRr1qvk2UyT+gFjBVGmPIwp40cRqEnLAdmkgAVySGTKAhN3Wa7nCEQN0BiRPBvkhR5dgEfIn707M1LwloEvCuSXAXW4XYLQIL0coBsdwQcmB8dKtxVTa52yyV0sghhD4qyjmajk0U8P4FWh+cXp6fn8cnpeDxenJ3Nw9Hp6GJxHMVnJ+OTxXErpEYPR6ZEeTRfZ/OjaxIrj4o8sj8iF18XR6tkuSwC2rOD61FAHCAB0hTfodmgqwQICL89OHEYVT3DA3UCSgbOKGwJGxYEix3swjwIJHtgRLaCLrGUepsvYaUWvHqBMoz8HSeLBZC1fl6Fxcp8Fv+0vvE3vAxBVghbPu3KZK1f/6lAGUI+rDPCiX6GTW6lH7JC/8wreIvb6jWyRTn+LdSEvtTg31BD9KW83WqeCVIEMt9X4YYXW5iIqggtSlHmWQpL7zksa1yYnmZAHvuQkvzz8MG71x/ePr8Mnr/+8ceX79mUOQfrNg8f/Pj6u8sfgpffYbU60evPby9/evnu5etXWOjQNQCVXz17//Kny+Dts/eXUHN0PBiA8vb+9b+/fBV8//oNvIJNDZQn4KtVoZOHD767fPHsww/vg59ev4RRvXz17v3bD89pXF+oJEi8fP/s3feX76D+ZxSkGXO0EE2kUPhCoBZSgTNhzmi8mM/nfDw8PR7OT/loHJ0cL+Lj0/gsOj2+ODnlg/FoNL4IT+NFfHEah/MRIHd4wk8uzhfHp+Hc8To6gv1zGQA7w0742cXoGHabwcXFIB4NT8Lw4nh8croYhfPh2clwFJ7Nz874RRQfj+fRxRwEvsEF4PnkLF5Ewyi+aOlELE5fv8BuTsfDwclgDo2fnZ3ML6Lz8GzEo/F8FJ/Dm7NofD4envERPxmM5sOLi/N4PBydL3i8OD8/Pz5dLDrHomRi7GQcArWdnvPzCz4fhcPBRTg6Gyz48fj8NOLDcMFPzsaDk2hwPj47CU9GxyG8g4eLi3gRAumMOjtBxYmQFZ2Px6Ozcx6exdF8NALEj+MRQB9ejEeL88Uc5uX4/Bwo8BjwNz8bDcPT8Xw8ng+jQXh8EmIHd7hUHj4AlNO2F0i5iceulKsmuMGhEgA7HCww+cg/CWGDHnus/w3+OxHwggaxy1OmGmCPdWn2RLfD+mzYY0dH+kUFh9LGAlKY3BueLFclwLEGceUKupmBbgJMc131rb9ICIDn/oDqVVGCErsJUV/KTaFK2HiAs8E+jDswB4kduA2K0ELu9olrY1PJgiQSCQRKp9Q3+xr3YQVbj5Fcc+vegBjBBqQl3KAtSH2XcBF2wgTEj5/C9Y5f5nmWu853Wo8CYVSP3oaPNi6CjzRyCWRPIRzHiMxxKqHrW9CJUiBgZ6haoMg0BdVmU/ue7jYo22R5AZ+vqjYfw1DqA5qJKlLBwfJDmNtrHBTOqdkVVhUfoHrVh25hUWYo2FWQI2SyYQlZlsdUoiAicnMU511zgD2PfeS303W4mcchSybM7btVV1fJjP3ehAmKJz3ZNoKXIGjUydVEATQzpkxCg+08mbKhReXym0G9JXQTBbiDurjfTWib8wQOJrhvEcm+glmVXZQctzSUUqe0Q/pInkEKBEkN+PgLsOv4qC6pOdeV/Btg8DxAsnArkLF7YEabbeFSx7BiUxQ5g7CIkmT6IlwX3CNDQlpORx5R3Q30mYpPPezv59TBaiB1AhlMnV256J/LVaEmpvBBkl/DHu1qeDwaQ69CCEkyASzHHcq7hUvDVxKOr3d5iQx6i5OtSzyTNd/QF2OMsDWRaQaWzjQI4iwKAg/nE+QT0NAC0K2LYqqbeRvefFdV+J6vty9UUWtMiMhgmWe7LU0H9umHcRxsQHIDLN0GpOEVoF2IUm6jHhVX43UdYehyDitG9jBAO8pD0zcCkRVa7CoVJqCyMAeYNT22gkFOnX/3DkSUbP4nDmotaN3Pf3iJmCtxKaDY+ZTeEJEAewPKz0EVR+XPnuo2AKDbVPVIu4JUE6fD/ZUKUOva643Gx/urxvwat/CqihNtd44aKvxmrvxCPDnaxeHVhGzLM2d/y8K40I+T3Eai6gifXEeUKo4qaaJ3z1iFIQF19NYhIx9Q4EvTxfe/PEVrHVo8lN5NUqJ/zwjIUNbfFKqfxToLjZ4G/kB19LyyHS1BeZ7z8obD9rIxrWp7e7NIbxN+6l/D/tlfzju6Pqm6/o5HCQisoGp7qHI/FRaDSpmf7+Ilxz1vfXsABVpwWNZhgCQkE9fUKaBdYJD5jmtCeZWRIXDdMM6Q+UnYbpda1z9sKaAYuwPx2yROnh6MRWWVc9pp43m22YR9Ze4RVuWmlY37S589EgrBzQotF/mjw6DXBwDt1I+gHDoSdW7QMZK3wggSC3t466nCU/bs3VsyeHOUDXBuePyl5GDatLoHpaD6AfqwjGKk6z1fQ5E31BBRRghrMkTrGgplhyEWrW55hudYEqKKMvXe9G2WgaiXKiX2GX2u4HwPpHsPG5aHXlwh/gt7EJv+fka2Nbh2bX0P/cGhs6IVJK96bTRp7u7GJFWvxWy9J1lRLNS+kI0LjnZ7WANzPMzgLUcIT/X2VlBVoCjvkBlE/kbt9GUfHVg434sF2ZAm93saGw7uaw0E3H5R8m3rtjIe7a+83CVxSFBEYSVw1GAY3QdC2S9WyaLs3HCG93HMW5iiLQeCLG/7C6DZLO/eQe5ZZllBZzZ9FESRb8Pu9mvbIsnxgIbunSLUOUDb+81XvLWiFsAcu3f9obHt/0C6CnIkaPGaDqpCXAyoVpp7/2GMLcbuoNdohZyymLoOnrPBC4dAGI/0z+Ep/pyr3z1jg6QqB3N0Hn7sr5NN0k1xF+dqsG+gsLbkXycho+OgchcL+QC3TQ5DFnh3B95w9oX7C0os6z6JEYdsLxEyrDTcFquM5HA8HdkKM3+2WCRRAt+psQOlDeodtYVij7xz6Fj0Sfl+sek7eewcwWrNOZ73gaa64WUSHekTXXFYVhsED/M1aIZBpVIJrfBjCvIXwlQoNSpZiMK+UGgMLVy8gBZIt0WZoHDNoqCKhrHQg2tKa78AzUjJ6YZRJynEAQ2or6IJmLskKk1rjYE9Lqw1StUS1hr4XYbk9GOoWVp0xv8EOtFG8ln89FH7msj3ZH+QP6EZ2VUgK91VzWC5j/zWq2wpctSg/G8AezWYoSj0CH+1gu70cREGJmgSFViW0JFqYEVn+Oazg1OPVbWG6SC1BEQtd7VuG+haOB9SmmRsEg/60c7AYzWZ0MmEfYa/d3W4JE6mCqQrKDRrgG7MYBOQUDJVV3Pd4B3SNDJXxXD3M+QaTD06hatRjjSvzKF+735sCM1LDFlQ0JxTXejbb5mblmFJAhDeIMhjWkYOYEpiQ74kDjLL9nJu+3jItIq2sfaR1jHzK0dOziwlX0K5f+bgaZv4ouG7JE1Rvd6/Bh7QMjpvx0JjyG0jxJ1nD3WCavnrQUN7Hjk+pMt/6ix1F6wN6jdY2L2DOjKw91vNczceTWwgLA0ktvRZ5rctb6spmJqtiuG0jJx/ijgouO57KERnEp5xPgFMr2A0qI6eauN+mZJDmuLwNH7g8fT5rnvYUoo0Rm5tSer7/Xjv6J/abPQv6xa8DKTUVriPH3+mbUnUkBVQRKkLLzWxBR99vTWaJz7ipAC/68IulaazHEn5PTadGq8DcoxU34zWbGnkTZ5do71W+RYiewDhUhiTkW8ZZmVk6uicRFKatACKgyc1dk2Wajgp+5oN6XBLDA527q/ZwH7xRBXtQ9Fvpmz0+PF41Akw+riq1SB0tmtB7mjpuWXUovqO5yBAAVegwFCjPQ3fIknx0EM3AJhzlIHCATU7IHE0qFRseql17/oHtF4Gy3mt9TQD6XwZqg4MgZnsCQHqXMZL1KXq75SaHUg12/hEym8gld9AKb9mJ1KjDSxF1ChAamrbV+NoS0pxdXw9aRmjMWWKeSx5idZFoko6YmvK0OgM4ieFaE5yGKaP/CSxuB1gVIsdaH+wV+5urvsF0NJnU8J9FDzy2KP+o17Fb0WH1M/nR6rbR0robYGIrwvOHhlYeXSnVBnN++0lYtAP+4ad7CF8w06u/NWQ86Ln6YqTbxjwDXgiPzSpuZorsoluAgC14oC0YkI8lhoA4mvf8GB6uAe4SrduQx5gilTkBoOQRA8jHw32tC5tFLppsapHg9lT8gqWdogI7a95WOD3BL13jxbkKiDsfI2+tf9xXTZyOzClKvQ0A9NNfF2jv/oAYuuMvs67RCcHzFUZkMGsAkC+EG4D6iXIggFa9pD57oFJWt8Igsoc2ICwgTdxJBVQhEoddfhwPhgMEKBGYSShixF83QOUcd5V6TfyfIkaxl5EK+z7XxqwNXg3e8wMV6j9CGkYaiuPihvbncJw82+SFR0xsq/Qq2u7c6S/qr/Yrdfk5e7mDh4run+YTH6On/T+4HhmvZ7aszvhfKdPbtRxpjx9nTA8v6ydWnaBNzXBEx9IcsT3ytDWCQI6wEojHfanLHPoi/r8zYenTD2jk+xHnqfAFcn7GamjOnjSfvAlntXVwIS9VrhdtpKYLoLCyZ4ZNcNS6NSoMOJSnpJjP7rcR8bZkWQIKqigFTASjjTiLFD3Tp0RE1ODR0YhPQ/JR14c9BTaOgT8PcmtqJkaUKJCIP2vFWDmW58OptB7oh3uFiWhOeccBCr0hOr2/SaJUQ/oqBrw/SDref0SsPeC/AwjS4oWKNuQL9lgJ7w6qEr2ugeg7uEYoNqNTlm7f6blKyQ0AOUXU6A5OpBhKy4tBNQOPRWXxCfav/XqCj7MyCgDf4HNJZvdRnribWB7UU8MdzHtEId1Koe4bylIxPCDp9hGPAigWJIyz9aFCE1YhNdZFaQiY1QYT1FxKCzPOAmJUhIkKCj17XV4kyfAaFpQu6g4pxOBLDQxzJS+RdwMnhnLLvTs7lIMIZgSgwaGCcMD9vzz1dV/+nk2e/Lz7D///O4JGucBubKCcG6YVBhC57VZJTBjgyieUMMTS9xV8+Litx6IPGoibPmwMd6F8zrlFElwnRQJTCe7yfL4SCKdCV8jIQmKoAmBDBLGQHPF3r6qVGcRAgITLR5vVqgnipfCJ5EgN2UZmDl0nqMiT5T7WlUXP5s1hTamxuowx/9TlsiPV6KVCVWCtma9HkoF7YigMlOzQ0RirbdaHbQO5wa00LI79JiL1friLXmNjmqKPE7dlpy5yI4ORHDlf/WH//uX//r3v/6Xv//1/8yufnYe/e0v//tvf/kfP896d7PH/waoAsp4Tyd//+t/+/tf/9ff//o/W8q0mXlASI0p3o7cHpPKgVC6JeIAPIWfHg4ZaLPgYQ6Sg4TQE9QFlfuIwlmzE6hV9dNhdxHTWhW76g9n7SXnOQj3xkGK8O8Jt3iY0zrBE/gwM21kiujgvSXi1ldJc62/Up4JsfRcFVsibljK0YYa0Sv6d+wnMgRQNNwu1Tp7rIoDk17zAkMVRUzOWxUdh9a1DUY8L4Q/sWhNLCby+EVWgTGSFFWMHq0Y/zPP0HJasUXSQoT3q4op9FVbFA9Kp4cqpgyXK47ryAgIf4pj0xtKlGeA6JgRgcZVa8g+Jb/EBYcrQkwMcpahvQjFB5xgWG6a45tHY5s5BXlOmSo6mqELJ/zvCatqt7MzVXvPUtatTohfqhpGg6g41yvJUsFhTFqDYbei/MMD9MxFEqwjZTSr1bBX6KzJKKJdWS1Y4C6EewvYXtvKRwCAa6DLMYZdqHVj17yaQOuzntf5HT5PZr321gNBRbIT+WSOGQv1KgnBpXK9Vv6hN+Op2bQxxURiRimrz05K0DwFl4bwWcaDELdicjXWew+L9VgTP4It9to7rmZXcTBS9xU8IMLMC9cmmr6JAUAfdOGZs9lrWqX3ct+Cr3lU0nLb4ATrsi0g11aOqopk67HqaTibWaKiimxWwiIJAwEaaVwM156I+CdpJJow6QeRbisH83IHevgVFcND7SsSLanczJAKn2GsJ3Kt23kYfSQHKYqcFAyTZGHLbcLTUd+mG8kSmGUlGGrkI7gwaOrUTbeo3OM/OEE4iJ7CO1aXuByit4ry2jiym6LN1G6czHVDf2DhDttmj6lVj312rBrOxG4B1imWg9eiuCP7hhfy1505CUAsIuYgQJGUnt06ttUfMxgFnQsxTq8P+E0WCW5mZYyRcBjQAxKwzDuCUdsivo3iiVcYuUtB+1uYnHApYkOxTSuZgfyojGYY3qiijxlF3/rsB+ndSjvgDkRv0e+tsndjmE4MIjmdISBfoH3tJmMaXpjsbI3OmItdKp0EZJwwZkOg6GRTPk3sqHvA1nXCb8jBxWc/kuutzm4AVRJMnSJBpkhXciKRHvMU7I5WpVJ7zKghS1+/ML013XU13iVEWx4BedmhjrQTBfjJdQxnbr03UyWlDwqn4MgXpNOitwq55yX1IAWfKrOM1M5lPhbDJJFnGe4k5FlutN/D8yWuzutFDDD5j9xVagrGhFAUlmBEAJQV1tfiFYKTTJEkU+qX9iY82XUf41uftFHXAdiuhrA/iRiUYrdYJJ/g7fbWPDnDOGRc17LFbscbp1e5n/ycUzwJRpXYpwgyftUvVuHo5NTFxn1qByrJVnr+in+KEwzoguX21VQPvFX1ertLMeZUKV8f1DoRmJSiWwyaFQJ/hxkiRKKcz1bE6J11Ritn4QqrICN31eA9woYsCeS9W7fOFSklxtnN3mBLb09BHSy5r1Cbk217FKE6MWohFE/NszX2qpwYLHyvheu6AZ1dBsEU8K7xBNtuQDFN8J6eDdHxEyxCXOFQlKafTtzNmg4WQb9B0acfBLilBUGvDk01QeJZRrYp0KpoV8KI8gkAZq/jh43PFEkM3+hf80OOMaS56ZXsyKBseC9/WUd92XLJc/gmI6n9JS9/oHcG7wkkj4Rhiqp3NeD93RaFDFeNdD8NzepIOrwdTWKNNjpJY1+4rSQZTBWA/ghFKY7PXWPdVHsKiEgV4lIoYXvA4RtcStiYP8/iW4uTGJ4bKZERdvZCtv0dXyj/LWAwak1+dgK5swVG/Eb1UkdlyNmY6Q0ClQYNNzGlUWNDsDiRI5M6qH6OtPFQ7a3CyL5Lw+swIbtfdd5tjF13anT3O5IIChwUpcAI01I0Fq6ROd+qJEQxSMcg4z21pYwkVxuq2eJNtlvHAI1KwpWsbw0RQ0kWrzKWkVFZCwiRyLBD+rFvTyZNGYnBJVAgpb/AkVVPGCKqyzW9NHVBMbViv32RZxtlQzEkcywAm1cgTciBVvBd/PIjEb+L/Uw1QoUXbwDLB4SKYnqldRCLQ6k+urmUXmrKSKl2oCavrPDToMOpbuaqhUhnXktVRVGtVTUpW1X/4/fB20urPL3RhXqWaK1G4rFO7qH3ntmeQrT3zCrJmsTM20Cyk3tFalzrjZwWPq4g6Ni1MtM5tDQdkZ7ungX6Vh0rtKS2I0JFUVRLxCIbi0zDoldqtwjwZeKa3l2rgYpfgfji1jZRKdxIaVLvk2In7v22Ipro6/+PkFaJ1aB17JPY9gkEKkNBs/TejczbW75B/geUFuugWmo6ZppSNfHKY4ncByYtYdOeXpP3qP64fAyV9C0QBIvEcaVIfEYWStGvmgVhxFSaaSO9nXUW9KXnmOapnwiwm+qh+PXgO3L3dwvpimQ1iFJhuNXsXQQEW8rKjg7GkrR0Rb3qwC+Qif6ArreczKsW2decjrpqp3Gyof2/brhk+/sbzLDWeaOWgPhrfcRjfML90GgUyacq07vvIKryjRTzLvK+SewTTJTfonlMrYxiBNiR6ZDxTeW20fCuuxcceVpeRYCqQ7COUDkLHrmGBfSeKdcThePqxRwpYqhBhLRkqkNSkBc4hII1rLaWxMkLCAloHsJ/24rlm8KRFjmT4OB1zyh+1+EuYR45ikxH6W6zvUUn3HSrzD90/kxrs3t/aPAaEd1rYBCzfE5l/lRcXwKAapVpmGCJGUmLbJqARvwC5cGp9GBCEk23ld8TFOj5aPK/nzzf2kld7Qgb0Rzmh0s54PRW5f2ziWJTmAbH4s+5MDzyMEVAHj8e9WpxQOgqh7W+ZgN/WINQIAj/PsavQPlQ0qotuZGK8A3EfLc2I/BMiTFhhYg8mQiThVtK44Rfp6PBAG3UoXga4hMmtlurjy0cjqZCcTI8JNVJpVpWsXI4+43XdNMA7rwRe4q5zmliF0nZtdBltjsZWpXa7lqt3phCw1SMi01tX58vnzOrrYOXmEj0QXF2W1QqRHihWzXW5F8iSLCLe6X85n7WVTV/P9fCXZB+97pZF/xtMqoucDWolvWh6XVDlnoZIWqZUvQwaoHeLcC3lRAgG18Me/0m/MgDPJF1q9wrwsemcupvlalstU0ctsgkSlK8kyRgfiKv/eZrTdyTmjAG30las99ODCs66bGUpnfL84VAA89VyIFSBU25qaEnCk6unmouUboFLampNz65MQgx3kGOK/wMdYUuC7jl9fIhrXwFVU0Q6K2O7hoRB8pfoO4aSN4DUxPLtbLkQuWx+lgrD60mltRH16JVD1OcBr+spnSwWdUixVidcpJLkbYP6EzS0zqF+G/hj8rwdanKuVqtls6I04pOrqpFMUMO7bxCKa2E7ZDzEnOFKhaIxRTvr9c21s1M+KC0tU6LEpMLqgxhUDC8cTHPFK2TWmI3gw9IvxxkJ3rwvjqWDhQOqCVPj9KrQCbUyXUjEmGL9F9Ts2XbRdt0BO719nuB08CqeDK+WIAmiimbKEgE8QXDpCmErc8CAJq1n+utCq9obMZEG3KZlMd7Ufc79iP6D1fmIEo5hmyBQnh2aS5TWCs8emhwoAhJt2dY0tTnyjHgC3F/wGxywEsNa9WMKQeCaTVL+/22lesaeTyKIANrolWDKGadn4jJ3T8Lsobp04fm6IZLpadmxjN7116TUzxzHlmwWgWFq1mvnu3OEHNqyQqvkLAInp5wNSHvo0Qd6GPqQmNctijaNWLbyYZdKVrb041luxNufpieUDp3aM9FqitfJjqXnSmokxM6jDhFBiud3TyjCpdp9rhO2Sdt41Y/ptIBRH+AL5DwmhAu8DaWKO8fNoKuUKMDdAoVVqMZhYy9QfmzLZE6mvbsDFhE85RelU3N/pE46m51AmY2kZhDH0k0eGtvIKp6NRHNWX5C4ov4MLG8AivC+y2oTQSvqeSceBoMg5Z2FRnXBgvCzFJ7xED/GOhzJMESdH0zXyNUrDLZCu/NgWf5tlHaz8c2ENJthFKriUiTluATGJ4Bk1WlgqWWwtSGtaZpGf3ZRm5TutUiRqylbds64GiL/ESLHuZnbXWfaFHEiuEjtMDHK5uQPzuyOyoAVKInEg1I2Jf0FHPsMeov5kzctbjAiWarhfxLspVz5DE7+yYde5kwWz0S/lACN1BrlLULGSj3GuGUjYFYNGKVF/dcwJw06tRmvLWSyEZrVLLJqL2Sihitl0YnpY5REZvOA1JQ3BYxXjk0o+flY1piZvXa7gtt1N7YVAYtArClTj4srjEQ8XNyljMUA3eFMvgaTq9P1c0irVeGVMFv1QUjOnWWoW6pi0gCOhlxyX4uHdXouqLW5KfiFGUqU98YVlJ0Y6rCpS1bOhRUlzsI3gBaSoM3aGMqtK8yxKyyrXJwhCot5pF6BUoLrPuohISm10TbedIPlLdODhEDkpY5VzdMGel/qtjC2lkSVMtE5lsBlLquIpAfFAbTazo4l0dGuLzlT7wHQRT15RGE26sdlIv3cob8NPWfQ3Pv0aKwzQo+jHvVibfswMcUBjGZr+2uCBCyaV+de+zEY8ceG3lsPLsHTR9SfTAmUSXHDYKyusND9FlYB+K1niemuzolMVWHPhXExkwvFgUvbVkO/+uLsrWXDGVE2Rh6cQNIVmv43xNVIE7WNJ+IocdMncWJOLoAbaX4AffBjvqSn3T3Y0RU2LKRHBTMwOD+QzfnOSG7VHNNFrfrbL0ztSmVV0Csmv5mty6T7frWpNJ76G3S5aLRSnlAcJOGM26D7mCAFHDUOiv4tX0mmsbJzrm5P91HN0ZNNO40bZtBAyJXsmLMhElts+KbLL8NVKYwt+KcmFgMQz/lg/AInt+WMkV7ty0JMChq+jpCFM/Imkp93Qq53O5EaDv1ArvKoG40JOjIkTZQoml8UGkZebancBFlWzIsYhAqelbs1E4lUsaieZgCSkCVRtYxx4u19HVdTtOCKegMY2v94jaNVnmGe4wrw3alRVANgvKR6/Ko3cqJ0SXsimo8nfVUAbsaWWZkzW+sKd3LMheOukFP1xbu0HR+hsigLLoT9ll9v4PmPxvt3zH6x0gh3xB9986tfuO112jMr3phCYdqitVoqjTA6Gv+VKaEl1f/4E1DYpiSN1V3BNmJNapRojRcPTUlFp2txa2LJeSWhMda6Lc5pN1sbCUIUBcCTCmS0NWxJFQJI796tuea1OrxWgQoUZXz2MXFRZ2/mGTYfhuCvAKhnjStACSXpJHpJiivjOhdxRnZ3wgSGQtifvpGh+4h/Mprvrr6gQbjUmZqjw1PUUP0mHo+Ph7azxj7b0XVEIalDC2RfHGKLUFB+H0ewL/4/wZuxBjbL6vwWuCkQVIuBqzTditFFWdgJGnHEJ9nDImDXX3LLv/DkL2bAcklILGjPwwmzMQ7KkNgS4vdGj0m6IT6AIuUvKZAXgiAEyLSpoC0pLKfy0FWMURSe54KA7gw2Ke3rqNBc7Ta32YJsJtFo4vWySnqa3S8p9ZCHxPSPMEcndD/ralBRClTtgbbQZuGQw0EVcSVNADX6naiTLbbiTfput5DJQokzmPzaLSOR6MniUwFtVob9RICWlipF2TvGUqfjG2e2Dk036kMmqC24g2lE5neWlwsUqlannUzHbZdeCr+2gypsUxR+AIQ/7e//HfAujCh9YwMm9V5FwyxZnjGwIzggNMkM8GVeVnD/emtDCbakCgG1noycl4tmJUAq8XErpNg3ZfX0+qCQv9RUNy6djIWlQ8L59i2PVYR4hh5eTLYH7uuEm5pNwTRMN77AdIkTD/N0CopxVFiH28Dgx0I77HDadMbrp25rfKrrB0feeaBV1Nb6YhIaiZZ685z0PTbak3FRxNTd9Ns6CEyk5wRCNOWPa5FoqkJfz/KvA/qntMoAVmFvcgok4uhOsuYGmnXxRT5ePRAGTcw7SNQz6OCxVmksu4ZADO6VIxAU0taeBJVbgrTDj+8yudOmVTXIQWw6fNmwbDssvvn1ujXaPTKofWrpxaNSRR+0m5OMhe7silVy1splnEC6tlH+OuKOKOC0k17IidIkH00s0/beeKs8AlKkofOQjLpl63dydSJdIWJJB5K9NXMY3sNLIfQ2lNusFb8A0xpskC+Om24HWDghrqpzat/CzCcohDWMPu+NksEJVIOkHQSNLdaDqWWodZEBLl/Gc81pwQxjST0yt91uyBaN+GfpqWUTMGWtRV93ncFXfclj+bq4i4dZCs/HeWeDBXEJT8WMs3LiWokAVKQ7sHHAo6ncd9r3/SMS4Zq1usWN6A2U6og8eaXmbe/gcpEXGugZtptbcc2SYsWzHetlbSdHvcPuTLFq1mvFVjT+CuhrF61d9FY6ZNuHlBr4K72rC92anDo2oauj41R4Zq07d+2L4zHApHGu30raHNZNN6TkEyJLeFbsbC+leKeWBI4kTMbN3EGq52+M1IZRgJ1c4tRhS6f99EBKVThpErawhR/61ujLF5Rr7um6+qVzVWk15JqvHh0zQRf8jC40ivr5+AqGyDwX6XEBNX5WZs1ZheHzqSRWtEwJCRFoIOCOhwpa5YoNJHAFmgEEz1Ft5Yq51i03VkHnKZ5hNKh0sCtMQsnhufC8KKv1iWVXJ6pzjnexovpLlbAj56iAyPdmIKnCiDYimw+0hRgeDTIS3+mFlb7bHx+TJvYSP1joUhW+rrL2Gl5GtO1xKICOR8aiReVIUmBZSMFxAs9ywaKlhpF6D8IGzHGcCts+aKasPx0YniL6T+lD6I0EoGyQKcrLqriEtwjEwgPg917yhLYPns4DpEqQDWLW0hhTyUliwtkPLCRPk571dATen+Ry57YwV19s0OXTVG6+embHozOjHSiBKxnwKDWlIhSP0Rd0TcpWCKxdl2m2xECkgN6TedP42vNt6LBXVwtXzAlT0xr0gQTzVEQPbHRqdFJ9bbBf3G9uCL2W/aKzkhO7QjMwXfyNIp25R4yA2zW7X2RSP1OYYzoX2QC5tXVE7VugVmkUsaGtcFz5KgLkLQpS4SVXq+xw4iLtQx3Ox2y4yPnxY1MHjCaMyehU6SN8acUPyypVlLLlP56TLirF7m6yBDIKg0otA7VVhkIUsTb0L6GQpwdchA9XetNbfBWibZTTXlo0bMl1CspeCoy3i+r14jdFterJsUtK9hOtVwM27H26iM33BktxMpp2LRX0qEjcYlKWMRKje1ER+MI3Aj7eiDce6lZ6THdcpzSGjqgPNCmlks2ZZ+zXbCn7Z7Znf7eeyN0iKV/1eFW+YWBOuhO/9V+H8v7onD2rExanc/QWXxtXoYmiFHm57bPjrUo+NTIonUDIpWM0LKXZRNr+9IaamXRCtoSiuLBWmNtPvDatI7greYSkscsC7zCU64RIv/WgzHFLqzjMJVVRicECfTVMU2e9EddSiTsNyZHpeGdWkl5DZHazu4titnvzEvORKbfqZn21/jclg5cbiQtX4yKbWnC5Upq+WJUbKQPF7Uar6072eg+q6kUh+nBBKWxok0VVAJbqiJC9WwWAc6uDCV0Z6VxZIDJpvU3K//0Ud1/Rixj4ZLa0ZxZpFyB2LQCPUtcKeU6wCR1Joae6W1GrhZN3XmhdMjPCR7rTAbj+C7AbPbBZzNVfnLn34TXTj1roKAww9cCjft0+a/OnyozHJKzloSkKQC8ALngkmpcqhSqAixZRecDEM367Pkqy9DiCkoRjJ+Z16BqQVj5mGvTdwa8CGlEemzL9P+PmduiHONrO6k1pUV8ot0fqqUvLR242J0q1tQ5yHKhWumwXIjdD9Q8l4Ce2mPw6CrhaZWyZ2IkJzqSLg3oEE3WBUekb5w6+Neh6zHmIZo3rbsLGY41AOx+dK2gEBYkKc6dehLDaHWNqjInpCRMiomo59ij04lp3WMBIFJ5PoY9EzSESO/tOV9iAuc8AOK6CfNYwKsht2Kr6heNSFdg4yph2x+4hUarEwNxsmmui5a7TUhYEffS0Upy8U9LirV068MqirONv6/UAXp3s3d5iq9BwJPwrg4O0FpqUYEgSYV52ZalUCRbyQvl8AlF2m/H0ad2Afkbe9URXjUTtRXZMWAt0tAxULs8hEiU4iGpSyaIe1oVRzNue4kWua27oO6/u4jQVO0xXw1mV46MeYEmLMy2ho105B7sdQ9SRJ7a8b51dLVPoYiBKT6a4kkllbyHL3swN8eIEnJcmg697mI49mJ6RfhQmJjtKy+8jYGfG7UsD+S91dERGiqKSahSluyropyjdbUqXcm+aopkoJr6eV9xk6BlLfp9Xz2YUFEcfuwrS865ahR1L97OwXSQlkhwRxsn+bc3d4AvV/BsNidXvNgQAqVBBElJBAiCO9Km15Smew3HxNpCpSwUrqlIUTihey6ZVIOi9sHZZef8o9J4pE+zyGJQ05cWefYLfKvcmNctl+3VeTOmf6nwImSAALYZvCumVIY7OTxPEUgbntq70YHYfXuOv5rad4Dswck9vrQ6zEwI6iBKiftBqNvO4bdf6FZHjM55MewuiwmWVXHyMfpqqia+5rY/29tIS0IBalRmFehYTl+Gt/2Eh5fWZ+iMoemirpDj+S95yejk1qm8PUdc7VHXxw9Y+dXeYZ7W17eujnvhOmQIuRhjXoLy7fb8aLvrQmAlnqg0u3oyw4Juk4M5keZgdGDZ3spLxPfQeyWnoPwv08DX9QUcpTwwawsLmU32U6cNM8D4C8+zwt3XorItVgPqGgQaNtWKV+jwaPNuL//FNovm9m6J8Xv4FAAP1Ieue2koF4lARbso3FiZ9ZPUA9He7gWt4EBmK+MtukxYrYwoWbSdy9bCPDogolsWKBKbfP6WUexaYV8tDe9ZAhK71iniAfyxtRv0H5PQeXZ8nX7fwcyaS+5Xzmr9fPwLZvVt5bYmbQntMys2P7zMuhbN0oEnPKdSUXbKy6SW7VqUrF1ohtmuW1qWvvWUmNzQTUU+WzSeu+Zb2BA3mCcfDZwlWWbaFLuFf5PjzoMmTN1+z9s/2x4rdnPiK86LH14/e99K5+kCU/hAB/ir1n77FGNBEZZUHDLBdHUtVhFvVCRTC7xderCCx9+lMPkfXZVPsd3iu4+C/uXZT0wl62Nxxgt1S1q0OkA2I08ttBrMC9dEQv9eHOCxGkaiGrU8NmxHr+jlmykb+IPhPx8lLyg+TzsiaywIMGB//2Y6/H0rOrJC5+3T8FjWmJY6hqlN4UZtlO0gf+7mdA4eQTrCAcxcUTK1htvr7VGRnEJENBZ2GGPTi6bBHNAnpvFybxO/jdeQbi+kgxqjMZOqDoPjV7k/NQDQrZiL4ai+2Pe1lPO12OS1XyKRHUaa0iXAe6oaWk4FyN4Q132tCTEJWvgCeamjvbvm670Loct7Do3QuJZbron+hwzRBwh30of227Dgl/QTGULHHcztxvNFCGszdrqKilnGkgvnM8kV9KbnqwTdxn3NtYPB32TsxAjNfOPAA9d18zLan2W2NHfvSYG6GkCNt36U+OsOD/+xkQpHyYXzLsTQLHEAlN5VqWPF+nTlXVtWY5jo53PdW1cz1btavNZAOP/DxqXmjgzbQYChAEGgjNu29V5sRO9uUfi5/JRg+hWMG7BduN3KfcqzNi2PDpleZeULdOYzXhnnTq13hiukUImKwjxy054WtwVIYzG87O2BFNfJ/wN7q3BJzqcAAA=="
  },
  "supertonic_samples.py": {
    "sha256": "fb4b25d31867306b9966682c83472d5e5c91c97c1627b5ee3085457874fe31d1",
    "gzip_base64": "H4sIAAAAAAAC/7V9+3PbRtLg7/wrJsh9taRNQnzoSS2zp9hK4m8T22fZ2auTVSgQGFCISACLh2xGp//9unsemAFASt7dS1VkEpjp6enp6dd0D7//7qAq8oNlnBzw5J5l2/I2TWY9x3GuSj8J/XWacHZVZTwv0yQO2IwV/iZb85x9ictb5rOwyv0yTpNRmqy3Q5bzqvCXa87iJOI5TwLOsrWfuL3eewLNZu5k8vIcXhelv16zpNpk0C1Nkq95lZTxhi8WE3cKrYasSKskjOI1h49BnG1ZlObs1ftPbu8n/PDp9YUGY/QfrbLqOqhCfwh/kuRGwptSW+6HLI3M5ue9quCsvOVs45fBbZysBOTJ1D19yYAELKhev33LzpjsUAAgeM7iIl37JQ8Z0C3O02TDk9LtvU6/JOsURkGIQYqkKjlLM56wd2/f/m/2hcer27JgSw6T4QgzgSHnPcZuIxaqzoUkOB+lRTHyc8Drnh8UehlGM/YZujA2GuX8Pi6A/sz3o+CYz6aHk2P/6Ozw8Hi8xC8zHgWHZ+FJxA8DfhweHkOfdRr461EY56wG6flFwcui94pW/LYss2J+cLCCRa6WLkzk4CmcXGgr6HXLgzuWVmVvws9Ozs742fEh9w+DceiHJ0HIz85my+OjYBacTmb+yXJ2MmN9pFbJv5YsAkqWHKDEQOdNluZA4QFwz+VXYjsklOBRE3XBkoULLIIEyVJ2kPnl7UGZGvhpiomJ0vQ7mkk6QLv7NAbu/W2KBPOBLWAJFQhC1bkAHKIoDmJY+fWWAeFWLOSbFPhM7AlWcJwMQClTmF5RElcUGefBrd44rgMAYV/0esDaOG1sE/LIr9aCnsDfsF7Qnf3J85T9DK1+/3Dxm8veZdjfX0P/ELgAhkG2n48ZcHTRi9IqFzz3QTCu4Gt4hexSiP0bwCee3wMi97BReT6Sr5kPe9dngZ9Bw9u04L2i2iByS75Ov8CAG//r6D73N6PVkm3ipEKC4ehAfgTIR5ulyz7ewhe2jjcxEDRI70FqENxi2EvSknF4sGVhDmPnBwGu+9fy4Eua3xWZD5OR80YSCdSBMLncNvyfVQwjMf41W8MClD1YVmj+ZVQlACyOYh4KdJZVuOLlOdtwv6hyXAiQF0Q/6FvyHMiHYssXZP+7v1qtOSLHl2l6BxsSuIpdwAyBKrC6IHYiHwRSKBDK8vQ+DmFWQBiQCDzP0/ycVQlwk+Bctsr97LYH2x/WOs0LkDFb3PMggGg8hT5wD8w/ileAYogiToPG0TPc4/xLDzpK7pCsg2/COADIQ4YE1bsIeC6FzkMW4Wpt4PN6iMjfp/TcZW9gp6NcTmA4sXsAAWATkG3Bugq5GCcl7gZ6Xb3++18KoPomvYdvIEAVT4YMZWBS9jI/DIFCQz0llvnIhkNi4RCEYA5sEhclbERYOBwSmrvsSmwGIGCC7MBg63AQwzzsAboIFIaAzbPx7ziRGACAnEZs85wHpct+w8kVNIpm7U0FW20JnAeyFPpHcV6U570gXUOb0s/LKqP2kq5iCsEWeZOh9KCVK3jmA5X5euuy942GcdGT/BQKwsNqAYsIVGFcsUl8wD+H+VZrP2e0AWHFBM+TEgPJEjfoHHKeDWHXrTeg8oCP0wpZBhYnw21V4o6CLqRNSMpmaQxrdgvcm6Q92BtC447+cfE7C0AcxSS1aMnPQTYUZbphJNUKtYOQpv46B7W4JW0FbIXKuyi3sA3+++rdW5h8lQRlJXS8/wVpFNxWyZ0k+S1Qi3HYY7AxQoIbVrB3N7CoPkhTmBNhTq17fwC6iFMAHHJXnNPcRTdaDu7fISVLWFfJ3CRKJM1sdnXRROn1QF1smOdFVQmL4XlSZQA4WBVCuej11LN8BesBkkx+B6rdruOl+ir+gQduVcZr9fSPIk3UZ+C7W/U5LdSnXAMsqiVgD8TV74qt/ojrL7AN/dIP1qhllIarH4kWqJQAEfX2PY5LL8pthuSTzy+Sba939e7Th1eX3ofL399cvXn3li2Y81yt6/T+cfnm518+Xlm9n2tGqN7eLxdXv1xeQdcHUo6OEk6eFk4umlvOnDnBjC/PJoeTw/Aoik79E386O1sCyDM+O+QnfLqEh6eTw3F4OjuBFsvJ4cnkdDkJjmdns+lRuHSGYgiUcZ5k7Br4CdhU4RH3T4PZZHJ2cnzGT/3jYHJ4ehwc8jA49mfL4HQGNDg+OZ5OgtPTyXK5PDk5XJ6djadRpIDfc8TZk5LGwP70dOYHp8en3B9PT454NAaaTo6mhwD6MJocL2fBeHZyDDicBLPp2I+Ol0ez6Gh5cjqGJ8eTQA+Q2oiPT49CfnIchqf8NDwCDMPjwD8+Pp0eH48n0fHk6Gx2Np2cRmdjfjQ5i07GHCY2O+bhFJb3cOkD3MderwcmAyMG94DTiz78uZ+D7i3Ka5CpN+z/srdo2C3onwEb/aA3hPvWB6sWVe6cEKSHObTUDS7yVYXm7Xt606dW+B/s3CCPyQpZeF6YBp43RCMd6Aby3iOWXmgoH/wvr+sOv/B19pNqShAHxuguSH6cBw1bD+iAkUF60hniZuAL3BtDdguwFg6KK2U4gXB59esbBtqqgm2GlkcO2hQFzuac3fFtgTYSaGpYhwJUCS8cAwVSoQsTkw3IFzQUPP4V1GMBBou3ytMq69cdbIwdYSM6T7wfoXdjzmU3DRxh2XZN/H1MgrHW1rXNCy5UjnYhKguwyJ3n0bi2kLuGUy4OyuIEDMRbUIBovflCuCNbH5BAJ1XjkS4pDpwn1tiRBjeMKNdw4fw2dfQcSY2wpQ8mNbDrORPaExXmgakpnzlDGmpEqHVN8VIZMYZONLowYjXDn1S4Pz1H9CTMKfLkmRij1xAqXMGs88saysQdK8x/AfdSGKSRD+iB3gfrD+1hYWkEPtpWhgEJ+DwDa4CUFWpwMDmMoY/VyD+hzbPiCVe+D/Y5r4cClKrEjyIQrzCR500avQyyNuAvNJUSlLaUwsQQRgKj2XhcPxSo/YiBBLLp/lnBELhHhA1zzv4Om99PDv7bz/wEnRXleAG2k+lYDriXNuUtmk/d1Dnc2zPppuj+lTC4wO42ne0fTbiIJu8FWaX3F3xGBwF9put5DNT6enOOplsGy1WQt+OhjfDjr5fea7AWXl1ePVeW7HHLjPX0yearlcWPabqGhVEu7gW9bi/2T/664M3lvgjukvTLmqzS8tYXftG7Dx9rpxb+RwuRrSqw8cHvBDsY3dw1H0kLTniIAY/RS3mSCyz9ZDvBz2LZk+PT5hxe8wDMjzX77UcmgYUYUimkFmPk3ZMTKjzq53Aq9M+qsi3WFRr4re+04yrO4JkiipqP0GvqZNGxIV7JPBaOJ7Y/ZyJqwRIRijg8nIzH7Jc/nyGbyM8cbYodsnHsjltSvQCti1GZJS+/cPKlyKHBCOYzuRp5a4SuCw1sL7FAoC2X3JZcepUWcSJcIAZsykE4C1zIozrXHlBh+ehIqzCmKCt4EOVzVn4FunmE9vMuEj1v1gnaa+v4T9gm4Kv9+7v3Y161Nu8H4UDiAOAWLsF0Y2P37BQ9ax+UbMjev/oNHcgMXaFdJPqmHWuEsnYQ6KjmIbUvf/5xiEu2SYuSHZ0bO5EJ0cZKGAsk3NNro0IaFEAHBJ5HS1sC7h3gi38/At8UJwbgg1sKACyuHSCkNzmGR/Rpeoiffvr13cVH58ZQErKVGIH7OUbLvNo6Fh4Hytuk9jtE4zgS7V1hss+by18AFPSvXTQmi77Z1kWV6qGF3Cc3D5Z64VRlNDpVskgOgEI8LugUADZ0X0EG/MHzHNRDGuShKF1fexIiWCTtV4YuEhl36fIP0HzOoMHiiPOD+OiC91PO5XM6mpAfAYwcypOdHjUU2NtIKoAC1qxGF9wx+i6bWzOUPfZMJXI+SaiEOXo3c/ZQUBirL7sPHo2pIK7QaijcI8RXIeLGJd8U/QbhAAtoju0eHNwFyCjSDXu0W3bgpnBiD/Dnu/yRQn2waj7OuETtOrBASCIuFMGvodtNEx2JeCF8WgqSqzXR5hsOgq+fh6GAiJRBPB+VgbDkLKnW6waOJga7h4FWBmv6ct/29cb2rkowPlEGqj29f88PBu0x8D9ABcTigiRXn/AaUKQeYLTa8zVgJWmF7emYJSm/CTC0fxZcEqLfABnYi8wF6vfYMUTB90MzqE0whxhNHnStCy4a9eoG2GCQN4kYgRiFZqf5pM0YJhHICX6aUcp8242H2pa0B/SWcA2qtan0NeBZyfq/4+tLnMCQfYT29HGA5xzQ4ttn3dgeIF8AjDl7LewpeLJzunpGIoZH4Y8bGZTqWcxUQ6NWzwXZhianBdLGU037L15o0UvtUHM1dVpbm+G3Qk9QxNEWC+OxxpTezXvdmucKfKo42sJCwHqut6RzKIZAkIHERmxoKA9kMLBixbccjRWNToeeSh6iKsKHIozj4UGnfLULJXlqivxqnY/69VmbctmRBzACg/u176AL66jogFO7wg4YVR6Zsp7w3A1qtNU2GEp+WeZEyCFBH+DEywFOpeMl+yub7FWID9jsUZ8D+SwDS5scC4DKVzxvkK+BLcCfjnfTqhGVqIcpGUht+DwdN+Cj1w5A98CkFgpQkiYJX5EnVANCmuERhBsXUZyAohZ8R3EhIhQ2GLvTI/bXBatf4bdDd9/ImTm0cokIEPLDIesDM+Z8Iw6uQ3GGGW8yH/jjnxWIiHI7eBJLctK8TTHQzKme7CeLcu40fgIoYfYtVEJ/ykM/S+FAxNKUMl/jw32rb7p+bbIRZub670AIPTIvXAp6LIvGwx8w+LQbBenOdRFFITICAPRg8gxkkP3RA/JWS5M4rPkSSXO0f1/onACJGkoJcJ6ObkByKc+psTlk7sJ3C0YRKbm2eLLmRmB7UVpOP3cwsNL/23z+OXw5+JszNPsOduMkgSt0OsJcz8RGSFQMZnl1MMtDQJ7w+HbgYIkpx/ATVVpEMxIlMhToMJSCUH9dHAHNzs1cAccGuj/IRjPYpLDaMPE6/YHyAfAQBGep4dm0oNlJB9vbLPdv1Ubga78w20nqznFfsIk3Ho/xf/bDos2Wxnv8/xtxBJl9b+mIljPflOYUafAwbqXZ4xRx04LXaADPJmfTvVgZcTNNtTEuDwJ1XdEdg2G1Bt7patajCJNt0dCfhu86FGZKr228yhiBaXMZtiwp76TRdr86NkattbK0ex5sQA2j8j/sQen1MuD2Gr4EOg6W8zT4FybnC3NtKdCQc8p5WeUJ8Yg8sF3iyYSfbz3Me0lW5W3/S5qHc3Jc8IxW+2aO4xiJGBj2kskeQxEWo7SKdAP7nF0mq3Vc3IJaWWLeikiDOEBxH/vrwsW8CYQIQ8RZxtFxwjHdnB70nc/O52o6noT09+wvg5vHmv1BJBccU/9AHl9/9zdsMj3+XM3G4+nnKorGE/w7iW5e/g9nqAcwyCcJMFXwVBOXJ2GBCWl9xzVNRXMKGDNRzUHa8bwvfEZbFm5y14hLykdF+1n7UdjumeVp1HpYtNv90X5UlK1H9+0xeRm0n7mr1rPY5a1nvrtpI2w/q51oufpAQ+TIfkO1glq9vhj9H3/0581nd/DSXLtWtM1aEr2bBHh7n1iLLb9MWlw0PJ8Tpx3S3xmxUUBstKS/foOZKA7AxnL/CKOdAoX4hzbOUKjWufAgYBPpJAi9lf7OeUYgg5KhDQ9bmOdiM2K+YywyriLMGhD7SqTxqnTO4iBYU3ab3kwwKxrU9kxyPwZca1e877yi6H49pOikhIbSQLbBRtkDYnMO9sH+qLL/QGzzTVaqswxxumHmgizY9Y1WJZjmRmmKuJvEWGBolOs44VYkEIWEdJJFe9GubyuQZNtf84SkGJqxYn44DD7BEQhMQ6K25hI5F2Cq3IEdi/EMdDro8JqIJ0A+0D+mshDLRuBt6CBbgCVIugL6YIyOrdct5NrREbDRwjhE9bxQgF6yepovxaaCcTqDTXVvSY/u8AtsK/+u9UYMN8RJwOAalHjwkk2a4Sd8/FeNWzHYMxlczOtOVMgwRuhEHPo21DTiSbXBM34uRriew5A3Axy7W5lRjGTaGudmV1yuxq+bTs/A///PHCadY93sXfIdc2isZnE9mthwxJ51fRB5Sdh3mONi7qSFrc1raneKFtBgfmPKXQFPCk3t7fb9KozTOWYSDplhtJLoHLJNvAaZwYMUpOJchGVJnkJzLUmvqgyP4AoGIi2UOxRRFfnlaQXiK483G0oNlrm99Flm92IOsJahIq2RKkIwUJlkPTmFAkP6C3jg+oWf5/5WID5kIZ3WwXNCbjYduEGabaVIEgnNC0xP7+e4rH3TLn9hTY8dsAkY2QMSFH0x4oAdHLCpFsUC3A+2fN9kMEB/7B6xEcO/LxDJIC368A9IUMqqQ/8XvmbxUMAYDAYwDwriGpgbNhIOfj2ntjfsxYKGab4e0eu5fn89nysekksuGqrEQJFW66Fbaa26DDnI5cVsZnnES03nJNlo0csKSHdNffBs7ZoULXW6qfXqRZbJgOZqnS7B3kDw5wxz/JIKqY6ZeMVtmoLjC6uAmUrlrZ/QeS4l5/1HmQFWoz8Zsxfwj5wmLPN0rLIZYH6eLMNZiJngeoBfif9gTEaygWxfBJiILdiJEq7oWPrAhDMQMsykoM5sVkOR+QL9zdnAGhL09vINDfvWMcB4dZ60MzfhDxvNCS40oX+bL/0I0zQtUHvpIPqrZNPi1p8eHfcxW3lOScrEJ8AXgh3CeIUVLwuVaO3K9oKaVH2CXV3MZu87+dKh0wmUEf5mbp1VLtdpcIdiOwZ0+2t/swz9uWxJJ8WwytNDjALAP7CFl47T0HsCF7fKUNr2CZ7lkcn3t/yr+ARIijni0bSnapL6GCQ3pqrlIDpePon0a2eFGRnMGb0SRiv1AZwcMJpH5Dni218uL147arfK+q2FkUDu5lXSl1CHIntTJEtg8hKluotcHvkQrTb6OAA7MYRXylys/TY5yHcL1sgYb1qUsljosh28ijCBhrL3VFqpBvvQAPp4zlagyh9Ug0eXvVJVYWTKZSJlVb13mxGoMM7LbZskutVuOoNTX1Z0BDEagQgJ+Bq2vviKf7PtAR5h4ynP1rmpN4RB4/pZB63rVB9Fc5lqIimvKE4T2GW176axhR8VdlCtHnoMCTCmKJlQUT6jxhLzF9ClqCujDCor/m0SGZP78NjUqnzAU4MAWX5Dp1meqsGyEsM0xCET5XaIuIP/1MTVzEfDGGdT+J1yPnjHuVQHgZxXIkRKlZFdSc4aGzHkJg0rktSNiYnnYmqIQx//1JSQKLn8K8xftO2LfyxRIR6BePifumqjDyD/5InYgT16xMjRe7/2ZeKGdk71Ny9GowprOPSTjV/c1Y9U4mz9ROTGqay8uT6IFyVYMDHM4xePpaEHU6lkLYYoJxMHDfOO3P+hKp/yhGVhe8+AQq3lfxYHaSFH4xRGptxKXXwVxsUfVJaExKcnaUTc6K9WOQaCuRVbdYVep5irDAi/eV3QySPKBsy8BpM2Tvw1cL/PVH4m0krUWyFXYU0kpQ+KAkiXQK6yytvwjSccRlGiZ4TehzINRXv3YvQRdEnzLYDzV7J20VUT74peL2T0uhXkunZevf90qVbgvVwAKfVxW5jkfipucKGOFnG2VNSpCjXjonlCKxDzhAO0wHXsGwhLr92Zw96dDK4nwu2BrxTTNSYm4yzkmoqDBAPWnug7JjM9FcWXh+6wFnJ1FmoMMLktwtQkrxs/QSuMgyBJQI8Dh0i4FOTHcDbL03QjKsmNcliVRWtt9dq57DjFaa+sHYt7aDl+jlqYEOwsc5GG7aYW7zpzc/odrcVbrGdNQo+Kj/lqi6VFd1ewvS+KD4oaTkdnqpIHYif3nr9epZ4IzWHvXy4/fXhz9fHNqye60UEuMIQu3aWqpq5OYeqhg+ahMS4SGTxhxmGPSVcPhF1Gs2kXyMf668AwbTu3nXh/IwUjZbYAlTzFa7V8lE+keySWSYYVKepVx+ZrxYUnAPeckvqoswscZ4CUuZPKeFow047pFhJkQtgCRmzI613Mx/aJm/r8SmAKZqBCp95JoAhI5oEz5HxKNLpaieia5EIkAhE5MBVIAH08r6f4oD49GpZdPaXvWjLTHP8lvBWb84tf1DsZ74MAcd1Zfy0PUSmvGw9II7wTYumDwQkKAottnX0GhhwYbX6hvC+TVazWFsT+uwQFCqbeJ6Uopj6nwhlVsxzgaKjhvoCMwfK3kSqDSVaCaUSgWErniHkeBsw9D5htHYmT7C6VbObMtN3hxivjSgtsAI96ugkO4yYZec32Q5n4RCdS1ot6rRc7zQgRLLHKQO3zgq5TXnvNv8faajBizdNteacHLPLb39+8fnOhLt4AdzG4g3WyQwd+0gCYrpEjiD9Ux2WVhAhvuUUboEpyLm7u0HX+1hUejVgeGOF0fgoURf8B+qJPGK7XhdMRXYVWrtmmD44AFZRuF047WXDHZlZHrAgMRYl/D0yP+f6mUGmP/ZRzsV9/yQItNZTL3nzD3SpO90Dk5Xdfp3LOwlTOUwyDGwhvPsA2aM99ueVY2F+mQAFYbbc9xKDBy+LijkUz+c2FvZuu77lxUoFTory4hdUV/BcqyrXhBtHKzlTv6+7QoSwLF985g+clrRNIMxAprCo10rXjc+fm2jFaODeN3uLQq4j/3NUZqyS9uhUAAOMrNssezMldwxzW2E1mwIE9jBFdL/KRb52b3g56S7cjjDctNCTAukWNwnNHtQRJv0m1YZMSwyZOA5Q4faojGrLZ+GQK5u7hYefB0/4d49CdC37oZ3hg1+VfW9cjqXs8xAFpGwGbjRskxVAZnWtQAuNchdgMfhP5j2YSpiV/H02qmSCBGFbDJ+ngkJche8utSufF5sy172SRQN5xZFYcqFt9Fo2Imk5erTfpoKWI0OFLca+qbu4nGAj08Hv1ro8RIHNXVqKBsK9BeIgdaoDGMl4P449KXhjFyhSksMWCWcqMcQ5M+9C9HgX4xhY3R6i/dAijurk5RaKTMWwfI+79BujBTVvf0l1FUoFoaBe/X7z59QKrN3+9ePvz1ZPHr5+MO2wQYIWG2YOG/10O1p7K/GEPu4Yxj2fTjGQzqrQrYSWL3Jyib7dxY4xkeWnmgbHjyYxitUjya6sDz9sdJnar2ohBC05iopUg3iLjvvvw0bu6/F+fLt9+fHPxa2N9VHhj0fDsxEZFYG9UNE7Or8mUMq8ZIQG2NPcFojasDeuFbXsNWgfVnZu+I99DtR3WcYLEnkh3qc/TjpGVeCnzsRsW2GtRnIhhob+A5KzKFAQHSAZtroN/neMxEYZ3ZU224WZQYlXhNoC+ScIYelf+Gq0ETFKPV3Trgb5bKecbXyTElSrkVF9GpIbGaJbbUIPCZ5MllR76Dug6NHeoJk3I6f4O4/337B9+vqH6Uft+JukxyvSY+E9dCU/3SIQiNrtpXkRU4xeHwB0Y6lKGipaG/WuHxqwyF8v1rvXWbBoLar2vd17McmOH1vE/dG2H7MFRgUrwvgkVR4ieMMMTJS2K3DAbyktZEFd4h/88GnaD9nxEcM0TtyyRNCNXuj5RrAONphLbH3AzQjYdURc66cJ4CsWrltuS43TGO4IuIlzk6XRcjDt0Oda02PP6Rrb6srKiK4yht5EinH7QCGnUhw8YPZNiwlh90el6fHNtZAx0Tn7HxGlpe8+a99u0URSv3543Eoqtqvn6XrVmehrNyQhlebB3taRRCJoTv24Ewm6eAqhjzM8DBzYpurDWRhl0jYHn1rdo3qt4pgJfs+eecGcDog5peEQ9RXsFUlvJT0Za2+mKz2Iy3VDRBYb2SFRBx/YOkqZoR2hL9e8PvkHh7ArhCWLD+iiJriUx8qKu3Tcvx6Nd9wWlKOayN6/dq1VDKxMT6SlvcEOfHwfAMwxQFymeIpgJ+uqYwN4JuLfOKX1exxDe/v7br6zka4AI2s3KwtTir6VDtAS0T8Ecx/nAsQAZy7/qK2hkr5HMQVOHfvo2UV/k/yl1glfFRWiThDqtQtkG4qyIqjPAL5k1rADhRSmpk2QuijnweyctL2soIQ1U+oXq0UproUN//14BrdWSdR2W0EVK/YhxYHXEh8cByL1mCAVgYmZBRhEmxFBO7UXTQ/w3AiaR87tAkjUoL89lQ9g2Gg8rDroXoceukIbmFYwwygChPj2sD+T0+aKhLdVRii08jLq2br1KDgMVd9/h9VLOH77zSFEYCW8B3uu4wSBypMm0zqWkiOi8iV2da6pYT+SIxUkzd1em7TZrz/dYQAQAjR97Ljc204HZ02a6b7CF/vP2EC10k5UNNI3UJng6YAfGYlENXnMPQCtrD+giLgVOl3shPKxi6lMT+AZoYInK+Fmboy4KlmQzrpKCLQAfwTF0mu6AuAUWD3RVxSvayOJKUSkm6lukf5DXeBD/0a4f1aaVlOikTWwbXhyJe0Q0PGBBar5oxdp03p2mS1IeHw66QKmQlgkYaNXwyMTOXrRja3RoiFhQHtWgAxf2shXMG7GJXmxDTHQg3RSs9jfkNk/jpuawc8DWiE32UgD+yias1h3AasZA/0Y0+rVa+zIdSc1jLDldOIDXFINc4Xj5Tc6jii7jxTu4Ez8IKiKnVHr7osRaSqm8Wi2m+jT5YS1thrSnRDKpSo0ttLIbNC1vglqL7mKbAIsjKaUAx/fzOidkyPJkJU8YQZ6tMPBJX+20MjK+sloq1AXymyWn3Nm2YGtfjfk8944uaVfftWDDEK0l2eBBQ7TVHRtOnyng1CQN60/bGEAKt6C75vPQE8mT3ZYGjWTluuxPpJVaA16lWMqAMCedUDpSR+s0L/JvRLopSWRD2VDd+82+3qjvsBWquxwthX6j917Dy7CROu8k3aesWk8fujdfksbF1msYWsPutoK3NktopzlwR9M97LMHdidL7eggl9BQsDsagnzIsSXSG1oaCwkPulZvF4bkq0ko9KXdsOHOtJR8xz74j9rEyvD0DAG66GD5tjHa22dSd4D9Fw5SaitaJFQqY9owndl9TD+qII9Xcj4yLpQ05vTQgdHjrpOVDqJ/z96pPEI6zyfPLwcbm64453UkD+0ceXc5XSSOt+23dZSqZOiZRk9cCN1Vp8nhvemYqk7uA+kyutFelNnqS+lwPFHdi+EVukVMOXxRTDcmuU31Uxd0ACmvx0Mm95CdMHgzZDuO0dpXJOiyAWnmeYJAClS/IxVxqPITjQoS2Us/sWtaJfYNYMCbRjewWgyw2mIxnklU5aXy3CMSIyGkgn0mVlrrYgq/MeJiYbVvkh7G6ckTro34lRIXA9PAJvp+cIlXlq630kSI72NxoEV3MayCsG+MaOE3aJQE1KBwhlZTJIuEbE3aeK7VpSNlHd4HBzaOukROUPJLDp6Ch0daRm7/kC4pnzcitY3QRclxzn6+pTJBmBqe/3v0IxlfCZYrPsOiOm65yfT1yLKbK4Yml1ATmg7ew2qTFX1EASvfMG7j+UDtWF4AStssKRfToQgJeYmfyEnhWJ8xAbx5Jm8kRqV0BrnGWh2Ny5BmoGiCJwvPuFJbs7Z5oVDzJiF5vbRxmVDErJuFdM09ZXzZtwvtTjIYAd851ghm4WizoPSpBMo4QQndVU6qKs1EoZQLMjFEhzJ3Pl9h5TB2GVjDTehiBNENLxAxgwkdR4+6yKC08jgnY9eFrljmWYr0qBEIbowYwEsCLmsO6LOuWe/Om1L386sfGKLKk0jszu/ZB3FEy1L1+ypilzElyQO8TVmG3BpHOyllMgGHYpZdrbvcvfdHtB6C2qW8gVb+lyVgANMk/SfsyZ8OxxPNda5QrpT4srmDv7Dx0P4pZK0I/wr866V3MltdGMpJHAGVrfPwGgyefxu5/6q1cegdRzYMlwYp2oUPPwGtL+mdWu13whRQv0EhOs7ZgwWPykfSFDcDCFpe3DLrMluZwy9+2cOjf5H7wRJxAetIJBLrAztOyX3QQGT5UQpCA0J3b7yk0Ryjp/zJJ4cUyYILObRLkT3aJ+Kt8Fz3DFsPYtYByOwhCdTgH5VuqjVDx7UkFK6vdai41wKTbev4h6iZtC9remFrFlE02VGcQKU7m36HFSJqzte+uLwECYMi2h77BaMicvkWdb6VQWsM8kwDxVKIHQpWcVtTv1mn/Q6ZinhWYGa7GIF+R5RFeaqqCZo26qKMtjJJxmzc/DmOVmtP5AOh/yUWXaTnGO2MhJG6scwiUoxSZ48YHeV9ns3znzu+pcCzvqQx6rhlURTEkbKq7wm0Lxi992XO6KB1DGT4TOIGvznpEHPyINHFfoB3JN6Nl8bCondnLHPdxlhvpF39zVw7tbVFQTCSTT0xUfHzjac3rG6qn5gVlpo11TXa8xYHdzfXYFsMf7ADebF9bJ5HtMxdZTTX0e8NL2/TkM647R8GGOm4OO3RYV13Vbs9I+n2qN+FwpMy5dqMMAUx1LXfxiZ58ULyoZ0KYPEiFasDVtdPVVRIfslcm2OazGFQpCGNOvpYjjJ1sZ50lV+oH8OyBmo63HLaRspiG5JwP02v1pNE/jbQqDP2zvSxlVmUaXmsXxmn7Q7+9hiuyI2qARaaywp8RzqUqzbE3PDUO3l47h5Hj6xg/Va7R9lM/+JTMTC4KFpXxa1ZftmND6XI1LjUmvYFaa65O4XRN2A5mr8Nhpcnq60/d2eI37m+ROWhVkyPT+JjeFKWPTPUCse+WExdBe5RcknT1RRHbRUYnIWnwpLULZFpDFJfvpA3klKAEXSrjEeKjqHwWMbGOaMOj7TPoRNAhOpP687GVdYa3ksz2y7K9RXORpMDE3HddumjF+x878jk4NkYPigA5LqN8BU+H3U0MGM8rSOYyPmc/1xHj64fYLDHG/aAPyxQw5gfIw/8FzBgjezjwYOB7CPyxcpmP3mHx8JpPGuygXGkq28hqWPBYvEMI5kuAV0YV4a+FF3sc9vmVaLdTtyVuKtz5yV8wuLbYe9HpsH/oK9SmY9n4aOHGDzgn0f3i39v1ffsMP53OwCZ/s0jbYATCXZZ0oR1zANx+YuRAUVnIUnmAmXDdKPuAffgeR9RHbSPoGXtnbZF6zthxD5q3YFu9BK3OpjateOWcUJTHTXpJApbJ7ejz43jKguIsuDq46VMnSbVB0kGBBFtXDRjY3RFSYIaK6Hp0hiDp2xkDXAo47femt9jhcai43YRmSUqr92Q3+ybMWqolPCJfLgsyO4nCGj3N3UCno7WDZo3k5nn3KJFfdJtgDVq37ROQredkPgBRKM7nvwrwW1D1sgaNWVMHQikRmI5ZA4QHZlro2/R1n90V2PlrxcPNfLwkPCE5vjP3D39r51h773BOAq/0fY1dl8k4m9mDEwupml1MvlbEgtaUwDhyQfGgiQRsh0AxE81PLumA9+50qiiLtJBFaPQA0yvSYDHRDFHy3q28Zo0tmwdVqySdZzc9eWvuNrBj30lEJfKAsEf1MT0P57bJxT272gZiex7Aor18aQwAa6FbXWjNvlTNi/9AtSc9ECHBUk3qguXzS43GHQlvGqWNZyOPZGPzhEFlxr2qbmuu5vr8Yzmu/wb3dkSPNDX+t5Vb6wcHdounlL66FrR1fitHk97J7b1bJzjPdPWE5YKGCfJlY8/6iSKKpJHnWWBqsUOf9WLCCanYWNYhwN4I2KMpaWYhel5lCbteRi39jyZK239zoDg96stWIkb0Mhln0LcAxVpEb8i8IbCjvJnBN5dyQ/mbwuYG2ZoXtfyiko4ZYXOjl8fULS4/PDh3Qf1mwIwQ+DhRbFFCxZ2nPELDi2cJwNx8AIde/8PkSoXScp+AAA="
  },
  "audio_quality_check.py": {
    "sha256": "fc4f16414bac0556db53dd4a378c13c50032472d7d094f6d9c1614d809369003",
    "gzip_base64": "H4sIAAAAAAAC/7U77ZLbuJH/9RQIUxWTtsT5yO5WVl465djjxLex1+exN3WlKCyMCEkYUySXIGcs+6bqXuNe757kuhsACX5IHiebqakZiWh0N/oL3Q3wt785qVV5ciWzE5HdsGJfbfPs9xPP894KnszyLN2zZ2/es9VWrD4ots5L9u7dJSvFKi8TmW0U41nC8qKSecZTJrNEFAL+ZBV7evk2nExeZqriacqyelfsCVjldZasZSpC9gLQzWZclVNWbQWrhKqEg83iWkkBdEoxnzBWyAKoaJxrDvDl7HYrVSHKKDoLz8MztqpKnqmUV+I8ir4JvwtPGb+BwW/Ds/AUMdAaGa8Tmce/1DyV1T6m9YXA4WxWil9qYETNrhWA2W8hfZvN8roq6kqPmcl2CNYxmXThJfDNUqkqlq9ZfnUtVpVit7LaGvIoBhJKJT5WU1aktWqWP7nOr2KZTNkuT0Q6ZUoI+ELiBmGVTKKU5XoPWmBrKdJEgUABH4mKZeIGYGCqhLEknFzCSLYSJ4Cq4ierVBYFTrzlZabVCHP4Kq9VJVdsK+pS4ic1ZVleses62eyAGjC3nsisEmkqN/JK4uoZ6DDjVV3yNBMKeAC9s1uwDibKMi815lW+KzigzDP4vuGoQFjJxDUXNKlNJj+J8oFipMJVKYtKM1CUOQoQfneyQhtRRf5BZERHhZM3dbaqao5yO1lxJYim3GR5KZLHaHpXotSWusozwL1CSCOnHEnflog2CyfvwAxbTvCjkDcg0iyHz2tRogxnqCxkaVdUIXv2/vlT1PNWJrAOIgISmNQKZqHngLj+ELJXqMNZmnN0GhQZGOyMtI/iWnOZ1iVMKEUFwmkVUea3sLzJ0zcv52xbVYWan5xswHzqqxBEenL5X5fv3j59fdJ1BPTeyWQN/LE4XtegGhHHTO6KvESpw4pJUmoysc/KDWhHCfsdTdd+3vFqaz/nyn4qG1i1bx5Wctc8rjO5giUnvOKakwLwpPLKsvEG0dJAtSdDNM+fZvvJZJKINZh0JisRa+X5NzytxRyHAzZ7wtYgyYr9N3udZxQVGIM1/yhE0QYPR4vAGJA4Ad2v0WlWgu1Azsgb+4/Ln17PFF+LEIWGiOSaETHUaYsef0A5dZnRQ3qmWWOR5kazGEwcSAMAGFGKoVR6Tb5+HjCRgqkSOr1ksmYfrWvOVFXSSjF4LODLslnmey1aDa2DCdr8SdE6AZhruYPY9El/wyjerM8OgRNFrprCZsD3Xr/48Zk3paAUhIh8naeJH4SlKFK+AoD/+5//hXHvgddZbylCWGECodkvvcU//v63ePnI/+P8gfkYPPSmDvkpyI1vVASz3r9++eyn5xeBI4cYgDYZBh2/cTySypRt9wVGQCVVK6ZErkhMUzSRVlavQOC7emfiUSIrAIS9A02ABJcI8BswDgp2U9xYCgjR/AqjsqXfSE58xEEKwijoFOSnNdYwGEzNk5ZFIyDw4ykEn7TeQdyJGMRi36IL2CN2NqVHGi89oGmrHHYSAF8sTpfsYTMfd2FwaBA4zzbCR+TBkibgiOyNtAZM6BZyidgiJpsZ1+0MQ2Iw6XS5uMZJ12NkgPsepR7Ws2btDkyHI0IOivA7ww4Im7EzAKN/IJ9GemaE/SYyWjEwwfQLmJYk9kNADqkekFaonLLraWsjfZV21Dlli6VRZ52RPj97sBsICsLenJ2CJ4HlidJ9oOorsMmqbp7dEQoI8WCbEneQ61aUEGEk7TzX9Lcvm6gnG7MTOqKPxsTcVRWuGK0Gh/DjzNhoKzjIRrJaHGLqy/RQ1l2arQ/yAvOEoXV8Hjwhz4dNRYDQOlL0puOwjffGmI58hGl6jePQrWc34NdHwK0qAKynlQMTtKIAvKOxIfDdiFG2qkA7W/RsCOQb9XR2TKcibRR4VHX/DrVZ7/j3qexXVxdu4/+cltpQMKYhNos6SlHiV5Z1E3m+Qti/um+MS+9Xdwgnyo7I+rqVdSvVEusoJXyN0JQ0EVP1ztdYQ8r7lB90UqFW6I78KK+hWbCc7n7RgrelzxDebigtNIFortBm6cPocFxCQdyAsJNeCgK+br+0aWmL6OFDvdgBakipkHAjLwegk4OipdnEFbPLE5tVPmZu3vrlmk1XSGA0DiWsSMsCRywpLKpaQULStzY60EVfIjkQogpL3OiSoFdnQh4oVlssN5VCuqGhd2dSVFuhxZBNVspXfFdA5U0FClTq9I0kPsf6D3AXbe1yIFU9P2U7xd6+umRKl+rAAyxgCnqBtUO9yD+0ZbvRBomHJ9eQk0MOQqW9W8c4fLAf3CBdcgkq/hkN9wLtwfcuCZQR6A5Wxq4EK3IlK6h+TZJ/y28EWH5WhFzxsuR7u+opSzCYRDBCldB33wSWA5wTZonc4b7hMGBw4b/FfErGthzOgZzunNoLoBl6qra8EJi9Dh+66cpweU+x26IXtkXSGRCE6n3P1iXfCdPEygTi3eXYr9hCnQx+YJau6za9+KaKQ+J6eJUKnunR2y3Yma8hpsTflJ0aJEhKu7NZerMeGk/qUjtB1AE9cdVIgNu8wGSZf9SJdw1B39X0Q3Yanp6fmnhERoQlhA+WWoIpYpLt4AdrxeewiQPaIKDMXT9psndIRzsTEFCz/EstBS5moStgWL/6paT/O5CIT3JZELa5qvICapiH7DwAKmCNp/jFn317CisEblvCyFGOTT7N+9KE1Q26oeUWSC6nbSVO9QjuOFDCqVhzBfMFRBGBMvH1o0ds8YJDcFsG3dTZTqHupF57v/YnURrSRKmbJ6ExjqPBkSEqsxy7Z/u0Ui0prM6meukLoqTzLKhoglFuGiGAsKm5BHYIdTvgRCl2rIfkdAWks1bGhhOUA42go55CyQI2pOlBGJWpg7dhwnfNCJik6cFXUaRHUcfeNemJU2lhcJc59lkWk/GcxiNZxErATpHgdqRX0uGkmzB4IHUHfiiqHrj1TWeOlu/4stvZd52SeFwSri1qzp+wU2NFAPiDKx6C1S6BO0LTfEKXg4CAwflKaccLAhudYNOgbk8DSqhiCIKfRJl3J7EnoP/w+++/xyXpICU/mTiXYQPNREKN6wwEMMCnYUbmQ6Ykk7jZryITVxfzGRR/v7PfzuYmHuJ2ZjYcYA/+4Tau2YSd76NUkQ1yKS83QlWxndKXiY7KXfJTTQDDc0D5j2vNT9iZToOsLaZ5hhRM3CV/XQytYqmjGAyieq3lBkhqzeu0ihqvKnfK5XMYNh8+PLcKbHr0kW0kUMQxunA3VQ1no4rXwMRmp/aajdkaxRNc4enp2REskAYoeQWWbbOPGLJJRR1/SrcanFZGZEHfHcGIgLGVjg6aMQrtsTknEM0ZAQJBDEw6SWLBsbnu0O1oH1f07THiCN1YgZGMnvuY2bWifFYfdE7I9VmROe2AagATuJY6anK4nR3TiuAgP8zxqphOgbxDlYMTUTBIjcUXTycwMOxu0e1wk8jMu8mSA6KLingkxNlHDjAGnRgZkVWdIFf4wBkHaXSG4bvLjbY6K/S1yeoBzox0yoeu8brQzZgDb61gVLNYP7lW4s7T22Zssm5n+WakW5vRXjgCbIdczNofWksfIaFB+rVMBxgzAW/eRBMH1njOQDxYnLq705G9B6P0uOVY+jzj6R5KeiyrdI0C0s/BDVKuaxEsWW4hU8lvFRBJ81uGXpD86cXlY5tHQyYmyo0U4+VbU0tZVyFj1R97VRfy8gl8wByy+sZx9Le5PizBqgr2B1XGdHBqajL8jrVarcwpQgq5bc039lDBVGggoDV9mLSnL92Cbdk7b8rwwFBBcMcjPcsLHiWs0prysTqrlT5RoCLEHBA2Z35NwYYt9DbEOymtwdnNaO2inUhDWRBt9HgSFwI7a61VUfpt7ghkAALXZHEEOgmmU3XQuN+sAYHMKdVnzzxFn9af7lyUYV0kyFY/RQV5Rx7GPZBEr8HUaiRqP3ZB2hI+GnaIer2NEYimszYy1raCRgY7jdPeeCvJogSf7K547f2nvhOgL0ywzzqDf8TO7k4+Y8elUdvdnH1GuW1E5T9oLwM8mLIHP1DTASwHz0yfPAjueoJbp7XaRu/KepSrqtz3Gve2PDmkYFNMDwGIOa9lzgumdOA2HzTfhhX3BUSIfWO6mRCJsjupufaAq0N0aBnB/TjWDOGx5L/KisS7DKWEkMhTOucc50SzS5xGdGZtOVm4YlniyajK0xvhd6fr+p8KoQgCC0DxxG9x2saJR3ng7889PFG65XsVnyek3y42MJcuWURalQ7CAbx1y167quULA18wQkZ7I5Fo4zPZNMTuvrKaWHu44LXJEuI+vLMv56M9YL3wJkZortQHnUu02HQyNcAw7JkfdJWR8HSPiNr/UUJflsFNYJ3D5EY+oQ1oV8I/OF1Hzo5ap0eB7U4W2Q/o0Yc76vbnSvBdjMVZ9O1xwErsCtx0YLfCAuY4MGY2kvJI+C1KcSPBeGL0sIh6L8dn3/AEbRvkex9o2gFQM6CnXaF6IbH/80WFxd0deOyHyjsNTQ0qo+n5cWU66A+e0tzv1GZwOtLvfHRvzRjKIUF9wYoOtEbGEQLMfdFRtJ5bIYR00e1+M+kqBUxd3Av8/kK7p/CQ/FdJ7l4SJKz3F19HGJgX43T885XTizK/4rqEHWfJAfgK1u7uDYmeQ7dwwG0aM9JXmGBksQzuhWn5ZdaOs3SYjHNcBLsL88LrXGaW04U242WIWULhBwcCAbl4YIEOknI25sFVJ5NdaHJTh6vgXviOB/I2nXYuWB6dQrugiXKRu8p7TDPZP/Z0IA/v1HyHSZHXRCPbLRSzzpZ8HFVz9y+Jm80Rt1Tcj0P7JPjarUJ8XAmwjgv6h40orvRx6vw+mnFFgrUfyoNWRCiitfcZc0GfvgVhHGdYlMdYJtCjOyczvScjDvEB4a8nSklY57q00/YcTZBAY0ZbLpJmC4TP3duDMGbqfLqNGvNyo3z4czNvb0GaO5/m3INqdHt7NXyNnbCCr0y2Rw/x2KABeFpuajTgNzTS+koitCuAKKM4TvJVHE/Rw3e8goXEq5QrqFEtlrf89nk74S8iLV5YUOfYQlMPeZLgOois7/Uul+MVS8z/31AtgEOyFG7mfwCJcwf9n0LRtl7o0rpHFxpx8bAj5qWIK5gND7ewtMh7W2ftpVp9obm9tP03fePYiSyYxn9ZDJQNe21f3LviIFlhm8kHZlnPdSeKrOH0gk50dR3HRIY9F6g56yoH5chVkybTpc+V07o+QK3aYrmmrISlPjLQZL85OjPJbzO86j0r87xyNaRnoV2DWZrJPVtv+sr4JDQ8dE/vzUShq1sLcuDU3jgXYjPOteOwr33ZrWC98wG7fT5/yy4FUl3joTnP9q2liAxqa2GudUO8uREZrgpf88AOP9iO+IhDUF39+c17FRK6XIEN3MgyzxYe3quPf355+fJPf72In1/8/PLZxaWu/LwB7E+v3sSv37+K3/3l7cXT55dtZezIMBjMevXjX+85y1xON++wQI1buI+bF1pwSK0nnaYk3pkFPw3RHpRGbYdiGkAyVB/5Ilvl2CeLvLpaz/4wU3LjBcHkaEcEikxUYnDs8kP3hRayE7xSQy8a0GxjKk2Z6nZLnT3KWMeUeYmkjiZuJM3xszVZAJ8fbkbRlX/9ukJsXlewYjSxhN6UmAw6hbq/EHWghtkOcaDXMBhLoBZdCchHitobu/e7g5gKsY8aMvjSxhhQUcfGMCLXSoaQYCp4feuDKFU0cv/NRogYI0TUGF3ncdCItPN47HrWMGdpFYgeg0zuva/NHw4ZgVdn/AYSCTQBsIAvJxEdrLpruqYrWg6ix+ZoAGN38/qLfusMEDXk75Bg0/8M3Mb54HCgdZFDlo3CtdsCdsKw+2/DJtlk1DmO092nOWtT2hjfIrD370hf2d7vNdBAYCbtoqoB++9SZzumtz6aIbv9rjkba5B7jVAMAH12xvUmOx91CpqsPYIW1PEJGnQdQt8T7biEhwvAw4BOLTB2Fc8qs33VLEv6Lwp2XhAzWQyFJoFTHps7cM0NqSQXutGInZ8rgNs27+I19/SacKCzJR1s9dYT7j4ksvT1F9M2AhJALs4/OIaFja+85OVeJ5FdVKR5NHV/MIRP2SMoIYGCDa4NrhDfNRM64jdyo2kJbC9otWbHzBS+uMXVSkrTDCOpZVV0jn3iNL8F8pkewldGvL9jKtTbRNxcLFfNCzwNN9PBwkxmY7z0kt/ADq2PLtBi76ygtddVOfhmf/lNL3zUWXU+cgrJCLiLDRbkJXGMqUkce/PJYO/Qu9rlHixid/FRVj5lMWZ3NPHMbze9Kb0J+TqvXuDebB4Rc/jC13OB91DpaTAS++zaCaCNYrAWwBmpvYICP4GHwWHuzoLJ/wMuQdBMWjsAAA=="
  }
}
SCRIPTS_DIR = ROOT / 'scripts' / '6facce3527a8'
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
for filename, record in EMBEDDED_SOURCES.items():
    data = gzip.decompress(base64.b64decode(record['gzip_base64']))
    if hashlib.sha256(data).hexdigest() != record['sha256']:
        raise RuntimeError('Embedded source checksum mismatch: ' + filename)
    compile(data.decode('utf-8'), filename, 'exec')
    destination = SCRIPTS_DIR / filename
    destination.write_bytes(data)
atomic_json(SCRIPTS_DIR / 'source_manifest.json', {name: {'sha256': record['sha256']} for name, record in EMBEDDED_SOURCES.items()})
print('Extracted verified adapters:', SCRIPTS_DIR)


## 4. Isolated environments, pinned sources and pretrained weights

Setup is sequential and uses **no GPU allocations**. Each model has its own
Python 3.11 environment and package versions. The notebook kernel's Torch is not
replaced. Shared UV caches/hardlinks reduce duplicate runtime storage.
All source commits, weight revisions and package versions are logged. Model
failures appear in the final report; other models can continue. Runtime CUDA
compatibility is confirmed only by the later monitored inference jobs.

Progress is printed per model; detailed install output is retained in setup logs.


In [ ]:
import hashlib
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.parse
import urllib.request
from pathlib import Path

MODELS = ("kokoro", "f5", "emotivoice", "styletts2", "omnivoice", "supertonic")
RUNTIME = Path("/tmp/tts_benchmark_runtime")
ENV_DIR = RUNTIME / "envs"
UV_CACHE = RUNTIME / "uv_cache"
HF_CACHE = RUNTIME / "huggingface"
ASSETS = RUNTIME / "assets"
REPOS = RUNTIME / "repos"
SETUP_LOGS = ROOT / "logs" / "setup"
for directory in (ENV_DIR, UV_CACHE, HF_CACHE, ASSETS, REPOS, SETUP_LOGS):
    directory.mkdir(parents=True, exist_ok=True)

BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "CUDA_VISIBLE_DEVICES": "", "HF_HOME": str(HF_CACHE),
    "HF_HUB_CACHE": str(HF_CACHE / "hub"), "UV_CACHE_DIR": str(UV_CACHE),
    "UV_PYTHON_INSTALL_DIR": str(RUNTIME / "python"),
    "GIT_LFS_SKIP_SMUDGE": "1", "TOKENIZERS_PARALLELISM": "false",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1", "PYTHONUNBUFFERED": "1",
    "NLTK_DATA": str(RUNTIME / "nltk_data"),
    "HF_HUB_DOWNLOAD_TIMEOUT": "120", "HF_HUB_ETAG_TIMEOUT": "60",
})
# Subprocesses get their own environments and therefore cannot contaminate
# the notebook kernel or accidentally initialize CUDA during installation.
os.environ["NLTK_DATA"] = BASE_ENV["NLTK_DATA"]

def setup_command(command, log, *, cwd=None, env=None):
    command = [str(value) for value in command]
    with Path(log).open("a", encoding="utf-8") as stream:
        stream.write("\nCOMMAND " + json.dumps(command) + "\n")
        stream.flush()
        started = time.perf_counter()
        process = subprocess.run(command, cwd=cwd, env=env or BASE_ENV,
                                 stdout=stream, stderr=subprocess.STDOUT, text=True)
        stream.write(f"EXIT {process.returncode}; elapsed={time.perf_counter()-started:.3f}s\n")
    if process.returncode:
        tail = Path(log).read_text(encoding="utf-8", errors="replace")[-5000:]
        raise RuntimeError(f"Command failed ({process.returncode}); see {log}\n{tail}")

def setup_json(path, value):
    temporary = Path(path).with_suffix(Path(path).suffix + ".part")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False,
                                    allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)

global_log = SETUP_LOGS / "bootstrap.log"
UV = [sys.executable, "-m", "uv"]
BOOTSTRAP_ERROR = None
try:
    if not globals().get("HARDWARE_READY", True):
        raise RuntimeError("Dual-GPU hardware preflight failed; installation and downloads were skipped.")
    if importlib.util.find_spec("uv") is None:
        setup_command([sys.executable, "-m", "pip", "install", "uv==0.6.17"], global_log)
    setup_command(UV + ["python", "install", "3.11"], global_log)
except Exception as exc:
    BOOTSTRAP_ERROR = str(exc)
    print("Python runtime bootstrap FAILED:", BOOTSTRAP_ERROR)

# Kaggle normally already supplies these. espeak-ng is required by StyleTTS2
# and is also the English fallback in Kokoro; the other packages support I/O.
missing_commands = [name for name in ("git", "ffmpeg", "espeak-ng")
                    if shutil.which(name) is None]
SYSTEM_STATUS = {"missing_before_setup": missing_commands}
if missing_commands and not globals().get("HARDWARE_READY", True):
    SYSTEM_STATUS.update(status="skipped", error="Hardware preflight failed; no system packages installed.")
elif missing_commands:
    try:
        apt = ["apt-get"] if os.geteuid() == 0 else ["sudo", "apt-get"]
        setup_command(apt + ["update", "-qq"], global_log)
        setup_command(apt + ["install", "-y", "-qq", "git", "ffmpeg",
                             "espeak-ng", "libespeak-ng1", "libsndfile1"], global_log)
        SYSTEM_STATUS["status"] = "ready"
    except Exception as exc:
        SYSTEM_STATUS.update(status="failed", error=str(exc))
else:
    SYSTEM_STATUS["status"] = "ready"

SOURCES = {
    "kokoro": ("https://github.com/hexgrad/kokoro.git", "dfb907a02bba8152ca444717ca5d78747ccb4bec"),
    "f5": ("https://github.com/SWivid/F5-TTS.git", "283252563dbf91be625e0c27926acfaac449186c"),
    "emotivoice": ("https://github.com/netease-youdao/EmotiVoice.git", "59f0f36de4db12825f4705dd4e0780d79dd6bb01"),
    "styletts2": ("https://github.com/yl4579/StyleTTS2.git", "5cedc71c333f8d8b8551ca59378bdcc7af4c9529"),
    "omnivoice": ("https://github.com/k2-fsa/OmniVoice.git", "08be0b4ccbac3e13e374e86fbfead4b4cac343e2"),
    "supertonic": ("https://github.com/supertone-oss-archive/supertonic.git", "1e9799e964ea4c0dad7cde993b65c3c813a7b373"),
}
COMMON = ["numpy==1.26.4", "scipy==1.14.1", "soundfile==0.13.1", "tqdm==4.67.1",
          "nvidia-ml-py==12.570.86"]
DEPENDENCIES = {
    "kokoro": ["transformers==5.3.0", "misaki[en]==0.9.4", "loguru==0.7.3",
               "spacy==3.8.7", "thinc==8.3.4", "spacy-curated-transformers==0.3.0",
               "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"],
    "f5": ["vocos==0.1.0", "pypinyin==0.54.0", "rjieba==0.1.1", "librosa==0.10.2.post1",
           "einops==0.8.1", "x_transformers==2.16.2", "torchdiffeq==0.2.5",
           "safetensors==0.5.3", "huggingface_hub==0.29.3", "wandb==0.19.8",
           "accelerate==1.6.0", "ema_pytorch==0.7.7", "datasets==3.5.0"],
    "emotivoice": ["transformers==4.26.1", "numba==0.61.2", "librosa==0.10.2.post1",
                   "yacs==0.1.8", "g2p_en==2.1.0", "jieba==0.42.1", "pypinyin==0.54.0",
                   "pypinyin_dict==0.9.0", "cn2an==0.5.23", "nltk==3.9.1",
                   "PyYAML==6.0.2", "huggingface_hub==0.29.3"],
    "styletts2": ["PyYAML==6.0.2", "munch==4.0.0", "transformers==4.44.2",
                  "einops==0.8.1", "einops-exts==0.0.4", "librosa==0.10.2.post1",
                  "phonemizer==3.3.0", "nltk==3.9.1", "matplotlib==3.10.1",
                  "huggingface_hub==0.29.3"],
    "omnivoice": ["transformers==5.3.0", "accelerate==1.12.0", "pydub==0.25.1",
                  "librosa==0.10.2.post1", "num2words==0.5.14", "WeTextProcessing==1.0.3"],
    "supertonic": ["onnxruntime-gpu[cuda,cudnn]==1.23.2", "huggingface_hub==0.29.3"],
}

def checkout_model(model, log):
    url, commit = SOURCES[model]
    destination = REPOS / model
    if not (destination / ".git").exists():
        setup_command(["git", "init", str(destination)], log)
        setup_command(["git", "-C", str(destination), "remote", "add", "origin", url], log)
    setup_command(["git", "-C", str(destination), "fetch", "--depth", "1", "origin", commit], log)
    setup_command(["git", "-C", str(destination), "checkout", "--detach", commit], log)
    actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"],
                                     text=True, env=BASE_ENV).strip()
    if actual != commit:
        raise RuntimeError(f"Source revision mismatch: {actual} != {commit}")
    return destination

def download_hf(python, model, log):
    destination = ASSETS / model
    destination.mkdir(parents=True, exist_ok=True)
    # Download before concurrent inference, so benchmark timing excludes network
    # and workers do not race on partial weight files.
    specifications = {
        "kokoro": [("hexgrad/Kokoro-82M", "f3ff3571791e39611d31c381e3a41a3af07b4987",
                    ["config.json", "kokoro-v1_0.pth", "voices/*.pt"], ".")],
        "f5": [("SWivid/F5-TTS", "84e5a410d9cead4de2f847e7c9369a6440bdfaca",
                ["F5TTS_v1_Base/model_1250000.safetensors", "F5TTS_v1_Base/vocab.txt"], "."),
               ("charactr/vocos-mel-24khz", "0feb3fdd929bcd6649e0e7c5a688cf7dd012ef21",
                ["config.yaml", "pytorch_model.bin"], "vocos")],
        "emotivoice": [("WangZeJun/simbert-base-chinese", "8096d1f9a7f7c2d9dbd7974511c4bc4a1755932b",
                        ["*.json", "*.txt", "*.bin", "*.safetensors"], "simbert")],
        "styletts2": [("yl4579/StyleTTS2-LibriTTS", "3aa7ba7f8f275ec13dce21682a61494c35089e2a",
                      ["Models/LibriTTS/config.yml", "Models/LibriTTS/epochs_2nd_00020.pth"], ".")],
        "omnivoice": [("k2-fsa/OmniVoice", "c5fdb5ccb189668d56333f77ba2629f4cd7535f4",
                      ["*.json", "*.safetensors", "*.model", "*.txt", "audio_tokenizer/**"], ".")],
        "supertonic": [("supertone-oss-archive/supertonic-3", "aafc6e32416a594460b32413efc49d7fe4ce6d46",
                       ["onnx/**", "voice_styles/**"], ".")],
    }
    script = """import json, sys
from huggingface_hub import snapshot_download
for repo, revision, patterns, subdir in json.loads(sys.argv[1]):
    snapshot_download(repo_id=repo, revision=revision, allow_patterns=patterns,
                      local_dir=str(__import__('pathlib').Path(sys.argv[2])/subdir), max_workers=1)
"""
    setup_command([python, "-c", script, json.dumps(specifications[model]), str(destination)], log)
    return destination

def download_emoti_checkpoints(log):
    # Git stores signed SHA-256 LFS pointers. Download only the two required
    # inference binaries, and validate bytes against their pinned Git pointers.
    commit = "d1332ad9fee7020df5f358fdbe8d257dbf0d1a7c"
    destination = ASSETS / "emotivoice" / "outputs"
    if not (destination / ".git").is_dir():
        setup_command(["git", "init", str(destination)], log)
        setup_command(["git", "-C", str(destination), "remote", "add", "origin",
                       "https://www.modelscope.cn/syq163/outputs.git"], log)
    setup_command(["git", "-C", str(destination), "fetch", "--depth", "1", "origin", commit], log)
    paths = ("prompt_tts_open_source_joint/ckpt/g_00140000", "style_encoder/ckpt/checkpoint_163431")
    records = []
    for relative in paths:
        pointer = subprocess.check_output(["git", "-C", str(destination), "show", f"{commit}:{relative}"],
                                          env=BASE_ENV, text=True)
        oid = re.search(r"^oid sha256:([0-9a-f]{64})$", pointer, re.M)
        size = re.search(r"^size (\d+)$", pointer, re.M)
        if not oid or not size:
            raise RuntimeError(f"Expected an authenticated Git LFS pointer for {relative}")
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        expected_size, expected_sha = int(size.group(1)), oid.group(1)
        def valid_file(path):
            if not path.is_file() or path.stat().st_size != expected_size:
                return False
            digest = hashlib.sha256()
            with path.open("rb") as stream:
                for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                    digest.update(chunk)
            return digest.hexdigest() == expected_sha
        if not valid_file(target):
            query = urllib.parse.urlencode({"Revision": commit, "FilePath": relative})
            url = "https://www.modelscope.cn/api/v1/models/syq163/outputs/repo?" + query
            temporary = target.with_name(target.name + ".part")
            with urllib.request.urlopen(url, timeout=120) as response, temporary.open("wb") as stream:
                shutil.copyfileobj(response, stream, length=8 * 1024 * 1024)
            if not valid_file(temporary):
                raise RuntimeError(f"Checkpoint SHA-256/size mismatch for {relative}; no inference will run.")
            temporary.replace(target)
        records.append({"file": str(target), "revision": commit,
                        "sha256": expected_sha, "size": expected_size})
    setup_json(ROOT / "emotivoice_checkpoint_provenance.json", records)
    return {"generator_checkpoint": str(destination / paths[0]),
            "style_checkpoint": str(destination / paths[1])}

def resources_for(model, repo, assets):
    resources = {"repo": str(repo)}
    if model == "kokoro":
        resources = {"checkpoint": str(assets / "kokoro-v1_0.pth"),
                     "model_config": str(assets / "config.json"),
                     "cache_dir": str(HF_CACHE / "hub")}
    elif model == "f5":
        resources.update(checkpoint=str(assets / "F5TTS_v1_Base/model_1250000.safetensors"),
                         vocab=str(assets / "F5TTS_v1_Base/vocab.txt"), vocos_dir=str(assets / "vocos"))
    elif model == "emotivoice":
        resources.update(bert_path=str(assets / "simbert"), local_files_only=True)
    elif model == "styletts2":
        resources.update(checkpoint=str(assets / "Models/LibriTTS/epochs_2nd_00020.pth"),
                         model_config=str(assets / "Models/LibriTTS/config.yml"))
    elif model == "omnivoice":
        resources = {"local_model": str(assets), "local_files_only": True}
    elif model == "supertonic":
        resources.update(assets_dir=str(assets))
    return resources

requested = CONFIG.get("install_models", list(MODELS))
if not isinstance(requested, list) or any(model not in MODELS for model in requested):
    raise ValueError("CONFIG['install_models'] must contain known model names.")
SETUP_RESULTS = {}
report_path = ROOT / "setup_report.json"
for model in MODELS:
    if model not in requested:
        SETUP_RESULTS[model] = {"status": "not_requested", "error": "Disabled in install_models"}
        continue
    log = SETUP_LOGS / f"{model}.log"
    python = ENV_DIR / model / "bin" / "python"
    entry = {"status": "installing", "python": str(python), "resources": {}, "log": str(log)}
    SETUP_RESULTS[model] = entry
    started = time.perf_counter()
    print(f"[{model}] installing isolated Python, dependencies, source, and weights...")
    try:
        if BOOTSTRAP_ERROR:
            raise RuntimeError("Python runtime bootstrap failed: " + BOOTSTRAP_ERROR)
        if model in ("styletts2", "kokoro") and shutil.which("espeak-ng") is None:
            raise RuntimeError("espeak-ng is unavailable. See bootstrap.log for the system dependency failure.")
        if shutil.which("git") is None:
            raise RuntimeError("git is unavailable; pinned source verification cannot proceed.")
        if not python.is_file():
            setup_command(UV + ["venv", "--python", "3.11", "--seed", str(ENV_DIR / model)], log)
        # Hardlink packages shared between isolated environments. All runtime
        # trees are under /tmp on one filesystem; zip contains outputs only.
        torch_version = "2.8.0" if model in ("omnivoice", "supertonic") else "2.5.1"
        variant = "cu128" if model in ("omnivoice", "supertonic") else "cu121"
        override = CONFIG.get("torch_cuda_variant")
        if isinstance(override, dict):
            variant = override.get(model, variant)
        elif override not in (None, "auto", ""):
            variant = override
        supported = {"2.5.1": {"cu118", "cu121", "cu124"}, "2.8.0": {"cu126", "cu128", "cu129"}}
        if variant not in supported[torch_version]:
            raise ValueError(f"Torch {torch_version} has no supported wheel variant {variant!r} in this setup.")
        if model == "supertonic" and variant != "cu128":
            raise ValueError("Supertonic ORT1.23.2 requires the configured CUDA12.8/cuDNN9 runtime; retain cu128.")
        if model == "supertonic" and CONFIG.get("strict_cuda_driver", False):
            versions = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                text=True, env=BASE_ENV).strip().splitlines()
            if not versions or any(tuple(map(int, version.strip().split("."))) < (570, 26)
                                   for version in versions):
                raise RuntimeError("Strict driver policy requires NVIDIA Linux driver>=570.26 for CUDA12.8.")
        constraints = RUNTIME / f"{model}_constraints.txt"
        constraints.write_text(f"torch=={torch_version}\ntorchaudio=={torch_version}\n"
                               "numpy==1.26.4\nscipy==1.14.1\n", encoding="utf-8")
        install = UV + ["pip", "install", "--python", str(python), "--link-mode", "hardlink"]
        reinstall = ["--reinstall"] if CONFIG.get("force_setup", False) else []
        setup_command(install + reinstall + [f"torch=={torch_version}", f"torchaudio=={torch_version}",
                                 "--index-url", f"https://download.pytorch.org/whl/{variant}"], log)
        setup_command(install + ["--constraint", str(constraints)] + COMMON + DEPENDENCIES[model], log)
        repo = checkout_model(model, log)
        if model in ("kokoro", "omnivoice"):
            url, commit = SOURCES[model]
            setup_command(install + reinstall + ["--no-deps", f"git+{url}@{commit}"], log)
        if model == "emotivoice":
            setup_command([python, "-m", "nltk.downloader", "averaged_perceptron_tagger_eng",
                           "averaged_perceptron_tagger", "cmudict", "punkt", "punkt_tab"], log)
        assets = download_hf(python, model, log)
        resources = resources_for(model, repo, assets)
        if model == "emotivoice":
            resources.update(download_emoti_checkpoints(log))
        entry.update(status="ready", resources=resources, source_revision=SOURCES[model][1],
                     torch_version=torch_version, torch_cuda_variant=variant,
                     elapsed_seconds=time.perf_counter()-started)
        # Exact installed package versions are useful when reproducing failures.
        with (SETUP_LOGS / f"{model}_packages.txt").open("w", encoding="utf-8") as stream:
            subprocess.run(UV + ["pip", "freeze", "--python", str(python)], env=BASE_ENV,
                           stdout=stream, stderr=subprocess.STDOUT, check=True, text=True)
        print(f"[{model}] ready ({entry['elapsed_seconds']:.1f}s); details: {log}")
    except Exception as exc:
        entry.update(status="failed", error=str(exc), elapsed_seconds=time.perf_counter()-started)
        print(f"[{model}] FAILED: {exc}")
    setup_json(report_path, {"models": SETUP_RESULTS, "system": SYSTEM_STATUS,
                             "runtime_directory": str(RUNTIME), "cuda_allocations_during_setup": False})
print("Setup statuses:", {model: item["status"] for model, item in SETUP_RESULTS.items()})


In [ ]:
# CPU-only import/API checks. CUDA context/session initialization is performed
# later by monitored benchmark jobs, not hidden in dependency setup.
PREFLIGHT_RESULTS = {}
API_CHECKS = {
    "kokoro": "from kokoro import KModel, KPipeline; from misaki import en; en.G2P(trf=False, british=False)",
    "f5": "from f5_tts.model import CFM, DiT; from f5_tts.model.utils import get_tokenizer; from vocos import Vocos",
    "emotivoice": "from frontend_en import G2p; from frontend import g2p_cn_en; from models.prompt_tts_modified.jets import JETSGenerator; from models.prompt_tts_modified.simbert import StyleEncoder; G2p()",
    "styletts2": "from models import build_model; from Utils.PLBERT.util import CustomAlbert; from Modules.diffusion.sampler import DiffusionSampler; from phonemizer.backend import EspeakBackend; EspeakBackend('en-us')",
    "omnivoice": "from omnivoice import OmniVoice; from transformers import HiggsAudioV2TokenizerModel",
    "supertonic": "import onnxruntime as ort; assert 'CUDAExecutionProvider' in ort.get_available_providers()",
}
SCRIPT_NAMES = {"f5": "f5_samples.py", "styletts2": "styletts2_samples.py",
                "emotivoice": "emotivoice_samples.py", "kokoro": "kokoro_samples.py",
                "omnivoice": "omnivoice_samples.py", "supertonic": "supertonic_samples.py"}
for model, item in SETUP_RESULTS.items():
    if item["status"] != "ready":
        PREFLIGHT_RESULTS[model] = {"status": "unavailable", "error": item.get("error")}
        continue
    log = SETUP_LOGS / f"{model}_preflight.log"
    try:
        repo = REPOS / model
        sys_path = str(repo / "src") if model == "f5" else str(repo)
        script = ("import os,sys,torch; assert os.environ.get('CUDA_VISIBLE_DEVICES')==''; "
                  f"sys.path.insert(0,{sys_path!r}); os.chdir({str(repo)!r}); " + API_CHECKS[model] +
                  "; print('API_IMPORTS_OK'); print('Torch:',torch.__version__,'CUDA runtime:',torch.version.cuda)")
        setup_command([item["python"], "-c", script], log)
        setup_command([item["python"], str(SCRIPTS_DIR / SCRIPT_NAMES[model]), "--help"], log)
        PREFLIGHT_RESULTS[model] = {"status": "passed", "log": str(log)}
        print(f"[{model}] CPU API preflight passed")
    except Exception as exc:
        PREFLIGHT_RESULTS[model] = {"status": "failed", "error": str(exc), "log": str(log)}
        item.update(status="failed", error="API preflight failed: " + str(exc))
        print(f"[{model}] API preflight FAILED: {exc}")
setup_json(ROOT / "preflight_report.json", PREFLIGHT_RESULTS)
setup_json(ROOT / "setup_report.json", {"models": SETUP_RESULTS, "system": SYSTEM_STATUS,
           "runtime_directory": str(RUNTIME), "cuda_allocations_during_setup": False})


## 5. Reference voice

The CPU bootstrap is a prerequisite for reference-based models; paired GPU
experiments begin after it finishes. No content-dependent trimming is used.
The reference and transcript are included in the ZIP when configured.


In [ ]:
# F5/StyleTTS2 need reference audio. Supply your own exact transcript for voice
# cloning, or generate this disclosed synthetic reference with Kokoro on CPU.
REFERENCE_AUDIO = CONFIG.get("reference_audio")
REFERENCE_TEXT = CONFIG.get("reference_text")
if CONFIG.get("speaker_mode") == "default":
    REFERENCE_AUDIO = REFERENCE_TEXT = None
REFERENCE_STATUS = {"status": "pending"}
if REFERENCE_AUDIO:
    try:
        REFERENCE_AUDIO = str(Path(REFERENCE_AUDIO).expanduser().resolve())
        if not Path(REFERENCE_AUDIO).is_file():
            raise ValueError("Set an existing reference_audio WAV.")
        if REFERENCE_TEXT is not None and (not isinstance(REFERENCE_TEXT, str) or not REFERENCE_TEXT.strip()):
            raise ValueError("reference_text must be the exact nonempty transcript or None.")
        checker_python = next((item["python"] for item in SETUP_RESULTS.values()
                               if item.get("status") == "ready"), sys.executable)
        checker = """import json,sys,numpy as np,soundfile as sf
wave,rate=sf.read(sys.argv[1],dtype='float32',always_2d=True)
assert wave.shape[1]==1, 'Reference must be mono.'
duration=wave.shape[0]/rate
assert 1<=duration<=8, f'Reference duration {duration:.3f}s outside 1-8s.'
assert np.isfinite(wave).all(), 'Reference contains nonfinite samples.'
assert np.max(np.abs(wave))>1e-5, 'Reference is silent.'
print(json.dumps({'duration_seconds':duration,'sample_rate':rate,'channels':1}))
"""
        checked = subprocess.run([checker_python, "-c", checker, REFERENCE_AUDIO], env=BASE_ENV,
                                 text=True, capture_output=True, check=True)
        facts = json.loads(checked.stdout.strip())
        REFERENCE_STATUS.update(status="provided", path=REFERENCE_AUDIO, transcript=REFERENCE_TEXT, **facts)
    except Exception as exc:
        REFERENCE_AUDIO = None
        details = (getattr(exc, "stderr", None) or str(exc)).strip()
        REFERENCE_STATUS.update(status="failed", error=details)
        print("Reference validation FAILED:", details)
elif CONFIG.get("speaker_mode") == "reference":
    REFERENCE_STATUS.update(status="failed", error="speaker_mode='reference' requires your reference_audio WAV; add its transcript for F5/OmniVoice.")
    print(REFERENCE_STATUS["error"])
else:
    REFERENCE_TEXT = "The morning sun brings gentle warmth to the quiet garden."
    item = SETUP_RESULTS.get("kokoro", {})
    if item.get("status") != "ready":
        REFERENCE_STATUS.update(status="failed", error="Kokoro unavailable; set reference_audio and reference_text.")
        print(REFERENCE_STATUS["error"])
    else:
        try:
            ref_root = ROOT / "reference"
            ref_root.mkdir(parents=True, exist_ok=True)
            parameters = {**item["resources"], "text": REFERENCE_TEXT,
                          "voice": str(ASSETS / "kokoro" / "voices" / "am_fenrir.pt"),
                          "language": "a", "device": "cpu", "n": 1, "seed": 42, "speed": 1.0,
                          "sample_rate": 24000, "output_dir": str(ref_root / "kokoro_runs")}
            settings = ref_root / "bootstrap_config.json"
            setup_json(settings, parameters)
            setup_command([item["python"], str(SCRIPTS_DIR / "kokoro_samples.py"),
                           "--config", str(settings)], SETUP_LOGS / "reference_bootstrap.log")
            candidates = sorted((ref_root / "kokoro_runs").glob("**/*.wav"), key=lambda path: path.stat().st_mtime)
            if not candidates:
                raise RuntimeError("Kokoro created no reference WAV.")
            reference = ref_root / "reference_kokoro_am_fenrir.wav"
            shutil.copy2(candidates[-1], reference)
            # WAV PCM frame/rate header gives total duration without loading a
            # model or removing content-dependent silence.
            import wave
            with wave.open(str(reference), "rb") as audio:
                duration = audio.getnframes() / audio.getframerate()
            if not 1 <= duration <= 8:
                raise RuntimeError(f"Synthetic reference duration {duration:.3f}s outside1–8s; supply a clean reference.")
            REFERENCE_AUDIO = str(reference)
            REFERENCE_STATUS.update(status="synthetic", path=REFERENCE_AUDIO,
                                    transcript=REFERENCE_TEXT, duration_seconds=duration,
                                    generator="Kokoro am_fenrir, CPU; complete untrimmed male waveform")
            print(f"Synthetic reference ready: {REFERENCE_AUDIO} ({duration:.3f}s)")
        except Exception as exc:
            REFERENCE_AUDIO = None
            REFERENCE_STATUS.update(status="failed", error=str(exc))
            print(f"Reference bootstrap FAILED: {exc}")
setup_json(ROOT / "reference_report.json", REFERENCE_STATUS)


## 6. Paired GPU experiments

Each job loads its model once and produces all requested takes. Both slots
display the worker's actual progress line, plus sampled VRAM. Jobs are named by
model, profile, word count, text/config hashes and seed. Full carriage-return
progress logs are saved. Failed/over-budget jobs and any partial recordings are
retained. If you interrupt, run the report cell afterward to package the results.


In [ ]:
SCRIPT_NAMES = {"kokoro": "kokoro_samples.py", "f5": "f5_samples.py", "emotivoice": "emotivoice_samples.py",
                "styletts2": "styletts2_samples.py", "omnivoice": "omnivoice_samples.py", "supertonic": "supertonic_samples.py"}
JOB_RESULTS = []
SAMPLE_RESULTS = []
TIMELINE = []

def slug(value, maximum=48):
    value = re.sub(r"[^a-zA-Z0-9_-]+", "-", str(value)).strip("-")
    return (value or "unnamed")[:maximum]

def profile_jobs():
    jobs = []
    speaker_mode = CONFIG.get("speaker_mode", "auto")
    if speaker_mode not in {"auto", "reference", "default"}:
        raise ValueError("speaker_mode must be auto, reference, or default")
    for case_index, case in enumerate(TEXT_CASES):
        text = case["text"].strip()
        count = len(text.split())
        if not 10 <= count <= 500:
            raise ValueError(f"Case {case['name']!r} has {count} words, outside 10-500")
        text_hash = hashlib.sha256(text.encode()).hexdigest()[:8]
        for model in CONFIG["install_models"]:
            if model not in SCRIPT_NAMES:
                raise ValueError(f"Unknown model {model}")
            for profile_index, profile in enumerate(MODEL_PROFILES.get(model, [])):
                if not profile.get("enabled", True):
                    continue
                parameters = dict(profile.get("parameters", {}))
                forbidden = set(parameters) & {"config", "text", "text_file", "device", "n", "seed", "output_dir", "max_vram_gb", "estimate_only", "self_test"}
                if forbidden:
                    raise ValueError(f"Profile {model}/{profile['name']} has supervisor-controlled keys: {sorted(forbidden)}")
                count_takes = profile.get("n", CONFIG["n"])
                seed = profile.get("seed", CONFIG["seed"])
                if type(count_takes) is not int or count_takes < 1 or type(seed) is not int or not 0 <= seed <= 2**63 - count_takes:
                    raise ValueError("Profile n and seed must be positive/nonnegative integers within signed 64-bit range")
                # Incorporate case ordinal/config identity: duplicate names cannot overwrite runs.
                config_hash = hashlib.sha256(json.dumps(profile, sort_keys=True).encode()).hexdigest()[:8]
                job_id = f"{model}__{slug(profile['name'])}__w{count:03d}__c{case_index:02d}p{profile_index:02d}__{text_hash}_{config_hash}"
                job_dir = RUN_DIR / "jobs" / job_id
                job_dir.mkdir(parents=True, exist_ok=False)
                text_path = job_dir / "text.txt"
                text_path.write_text(text, encoding="utf-8")
                settings = {"text_file": str(text_path), "n": count_takes, "seed": seed,
                            "device": "cuda:0", "sample_rate": CONFIG["sample_rate"], "pause_ms": CONFIG["pause_ms"],
                            "max_vram_gb": CONFIG["max_vram_gb"], "output_dir": str(job_dir / "raw")}
                setup = SETUP_RESULTS.get(model, {"status": "failed", "error": "Model setup not requested/completed"})
                resources = dict(setup.get("resources", {}))
                settings.update({k: str(v) if isinstance(v, Path) else v for k, v in resources.items() if v is not None})
                settings.update({k: v for k, v in parameters.items() if v is not None})
                if model == "kokoro":
                    voice_dir = Path(resources.get("checkpoint", ".")).parent / "voices"
                    voices = [voice.strip() for voice in settings.get("voice", "am_fenrir").split(",")]
                    settings["voice"] = ",".join(str(voice_dir / f"{voice}.pt") if (voice_dir / f"{voice}.pt").is_file() else voice for voice in voices)
                error = None
                blocked_status = "setup_failed"
                speaker_source = "preset"
                if setup.get("status") != "ready":
                    error = setup.get("error", "Model environment is unavailable")
                if model in {"f5", "styletts2"}:
                    speaker_source = "user_reference" if parameters.get("reference_audio") or (CONFIG.get("reference_audio") and speaker_mode != "default") else "synthetic_male_reference"
                    settings.setdefault("reference_audio", str(REFERENCE_AUDIO) if REFERENCE_AUDIO else None)
                    if model == "f5":
                        settings.setdefault("reference_text", REFERENCE_TEXT)
                    if not settings.get("reference_audio") or (model == "f5" and not settings.get("reference_text")):
                        error = "Reference WAV/transcript preparation failed; see reference_report.json. F5 requires an exact transcript."
                if model == "omnivoice":
                    reference_choice = profile.get("reference", "auto")
                    if type(reference_choice) is not bool and reference_choice != "auto":
                        raise ValueError("OmniVoice profile reference must be True, False, or 'auto'")
                    cached_prompt = bool(settings.get("prompt_cache") and Path(settings["prompt_cache"]).is_file())
                    use_reference = bool(settings.get("ref_audio")) or cached_prompt or reference_choice is True or speaker_mode == "reference" or (reference_choice == "auto" and speaker_mode == "auto" and bool(CONFIG.get("reference_audio")))
                    if use_reference:
                        if not cached_prompt:
                            settings.setdefault("ref_audio", str(REFERENCE_AUDIO) if REFERENCE_AUDIO else None)
                            settings.setdefault("ref_text", REFERENCE_TEXT)
                        # Preset voice attributes must not conflict with the supplied person's voice.
                        clone_parameters = profile.get("clone_parameters", {})
                        forbidden_clone = set(clone_parameters) & {"config", "text", "text_file", "device", "n", "seed", "output_dir", "max_vram_gb", "estimate_only", "self_test"}
                        if forbidden_clone:
                            raise ValueError(f"clone_parameters contains supervisor-controlled keys: {sorted(forbidden_clone)}")
                        if "instruct" not in clone_parameters:
                            settings.pop("instruct", None)
                        settings.update(clone_parameters)
                        speaker_source = "cached_reference_prompt" if cached_prompt else "user_reference" if parameters.get("ref_audio") or (CONFIG.get("reference_audio") and speaker_mode != "default") else "synthetic_male_reference"
                        if not cached_prompt and (not settings.get("ref_audio") or not settings.get("ref_text")):
                            error = "Reference WAV/transcript preparation failed; see reference_report.json. OmniVoice requires an exact transcript in this adapter."
                    else:
                        speaker_source = "voice_design"
                if speaker_mode == "reference" and model not in {"f5", "styletts2", "omnivoice"}:
                    error = "This model supports preset speakers rather than arbitrary reference-WAV cloning."
                    blocked_status = "reference_unsupported"
                if model in {"emotivoice", "supertonic"}:
                    settings["allow_unverified_cuda_budget"] = True
                settings = {k: v for k, v in settings.items() if v is not None}
                config_path = job_dir / "config.json"
                atomic_json(config_path, settings)
                job = {"job_id": job_id, "model": model, "profile": profile["name"], "case": case["name"],
                       "word_count": count, "text_sha256": hashlib.sha256(text.encode()).hexdigest(),
                       "n": count_takes, "seed": seed, "job_dir": str(job_dir), "settings": settings,
                       "speaker_source": speaker_source, "blocked_status": blocked_status,
                       "config_path": str(config_path), "python": setup.get("python"), "blocked_reason": error}
                atomic_json(job_dir / "job.json", job)
                jobs.append(job)
    return jobs

def process_tree(pid):
    ids = {pid}
    if pid:
        try:
            ids.update(child.pid for child in psutil.Process(pid).children(recursive=True))
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return ids - {None}

def nvml_process_usage(handle, tracked):
    allocations = {}
    available = False
    for getter_name in ("nvmlDeviceGetComputeRunningProcesses", "nvmlDeviceGetGraphicsRunningProcesses"):
        try:
            for proc in getattr(pynvml, getter_name)(handle):
                if proc.pid in tracked:
                    value = proc.usedGpuMemory
                    if isinstance(value, int) and 0 <= value < 2**60:
                        allocations[proc.pid] = max(value, allocations.get(proc.pid, 0))
                        available = True
        except pynvml.NVMLError:
            continue
    return (sum(allocations.values()) if available else None), sorted(allocations)

class JobMonitor:
    """Monitor external allocations from before model loading through process exit."""
    def __init__(self, job, slot):
        self.job, self.slot, self.pid = job, slot, None
        self.handle = GPU_HANDLES[slot]
        self.stop_event = threading.Event()
        self.lock = threading.Lock()
        self.process_ids = set()
        self.process_births = {}
        self.device_peak = 0
        self.process_peak = None
        self.last_device = 0
        self.last_process = None
        self.violation = None
        self.error = None
        self.poll_count = 0
        self.started = time.perf_counter()
        self.thread = threading.Thread(target=self.observe, daemon=True)

    def start(self):
        self.thread.start()

    def observe(self):
        path = Path(self.job["job_dir"]) / "vram_trace.csv"
        budget = int(CONFIG["max_vram_gb"] * 1e9)
        interval = CONFIG["nvml_interval_ms"] / 1000
        fields = ["elapsed_seconds", "device_used_bytes", "process_tree_used_bytes", "gpu_utilization_percent", "memory_utilization_percent", "worker_pid", "tracked_gpu_pids"]
        try:
            with path.open("w", newline="", encoding="utf-8") as stream:
                writer = csv.DictWriter(stream, fieldnames=fields)
                writer.writeheader()
                while not self.stop_event.is_set():
                    tick = time.perf_counter()
                    if self.pid:
                        self.process_ids.update(process_tree(self.pid))
                        for pid in self.process_ids:
                            if pid not in self.process_births:
                                try:
                                    self.process_births[pid] = psutil.Process(pid).create_time()
                                except (psutil.NoSuchProcess, psutil.AccessDenied):
                                    pass
                    info = pynvml.nvmlDeviceGetMemoryInfo(self.handle)
                    usage, gpu_pids = nvml_process_usage(self.handle, self.process_ids)
                    utilization = pynvml.nvmlDeviceGetUtilizationRates(self.handle)
                    self.last_device, self.last_process = info.used, usage
                    self.device_peak = max(self.device_peak, info.used)
                    if usage is not None:
                        self.process_peak = max(self.process_peak or 0, usage)
                    self.poll_count += 1
                    writer.writerow({"elapsed_seconds": f"{tick-self.started:.6f}", "device_used_bytes": info.used,
                        "process_tree_used_bytes": usage, "gpu_utilization_percent": utilization.gpu,
                        "memory_utilization_percent": utilization.memory, "worker_pid": self.pid,
                        "tracked_gpu_pids": ";".join(map(str, gpu_pids))})
                    observed = usage if CONFIG["budget_scope"] == "process_tree" and usage is not None else info.used
                    scope = "process_tree" if CONFIG["budget_scope"] == "process_tree" and usage is not None else "device_total"
                    if observed > budget and self.violation is None:
                        self.violation = {"observed_bytes": observed, "scope": scope,
                                          "budget_bytes": budget, "elapsed_seconds": tick-self.started}
                    if self.poll_count % 100 == 0:
                        stream.flush()
                    self.stop_event.wait(max(0.001, interval-(time.perf_counter()-tick)))
        except Exception as exc:
            self.error = f"{type(exc).__name__}: {exc}"

    def finish(self):
        self.stop_event.set()
        self.thread.join(timeout=5)
        if self.thread.is_alive():
            self.error = "Monitor thread did not finish"
        result = {"nvml_sampled_device_peak_bytes": self.device_peak,
                  "nvml_sampled_process_tree_peak_bytes": self.process_peak,
                  "requested_interval_ms": CONFIG["nvml_interval_ms"], "samples": self.poll_count,
                  "budget_violation": self.violation, "monitor_error": self.error,
                  "hard_instantaneous_peak_certified": False}
        atomic_json(Path(self.job["job_dir"]) / "memory_monitor.json", result)
        return result

class Worker:
    def __init__(self, job, slot):
        import codecs
        self.job, self.slot = job, slot
        self.started = time.perf_counter()
        self.latest_line = "Starting interpreter and loading model"
        self.tail = deque(maxlen=40)
        self.abort_reason = None
        self.monitor = JobMonitor(job, slot)
        self.monitor.start()
        env = globals().get("BASE_ENV", os.environ).copy()
        env.update({"CUDA_VISIBLE_DEVICES": GPU_INVENTORY[slot]["uuid"], "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
                    "PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": str(CONFIG["cpu_threads_per_job"]),
                    "MKL_NUM_THREADS": str(CONFIG["cpu_threads_per_job"]), "TOKENIZERS_PARALLELISM": "false"})
        espeak_library = SETUP_RESULTS[self.job["model"]].get("espeak_library") or env.get("PHONEMIZER_ESPEAK_LIBRARY")
        if espeak_library:
            env["PHONEMIZER_ESPEAK_LIBRARY"] = espeak_library
        else:
            env.pop("PHONEMIZER_ESPEAK_LIBRARY", None)
        command = [job["python"], str(SCRIPTS_DIR / SCRIPT_NAMES[job["model"]]), "--config", job["config_path"]]
        atomic_json(Path(job["job_dir"]) / "command.json", {"argv": command, "gpu_uuid": env["CUDA_VISIBLE_DEVICES"],
                    "logical_device": "cuda:0", "cpu_threads": CONFIG["cpu_threads_per_job"]})
        try:
            self.process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                bufsize=0, start_new_session=True, cwd=str(ROOT))
            self.monitor.pid = self.process.pid
            self.monitor.process_ids.add(self.process.pid)
        except Exception:
            self.monitor.finish()
            raise

        def read_output():
            decoder = codecs.getincrementaldecoder("utf-8")("replace")
            remainder = ""
            try:
                with (Path(job["job_dir"]) / "generation.log").open("w", encoding="utf-8") as log:
                    while True:
                        data = os.read(self.process.stdout.fileno(), 65536)
                        if not data:
                            break
                        decoded = decoder.decode(data)
                        log.write(decoded)
                        log.flush()
                        fragments = re.split(r"[\r\n]", remainder + decoded)
                        remainder = fragments.pop()
                        for fragment in fragments:
                            clean = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", fragment).strip()
                            if clean:
                                self.latest_line = clean[-500:]
                                self.tail.append(clean)
                    final = remainder + decoder.decode(b"", final=True)
                    if final.strip():
                        self.latest_line = final.strip()[-500:]
                        self.tail.append(final.strip())
            except Exception as exc:
                self.latest_line = "Log reader error: " + str(exc)
        self.reader = threading.Thread(target=read_output, daemon=True)
        self.reader.start()
        TIMELINE.append({"event": "start", "job_id": job["job_id"], "model": job["model"],
                         "gpu_slot": slot, "gpu_uuid": GPU_INVENTORY[slot]["uuid"], "timestamp": time.time()})

    def terminate(self, reason):
        self.abort_reason = reason
        self.monitor.process_ids.update(process_tree(self.process.pid))
        try:
            os.killpg(self.process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            self.process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(self.process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            self.process.wait(timeout=10)
        for pid in self.monitor.process_ids - {self.process.pid}:
            try:
                descendant = psutil.Process(pid)
                if descendant.create_time() == self.monitor.process_births.get(pid):
                    descendant.kill()
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass

    def finish(self):
        self.process.wait()
        # A parent can leave a child holding pipe/GPU handles. Kill only tracked
        # descendants whose creation time still matches, avoiding reused PIDs.
        for pid in self.monitor.process_ids - {self.process.pid}:
            try:
                descendant = psutil.Process(pid)
                if descendant.create_time() == self.monitor.process_births.get(pid):
                    descendant.kill()
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        self.reader.join(timeout=10)
        exit_code = self.process.returncode
        status = "completed" if exit_code == 0 and not self.abort_reason else "failed"
        if self.abort_reason:
            status = "budget_exceeded" if self.abort_reason.startswith("VRAM") else "failed"
        # Reap remaining descendants even after a nominally successful parent exit.
        deadline = time.perf_counter() + CONFIG["gpu_release_timeout_seconds"]
        busy = []
        while time.perf_counter() < deadline:
            _, busy = nvml_process_usage(GPU_HANDLES[self.slot], self.monitor.process_ids)
            if not busy:
                break
            time.sleep(0.05)
        if busy:
            status = "gpu_release_failed"
        memory = self.monitor.finish()
        if memory["budget_violation"] and CONFIG["enforce_observed_budget"]:
            status = "budget_exceeded" if not busy else status
            self.abort_reason = self.abort_reason or "VRAM exceeded observed budget: " + json.dumps(memory["budget_violation"])
        if memory["monitor_error"]:
            status = "monitor_failed"
        result = {**self.job, "gpu_slot": self.slot, "gpu_uuid": GPU_INVENTORY[self.slot]["uuid"],
                  "status": status, "exit_code": exit_code, "process_wall_seconds": time.perf_counter()-self.started,
                  "error": self.abort_reason or ("\n".join(self.tail)[-5000:] if exit_code else None),
                  "gpu_release_failed_pids": busy, **memory}
        atomic_json(Path(self.job["job_dir"]) / "supervisor_result.json", result)
        TIMELINE.append({"event": "finish", "job_id": self.job["job_id"], "model": self.job["model"],
                         "gpu_slot": self.slot, "timestamp": time.time(), "status": status})
        return result

def preferred_job_index(pending, active_models):
    for index, job in enumerate(pending):
        if job["model"] not in active_models:
            return index
    return 0 if pending else None

def live_html(active, completed, total):
    rows = []
    for slot in range(2):
        worker = active.get(slot)
        if worker:
            label = html.escape(worker.job["job_id"])
            line = html.escape(worker.latest_line)
            memory = worker.monitor.last_device / 1e9
            process_memory = worker.monitor.last_process
            process_text = "unavailable" if process_memory is None else f"{process_memory/1e9:.3f} GB"
            rows.append(f"<tr><td>GPU {slot}</td><td>{label}<br><pre>{line}</pre></td><td>{memory:.3f} GB device<br>{process_text} process</td></tr>")
        else:
            rows.append(f"<tr><td>GPU {slot}</td><td>Idle</td><td></td></tr>")
    return f"<p>{completed}/{total} jobs finished. One isolated worker per GPU; distinct models preferred.</p><table><tr><th>Slot</th><th>Model progress</th><th>Live NVML usage</th></tr>{''.join(rows)}</table>"

def run_benchmark(jobs):
    pending = []
    for job in jobs:
        reason = job["blocked_reason"] or (hardware_error if not HARDWARE_READY else None)
        if reason:
            result = {**job, "status": job.get("blocked_status", "setup_failed"), "error": reason, "exit_code": None, "gpu_slot": None,
                      "gpu_uuid": None, "process_wall_seconds": None, "hard_instantaneous_peak_certified": False}
            JOB_RESULTS.append(result)
            atomic_json(Path(job["job_dir"]) / "supervisor_result.json", result)
        else:
            pending.append(job)
    active = {}
    disabled_slots = set()
    progress = tqdm(total=len(jobs), initial=len(JOB_RESULTS), desc="Benchmark jobs", unit="job")
    live = display(HTML(live_html(active, len(JOB_RESULTS), len(jobs))), display_id=True)
    try:
        while pending or active:
            for slot in range(2):
                if slot in active or slot in disabled_slots or not pending:
                    continue
                index = preferred_job_index(pending, {worker.job["model"] for worker in active.values()})
                job = pending.pop(index)
                try:
                    active[slot] = Worker(job, slot)
                except Exception as exc:
                    result = {**job, "status": "launch_failed", "error": f"{type(exc).__name__}: {exc}",
                              "exit_code": None, "gpu_slot": slot, "gpu_uuid": GPU_INVENTORY[slot]["uuid"]}
                    JOB_RESULTS.append(result)
                    atomic_json(Path(job["job_dir"]) / "supervisor_result.json", result)
                    progress.update(1)
            for slot, worker in list(active.items()):
                elapsed = time.perf_counter()-worker.started
                if worker.process.poll() is None:
                    if worker.monitor.error:
                        worker.terminate("NVML monitor failed: " + worker.monitor.error)
                    elif worker.monitor.violation and CONFIG["enforce_observed_budget"]:
                        worker.terminate("VRAM exceeded observed budget: " + json.dumps(worker.monitor.violation))
                    elif elapsed > CONFIG["job_timeout_seconds"]:
                        worker.terminate("Job timeout")
                if worker.process.poll() is not None:
                    result = worker.finish()
                    JOB_RESULTS.append(result)
                    progress.update(1)
                    if result.get("gpu_release_failed_pids"):
                        disabled_slots.add(slot)
                    del active[slot]
            if len(disabled_slots) == 2 and pending:
                for job in pending:
                    result = {**job, "status": "gpu_unavailable", "error": "Both GPUs retain tracked allocations", "exit_code": None}
                    JOB_RESULTS.append(result)
                    atomic_json(Path(job["job_dir"]) / "supervisor_result.json", result)
                    progress.update(1)
                pending.clear()
            live.update(HTML(live_html(active, len(JOB_RESULTS), len(jobs))))
            time.sleep(0.2)
    except BaseException as supervisor_error:
        for worker in active.values():
            worker.terminate("Interrupted by user" if isinstance(supervisor_error, KeyboardInterrupt) else "Supervisor error: " + str(supervisor_error))
            JOB_RESULTS.append(worker.finish())
        for job in pending:
            result = {**job, "status": "cancelled", "error": "Supervisor interrupted: " + str(supervisor_error), "exit_code": None}
            JOB_RESULTS.append(result)
            atomic_json(Path(job["job_dir"]) / "supervisor_result.json", result)
        print("Supervisor stopped:", type(supervisor_error).__name__, str(supervisor_error))
        print("Run the report/ZIP cell to package all completed and partial outputs.")
    finally:
        progress.close()
        atomic_json(RUN_DIR / "supervisor_results.json", JOB_RESULTS)
        atomic_json(RUN_DIR / "concurrency_timeline.json", TIMELINE)
        live.update(HTML(live_html({}, len(JOB_RESULTS), len(jobs))))

jobs = profile_jobs()
atomic_json(RUN_DIR / "experiment_plan.json", {"config": CONFIG, "cases": TEXT_CASES, "profiles": MODEL_PROFILES, "jobs": jobs})
print(f"{len(jobs)} model/profile/text jobs; {sum(j['n'] for j in jobs)} planned WAVs")
if KERNEL_READY:
    run_benchmark(jobs)
else:
    JOB_RESULTS = [{**job, "status": "kernel_failed", "error": hardware_error, "exit_code": None} for job in jobs]
    atomic_json(RUN_DIR / "supervisor_results.json", JOB_RESULTS)


## 7. Reports, listening players and ZIP download

`jobs.csv` includes every scheduled job, including setup/runtime failures and
duration plans available before a failure. `samples.csv` independently compares
predicted and actual duration/rate/sample count for each created WAV. Planner
timing scopes differ by model and are labeled explicitly. VRAM CSV traces and
native manifests remain available per job.

`quality.csv`/`quality.json` include acoustic gaps/clipping statistics, independent
Whisper transcripts, word edits, and ASR word error rate. Checks run on **CPU**
after generation and do not enter model VRAM or estimator latency measurements.
ASR errors are screening evidence: pronunciation, punctuation and recognition
errors can contribute. Listen to flagged samples before deciding a word is missing.
Disable ASR with `asr_quality_check=False` to retain acoustic checks without its
additional download/runtime, or disable both with `quality_checks=False`.

Download the ZIP from the link or Kaggle's **Output** panel. After extracting,
open `index.html` to compare the report and listen. Dependencies and pretrained
weights are excluded from the ZIP. A full kernel/VM shutdown can interrupt
packaging; completed files remain under `/kaggle/working/tts_benchmark/runs`
for as long as Kaggle retains that filesystem.


In [ ]:
PLANNER_SCOPE = {
    "kokoro": "First loaded voice-conditioned G2P/duration pass; not a warmed latency guarantee",
    "f5": "Allocated frame plan including CFM minimum-length clamp; no waveform generation",
    "emotivoice": "Warmed speaker/style-conditioned duration preflight",
    "styletts2": "BERT/style diffusion/duration preflight for all requested takes",
    "omnivoice": "Upstream rule token allocation; separate cold preflight/import/reference timing",
    "supertonic": "Warmed voice-conditioned duration predictor; startup probes excluded",
}

def read_json_safely(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None

def find_manifest(job):
    raw = Path(job["job_dir"]) / "raw"
    preferred = {"kokoro": ["manifest.json", "preview.json"], "f5": ["manifest.json"],
                 "styletts2": ["manifest.json"], "emotivoice": ["emotivoice_manifest.json"],
                 "supertonic": ["supertonic_manifest.json"], "omnivoice": ["manifest.json", "estimate.json"]}
    for filename in preferred[job["model"]]:
        for path in sorted(raw.rglob(filename)):
            data = read_json_safely(path)
            if data is not None:
                return path, data
    return None, {}

def native_schema(job, data):
    model = job["model"]
    samples = data.get("samples", data.get("outputs", data.get("runs", [])))
    planner = cold = None
    prediction = None
    export_rate = data.get("export_sample_rate", data.get("export_rate"))
    predictions = []
    if model == "kokoro":
        prediction = data.get("predicted_export_samples")
        planner, cold = data.get("preflight_seconds"), data.get("startup_seconds")
    elif model == "f5":
        preview = data.get("preview", {})
        prediction = preview.get("exported_samples")
        planner, cold = preview.get("preflight_seconds"), preview.get("startup_seconds")
        export_rate = data.get("settings", {}).get("sample_rate")
    elif model == "styletts2":
        planner, cold = data.get("preflight_seconds"), data.get("startup_seconds")
        export_rate = data.get("settings", {}).get("sample_rate")
        predictions = [{"seed": p.get("seed"), "predicted_samples": p.get("exported_samples"), "predicted_seconds": p.get("seconds")} for p in data.get("preview", [])]
    elif model in {"emotivoice", "supertonic"}:
        prediction = data.get("predicted_samples")
        planner, cold = data.get("warm_preview_seconds"), data.get("startup_seconds")
    elif model == "omnivoice":
        plan = data.get("plan", {})
        prediction = plan.get("predicted_export_samples")
        planner = plan.get("planner_ms", 0) / 1000 if "planner_ms" in plan else None
        cold = plan.get("cold_preflight_ms", 0) / 1000 if "cold_preflight_ms" in plan else None
        export_rate = plan.get("sample_rate")
    export_rate = export_rate or job.get("settings", {}).get("sample_rate")
    if not export_rate:
        export_rate = {"emotivoice": 16000, "supertonic": 44100}.get(model, 24000)
    if prediction is not None:
        predictions = [{"seed": job["seed"] + index, "predicted_samples": prediction, "predicted_seconds": prediction/export_rate} for index in range(job["n"])]
    rows = []
    for index, record in enumerate(samples):
        seed = record.get("seed", job["seed"] + index)
        predicted = record.get("predicted_samples", prediction)
        if model == "styletts2":
            previews = data.get("preview", [])
            match = next((p for p in previews if p.get("seed") == seed), None)
            if match:
                predicted = match.get("exported_samples")
        rows.append({"seed": seed, "path": record.get("path", record.get("file")),
                     "predicted_samples": predicted, "generation_seconds": record.get("generation_seconds"),
                     "native_record": record})
    return {"rows": rows, "prediction": prediction, "predictions": predictions, "export_rate": export_rate,
            "planner_seconds": planner, "startup_seconds": cold}

def allocator_peaks(data):
    allocated, reserved = [], []
    def visit(value):
        if isinstance(value, dict):
            for key, item in value.items():
                if isinstance(item, (int, float)) and not isinstance(item, bool):
                    if key in {"torch_peak_allocated_bytes", "peak_torch_allocated_bytes"}:
                        allocated.append(item)
                    elif key in {"torch_peak_reserved_bytes", "peak_torch_reserved_bytes"}:
                        reserved.append(item)
                visit(item)
        elif isinstance(value, list):
            for item in value:
                visit(item)
    visit(data)
    return max(allocated, default=None), max(reserved, default=None)

def collect_reports():
    audio_dir = PACKAGE_DIR / "audio"
    audio_dir.mkdir(parents=True, exist_ok=True)
    job_rows, sample_rows = [], []
    for job in JOB_RESULTS:
        manifest_path, manifest = find_manifest(job)
        schema = native_schema(job, manifest)
        allocated, reserved = allocator_peaks(manifest)
        listed = {Path(row["path"]).resolve(): row for row in schema["rows"] if row["path"]}
        found = sorted((Path(job["job_dir"]) / "raw").rglob("*.wav"))
        observed_errors = []
        final_count = 0
        for index, source in enumerate(found):
            native = listed.get(source.resolve())
            seed = native["seed"] if native else None
            predicted = native["predicted_samples"] if native else schema["prediction"]
            filename = f"{job['job_id']}__seed{seed if seed is not None else 'unknown'}__take{index+1:03d}"
            row = {"job_id": job["job_id"], "model": job["model"], "profile": job["profile"], "case": job["case"],
                   "word_count": job["word_count"], "seed": seed, "take": index+1,
                   "speaker_source": job.get("speaker_source"),
                   "job_status": job["status"], "original_path": str(source), "predicted_samples": predicted,
                   "planner_seconds": schema["planner_seconds"], "planner_scope": PLANNER_SCOPE[job["model"]],
                   "generation_seconds": native["generation_seconds"] if native else None,
                   "gpu_slot": job.get("gpu_slot"), "gpu_uuid": job.get("gpu_uuid"),
                   "nvml_sampled_process_tree_peak_bytes": job.get("nvml_sampled_process_tree_peak_bytes"),
                   "nvml_sampled_device_peak_bytes": job.get("nvml_sampled_device_peak_bytes"),
                   "torch_peak_allocated_bytes": allocated, "torch_peak_reserved_bytes": reserved,
                   "actual_samples": None, "sample_rate": None, "channels": None,
                   "planned_sample_rate": schema["export_rate"], "sample_rate_match": None,
                   "actual_seconds": None, "predicted_seconds": predicted / schema["export_rate"] if predicted is not None else None,
                   "relative_duration_error": None, "duration_error_percent": None,
                   "exact_sample_count_match": None, "under_one_percent": None}
            try:
                info = sf.info(source)
                actual = info.frames
                actual_seconds = actual / info.samplerate
                planned_rate = schema["export_rate"]
                predicted_seconds = predicted / planned_rate if predicted is not None else None
                error = abs(actual_seconds-predicted_seconds) / actual_seconds if actual_seconds > 0 and predicted_seconds is not None else None
                is_temporary = "temporary" in source.name or bool(re.search(r"\.(?:tmp|part)\.wav$", source.name, re.I))
                valid = native is not None and error is not None and error < 0.01 and actual == predicted and info.samplerate == planned_rate and info.channels == 1 and not is_temporary
                partial = not valid
                destination = audio_dir / (("PARTIAL__" if partial else "") + filename + f"__sr{info.samplerate}.wav")
                shutil.copy2(source, destination)
                row.update({"audio_file": str(destination.relative_to(PACKAGE_DIR)), "actual_samples": actual,
                            "sample_rate": info.samplerate, "channels": info.channels,
                            "planned_sample_rate": planned_rate, "sample_rate_match": info.samplerate == planned_rate,
                            "actual_seconds": actual_seconds, "predicted_seconds": predicted_seconds,
                            "relative_duration_error": error, "duration_error_percent": error*100 if error is not None else None,
                            "exact_sample_count_match": actual == predicted if predicted is not None else None,
                            "under_one_percent": error < 0.01 if error is not None else None,
                            "audio_status": "partial" if partial else "verified", "error": None})
                if error is not None:
                    observed_errors.append(error)
                if valid:
                    final_count += 1
            except Exception as exc:
                destination = audio_dir / ("UNREADABLE__" + filename + ".wav")
                shutil.copy2(source, destination)
                row.update({"audio_file": str(destination.relative_to(PACKAGE_DIR)), "audio_status": "unreadable", "error": str(exc)})
            sample_rows.append(row)
        if job["status"] == "completed" and final_count != job["n"]:
            job["status"] = "report_validation_failed"
            job["error"] = f"Expected {job['n']} exact exports, independently verified {final_count}"
        for row in sample_rows:
            if row["job_id"] == job["job_id"]:
                row["job_status"] = job["status"]
        job_rows.append({"job_id": job["job_id"], "model": job["model"], "profile": job["profile"], "case": job["case"],
            "speaker_source": job.get("speaker_source"),
            "word_count": job["word_count"], "requested_samples": job["n"], "created_wavs": len(found), "verified_wavs": final_count,
            "status": job["status"], "error": job.get("error"), "gpu_slot": job.get("gpu_slot"), "gpu_uuid": job.get("gpu_uuid"),
            "exit_code": job.get("exit_code"), "process_wall_seconds": job.get("process_wall_seconds"),
            "planner_seconds": schema["planner_seconds"], "startup_seconds": schema["startup_seconds"],
            "planner_seconds_per_take_amortized": schema["planner_seconds"]/job["n"] if schema["planner_seconds"] is not None and job["model"] == "styletts2" else None,
            "planner_scope": PLANNER_SCOPE[job["model"]], "max_duration_error_percent": max(observed_errors)*100 if observed_errors else None,
            "planned_samples_json": json.dumps(schema["predictions"]), "planned_sample_rate": schema["export_rate"],
            "nvml_sampled_process_tree_peak_bytes": job.get("nvml_sampled_process_tree_peak_bytes"),
            "nvml_sampled_device_peak_bytes": job.get("nvml_sampled_device_peak_bytes"),
            "torch_peak_allocated_bytes": allocated, "torch_peak_reserved_bytes": reserved,
            "budget_violation": json.dumps(job["budget_violation"]) if job.get("budget_violation") else None,
            "monitor_error": job.get("monitor_error"), "trace_samples": job.get("samples"),
            "monitor_interval_ms": CONFIG["nvml_interval_ms"], "instantaneous_peak_certified": False,
            "native_manifest": str(manifest_path) if manifest_path else None})
    return job_rows, sample_rows

def run_quality_checks(sample_rows):
    """Independent CPU diagnostics; errors never prevent audio/report packaging."""
    quality_dir = RUN_DIR / "quality"
    quality_dir.mkdir(parents=True, exist_ok=True)
    output = quality_dir / "quality_report.json"
    if not CONFIG.get("quality_checks", False) or not sample_rows:
        result = {"status": "disabled" if sample_rows else "no_audio", "rows": []}
        atomic_json(output, result)
        return result
    jobs_by_id = {job["job_id"]: job for job in JOB_RESULTS}
    requests = []
    for row in sample_rows:
        job = jobs_by_id[row["job_id"]]
        text = (Path(job["job_dir"]) / "text.txt").read_text(encoding="utf-8")
        requests.append({"job_id": row["job_id"], "model": row["model"], "seed": row["seed"],
                         "audio_file": str((PACKAGE_DIR / row["audio_file"]).resolve()), "text": text})
    request_path = quality_dir / "requests.json"
    atomic_json(request_path, requests)
    log_path = quality_dir / "quality_check.log"
    setup_error = None
    try:
        python = ENV_DIR / "quality_checks" / "bin" / "python"
        if not python.is_file():
            setup_command(UV + ["venv", "--python", "3.11", "--seed", str(python.parent.parent)], log_path)
        install = UV + ["pip", "install", "--python", str(python), "--link-mode", "hardlink"]
        setup_command(install + ["numpy==1.26.4", "soundfile==0.13.1"], log_path)
        use_asr = bool(CONFIG.get("asr_quality_check", False))
        model = CONFIG.get("asr_model", "base.en")
        if use_asr:
            try:
                setup_command(install + ["faster-whisper==1.2.1", "ctranslate2==4.6.0",
                              "onnxruntime==1.23.2", "huggingface_hub==0.29.3", "av==15.1.0"], log_path)
                if model == "base.en":
                    model = str(ASSETS / "quality_asr" / "base.en")
                    downloader = "from huggingface_hub import snapshot_download; import sys; snapshot_download('Systran/faster-whisper-base.en', revision='3d3d5dee26484f91867d81cb899cfcf72b96be6c',local_dir=sys.argv[1],max_workers=1)"
                    setup_command([python, "-c", downloader, model], log_path)
            except Exception as exc:
                use_asr = False
                setup_error = str(exc)
                print("ASR setup failed; continuing with waveform diagnostics:", exc)
        command = [str(python), str(SCRIPTS_DIR / "audio_quality_check.py"), "--requests-json", str(request_path),
                   "--output-json", str(output), "--model", str(model), "--language", CONFIG.get("asr_language", "en"),
                   "--threads", str(CONFIG["cpu_threads_per_job"])]
        if use_asr:
            command.append("--asr")
        print(f"Checking {len(requests)} WAVs on CPU; ASR={use_asr}. Details: {log_path}")
        # Explicit CUDA hiding and CPU/int8 ASR keep diagnostics out of TTS VRAM/timing measurements.
        with log_path.open("a", encoding="utf-8") as log:
            subprocess.run(command, env=BASE_ENV, stdout=log, stderr=subprocess.STDOUT, check=True,
                           timeout=CONFIG.get("quality_timeout_seconds", 7200))
        result = read_json_safely(output)
        if not isinstance(result, dict):
            raise RuntimeError("Quality helper returned no valid report")
        result["asr_setup_error"] = setup_error
        result["asr_completed_files"] = sum(row.get("asr_status") == "completed" for row in result.get("rows", []))
        result["asr_failed_files"] = sum(row.get("asr_status") == "failed" for row in result.get("rows", []))
        if CONFIG.get("asr_model", "base.en") == "base.en" and use_asr:
            result["asr_weight_revision"] = "3d3d5dee26484f91867d81cb899cfcf72b96be6c"
        atomic_json(output, result)
    except Exception as exc:
        result = {"status": "failed", "error": str(exc), "rows": [], "asr_setup_error": setup_error}
        atomic_json(output, result)
        print("Quality diagnostics failed; WAVs and timing reports will still be packaged:", exc)
    lookup = {(r.get("job_id"), str(Path(r.get("audio_file", "")).resolve())): r for r in result.get("rows", [])}
    for row in sample_rows:
        diagnostics = lookup.get((row["job_id"], str((PACKAGE_DIR / row["audio_file"]).resolve())))
        if diagnostics:
            row["quality_status"] = diagnostics.get("status")
            for field in ("asr_status", "asr_error", "longest_interior_silence_seconds", "clipped_sample_fraction", "word_error_rate", "deletions", "insertions", "substitutions", "transcript"):
                row[field] = diagnostics.get(field)
    return result

def package_results(job_rows, sample_rows):
    reports = PACKAGE_DIR / "reports"
    reports.mkdir(parents=True, exist_ok=True)
    quality_report = run_quality_checks(sample_rows)
    atomic_json(reports / "quality.json", quality_report)
    pd.DataFrame(quality_report.get("rows", [])).to_csv(reports / "quality.csv", index=False)
    shutil.copytree(RUN_DIR / "quality", PACKAGE_DIR / "quality_details", dirs_exist_ok=True)
    jobs_frame, samples_frame = pd.DataFrame(job_rows), pd.DataFrame(sample_rows)
    jobs_frame.to_csv(reports / "jobs.csv", index=False)
    samples_frame.to_csv(reports / "samples.csv", index=False)
    atomic_json(reports / "jobs.json", job_rows)
    atomic_json(reports / "samples.json", sample_rows)
    atomic_json(reports / "experiment.json", {"config": CONFIG, "cases": TEXT_CASES, "profiles": MODEL_PROFILES,
                "hardware": GPU_INVENTORY, "setup": SETUP_RESULTS, "preflight": globals().get("PREFLIGHT_RESULTS", {}),
                "runtime_validation": "This run's data only; failures are included. Sampled NVML does not certify instantaneous peaks."})
    sources = PACKAGE_DIR / "source"
    sources.mkdir(exist_ok=True)
    for path in SCRIPTS_DIR.glob("*.py"):
        shutil.copy2(path, sources / path.name)
    if (SCRIPTS_DIR / "source_manifest.json").exists():
        shutil.copy2(SCRIPTS_DIR / "source_manifest.json", sources / "source_manifest.json")
    for name in ("hardware.json", "supervisor_results.json", "concurrency_timeline.json", "experiment_plan.json", "reference_report.json"):
        source = RUN_DIR / name
        if source.exists():
            shutil.copy2(source, reports / name)
    for source in (ROOT / "setup_report.json", ROOT / "preflight_report.json", ROOT / "reference_report.json", ROOT / "emotivoice_checkpoint_provenance.json"):
        if source.exists():
            shutil.copy2(source, reports / source.name)
    reference_root = ROOT / "reference"
    if reference_root.exists():
        for source in reference_root.rglob("*.json"):
            destination = reports / "reference_provenance" / source.relative_to(reference_root)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
    logs_dir = ROOT / "logs" / "setup"
    if logs_dir.exists():
        shutil.copytree(logs_dir, PACKAGE_DIR / "setup_logs", dirs_exist_ok=True)
    for job in JOB_RESULTS:
        base = Path(job["job_dir"])
        target = PACKAGE_DIR / "job_details" / job["job_id"]
        target.mkdir(parents=True, exist_ok=True)
        for source in base.rglob("*"):
            if source.is_file() and source.suffix.lower() in {".json", ".txt", ".log", ".csv", ".pt"}:
                # .pt here is the small StyleTTS2 cached duration plan, never model checkpoints.
                relative = source.relative_to(base)
                destination = target / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
    if CONFIG["include_reference_in_zip"] and globals().get("REFERENCE_AUDIO"):
        reference = Path(REFERENCE_AUDIO)
        if reference.exists():
            target = PACKAGE_DIR / "reference"
            target.mkdir(exist_ok=True)
            shutil.copy2(reference, target / reference.name)
            (target / "transcript.txt").write_text(REFERENCE_TEXT or "", encoding="utf-8")
    # A static report links to every named clip; suitable after unzipping.
    clips = []
    for row in sample_rows:
        href = html.escape(row["audio_file"], quote=True)
        label = html.escape(f"{row['model']} / {row['profile']} / {row['case']} / seed {row.get('seed')} / {row['audio_status']} / speaker: {row.get('speaker_source')}")
        clips.append(f'<p>{label}</p><audio controls preload="none" src="{href}"></audio><a href="{href}">WAV</a>')
    table = jobs_frame.to_html(index=False, escape=True) if not jobs_frame.empty else "<p>No jobs completed.</p>"
    figures_html = ""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        if job_rows:
            y = list(range(len(job_rows)))
            figure, axis = plt.subplots(figsize=(12, max(4, len(y)*0.42)))
            labels = [f"{r['model']}/{r['profile']}/{r['case']} GPU{r.get('gpu_slot')}" for r in job_rows]
            for field, offset, label in (("nvml_sampled_device_peak_bytes", -.24, "NVML device sampled"),
                                         ("nvml_sampled_process_tree_peak_bytes", 0, "NVML process tree sampled"),
                                         ("torch_peak_allocated_bytes", .24, "PyTorch allocator peak")):
                values = [r[field]/1e9 if r.get(field) is not None else float("nan") for r in job_rows]
                axis.barh([i+offset for i in y], values, height=.22, label=label)
            axis.axvline(CONFIG["max_vram_gb"], color="red", linestyle="--", label="Configured VRAM target")
            axis.set_yticks(y, labels)
            axis.invert_yaxis()
            axis.set_xlabel("GPU memory (decimal GB); absent measurements are blank")
            axis.legend(loc="best")
            figure.tight_layout()
            figure.savefig(reports / "vram_peaks.png", dpi=150)
            plt.close(figure)
            figures_html += '<h2>Memory measurements</h2><img style="max-width:100%" src="reports/vram_peaks.png">'
        timed = [r for r in job_rows if r.get("planner_seconds") is not None and r["planner_seconds"] > 0]
        if timed:
            figure, axis = plt.subplots(figsize=(10, 5))
            for model in sorted({r["model"] for r in timed}):
                rows = sorted([r for r in timed if r["model"] == model], key=lambda r:r["word_count"])
                axis.plot([r["word_count"] for r in rows], [r["planner_seconds"]*1000 for r in rows], "o", label=model)
            axis.set_yscale("log")
            axis.set_xlabel("Words")
            axis.set_ylabel("Measured planning time (ms, log scale)")
            axis.set_title("Model-specific timing scopes differ; StyleTTS2 includes all requested takes")
            axis.legend()
            axis.grid(True, alpha=.3)
            figure.tight_layout()
            figure.savefig(reports / "planning_latency.png", dpi=150)
            plt.close(figure)
            figures_html += '<h2>Planning latency</h2><img style="max-width:100%" src="reports/planning_latency.png">'
    except Exception as exc:
        atomic_json(reports / "plot_failure.json", {"error": str(exc)})
    quality_frame = pd.DataFrame(quality_report.get("rows", []))
    quality_columns = ["job_id", "model", "seed", "status", "asr_status", "asr_error",
                       "word_error_rate", "deletions", "insertions", "substitutions",
                       "longest_interior_silence_seconds", "clipped_sample_fraction", "transcript"]
    quality_table = (quality_frame.reindex(columns=[name for name in quality_columns if name in quality_frame.columns])
                     .to_html(index=False, escape=True) if not quality_frame.empty
                     else html.escape(str(quality_report.get("status"))))
    report_html = ('<!doctype html><meta charset="utf-8"><title>TTS benchmark</title>'
        '<style>body{font-family:system-ui;margin:24px}table{border-collapse:collapse;font-size:12px}td,th{padding:6px;border:1px solid #ddd}audio{width:500px;max-width:100%}</style>'
        '<h1>TTS timing and GPU memory benchmark</h1><p>NVML peaks are sampled; allocator peaks exclude driver/library memory. Failed and partial jobs are retained.</p>'
        + figures_html + table + '<h2>Word completeness and acoustic diagnostics</h2><p>ASR disagreements are screening results, not proof of a synthesis error. Silence can be intentional; inspect the transcript and listen.</p>'
        + quality_table + '<p>Full word alignments, silence spans, and ASR timestamps are in reports/quality.json.</p>'
        + '<h2>Recordings</h2>' + ''.join(clips))
    (PACKAGE_DIR / "index.html").write_text(report_html, encoding="utf-8")
    (PACKAGE_DIR / "README.txt").write_text(
        "Open index.html to compare reports and play recordings.\n"
        "reports/jobs.csv: every planned job, including setup/runtime/budget failures.\n"
        "reports/samples.csv: independent WAV header duration checks and canonical filenames.\n"
        "reports/quality.csv and quality.json: CPU acoustic checks, ASR transcripts and word edits; ASR is fallible.\n"
        "job_details/: exact config, text, native manifests, progress logs and NVML time series.\n"
        "source/: embedded standalone adapters and SHA256 provenance.\n"
        "PARTIAL/UNREADABLE filenames are not verified final samples.\n"
        "Audio length accuracy does not establish text intelligibility or natural speaking time.\n"
        "Dependency environments and model weights are excluded from this archive.\n", encoding="utf-8")
    archive = ROOT.parent / f"tts_benchmark_{RUN_ID}.zip"
    files = [path for path in PACKAGE_DIR.rglob("*") if path.is_file()]
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as zipped:
        for source in tqdm(files, desc="Packaging WAVs and reports", unit="file"):
            zipped.write(source, source.relative_to(PACKAGE_DIR))
    return jobs_frame, samples_frame, archive

if KERNEL_READY:
    job_rows, sample_rows = collect_reports()
    JOBS_TABLE, SAMPLES_TABLE, ZIP_PATH = package_results(job_rows, sample_rows)
    print(f"ZIP: {ZIP_PATH} ({ZIP_PATH.stat().st_size/1e6:.1f} MB)")
    display(FileLink(str(ZIP_PATH)))
    display(FileLink(str(PACKAGE_DIR / "index.html")))
    if not JOBS_TABLE.empty:
        display(JOBS_TABLE.reindex(columns=["model", "profile", "word_count", "status", "verified_wavs", "planner_seconds", "max_duration_error_percent", "nvml_sampled_process_tree_peak_bytes", "nvml_sampled_device_peak_bytes"]))
    if not SAMPLES_TABLE.empty:
        display(SAMPLES_TABLE.reindex(columns=["model", "profile", "word_count", "seed", "speaker_source", "predicted_seconds", "actual_seconds", "duration_error_percent", "longest_interior_silence_seconds", "word_error_rate", "audio_status", "audio_file"]))
    played_models = set()
    for row in sample_rows:
        if len(played_models) >= CONFIG["preview_players"]:
            break
        if row["audio_status"] == "verified" and row["model"] not in played_models:
            print(row["model"], row["profile"], row["case"], "seed", row["seed"])
            display(Audio(filename=str(PACKAGE_DIR / row["audio_file"])))
            played_models.add(row["model"])
else:
    ZIP_PATH = ROOT.parent / f"tts_benchmark_{RUN_ID}_failure.zip"
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
        for file in RUN_DIR.rglob("*.json"):
            archive.write(file, file.relative_to(RUN_DIR))
        for file in ROOT.glob("*report.json"):
            archive.write(file, "reports/" + file.name)
    print("Kernel dependency setup failed. Diagnostic ZIP saved:", ZIP_PATH)
    from IPython.display import FileLink, display
    display(FileLink(str(ZIP_PATH)))


## Primary references

- [Kaggle notebook settings](https://www.kaggle.com/docs/notebooks)
- [Kaggle accelerator identifiers](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md)
- [ONNX Runtime CUDA requirements and arena limits](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html)
- [PyTorch CUDA wheel versions](https://pytorch.org/get-started/previous-versions/)
- [Kokoro](https://github.com/hexgrad/kokoro/tree/dfb907a02bba8152ca444717ca5d78747ccb4bec)
- [F5-TTS](https://github.com/SWivid/F5-TTS/tree/283252563dbf91be625e0c27926acfaac449186c)
- [EmotiVoice](https://github.com/netease-youdao/EmotiVoice/tree/59f0f36de4db12825f4705dd4e0780d79dd6bb01)
- [StyleTTS2](https://github.com/yl4579/StyleTTS2/tree/5cedc71c333f8d8b8551ca59378bdcc7af4c9529)
- [OmniVoice](https://github.com/k2-fsa/OmniVoice/tree/08be0b4ccbac3e13e374e86fbfead4b4cac343e2)
- [Supertonic 3](https://github.com/supertone-oss-archive/supertonic/tree/1e9799e964ea4c0dad7cde993b65c3c813a7b373)
- [Supertonic speaker descriptions](https://github.com/supertone-oss-archive/supertonic-py/blob/main/docs/voices.md)
- [EmotiVoice speaker descriptions](https://github.com/netease-youdao/EmotiVoice/tree/main/data/youdao/text)
- [OmniVoice design attributes](https://github.com/k2-fsa/OmniVoice/blob/08be0b4ccbac3e13e374e86fbfead4b4cac343e2/docs/voice-design.md)
- [CPU ASR implementation](https://github.com/SYSTRAN/faster-whisper)

Use pretrained model licenses appropriate to your intended use; for example,
F5's published checkpoint is CC-BY-NC-4.0. This notebook is a benchmarking tool.
